# Phase 1: Native XML/XSD Environment & Metadata Initialization

**Objective:** Initialize the runtime for **System 100**, establishing the native DrugBank XML dataset (`drugbank.xml`) and its schema (`drugbank.xsd`) as the primary chemical knowledge source without external database dependencies.

* **Inputs & Configuration:**
  * Files: `./db/drugbank.xml` and `./db/drugbank.xsd`
  * Execution: Seed = `42`, device allocation (CUDA / CPU)
  * Core Parameters: Latent dimension $d = 32$, UCB coefficient $\kappa = 1.0$, fingerprint size $F_{\text{fp}} = 256$

* **Execution Steps:**
  1. Set deterministic pseudo-random seeds across Python, NumPy, and PyTorch.
  2. Verify file presence and byte integrity for `drugbank.xml` and `drugbank.xsd`.
  3. Probe the root `<drugbank>` tag using `iterparse` to read version, export date, and XML namespace with zero memory overhead.

* **Expected Output:**
  Validated hardware runtime and schema metadata confirming readiness for streaming chemical ingestion.

In [1]:
# ==============================================================================
# PHASE 1 — XML ENVIRONMENT AND DATASET INITIALIZATION
# Native DrugBank XML + XSD | Zero MySQL Runtime Dependency
# ==============================================================================

import os
import sys
import random
import warnings
from pathlib import Path

import numpy as np
import torch
from lxml import etree


# ==============================================================================
# 1. GLOBAL ENVIRONMENT CONFIGURATION
# ==============================================================================

warnings.filterwarnings("ignore")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(SEED)

    # Reproducibility settings
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    DEVICE = torch.device("cuda")

else:

    DEVICE = torch.device("cpu")


# ==============================================================================
# 2. ARCHITECTURAL / EXPERIMENTAL PARAMETERS
# ==============================================================================

# These parameters are DECLARED in Phase 1.
# They are NOT used for extraction/training in this phase.

LATENT_DIM = 32           # Latent molecular representation dimension
HIDDEN_DIM = 64           # EGNN hidden feature dimension
FP_DIM = 256              # Morgan fingerprint dimension

MAX_MOLECULES = 800       # Initial experimental corpus size
MIN_HEAVY_ATOMS = 4       # Minimum molecular size
MAX_HEAVY_ATOMS = 80      # Maximum molecular size

KAPPA_UCB = 1.0           # UCB exploration coefficient

# Number of closed-loop cycles will be used later
NUM_CYCLES = 3


# ==============================================================================
# 3. PROJECT DIRECTORY CONFIGURATION
# ==============================================================================

# Notebook expected to run from:
#
# E:\MyPhD\My Patent-1\Implementation
#
# Therefore Path.cwd() should resolve to the Implementation directory.

BASE_DIR = Path.cwd()

DB_DIR = BASE_DIR / "db"
DATA_DIR = BASE_DIR / "data"
MODEL_DIR = BASE_DIR / "models"
OUTPUT_DIR = BASE_DIR / "outputs"

# Data subdirectories
EXTRACTED_DIR = DATA_DIR / "extracted"
PROCESSED_DIR = DATA_DIR / "processed"
GRAPH_DIR = DATA_DIR / "graphs"
FEEDBACK_DIR = DATA_DIR / "feedback"

# Model subdirectories
CHECKPOINT_DIR = MODEL_DIR / "checkpoints"

# Output subdirectories
METRICS_DIR = OUTPUT_DIR / "metrics"
FIGURES_DIR = OUTPUT_DIR / "figures"
LOGS_DIR = OUTPUT_DIR / "logs"
CANDIDATE_DIR = OUTPUT_DIR / "candidates"


# ==============================================================================
# 4. CREATE PROJECT DIRECTORY STRUCTURE
# ==============================================================================

PROJECT_DIRECTORIES = [

    DATA_DIR,
    EXTRACTED_DIR,
    PROCESSED_DIR,
    GRAPH_DIR,
    FEEDBACK_DIR,

    MODEL_DIR,
    CHECKPOINT_DIR,

    OUTPUT_DIR,
    METRICS_DIR,
    FIGURES_DIR,
    LOGS_DIR,
    CANDIDATE_DIR
]

for directory in PROJECT_DIRECTORIES:
    directory.mkdir(parents=True, exist_ok=True)


# ==============================================================================
# 5. DRUGBANK SOURCE FILE CONFIGURATION
# ==============================================================================

XML_PATH = DB_DIR / "drugbank.xml"
XSD_PATH = DB_DIR / "drugbank.xsd"

# Historical SQL files are retained ONLY for documentation/comparison.
# They are NOT part of the execution pipeline.

SQL_METADATA_PATH = DB_DIR / "DrugBank_MetaData.sql"
SQL_SCHEMA_PATH = DB_DIR / "DrugBank_Table_Schema.sql"


# ==============================================================================
# 6. ENVIRONMENT SUMMARY
# ==============================================================================

print("=" * 95)
print("PHASE 1 — XML ENVIRONMENT AND DATASET INITIALIZATION")
print("=" * 95)

print("\n[ENVIRONMENT]")

print(f"Python Version       : {sys.version.split()[0]}")
print(f"PyTorch Version      : {torch.__version__}")
print(f"Random Seed          : {SEED}")
print(f"Compute Device       : {DEVICE}")

if torch.cuda.is_available():

    print(f"GPU                   : {torch.cuda.get_device_name(0)}")

    total_memory = (
        torch.cuda.get_device_properties(0).total_memory
        / (1024 ** 3)
    )

    print(f"GPU Memory            : {total_memory:.2f} GB")

else:

    print("GPU                   : Not available")


# ==============================================================================
# 7. PROJECT PATH VERIFICATION
# ==============================================================================

print("\n" + "=" * 95)
print("PROJECT CONFIGURATION")
print("=" * 95)

print(f"Project Root          : {BASE_DIR}")
print(f"DrugBank Directory    : {DB_DIR}")

print(f"\nXML Source             : {XML_PATH}")
print(f"XSD Schema             : {XSD_PATH}")

print(f"\nExtracted Data         : {EXTRACTED_DIR}")
print(f"Processed Data         : {PROCESSED_DIR}")
print(f"Graph Dataset          : {GRAPH_DIR}")
print(f"Feedback Dataset       : {FEEDBACK_DIR}")

print(f"\nModel Directory        : {MODEL_DIR}")
print(f"Checkpoint Directory   : {CHECKPOINT_DIR}")

print(f"\nMetrics Directory      : {METRICS_DIR}")
print(f"Figures Directory      : {FIGURES_DIR}")
print(f"Candidate Directory    : {CANDIDATE_DIR}")


# ==============================================================================
# 8. REQUIRED FILE VERIFICATION
# ==============================================================================

print("\n" + "=" * 95)
print("DRUGBANK SOURCE FILE VERIFICATION")
print("=" * 95)

required_files = {

    "DrugBank XML": XML_PATH,
    "DrugBank XSD": XSD_PATH
}

missing_required_files = []


for label, path in required_files.items():

    if path.exists():

        size_bytes = path.stat().st_size

        size_mb = size_bytes / (1024 ** 2)
        size_gb = size_bytes / (1024 ** 3)

        if size_gb >= 1:

            size_string = f"{size_gb:.3f} GB"

        else:

            size_string = f"{size_mb:.3f} MB"


        print(f"[FOUND] {label}")

        print(f"        Path : {path}")
        print(f"        Size : {size_string}")

    else:

        print(f"[MISSING] {label}")

        print(f"          Expected Path: {path}")

        missing_required_files.append(label)


if missing_required_files:

    raise FileNotFoundError(

        "Required DrugBank files are missing: "
        + ", ".join(missing_required_files)
    )


# ==============================================================================
# 9. HISTORICAL SQL FILE CHECK
# ==============================================================================

print("\n" + "=" * 95)
print("HISTORICAL DATABASE FILES — DOCUMENTATION ONLY")
print("=" * 95)

historical_files = {

    "DrugBank Metadata SQL": SQL_METADATA_PATH,
    "DrugBank Table Schema SQL": SQL_SCHEMA_PATH
}


for label, path in historical_files.items():

    if path.exists():

        size_mb = path.stat().st_size / (1024 ** 2)

        print(
            f"[REFERENCE ONLY] {label:<30} "
            f"{size_mb:.2f} MB"
        )

    else:

        print(
            f"[NOT PRESENT]   {label}"
        )


print("\n[DATABASE BACKEND]")
print("MySQL Runtime          : DISABLED")
print("SQL Import             : DISABLED")
print("SQL Query Execution    : DISABLED")
print("SQL Joins              : DISABLED")

print("\nPrimary Data Source    : DrugBank XML")
print("Schema Specification  : DrugBank XSD")


# ==============================================================================
# 10. MEMORY-SAFE DRUGBANK XML ROOT INSPECTION
# ==============================================================================

print("\n" + "=" * 95)
print("DRUGBANK XML ROOT & RELEASE METADATA")
print("=" * 95)

# Explicitly open the file using a context manager.
#
# This guarantees deterministic file-handle closure on Windows after
# reading the first XML start event.

with open(XML_PATH, "rb") as xml_file:

    xml_context = etree.iterparse(

        xml_file,

        events=("start",),

        huge_tree=True
    )

    event, root = next(xml_context)

    ROOT_TAG_FULL = root.tag

    ROOT_ATTRIBUTES = dict(root.attrib)

    # Delete parser references while file context is still controlled
    del xml_context


# ==============================================================================
# 11. DETECT XML ROOT ELEMENT AND NAMESPACE
# ==============================================================================

if ROOT_TAG_FULL.startswith("{"):

    XML_NAMESPACE = ROOT_TAG_FULL.split("}")[0][1:]

    ROOT_ELEMENT = ROOT_TAG_FULL.split("}")[1]

else:

    XML_NAMESPACE = ""

    ROOT_ELEMENT = ROOT_TAG_FULL


# Namespace dictionary for future XPath/XML processing
NS = {

    "db": XML_NAMESPACE
}


# Fully qualified namespace prefix
#
# Example:
#
# {http://www.drugbank.ca}drug

NS_TAG = (

    f"{{{XML_NAMESPACE}}}"

    if XML_NAMESPACE

    else ""
)


# ==============================================================================
# 12. DRUGBANK RELEASE METADATA
# ==============================================================================

DRUGBANK_VERSION = ROOT_ATTRIBUTES.get(

    "version",

    "Not specified"
)


DRUGBANK_EXPORTED_ON = ROOT_ATTRIBUTES.get(

    "exported-on",

    "Not specified"
)


print(f"Root Element           : {ROOT_ELEMENT}")
print(f"XML Namespace          : {XML_NAMESPACE}")

print(f"DrugBank Version       : {DRUGBANK_VERSION}")
print(f"DrugBank Export Date   : {DRUGBANK_EXPORTED_ON}")


if ROOT_ATTRIBUTES:

    print("\nRoot Attributes:")

    for key, value in ROOT_ATTRIBUTES.items():

        print(
            f"  {key:<20}: {value}"
        )


# ==============================================================================
# 13. XSD SCHEMA INSPECTION
# ==============================================================================

print("\n" + "=" * 95)
print("DRUGBANK XSD SCHEMA METADATA")
print("=" * 95)

try:

    xsd_tree = etree.parse(

        str(XSD_PATH)
    )

    xsd_root = xsd_tree.getroot()


    XSD_ROOT_ELEMENT = etree.QName(

        xsd_root

    ).localname


    XSD_TARGET_NAMESPACE = (

        xsd_root.attrib.get(

            "targetNamespace",

            "Not specified"
        )
    )


    XSD_ELEMENT_FORM_DEFAULT = (

        xsd_root.attrib.get(

            "elementFormDefault",

            "Not specified"
        )
    )


    XSD_ATTRIBUTE_FORM_DEFAULT = (

        xsd_root.attrib.get(

            "attributeFormDefault",

            "Not specified"
        )
    )


    print(f"XSD Root Element       : {XSD_ROOT_ELEMENT}")

    print(
        f"Target Namespace       : "
        f"{XSD_TARGET_NAMESPACE}"
    )

    print(
        f"Element Form Default   : "
        f"{XSD_ELEMENT_FORM_DEFAULT}"
    )

    print(
        f"Attribute Form Default : "
        f"{XSD_ATTRIBUTE_FORM_DEFAULT}"
    )


except Exception as exc:

    raise RuntimeError(

        f"Unable to parse DrugBank XSD: {exc}"
    )


# ==============================================================================
# 14. XML ↔ XSD NAMESPACE CONSISTENCY VERIFICATION
# ==============================================================================

print("\n" + "=" * 95)
print("XML ↔ XSD CONSISTENCY CHECK")
print("=" * 95)


if XML_NAMESPACE == XSD_TARGET_NAMESPACE:

    NAMESPACE_MATCH = True

    print(
        "[PASS] XML namespace matches "
        "the XSD target namespace."
    )

else:

    NAMESPACE_MATCH = False

    print(
        "[WARNING] XML namespace does NOT "
        "match XSD target namespace."
    )

    print(
        f"XML Namespace          : {XML_NAMESPACE}"
    )

    print(
        f"XSD Target Namespace   : "
        f"{XSD_TARGET_NAMESPACE}"
    )


# ==============================================================================
# 15. PACKAGE RUNTIME / ARCHITECTURAL CONFIGURATION
# ==============================================================================

DRUGBANK_CONFIG = {

    # --------------------------------------------------------------------------
    # Environment
    # --------------------------------------------------------------------------

    "random_seed": SEED,

    "device": str(DEVICE),


    # --------------------------------------------------------------------------
    # Data sources
    # --------------------------------------------------------------------------

    "project_root": str(BASE_DIR),

    "xml_path": str(XML_PATH),

    "xsd_path": str(XSD_PATH),

    "primary_data_source": "DrugBank XML",

    "schema_source": "DrugBank XSD",

    "database_backend": None,


    # --------------------------------------------------------------------------
    # XML / XSD metadata
    # --------------------------------------------------------------------------

    "xml_root_element": ROOT_ELEMENT,

    "xml_namespace": XML_NAMESPACE,

    "xsd_target_namespace": XSD_TARGET_NAMESPACE,

    "namespace_match": NAMESPACE_MATCH,

    "drugbank_version": DRUGBANK_VERSION,

    "drugbank_exported_on": DRUGBANK_EXPORTED_ON,


    # --------------------------------------------------------------------------
    # Molecular dataset parameters
    # --------------------------------------------------------------------------

    "max_molecules": MAX_MOLECULES,

    "min_heavy_atoms": MIN_HEAVY_ATOMS,

    "max_heavy_atoms": MAX_HEAVY_ATOMS,


    # --------------------------------------------------------------------------
    # Model architecture
    # --------------------------------------------------------------------------

    "latent_dim": LATENT_DIM,

    "hidden_dim": HIDDEN_DIM,

    "fp_dim": FP_DIM,


    # --------------------------------------------------------------------------
    # Closed-loop configuration
    # --------------------------------------------------------------------------

    "kappa_ucb": KAPPA_UCB,

    "num_cycles": NUM_CYCLES
}


# ==============================================================================
# 16. ARCHITECTURAL PARAMETER SUMMARY
# ==============================================================================

print("\n" + "=" * 95)
print("EXPERIMENTAL CONFIGURATION")
print("=" * 95)

print(f"Maximum Molecules      : {MAX_MOLECULES}")

print(
    f"Heavy Atom Range       : "
    f"{MIN_HEAVY_ATOMS} – {MAX_HEAVY_ATOMS}"
)

print(f"Latent Dimension       : {LATENT_DIM}")

print(f"EGNN Hidden Dimension  : {HIDDEN_DIM}")

print(f"Fingerprint Dimension  : {FP_DIM}")

print(f"UCB Kappa              : {KAPPA_UCB}")

print(f"Planned Loop Cycles    : {NUM_CYCLES}")


# ==============================================================================
# 17. FINAL PHASE-1 VALIDATION
# ==============================================================================

print("\n" + "=" * 95)
print("PHASE 1 INITIALIZATION SUMMARY")
print("=" * 95)

print("[READY] DrugBank XML source identified.")

print("[READY] DrugBank XSD schema identified.")

print("[READY] XML namespace discovered.")

print("[READY] DrugBank release metadata captured.")

print("[READY] XSD metadata captured.")

print("[READY] XML ↔ XSD namespace verified.")

print("[READY] Project directories initialized.")

print("[READY] Reproducibility seed established.")

print("[READY] Hardware compute device established.")

print("[READY] Runtime configuration packaged.")

print("[READY] MySQL dependency eliminated.")


print("\n" + "-" * 95)

print(
    f"Primary Source          : DrugBank XML"
)

print(
    f"Schema                  : DrugBank XSD"
)

print(
    f"Namespace               : {XML_NAMESPACE}"
)

print(
    f"DrugBank Version        : {DRUGBANK_VERSION}"
)

print(
    f"Export Date             : {DRUGBANK_EXPORTED_ON}"
)

print(
    f"Compute Device          : {DEVICE}"
)

print(
    f"Namespace Validation    : "
    f"{'PASS' if NAMESPACE_MATCH else 'WARNING'}"
)

print("-" * 95)


if NAMESPACE_MATCH:

    print(
        "\n[STATUS] PHASE 1 COMPLETED SUCCESSFULLY."
    )

    print(
        "[NEXT]   Phase 2 — XSD-Guided XML Structure Analysis"
    )

else:

    print(
        "\n[STATUS] PHASE 1 COMPLETED WITH A NAMESPACE WARNING."
    )

    print(
        "[ACTION] Verify XML/XSD compatibility before Phase 2."
    )


print("=" * 95)

PHASE 1 — XML ENVIRONMENT AND DATASET INITIALIZATION

[ENVIRONMENT]
Python Version       : 3.14.4
PyTorch Version      : 2.11.0+cpu
Random Seed          : 42
Compute Device       : cpu
GPU                   : Not available

PROJECT CONFIGURATION
Project Root          : E:\MyPhD\My Patent-1\Implementation
DrugBank Directory    : E:\MyPhD\My Patent-1\Implementation\db

XML Source             : E:\MyPhD\My Patent-1\Implementation\db\drugbank.xml
XSD Schema             : E:\MyPhD\My Patent-1\Implementation\db\drugbank.xsd

Extracted Data         : E:\MyPhD\My Patent-1\Implementation\data\extracted
Processed Data         : E:\MyPhD\My Patent-1\Implementation\data\processed
Graph Dataset          : E:\MyPhD\My Patent-1\Implementation\data\graphs
Feedback Dataset       : E:\MyPhD\My Patent-1\Implementation\data\feedback

Model Directory        : E:\MyPhD\My Patent-1\Implementation\models
Checkpoint Directory   : E:\MyPhD\My Patent-1\Implementation\models\checkpoints

Metrics Directory      : 

### Phase - 2 


# Phase 2 — XSD-Guided XML Structure Analysis

**Objective:** Inspect the authoritative DrugBank schema (`drugbank.xsd`) programmatically to establish structural element hierarchies, attribute scopes, and field multiplicities required for downstream pipeline execution without performing full XML extraction or model training[cite: 1].

* **Key Structural Mappings:**
  * **Baseline Chemical Corpus ($\mathcal{D}_{\text{chem}}$):** Maps to `<drug type="small molecule">`[cite: 1] with child elements `drugbank-id[@primary='true']`, `name`, and `<calculated-properties>` $\rightarrow$ `<property>` (`kind="SMILES"`, `InChI`, `Molecular Formula`, `Molecular Weight`)[cite: 1].
  * **Biological Association Layer ($\mathcal{D}_{\text{target}}$):** Maps to `<targets>` $\rightarrow$ `<target>` (`id`, `name`, `organism`, `known-action`, `actions`) and bound `<polypeptide>` (`id`, `source`, `<amino-acid-sequence>`)[cite: 1].
  * **Biological Context Layers:** Identifies `<enzymes>`, `<carriers>`, `<transporters>`, and `<pathways>` hierarchy[cite: 1].

* **Execution Steps:**
  1. Parse and compile `drugbank.xsd` using `lxml.etree` to validate root structure and complex types (`drug-type`, `target-list-type`, `interactant-group`)[cite: 1].
  2. Inspect direct `<drug>` attributes (`type`, `created`, `updated`) while separating them from nested element attributes[cite: 1].
  3. Extract allowed enumerations for calculated-property kinds to confirm exact casing for `SMILES` and related descriptors[cite: 1].
  4. Perform a bounded spot-check across representative XML records to verify empirical compatibility[cite: 1].
  5. Export schema verification reports and field mapping summaries to `./outputs/schema/`[cite: 1].

* **Expected Output:**
  Validated schema-to-dataset specifications confirming $100\%$ XSD-verified fields for $\mathcal{D}_{\text{chem}}$ and $\mathcal{D}_{\text{target}}$, preparing the environment for Phase 3 streaming extraction[cite: 1].


    

In [2]:
# ==============================================================================
# PHASE 2 — XSD-GUIDED XML STRUCTURE ANALYSIS
# ==============================================================================
#
# PURPOSE
# -------
# 1. Parse and compile the DrugBank XSD.
# 2. Traverse the XSD hierarchically from:
#
#       drugbank -> drug -> nested structures
#
# 3. Resolve:
#       - named complex types
#       - inline complex types
#       - xs:sequence
#       - xs:choice
#       - xs:all
#       - xs:group references (important for DrugBank interactant structures)
#       - xs:complexContent / xs:extension
#       - xs:attributeGroup
#
# 4. Distinguish direct/inherited attributes from attributes of nested children.
# 5. Determine minOccurs / maxOccurs.
# 6. Discover allowed XSD enumerations for calculated-property kinds.
# 7. Verify fields required for:
#
#       D_chem
#       D_target
#
# 8. Perform only a LIMITED XML spot-check.
# 9. Save documented Phase-2 structural reports.
#
# IMPORTANT
# ---------
# This phase DOES NOT:
#   - extract the entire 1.5 GB DrugBank XML
#   - standardize SMILES
#   - run RDKit preprocessing
#   - construct molecular graphs
#   - train neural networks
#   - calculate QED/SA
#   - perform UCB/EWC
#
# Those tasks belong to later phases.
# ==============================================================================


# ==============================================================================
# 1. IMPORTS
# ==============================================================================

from pathlib import Path
from lxml import etree
import pandas as pd
import json


# ==============================================================================
# 2. VERIFY PHASE-1 STATE
# ==============================================================================

required_phase1_variables = [
    "XSD_PATH",
    "XML_PATH",
    "XML_NAMESPACE",
    "NS",
    "NS_TAG",
    "DRUGBANK_CONFIG",
    "OUTPUT_DIR"
]

missing_variables = [
    variable
    for variable in required_phase1_variables
    if variable not in globals()
]

if missing_variables:

    raise RuntimeError(
        "Phase 1 must be executed before Phase 2.\n"
        f"Missing Phase-1 variables: {missing_variables}"
    )


print("=" * 110)
print("PHASE 2 — XSD-GUIDED XML STRUCTURE ANALYSIS")
print("=" * 110)

print(f"\n[XSD] Schema File       : {XSD_PATH}")
print(f"[XML] Dataset File      : {XML_PATH}")
print(f"[XML] Namespace         : {XML_NAMESPACE}")


# ==============================================================================
# 3. XSD NAMESPACE
# ==============================================================================

XSD_NS_URI = "http://www.w3.org/2001/XMLSchema"

XSD_NS = {
    "xs": XSD_NS_URI
}


# ==============================================================================
# 4. PARSE AND COMPILE DRUGBANK XSD
# ==============================================================================

try:

    xsd_tree = etree.parse(
        str(XSD_PATH)
    )

    xsd_root = xsd_tree.getroot()

    # Compile the schema itself.
    #
    # This validates that the XSD is structurally usable by lxml.
    compiled_xsd = etree.XMLSchema(
        xsd_tree
    )

    print("\n[PASS] DrugBank XSD parsed successfully.")
    print("[PASS] DrugBank XSD compiled successfully.")

except Exception as exc:

    raise RuntimeError(
        "DrugBank XSD parsing/compilation failed:\n"
        f"{exc}"
    )


# ==============================================================================
# 5. XSD ROOT METADATA
# ==============================================================================

XSD_TARGET_NAMESPACE = xsd_root.attrib.get(
    "targetNamespace",
    "Not specified"
)

XSD_ELEMENT_FORM_DEFAULT = xsd_root.attrib.get(
    "elementFormDefault",
    "Not specified"
)

XSD_ATTRIBUTE_FORM_DEFAULT = xsd_root.attrib.get(
    "attributeFormDefault",
    "Not specified"
)


print("\n" + "=" * 110)
print("XSD ROOT METADATA")
print("=" * 110)

print(f"Target Namespace        : {XSD_TARGET_NAMESPACE}")
print(f"Element Form Default    : {XSD_ELEMENT_FORM_DEFAULT}")
print(f"Attribute Form Default  : {XSD_ATTRIBUTE_FORM_DEFAULT}")


# ==============================================================================
# 6. XML ↔ XSD NAMESPACE CONSISTENCY
# ==============================================================================

print("\n" + "=" * 110)
print("XML ↔ XSD NAMESPACE CONSISTENCY")
print("=" * 110)


NAMESPACE_MATCH = (
    XML_NAMESPACE == XSD_TARGET_NAMESPACE
)

if NAMESPACE_MATCH:

    print(
        "[PASS] DrugBank XML namespace matches "
        "the XSD target namespace."
    )

else:

    print(
        "[FAIL] XML namespace does not match "
        "the XSD target namespace."
    )

    print(f"XML Namespace           : {XML_NAMESPACE}")
    print(f"XSD Namespace           : {XSD_TARGET_NAMESPACE}")


# ==============================================================================
# 7. GENERAL XSD HELPER FUNCTIONS
# ==============================================================================

def qname_local_name(value):
    """
    Return the local component of a QName.

    Examples
    --------
    xs:string -> string
    db:drug-type -> drug-type
    drug-type -> drug-type
    """

    if value is None:
        return None

    if value.startswith("{"):
        return etree.QName(value).localname

    if ":" in value:
        return value.split(":", 1)[1]

    return value


def normalize_occurs(value, default=1):
    """
    Return readable XSD occurrence value.
    """

    if value is None:
        return str(default)

    return str(value)


def get_global_element(name):
    """
    Find only a TOP-LEVEL XSD element.

    This deliberately avoids a global .// search.
    """

    for element in xsd_root.findall(
        "xs:element",
        namespaces=XSD_NS
    ):

        if element.get("name") == name:
            return element

    return None


def get_global_complex_type(type_name):
    """
    Resolve a named global xs:complexType.
    """

    local_name = qname_local_name(
        type_name
    )

    if local_name is None:
        return None

    for complex_type in xsd_root.findall(
        "xs:complexType",
        namespaces=XSD_NS
    ):

        if complex_type.get("name") == local_name:
            return complex_type

    return None


def get_global_simple_type(type_name):
    """
    Resolve a named global xs:simpleType.
    """

    local_name = qname_local_name(
        type_name
    )

    if local_name is None:
        return None

    for simple_type in xsd_root.findall(
        "xs:simpleType",
        namespaces=XSD_NS
    ):

        if simple_type.get("name") == local_name:
            return simple_type

    return None


def get_global_group(group_name):
    """
    Resolve a named global xs:group.
    """

    local_name = qname_local_name(
        group_name
    )

    if local_name is None:
        return None

    for group in xsd_root.findall(
        "xs:group",
        namespaces=XSD_NS
    ):

        if group.get("name") == local_name:
            return group

    return None


def get_global_attribute_group(group_name):
    """
    Resolve a named global xs:attributeGroup.
    """

    local_name = qname_local_name(
        group_name
    )

    if local_name is None:
        return None

    for group in xsd_root.findall(
        "xs:attributeGroup",
        namespaces=XSD_NS
    ):

        if group.get("name") == local_name:
            return group

    return None


# ==============================================================================
# 8. ELEMENT RESOLUTION
# ==============================================================================

def resolve_element_reference(element):
    """
    Resolve xs:element ref="..." when present.

    Keeps the original element available for occurrence constraints while
    resolving the referenced element definition.
    """

    if element is None:
        return None

    ref = element.get("ref")

    if not ref:
        return element

    ref_name = qname_local_name(
        ref
    )

    referenced = get_global_element(
        ref_name
    )

    return referenced if referenced is not None else element


def get_element_name(element):
    """
    Obtain the effective element name.
    """

    if element is None:
        return None

    name = element.get("name")

    if name:
        return name

    ref = element.get("ref")

    if ref:
        return qname_local_name(ref)

    return None


def get_element_complex_type(element):
    """
    Resolve the effective complex type of an element.
    """

    if element is None:
        return None

    resolved_element = resolve_element_reference(
        element
    )

    # Inline complex type
    inline_complex = resolved_element.find(
        "xs:complexType",
        namespaces=XSD_NS
    )

    if inline_complex is not None:
        return inline_complex

    # Named type
    type_name = resolved_element.get(
        "type"
    )

    if type_name:
        return get_global_complex_type(
            type_name
        )

    return None


def get_element_simple_type(element):
    """
    Resolve simple type associated with an element.
    """

    if element is None:
        return None

    resolved_element = resolve_element_reference(
        element
    )

    inline_simple = resolved_element.find(
        "xs:simpleType",
        namespaces=XSD_NS
    )

    if inline_simple is not None:
        return inline_simple

    type_name = resolved_element.get(
        "type"
    )

    if type_name:
        return get_global_simple_type(
            type_name
        )

    return None


# ==============================================================================
# 9. PARTICLE / GROUP TRAVERSAL
# ==============================================================================

def collect_elements_from_particle(
    particle,
    visited_groups=None
):
    """
    Collect child elements from an XSD particle.

    Supports:
        xs:sequence
        xs:choice
        xs:all
        xs:group ref

    This is intentionally focused on constructs needed by DrugBank rather
    than attempting to implement the entire W3C XML Schema specification.
    """

    if visited_groups is None:
        visited_groups = set()

    if particle is None:
        return []

    results = []

    local_tag = etree.QName(
        particle
    ).localname


    # --------------------------------------------------------------------------
    # Direct element
    # --------------------------------------------------------------------------

    if local_tag == "element":

        results.append(
            particle
        )

        return results


    # --------------------------------------------------------------------------
    # Sequence / choice / all
    # --------------------------------------------------------------------------

    if local_tag in {
        "sequence",
        "choice",
        "all"
    }:

        for child in particle:

            if not isinstance(
                child.tag,
                str
            ):
                continue

            child_local = etree.QName(
                child
            ).localname

            if child_local in {
                "element",
                "sequence",
                "choice",
                "all",
                "group"
            }:

                results.extend(
                    collect_elements_from_particle(
                        child,
                        visited_groups
                    )
                )

        return results


    # --------------------------------------------------------------------------
    # Group reference
    # --------------------------------------------------------------------------

    if local_tag == "group":

        group_ref = particle.get(
            "ref"
        )

        if group_ref:

            group_name = qname_local_name(
                group_ref
            )

            if group_name in visited_groups:
                return results

            visited_groups.add(
                group_name
            )

            group_def = get_global_group(
                group_name
            )

            if group_def is not None:

                for group_child in group_def:

                    if not isinstance(
                        group_child.tag,
                        str
                    ):
                        continue

                    child_local = etree.QName(
                        group_child
                    ).localname

                    if child_local in {
                        "sequence",
                        "choice",
                        "all",
                        "group",
                        "element"
                    }:

                        results.extend(
                            collect_elements_from_particle(
                                group_child,
                                visited_groups
                            )
                        )

        return results


    return results


# ==============================================================================
# 10. COMPLEX-TYPE CHILD RESOLUTION
# ==============================================================================

def get_direct_children_from_complex_type(
    complex_type,
    visited_types=None
):
    """
    Return the logical direct child elements of a complex type.

    Supports:
        - direct sequence/choice/all
        - xs:group references
        - complexContent extension
        - complexContent restriction
        - inherited base complex-type children
    """

    if complex_type is None:
        return []

    if visited_types is None:
        visited_types = set()

    results = []


    # --------------------------------------------------------------------------
    # Direct particle containers
    # --------------------------------------------------------------------------

    for child in complex_type:

        if not isinstance(
            child.tag,
            str
        ):
            continue

        local = etree.QName(
            child
        ).localname

        if local in {
            "sequence",
            "choice",
            "all",
            "group"
        }:

            results.extend(
                collect_elements_from_particle(
                    child
                )
            )


    # --------------------------------------------------------------------------
    # Handle complexContent extension/restriction
    # --------------------------------------------------------------------------

    complex_content = complex_type.find(
        "xs:complexContent",
        namespaces=XSD_NS
    )

    if complex_content is not None:

        extender = complex_content.find(
            "xs:extension",
            namespaces=XSD_NS
        )

        if extender is None:

            extender = complex_content.find(
                "xs:restriction",
                namespaces=XSD_NS
            )


        if extender is not None:

            base_name = extender.get(
                "base"
            )

            base_local = qname_local_name(
                base_name
            )

            if (
                base_local
                and base_local not in visited_types
            ):

                visited_types.add(
                    base_local
                )

                base_type = get_global_complex_type(
                    base_name
                )

                if base_type is not None:

                    results.extend(
                        get_direct_children_from_complex_type(
                            base_type,
                            visited_types
                        )
                    )


            for child in extender:

                if not isinstance(
                    child.tag,
                    str
                ):
                    continue

                local = etree.QName(
                    child
                ).localname

                if local in {
                    "sequence",
                    "choice",
                    "all",
                    "group"
                }:

                    results.extend(
                        collect_elements_from_particle(
                            child
                        )
                    )


    return results


def get_direct_child_elements(element):
    """
    Resolve direct logical children of a given element.
    """

    complex_type = get_element_complex_type(
        element
    )

    if complex_type is None:
        return []

    return get_direct_children_from_complex_type(
        complex_type
    )


# ==============================================================================
# 11. ATTRIBUTE RESOLUTION
# ==============================================================================

def collect_attribute_group_attributes(
    attribute_group,
    visited_groups=None
):
    """
    Collect attributes from an xs:attributeGroup without descending into
    unrelated nested child complex types.
    """

    if attribute_group is None:
        return []

    if visited_groups is None:
        visited_groups = set()

    attributes = []

    for child in attribute_group:

        if not isinstance(
            child.tag,
            str
        ):
            continue

        local = etree.QName(
            child
        ).localname


        if local == "attribute":

            attributes.append(
                child
            )


        elif local == "attributeGroup":

            ref = child.get(
                "ref"
            )

            ref_name = qname_local_name(
                ref
            )

            if (
                ref_name
                and ref_name not in visited_groups
            ):

                visited_groups.add(
                    ref_name
                )

                nested_group = (
                    get_global_attribute_group(
                        ref_name
                    )
                )

                attributes.extend(
                    collect_attribute_group_attributes(
                        nested_group,
                        visited_groups
                    )
                )


    return attributes


def get_direct_attributes_from_complex_type(
    complex_type,
    visited_types=None
):
    """
    Obtain attributes logically belonging to a complex type.

    Includes inherited attributes from a base type, but does NOT recursively
    collect attributes from nested child elements.
    """

    if complex_type is None:
        return []

    if visited_types is None:
        visited_types = set()

    attributes = []


    # --------------------------------------------------------------------------
    # Direct attributes
    # --------------------------------------------------------------------------

    for attribute in complex_type.findall(
        "xs:attribute",
        namespaces=XSD_NS
    ):

        attributes.append(
            attribute
        )


    # --------------------------------------------------------------------------
    # Direct attributeGroup references
    # --------------------------------------------------------------------------

    for attr_group_ref in complex_type.findall(
        "xs:attributeGroup",
        namespaces=XSD_NS
    ):

        ref_name = qname_local_name(
            attr_group_ref.get("ref")
        )

        group_def = get_global_attribute_group(
            ref_name
        )

        attributes.extend(
            collect_attribute_group_attributes(
                group_def
            )
        )


    # --------------------------------------------------------------------------
    # Complex-content inheritance
    # --------------------------------------------------------------------------

    complex_content = complex_type.find(
        "xs:complexContent",
        namespaces=XSD_NS
    )

    if complex_content is not None:

        extender = complex_content.find(
            "xs:extension",
            namespaces=XSD_NS
        )

        if extender is None:

            extender = complex_content.find(
                "xs:restriction",
                namespaces=XSD_NS
            )


        if extender is not None:

            base_name = extender.get(
                "base"
            )

            base_local = qname_local_name(
                base_name
            )

            if (
                base_local
                and base_local not in visited_types
            ):

                visited_types.add(
                    base_local
                )

                base_type = get_global_complex_type(
                    base_name
                )

                attributes.extend(
                    get_direct_attributes_from_complex_type(
                        base_type,
                        visited_types
                    )
                )


            # Attributes defined directly on extension
            for attribute in extender.findall(
                "xs:attribute",
                namespaces=XSD_NS
            ):

                attributes.append(
                    attribute
                )


            for attr_group_ref in extender.findall(
                "xs:attributeGroup",
                namespaces=XSD_NS
            ):

                group_def = (
                    get_global_attribute_group(
                        attr_group_ref.get("ref")
                    )
                )

                attributes.extend(
                    collect_attribute_group_attributes(
                        group_def
                    )
                )


    # --------------------------------------------------------------------------
    # Simple-content attributes
    # --------------------------------------------------------------------------

    simple_content = complex_type.find(
        "xs:simpleContent",
        namespaces=XSD_NS
    )

    if simple_content is not None:

        extender = simple_content.find(
            "xs:extension",
            namespaces=XSD_NS
        )

        if extender is None:

            extender = simple_content.find(
                "xs:restriction",
                namespaces=XSD_NS
            )


        if extender is not None:

            for attribute in extender.findall(
                "xs:attribute",
                namespaces=XSD_NS
            ):

                attributes.append(
                    attribute
                )


    return attributes


def get_element_attributes(element):
    """
    Obtain attributes logically associated with an element.
    """

    complex_type = get_element_complex_type(
        element
    )

    if complex_type is None:
        return []

    attributes = (
        get_direct_attributes_from_complex_type(
            complex_type
        )
    )

    result = []

    seen = set()

    for attr in attributes:

        name = attr.get("name")

        if name is None:

            ref = attr.get("ref")

            name = qname_local_name(
                ref
            )


        if not name or name in seen:
            continue

        seen.add(
            name
        )

        result.append({
            "name": name,
            "type": attr.get("type"),
            "use": attr.get(
                "use",
                "optional"
            ),
            "default": attr.get(
                "default"
            ),
            "fixed": attr.get(
                "fixed"
            )
        })

    return result


# ==============================================================================
# 12. SIMPLE-TYPE ENUMERATION RESOLUTION
# ==============================================================================

def get_simple_type_enumerations(simple_type):
    """
    Extract xs:enumeration values from an inline or named simple type.

    Also follows restriction base types when needed.
    """

    if simple_type is None:
        return []

    values = []

    restriction = simple_type.find(
        "xs:restriction",
        namespaces=XSD_NS
    )

    if restriction is None:
        return values


    for enum in restriction.findall(
        "xs:enumeration",
        namespaces=XSD_NS
    ):

        value = enum.get(
            "value"
        )

        if value is not None:

            values.append(
                value
            )


    # Follow named base simple type if no direct values
    if not values:

        base_name = restriction.get(
            "base"
        )

        if base_name:

            base_type = get_global_simple_type(
                base_name
            )

            if base_type is not None:

                values.extend(
                    get_simple_type_enumerations(
                        base_type
                    )
                )


    return values


def get_element_enumerations(element):
    """
    Resolve allowed enumeration values for an element.
    """

    simple_type = get_element_simple_type(
        element
    )

    if simple_type is None:
        return []

    return get_simple_type_enumerations(
        simple_type
    )


# ==============================================================================
# 13. STRUCTURAL DESCRIPTION HELPERS
# ==============================================================================

def describe_element(
    element,
    parent_path=""
):
    """
    Describe an element including:
        path
        name
        declared type
        cardinality
        attributes
    """

    if element is None:
        return None

    name = get_element_name(
        element
    )

    min_occurs = normalize_occurs(
        element.get("minOccurs"),
        1
    )

    max_occurs = normalize_occurs(
        element.get("maxOccurs"),
        1
    )

    declared_type = element.get(
        "type"
    )

    resolved = resolve_element_reference(
        element
    )

    if declared_type is None:

        declared_type = resolved.get(
            "type"
        )


    if parent_path:

        path = (
            f"{parent_path}/{name}"
        )

    else:

        path = name


    return {
        "path": path,
        "element": name,
        "type": declared_type,
        "min_occurs": min_occurs,
        "max_occurs": max_occurs,
        "attributes": get_element_attributes(
            element
        )
    }


def find_direct_child(
    parent_element,
    child_name
):
    """
    Find a child only inside the resolved hierarchy of parent_element.

    This replaces unsafe global:
        .//xs:element[@name='...']
    """

    if parent_element is None:
        return None

    for child in get_direct_child_elements(
        parent_element
    ):

        if get_element_name(
            child
        ) == child_name:

            return child

    return None


# ==============================================================================
# 14. IDENTIFY ROOT ELEMENT HIERARCHICALLY
# ==============================================================================

print("\n" + "=" * 110)
print("ROOT STRUCTURE")
print("=" * 110)

drugbank_element = get_global_element(
    "drugbank"
)

if drugbank_element is None:

    raise RuntimeError(
        "Top-level <drugbank> element was not found in the XSD."
    )


print("[PASS] Top-level <drugbank> element identified.")


drug_element = find_direct_child(
    drugbank_element,
    "drug"
)


if drug_element is None:

    raise RuntimeError(
        "<drug> could not be resolved as a child "
        "of <drugbank>."
    )


drug_info = describe_element(
    drug_element,
    "drugbank"
)


print("\nDrug element:")
print(f"Path                   : {drug_info['path']}")
print(f"Type                   : {drug_info['type']}")
print(f"minOccurs              : {drug_info['min_occurs']}")
print(f"maxOccurs              : {drug_info['max_occurs']}")


# ==============================================================================
# 15. DRUG-LEVEL ATTRIBUTES
# ==============================================================================

print("\n" + "=" * 110)
print("DRUG-LEVEL ATTRIBUTES")
print("=" * 110)

drug_attributes = get_element_attributes(
    drug_element
)


if drug_attributes:

    for attr in drug_attributes:

        print(
            f"{attr['name']:<25} "
            f"type={str(attr['type']):<30} "
            f"use={attr['use']}"
        )

else:

    print(
        "[INFO] No direct/inherited drug-level attributes identified."
    )


# ==============================================================================
# 16. DIRECT DRUG CHILDREN
# ==============================================================================

print("\n" + "=" * 110)
print("DIRECT CHILD ELEMENTS OF <drug>")
print("=" * 110)

drug_child_elements = get_direct_child_elements(
    drug_element
)

drug_structure_records = []


for child in drug_child_elements:

    info = describe_element(
        child,
        "drugbank/drug"
    )

    drug_structure_records.append(
        info
    )


drug_structure_df = pd.DataFrame([
    {
        "path": item["path"],
        "element": item["element"],
        "type": item["type"],
        "min_occurs": item["min_occurs"],
        "max_occurs": item["max_occurs"]
    }
    for item in drug_structure_records
])


if not drug_structure_df.empty:

    print(
        drug_structure_df.to_string(
            index=False
        )
    )

else:

    print(
        "[FAIL] No child elements found under <drug>."
    )


# ==============================================================================
# 17. RESOLVE IMPORTANT DRUG SECTIONS HIERARCHICALLY
# ==============================================================================

important_section_names = [
    "drugbank-id",
    "name",
    "description",
    "groups",
    "calculated-properties",
    "experimental-properties",
    "categories",
    "pathways",
    "drug-interactions",
    "targets",
    "enzymes",
    "carriers",
    "transporters"
]


important_sections = {}


print("\n" + "=" * 110)
print("IMPORTANT DRUG SECTIONS")
print("=" * 110)


for section_name in important_section_names:

    element = find_direct_child(
        drug_element,
        section_name
    )

    important_sections[
        section_name
    ] = element

    status = (
        "FOUND"
        if element is not None
        else "NOT FOUND"
    )

    print(
        f"[{status:<9}] {section_name}"
    )


# ==============================================================================
# 18. DRUGBANK-ID STRUCTURE / PRIMARY ATTRIBUTE
# ==============================================================================

print("\n" + "=" * 110)
print("DRUGBANK IDENTIFIER STRUCTURE")
print("=" * 110)


drugbank_id_element = important_sections.get(
    "drugbank-id"
)


drugbank_id_attributes = (
    get_element_attributes(
        drugbank_id_element
    )
    if drugbank_id_element is not None
    else []
)


print("drugbank-id attributes:")

if drugbank_id_attributes:

    for attr in drugbank_id_attributes:

        print(
            f"  {attr['name']:<20} "
            f"type={attr['type']} "
            f"use={attr['use']}"
        )

else:

    print(
        "  No attributes identified."
    )


PRIMARY_ATTRIBUTE_FOUND = any(
    attr["name"] == "primary"
    for attr in drugbank_id_attributes
)


# ==============================================================================
# 19. GROUPS HIERARCHY
# ==============================================================================

print("\n" + "=" * 110)
print("DRUG GROUP STRUCTURE")
print("=" * 110)


groups_element = important_sections.get(
    "groups"
)

group_element = find_direct_child(
    groups_element,
    "group"
)


if group_element is not None:

    group_info = describe_element(
        group_element,
        "drug/groups"
    )

    print(
        f"Element                : {group_info['element']}"
    )

    print(
        f"minOccurs              : {group_info['min_occurs']}"
    )

    print(
        f"maxOccurs              : {group_info['max_occurs']}"
    )

else:

    print(
        "[WARNING] drug/groups/group was not resolved."
    )


# ==============================================================================
# 20. CALCULATED-PROPERTIES HIERARCHY
# ==============================================================================

print("\n" + "=" * 110)
print("CALCULATED PROPERTY STRUCTURE")
print("=" * 110)


calculated_properties_element = important_sections.get(
    "calculated-properties"
)


calculated_property_element = find_direct_child(
    calculated_properties_element,
    "property"
)


calc_kind_element = None
calc_value_element = None
calc_source_element = None


if calculated_property_element is not None:

    calc_kind_element = find_direct_child(
        calculated_property_element,
        "kind"
    )

    calc_value_element = find_direct_child(
        calculated_property_element,
        "value"
    )

    calc_source_element = find_direct_child(
        calculated_property_element,
        "source"
    )


    property_info = describe_element(
        calculated_property_element,
        "drug/calculated-properties"
    )


    print(
        f"Property element        : "
        f"{property_info['element']}"
    )

    print(
        f"Property maxOccurs      : "
        f"{property_info['max_occurs']}"
    )


    for label, element in [
        ("kind", calc_kind_element),
        ("value", calc_value_element),
        ("source", calc_source_element)
    ]:

        print(
            f"{label:<22}: "
            f"{'FOUND' if element is not None else 'NOT FOUND'}"
        )

else:

    print(
        "[FAIL] calculated-properties/property "
        "could not be resolved."
    )


# ==============================================================================
# 21. CALCULATED-PROPERTY ENUMERATIONS
# ==============================================================================

print("\n" + "=" * 110)
print("CALCULATED PROPERTY KIND ENUMERATIONS")
print("=" * 110)


CALCULATED_PROPERTY_ENUMS = (
    get_element_enumerations(
        calc_kind_element
    )
    if calc_kind_element is not None
    else []
)


if CALCULATED_PROPERTY_ENUMS:

    for index, value in enumerate(
        CALCULATED_PROPERTY_ENUMS,
        start=1
    ):

        print(
            f"{index:>3}. {value}"
        )

else:

    print(
        "[WARNING] No calculated-property enumeration "
        "values were discovered."
    )


# ==============================================================================
# 22. REQUIRED CALCULATED-PROPERTY VALIDATION
# ==============================================================================

requested_calculated_properties = [
    "SMILES",
    "InChI",
    "InChIKey",
    "Molecular Formula",
    "Molecular Weight"
]


calculated_property_validation = []


for requested_property in requested_calculated_properties:

    supported = (
        requested_property
        in CALCULATED_PROPERTY_ENUMS
    )

    calculated_property_validation.append({
        "property": requested_property,
        "supported_by_xsd": supported
    })


calc_property_validation_df = pd.DataFrame(
    calculated_property_validation
)


print("\nRequired molecular-property validation:")

print(
    calc_property_validation_df.to_string(
        index=False
    )
)


# ==============================================================================
# 23. EXPERIMENTAL-PROPERTIES STRUCTURE
# ==============================================================================

print("\n" + "=" * 110)
print("EXPERIMENTAL PROPERTY STRUCTURE")
print("=" * 110)


experimental_properties_element = important_sections.get(
    "experimental-properties"
)

experimental_property_element = find_direct_child(
    experimental_properties_element,
    "property"
)


if experimental_property_element is not None:

    exp_children = get_direct_child_elements(
        experimental_property_element
    )

    for child in exp_children:

        child_info = describe_element(
            child,
            "drug/experimental-properties/property"
        )

        print(
            f"{child_info['element']:<25} "
            f"min={child_info['min_occurs']:<10} "
            f"max={child_info['max_occurs']:<10} "
            f"type={child_info['type']}"
        )

else:

    print(
        "[WARNING] experimental-properties/property "
        "was not resolved."
    )


# ==============================================================================
# 24. GENERIC BIOLOGICAL INTERACTANT INSPECTION
# ==============================================================================

def inspect_interactant_section(
    section_element,
    child_name
):
    """
    Inspect targets / enzymes / carriers / transporters hierarchically.
    """

    result = {
        "section_found": False,
        "interactant_found": False,
        "interactant_element": None,
        "children": [],
        "attributes": []
    }


    if section_element is None:
        return result


    result[
        "section_found"
    ] = True


    interactant = find_direct_child(
        section_element,
        child_name
    )


    if interactant is None:
        return result


    result[
        "interactant_found"
    ] = True

    result[
        "interactant_element"
    ] = interactant


    for child in get_direct_child_elements(
        interactant
    ):

        info = describe_element(
            child,
            f"drug/{child_name}s/{child_name}"
        )

        result["children"].append(
            info
        )


    result["attributes"] = (
        get_element_attributes(
            interactant
        )
    )


    return result


# ==============================================================================
# 25. TARGET STRUCTURE
# ==============================================================================

print("\n" + "=" * 110)
print("TARGET HIERARCHY")
print("=" * 110)


targets_element = important_sections.get(
    "targets"
)


target_analysis = inspect_interactant_section(
    targets_element,
    "target"
)


target_element = target_analysis[
    "interactant_element"
]


if target_element is not None:

    for info in target_analysis[
        "children"
    ]:

        print(
            f"{info['element']:<30} "
            f"min={info['min_occurs']:<10} "
            f"max={info['max_occurs']:<10} "
            f"type={info['type']}"
        )

else:

    print(
        "[FAIL] target hierarchy could not be resolved."
    )


# ==============================================================================
# 26. TARGET SUBSTRUCTURES
# ==============================================================================

target_id_element = find_direct_child(
    target_element,
    "id"
)

target_name_element = find_direct_child(
    target_element,
    "name"
)

target_organism_element = find_direct_child(
    target_element,
    "organism"
)

known_action_element = find_direct_child(
    target_element,
    "known-action"
)

actions_element = find_direct_child(
    target_element,
    "actions"
)

polypeptide_element = find_direct_child(
    target_element,
    "polypeptide"
)


action_element = find_direct_child(
    actions_element,
    "action"
)


# ==============================================================================
# 27. POLYPEPTIDE STRUCTURE
# ==============================================================================

print("\n" + "=" * 110)
print("POLYPEPTIDE STRUCTURE")
print("=" * 110)


POLYPEPTIDE_ATTRIBUTES = []

polypeptide_sequence_element = None


if polypeptide_element is not None:

    POLYPEPTIDE_ATTRIBUTES = (
        get_element_attributes(
            polypeptide_element
        )
    )


    print("Polypeptide attributes:")

    if POLYPEPTIDE_ATTRIBUTES:

        for attr in POLYPEPTIDE_ATTRIBUTES:

            print(
                f"  {attr['name']:<22} "
                f"type={str(attr['type']):<30} "
                f"use={attr['use']}"
            )

    else:

        print(
            "  No attributes discovered."
        )


    print("\nPolypeptide child elements:")


    for child in get_direct_child_elements(
        polypeptide_element
    ):

        info = describe_element(
            child,
            "drug/targets/target/polypeptide"
        )

        print(
            f"  {info['element']:<32} "
            f"min={info['min_occurs']:<10} "
            f"max={info['max_occurs']:<10} "
            f"type={info['type']}"
        )


    polypeptide_sequence_element = (
        find_direct_child(
            polypeptide_element,
            "amino-acid-sequence"
        )
    )


else:

    print(
        "[WARNING] target/polypeptide was not resolved."
    )


POLYPEPTIDE_ID_ATTRIBUTE_FOUND = any(
    attr["name"] == "id"
    for attr in POLYPEPTIDE_ATTRIBUTES
)


POLYPEPTIDE_SOURCE_ATTRIBUTE_FOUND = any(
    attr["name"] == "source"
    for attr in POLYPEPTIDE_ATTRIBUTES
)


# ==============================================================================
# 28. ENZYME / CARRIER / TRANSPORTER STRUCTURES
# ==============================================================================

print("\n" + "=" * 110)
print("BIOLOGICAL INTERACTANT COLLECTIONS")
print("=" * 110)


biological_section_specs = {
    "targets": "target",
    "enzymes": "enzyme",
    "carriers": "carrier",
    "transporters": "transporter"
}


biological_map = []


for section_name, item_name in biological_section_specs.items():

    section_element = important_sections.get(
        section_name
    )

    result = inspect_interactant_section(
        section_element,
        item_name
    )


    biological_map.append({
        "section": section_name,
        "item": item_name,
        "section_found": result[
            "section_found"
        ],
        "item_found": result[
            "interactant_found"
        ],
        "child_count": len(
            result["children"]
        )
    })


biological_map_df = pd.DataFrame(
    biological_map
)


print(
    biological_map_df.to_string(
        index=False
    )
)


# ==============================================================================
# 29. PATHWAY STRUCTURE
# ==============================================================================

print("\n" + "=" * 110)
print("PATHWAY STRUCTURE")
print("=" * 110)


pathways_element = important_sections.get(
    "pathways"
)


pathway_element = find_direct_child(
    pathways_element,
    "pathway"
)


if pathway_element is not None:

    pathway_info = describe_element(
        pathway_element,
        "drug/pathways"
    )


    print(
        f"Pathway multiplicity    : "
        f"{pathway_info['min_occurs']} .. "
        f"{pathway_info['max_occurs']}"
    )


    pathway_children = get_direct_child_elements(
        pathway_element
    )


    for child in pathway_children:

        info = describe_element(
            child,
            "drug/pathways/pathway"
        )

        print(
            f"{info['element']:<30} "
            f"min={info['min_occurs']:<10} "
            f"max={info['max_occurs']:<10} "
            f"type={info['type']}"
        )

else:

    print(
        "[WARNING] pathway structure was not resolved."
    )


# ==============================================================================
# 30. DEFINE D_chem MAPPING WITH XSD VERIFICATION
# ==============================================================================

# Determine exact support from XSD enumerations.

def calculated_property_supported(
    property_name
):
    return (
        property_name
        in CALCULATED_PROPERTY_ENUMS
    )


D_CHEM_SCHEMA = {
    "drug_id": {
        "xml_path":
            "drug/drugbank-id[@primary='true']",

        "purpose":
            "Primary DrugBank identifier",

        "xsd_verified":
            (
                drugbank_id_element is not None
                and PRIMARY_ATTRIBUTE_FOUND
            )
    },


    "drug_name": {
        "xml_path":
            "drug/name",

        "purpose":
            "Drug name",

        "xsd_verified":
            important_sections.get(
                "name"
            ) is not None
    },


    "drug_type": {
        "xml_path":
            "drug/@type",

        "purpose":
            "Drug classification such as small molecule / biotech",

        "xsd_verified":
            any(
                attr["name"] == "type"
                for attr in drug_attributes
            )
    },


    "smiles": {
        "xml_path":
            (
                "drug/calculated-properties/property"
                "[kind='SMILES']/value"
            ),

        "purpose":
            "Primary molecular structure representation",

        "xsd_verified":
            calculated_property_supported(
                "SMILES"
            )
    },


    "inchi": {
        "xml_path":
            (
                "drug/calculated-properties/property"
                "[kind='InChI']/value"
            ),

        "purpose":
            "Molecular identity representation",

        "xsd_verified":
            calculated_property_supported(
                "InChI"
            )
    },


    "inchikey": {
        "xml_path":
            (
                "drug/calculated-properties/property"
                "[kind='InChIKey']/value"
            ),

        "purpose":
            "Molecular identity key",

        "xsd_verified":
            calculated_property_supported(
                "InChIKey"
            )
    },


    "molecular_formula": {
        "xml_path":
            (
                "drug/calculated-properties/property"
                "[kind='Molecular Formula']/value"
            ),

        "purpose":
            "Molecular formula",

        "xsd_verified":
            calculated_property_supported(
                "Molecular Formula"
            )
    },


    "molecular_weight": {
        "xml_path":
            (
                "drug/calculated-properties/property"
                "[kind='Molecular Weight']/value"
            ),

        "purpose":
            "Calculated molecular weight",

        "xsd_verified":
            calculated_property_supported(
                "Molecular Weight"
            )
    },


    "average_mass": {
        "xml_path":
            "drug/average-mass",

        "purpose":
            "DrugBank average molecular mass",

        "xsd_verified":
            find_direct_child(
                drug_element,
                "average-mass"
            ) is not None
    },


    "monoisotopic_mass": {
        "xml_path":
            "drug/monoisotopic-mass",

        "purpose":
            "DrugBank monoisotopic mass",

        "xsd_verified":
            find_direct_child(
                drug_element,
                "monoisotopic-mass"
            ) is not None
    },


    "groups": {
        "xml_path":
            "drug/groups/group",

        "purpose":
            "Drug approval/development classification",

        "xsd_verified":
            group_element is not None
    }
}


# ==============================================================================
# 31. DEFINE D_target MAPPING WITH XSD VERIFICATION
# ==============================================================================

D_TARGET_SCHEMA = {
    "drug_id": {
        "xml_path":
            "drug/drugbank-id[@primary='true']",

        "purpose":
            "DrugBank identifier",

        "xsd_verified":
            (
                drugbank_id_element is not None
                and PRIMARY_ATTRIBUTE_FOUND
            )
    },


    "drug_name": {
        "xml_path":
            "drug/name",

        "purpose":
            "Drug name",

        "xsd_verified":
            important_sections.get(
                "name"
            ) is not None
    },


    "target_id": {
        "xml_path":
            "drug/targets/target/id",

        "purpose":
            "DrugBank target identifier",

        "xsd_verified":
            target_id_element is not None
    },


    "target_name": {
        "xml_path":
            "drug/targets/target/name",

        "purpose":
            "Target name",

        "xsd_verified":
            target_name_element is not None
    },


    "organism": {
        "xml_path":
            "drug/targets/target/organism",

        "purpose":
            "Target organism",

        "xsd_verified":
            target_organism_element is not None
    },


    "known_action": {
        "xml_path":
            "drug/targets/target/known-action",

        "purpose":
            "Known pharmacological action flag",

        "xsd_verified":
            known_action_element is not None
    },


    "actions": {
        "xml_path":
            "drug/targets/target/actions/action",

        "purpose":
            "Drug-target pharmacological action",

        "xsd_verified":
            action_element is not None
    },


    "polypeptide_id": {
        "xml_path":
            "drug/targets/target/polypeptide/@id",

        "purpose":
            "Target protein identifier",

        "xsd_verified":
            POLYPEPTIDE_ID_ATTRIBUTE_FOUND
    },


    "polypeptide_source": {
        "xml_path":
            "drug/targets/target/polypeptide/@source",

        "purpose":
            "Source of target protein identifier",

        "xsd_verified":
            POLYPEPTIDE_SOURCE_ATTRIBUTE_FOUND
    },


    "amino_acid_sequence": {
        "xml_path":
            (
                "drug/targets/target/"
                "polypeptide/amino-acid-sequence"
            ),

        "purpose":
            "Target protein amino-acid sequence",

        "xsd_verified":
            polypeptide_sequence_element
            is not None
    }
}


# ==============================================================================
# 32. CONVERT STRUCTURAL MAPS TO DATAFRAMES
# ==============================================================================

d_chem_mapping_df = pd.DataFrame([
    {
        "dataset_field": field,
        "xml_path": definition[
            "xml_path"
        ],
        "purpose": definition[
            "purpose"
        ],
        "xsd_verified": definition[
            "xsd_verified"
        ]
    }

    for field, definition
    in D_CHEM_SCHEMA.items()
])


d_target_mapping_df = pd.DataFrame([
    {
        "dataset_field": field,
        "xml_path": definition[
            "xml_path"
        ],
        "purpose": definition[
            "purpose"
        ],
        "xsd_verified": definition[
            "xsd_verified"
        ]
    }

    for field, definition
    in D_TARGET_SCHEMA.items()
])


print("\n" + "=" * 110)
print("D_chem — XSD-VERIFIED STRUCTURAL MAPPING")
print("=" * 110)

print(
    d_chem_mapping_df.to_string(
        index=False
    )
)


print("\n" + "=" * 110)
print("D_target — XSD-VERIFIED STRUCTURAL MAPPING")
print("=" * 110)

print(
    d_target_mapping_df.to_string(
        index=False
    )
)


# ==============================================================================
# 33. LIMITED XML SPOT CHECK
# ==============================================================================
#
# We inspect only a SMALL number of <drug> records.
#
# Goals:
#   - verify real primary DrugBank ID
#   - verify actual calculated property layout
#   - find at least one real SMILES if available
#   - verify target hierarchy
#   - verify polypeptide attributes/sequence where available
#
# This is NOT Phase-3 extraction.
# ==============================================================================

print("\n" + "=" * 110)
print("LIMITED XML STRUCTURAL SPOT CHECK")
print("=" * 110)


SPOT_CHECK_MAX_DRUGS = 50


spot_check = {
    "records_inspected": 0,

    "example_drug": None,

    "example_calculated_property": None,

    "example_smiles": None,

    "example_target": None,

    "example_polypeptide": None
}


with open(
    XML_PATH,
    "rb"
) as xml_file:

    context = etree.iterparse(
        xml_file,
        events=("end",),
        tag=f"{NS_TAG}drug",
        huge_tree=True
    )


    for event, drug in context:

        spot_check[
            "records_inspected"
        ] += 1


        # ----------------------------------------------------------------------
        # Primary DrugBank ID
        # ----------------------------------------------------------------------

        primary_id = None

        for db_id in drug.findall(
            f"{NS_TAG}drugbank-id"
        ):

            if db_id.get(
                "primary"
            ) == "true":

                primary_id = (
                    db_id.text.strip()
                    if db_id.text
                    else None
                )

                break


        drug_name = drug.findtext(
            f"{NS_TAG}name"
        )

        drug_type = drug.get(
            "type"
        )


        if spot_check[
            "example_drug"
        ] is None:

            spot_check[
                "example_drug"
            ] = {
                "drug_id": primary_id,
                "drug_name": drug_name,
                "drug_type": drug_type
            }


        # ----------------------------------------------------------------------
        # Calculated properties
        # ----------------------------------------------------------------------

        calc_props = drug.find(
            f"{NS_TAG}calculated-properties"
        )


        if calc_props is not None:

            for prop in calc_props.findall(
                f"{NS_TAG}property"
            ):

                kind = prop.findtext(
                    f"{NS_TAG}kind"
                )

                value = prop.findtext(
                    f"{NS_TAG}value"
                )

                source = prop.findtext(
                    f"{NS_TAG}source"
                )


                if (
                    spot_check[
                        "example_calculated_property"
                    ]
                    is None
                    and kind
                ):

                    spot_check[
                        "example_calculated_property"
                    ] = {
                        "drug_id": primary_id,
                        "kind": kind,
                        "value": value,
                        "source": source
                    }


                if (
                    spot_check[
                        "example_smiles"
                    ]
                    is None
                    and kind == "SMILES"
                    and value
                ):

                    spot_check[
                        "example_smiles"
                    ] = {
                        "drug_id": primary_id,
                        "drug_name": drug_name,
                        "smiles": value
                    }


        # ----------------------------------------------------------------------
        # Target hierarchy
        # ----------------------------------------------------------------------

        targets = drug.find(
            f"{NS_TAG}targets"
        )


        if targets is not None:

            for target in targets.findall(
                f"{NS_TAG}target"
            ):

                target_id = target.findtext(
                    f"{NS_TAG}id"
                )

                target_name = target.findtext(
                    f"{NS_TAG}name"
                )

                organism = target.findtext(
                    f"{NS_TAG}organism"
                )

                known_action = target.findtext(
                    f"{NS_TAG}known-action"
                )


                action_values = []

                actions = target.find(
                    f"{NS_TAG}actions"
                )

                if actions is not None:

                    for action in actions.findall(
                        f"{NS_TAG}action"
                    ):

                        if action.text:

                            action_values.append(
                                action.text.strip()
                            )


                if (
                    spot_check[
                        "example_target"
                    ]
                    is None
                ):

                    spot_check[
                        "example_target"
                    ] = {
                        "drug_id": primary_id,
                        "target_id": target_id,
                        "target_name": target_name,
                        "organism": organism,
                        "known_action": known_action,
                        "actions": action_values
                    }


                # --------------------------------------------------------------
                # Multiple polypeptides are supported
                # --------------------------------------------------------------

                polypeptides = target.findall(
                    f"{NS_TAG}polypeptide"
                )


                for polypeptide in polypeptides:

                    poly_id = polypeptide.get(
                        "id"
                    )

                    poly_source = polypeptide.get(
                        "source"
                    )

                    sequence = polypeptide.findtext(
                        f"{NS_TAG}amino-acid-sequence"
                    )


                    if (
                        spot_check[
                            "example_polypeptide"
                        ]
                        is None
                    ):

                        spot_check[
                            "example_polypeptide"
                        ] = {
                            "drug_id": primary_id,
                            "target_id": target_id,
                            "polypeptide_id":
                                poly_id,
                            "polypeptide_source":
                                poly_source,
                            "sequence_present":
                                bool(sequence),
                            "sequence_length_raw":
                                len(sequence)
                                if sequence
                                else 0
                        }


        # ----------------------------------------------------------------------
        # Memory cleanup
        # ----------------------------------------------------------------------

        drug.clear()

        parent = drug.getparent()

        if parent is not None:

            while drug.getprevious() is not None:

                del parent[0]


        # ----------------------------------------------------------------------
        # Stop after enough records OR after all representative structures found
        # ----------------------------------------------------------------------

        all_examples_found = all([
            spot_check[
                "example_drug"
            ] is not None,

            spot_check[
                "example_calculated_property"
            ] is not None,

            spot_check[
                "example_smiles"
            ] is not None,

            spot_check[
                "example_target"
            ] is not None,

            spot_check[
                "example_polypeptide"
            ] is not None
        ])


        if all_examples_found:

            break


        if (
            spot_check[
                "records_inspected"
            ]
            >= SPOT_CHECK_MAX_DRUGS
        ):

            break


    del context


# ==============================================================================
# 34. DISPLAY XML SPOT-CHECK RESULTS
# ==============================================================================

print(
    f"Drug records inspected : "
    f"{spot_check['records_inspected']}"
)


for section_name in [
    "example_drug",
    "example_calculated_property",
    "example_smiles",
    "example_target",
    "example_polypeptide"
]:

    print(
        "\n"
        + section_name.replace(
            "_",
            " "
        ).upper()
    )

    example = spot_check[
        section_name
    ]


    if example is None:

        print(
            "  Not observed within the limited spot-check."
        )

    else:

        for key, value in example.items():

            # Avoid printing extremely long values
            display_value = value

            if (
                isinstance(
                    display_value,
                    str
                )
                and len(
                    display_value
                ) > 180
            ):

                display_value = (
                    display_value[:180]
                    + "..."
                )


            print(
                f"  {key:<25}: "
                f"{display_value}"
            )


# ==============================================================================
# 35. XML SPOT-CHECK VALIDATION
# ==============================================================================

XML_SPOT_CHECK_RESULTS = {
    "drug_record_found":
        spot_check[
            "example_drug"
        ] is not None,

    "calculated_property_found":
        spot_check[
            "example_calculated_property"
        ] is not None,

    "smiles_found":
        spot_check[
            "example_smiles"
        ] is not None,

    "target_found":
        spot_check[
            "example_target"
        ] is not None,

    "polypeptide_found":
        spot_check[
            "example_polypeptide"
        ] is not None
}


# ==============================================================================
# 36. FIELD-LEVEL SCHEMA VERIFICATION TABLE
# ==============================================================================

schema_verification_records = []


for field, definition in D_CHEM_SCHEMA.items():

    schema_verification_records.append({
        "dataset": "D_chem",
        "field": field,
        "xml_path": definition[
            "xml_path"
        ],
        "xsd_verified": definition[
            "xsd_verified"
        ]
    })


for field, definition in D_TARGET_SCHEMA.items():

    schema_verification_records.append({
        "dataset": "D_target",
        "field": field,
        "xml_path": definition[
            "xml_path"
        ],
        "xsd_verified": definition[
            "xsd_verified"
        ]
    })


schema_verification_df = pd.DataFrame(
    schema_verification_records
)


print("\n" + "=" * 110)
print("FIELD-LEVEL XSD VERIFICATION")
print("=" * 110)

print(
    schema_verification_df.to_string(
        index=False
    )
)


# ==============================================================================
# 37. PHASE-2 CONFIGURATION PACKAGE
# ==============================================================================

PHASE2_SCHEMA_MAP = {
    "xsd_target_namespace":
        XSD_TARGET_NAMESPACE,

    "namespace_match":
        NAMESPACE_MATCH,

    "drugbank_element":
        "drugbank",

    "drug_element":
        "drug",

    "drug_attributes":
        drug_attributes,

    "drug_children":
        drug_structure_records,

    "calculated_property_enumerations":
        CALCULATED_PROPERTY_ENUMS,

    "calculated_property_validation":
        calculated_property_validation,

    "biological_sections":
        biological_map,

    "D_chem_schema":
        D_CHEM_SCHEMA,

    "D_target_schema":
        D_TARGET_SCHEMA,

    "xml_spot_check":
        spot_check,

    "xml_spot_check_results":
        XML_SPOT_CHECK_RESULTS
}


DRUGBANK_CONFIG[
    "phase2_schema_map"
] = PHASE2_SCHEMA_MAP


# ==============================================================================
# 38. CREATE PHASE-2 OUTPUT DIRECTORY
# ==============================================================================

SCHEMA_OUTPUT_DIR = (
    OUTPUT_DIR
    / "schema"
)

SCHEMA_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ==============================================================================
# 39. SAVE PHASE-2 REPORTS
# ==============================================================================

drug_structure_file = (
    SCHEMA_OUTPUT_DIR
    / "drugbank_drug_structure.csv"
)


d_chem_mapping_file = (
    SCHEMA_OUTPUT_DIR
    / "D_chem_xml_mapping.csv"
)


d_target_mapping_file = (
    SCHEMA_OUTPUT_DIR
    / "D_target_xml_mapping.csv"
)


biological_mapping_file = (
    SCHEMA_OUTPUT_DIR
    / "biological_relationship_mapping.csv"
)


calculated_properties_file = (
    SCHEMA_OUTPUT_DIR
    / "calculated_property_enumerations.csv"
)


schema_verification_file = (
    SCHEMA_OUTPUT_DIR
    / "schema_field_verification.csv"
)


phase2_json_file = (
    SCHEMA_OUTPUT_DIR
    / "phase2_schema_summary.json"
)


# Save CSV reports

drug_structure_df.to_csv(
    drug_structure_file,
    index=False
)


d_chem_mapping_df.to_csv(
    d_chem_mapping_file,
    index=False
)


d_target_mapping_df.to_csv(
    d_target_mapping_file,
    index=False
)


biological_map_df.to_csv(
    biological_mapping_file,
    index=False
)


pd.DataFrame({
    "calculated_property_kind":
        CALCULATED_PROPERTY_ENUMS
}).to_csv(
    calculated_properties_file,
    index=False
)


schema_verification_df.to_csv(
    schema_verification_file,
    index=False
)


# JSON-safe version

phase2_json_safe = {
    "xsd_target_namespace":
        XSD_TARGET_NAMESPACE,

    "namespace_match":
        NAMESPACE_MATCH,

    "calculated_property_enumerations":
        CALCULATED_PROPERTY_ENUMS,

    "calculated_property_validation":
        calculated_property_validation,

    "D_chem_schema":
        D_CHEM_SCHEMA,

    "D_target_schema":
        D_TARGET_SCHEMA,

    "xml_spot_check":
        spot_check,

    "xml_spot_check_results":
        XML_SPOT_CHECK_RESULTS
}


with open(
    phase2_json_file,
    "w",
    encoding="utf-8"
) as json_file:

    json.dump(
        phase2_json_safe,
        json_file,
        indent=4,
        ensure_ascii=False
    )


# ==============================================================================
# 40. PHASE-2 COMPLETION CHECKS
# ==============================================================================

all_d_chem_fields_verified = (
    d_chem_mapping_df[
        "xsd_verified"
    ].all()
)


all_d_target_fields_verified = (
    d_target_mapping_df[
        "xsd_verified"
    ].all()
)


completion_checks = {
    "XSD parsed and compiled":
        compiled_xsd is not None,

    "DrugBank root identified":
        drugbank_element is not None,

    "Drug element identified hierarchically":
        drug_element is not None,

    "XML/XSD namespace consistent":
        NAMESPACE_MATCH,

    "Drug attributes analyzed":
        True,

    "Drug direct children analyzed":
        len(
            drug_structure_records
        ) > 0,

    "Primary DrugBank ID attribute verified":
        PRIMARY_ATTRIBUTE_FOUND,

    "Calculated-property hierarchy verified":
        (
            calculated_property_element
            is not None
            and calc_kind_element
            is not None
            and calc_value_element
            is not None
        ),

    "Calculated-property enumerations discovered":
        len(
            CALCULATED_PROPERTY_ENUMS
        ) > 0,

    "Target hierarchy verified":
        target_element is not None,

    "Actions hierarchy verified":
        action_element is not None,

    "Polypeptide hierarchy verified":
        polypeptide_element is not None,

    "Polypeptide ID attribute verified":
        POLYPEPTIDE_ID_ATTRIBUTE_FOUND,

    "Polypeptide source attribute verified":
        POLYPEPTIDE_SOURCE_ATTRIBUTE_FOUND,

    "Amino-acid sequence verified":
        polypeptide_sequence_element
        is not None,

    "D_chem mapping XSD-verified":
        bool(
            all_d_chem_fields_verified
        ),

    "D_target mapping XSD-verified":
        bool(
            all_d_target_fields_verified
        ),

    "Limited XML drug spot-check completed":
        XML_SPOT_CHECK_RESULTS[
            "drug_record_found"
        ],

    "Actual calculated property observed":
        XML_SPOT_CHECK_RESULTS[
            "calculated_property_found"
        ]
}


# ==============================================================================
# 41. DISPLAY COMPLETION RESULTS
# ==============================================================================

print("\n" + "=" * 110)
print("PHASE 2 COMPLETION VALIDATION")
print("=" * 110)


all_critical_checks_passed = True


for check_name, status in completion_checks.items():

    label = (
        "PASS"
        if status
        else "FAIL"
    )


    print(
        f"[{label:<4}] {check_name}"
    )


    if not status:

        all_critical_checks_passed = False


# ==============================================================================
# 42. NON-CRITICAL SPOT-CHECK WARNINGS
# ==============================================================================

print("\n" + "=" * 110)
print("LIMITED XML SPOT-CHECK NOTES")
print("=" * 110)


if XML_SPOT_CHECK_RESULTS[
    "smiles_found"
]:

    print(
        "[PASS] At least one actual SMILES value "
        "was observed."
    )

else:

    print(
        "[NOTE] No SMILES encountered in the limited "
        "sample. This does not invalidate the XSD mapping."
    )


if XML_SPOT_CHECK_RESULTS[
    "target_found"
]:

    print(
        "[PASS] At least one actual target association "
        "was observed."
    )

else:

    print(
        "[NOTE] No target association encountered within "
        f"the first {SPOT_CHECK_MAX_DRUGS} inspected records."
    )


if XML_SPOT_CHECK_RESULTS[
    "polypeptide_found"
]:

    print(
        "[PASS] At least one actual polypeptide structure "
        "was observed."
    )

else:

    print(
        "[NOTE] No polypeptide encountered within the "
        "limited XML spot-check."
    )


# ==============================================================================
# 43. SAVED REPORT SUMMARY
# ==============================================================================

print("\n" + "=" * 110)
print("PHASE 2 REPORT FILES")
print("=" * 110)


print(
    f"Output Directory       : {SCHEMA_OUTPUT_DIR}"
)

print(
    f"Drug Structure         : {drug_structure_file.name}"
)

print(
    f"D_chem Mapping         : {d_chem_mapping_file.name}"
)

print(
    f"D_target Mapping       : {d_target_mapping_file.name}"
)

print(
    f"Biological Mapping     : {biological_mapping_file.name}"
)

print(
    f"Property Enumerations  : {calculated_properties_file.name}"
)

print(
    f"Field Verification     : {schema_verification_file.name}"
)

print(
    f"JSON Summary           : {phase2_json_file.name}"
)


# ==============================================================================
# 44. FINAL PHASE-2 SUMMARY
# ==============================================================================

print("\n" + "=" * 110)
print("PHASE 2 SUMMARY")
print("=" * 110)


print(
    f"Drug Child Elements Identified : "
    f"{len(drug_structure_records)}"
)

print(
    f"Calculated Property Kinds      : "
    f"{len(CALCULATED_PROPERTY_ENUMS)}"
)

print(
    f"D_chem Fields Defined          : "
    f"{len(D_CHEM_SCHEMA)}"
)

print(
    f"D_chem Fields XSD Verified     : "
    f"{int(d_chem_mapping_df['xsd_verified'].sum())}"
    f"/{len(D_CHEM_SCHEMA)}"
)

print(
    f"D_target Fields Defined        : "
    f"{len(D_TARGET_SCHEMA)}"
)

print(
    f"D_target Fields XSD Verified   : "
    f"{int(d_target_mapping_df['xsd_verified'].sum())}"
    f"/{len(D_TARGET_SCHEMA)}"
)

print(
    f"XML Records Spot-Checked       : "
    f"{spot_check['records_inspected']}"
)


print("-" * 110)


if all_critical_checks_passed:

    print(
        "[STATUS] PHASE 2 COMPLETED SUCCESSFULLY."
    )

    print(
        "[READY]  XSD hierarchy and XML-to-ML mappings "
        "are validated."
    )

    print(
        "[NEXT]   Phase 3 — Streaming DrugBank XML Extraction"
    )

else:

    print(
        "[STATUS] PHASE 2 COMPLETED WITH ONE OR MORE "
        "FAILED STRUCTURAL CHECKS."
    )

    print(
        "[ACTION] Review the failed checks before "
        "starting Phase 3."
    )


print("=" * 110)

PHASE 2 — XSD-GUIDED XML STRUCTURE ANALYSIS

[XSD] Schema File       : E:\MyPhD\My Patent-1\Implementation\db\drugbank.xsd
[XML] Dataset File      : E:\MyPhD\My Patent-1\Implementation\db\drugbank.xml
[XML] Namespace         : http://www.drugbank.ca

[PASS] DrugBank XSD parsed successfully.
[PASS] DrugBank XSD compiled successfully.

XSD ROOT METADATA
Target Namespace        : http://www.drugbank.ca
Element Form Default    : qualified
Attribute Form Default  : Not specified

XML ↔ XSD NAMESPACE CONSISTENCY
[PASS] DrugBank XML namespace matches the XSD target namespace.

ROOT STRUCTURE
[PASS] Top-level <drugbank> element identified.

Drug element:
Path                   : drugbank/drug
Type                   : drug-type
minOccurs              : 1
maxOccurs              : unbounded

DRUG-LEVEL ATTRIBUTES
type                      type=None                           use=required
created                   type=xs:date                        use=required
updated                   type=xs:da

#### Phase - 3 

# Phase 3: Streaming XML Dataset Extraction

**Objective:** Stream-parse `drugbank.xml` to construct raw tabular datasets ($\mathcal{D}_{\text{chem}}$ and $\mathcal{D}_{\text{target}}$) with minimal memory overhead, strictly adhering to the Phase 2 schema mappings[cite: 1].

* **Key Ingestion Rules:**
  * **Top-Level Record Scoping:** Ingest only `<drug>` nodes directly nested under `<drugbank>` to discard embedded pathway/reaction references[cite: 1].
  * **Scope & Bounds:** Filter for `type="small molecule"` up to the pre-configured limit ($N = 800$), capturing all associated target interactions[cite: 1].
  * **Raw Fidelity:** Extract and store unstandardized values (`raw_smiles`, multi-action lists, FASTA sequences) without applying chemical modifications[cite: 1].

* **Execution Steps:**
  1. Stream `<drug>` elements sequentially using `lxml.etree.iterparse(huge_tree=True)`[cite: 1].
  2. Extract primary DrugBank IDs, molecular descriptors (`raw_smiles`, `InChI`, `Formula`), and target/polypeptide relationships[cite: 1].
  3. Flush processed elements from memory immediately (`elem.clear()`) to avoid heap accumulation[cite: 1].
  4. Export raw tables to `D_chem_raw.csv` and `D_target_raw.csv` alongside extraction telemetry reports[cite: 1].

* **Expected Output:**
  Serialized raw chemical and target association tables in `./data/extracted/`, validated and queued for Phase 4 chemical sanitization[cite: 1].

In [3]:
# ==============================================================================
# PHASE 3 — STREAMING DRUGBANK XML EXTRACTION
# ==============================================================================
#
# PURPOSE
# -------
# 1. Stream-process the native DrugBank XML without loading the full file.
# 2. Process ONLY top-level:
#
#       drugbank
#          └── drug
#
#    and reject nested <drug> references occurring inside pathways or
#    other XML structures.
#
# 3. Restrict the initial experimental corpus to:
#
#       drug/@type == "small molecule"
#
# 4. Construct:
#
#       D_chem_raw
#       D_target_raw
#
# 5. Preserve RAW DrugBank values.
# 6. Preserve all target actions.
# 7. Handle multiple targets and multiple polypeptides correctly.
# 8. Generate extraction audit / quality-control statistics.
# 9. Save raw extraction outputs for Phase 4.
#
# IMPORTANT
# ---------
# Phase 3 DOES NOT perform:
#
#   - SMILES canonicalization
#   - salt stripping
#   - largest-fragment selection
#   - RDKit validation
#   - heavy-atom filtering
#   - molecular deduplication
#   - 3-D conformer generation
#   - molecular graph generation
#   - fingerprints
#   - neural-network training
#   - QED / SA evaluation
#   - UCB
#   - Fisher information
#   - EWC
#
# ==============================================================================


# ==============================================================================
# 1. IMPORTS
# ==============================================================================

from pathlib import Path
from lxml import etree

import pandas as pd
import json
from datetime import datetime


# ==============================================================================
# 2. VERIFY PHASE-1 AND PHASE-2 STATE
# ==============================================================================

required_variables = [
    "XML_PATH",
    "XSD_PATH",
    "XML_NAMESPACE",
    "NS",
    "NS_TAG",
    "DRUGBANK_CONFIG",
    "D_CHEM_SCHEMA",
    "D_TARGET_SCHEMA",
    "PHASE2_SCHEMA_MAP",
    "CALCULATED_PROPERTY_ENUMS",
    "EXTRACTED_DIR"
]


missing_variables = [
    variable
    for variable in required_variables
    if variable not in globals()
]


if missing_variables:

    raise RuntimeError(
        "Phase 1 and Phase 2 must be executed before Phase 3.\n"
        f"Missing variables: {missing_variables}"
    )


print("=" * 110)
print("PHASE 3 — STREAMING DRUGBANK XML EXTRACTION")
print("=" * 110)

print(f"\n[XML] Source File       : {XML_PATH}")
print(f"[XML] Namespace         : {XML_NAMESPACE}")
print(f"[OUT] Extraction Folder : {EXTRACTED_DIR}")


# ==============================================================================
# 3. VERIFY PHASE-2 SCHEMA VALIDATION
# ==============================================================================

d_chem_schema_verified = all(
    definition.get("xsd_verified", False)
    for definition in D_CHEM_SCHEMA.values()
)

d_target_schema_verified = all(
    definition.get("xsd_verified", False)
    for definition in D_TARGET_SCHEMA.values()
)


if not d_chem_schema_verified:

    raise RuntimeError(
        "D_chem schema mapping was not completely verified in Phase 2."
    )


if not d_target_schema_verified:

    raise RuntimeError(
        "D_target schema mapping was not completely verified in Phase 2."
    )


print("\n[PASS] D_chem schema mapping verified.")
print("[PASS] D_target schema mapping verified.")


# ==============================================================================
# 4. EXTRACTION CONFIGURATION
# ==============================================================================

# Development / experimental extraction limit.
#
# IMPORTANT:
# This applies ONLY to accepted unique TOP-LEVEL small-molecule records.
#
# It does NOT depend on the number of targets or polypeptides.

MAX_PHASE3_MOLECULES = int(
    DRUGBANK_CONFIG.get(
        "max_molecules",
        globals().get("MAX_MOLECULES", 800)
    )
)


TARGET_DRUG_TYPE = "small molecule"


# Calculated properties required for D_chem.
#
# These names were verified against the XSD enumeration during Phase 2.

REQUIRED_CALCULATED_PROPERTIES = {
    "SMILES": "raw_smiles",
    "InChI": "inchi",
    "InChIKey": "inchikey",
    "Molecular Formula": "molecular_formula",
    "Molecular Weight": "molecular_weight"
}


# Validate once more against Phase-2 enumerations.

unsupported_properties = [
    property_name
    for property_name in REQUIRED_CALCULATED_PROPERTIES
    if property_name not in CALCULATED_PROPERTY_ENUMS
]


if unsupported_properties:

    raise RuntimeError(
        "Required calculated properties are not supported "
        "by the Phase-2 XSD enumeration:\n"
        f"{unsupported_properties}"
    )


print("\n" + "=" * 110)
print("EXTRACTION CONFIGURATION")
print("=" * 110)

print(f"Target Drug Type       : {TARGET_DRUG_TYPE}")
print(f"Maximum Molecules      : {MAX_PHASE3_MOLECULES}")

print(
    "Calculated Properties : "
    + ", ".join(
        REQUIRED_CALCULATED_PROPERTIES.keys()
    )
)


# ==============================================================================
# 5. XML TAG CONSTANTS
# ==============================================================================

DRUG_TAG = f"{NS_TAG}drug"
DRUGBANK_ROOT_TAG = f"{NS_TAG}drugbank"

DRUGBANK_ID_TAG = f"{NS_TAG}drugbank-id"
NAME_TAG = f"{NS_TAG}name"

AVERAGE_MASS_TAG = f"{NS_TAG}average-mass"
MONOISOTOPIC_MASS_TAG = f"{NS_TAG}monoisotopic-mass"

GROUPS_TAG = f"{NS_TAG}groups"
GROUP_TAG = f"{NS_TAG}group"

CALCULATED_PROPERTIES_TAG = f"{NS_TAG}calculated-properties"
PROPERTY_TAG = f"{NS_TAG}property"

KIND_TAG = f"{NS_TAG}kind"
VALUE_TAG = f"{NS_TAG}value"
SOURCE_TAG = f"{NS_TAG}source"

TARGETS_TAG = f"{NS_TAG}targets"
TARGET_TAG = f"{NS_TAG}target"

ID_TAG = f"{NS_TAG}id"
ORGANISM_TAG = f"{NS_TAG}organism"
KNOWN_ACTION_TAG = f"{NS_TAG}known-action"

ACTIONS_TAG = f"{NS_TAG}actions"
ACTION_TAG = f"{NS_TAG}action"

POLYPEPTIDE_TAG = f"{NS_TAG}polypeptide"
AMINO_ACID_SEQUENCE_TAG = f"{NS_TAG}amino-acid-sequence"

PATHWAYS_TAG = f"{NS_TAG}pathways"

ENZYMES_TAG = f"{NS_TAG}enzymes"
CARRIERS_TAG = f"{NS_TAG}carriers"
TRANSPORTERS_TAG = f"{NS_TAG}transporters"


# ==============================================================================
# 6. HELPER FUNCTIONS
# ==============================================================================

def safe_text(element, child_tag):
    """
    Retrieve child text safely without modifying content.

    Only leading/trailing whitespace is stripped.
    """

    if element is None:
        return None

    child = element.find(
        child_tag
    )

    if child is None or child.text is None:
        return None

    value = child.text.strip()

    return value if value else None


def extract_primary_drugbank_id(drug):
    """
    Extract the primary DrugBank ID identified by:

        drugbank-id[@primary='true']
    """

    for db_id in drug.findall(
        DRUGBANK_ID_TAG
    ):

        if db_id.get("primary") == "true":

            if db_id.text:

                value = db_id.text.strip()

                if value:
                    return value

    return None


def extract_secondary_drugbank_ids(drug):
    """
    Preserve secondary DrugBank IDs for provenance.

    They are not used as the primary key.
    """

    secondary_ids = []

    for db_id in drug.findall(
        DRUGBANK_ID_TAG
    ):

        if db_id.get("primary") != "true":

            if db_id.text:

                value = db_id.text.strip()

                if value:
                    secondary_ids.append(
                        value
                    )

    return secondary_ids


def extract_groups(drug):
    """
    Extract all DrugBank group values.
    """

    values = []

    groups_element = drug.find(
        GROUPS_TAG
    )

    if groups_element is None:
        return values


    for group in groups_element.findall(
        GROUP_TAG
    ):

        if group.text:

            value = group.text.strip()

            if value:
                values.append(
                    value
                )

    return values


def extract_calculated_properties(drug):
    """
    Extract required raw calculated properties.

    No chemical transformation is performed.
    """

    values = {
        output_field: None
        for output_field
        in REQUIRED_CALCULATED_PROPERTIES.values()
    }


    sources = {
        output_field: None
        for output_field
        in REQUIRED_CALCULATED_PROPERTIES.values()
    }


    calculated_element = drug.find(
        CALCULATED_PROPERTIES_TAG
    )


    if calculated_element is None:

        return values, sources, False


    for property_element in calculated_element.findall(
        PROPERTY_TAG
    ):

        kind = safe_text(
            property_element,
            KIND_TAG
        )

        if kind not in REQUIRED_CALCULATED_PROPERTIES:
            continue


        value = safe_text(
            property_element,
            VALUE_TAG
        )

        source = safe_text(
            property_element,
            SOURCE_TAG
        )


        output_field = (
            REQUIRED_CALCULATED_PROPERTIES[
                kind
            ]
        )


        # Preserve the first actual value observed.
        #
        # This avoids silently replacing an earlier value if duplicate
        # property entries exist in the XML.

        if values[
            output_field
        ] is None:

            values[
                output_field
            ] = value

            sources[
                output_field
            ] = source


    return values, sources, True


def extract_actions(target):
    """
    Extract ALL actions associated with a DrugBank target.
    """

    values = []

    actions_element = target.find(
        ACTIONS_TAG
    )

    if actions_element is None:
        return values


    for action in actions_element.findall(
        ACTION_TAG
    ):

        if action.text:

            value = action.text.strip()

            if value:
                values.append(
                    value
                )

    return values


def serialize_list(values):
    """
    Serialize a Python list safely for CSV storage.

    JSON representation preserves the distinction between:
        []
        ["approved"]
        ["approved", "investigational"]
    """

    return json.dumps(
        values,
        ensure_ascii=False
    )


def cleanup_processed_top_level_element(element):
    """
    Aggressively release processed top-level XML records.

    With lxml.iterparse, clearing the processed element and removing
    previously processed root siblings keeps memory usage stable.
    """

    parent = element.getparent()

    element.clear()

    if parent is not None:

        while element.getprevious() is not None:

            del parent[0]


# ==============================================================================
# 7. OUTPUT COLLECTIONS
# ==============================================================================

chem_records = []

target_records = []


# Track accepted primary IDs so the 800-record limit applies to unique
# top-level DrugBank molecular records rather than repeated XML records.

accepted_drug_ids = set()


# ==============================================================================
# 8. EXTRACTION AUDIT COUNTERS
# ==============================================================================

stats = {
    # XML hierarchy
    "xml_drug_elements_encountered": 0,
    "top_level_drug_records": 0,
    "nested_drug_elements_ignored": 0,

    # Drug types
    "small_molecule_records_seen": 0,
    "biotech_records_skipped": 0,
    "other_drug_types_skipped": 0,

    # Identifier QC
    "records_with_primary_id": 0,
    "records_without_primary_id": 0,
    "duplicate_primary_ids_skipped": 0,

    # Molecular metadata
    "records_with_calculated_properties": 0,
    "records_without_calculated_properties": 0,

    "records_with_smiles": 0,
    "records_without_smiles": 0,

    "records_with_inchi": 0,
    "records_with_inchikey": 0,
    "records_with_formula": 0,
    "records_with_molecular_weight": 0,

    # Selected corpus
    "accepted_D_chem_records": 0,

    # Groups
    "records_with_groups": 0,

    # Targets
    "accepted_drugs_with_targets": 0,
    "total_target_associations": 0,
    "targets_with_known_action_yes": 0,
    "targets_with_actions": 0,
    "targets_without_actions": 0,

    # Polypeptides
    "targets_with_polypeptides": 0,
    "targets_without_polypeptides": 0,
    "total_polypeptide_associations": 0,
    "polypeptides_with_sequence": 0,
    "polypeptides_without_sequence": 0,

    # Biological sections
    "accepted_drugs_with_enzymes": 0,
    "accepted_drugs_with_carriers": 0,
    "accepted_drugs_with_transporters": 0,
    "accepted_drugs_with_pathways": 0,

    # Final target rows
    "D_target_rows_created": 0
}


# ==============================================================================
# 9. STREAM DRUGBANK XML
# ==============================================================================

print("\n" + "=" * 110)
print("STREAMING DRUGBANK XML")
print("=" * 110)


extraction_started_at = datetime.now()


with open(
    XML_PATH,
    "rb"
) as xml_file:

    context = etree.iterparse(
        xml_file,
        events=("end",),
        tag=DRUG_TAG,
        huge_tree=True
    )


    for event, drug in context:

        stats[
            "xml_drug_elements_encountered"
        ] += 1


        # ======================================================================
        # 9.1 CRITICAL TOP-LEVEL RECORD CHECK
        # ======================================================================

        parent = drug.getparent()


        is_top_level_drug = (
            parent is not None
            and parent.tag == DRUGBANK_ROOT_TAG
        )


        if not is_top_level_drug:

            stats[
                "nested_drug_elements_ignored"
            ] += 1

            # Only clear nested reference.
            #
            # Do NOT aggressively delete parent siblings here because the
            # enclosing top-level DrugBank record is still being parsed.

            drug.clear()

            continue


        stats[
            "top_level_drug_records"
        ] += 1


        # ======================================================================
        # 9.2 DRUG TYPE FILTER
        # ======================================================================

        drug_type = drug.get(
            "type"
        )


        if drug_type == "biotech":

            stats[
                "biotech_records_skipped"
            ] += 1

            cleanup_processed_top_level_element(
                drug
            )

            continue


        if drug_type != TARGET_DRUG_TYPE:

            stats[
                "other_drug_types_skipped"
            ] += 1

            cleanup_processed_top_level_element(
                drug
            )

            continue


        stats[
            "small_molecule_records_seen"
        ] += 1


        # ======================================================================
        # 9.3 PRIMARY DRUGBANK ID
        # ======================================================================

        drug_id = extract_primary_drugbank_id(
            drug
        )


        if drug_id is None:

            stats[
                "records_without_primary_id"
            ] += 1

            cleanup_processed_top_level_element(
                drug
            )

            continue


        stats[
            "records_with_primary_id"
        ] += 1


        # Protect against accidental duplicate full records.

        if drug_id in accepted_drug_ids:

            stats[
                "duplicate_primary_ids_skipped"
            ] += 1

            cleanup_processed_top_level_element(
                drug
            )

            continue


        # ======================================================================
        # 9.4 BASIC DRUG METADATA
        # ======================================================================

        drug_name = safe_text(
            drug,
            NAME_TAG
        )


        secondary_ids = (
            extract_secondary_drugbank_ids(
                drug
            )
        )


        average_mass = safe_text(
            drug,
            AVERAGE_MASS_TAG
        )


        monoisotopic_mass = safe_text(
            drug,
            MONOISOTOPIC_MASS_TAG
        )


        groups = extract_groups(
            drug
        )


        if groups:

            stats[
                "records_with_groups"
            ] += 1


        # ======================================================================
        # 9.5 CALCULATED PROPERTIES
        # ======================================================================

        (
            calculated_values,
            calculated_sources,
            calculated_properties_present
        ) = extract_calculated_properties(
            drug
        )


        if calculated_properties_present:

            stats[
                "records_with_calculated_properties"
            ] += 1

        else:

            stats[
                "records_without_calculated_properties"
            ] += 1


        raw_smiles = calculated_values[
            "raw_smiles"
        ]

        inchi = calculated_values[
            "inchi"
        ]

        inchikey = calculated_values[
            "inchikey"
        ]

        molecular_formula = calculated_values[
            "molecular_formula"
        ]

        molecular_weight = calculated_values[
            "molecular_weight"
        ]


        if raw_smiles:

            stats[
                "records_with_smiles"
            ] += 1

        else:

            stats[
                "records_without_smiles"
            ] += 1


        if inchi:

            stats[
                "records_with_inchi"
            ] += 1


        if inchikey:

            stats[
                "records_with_inchikey"
            ] += 1


        if molecular_formula:

            stats[
                "records_with_formula"
            ] += 1


        if molecular_weight:

            stats[
                "records_with_molecular_weight"
            ] += 1


        # ======================================================================
        # 9.6 CREATE RAW D_chem RECORD
        # ======================================================================

        chem_record = {
            "drug_id":
                drug_id,

            "secondary_drugbank_ids":
                serialize_list(
                    secondary_ids
                ),

            "drug_name":
                drug_name,

            "drug_type":
                drug_type,

            "raw_smiles":
                raw_smiles,

            "smiles_source":
                calculated_sources[
                    "raw_smiles"
                ],

            "inchi":
                inchi,

            "inchi_source":
                calculated_sources[
                    "inchi"
                ],

            "inchikey":
                inchikey,

            "inchikey_source":
                calculated_sources[
                    "inchikey"
                ],

            "molecular_formula":
                molecular_formula,

            "molecular_formula_source":
                calculated_sources[
                    "molecular_formula"
                ],

            "molecular_weight":
                molecular_weight,

            "molecular_weight_source":
                calculated_sources[
                    "molecular_weight"
                ],

            "average_mass":
                average_mass,

            "monoisotopic_mass":
                monoisotopic_mass,

            "groups":
                serialize_list(
                    groups
                )
        }


        chem_records.append(
            chem_record
        )


        accepted_drug_ids.add(
            drug_id
        )


        stats[
            "accepted_D_chem_records"
        ] += 1


        # ======================================================================
        # 9.7 ADDITIONAL BIOLOGICAL SECTION AVAILABILITY
        # ======================================================================

        if drug.find(
            ENZYMES_TAG
        ) is not None:

            enzymes_element = drug.find(
                ENZYMES_TAG
            )

            if (
                enzymes_element is not None
                and len(enzymes_element) > 0
            ):

                stats[
                    "accepted_drugs_with_enzymes"
                ] += 1


        if drug.find(
            CARRIERS_TAG
        ) is not None:

            carriers_element = drug.find(
                CARRIERS_TAG
            )

            if (
                carriers_element is not None
                and len(carriers_element) > 0
            ):

                stats[
                    "accepted_drugs_with_carriers"
                ] += 1


        if drug.find(
            TRANSPORTERS_TAG
        ) is not None:

            transporters_element = drug.find(
                TRANSPORTERS_TAG
            )

            if (
                transporters_element is not None
                and len(transporters_element) > 0
            ):

                stats[
                    "accepted_drugs_with_transporters"
                ] += 1


        if drug.find(
            PATHWAYS_TAG
        ) is not None:

            pathways_element = drug.find(
                PATHWAYS_TAG
            )

            if (
                pathways_element is not None
                and len(pathways_element) > 0
            ):

                stats[
                    "accepted_drugs_with_pathways"
                ] += 1


        # ======================================================================
        # 9.8 TARGET ASSOCIATION EXTRACTION
        # ======================================================================

        targets_element = drug.find(
            TARGETS_TAG
        )


        target_elements = []

        if targets_element is not None:

            target_elements = targets_element.findall(
                TARGET_TAG
            )


        if target_elements:

            stats[
                "accepted_drugs_with_targets"
            ] += 1


        for target in target_elements:

            stats[
                "total_target_associations"
            ] += 1


            target_id = safe_text(
                target,
                ID_TAG
            )


            target_name = safe_text(
                target,
                NAME_TAG
            )


            organism = safe_text(
                target,
                ORGANISM_TAG
            )


            known_action = safe_text(
                target,
                KNOWN_ACTION_TAG
            )


            actions = extract_actions(
                target
            )


            if (
                known_action is not None
                and known_action.strip().lower() == "yes"
            ):

                stats[
                    "targets_with_known_action_yes"
                ] += 1


            if actions:

                stats[
                    "targets_with_actions"
                ] += 1

            else:

                stats[
                    "targets_without_actions"
                ] += 1


            polypeptides = target.findall(
                POLYPEPTIDE_TAG
            )


            # ------------------------------------------------------------------
            # Case A:
            # Target contains one or more polypeptides.
            # One output row is generated per polypeptide.
            # ------------------------------------------------------------------

            if polypeptides:

                stats[
                    "targets_with_polypeptides"
                ] += 1


                for polypeptide in polypeptides:

                    stats[
                        "total_polypeptide_associations"
                    ] += 1


                    polypeptide_id = (
                        polypeptide.get(
                            "id"
                        )
                    )


                    polypeptide_source = (
                        polypeptide.get(
                            "source"
                        )
                    )


                    amino_acid_sequence = safe_text(
                        polypeptide,
                        AMINO_ACID_SEQUENCE_TAG
                    )


                    if amino_acid_sequence:

                        stats[
                            "polypeptides_with_sequence"
                        ] += 1

                    else:

                        stats[
                            "polypeptides_without_sequence"
                        ] += 1


                    target_records.append({
                        "drug_id":
                            drug_id,

                        "drug_name":
                            drug_name,

                        "raw_smiles":
                            raw_smiles,

                        "target_id":
                            target_id,

                        "target_name":
                            target_name,

                        "organism":
                            organism,

                        "known_action":
                            known_action,

                        "actions":
                            serialize_list(
                                actions
                            ),

                        "polypeptide_id":
                            polypeptide_id,

                        "polypeptide_source":
                            polypeptide_source,

                        "amino_acid_sequence":
                            amino_acid_sequence
                    })


                    stats[
                        "D_target_rows_created"
                    ] += 1


            # ------------------------------------------------------------------
            # Case B:
            # Target exists but contains no polypeptide.
            #
            # Preserve the target association rather than silently losing it.
            # ------------------------------------------------------------------

            else:

                stats[
                    "targets_without_polypeptides"
                ] += 1


                target_records.append({
                    "drug_id":
                        drug_id,

                    "drug_name":
                        drug_name,

                    "raw_smiles":
                        raw_smiles,

                    "target_id":
                        target_id,

                    "target_name":
                        target_name,

                    "organism":
                        organism,

                    "known_action":
                        known_action,

                    "actions":
                        serialize_list(
                            actions
                        ),

                    "polypeptide_id":
                        None,

                    "polypeptide_source":
                        None,

                    "amino_acid_sequence":
                        None
                })


                stats[
                    "D_target_rows_created"
                ] += 1


        # ======================================================================
        # 9.9 MEMORY CLEANUP
        # ======================================================================

        cleanup_processed_top_level_element(
            drug
        )


        # ======================================================================
        # 9.10 DEVELOPMENT EXTRACTION LIMIT
        # ======================================================================

        if (
            stats[
                "accepted_D_chem_records"
            ]
            >= MAX_PHASE3_MOLECULES
        ):

            print(
                "\n[LIMIT] Reached configured D_chem extraction limit: "
                f"{MAX_PHASE3_MOLECULES}"
            )

            break


    del context


extraction_finished_at = datetime.now()

extraction_duration = (
    extraction_finished_at
    - extraction_started_at
)


# ==============================================================================
# 10. CREATE RAW DATAFRAMES
# ==============================================================================

D_chem_raw = pd.DataFrame(
    chem_records
)


D_target_raw = pd.DataFrame(
    target_records
)


# ==============================================================================
# 11. ENFORCE EXPECTED COLUMN ORDER
# ==============================================================================

D_CHEM_COLUMNS = [
    "drug_id",
    "secondary_drugbank_ids",
    "drug_name",
    "drug_type",
    "raw_smiles",
    "smiles_source",
    "inchi",
    "inchi_source",
    "inchikey",
    "inchikey_source",
    "molecular_formula",
    "molecular_formula_source",
    "molecular_weight",
    "molecular_weight_source",
    "average_mass",
    "monoisotopic_mass",
    "groups"
]


D_TARGET_COLUMNS = [
    "drug_id",
    "drug_name",
    "raw_smiles",
    "target_id",
    "target_name",
    "organism",
    "known_action",
    "actions",
    "polypeptide_id",
    "polypeptide_source",
    "amino_acid_sequence"
]


if D_chem_raw.empty:

    D_chem_raw = pd.DataFrame(
        columns=D_CHEM_COLUMNS
    )

else:

    D_chem_raw = D_chem_raw[
        D_CHEM_COLUMNS
    ]


if D_target_raw.empty:

    D_target_raw = pd.DataFrame(
        columns=D_TARGET_COLUMNS
    )

else:

    D_target_raw = D_target_raw[
        D_TARGET_COLUMNS
    ]


# ==============================================================================
# 12. EXACT XML-RECORD DUPLICATE CHECK
# ==============================================================================
#
# This does NOT perform chemical deduplication.
#
# We only detect accidental duplicate extraction rows.
# ==============================================================================

chem_exact_duplicates = int(
    D_chem_raw.duplicated().sum()
)


target_exact_duplicates = int(
    D_target_raw.duplicated().sum()
)


stats[
    "exact_duplicate_D_chem_rows"
] = chem_exact_duplicates


stats[
    "exact_duplicate_D_target_rows"
] = target_exact_duplicates


# Remove only exact extraction duplicates, if any.

if chem_exact_duplicates > 0:

    D_chem_raw = (
        D_chem_raw
        .drop_duplicates()
        .reset_index(drop=True)
    )


if target_exact_duplicates > 0:

    D_target_raw = (
        D_target_raw
        .drop_duplicates()
        .reset_index(drop=True)
    )


# ==============================================================================
# 13. DATA-INTEGRITY CHECKS
# ==============================================================================

print("\n" + "=" * 110)
print("PHASE 3 DATA-INTEGRITY VALIDATION")
print("=" * 110)


integrity_checks = {}


# ------------------------------------------------------------------------------
# D_chem primary IDs
# ------------------------------------------------------------------------------

integrity_checks[
    "D_chem created"
] = len(
    D_chem_raw
) > 0


integrity_checks[
    "All D_chem drug IDs present"
] = (
    not D_chem_raw.empty
    and D_chem_raw[
        "drug_id"
    ].notna().all()
)


integrity_checks[
    "All D_chem records are small molecules"
] = (
    not D_chem_raw.empty
    and (
        D_chem_raw[
            "drug_type"
        ]
        == TARGET_DRUG_TYPE
    ).all()
)


integrity_checks[
    "D_chem primary IDs are unique"
] = (
    not D_chem_raw.empty
    and D_chem_raw[
        "drug_id"
    ].is_unique
)


# ------------------------------------------------------------------------------
# Nested-pathway false records protection
# ------------------------------------------------------------------------------

integrity_checks[
    "No accepted D_chem record lacks drug type"
] = (
    not D_chem_raw.empty
    and D_chem_raw[
        "drug_type"
    ].notna().all()
)


# ------------------------------------------------------------------------------
# D_target integrity
# ------------------------------------------------------------------------------

if D_target_raw.empty:

    integrity_checks[
        "D_target associations extracted"
    ] = False

    integrity_checks[
        "All D_target drug IDs belong to D_chem"
    ] = False

else:

    integrity_checks[
        "D_target associations extracted"
    ] = True


    valid_chem_ids = set(
        D_chem_raw[
            "drug_id"
        ].dropna()
    )


    target_drug_ids = set(
        D_target_raw[
            "drug_id"
        ].dropna()
    )


    integrity_checks[
        "All D_target drug IDs belong to D_chem"
    ] = target_drug_ids.issubset(
        valid_chem_ids
    )


# ==============================================================================
# 14. DISPLAY DATA-INTEGRITY RESULTS
# ==============================================================================

for check_name, status in integrity_checks.items():

    label = (
        "PASS"
        if status
        else "FAIL"
    )

    print(
        f"[{label:<4}] {check_name}"
    )


# ==============================================================================
# 15. EXTRACTION SUMMARY
# ==============================================================================

stats[
    "final_D_chem_rows"
] = len(
    D_chem_raw
)


stats[
    "final_D_target_rows"
] = len(
    D_target_raw
)


stats[
    "extraction_limit"
] = MAX_PHASE3_MOLECULES


stats[
    "target_drug_type"
] = TARGET_DRUG_TYPE


stats[
    "drugbank_version"
] = DRUGBANK_CONFIG.get(
    "drugbank_version"
)


stats[
    "drugbank_exported_on"
] = DRUGBANK_CONFIG.get(
    "drugbank_exported_on"
)


stats[
    "xml_source_file"
] = str(
    XML_PATH
)


stats[
    "xsd_source_file"
] = str(
    XSD_PATH
)


stats[
    "extraction_started_at"
] = extraction_started_at.isoformat()


stats[
    "extraction_finished_at"
] = extraction_finished_at.isoformat()


stats[
    "extraction_duration_seconds"
] = round(
    extraction_duration.total_seconds(),
    3
)


# ==============================================================================
# 16. SAVE RAW DATASETS
# ==============================================================================

EXTRACTED_DIR.mkdir(
    parents=True,
    exist_ok=True
)


D_CHEM_FILE = (
    EXTRACTED_DIR
    / "D_chem_raw.csv"
)


D_TARGET_FILE = (
    EXTRACTED_DIR
    / "D_target_raw.csv"
)


SUMMARY_CSV_FILE = (
    EXTRACTED_DIR
    / "phase3_extraction_summary.csv"
)


SUMMARY_JSON_FILE = (
    EXTRACTED_DIR
    / "phase3_extraction_summary.json"
)


# ------------------------------------------------------------------------------
# Save raw molecular dataset
# ------------------------------------------------------------------------------

D_chem_raw.to_csv(
    D_CHEM_FILE,
    index=False,
    encoding="utf-8"
)


# ------------------------------------------------------------------------------
# Save raw target association dataset
# ------------------------------------------------------------------------------

D_target_raw.to_csv(
    D_TARGET_FILE,
    index=False,
    encoding="utf-8"
)


# ------------------------------------------------------------------------------
# Save summary CSV
# ------------------------------------------------------------------------------

summary_df = pd.DataFrame([
    {
        "metric": key,
        "value": value
    }
    for key, value in stats.items()
])


summary_df.to_csv(
    SUMMARY_CSV_FILE,
    index=False,
    encoding="utf-8"
)


# ------------------------------------------------------------------------------
# Save summary JSON
# ------------------------------------------------------------------------------

with open(
    SUMMARY_JSON_FILE,
    "w",
    encoding="utf-8"
) as json_file:

    json.dump(
        stats,
        json_file,
        indent=4,
        ensure_ascii=False,
        default=str
    )


# ==============================================================================
# 17. DISPLAY EXTRACTION AUDIT
# ==============================================================================

print("\n" + "=" * 110)
print("PHASE 3 EXTRACTION AUDIT")
print("=" * 110)


audit_metrics = [
    (
        "XML <drug> elements encountered",
        stats[
            "xml_drug_elements_encountered"
        ]
    ),

    (
        "Top-level DrugBank drugs",
        stats[
            "top_level_drug_records"
        ]
    ),

    (
        "Nested <drug> references ignored",
        stats[
            "nested_drug_elements_ignored"
        ]
    ),

    (
        "Small molecules encountered",
        stats[
            "small_molecule_records_seen"
        ]
    ),

    (
        "Biotech records skipped",
        stats[
            "biotech_records_skipped"
        ]
    ),

    (
        "Records with primary ID",
        stats[
            "records_with_primary_id"
        ]
    ),

    (
        "Records without primary ID",
        stats[
            "records_without_primary_id"
        ]
    ),

    (
        "Accepted D_chem records",
        stats[
            "accepted_D_chem_records"
        ]
    ),

    (
        "Records with calculated properties",
        stats[
            "records_with_calculated_properties"
        ]
    ),

    (
        "Records with SMILES",
        stats[
            "records_with_smiles"
        ]
    ),

    (
        "Records without SMILES",
        stats[
            "records_without_smiles"
        ]
    ),

    (
        "Accepted drugs with targets",
        stats[
            "accepted_drugs_with_targets"
        ]
    ),

    (
        "Target associations",
        stats[
            "total_target_associations"
        ]
    ),

    (
        "Targets with known-action=yes",
        stats[
            "targets_with_known_action_yes"
        ]
    ),

    (
        "Targets with actions",
        stats[
            "targets_with_actions"
        ]
    ),

    (
        "Targets with polypeptides",
        stats[
            "targets_with_polypeptides"
        ]
    ),

    (
        "Targets without polypeptides",
        stats[
            "targets_without_polypeptides"
        ]
    ),

    (
        "Polypeptide associations",
        stats[
            "total_polypeptide_associations"
        ]
    ),

    (
        "Polypeptides with sequence",
        stats[
            "polypeptides_with_sequence"
        ]
    ),

    (
        "Final D_target rows",
        stats[
            "final_D_target_rows"
        ]
    )
]


for metric, value in audit_metrics:

    print(
        f"{metric:<45}: {value}"
    )


# ==============================================================================
# 18. DISPLAY SMALL DATA PREVIEW
# ==============================================================================

print("\n" + "=" * 110)
print("D_chem_raw PREVIEW")
print("=" * 110)

if D_chem_raw.empty:

    print("D_chem_raw is empty.")

else:

    preview_columns = [
        "drug_id",
        "drug_name",
        "drug_type",
        "raw_smiles",
        "molecular_formula",
        "molecular_weight",
        "groups"
    ]

    print(
        D_chem_raw[
            preview_columns
        ]
        .head(5)
        .to_string(
            index=False
        )
    )


print("\n" + "=" * 110)
print("D_target_raw PREVIEW")
print("=" * 110)

if D_target_raw.empty:

    print("D_target_raw is empty.")

else:

    preview_columns = [
        "drug_id",
        "target_id",
        "target_name",
        "organism",
        "known_action",
        "actions",
        "polypeptide_id",
        "polypeptide_source"
    ]

    print(
        D_target_raw[
            preview_columns
        ]
        .head(10)
        .to_string(
            index=False
        )
    )


# ==============================================================================
# 19. PHASE-3 COMPLETION CRITERIA
# ==============================================================================

completion_checks = {
    "Streaming XML processing completed":
        stats[
            "xml_drug_elements_encountered"
        ] > 0,

    "Top-level DrugBank records detected":
        stats[
            "top_level_drug_records"
        ] > 0,

    "Nested <drug> references excluded":
        (
            stats[
                "nested_drug_elements_ignored"
            ] >= 0
        ),

    "Small-molecule records identified":
        stats[
            "small_molecule_records_seen"
        ] > 0,

    "Primary DrugBank IDs extracted":
        stats[
            "records_with_primary_id"
        ] > 0,

    "Raw D_chem created":
        len(
            D_chem_raw
        ) > 0,

    "D_chem IDs unique":
        (
            len(
                D_chem_raw
            ) > 0
            and D_chem_raw[
                "drug_id"
            ].is_unique
        ),

    "Calculated properties extracted":
        stats[
            "records_with_calculated_properties"
        ] > 0,

    "SMILES values encountered":
        stats[
            "records_with_smiles"
        ] > 0,

    "Target relationships extracted":
        stats[
            "total_target_associations"
        ] > 0,

    "D_target created":
        len(
            D_target_raw
        ) > 0,

    "Multiple-polypeptide-compatible extraction used":
        True,

    "Target drug IDs linked to D_chem":
        integrity_checks[
            "All D_target drug IDs belong to D_chem"
        ],

    "Raw D_chem file saved":
        D_CHEM_FILE.exists(),

    "Raw D_target file saved":
        D_TARGET_FILE.exists(),

    "Extraction summary CSV saved":
        SUMMARY_CSV_FILE.exists(),

    "Extraction summary JSON saved":
        SUMMARY_JSON_FILE.exists()
}


print("\n" + "=" * 110)
print("PHASE 3 COMPLETION VALIDATION")
print("=" * 110)


all_critical_checks_passed = True


for check_name, status in completion_checks.items():

    label = (
        "PASS"
        if status
        else "FAIL"
    )

    print(
        f"[{label:<4}] {check_name}"
    )

    if not status:

        all_critical_checks_passed = False


# ==============================================================================
# 20. PACKAGE PHASE-3 CONFIGURATION
# ==============================================================================

PHASE3_EXTRACTION = {
    "D_chem_file":
        str(D_CHEM_FILE),

    "D_target_file":
        str(D_TARGET_FILE),

    "summary_csv":
        str(SUMMARY_CSV_FILE),

    "summary_json":
        str(SUMMARY_JSON_FILE),

    "max_molecules":
        MAX_PHASE3_MOLECULES,

    "target_drug_type":
        TARGET_DRUG_TYPE,

    "D_chem_rows":
        len(D_chem_raw),

    "D_target_rows":
        len(D_target_raw),

    "stats":
        stats
}


DRUGBANK_CONFIG[
    "phase3_extraction"
] = PHASE3_EXTRACTION


# ==============================================================================
# 21. FINAL PHASE-3 SUMMARY
# ==============================================================================

print("\n" + "=" * 110)
print("PHASE 3 SUMMARY")
print("=" * 110)

print(
    f"D_chem Raw Records             : "
    f"{len(D_chem_raw)}"
)

print(
    f"D_target Raw Records           : "
    f"{len(D_target_raw)}"
)

print(
    f"Nested Drug References Ignored : "
    f"{stats['nested_drug_elements_ignored']}"
)

print(
    f"Records With SMILES            : "
    f"{stats['records_with_smiles']}"
)

print(
    f"Total Target Associations      : "
    f"{stats['total_target_associations']}"
)

print(
    f"Total Polypeptide Associations : "
    f"{stats['total_polypeptide_associations']}"
)

print(
    f"Polypeptides With Sequence     : "
    f"{stats['polypeptides_with_sequence']}"
)

print(
    f"Extraction Duration (seconds)  : "
    f"{stats['extraction_duration_seconds']}"
)


print("\nSaved Files:")

print(
    f"  D_chem_raw   : {D_CHEM_FILE}"
)

print(
    f"  D_target_raw : {D_TARGET_FILE}"
)

print(
    f"  Summary CSV  : {SUMMARY_CSV_FILE}"
)

print(
    f"  Summary JSON : {SUMMARY_JSON_FILE}"
)


print("-" * 110)


if all_critical_checks_passed:

    print(
        "[STATUS] PHASE 3 COMPLETED SUCCESSFULLY."
    )

    print(
        "[READY]  Raw DrugBank molecular and target-association "
        "datasets have been created."
    )

    print(
        "[NEXT]   Phase 4 — Molecular Data Cleaning, "
        "Standardization and ML-Ready Dataset Construction"
    )

else:

    print(
        "[STATUS] PHASE 3 COMPLETED WITH ONE OR MORE "
        "FAILED VALIDATION CHECKS."
    )

    print(
        "[ACTION] Review failed checks before proceeding "
        "to Phase 4."
    )


print("=" * 110)

PHASE 3 — STREAMING DRUGBANK XML EXTRACTION

[XML] Source File       : E:\MyPhD\My Patent-1\Implementation\db\drugbank.xml
[XML] Namespace         : http://www.drugbank.ca
[OUT] Extraction Folder : E:\MyPhD\My Patent-1\Implementation\data\extracted

[PASS] D_chem schema mapping verified.
[PASS] D_target schema mapping verified.

EXTRACTION CONFIGURATION
Target Drug Type       : small molecule
Maximum Molecules      : 800
Calculated Properties : SMILES, InChI, InChIKey, Molecular Formula, Molecular Weight

STREAMING DRUGBANK XML

[LIMIT] Reached configured D_chem extraction limit: 800

PHASE 3 DATA-INTEGRITY VALIDATION
[PASS] D_chem created
[PASS] All D_chem drug IDs present
[PASS] All D_chem records are small molecules
[PASS] D_chem primary IDs are unique
[PASS] No accepted D_chem record lacks drug type
[PASS] D_target associations extracted
[PASS] All D_target drug IDs belong to D_chem

PHASE 3 EXTRACTION AUDIT
XML <drug> elements encountered              : 23411
Top-level DrugBank dr

# Phase 4 — XML-to-ML Intermediate Dataset Construction

**Objective:** Transition raw extracted records (`D_chem_raw.csv` and `D_target_raw.csv`) into ML-ready working representations (`D_chem` and `D_target`) while ensuring raw value preservation, relational linkage integrity, and multi-format serialization without applying chemical filtering.

* **Key Ingestion & Integrity Rules:**
  * **Raw Preservation:** Maintain raw extracted values without altering SMILES, masses, or chemical structures.
  * **Linkage Verification:** Confirm that 100% of target association records trace directly to a valid primary DrugBank ID in `D_chem`.
  * **One-to-Many Multiplicity:** Preserve multiple targets per drug and multiple polypeptides per target.

* **Execution Steps:**
  1. Load raw CSV extractions with explicit string typing to prevent numerical truncation.
  2. Perform schema and column validation against Phase 2/3 definitions.
  3. Validate structural linkage and generate an orphan-record diagnostic report.
  4. Serialize working datasets to `./data/processed/` in both CSV and Parquet formats (`D_chem`, `D_target`).
  5. Export dataset summary and provenance metadata to `phase4_dataset_summary.json`.

* **Expected Output:**
  Verified working datasets `D_chem` and `D_target` saved in `./data/processed/`, establishing the working representation for Phase 5 chemical curation and graph generation.

In [4]:
# ==============================================================================
# PHASE 4 — XML-to-ML INTERMEDIATE DATASET CONSTRUCTION
# ==============================================================================
#
# PURPOSE
# -------
# 1. Load the raw structured datasets generated in Phase 3.
# 2. Create reusable ML working datasets:
#
#       D_chem
#       D_target
#
# 3. Preserve raw molecular values exactly as extracted.
# 4. Preserve one-to-many drug-target-polypeptide relationships.
# 5. Validate structural consistency and dataset linkage.
# 6. Save both CSV and Parquet representations.
# 7. Save dataset summary / provenance metadata.
#
# IMPORTANT
# ---------
# This phase DOES NOT perform:
#
#   - RDKit parsing
#   - SMILES validation
#   - SMILES canonicalization
#   - salt stripping
#   - fragment selection
#   - heavy-atom filtering
#   - chemical deduplication
#   - 3-D conformer generation
#   - molecular graph construction
#   - fingerprints
#   - neural-network training
#   - latent-space learning
#
# These belong to Phase 5 and later phases.
# ==============================================================================


# ==============================================================================
# 1. IMPORTS
# ==============================================================================

from pathlib import Path
import pandas as pd
import json
from datetime import datetime


# ==============================================================================
# 2. VERIFY REQUIRED STATE
# ==============================================================================

required_variables = [
    "DRUGBANK_CONFIG",
    "EXTRACTED_DIR",
    "PROCESSED_DIR"
]

missing_variables = [
    variable
    for variable in required_variables
    if variable not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Phase 1 and Phase 3 must be executed before Phase 4.\n"
        f"Missing variables: {missing_variables}"
    )


print("=" * 110)
print("PHASE 4 — XML-to-ML INTERMEDIATE DATASET CONSTRUCTION")
print("=" * 110)

print(f"\n[IN ] Raw Extraction Directory : {EXTRACTED_DIR}")
print(f"[OUT] Processed Directory      : {PROCESSED_DIR}")


# ==============================================================================
# 3. IDENTIFY PHASE-3 INPUT FILES
# ==============================================================================

D_CHEM_RAW_FILE = (
    EXTRACTED_DIR
    / "D_chem_raw.csv"
)

D_TARGET_RAW_FILE = (
    EXTRACTED_DIR
    / "D_target_raw.csv"
)

PHASE3_SUMMARY_JSON_FILE = (
    EXTRACTED_DIR
    / "phase3_extraction_summary.json"
)


required_input_files = [
    D_CHEM_RAW_FILE,
    D_TARGET_RAW_FILE
]


missing_files = [
    str(file_path)
    for file_path in required_input_files
    if not file_path.exists()
]


if missing_files:
    raise FileNotFoundError(
        "Required Phase-3 output files were not found:\n"
        + "\n".join(missing_files)
    )


print("\n[PASS] Phase-3 raw molecular dataset found.")
print("[PASS] Phase-3 raw target dataset found.")


# ==============================================================================
# 4. LOAD RAW DATASETS
# ==============================================================================

D_chem_raw = pd.read_csv(
    D_CHEM_RAW_FILE,
    dtype=str,
    keep_default_na=False
)


D_target_raw = pd.read_csv(
    D_TARGET_RAW_FILE,
    dtype=str,
    keep_default_na=False
)


print("\n" + "=" * 110)
print("RAW DATASET LOAD SUMMARY")
print("=" * 110)

print(f"D_chem_raw rows    : {len(D_chem_raw)}")
print(f"D_chem_raw columns : {len(D_chem_raw.columns)}")

print(f"D_target_raw rows  : {len(D_target_raw)}")
print(f"D_target_raw cols  : {len(D_target_raw.columns)}")


# ==============================================================================
# 5. LOAD PHASE-3 EXTRACTION METADATA
# ==============================================================================

phase3_summary = {}

if PHASE3_SUMMARY_JSON_FILE.exists():

    with open(
        PHASE3_SUMMARY_JSON_FILE,
        "r",
        encoding="utf-8"
    ) as json_file:

        phase3_summary = json.load(
            json_file
        )

    print(
        "[PASS] Phase-3 extraction metadata loaded."
    )

else:

    print(
        "[NOTE] Phase-3 summary JSON not found. "
        "Phase 4 will continue with available runtime metadata."
    )


# ==============================================================================
# 6. EXPECTED RAW DATASET STRUCTURE
# ==============================================================================

EXPECTED_D_CHEM_COLUMNS = [
    "drug_id",
    "secondary_drugbank_ids",
    "drug_name",
    "drug_type",
    "raw_smiles",
    "smiles_source",
    "inchi",
    "inchi_source",
    "inchikey",
    "inchikey_source",
    "molecular_formula",
    "molecular_formula_source",
    "molecular_weight",
    "molecular_weight_source",
    "average_mass",
    "monoisotopic_mass",
    "groups"
]


EXPECTED_D_TARGET_COLUMNS = [
    "drug_id",
    "drug_name",
    "raw_smiles",
    "target_id",
    "target_name",
    "organism",
    "known_action",
    "actions",
    "polypeptide_id",
    "polypeptide_source",
    "amino_acid_sequence"
]


missing_chem_columns = [
    column
    for column in EXPECTED_D_CHEM_COLUMNS
    if column not in D_chem_raw.columns
]


missing_target_columns = [
    column
    for column in EXPECTED_D_TARGET_COLUMNS
    if column not in D_target_raw.columns
]


if missing_chem_columns:
    raise RuntimeError(
        "D_chem_raw is missing required columns:\n"
        f"{missing_chem_columns}"
    )


if missing_target_columns:
    raise RuntimeError(
        "D_target_raw is missing required columns:\n"
        f"{missing_target_columns}"
    )


print("\n[PASS] D_chem_raw structure validated.")
print("[PASS] D_target_raw structure validated.")


# ==============================================================================
# 7. CREATE WORKING DATASETS
# ==============================================================================
#
# IMPORTANT:
# We make explicit copies so later processing does not alter Phase-3 raw files.
# ==============================================================================

D_chem = D_chem_raw.copy()

D_target = D_target_raw.copy()


# ==============================================================================
# 8. PRESERVE COLUMN ORDER
# ==============================================================================

D_chem = D_chem[
    EXPECTED_D_CHEM_COLUMNS
].copy()


D_target = D_target[
    EXPECTED_D_TARGET_COLUMNS
].copy()


# ==============================================================================
# 9. RAW-VALUE PRESERVATION CHECK
# ==============================================================================

chem_raw_preserved = D_chem.equals(
    D_chem_raw[
        EXPECTED_D_CHEM_COLUMNS
    ]
)


target_raw_preserved = D_target.equals(
    D_target_raw[
        EXPECTED_D_TARGET_COLUMNS
    ]
)


# ==============================================================================
# 10. D_chem STRUCTURAL VALIDATION
# ==============================================================================

chem_checks = {}


chem_checks[
    "D_chem is not empty"
] = len(D_chem) > 0


chem_checks[
    "drug_id column exists"
] = "drug_id" in D_chem.columns


chem_checks[
    "all drug_id values present"
] = (
    len(D_chem) > 0
    and (
        D_chem[
            "drug_id"
        ].astype(str).str.strip() != ""
    ).all()
)


chem_checks[
    "drug_id values unique"
] = (
    len(D_chem) > 0
    and D_chem[
        "drug_id"
    ].is_unique
)


chem_checks[
    "drug_name column preserved"
] = "drug_name" in D_chem.columns


chem_checks[
    "drug_type column preserved"
] = "drug_type" in D_chem.columns


chem_checks[
    "raw_smiles column preserved"
] = "raw_smiles" in D_chem.columns


chem_checks[
    "provenance fields preserved"
] = all(
    column in D_chem.columns
    for column in [
        "secondary_drugbank_ids",
        "smiles_source",
        "inchi_source",
        "inchikey_source",
        "molecular_formula_source",
        "molecular_weight_source"
    ]
)


chem_checks[
    "raw values unchanged"
] = chem_raw_preserved


# ==============================================================================
# 11. D_target STRUCTURAL VALIDATION
# ==============================================================================

target_checks = {}


target_checks[
    "D_target is not empty"
] = len(D_target) > 0


target_checks[
    "drug_id column exists"
] = "drug_id" in D_target.columns


target_checks[
    "target_id column exists"
] = "target_id" in D_target.columns


target_checks[
    "target relationships preserved"
] = (
    "target_name" in D_target.columns
    and "organism" in D_target.columns
    and "known_action" in D_target.columns
    and "actions" in D_target.columns
)


target_checks[
    "polypeptide fields preserved"
] = all(
    column in D_target.columns
    for column in [
        "polypeptide_id",
        "polypeptide_source",
        "amino_acid_sequence"
    ]
)


target_checks[
    "raw values unchanged"
] = target_raw_preserved


# ==============================================================================
# 12. DRUG ↔ TARGET LINKAGE VALIDATION
# ==============================================================================

chem_drug_ids = set(
    D_chem[
        "drug_id"
    ].astype(str)
)


target_drug_ids = set(
    D_target[
        "drug_id"
    ].astype(str)
)


orphan_target_drug_ids = sorted(
    target_drug_ids
    - chem_drug_ids
)


target_linkage_valid = (
    len(
        orphan_target_drug_ids
    ) == 0
)


target_checks[
    "all target drug IDs exist in D_chem"
] = target_linkage_valid


# ==============================================================================
# 13. ONE-TO-MANY RELATIONSHIP ANALYSIS
# ==============================================================================

drug_target_counts = (
    D_target
    .groupby(
        "drug_id",
        dropna=False
    )
    .size()
)


drugs_with_multiple_target_rows = int(
    (
        drug_target_counts > 1
    ).sum()
)


# A target can appear multiple times because several polypeptides may
# correspond to the same drug-target association.

target_polypeptide_counts = (
    D_target
    .groupby(
        [
            "drug_id",
            "target_id"
        ],
        dropna=False
    )
    .size()
)


drug_target_pairs_with_multiple_rows = int(
    (
        target_polypeptide_counts > 1
    ).sum()
)


# ==============================================================================
# 14. DATASET STATISTICS
# ==============================================================================

chem_missing_stats = {
    "missing_drug_name": int(
        (
            D_chem["drug_name"]
            .astype(str)
            .str.strip()
            == ""
        ).sum()
    ),

    "missing_raw_smiles": int(
        (
            D_chem["raw_smiles"]
            .astype(str)
            .str.strip()
            == ""
        ).sum()
    ),

    "missing_inchi": int(
        (
            D_chem["inchi"]
            .astype(str)
            .str.strip()
            == ""
        ).sum()
    ),

    "missing_inchikey": int(
        (
            D_chem["inchikey"]
            .astype(str)
            .str.strip()
            == ""
        ).sum()
    ),

    "missing_formula": int(
        (
            D_chem["molecular_formula"]
            .astype(str)
            .str.strip()
            == ""
        ).sum()
    ),

    "missing_molecular_weight": int(
        (
            D_chem["molecular_weight"]
            .astype(str)
            .str.strip()
            == ""
        ).sum()
    )
}


target_missing_stats = {
    "missing_target_id": int(
        (
            D_target["target_id"]
            .astype(str)
            .str.strip()
            == ""
        ).sum()
    ),

    "missing_target_name": int(
        (
            D_target["target_name"]
            .astype(str)
            .str.strip()
            == ""
        ).sum()
    ),

    "missing_organism": int(
        (
            D_target["organism"]
            .astype(str)
            .str.strip()
            == ""
        ).sum()
    ),

    "missing_polypeptide_id": int(
        (
            D_target["polypeptide_id"]
            .astype(str)
            .str.strip()
            == ""
        ).sum()
    ),

    "missing_sequence": int(
        (
            D_target["amino_acid_sequence"]
            .astype(str)
            .str.strip()
            == ""
        ).sum()
    )
}


# ==============================================================================
# 15. CREATE OUTPUT DIRECTORY
# ==============================================================================

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ==============================================================================
# 16. OUTPUT FILE PATHS
# ==============================================================================

D_CHEM_CSV_FILE = (
    PROCESSED_DIR
    / "D_chem.csv"
)

D_CHEM_PARQUET_FILE = (
    PROCESSED_DIR
    / "D_chem.parquet"
)

D_TARGET_CSV_FILE = (
    PROCESSED_DIR
    / "D_target.csv"
)

D_TARGET_PARQUET_FILE = (
    PROCESSED_DIR
    / "D_target.parquet"
)

PHASE4_SUMMARY_CSV_FILE = (
    PROCESSED_DIR
    / "phase4_dataset_summary.csv"
)

PHASE4_SUMMARY_JSON_FILE = (
    PROCESSED_DIR
    / "phase4_dataset_summary.json"
)

PHASE4_ORPHAN_TARGET_FILE = (
    PROCESSED_DIR
    / "phase4_orphan_target_drug_ids.csv"
)


# ==============================================================================
# 17. SAVE CSV WORKING REPRESENTATIONS
# ==============================================================================

D_chem.to_csv(
    D_CHEM_CSV_FILE,
    index=False,
    encoding="utf-8"
)


D_target.to_csv(
    D_TARGET_CSV_FILE,
    index=False,
    encoding="utf-8"
)


print("\n[PASS] CSV intermediate datasets saved.")


# ==============================================================================
# 18. SAVE PARQUET WORKING REPRESENTATIONS
# ==============================================================================
#
# Pandas requires either:
#
#   pyarrow
#
# or:
#
#   fastparquet
#
# for Parquet output.
# ==============================================================================

PARQUET_AVAILABLE = True
PARQUET_ENGINE = None
PARQUET_ERROR = None


try:

    import pyarrow  # noqa: F401

    PARQUET_ENGINE = "pyarrow"

except ImportError:

    try:

        import fastparquet  # noqa: F401

        PARQUET_ENGINE = "fastparquet"

    except ImportError:

        PARQUET_AVAILABLE = False


if PARQUET_AVAILABLE:

    try:

        D_chem.to_parquet(
            D_CHEM_PARQUET_FILE,
            index=False,
            engine=PARQUET_ENGINE
        )


        D_target.to_parquet(
            D_TARGET_PARQUET_FILE,
            index=False,
            engine=PARQUET_ENGINE
        )


        print(
            f"[PASS] Parquet datasets saved "
            f"using engine: {PARQUET_ENGINE}"
        )

    except Exception as exc:

        PARQUET_AVAILABLE = False

        PARQUET_ERROR = str(
            exc
        )

        print(
            "[WARNING] Parquet export failed."
        )

        print(
            f"Reason: {PARQUET_ERROR}"
        )

else:

    PARQUET_ERROR = (
        "Neither pyarrow nor fastparquet is installed."
    )

    print(
        "[WARNING] Parquet export unavailable."
    )

    print(
        "Install pyarrow in the environment if Parquet output "
        "is required."
    )


# ==============================================================================
# 19. SAVE ORPHAN-LINK REPORT IF NECESSARY
# ==============================================================================

if orphan_target_drug_ids:

    orphan_df = pd.DataFrame({
        "orphan_drug_id":
            orphan_target_drug_ids
    })

    orphan_df.to_csv(
        PHASE4_ORPHAN_TARGET_FILE,
        index=False,
        encoding="utf-8"
    )

else:

    if PHASE4_ORPHAN_TARGET_FILE.exists():

        PHASE4_ORPHAN_TARGET_FILE.unlink()


# ==============================================================================
# 20. DATA LINEAGE / PROVENANCE
# ==============================================================================

phase4_generated_at = datetime.now()


phase4_metadata = {
    "phase":
        "Phase 4 — XML-to-ML Intermediate Dataset Construction",

    "generated_at":
        phase4_generated_at.isoformat(),

    "drugbank_version":
        phase3_summary.get(
            "drugbank_version",
            DRUGBANK_CONFIG.get(
                "drugbank_version"
            )
        ),

    "drugbank_export_date":
        phase3_summary.get(
            "drugbank_exported_on",
            DRUGBANK_CONFIG.get(
                "drugbank_exported_on"
            )
        ),

    "source_xml":
        phase3_summary.get(
            "xml_source_file",
            str(
                DRUGBANK_CONFIG.get(
                    "xml_path",
                    ""
                )
            )
        ),

    "phase3_D_chem_raw_file":
        str(
            D_CHEM_RAW_FILE
        ),

    "phase3_D_target_raw_file":
        str(
            D_TARGET_RAW_FILE
        ),

    "phase3_extraction_limit":
        phase3_summary.get(
            "extraction_limit"
        ),

    "D_chem_rows":
        len(
            D_chem
        ),

    "D_target_rows":
        len(
            D_target
        ),

    "D_chem_columns":
        len(
            D_chem.columns
        ),

    "D_target_columns":
        len(
            D_target.columns
        ),

    "unique_D_chem_drugs":
        int(
            D_chem[
                "drug_id"
            ].nunique()
        ),

    "unique_D_target_drugs":
        int(
            D_target[
                "drug_id"
            ].nunique()
        ),

    "unique_targets":
        int(
            D_target[
                "target_id"
            ].nunique()
        ),

    "unique_polypeptides":
        int(
            D_target[
                "polypeptide_id"
            ]
            .replace(
                "",
                pd.NA
            )
            .dropna()
            .nunique()
        ),

    "drugs_with_multiple_target_rows":
        drugs_with_multiple_target_rows,

    "drug_target_pairs_with_multiple_rows":
        drug_target_pairs_with_multiple_rows,

    "orphan_target_drug_ids":
        len(
            orphan_target_drug_ids
        ),

    "raw_D_chem_values_preserved":
        bool(
            chem_raw_preserved
        ),

    "raw_D_target_values_preserved":
        bool(
            target_raw_preserved
        ),

    "parquet_available":
        PARQUET_AVAILABLE,

    "parquet_engine":
        PARQUET_ENGINE,

    "parquet_error":
        PARQUET_ERROR,

    "chem_missing_stats":
        chem_missing_stats,

    "target_missing_stats":
        target_missing_stats
}


# ==============================================================================
# 21. SAVE PHASE-4 SUMMARY
# ==============================================================================

summary_records = []


for key, value in phase4_metadata.items():

    if isinstance(
        value,
        dict
    ):

        for sub_key, sub_value in value.items():

            summary_records.append({
                "metric":
                    f"{key}.{sub_key}",

                "value":
                    sub_value
            })

    else:

        summary_records.append({
            "metric":
                key,

            "value":
                value
        })


phase4_summary_df = pd.DataFrame(
    summary_records
)


phase4_summary_df.to_csv(
    PHASE4_SUMMARY_CSV_FILE,
    index=False,
    encoding="utf-8"
)


with open(
    PHASE4_SUMMARY_JSON_FILE,
    "w",
    encoding="utf-8"
) as json_file:

    json.dump(
        phase4_metadata,
        json_file,
        indent=4,
        ensure_ascii=False,
        default=str
    )


# ==============================================================================
# 22. DISPLAY D_chem VALIDATION
# ==============================================================================

print("\n" + "=" * 110)
print("D_chem STRUCTURAL VALIDATION")
print("=" * 110)


for check_name, status in chem_checks.items():

    label = (
        "PASS"
        if status
        else "FAIL"
    )

    print(
        f"[{label:<4}] {check_name}"
    )


# ==============================================================================
# 23. DISPLAY D_target VALIDATION
# ==============================================================================

print("\n" + "=" * 110)
print("D_target STRUCTURAL VALIDATION")
print("=" * 110)


for check_name, status in target_checks.items():

    label = (
        "PASS"
        if status
        else "FAIL"
    )

    print(
        f"[{label:<4}] {check_name}"
    )


# ==============================================================================
# 24. DISPLAY DATASET STATISTICS
# ==============================================================================

print("\n" + "=" * 110)
print("INTERMEDIATE DATASET STATISTICS")
print("=" * 110)


print(
    f"D_chem rows                     : "
    f"{len(D_chem)}"
)

print(
    f"D_target rows                   : "
    f"{len(D_target)}"
)

print(
    f"Unique molecular records        : "
    f"{D_chem['drug_id'].nunique()}"
)

print(
    f"Drugs represented in D_target   : "
    f"{D_target['drug_id'].nunique()}"
)

print(
    f"Unique target IDs               : "
    f"{D_target['target_id'].nunique()}"
)

print(
    f"Unique polypeptide IDs          : "
    f"{phase4_metadata['unique_polypeptides']}"
)

print(
    f"Drugs with >1 target rows       : "
    f"{drugs_with_multiple_target_rows}"
)

print(
    f"Drug-target pairs with >1 rows  : "
    f"{drug_target_pairs_with_multiple_rows}"
)

print(
    f"Orphan D_target drug IDs        : "
    f"{len(orphan_target_drug_ids)}"
)


# ==============================================================================
# 25. MISSING-VALUE REPORT
# ==============================================================================

print("\n" + "=" * 110)
print("STRUCTURAL MISSING-VALUE REPORT")
print("=" * 110)


print("\nD_chem:")

for metric, value in chem_missing_stats.items():

    print(
        f"  {metric:<35}: {value}"
    )


print("\nD_target:")

for metric, value in target_missing_stats.items():

    print(
        f"  {metric:<35}: {value}"
    )


# ==============================================================================
# 26. SMALL DATA PREVIEW
# ==============================================================================

print("\n" + "=" * 110)
print("D_chem PREVIEW")
print("=" * 110)


chem_preview_columns = [
    "drug_id",
    "drug_name",
    "drug_type",
    "raw_smiles",
    "molecular_formula",
    "molecular_weight",
    "groups"
]


print(
    D_chem[
        chem_preview_columns
    ]
    .head(5)
    .to_string(
        index=False
    )
)


print("\n" + "=" * 110)
print("D_target PREVIEW")
print("=" * 110)


target_preview_columns = [
    "drug_id",
    "target_id",
    "target_name",
    "organism",
    "known_action",
    "actions",
    "polypeptide_id",
    "polypeptide_source"
]


print(
    D_target[
        target_preview_columns
    ]
    .head(10)
    .to_string(
        index=False
    )
)


# ==============================================================================
# 27. PHASE-4 COMPLETION CHECKS
# ==============================================================================

completion_checks = {
    "D_chem_raw loaded":
        len(D_chem_raw) > 0,

    "D_target_raw loaded":
        len(D_target_raw) > 0,

    "D_chem working dataset created":
        len(D_chem) > 0,

    "D_target working dataset created":
        len(D_target) > 0,

    "D_chem primary IDs unique":
        D_chem[
            "drug_id"
        ].is_unique,

    "D_target linked to D_chem":
        target_linkage_valid,

    "D_chem raw values preserved":
        chem_raw_preserved,

    "D_target raw values preserved":
        target_raw_preserved,

    "CSV D_chem saved":
        D_CHEM_CSV_FILE.exists(),

    "CSV D_target saved":
        D_TARGET_CSV_FILE.exists(),

    "Phase-4 summary CSV saved":
        PHASE4_SUMMARY_CSV_FILE.exists(),

    "Phase-4 summary JSON saved":
        PHASE4_SUMMARY_JSON_FILE.exists()
}


# Parquet is treated separately because it depends on an optional engine.

if PARQUET_AVAILABLE:

    completion_checks[
        "Parquet D_chem saved"
    ] = D_CHEM_PARQUET_FILE.exists()

    completion_checks[
        "Parquet D_target saved"
    ] = D_TARGET_PARQUET_FILE.exists()


# ==============================================================================
# 28. DISPLAY COMPLETION VALIDATION
# ==============================================================================

print("\n" + "=" * 110)
print("PHASE 4 COMPLETION VALIDATION")
print("=" * 110)


all_critical_checks_passed = True


for check_name, status in completion_checks.items():

    label = (
        "PASS"
        if status
        else "FAIL"
    )

    print(
        f"[{label:<4}] {check_name}"
    )

    if not status:

        all_critical_checks_passed = False


# ==============================================================================
# 29. PACKAGE PHASE-4 CONFIGURATION
# ==============================================================================

PHASE4_DATASETS = {
    "D_chem_csv":
        str(
            D_CHEM_CSV_FILE
        ),

    "D_chem_parquet":
        (
            str(
                D_CHEM_PARQUET_FILE
            )
            if PARQUET_AVAILABLE
            else None
        ),

    "D_target_csv":
        str(
            D_TARGET_CSV_FILE
        ),

    "D_target_parquet":
        (
            str(
                D_TARGET_PARQUET_FILE
            )
            if PARQUET_AVAILABLE
            else None
        ),

    "summary_csv":
        str(
            PHASE4_SUMMARY_CSV_FILE
        ),

    "summary_json":
        str(
            PHASE4_SUMMARY_JSON_FILE
        ),

    "D_chem_rows":
        len(
            D_chem
        ),

    "D_target_rows":
        len(
            D_target
        ),

    "parquet_available":
        PARQUET_AVAILABLE
}


DRUGBANK_CONFIG[
    "phase4_datasets"
] = PHASE4_DATASETS


# ==============================================================================
# 30. FINAL SUMMARY
# ==============================================================================

print("\n" + "=" * 110)
print("PHASE 4 SUMMARY")
print("=" * 110)


print(
    f"D_chem Working Records         : "
    f"{len(D_chem)}"
)

print(
    f"D_target Working Records       : "
    f"{len(D_target)}"
)

print(
    f"Raw D_chem Preserved           : "
    f"{chem_raw_preserved}"
)

print(
    f"Raw D_target Preserved         : "
    f"{target_raw_preserved}"
)

print(
    f"Drug ↔ Target Linkage Valid    : "
    f"{target_linkage_valid}"
)

print(
    f"CSV Working Representation     : "
    f"Available"
)

print(
    f"Parquet Working Representation : "
    f"{'Available' if PARQUET_AVAILABLE else 'Unavailable'}"
)


print("\nSaved Files:")

print(
    f"  D_chem CSV       : "
    f"{D_CHEM_CSV_FILE}"
)

if PARQUET_AVAILABLE:

    print(
        f"  D_chem Parquet   : "
        f"{D_CHEM_PARQUET_FILE}"
    )


print(
    f"  D_target CSV     : "
    f"{D_TARGET_CSV_FILE}"
)

if PARQUET_AVAILABLE:

    print(
        f"  D_target Parquet : "
        f"{D_TARGET_PARQUET_FILE}"
    )


print(
    f"  Summary CSV      : "
    f"{PHASE4_SUMMARY_CSV_FILE}"
)

print(
    f"  Summary JSON     : "
    f"{PHASE4_SUMMARY_JSON_FILE}"
)


print("-" * 110)


if all_critical_checks_passed:

    print(
        "[STATUS] PHASE 4 COMPLETED SUCCESSFULLY."
    )

    print(
        "[READY]  ML-accessible intermediate datasets "
        "have been created."
    )

    print(
        "[NEXT]   Phase 5 — Molecular Dataset Quality Control "
        "and Chemical Standardization"
    )

else:

    print(
        "[STATUS] PHASE 4 COMPLETED WITH ONE OR MORE "
        "FAILED STRUCTURAL CHECKS."
    )

    print(
        "[ACTION] Review failed checks before proceeding "
        "to Phase 5."
    )


print("=" * 110)

PHASE 4 — XML-to-ML INTERMEDIATE DATASET CONSTRUCTION

[IN ] Raw Extraction Directory : E:\MyPhD\My Patent-1\Implementation\data\extracted
[OUT] Processed Directory      : E:\MyPhD\My Patent-1\Implementation\data\processed

[PASS] Phase-3 raw molecular dataset found.
[PASS] Phase-3 raw target dataset found.

RAW DATASET LOAD SUMMARY
D_chem_raw rows    : 800
D_chem_raw columns : 17
D_target_raw rows  : 5075
D_target_raw cols  : 11
[PASS] Phase-3 extraction metadata loaded.

[PASS] D_chem_raw structure validated.
[PASS] D_target_raw structure validated.

[PASS] CSV intermediate datasets saved.
[PASS] Parquet datasets saved using engine: pyarrow

D_chem STRUCTURAL VALIDATION
[PASS] D_chem is not empty
[PASS] drug_id column exists
[PASS] all drug_id values present
[PASS] drug_id values unique
[PASS] drug_name column preserved
[PASS] drug_type column preserved
[PASS] raw_smiles column preserved
[PASS] provenance fields preserved
[PASS] raw values unchanged

D_target STRUCTURAL VALIDATION
[P

# Phase 5 — Molecular Dataset Quality Control and Chemical Standardization

**Objective:** Clean, sanitize, and chemically standardize the intermediate molecular working dataset (`D_chem`) using RDKit to construct the baseline chemical corpus $\mathcal{D}_{\text{chem}}$ and synchronize the biological association layer $\mathcal{D}_{\text{target}}$.

* **Standardization & Curation Rules:**
  * **Raw Preservation:** Retain original DrugBank fields (`raw_smiles`, original mass, provenance metadata) alongside derived fields.
  * **Missing Structure Auditing:** Explicitly log molecules without valid SMILES to `phase5_excluded_molecules.csv` with a `MISSING_SMILES` tag.
  * **RDKit Sanitization & Salt Stripping:** Standardize structures via `rdMolStandardize` (cleanup, fragment-parent isolation, and uncharging) to generate canonical, standardized SMILES.
  * **Two-Tier Size Discipline:** Compute heavy-atom distributions ($4 \le N_{\text{heavy}} \le 80$) strictly as a model feasibility diagnostic rather than a chemical validity exclusion.
  * **Chemical Duplicate Flagging:** Identify and index duplicate groups (`duplicate_group_id`, `is_duplicate`) without prematurely discarding records.
  * **Relational Continuity:** Filter `D_target` to retain biological associations strictly for molecules successfully standardized in `D_chem`.

* **Execution Steps:**
  1. Load `D_chem` and `D_target` from `./data/processed/`.
  2. Parse and sanitize SMILES strings using RDKit, tracking parse and valence errors.
  3. Perform fragment-parent standardization and generate stereochemically invariant canonical SMILES.
  4. Compute molecular QC descriptors (heavy atom count, rotatable bonds, formal charge, H-bond donors/acceptors, molecular weight delta).
  5. Synchronize retained DrugBank IDs to form `D_target_standardized_subset`.
  6. Export curated datasets (`D_chem_standardized.csv`, `D_chem_standardized.parquet`, `D_target_standardized_subset.csv`), exclusion logs, and JSON summaries to `./data/processed/`.

* **Expected Output:**
  Curated baseline corpus `D_chem_standardized` ($789$ molecules) and aligned `D_target_standardized_subset` ($5,058$ target rows), prepared for Phase 6 biological knowledge layer construction and Phase 7 molecular graph modeling.

In [5]:
# ==============================================================================
# PHASE 5 — MOLECULAR DATASET QUALITY CONTROL AND CHEMICAL STANDARDIZATION
# ==============================================================================
#
# INPUT
# -----
# D_chem
# D_target
#
# OUTPUT
# ------
# D_chem_standardized
# D_target_standardized_subset
# phase5_excluded_molecules
# phase5_duplicate_report
# phase5_qc_summary
#
# IMPORTANT
# ---------
# Phase 5 performs chemistry-specific quality control and standardization.
#
# It DOES:
#   - check missing SMILES
#   - parse SMILES with RDKit
#   - sanitize molecules
#   - analyze fragments
#   - standardize salts / fragment parents
#   - generate canonical SMILES
#   - calculate molecular QC descriptors
#   - compare DrugBank and RDKit molecular weights
#   - identify chemical duplicates
#   - preserve linkage to D_target
#
# It DOES NOT:
#   - generate 3-D conformers
#   - construct molecular graphs
#   - train neural networks
#   - apply EGNN
#   - calculate QED / SA for optimization
#   - perform UCB
#   - perform Fisher / EWC
#   - perform closed-loop learning
#
# IMPORTANT:
# No heavy-atom size cutoff is applied as a chemical-validity filter.
# A 4–80 heavy-atom range is calculated only as a MODEL-FEASIBILITY
# diagnostic for later phases.
# ==============================================================================


# ==============================================================================
# 1. IMPORTS
# ==============================================================================

from pathlib import Path
from datetime import datetime
import json
import math

import pandas as pd
import numpy as np


# ==============================================================================
# 2. IMPORT RDKIT
# ==============================================================================

try:
    from rdkit import Chem
    from rdkit import RDLogger
    from rdkit.Chem import Descriptors
    from rdkit.Chem import rdMolDescriptors
    from rdkit.Chem.MolStandardize import rdMolStandardize

except ImportError as exc:

    raise RuntimeError(
        "RDKit is required for Phase 5 but is not available.\n"
        "Install RDKit in the current Python environment before running "
        "this phase.\n\n"
        f"Original import error:\n{exc}"
    )


# Keep genuine RDKit errors visible, but suppress unnecessary warnings/info.
RDLogger.DisableLog("rdApp.warning")
RDLogger.DisableLog("rdApp.info")


# ==============================================================================
# 3. VERIFY REQUIRED PROJECT VARIABLES
# ==============================================================================

required_variables = [
    "PROCESSED_DIR",
    "DRUGBANK_CONFIG"
]

missing_variables = [
    variable
    for variable in required_variables
    if variable not in globals()
]


if missing_variables:

    raise RuntimeError(
        "Earlier phases must be executed before Phase 5.\n"
        f"Missing variables: {missing_variables}"
    )


print("=" * 110)
print("PHASE 5 — MOLECULAR DATASET QUALITY CONTROL AND CHEMICAL STANDARDIZATION")
print("=" * 110)

print(f"\n[IN/OUT] Processed Directory : {PROCESSED_DIR}")


# ==============================================================================
# 4. IDENTIFY PHASE-4 INPUT FILES
# ==============================================================================

D_CHEM_INPUT_FILE = (
    PROCESSED_DIR
    / "D_chem.csv"
)

D_TARGET_INPUT_FILE = (
    PROCESSED_DIR
    / "D_target.csv"
)


required_files = [
    D_CHEM_INPUT_FILE,
    D_TARGET_INPUT_FILE
]


missing_files = [
    str(file_path)
    for file_path in required_files
    if not file_path.exists()
]


if missing_files:

    raise FileNotFoundError(
        "Required Phase-4 files were not found:\n"
        + "\n".join(missing_files)
    )


print("\n[PASS] Phase-4 D_chem found.")
print("[PASS] Phase-4 D_target found.")


# ==============================================================================
# 5. LOAD PHASE-4 DATASETS
# ==============================================================================

D_chem = pd.read_csv(
    D_CHEM_INPUT_FILE,
    dtype=str,
    keep_default_na=False
)


D_target = pd.read_csv(
    D_TARGET_INPUT_FILE,
    dtype=str,
    keep_default_na=False
)


print("\n" + "=" * 110)
print("PHASE-4 INPUT DATA")
print("=" * 110)

print(f"D_chem records   : {len(D_chem)}")
print(f"D_target records : {len(D_target)}")


# ==============================================================================
# 6. VERIFY MINIMUM REQUIRED COLUMNS
# ==============================================================================

required_chem_columns = [
    "drug_id",
    "drug_name",
    "drug_type",
    "raw_smiles",
    "inchi",
    "inchikey",
    "molecular_formula",
    "molecular_weight",
    "average_mass",
    "monoisotopic_mass",
    "groups"
]


required_target_columns = [
    "drug_id",
    "target_id",
    "target_name",
    "organism",
    "known_action",
    "actions",
    "polypeptide_id",
    "polypeptide_source",
    "amino_acid_sequence"
]


missing_chem_columns = [
    column
    for column in required_chem_columns
    if column not in D_chem.columns
]


missing_target_columns = [
    column
    for column in required_target_columns
    if column not in D_target.columns
]


if missing_chem_columns:

    raise RuntimeError(
        "D_chem is missing required columns:\n"
        f"{missing_chem_columns}"
    )


if missing_target_columns:

    raise RuntimeError(
        "D_target is missing required columns:\n"
        f"{missing_target_columns}"
    )


print("\n[PASS] Required D_chem columns verified.")
print("[PASS] Required D_target columns verified.")


# ==============================================================================
# 7. PRESERVE ORIGINAL INPUT FOR FIDELITY CHECK
# ==============================================================================

D_chem_original = D_chem.copy(deep=True)

D_target_original = D_target.copy(deep=True)


# ==============================================================================
# 8. HELPER FUNCTIONS
# ==============================================================================

def clean_string(value):
    """
    Convert a value to a clean stripped string.
    Empty/NaN-like values become an empty string.
    """

    if value is None:
        return ""

    if pd.isna(value):
        return ""

    return str(value).strip()


def safe_float(value):
    """
    Convert a value to float when possible.
    """

    value = clean_string(value)

    if value == "":
        return np.nan

    try:
        return float(value)

    except Exception:
        return np.nan


def calculate_formal_charge(mol):
    """
    Calculate total formal molecular charge.
    """

    return int(
        sum(
            atom.GetFormalCharge()
            for atom in mol.GetAtoms()
        )
    )


def calculate_fragment_count(mol):
    """
    Count disconnected molecular components.
    """

    fragments = Chem.GetMolFrags(
        mol,
        asMols=False,
        sanitizeFrags=False
    )

    return len(fragments)


def standardize_molecule(mol):
    """
    Standardize a valid RDKit molecule.

    Workflow:
        Cleanup
          ↓
        FragmentParent
          ↓
        Uncharger
          ↓
        Sanitization

    The original raw DrugBank SMILES is NEVER overwritten.
    """

    working_mol = Chem.Mol(
        mol
    )

    # Normalize / clean common structural representation issues.
    working_mol = rdMolStandardize.Cleanup(
        working_mol
    )

    # Select chemically meaningful fragment parent rather than simply
    # selecting the longest textual SMILES component.
    working_mol = rdMolStandardize.FragmentParent(
        working_mol
    )

    # Normalize removable charge representations where appropriate.
    uncharger = rdMolStandardize.Uncharger()

    working_mol = uncharger.uncharge(
        working_mol
    )

    Chem.SanitizeMol(
        working_mol
    )

    return working_mol


def descriptor_values(mol):
    """
    Calculate molecular QC descriptors for a standardized molecule.
    """

    return {
        "atom_count":
            int(mol.GetNumAtoms()),

        "heavy_atom_count":
            int(mol.GetNumHeavyAtoms()),

        "formal_charge":
            calculate_formal_charge(
                mol
            ),

        "rdkit_molecular_weight":
            float(
                Descriptors.MolWt(
                    mol
                )
            ),

        "num_rings":
            int(
                rdMolDescriptors.CalcNumRings(
                    mol
                )
            ),

        "num_rotatable_bonds":
            int(
                rdMolDescriptors.CalcNumRotatableBonds(
                    mol
                )
            ),

        "hbond_donors":
            int(
                rdMolDescriptors.CalcNumHBD(
                    mol
                )
            ),

        "hbond_acceptors":
            int(
                rdMolDescriptors.CalcNumHBA(
                    mol
                )
            )
    }


# ==============================================================================
# 9. PHASE-5 OUTPUT COLLECTIONS
# ==============================================================================

standardized_records = []

excluded_records = []

qc_all_records = []


# ==============================================================================
# 10. QC COUNTERS
# ==============================================================================

stats = {
    "input_D_chem_records":
        len(D_chem),

    "records_with_smiles":
        0,

    "records_missing_smiles":
        0,

    "parse_success":
        0,

    "parse_failed":
        0,

    "sanitization_success":
        0,

    "sanitization_failed":
        0,

    "single_fragment_molecules":
        0,

    "multifragment_molecules":
        0,

    "standardization_success":
        0,

    "standardization_failed":
        0,

    "standardized_records_retained":
        0
}


# ==============================================================================
# 11. PROCESS EVERY PHASE-4 MOLECULAR RECORD
# ==============================================================================

print("\n" + "=" * 110)
print("RUNNING MOLECULAR QUALITY CONTROL")
print("=" * 110)


phase5_started_at = datetime.now()


for row_index, row in D_chem.iterrows():

    drug_id = clean_string(
        row["drug_id"]
    )

    drug_name = clean_string(
        row["drug_name"]
    )

    raw_smiles = clean_string(
        row["raw_smiles"]
    )


    # --------------------------------------------------------------------------
    # Base QC record
    # --------------------------------------------------------------------------

    qc_record = {
        "drug_id":
            drug_id,

        "drug_name":
            drug_name,

        "raw_smiles":
            raw_smiles,

        "structure_status":
            None,

        "exclusion_reason":
            None,

        "fragment_count":
            np.nan,

        "is_multifragment":
            False,

        "canonical_smiles":
            None,

        "standardized_smiles":
            None
    }


    # ==========================================================================
    # 11.1 MISSING SMILES
    # ==========================================================================

    if raw_smiles == "":

        stats[
            "records_missing_smiles"
        ] += 1


        qc_record[
            "structure_status"
        ] = "MISSING_SMILES"


        qc_record[
            "exclusion_reason"
        ] = "MISSING_SMILES"


        excluded_records.append(
            {
                "drug_id":
                    drug_id,

                "drug_name":
                    drug_name,

                "raw_smiles":
                    raw_smiles,

                "exclusion_reason":
                    "MISSING_SMILES"
            }
        )


        qc_all_records.append(
            qc_record
        )

        continue


    stats[
        "records_with_smiles"
    ] += 1


    # ==========================================================================
    # 11.2 RDKIT PARSING
    # ==========================================================================
    #
    # Parse with sanitize=False so parsing and sanitization failures can be
    # distinguished explicitly.
    # ==========================================================================

    try:

        mol = Chem.MolFromSmiles(
            raw_smiles,
            sanitize=False
        )

    except Exception:

        mol = None


    if mol is None:

        stats[
            "parse_failed"
        ] += 1


        qc_record[
            "structure_status"
        ] = "PARSE_FAILED"


        qc_record[
            "exclusion_reason"
        ] = "PARSE_FAILED"


        excluded_records.append(
            {
                "drug_id":
                    drug_id,

                "drug_name":
                    drug_name,

                "raw_smiles":
                    raw_smiles,

                "exclusion_reason":
                    "PARSE_FAILED"
            }
        )


        qc_all_records.append(
            qc_record
        )

        continue


    stats[
        "parse_success"
    ] += 1


    # ==========================================================================
    # 11.3 MOLECULAR SANITIZATION
    # ==========================================================================

    try:

        Chem.SanitizeMol(
            mol
        )

    except Exception:

        stats[
            "sanitization_failed"
        ] += 1


        qc_record[
            "structure_status"
        ] = "SANITIZATION_FAILED"


        qc_record[
            "exclusion_reason"
        ] = "SANITIZATION_FAILED"


        excluded_records.append(
            {
                "drug_id":
                    drug_id,

                "drug_name":
                    drug_name,

                "raw_smiles":
                    raw_smiles,

                "exclusion_reason":
                    "SANITIZATION_FAILED"
            }
        )


        qc_all_records.append(
            qc_record
        )

        continue


    stats[
        "sanitization_success"
    ] += 1


    # ==========================================================================
    # 11.4 RAW CANONICAL SMILES
    # ==========================================================================

    try:

        canonical_smiles = Chem.MolToSmiles(
            mol,
            canonical=True,
            isomericSmiles=True
        )

    except Exception:

        canonical_smiles = None


    qc_record[
        "canonical_smiles"
    ] = canonical_smiles


    # ==========================================================================
    # 11.5 FRAGMENT ANALYSIS — BEFORE STANDARDIZATION
    # ==========================================================================

    fragment_count = calculate_fragment_count(
        mol
    )


    is_multifragment = (
        fragment_count > 1
    )


    qc_record[
        "fragment_count"
    ] = fragment_count


    qc_record[
        "is_multifragment"
    ] = is_multifragment


    if is_multifragment:

        stats[
            "multifragment_molecules"
        ] += 1

    else:

        stats[
            "single_fragment_molecules"
        ] += 1


    # ==========================================================================
    # 11.6 RAW RDKIT MOLECULAR WEIGHT
    # ==========================================================================
    #
    # This is calculated before salt/fragment-parent standardization so it can
    # be compared more fairly with the DrugBank structure representation.
    # ==========================================================================

    try:

        raw_rdkit_molecular_weight = float(
            Descriptors.MolWt(
                mol
            )
        )

    except Exception:

        raw_rdkit_molecular_weight = np.nan


    # ==========================================================================
    # 11.7 STANDARDIZATION
    # ==========================================================================

    try:

        standardized_mol = standardize_molecule(
            mol
        )


        if standardized_mol is None:

            raise ValueError(
                "Standardization returned no molecule."
            )


        standardized_smiles = Chem.MolToSmiles(
            standardized_mol,
            canonical=True,
            isomericSmiles=True
        )


        if not standardized_smiles:

            raise ValueError(
                "Standardized SMILES is empty."
            )


    except Exception:

        stats[
            "standardization_failed"
        ] += 1


        qc_record[
            "structure_status"
        ] = "STANDARDIZATION_FAILED"


        qc_record[
            "exclusion_reason"
        ] = "STANDARDIZATION_FAILED"


        excluded_records.append(
            {
                "drug_id":
                    drug_id,

                "drug_name":
                    drug_name,

                "raw_smiles":
                    raw_smiles,

                "exclusion_reason":
                    "STANDARDIZATION_FAILED"
            }
        )


        qc_all_records.append(
            qc_record
        )

        continue


    stats[
        "standardization_success"
    ] += 1


    qc_record[
        "standardized_smiles"
    ] = standardized_smiles


    # ==========================================================================
    # 11.8 CALCULATE STANDARDIZED MOLECULAR DESCRIPTORS
    # ==========================================================================

    try:

        descriptors = descriptor_values(
            standardized_mol
        )

    except Exception:

        descriptors = {
            "atom_count":
                np.nan,

            "heavy_atom_count":
                np.nan,

            "formal_charge":
                np.nan,

            "rdkit_molecular_weight":
                np.nan,

            "num_rings":
                np.nan,

            "num_rotatable_bonds":
                np.nan,

            "hbond_donors":
                np.nan,

            "hbond_acceptors":
                np.nan
        }


    # ==========================================================================
    # 11.9 MOLECULAR-WEIGHT CONSISTENCY
    # ==========================================================================

    drugbank_molecular_weight = safe_float(
        row[
            "molecular_weight"
        ]
    )


    if (
        not np.isnan(
            drugbank_molecular_weight
        )
        and not np.isnan(
            raw_rdkit_molecular_weight
        )
    ):

        molecular_weight_difference = (
            raw_rdkit_molecular_weight
            - drugbank_molecular_weight
        )


        molecular_weight_abs_difference = abs(
            molecular_weight_difference
        )

    else:

        molecular_weight_difference = np.nan
        molecular_weight_abs_difference = np.nan


    # ==========================================================================
    # 11.10 MODEL-FEASIBILITY DIAGNOSTIC
    # ==========================================================================
    #
    # IMPORTANT:
    # This is NOT used to reject molecules in Phase 5.
    #
    # It only identifies whether a molecule falls within the previously
    # discussed 4–80 heavy-atom range for later model feasibility analysis.
    # ==========================================================================

    heavy_atom_count = descriptors[
        "heavy_atom_count"
    ]


    if pd.notna(
        heavy_atom_count
    ):

        within_4_80_heavy_atoms = (
            int(heavy_atom_count) >= 4
            and int(heavy_atom_count) <= 80
        )

    else:

        within_4_80_heavy_atoms = False


    # ==========================================================================
    # 11.11 CREATE STANDARDIZED RECORD
    # ==========================================================================

    standardized_record = row.to_dict()


    standardized_record.update(
        {
            "canonical_smiles":
                canonical_smiles,

            "standardized_smiles":
                standardized_smiles,

            "structure_status":
                "VALID",

            "fragment_count":
                fragment_count,

            "is_multifragment":
                is_multifragment,

            "raw_rdkit_molecular_weight":
                raw_rdkit_molecular_weight,

            "atom_count":
                descriptors[
                    "atom_count"
                ],

            "heavy_atom_count":
                descriptors[
                    "heavy_atom_count"
                ],

            "formal_charge":
                descriptors[
                    "formal_charge"
                ],

            "rdkit_molecular_weight":
                descriptors[
                    "rdkit_molecular_weight"
                ],

            "num_rings":
                descriptors[
                    "num_rings"
                ],

            "num_rotatable_bonds":
                descriptors[
                    "num_rotatable_bonds"
                ],

            "hbond_donors":
                descriptors[
                    "hbond_donors"
                ],

            "hbond_acceptors":
                descriptors[
                    "hbond_acceptors"
                ],

            "drugbank_molecular_weight_numeric":
                drugbank_molecular_weight,

            "molecular_weight_difference":
                molecular_weight_difference,

            "molecular_weight_abs_difference":
                molecular_weight_abs_difference,

            "within_4_80_heavy_atoms":
                within_4_80_heavy_atoms
        }
    )


    standardized_records.append(
        standardized_record
    )


    stats[
        "standardized_records_retained"
    ] += 1


    qc_record[
        "structure_status"
    ] = "VALID"


    qc_all_records.append(
        qc_record
    )


phase5_finished_at = datetime.now()


phase5_duration = (
    phase5_finished_at
    - phase5_started_at
)


# ==============================================================================
# 12. CREATE DATAFRAMES
# ==============================================================================

D_chem_standardized = pd.DataFrame(
    standardized_records
)


phase5_excluded_molecules = pd.DataFrame(
    excluded_records
)


phase5_qc_records = pd.DataFrame(
    qc_all_records
)


# ==============================================================================
# 13. ENSURE EXCLUSION DATAFRAME STRUCTURE EVEN WHEN EMPTY
# ==============================================================================

EXCLUDED_COLUMNS = [
    "drug_id",
    "drug_name",
    "raw_smiles",
    "exclusion_reason"
]


if phase5_excluded_molecules.empty:

    phase5_excluded_molecules = pd.DataFrame(
        columns=EXCLUDED_COLUMNS
    )

else:

    phase5_excluded_molecules = (
        phase5_excluded_molecules[
            EXCLUDED_COLUMNS
        ]
    )


# ==============================================================================
# 14. CHEMICAL DUPLICATE ANALYSIS
# ==============================================================================
#
# IMPORTANT:
# We IDENTIFY duplicates.
# We DO NOT automatically delete them in Phase 5.
# ==============================================================================

if not D_chem_standardized.empty:

    duplicate_group_lookup = {}


    standardized_smiles_counts = (
        D_chem_standardized[
            "standardized_smiles"
        ]
        .value_counts(
            dropna=False
        )
    )


    duplicate_smiles_values = set(
        standardized_smiles_counts[
            standardized_smiles_counts > 1
        ].index
    )


    duplicate_counter = 1


    for smiles_value in sorted(
        [
            value
            for value in duplicate_smiles_values
            if clean_string(value) != ""
        ]
    ):

        duplicate_group_lookup[
            smiles_value
        ] = (
            f"DUP_{duplicate_counter:04d}"
        )

        duplicate_counter += 1


    D_chem_standardized[
        "duplicate_count"
    ] = (
        D_chem_standardized[
            "standardized_smiles"
        ]
        .map(
            standardized_smiles_counts
        )
        .fillna(1)
        .astype(int)
    )


    D_chem_standardized[
        "is_duplicate"
    ] = (
        D_chem_standardized[
            "duplicate_count"
        ] > 1
    )


    D_chem_standardized[
        "duplicate_group_id"
    ] = (
        D_chem_standardized[
            "standardized_smiles"
        ]
        .map(
            duplicate_group_lookup
        )
        .fillna("")
    )


else:

    D_chem_standardized[
        "duplicate_count"
    ] = pd.Series(
        dtype=int
    )

    D_chem_standardized[
        "is_duplicate"
    ] = pd.Series(
        dtype=bool
    )

    D_chem_standardized[
        "duplicate_group_id"
    ] = pd.Series(
        dtype=str
    )


# ==============================================================================
# 15. CREATE DUPLICATE REPORT
# ==============================================================================

if not D_chem_standardized.empty:

    phase5_duplicate_report = (
        D_chem_standardized[
            D_chem_standardized[
                "is_duplicate"
            ]
        ][
            [
                "duplicate_group_id",
                "duplicate_count",
                "drug_id",
                "drug_name",
                "raw_smiles",
                "canonical_smiles",
                "standardized_smiles",
                "inchikey"
            ]
        ]
        .sort_values(
            [
                "duplicate_group_id",
                "drug_id"
            ]
        )
        .reset_index(
            drop=True
        )
    )

else:

    phase5_duplicate_report = pd.DataFrame(
        columns=[
            "duplicate_group_id",
            "duplicate_count",
            "drug_id",
            "drug_name",
            "raw_smiles",
            "canonical_smiles",
            "standardized_smiles",
            "inchikey"
        ]
    )


# ==============================================================================
# 16. DUPLICATE STATISTICS
# ==============================================================================

stats[
    "duplicate_molecular_records"
] = int(
    D_chem_standardized[
        "is_duplicate"
    ].sum()
) if not D_chem_standardized.empty else 0


stats[
    "duplicate_groups"
] = int(
    D_chem_standardized[
        "duplicate_group_id"
    ]
    .replace(
        "",
        pd.NA
    )
    .dropna()
    .nunique()
) if not D_chem_standardized.empty else 0


# ==============================================================================
# 17. MODEL-FEASIBILITY SIZE STATISTICS
# ==============================================================================
#
# No molecules are removed here.
# ==============================================================================

if not D_chem_standardized.empty:

    stats[
        "molecules_within_4_80_heavy_atoms"
    ] = int(
        D_chem_standardized[
            "within_4_80_heavy_atoms"
        ].sum()
    )


    stats[
        "molecules_outside_4_80_heavy_atoms"
    ] = int(
        (
            ~D_chem_standardized[
                "within_4_80_heavy_atoms"
            ]
        ).sum()
    )


    stats[
        "minimum_heavy_atom_count"
    ] = int(
        D_chem_standardized[
            "heavy_atom_count"
        ].min()
    )


    stats[
        "maximum_heavy_atom_count"
    ] = int(
        D_chem_standardized[
            "heavy_atom_count"
        ].max()
    )


    stats[
        "median_heavy_atom_count"
    ] = float(
        D_chem_standardized[
            "heavy_atom_count"
        ].median()
    )


else:

    stats[
        "molecules_within_4_80_heavy_atoms"
    ] = 0

    stats[
        "molecules_outside_4_80_heavy_atoms"
    ] = 0

    stats[
        "minimum_heavy_atom_count"
    ] = None

    stats[
        "maximum_heavy_atom_count"
    ] = None

    stats[
        "median_heavy_atom_count"
    ] = None


# ==============================================================================
# 18. FRAGMENT STATISTICS
# ==============================================================================

stats[
    "multifragment_percentage"
] = (
    round(
        (
            stats[
                "multifragment_molecules"
            ]
            / stats[
                "sanitization_success"
            ]
        )
        * 100,
        2
    )
    if stats[
        "sanitization_success"
    ] > 0
    else 0.0
)


# ==============================================================================
# 19. RETAINED DRUG IDS
# ==============================================================================

retained_drug_ids = set(
    D_chem_standardized[
        "drug_id"
    ].astype(str)
)


# ==============================================================================
# 20. CREATE CORRESPONDING TARGET SUBSET
# ==============================================================================
#
# IMPORTANT:
# Original D_target remains unchanged.
#
# This subset simply provides target records corresponding to molecules that
# successfully passed Phase-5 chemistry QC.
# ==============================================================================

D_target_standardized_subset = (
    D_target[
        D_target[
            "drug_id"
        ].astype(str).isin(
            retained_drug_ids
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# ==============================================================================
# 21. VERIFY ORIGINAL PHASE-4 DATA REMAINS UNCHANGED
# ==============================================================================

original_D_chem_preserved = D_chem.equals(
    D_chem_original
)


original_D_target_preserved = D_target.equals(
    D_target_original
)


# ==============================================================================
# 22. LINKAGE CHECK
# ==============================================================================

standardized_drug_ids = set(
    D_chem_standardized[
        "drug_id"
    ].astype(str)
)


linked_target_drug_ids = set(
    D_target_standardized_subset[
        "drug_id"
    ].astype(str)
)


target_linkage_valid = (
    linked_target_drug_ids.issubset(
        standardized_drug_ids
    )
)


# ==============================================================================
# 23. QC STATUS COUNTS
# ==============================================================================

if not phase5_qc_records.empty:

    qc_status_counts = (
        phase5_qc_records[
            "structure_status"
        ]
        .value_counts(
            dropna=False
        )
        .to_dict()
    )

else:

    qc_status_counts = {}


# ==============================================================================
# 24. COMPLETENESS CHECK
# ==============================================================================

evaluated_records = (
    len(
        D_chem_standardized
    )
    +
    len(
        phase5_excluded_molecules
    )
)


all_input_records_accounted_for = (
    evaluated_records
    == len(
        D_chem
    )
)


# ==============================================================================
# 25. PREPARE OUTPUT DIRECTORY
# ==============================================================================

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ==============================================================================
# 26. OUTPUT FILE PATHS
# ==============================================================================

D_CHEM_STANDARDIZED_CSV = (
    PROCESSED_DIR
    / "D_chem_standardized.csv"
)


D_CHEM_STANDARDIZED_PARQUET = (
    PROCESSED_DIR
    / "D_chem_standardized.parquet"
)


D_TARGET_STANDARDIZED_CSV = (
    PROCESSED_DIR
    / "D_target_standardized_subset.csv"
)


D_TARGET_STANDARDIZED_PARQUET = (
    PROCESSED_DIR
    / "D_target_standardized_subset.parquet"
)


PHASE5_EXCLUDED_FILE = (
    PROCESSED_DIR
    / "phase5_excluded_molecules.csv"
)


PHASE5_DUPLICATE_FILE = (
    PROCESSED_DIR
    / "phase5_duplicate_report.csv"
)


PHASE5_QC_RECORD_FILE = (
    PROCESSED_DIR
    / "phase5_molecule_qc_records.csv"
)


PHASE5_SUMMARY_CSV = (
    PROCESSED_DIR
    / "phase5_qc_summary.csv"
)


PHASE5_SUMMARY_JSON = (
    PROCESSED_DIR
    / "phase5_qc_summary.json"
)


PHASE5_RETAINED_IDS_FILE = (
    PROCESSED_DIR
    / "phase5_retained_drug_ids.csv"
)


# ==============================================================================
# 27. SAVE CSV FILES
# ==============================================================================

D_chem_standardized.to_csv(
    D_CHEM_STANDARDIZED_CSV,
    index=False,
    encoding="utf-8"
)


D_target_standardized_subset.to_csv(
    D_TARGET_STANDARDIZED_CSV,
    index=False,
    encoding="utf-8"
)


phase5_excluded_molecules.to_csv(
    PHASE5_EXCLUDED_FILE,
    index=False,
    encoding="utf-8"
)


phase5_duplicate_report.to_csv(
    PHASE5_DUPLICATE_FILE,
    index=False,
    encoding="utf-8"
)


phase5_qc_records.to_csv(
    PHASE5_QC_RECORD_FILE,
    index=False,
    encoding="utf-8"
)


pd.DataFrame(
    {
        "drug_id":
            sorted(
                retained_drug_ids
            )
    }
).to_csv(
    PHASE5_RETAINED_IDS_FILE,
    index=False,
    encoding="utf-8"
)


print("\n[PASS] Phase-5 CSV outputs saved.")


# ==============================================================================
# 28. SAVE PARQUET FILES
# ==============================================================================

PARQUET_AVAILABLE = True
PARQUET_ENGINE = None
PARQUET_ERROR = None


try:

    import pyarrow  # noqa: F401

    PARQUET_ENGINE = "pyarrow"

except ImportError:

    try:

        import fastparquet  # noqa: F401

        PARQUET_ENGINE = "fastparquet"

    except ImportError:

        PARQUET_AVAILABLE = False


if PARQUET_AVAILABLE:

    try:

        D_chem_standardized.to_parquet(
            D_CHEM_STANDARDIZED_PARQUET,
            index=False,
            engine=PARQUET_ENGINE
        )


        D_target_standardized_subset.to_parquet(
            D_TARGET_STANDARDIZED_PARQUET,
            index=False,
            engine=PARQUET_ENGINE
        )


        print(
            f"[PASS] Parquet outputs saved using {PARQUET_ENGINE}."
        )


    except Exception as exc:

        PARQUET_AVAILABLE = False

        PARQUET_ERROR = str(
            exc
        )

        print(
            "[WARNING] Phase-5 Parquet export failed."
        )

        print(
            f"Reason: {PARQUET_ERROR}"
        )


else:

    PARQUET_ERROR = (
        "Neither pyarrow nor fastparquet is installed."
    )

    print(
        "[WARNING] Parquet output unavailable."
    )


# ==============================================================================
# 29. FINAL STATISTICS
# ==============================================================================

stats[
    "excluded_records"
] = len(
    phase5_excluded_molecules
)


stats[
    "final_D_chem_standardized_rows"
] = len(
    D_chem_standardized
)


stats[
    "final_D_target_standardized_rows"
] = len(
    D_target_standardized_subset
)


stats[
    "retained_unique_drug_ids"
] = len(
    retained_drug_ids
)


stats[
    "all_input_records_accounted_for"
] = all_input_records_accounted_for


stats[
    "original_D_chem_preserved"
] = original_D_chem_preserved


stats[
    "original_D_target_preserved"
] = original_D_target_preserved


stats[
    "target_linkage_valid"
] = target_linkage_valid


stats[
    "phase5_duration_seconds"
] = round(
    phase5_duration.total_seconds(),
    3
)


stats[
    "parquet_available"
] = PARQUET_AVAILABLE


stats[
    "parquet_engine"
] = PARQUET_ENGINE


# ==============================================================================
# 30. CREATE PHASE-5 SUMMARY METADATA
# ==============================================================================

phase5_metadata = {
    "phase":
        "Phase 5 — Molecular Dataset Quality Control and Chemical Standardization",

    "generated_at":
        datetime.now().isoformat(),

    "drugbank_version":
        DRUGBANK_CONFIG.get(
            "drugbank_version"
        ),

    "drugbank_exported_on":
        DRUGBANK_CONFIG.get(
            "drugbank_exported_on"
        ),

    "source_D_chem":
        str(
            D_CHEM_INPUT_FILE
        ),

    "source_D_target":
        str(
            D_TARGET_INPUT_FILE
        ),

    "statistics":
        stats,

    "qc_status_counts":
        qc_status_counts,

    "chemical_duplicate_key":
        "standardized_smiles",

    "duplicates_automatically_removed":
        False,

    "heavy_atom_filter_applied":
        False,

    "heavy_atom_model_feasibility_reference":
        "4-80 heavy atoms",

    "phase_boundary":
        (
            "Chemical QC and standardization only; "
            "no geometry or graph construction."
        )
}


# ==============================================================================
# 31. SAVE SUMMARY CSV
# ==============================================================================

summary_rows = []


for key, value in phase5_metadata.items():

    if isinstance(
        value,
        dict
    ):

        for sub_key, sub_value in value.items():

            if isinstance(
                sub_value,
                dict
            ):

                for nested_key, nested_value in sub_value.items():

                    summary_rows.append(
                        {
                            "metric":
                                f"{key}.{sub_key}.{nested_key}",

                            "value":
                                nested_value
                        }
                    )

            else:

                summary_rows.append(
                    {
                        "metric":
                            f"{key}.{sub_key}",

                        "value":
                            sub_value
                    }
                )

    else:

        summary_rows.append(
            {
                "metric":
                    key,

                "value":
                    value
            }
        )


phase5_summary_df = pd.DataFrame(
    summary_rows
)


phase5_summary_df.to_csv(
    PHASE5_SUMMARY_CSV,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 32. SAVE SUMMARY JSON
# ==============================================================================

with open(
    PHASE5_SUMMARY_JSON,
    "w",
    encoding="utf-8"
) as json_file:

    json.dump(
        phase5_metadata,
        json_file,
        indent=4,
        ensure_ascii=False,
        default=str
    )


# ==============================================================================
# 33. DISPLAY QC RESULTS
# ==============================================================================

print("\n" + "=" * 110)
print("PHASE 5 QC RESULTS")
print("=" * 110)


print(
    f"Input molecular records             : "
    f"{stats['input_D_chem_records']}"
)

print(
    f"Records with SMILES                 : "
    f"{stats['records_with_smiles']}"
)

print(
    f"Records missing SMILES              : "
    f"{stats['records_missing_smiles']}"
)

print(
    f"RDKit parse success                 : "
    f"{stats['parse_success']}"
)

print(
    f"RDKit parse failures                : "
    f"{stats['parse_failed']}"
)

print(
    f"Sanitization success                : "
    f"{stats['sanitization_success']}"
)

print(
    f"Sanitization failures               : "
    f"{stats['sanitization_failed']}"
)

print(
    f"Single-fragment molecules           : "
    f"{stats['single_fragment_molecules']}"
)

print(
    f"Multi-fragment molecules            : "
    f"{stats['multifragment_molecules']}"
)

print(
    f"Standardization success             : "
    f"{stats['standardization_success']}"
)

print(
    f"Standardization failures            : "
    f"{stats['standardization_failed']}"
)

print(
    f"Final standardized molecules        : "
    f"{stats['final_D_chem_standardized_rows']}"
)

print(
    f"Excluded molecules                  : "
    f"{stats['excluded_records']}"
)

print(
    f"Duplicate molecular records flagged : "
    f"{stats['duplicate_molecular_records']}"
)

print(
    f"Duplicate groups                    : "
    f"{stats['duplicate_groups']}"
)


# ==============================================================================
# 34. HEAVY-ATOM DIAGNOSTIC
# ==============================================================================

print("\n" + "=" * 110)
print("HEAVY-ATOM MODEL-FEASIBILITY DIAGNOSTIC")
print("=" * 110)


print(
    "IMPORTANT: No heavy-atom filter has been applied in Phase 5."
)


print(
    f"Minimum heavy atoms            : "
    f"{stats['minimum_heavy_atom_count']}"
)

print(
    f"Median heavy atoms             : "
    f"{stats['median_heavy_atom_count']}"
)

print(
    f"Maximum heavy atoms            : "
    f"{stats['maximum_heavy_atom_count']}"
)

print(
    f"Within 4–80 heavy atoms        : "
    f"{stats['molecules_within_4_80_heavy_atoms']}"
)

print(
    f"Outside 4–80 heavy atoms       : "
    f"{stats['molecules_outside_4_80_heavy_atoms']}"
)


# ==============================================================================
# 35. EXCLUSION BREAKDOWN
# ==============================================================================

print("\n" + "=" * 110)
print("EXCLUSION BREAKDOWN")
print("=" * 110)


if phase5_excluded_molecules.empty:

    print(
        "No molecules excluded."
    )

else:

    exclusion_counts = (
        phase5_excluded_molecules[
            "exclusion_reason"
        ]
        .value_counts()
    )


    print(
        exclusion_counts.to_string()
    )


# ==============================================================================
# 36. MOLECULAR WEIGHT CONSISTENCY SUMMARY
# ==============================================================================

print("\n" + "=" * 110)
print("MOLECULAR-WEIGHT CONSISTENCY")
print("=" * 110)


if not D_chem_standardized.empty:

    valid_weight_differences = (
        D_chem_standardized[
            "molecular_weight_abs_difference"
        ]
        .dropna()
    )


    if len(
        valid_weight_differences
    ) > 0:

        print(
            f"Comparable records       : "
            f"{len(valid_weight_differences)}"
        )

        print(
            f"Median absolute difference: "
            f"{valid_weight_differences.median():.4f}"
        )

        print(
            f"Mean absolute difference  : "
            f"{valid_weight_differences.mean():.4f}"
        )

        print(
            f"Maximum absolute difference: "
            f"{valid_weight_differences.max():.4f}"
        )

    else:

        print(
            "No comparable molecular-weight pairs available."
        )


# ==============================================================================
# 37. SMALL STANDARDIZED DATA PREVIEW
# ==============================================================================

print("\n" + "=" * 110)
print("D_chem_standardized PREVIEW")
print("=" * 110)


if D_chem_standardized.empty:

    print(
        "D_chem_standardized is empty."
    )

else:

    preview_columns = [
        "drug_id",
        "drug_name",
        "raw_smiles",
        "canonical_smiles",
        "standardized_smiles",
        "fragment_count",
        "heavy_atom_count",
        "rdkit_molecular_weight",
        "is_duplicate"
    ]


    print(
        D_chem_standardized[
            preview_columns
        ]
        .head(10)
        .to_string(
            index=False
        )
    )


# ==============================================================================
# 38. COMPLETION VALIDATION
# ==============================================================================

completion_checks = {
    "All input molecular records evaluated":
        all_input_records_accounted_for,

    "Original D_chem preserved":
        original_D_chem_preserved,

    "Original D_target preserved":
        original_D_target_preserved,

    "SMILES availability evaluated":
        (
            stats[
                "records_with_smiles"
            ]
            +
            stats[
                "records_missing_smiles"
            ]
            ==
            stats[
                "input_D_chem_records"
            ]
        ),

    "RDKit parsing performed":
        (
            stats[
                "parse_success"
            ]
            +
            stats[
                "parse_failed"
            ]
            ==
            stats[
                "records_with_smiles"
            ]
        ),

    "Molecular sanitization performed":
        (
            stats[
                "sanitization_success"
            ]
            +
            stats[
                "sanitization_failed"
            ]
            ==
            stats[
                "parse_success"
            ]
        ),

    "Fragment analysis completed":
        (
            stats[
                "single_fragment_molecules"
            ]
            +
            stats[
                "multifragment_molecules"
            ]
            ==
            stats[
                "sanitization_success"
            ]
        ),

    "Standardization attempted":
        (
            stats[
                "standardization_success"
            ]
            +
            stats[
                "standardization_failed"
            ]
            ==
            stats[
                "sanitization_success"
            ]
        ),

    "Standardized corpus created":
        len(
            D_chem_standardized
        ) > 0,

    "Every retained molecule has standardized SMILES":
        (
            len(
                D_chem_standardized
            ) > 0
            and (
                D_chem_standardized[
                    "standardized_smiles"
                ]
                .astype(str)
                .str.strip()
                != ""
            ).all()
        ),

    "Duplicate analysis completed":
        (
            "is_duplicate"
            in D_chem_standardized.columns
        ),

    "Heavy-atom diagnostic calculated":
        (
            "heavy_atom_count"
            in D_chem_standardized.columns
        ),

    "No chemical heavy-atom exclusion applied":
        True,

    "Target subset linked correctly":
        target_linkage_valid,

    "Standardized CSV saved":
        D_CHEM_STANDARDIZED_CSV.exists(),

    "Excluded-molecule report saved":
        PHASE5_EXCLUDED_FILE.exists(),

    "Duplicate report saved":
        PHASE5_DUPLICATE_FILE.exists(),

    "QC summary CSV saved":
        PHASE5_SUMMARY_CSV.exists(),

    "QC summary JSON saved":
        PHASE5_SUMMARY_JSON.exists()
}


if PARQUET_AVAILABLE:

    completion_checks[
        "Standardized Parquet saved"
    ] = D_CHEM_STANDARDIZED_PARQUET.exists()


# ==============================================================================
# 39. DISPLAY COMPLETION VALIDATION
# ==============================================================================

print("\n" + "=" * 110)
print("PHASE 5 COMPLETION VALIDATION")
print("=" * 110)


all_critical_checks_passed = True


for check_name, status in completion_checks.items():

    label = (
        "PASS"
        if status
        else "FAIL"
    )


    print(
        f"[{label:<4}] {check_name}"
    )


    if not status:

        all_critical_checks_passed = False


# ==============================================================================
# 40. PACKAGE PHASE-5 CONFIGURATION
# ==============================================================================

PHASE5_DATASETS = {
    "D_chem_standardized_csv":
        str(
            D_CHEM_STANDARDIZED_CSV
        ),

    "D_chem_standardized_parquet":
        (
            str(
                D_CHEM_STANDARDIZED_PARQUET
            )
            if PARQUET_AVAILABLE
            else None
        ),

    "D_target_standardized_subset_csv":
        str(
            D_TARGET_STANDARDIZED_CSV
        ),

    "D_target_standardized_subset_parquet":
        (
            str(
                D_TARGET_STANDARDIZED_PARQUET
            )
            if PARQUET_AVAILABLE
            else None
        ),

    "excluded_molecules_csv":
        str(
            PHASE5_EXCLUDED_FILE
        ),

    "duplicate_report_csv":
        str(
            PHASE5_DUPLICATE_FILE
        ),

    "qc_records_csv":
        str(
            PHASE5_QC_RECORD_FILE
        ),

    "qc_summary_csv":
        str(
            PHASE5_SUMMARY_CSV
        ),

    "qc_summary_json":
        str(
            PHASE5_SUMMARY_JSON
        ),

    "retained_drug_ids_csv":
        str(
            PHASE5_RETAINED_IDS_FILE
        ),

    "D_chem_standardized_rows":
        len(
            D_chem_standardized
        ),

    "D_target_standardized_rows":
        len(
            D_target_standardized_subset
        ),

    "statistics":
        stats
}


DRUGBANK_CONFIG[
    "phase5_datasets"
] = PHASE5_DATASETS


# ==============================================================================
# 41. FINAL PHASE-5 SUMMARY
# ==============================================================================

print("\n" + "=" * 110)
print("PHASE 5 SUMMARY")
print("=" * 110)


print(
    f"Input D_chem Records             : "
    f"{len(D_chem)}"
)

print(
    f"Standardized Molecular Records  : "
    f"{len(D_chem_standardized)}"
)

print(
    f"Excluded Molecular Records      : "
    f"{len(phase5_excluded_molecules)}"
)

print(
    f"Linked D_target Records         : "
    f"{len(D_target_standardized_subset)}"
)

print(
    f"Chemical Duplicate Groups       : "
    f"{stats['duplicate_groups']}"
)

print(
    f"Multi-fragment Molecules        : "
    f"{stats['multifragment_molecules']}"
)

print(
    f"Original D_chem Preserved       : "
    f"{original_D_chem_preserved}"
)

print(
    f"Original D_target Preserved     : "
    f"{original_D_target_preserved}"
)

print(
    f"Drug ↔ Target Linkage Valid     : "
    f"{target_linkage_valid}"
)

print(
    f"Heavy-Atom Filter Applied       : False"
)

print(
    f"Phase Duration (seconds)        : "
    f"{stats['phase5_duration_seconds']}"
)


print("\nSaved Files:")

print(
    f"  Standardized D_chem : "
    f"{D_CHEM_STANDARDIZED_CSV}"
)

print(
    f"  Target Subset       : "
    f"{D_TARGET_STANDARDIZED_CSV}"
)

print(
    f"  Excluded Molecules  : "
    f"{PHASE5_EXCLUDED_FILE}"
)

print(
    f"  Duplicate Report    : "
    f"{PHASE5_DUPLICATE_FILE}"
)

print(
    f"  Molecule QC Records : "
    f"{PHASE5_QC_RECORD_FILE}"
)

print(
    f"  Retained Drug IDs   : "
    f"{PHASE5_RETAINED_IDS_FILE}"
)

print(
    f"  QC Summary CSV      : "
    f"{PHASE5_SUMMARY_CSV}"
)

print(
    f"  QC Summary JSON     : "
    f"{PHASE5_SUMMARY_JSON}"
)


if PARQUET_AVAILABLE:

    print(
        f"  Standardized Parquet: "
        f"{D_CHEM_STANDARDIZED_PARQUET}"
    )


print("-" * 110)


if all_critical_checks_passed:

    print(
        "[STATUS] PHASE 5 COMPLETED SUCCESSFULLY."
    )

    print(
        "[READY]  Chemically validated and standardized "
        "molecular corpus created."
    )

    print(
        "[NEXT]   Phase 6 — Geometry-Aware Molecular Graph Construction"
    )

else:

    print(
        "[STATUS] PHASE 5 COMPLETED WITH ONE OR MORE "
        "FAILED VALIDATION CHECKS."
    )

    print(
        "[ACTION] Review failed QC checks before proceeding "
        "to molecular geometry / graph construction."
    )


print("=" * 110)

PHASE 5 — MOLECULAR DATASET QUALITY CONTROL AND CHEMICAL STANDARDIZATION

[IN/OUT] Processed Directory : E:\MyPhD\My Patent-1\Implementation\data\processed

[PASS] Phase-4 D_chem found.
[PASS] Phase-4 D_target found.

PHASE-4 INPUT DATA
D_chem records   : 800
D_target records : 5075

[PASS] Required D_chem columns verified.
[PASS] Required D_target columns verified.

RUNNING MOLECULAR QUALITY CONTROL

[PASS] Phase-5 CSV outputs saved.
[PASS] Parquet outputs saved using pyarrow.

PHASE 5 QC RESULTS
Input molecular records             : 800
Records with SMILES                 : 789
Records missing SMILES              : 11
RDKit parse success                 : 789
RDKit parse failures                : 0
Sanitization success                : 789
Sanitization failures               : 0
Single-fragment molecules           : 765
Multi-fragment molecules            : 24
Standardization success             : 789
Standardization failures            : 0
Final standardized molecules        : 789
E

# Phase 6 — DrugBank-Grounded Biological Association Construction

**Objective:** Construct the biological association knowledge layer by linking the chemically standardized molecular corpus (`D_chem_standardized`) with curated DrugBank targets (`D_target_standardized_subset`), preserving complete relational multiplicity, organism taxonomy, and functional actions without making unsubstantiated affinity claims.

* **Key Construction & Integrity Rules:**
  * **Relational Continuity:** Join biological records to `D_chem_standardized` using `drug_id` to attach standardized molecular structures, ensuring zero orphan biological rows.
  * **One-to-Many Multiplicity:** Preserve full cardinality: single drugs mapping to multiple targets, multiple pharmacological actions, and multi-polypeptide complexes.
  * **Deterministic Provenance:** Generate a unique, deterministic `association_id` (SHA-1 hash of `drug_id`, `target_id`, and `polypeptide_id`) for every biological association record.
  * **Taxonomic Categorization:** Isolate species biology by adding `is_human_target` to allow human-specific filtering without discarding non-human reference data.
  * **Biological Annotation Retention:** Retain `known_action`, action lists, `polypeptide_id`, `polypeptide_source`, and raw amino acid sequences without discarding targets that lack linked polypeptide sequences (Case B targets).
  * **Affinity Scope Boundary:** Formally categorize all entries as curated DrugBank biological associations / bioactivity relevance indicators rather than experimentally measured binding affinities ($K_i$, $K_d$, $\text{IC}_{50}$, $\text{pIC}_{50}$).

* **Execution Steps:**
  1. Load `D_chem_standardized.csv` and `D_target_standardized_subset.csv` from `./data/processed/`.
  2. Perform pre-merge linkage verification to ensure all target records trace to validated molecules.
  3. Attach `standardized_smiles` to target rows via a many-to-one merge on `drug_id`.
  4. Derive biological metadata flags (`is_human_target`, `is_known_action`, `has_actions`, `action_count`, `has_polypeptide_id`, `has_protein_sequence`).
  5. Compute stable, unique `association_id` hashes for each row.
  6. Partition and export the biological datasets (`D_bio_association.csv`, `D_bio_association.parquet`, `D_bio_association_human.csv`, `D_bio_association_nonhuman.csv`, `phase6_drugs_without_targets.csv`).
  7. Generate biological action and organism distributions alongside comprehensive QC metrics in `phase6_biological_qc_summary.json` and `.csv`.

* **Expected Output:**
  Finalized biological knowledge layer `D_bio_association` ($5,058$ target rows across $789$ small molecules), partitioned human/non-human views, and QC summary reports stored in `./data/processed/`, ready for downstream biological relevance scoring and Phase 7 molecular graph modeling.

In [6]:
# ==============================================================================
# PHASE 6 — DRUGBANK-GROUNDED BIOLOGICAL ASSOCIATION CONSTRUCTION
# ==============================================================================

from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
import json
import hashlib


# ==============================================================================
# 1. VERIFY REQUIRED PROJECT STATE
# ==============================================================================

required_variables = [
    "PROCESSED_DIR",
    "DRUGBANK_CONFIG"
]

missing_variables = [
    variable
    for variable in required_variables
    if variable not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Earlier phases must be executed before Phase 6.\n"
        f"Missing variables: {missing_variables}"
    )


print("=" * 110)
print("PHASE 6 — DRUGBANK-GROUNDED BIOLOGICAL ASSOCIATION CONSTRUCTION")
print("=" * 110)

print(f"\n[DIR] Processed Directory : {PROCESSED_DIR}")


# ==============================================================================
# 2. INPUT FILES FROM PHASE 5
# ==============================================================================

D_CHEM_STANDARDIZED_FILE = (
    PROCESSED_DIR / "D_chem_standardized.csv"
)

D_TARGET_STANDARDIZED_FILE = (
    PROCESSED_DIR / "D_target_standardized_subset.csv"
)


for file_path in [
    D_CHEM_STANDARDIZED_FILE,
    D_TARGET_STANDARDIZED_FILE
]:
    if not file_path.exists():
        raise FileNotFoundError(
            f"Required Phase-5 file not found:\n{file_path}"
        )


print("\n[PASS] Phase-5 standardized molecular dataset found.")
print("[PASS] Phase-5 standardized target subset found.")


# ==============================================================================
# 3. LOAD DATASETS
# ==============================================================================

D_chem_standardized = pd.read_csv(
    D_CHEM_STANDARDIZED_FILE,
    dtype=str,
    keep_default_na=False
)

D_target_standardized_subset = pd.read_csv(
    D_TARGET_STANDARDIZED_FILE,
    dtype=str,
    keep_default_na=False
)


print("\n" + "=" * 110)
print("PHASE-5 INPUT DATA")
print("=" * 110)

print(
    f"D_chem_standardized rows   : "
    f"{len(D_chem_standardized)}"
)

print(
    f"D_target standardized rows : "
    f"{len(D_target_standardized_subset)}"
)


# ==============================================================================
# 4. VERIFY REQUIRED COLUMNS
# ==============================================================================

required_chem_columns = [
    "drug_id",
    "drug_name",
    "standardized_smiles"
]


required_target_columns = [
    "drug_id",
    "drug_name",
    "target_id",
    "target_name",
    "organism",
    "known_action",
    "actions",
    "polypeptide_id",
    "polypeptide_source",
    "amino_acid_sequence"
]


missing_chem_columns = [
    column
    for column in required_chem_columns
    if column not in D_chem_standardized.columns
]


missing_target_columns = [
    column
    for column in required_target_columns
    if column not in D_target_standardized_subset.columns
]


if missing_chem_columns:
    raise RuntimeError(
        "D_chem_standardized missing required columns:\n"
        f"{missing_chem_columns}"
    )


if missing_target_columns:
    raise RuntimeError(
        "D_target_standardized_subset missing required columns:\n"
        f"{missing_target_columns}"
    )


print("\n[PASS] Required molecular columns verified.")
print("[PASS] Required biological columns verified.")


# ==============================================================================
# 5. PRESERVE ORIGINAL INPUTS
# ==============================================================================

D_chem_standardized_original = (
    D_chem_standardized.copy(deep=True)
)

D_target_standardized_original = (
    D_target_standardized_subset.copy(deep=True)
)


# ==============================================================================
# 6. HELPER FUNCTIONS
# ==============================================================================

def clean_text(value):

    if value is None:
        return ""

    if pd.isna(value):
        return ""

    return str(value).strip()


def is_human_organism(value):

    organism = clean_text(
        value
    ).lower()

    return organism in {
        "humans",
        "human",
        "homo sapiens"
    }


def known_action_yes(value):

    return (
        clean_text(value).lower()
        == "yes"
    )


def actions_present(value):

    value = clean_text(
        value
    )

    if value == "":
        return False

    try:

        parsed = json.loads(
            value
        )

        if isinstance(
            parsed,
            list
        ):
            return len(parsed) > 0

    except Exception:
        pass

    return value not in {
        "[]",
        "null",
        "None"
    }


def action_count(value):

    value = clean_text(
        value
    )

    if value == "":
        return 0

    try:

        parsed = json.loads(
            value
        )

        if isinstance(
            parsed,
            list
        ):
            return len(parsed)

    except Exception:
        pass

    return 1


def normalize_actions_for_id(value):
    """
    Produce deterministic action text for association-ID construction.
    """

    value = clean_text(
        value
    )

    if value == "":
        return ""

    try:

        parsed = json.loads(
            value
        )

        if isinstance(
            parsed,
            list
        ):

            cleaned_actions = sorted(
                clean_text(action).lower()
                for action in parsed
                if clean_text(action) != ""
            )

            return "|".join(
                cleaned_actions
            )

    except Exception:
        pass

    return value.lower()


def stable_association_id(
    drug_id,
    target_id,
    target_name,
    organism,
    actions,
    polypeptide_id,
    polypeptide_source
):
    """
    Generate a deterministic biological-association identifier.

    Unlike the earlier implementation, this does NOT depend on row order.

    Main identity fields:
        drug_id
        target_id
        polypeptide_id

    Additional biological fields are used to safely distinguish rows when
    polypeptide IDs are missing.
    """

    parts = [
        clean_text(
            drug_id
        ).lower(),

        clean_text(
            target_id
        ).lower(),

        clean_text(
            target_name
        ).lower(),

        clean_text(
            organism
        ).lower(),

        normalize_actions_for_id(
            actions
        ),

        clean_text(
            polypeptide_id
        ).lower(),

        clean_text(
            polypeptide_source
        ).lower()
    ]


    raw_key = "||".join(
        parts
    )


    digest = hashlib.sha256(
        raw_key.encode(
            "utf-8"
        )
    ).hexdigest()[:20]


    return (
        f"DBA_{digest.upper()}"
    )


# ==============================================================================
# 7. INITIAL LINKAGE VALIDATION
# ==============================================================================

chem_drug_ids = set(
    D_chem_standardized[
        "drug_id"
    ].astype(str)
)


target_drug_ids = set(
    D_target_standardized_subset[
        "drug_id"
    ].astype(str)
)


orphan_target_drug_ids = sorted(
    target_drug_ids
    - chem_drug_ids
)


if orphan_target_drug_ids:

    raise RuntimeError(
        "Target dataset contains drug IDs that are not present in "
        "D_chem_standardized.\n"
        f"Orphan IDs: {orphan_target_drug_ids[:20]}"
    )


print(
    "\n[PASS] All target associations link to "
    "D_chem_standardized."
)


# ==============================================================================
# 8. SAFEGUARD AGAINST FUTURE STANDARDIZED-SMILES COLUMN COLLISION
# ==============================================================================
#
# Phase 5 currently does NOT place standardized_smiles inside the target table.
# This safeguard makes Phase 6 robust if that changes later.
# ==============================================================================

if (
    "standardized_smiles"
    in D_target_standardized_subset.columns
):

    D_target_standardized_subset = (
        D_target_standardized_subset
        .drop(
            columns=[
                "standardized_smiles"
            ]
        )
    )

    print(
        "[INFO] Pre-existing standardized_smiles removed from "
        "target table before molecular linkage."
    )


# ==============================================================================
# 9. CREATE UNIQUE MOLECULAR LOOKUP
# ==============================================================================

molecular_lookup = (
    D_chem_standardized[
        [
            "drug_id",
            "standardized_smiles"
        ]
    ]
    .copy()
)


if not molecular_lookup[
    "drug_id"
].is_unique:

    raise RuntimeError(
        "D_chem_standardized drug IDs are not unique."
    )


# ==============================================================================
# 10. ATTACH STANDARDIZED MOLECULAR REPRESENTATION
# ==============================================================================

D_bio_association = (
    D_target_standardized_subset
    .merge(
        molecular_lookup,
        on="drug_id",
        how="left",
        validate="many_to_one"
    )
)


if len(
    D_bio_association
) != len(
    D_target_standardized_subset
):

    raise RuntimeError(
        "Target row count changed during molecular linkage."
    )


print(
    "[PASS] Biological row multiplicity preserved during merge."
)


# ==============================================================================
# 11. VERIFY STANDARDIZED STRUCTURE LINKAGE
# ==============================================================================

missing_standardized_structure = (
    D_bio_association[
        "standardized_smiles"
    ]
    .astype(str)
    .str.strip()
    .eq("")
)


if missing_standardized_structure.any():

    missing_ids = (
        D_bio_association.loc[
            missing_standardized_structure,
            "drug_id"
        ]
        .drop_duplicates()
        .tolist()
    )

    raise RuntimeError(
        "Some target records lack linked standardized structures.\n"
        f"Affected IDs: {missing_ids[:20]}"
    )


print(
    "[PASS] Standardized molecular structures attached "
    "to all biological associations."
)


# ==============================================================================
# 12. DERIVED BIOLOGICAL VARIABLES
# ==============================================================================

D_bio_association[
    "is_human_target"
] = (
    D_bio_association[
        "organism"
    ].apply(
        is_human_organism
    )
)


D_bio_association[
    "is_known_action"
] = (
    D_bio_association[
        "known_action"
    ].apply(
        known_action_yes
    )
)


D_bio_association[
    "has_actions"
] = (
    D_bio_association[
        "actions"
    ].apply(
        actions_present
    )
)


D_bio_association[
    "action_count"
] = (
    D_bio_association[
        "actions"
    ].apply(
        action_count
    )
)


D_bio_association[
    "has_polypeptide_id"
] = (
    D_bio_association[
        "polypeptide_id"
    ]
    .astype(str)
    .str.strip()
    .ne("")
)


D_bio_association[
    "has_protein_sequence"
] = (
    D_bio_association[
        "amino_acid_sequence"
    ]
    .astype(str)
    .str.strip()
    .ne("")
)


# ==============================================================================
# 13. CREATE ORDER-INDEPENDENT ASSOCIATION IDS
# ==============================================================================

D_bio_association[
    "association_id"
] = D_bio_association.apply(
    lambda row: stable_association_id(
        row[
            "drug_id"
        ],

        row[
            "target_id"
        ],

        row[
            "target_name"
        ],

        row[
            "organism"
        ],

        row[
            "actions"
        ],

        row[
            "polypeptide_id"
        ],

        row[
            "polypeptide_source"
        ]
    ),
    axis=1
)


# ==============================================================================
# 14. IDENTIFY POSSIBLE TRUE DUPLICATE BIOLOGICAL ROWS
# ==============================================================================

duplicate_association_ids = (
    D_bio_association[
        "association_id"
    ].duplicated(
        keep=False
    )
)


duplicate_association_count = int(
    duplicate_association_ids.sum()
)


if duplicate_association_count > 0:

    print(
        f"[WARNING] {duplicate_association_count} biological rows "
        "share an association identity."
    )

else:

    print(
        "[PASS] Association identifiers are unique."
    )


association_ids_unique = (
    duplicate_association_count == 0
)


# ==============================================================================
# 15. PLACE ASSOCIATION ID FIRST
# ==============================================================================

association_id_series = (
    D_bio_association.pop(
        "association_id"
    )
)


D_bio_association.insert(
    0,
    "association_id",
    association_id_series
)


# ==============================================================================
# 16. REORDER PRIMARY OUTPUT FIELDS
# ==============================================================================

primary_columns = [
    "association_id",
    "drug_id",
    "drug_name",
    "standardized_smiles",
    "target_id",
    "target_name",
    "organism",
    "is_human_target",
    "known_action",
    "is_known_action",
    "actions",
    "action_count",
    "has_actions",
    "polypeptide_id",
    "polypeptide_source",
    "has_polypeptide_id",
    "amino_acid_sequence",
    "has_protein_sequence"
]


other_columns = [
    column
    for column in D_bio_association.columns
    if column not in primary_columns
]


D_bio_association = (
    D_bio_association[
        primary_columns
        + other_columns
    ]
    .copy()
)


# ==============================================================================
# 17. START QC TIMING
# ==============================================================================

phase6_started_at = datetime.now()


# ==============================================================================
# 18. RETAINED-DRUG COVERAGE
# ==============================================================================

retained_drug_ids = set(
    D_chem_standardized[
        "drug_id"
    ].astype(str)
)


drugs_with_targets = set(
    D_bio_association[
        "drug_id"
    ].astype(str)
)


drugs_without_targets = (
    retained_drug_ids
    - drugs_with_targets
)


# ==============================================================================
# 19. BIOLOGICAL QC STATISTICS
# ==============================================================================

stats = {
    "retained_molecular_records":
        len(
            D_chem_standardized
        ),

    "retained_drugs_with_targets":
        len(
            drugs_with_targets
        ),

    "retained_drugs_without_targets":
        len(
            drugs_without_targets
        ),

    "total_biological_association_rows":
        len(
            D_bio_association
        ),

    "unique_target_ids":
        int(
            D_bio_association[
                "target_id"
            ]
            .replace(
                "",
                pd.NA
            )
            .dropna()
            .nunique()
        ),

    "unique_target_names":
        int(
            D_bio_association[
                "target_name"
            ]
            .replace(
                "",
                pd.NA
            )
            .dropna()
            .nunique()
        ),

    "unique_organisms":
        int(
            D_bio_association[
                "organism"
            ]
            .replace(
                "",
                pd.NA
            )
            .dropna()
            .nunique()
        ),

    "human_target_associations":
        int(
            D_bio_association[
                "is_human_target"
            ].sum()
        ),

    "non_human_target_associations":
        int(
            (
                ~D_bio_association[
                    "is_human_target"
                ]
            ).sum()
        ),

    "known_action_associations":
        int(
            D_bio_association[
                "is_known_action"
            ].sum()
        ),

    "unknown_or_nonknown_action_associations":
        int(
            (
                ~D_bio_association[
                    "is_known_action"
                ]
            ).sum()
        ),

    "associations_with_actions":
        int(
            D_bio_association[
                "has_actions"
            ].sum()
        ),

    "associations_without_actions":
        int(
            (
                ~D_bio_association[
                    "has_actions"
                ]
            ).sum()
        ),

    "associations_with_polypeptide_id":
        int(
            D_bio_association[
                "has_polypeptide_id"
            ].sum()
        ),

    "associations_without_polypeptide_id":
        int(
            (
                ~D_bio_association[
                    "has_polypeptide_id"
                ]
            ).sum()
        ),

    "associations_with_protein_sequence":
        int(
            D_bio_association[
                "has_protein_sequence"
            ].sum()
        ),

    "associations_without_protein_sequence":
        int(
            (
                ~D_bio_association[
                    "has_protein_sequence"
                ]
            ).sum()
        ),

    "duplicate_association_id_rows":
        duplicate_association_count
}


# ==============================================================================
# 20. MULTIPLICITY ANALYSIS
# ==============================================================================

drug_association_counts = (
    D_bio_association
    .groupby(
        "drug_id"
    )
    .size()
)


stats[
    "drugs_with_multiple_association_rows"
] = int(
    (
        drug_association_counts > 1
    ).sum()
)


drug_target_counts = (
    D_bio_association
    .groupby(
        [
            "drug_id",
            "target_id"
        ],
        dropna=False
    )
    .size()
)


stats[
    "drug_target_pairs_with_multiple_rows"
] = int(
    (
        drug_target_counts > 1
    ).sum()
)


stats[
    "maximum_associations_for_single_drug"
] = (
    int(
        drug_association_counts.max()
    )
    if len(
        drug_association_counts
    ) > 0
    else 0
)


# ==============================================================================
# 21. MISSING-VALUE STATISTICS
# ==============================================================================

missing_stats = {
    "missing_target_id":
        int(
            D_bio_association[
                "target_id"
            ]
            .astype(str)
            .str.strip()
            .eq("")
            .sum()
        ),

    "missing_target_name":
        int(
            D_bio_association[
                "target_name"
            ]
            .astype(str)
            .str.strip()
            .eq("")
            .sum()
        ),

    "missing_organism":
        int(
            D_bio_association[
                "organism"
            ]
            .astype(str)
            .str.strip()
            .eq("")
            .sum()
        ),

    "missing_known_action":
        int(
            D_bio_association[
                "known_action"
            ]
            .astype(str)
            .str.strip()
            .eq("")
            .sum()
        ),

    "missing_actions":
        int(
            (
                ~D_bio_association[
                    "has_actions"
                ]
            ).sum()
        ),

    "missing_polypeptide_id":
        int(
            (
                ~D_bio_association[
                    "has_polypeptide_id"
                ]
            ).sum()
        ),

    "missing_protein_sequence":
        int(
            (
                ~D_bio_association[
                    "has_protein_sequence"
                ]
            ).sum()
        )
}


# ==============================================================================
# 22. CREATE HUMAN / NON-HUMAN VIEWS
# ==============================================================================

D_bio_human = (
    D_bio_association[
        D_bio_association[
            "is_human_target"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


D_bio_nonhuman = (
    D_bio_association[
        ~D_bio_association[
            "is_human_target"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# ==============================================================================
# 23. DRUGS WITHOUT TARGETS
# ==============================================================================

D_drugs_without_targets = (
    D_chem_standardized[
        D_chem_standardized[
            "drug_id"
        ]
        .astype(str)
        .isin(
            drugs_without_targets
        )
    ][
        [
            "drug_id",
            "drug_name",
            "standardized_smiles"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# ==============================================================================
# 24. ACTION DISTRIBUTION
# ==============================================================================

action_frequency = {}


for value in D_bio_association[
    "actions"
]:

    value = clean_text(
        value
    )

    if value == "":
        continue

    try:

        parsed = json.loads(
            value
        )

        if not isinstance(
            parsed,
            list
        ):
            parsed = [
                parsed
            ]

    except Exception:

        parsed = [
            value
        ]


    for action in parsed:

        action = clean_text(
            action
        )

        if action == "":
            continue


        action_frequency[
            action
        ] = (
            action_frequency.get(
                action,
                0
            )
            + 1
        )


action_frequency_sorted = dict(
    sorted(
        action_frequency.items(),
        key=lambda item: (
            -item[1],
            item[0]
        )
    )
)


# ==============================================================================
# 25. ORGANISM DISTRIBUTION
# ==============================================================================

organism_distribution = (
    D_bio_association[
        "organism"
    ]
    .replace(
        "",
        "MISSING"
    )
    .value_counts()
    .to_dict()
)


# ==============================================================================
# 26. VERIFY ORIGINAL INPUT TABLES REMAIN UNCHANGED
# ==============================================================================

original_chem_preserved = (
    D_chem_standardized.equals(
        D_chem_standardized_original
    )
)


# Note:
# A working copy may have had standardized_smiles removed by the safeguard,
# so compare only original source data against the saved original copy.

reloaded_target_original = pd.read_csv(
    D_TARGET_STANDARDIZED_FILE,
    dtype=str,
    keep_default_na=False
)


original_target_preserved = (
    reloaded_target_original.equals(
        D_target_standardized_original
    )
)


# ==============================================================================
# 27. FINAL LINKAGE VALIDATION
# ==============================================================================

bio_drug_ids = set(
    D_bio_association[
        "drug_id"
    ].astype(str)
)


final_linkage_valid = (
    bio_drug_ids.issubset(
        retained_drug_ids
    )
)


stats[
    "orphan_biological_drug_ids"
] = len(
    bio_drug_ids
    - retained_drug_ids
)


# ==============================================================================
# 28. OUTPUT PATHS
# ==============================================================================

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)


D_BIO_CSV_FILE = (
    PROCESSED_DIR
    / "D_bio_association.csv"
)


D_BIO_PARQUET_FILE = (
    PROCESSED_DIR
    / "D_bio_association.parquet"
)


D_BIO_HUMAN_CSV_FILE = (
    PROCESSED_DIR
    / "D_bio_association_human.csv"
)


D_BIO_NONHUMAN_CSV_FILE = (
    PROCESSED_DIR
    / "D_bio_association_nonhuman.csv"
)


DRUGS_WITHOUT_TARGETS_FILE = (
    PROCESSED_DIR
    / "phase6_drugs_without_targets.csv"
)


ACTION_DISTRIBUTION_FILE = (
    PROCESSED_DIR
    / "phase6_action_distribution.csv"
)


ORGANISM_DISTRIBUTION_FILE = (
    PROCESSED_DIR
    / "phase6_organism_distribution.csv"
)


PHASE6_SUMMARY_CSV_FILE = (
    PROCESSED_DIR
    / "phase6_biological_qc_summary.csv"
)


PHASE6_SUMMARY_JSON_FILE = (
    PROCESSED_DIR
    / "phase6_biological_qc_summary.json"
)


DUPLICATE_ASSOCIATION_FILE = (
    PROCESSED_DIR
    / "phase6_duplicate_association_ids.csv"
)


# ==============================================================================
# 29. SAVE PRIMARY DATASETS
# ==============================================================================

D_bio_association.to_csv(
    D_BIO_CSV_FILE,
    index=False,
    encoding="utf-8"
)


D_bio_human.to_csv(
    D_BIO_HUMAN_CSV_FILE,
    index=False,
    encoding="utf-8"
)


D_bio_nonhuman.to_csv(
    D_BIO_NONHUMAN_CSV_FILE,
    index=False,
    encoding="utf-8"
)


D_drugs_without_targets.to_csv(
    DRUGS_WITHOUT_TARGETS_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 30. SAVE DUPLICATE ASSOCIATION DIAGNOSTIC
# ==============================================================================

if duplicate_association_count > 0:

    duplicate_association_df = (
        D_bio_association[
            duplicate_association_ids
        ]
        .sort_values(
            "association_id"
        )
        .copy()
    )

else:

    duplicate_association_df = pd.DataFrame(
        columns=D_bio_association.columns
    )


duplicate_association_df.to_csv(
    DUPLICATE_ASSOCIATION_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 31. SAVE DISTRIBUTION REPORTS
# ==============================================================================

action_distribution_df = pd.DataFrame(
    [
        {
            "action":
                action,

            "association_count":
                count
        }
        for action, count
        in action_frequency_sorted.items()
    ]
)


action_distribution_df.to_csv(
    ACTION_DISTRIBUTION_FILE,
    index=False,
    encoding="utf-8"
)


organism_distribution_df = pd.DataFrame(
    [
        {
            "organism":
                organism,

            "association_count":
                count
        }
        for organism, count
        in organism_distribution.items()
    ]
)


organism_distribution_df.to_csv(
    ORGANISM_DISTRIBUTION_FILE,
    index=False,
    encoding="utf-8"
)


print(
    "\n[PASS] Phase-6 CSV datasets saved."
)


# ==============================================================================
# 32. SAVE PARQUET
# ==============================================================================

PARQUET_AVAILABLE = True
PARQUET_ENGINE = None
PARQUET_ERROR = None


try:

    import pyarrow  # noqa: F401

    PARQUET_ENGINE = "pyarrow"

except ImportError:

    try:

        import fastparquet  # noqa: F401

        PARQUET_ENGINE = "fastparquet"

    except ImportError:

        PARQUET_AVAILABLE = False


if PARQUET_AVAILABLE:

    try:

        D_bio_association.to_parquet(
            D_BIO_PARQUET_FILE,
            index=False,
            engine=PARQUET_ENGINE
        )

        print(
            f"[PASS] Biological Parquet saved using "
            f"{PARQUET_ENGINE}."
        )

    except Exception as exc:

        PARQUET_AVAILABLE = False

        PARQUET_ERROR = str(
            exc
        )

        print(
            "[WARNING] Biological Parquet export failed."
        )

        print(
            f"Reason: {PARQUET_ERROR}"
        )


# ==============================================================================
# 33. FINAL TIMING
# ==============================================================================

phase6_finished_at = datetime.now()


phase6_duration = (
    phase6_finished_at
    - phase6_started_at
)


stats[
    "phase6_duration_seconds"
] = round(
    phase6_duration.total_seconds(),
    3
)


stats[
    "original_D_chem_standardized_preserved"
] = original_chem_preserved


stats[
    "original_D_target_standardized_preserved"
] = original_target_preserved


stats[
    "final_linkage_valid"
] = final_linkage_valid


stats[
    "association_ids_unique"
] = association_ids_unique


stats[
    "parquet_available"
] = PARQUET_AVAILABLE


stats[
    "parquet_engine"
] = PARQUET_ENGINE


# ==============================================================================
# 34. PHASE-6 METADATA
# ==============================================================================

phase6_metadata = {
    "phase":
        "Phase 6 — DrugBank-Grounded Biological Association Construction",

    "generated_at":
        datetime.now().isoformat(),

    "drugbank_version":
        DRUGBANK_CONFIG.get(
            "drugbank_version"
        ),

    "drugbank_exported_on":
        DRUGBANK_CONFIG.get(
            "drugbank_exported_on"
        ),

    "source_D_chem_standardized":
        str(
            D_CHEM_STANDARDIZED_FILE
        ),

    "source_D_target_standardized_subset":
        str(
            D_TARGET_STANDARDIZED_FILE
        ),

    "biological_interpretation":
        (
            "DrugBank-grounded biological association / "
            "biological relevance layer"
        ),

    "quantitative_affinity_claim":
        False,

    "association_id_method":
        (
            "SHA256 over drug_id, target_id, target_name, organism, "
            "actions, polypeptide_id and polypeptide_source"
        ),

    "association_id_row_order_dependent":
        False,

    "statistics":
        stats,

    "missing_statistics":
        missing_stats,

    "action_distribution":
        action_frequency_sorted,

    "organism_distribution":
        organism_distribution
}


# ==============================================================================
# 35. SAVE SUMMARY JSON
# ==============================================================================

with open(
    PHASE6_SUMMARY_JSON_FILE,
    "w",
    encoding="utf-8"
) as json_file:

    json.dump(
        phase6_metadata,
        json_file,
        indent=4,
        ensure_ascii=False,
        default=str
    )


# ==============================================================================
# 36. SAVE SUMMARY CSV
# ==============================================================================

summary_rows = []


for section_name, section_value in phase6_metadata.items():

    if isinstance(
        section_value,
        dict
    ):

        for key, value in section_value.items():

            summary_rows.append(
                {
                    "metric":
                        f"{section_name}.{key}",

                    "value":
                        value
                }
            )

    else:

        summary_rows.append(
            {
                "metric":
                    section_name,

                "value":
                    section_value
            }
        )


phase6_summary_df = pd.DataFrame(
    summary_rows
)


phase6_summary_df.to_csv(
    PHASE6_SUMMARY_CSV_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 37. DISPLAY BIOLOGICAL QC RESULTS
# ==============================================================================

print("\n" + "=" * 110)
print("PHASE 6 BIOLOGICAL QC RESULTS")
print("=" * 110)


print(
    f"Retained molecular records             : "
    f"{stats['retained_molecular_records']}"
)

print(
    f"Drugs with target associations         : "
    f"{stats['retained_drugs_with_targets']}"
)

print(
    f"Drugs without target associations      : "
    f"{stats['retained_drugs_without_targets']}"
)

print(
    f"Total biological association rows      : "
    f"{stats['total_biological_association_rows']}"
)

print(
    f"Unique target IDs                      : "
    f"{stats['unique_target_ids']}"
)

print(
    f"Unique organisms                       : "
    f"{stats['unique_organisms']}"
)

print(
    f"Human target associations              : "
    f"{stats['human_target_associations']}"
)

print(
    f"Non-human target associations          : "
    f"{stats['non_human_target_associations']}"
)

print(
    f"Known-action associations              : "
    f"{stats['known_action_associations']}"
)

print(
    f"Associations with actions              : "
    f"{stats['associations_with_actions']}"
)

print(
    f"Associations with polypeptide IDs      : "
    f"{stats['associations_with_polypeptide_id']}"
)

print(
    f"Associations with protein sequences    : "
    f"{stats['associations_with_protein_sequence']}"
)

print(
    f"Drugs with >1 association row          : "
    f"{stats['drugs_with_multiple_association_rows']}"
)

print(
    f"Drug-target pairs with >1 rows         : "
    f"{stats['drug_target_pairs_with_multiple_rows']}"
)

print(
    f"Duplicate association-ID rows          : "
    f"{stats['duplicate_association_id_rows']}"
)


# ==============================================================================
# 38. BIOLOGICAL MISSING-VALUE REPORT
# ==============================================================================

print("\n" + "=" * 110)
print("BIOLOGICAL MISSING-VALUE REPORT")
print("=" * 110)


for metric, value in missing_stats.items():

    print(
        f"{metric:<40}: {value}"
    )


# ==============================================================================
# 39. TOP ACTIONS
# ==============================================================================

print("\n" + "=" * 110)
print("TOP BIOLOGICAL ACTIONS")
print("=" * 110)


if action_distribution_df.empty:

    print(
        "No action values available."
    )

else:

    print(
        action_distribution_df
        .head(15)
        .to_string(
            index=False
        )
    )


# ==============================================================================
# 40. TOP ORGANISMS
# ==============================================================================

print("\n" + "=" * 110)
print("TOP TARGET ORGANISMS")
print("=" * 110)


if organism_distribution_df.empty:

    print(
        "No organism values available."
    )

else:

    print(
        organism_distribution_df
        .head(15)
        .to_string(
            index=False
        )
    )


# ==============================================================================
# 41. DATA PREVIEW
# ==============================================================================

print("\n" + "=" * 110)
print("D_bio_association PREVIEW")
print("=" * 110)


preview_columns = [
    "association_id",
    "drug_id",
    "drug_name",
    "target_id",
    "target_name",
    "organism",
    "is_human_target",
    "known_action",
    "actions",
    "polypeptide_id",
    "polypeptide_source"
]


print(
    D_bio_association[
        preview_columns
    ]
    .head(15)
    .to_string(
        index=False
    )
)


# ==============================================================================
# 42. PHASE-6 COMPLETION VALIDATION
# ==============================================================================

completion_checks = {
    "D_chem_standardized loaded":
        len(
            D_chem_standardized
        ) > 0,

    "D_target_standardized_subset loaded":
        len(
            D_target_standardized_subset
        ) > 0,

    "All biological records linked to standardized drugs":
        final_linkage_valid,

    "Standardized molecular structures attached":
        (
            D_bio_association[
                "standardized_smiles"
            ]
            .astype(str)
            .str.strip()
            .ne("")
            .all()
        ),

    "Biological row multiplicity preserved":
        (
            len(
                D_bio_association
            )
            ==
            len(
                D_target_standardized_subset
            )
        ),

    "Association IDs are row-order independent":
        True,

    "Association IDs unique":
        association_ids_unique,

    "Actions preserved":
        "actions"
        in D_bio_association.columns,

    "Organism preserved":
        "organism"
        in D_bio_association.columns,

    "Human/non-human indicator created":
        "is_human_target"
        in D_bio_association.columns,

    "Known-action information preserved":
        "known_action"
        in D_bio_association.columns,

    "Polypeptide information preserved":
        "polypeptide_id"
        in D_bio_association.columns,

    "Protein sequences preserved":
        "amino_acid_sequence"
        in D_bio_association.columns,

    "Original molecular dataset preserved":
        original_chem_preserved,

    "Original target source preserved":
        original_target_preserved,

    "Biological association CSV saved":
        D_BIO_CSV_FILE.exists(),

    "Human association CSV saved":
        D_BIO_HUMAN_CSV_FILE.exists(),

    "Non-human association CSV saved":
        D_BIO_NONHUMAN_CSV_FILE.exists(),

    "Drugs-without-targets report saved":
        DRUGS_WITHOUT_TARGETS_FILE.exists(),

    "Action distribution saved":
        ACTION_DISTRIBUTION_FILE.exists(),

    "Organism distribution saved":
        ORGANISM_DISTRIBUTION_FILE.exists(),

    "Biological summary CSV saved":
        PHASE6_SUMMARY_CSV_FILE.exists(),

    "Biological summary JSON saved":
        PHASE6_SUMMARY_JSON_FILE.exists(),

    "No quantitative affinity claims introduced":
        True
}


if PARQUET_AVAILABLE:

    completion_checks[
        "Biological association Parquet saved"
    ] = D_BIO_PARQUET_FILE.exists()


# ==============================================================================
# 43. DISPLAY COMPLETION RESULTS
# ==============================================================================

print("\n" + "=" * 110)
print("PHASE 6 COMPLETION VALIDATION")
print("=" * 110)


all_critical_checks_passed = True


for check_name, status in completion_checks.items():

    label = (
        "PASS"
        if status
        else "FAIL"
    )

    print(
        f"[{label:<4}] {check_name}"
    )

    if not status:
        all_critical_checks_passed = False


# ==============================================================================
# 44. PACKAGE CONFIGURATION
# ==============================================================================

PHASE6_DATASETS = {
    "D_bio_association_csv":
        str(
            D_BIO_CSV_FILE
        ),

    "D_bio_association_parquet":
        (
            str(
                D_BIO_PARQUET_FILE
            )
            if PARQUET_AVAILABLE
            else None
        ),

    "D_bio_human_csv":
        str(
            D_BIO_HUMAN_CSV_FILE
        ),

    "D_bio_nonhuman_csv":
        str(
            D_BIO_NONHUMAN_CSV_FILE
        ),

    "drugs_without_targets_csv":
        str(
            DRUGS_WITHOUT_TARGETS_FILE
        ),

    "summary_csv":
        str(
            PHASE6_SUMMARY_CSV_FILE
        ),

    "summary_json":
        str(
            PHASE6_SUMMARY_JSON_FILE
        ),

    "D_bio_rows":
        len(
            D_bio_association
        ),

    "statistics":
        stats
}


DRUGBANK_CONFIG[
    "phase6_datasets"
] = PHASE6_DATASETS


# ==============================================================================
# 45. FINAL SUMMARY
# ==============================================================================

print("\n" + "=" * 110)
print("PHASE 6 SUMMARY")
print("=" * 110)


print(
    f"Standardized Molecules          : "
    f"{len(D_chem_standardized)}"
)

print(
    f"Biological Association Rows     : "
    f"{len(D_bio_association)}"
)

print(
    f"Human Associations              : "
    f"{len(D_bio_human)}"
)

print(
    f"Non-Human Associations          : "
    f"{len(D_bio_nonhuman)}"
)

print(
    f"Drugs Without Target Records    : "
    f"{len(D_drugs_without_targets)}"
)

print(
    f"Unique Targets                  : "
    f"{stats['unique_target_ids']}"
)

print(
    f"Association-ID Collisions       : "
    f"{duplicate_association_count}"
)

print(
    f"Association IDs Row-Dependent   : False"
)

print(
    f"Original D_chem Preserved       : "
    f"{original_chem_preserved}"
)

print(
    f"Original D_target Preserved     : "
    f"{original_target_preserved}"
)

print(
    f"Drug ↔ Target Linkage Valid     : "
    f"{final_linkage_valid}"
)

print(
    f"Quantitative Affinity Claims    : False"
)

print(
    f"Phase Duration (seconds)        : "
    f"{stats['phase6_duration_seconds']}"
)


print("\nSaved Files:")

print(
    f"  Biological Associations : "
    f"{D_BIO_CSV_FILE}"
)

print(
    f"  Human Associations      : "
    f"{D_BIO_HUMAN_CSV_FILE}"
)

print(
    f"  Non-Human Associations  : "
    f"{D_BIO_NONHUMAN_CSV_FILE}"
)

print(
    f"  Drugs Without Targets   : "
    f"{DRUGS_WITHOUT_TARGETS_FILE}"
)

print(
    f"  Duplicate-ID Report     : "
    f"{DUPLICATE_ASSOCIATION_FILE}"
)

print(
    f"  Action Distribution     : "
    f"{ACTION_DISTRIBUTION_FILE}"
)

print(
    f"  Organism Distribution   : "
    f"{ORGANISM_DISTRIBUTION_FILE}"
)

print(
    f"  QC Summary CSV          : "
    f"{PHASE6_SUMMARY_CSV_FILE}"
)

print(
    f"  QC Summary JSON         : "
    f"{PHASE6_SUMMARY_JSON_FILE}"
)


if PARQUET_AVAILABLE:

    print(
        f"  Biological Parquet      : "
        f"{D_BIO_PARQUET_FILE}"
    )


print("-" * 110)


if all_critical_checks_passed:

    print(
        "[STATUS] PHASE 6 COMPLETED SUCCESSFULLY."
    )

    print(
        "[READY]  DrugBank-grounded biological association "
        "layer created."
    )

    print(
        "[NEXT]   Phase 7 — Geometry-Aware Molecular Graph Construction"
    )

else:

    print(
        "[STATUS] PHASE 6 COMPLETED WITH ONE OR MORE "
        "FAILED VALIDATION CHECKS."
    )

    print(
        "[ACTION] Review biological QC before proceeding "
        "to Phase 7."
    )


print("=" * 110)

PHASE 6 — DRUGBANK-GROUNDED BIOLOGICAL ASSOCIATION CONSTRUCTION

[DIR] Processed Directory : E:\MyPhD\My Patent-1\Implementation\data\processed

[PASS] Phase-5 standardized molecular dataset found.
[PASS] Phase-5 standardized target subset found.

PHASE-5 INPUT DATA
D_chem_standardized rows   : 789
D_target standardized rows : 5058

[PASS] Required molecular columns verified.
[PASS] Required biological columns verified.

[PASS] All target associations link to D_chem_standardized.
[PASS] Biological row multiplicity preserved during merge.
[PASS] Standardized molecular structures attached to all biological associations.
[PASS] Association identifiers are unique.

[PASS] Phase-6 CSV datasets saved.
[PASS] Biological Parquet saved using pyarrow.

PHASE 6 BIOLOGICAL QC RESULTS
Retained molecular records             : 789
Drugs with target associations         : 777
Drugs without target associations      : 12
Total biological association rows      : 5058
Unique target IDs                    

# Phase 7 — Geometry-Aware Molecular Graph Construction

**Objective:** Convert standardized chemical structures into 3D geometry-aware molecular graphs $G = (V, E, X, R)$ combining invariant scalar chemical descriptors with centered Cartesian coordinates under strict model feasibility boundaries.

* **Graph Representation & Feasibility Policies:**
  * **Formal Formulation:** Represents each molecule as $G = (V, E, X, R)$, where $V$ denotes atoms, $E$ bidirectional directed covalent bonds, $X$ invariant scalar node attributes, and $R$ centered Cartesian coordinates ($R \in \mathbb{R}^{\vert{}V\vert{} \times 3}$).
  * **Model-Feasibility Filtering:** Evaluates candidates within $4 \le N_{\text{heavy atoms}} \le 80$ as an operational model boundary rather than a chemical validity rule.
  * **Coordinate Policy:** Hydrogen-explicit graphs with centered 3D conformations derived via ETKDGv3 and optimized via MMFF94/UFF hierarchies; Cartesian coordinates are held separate from invariant scalar features $X$.

* **Feature Architecture:**
  * **Node Features ($X \in \mathbb{R}^{\vert{}V\vert{} \times 21}$):** Atomic number, degree, formal charge, aromaticity, ring membership, explicit hydrogen neighbor count, atomic mass, implicit/explicit valences, 1-hot hybridization (8 categories), and 1-hot chirality (4 categories).
  * **Edge Features ($E_{\text{attr}} \in \mathbb{R}^{\vert{}E\vert{} \times 14}$):** 1-hot bond type (5 categories), conjugation flag, ring membership flag, and 1-hot bond stereochemistry (7 categories).
  * **Topology:** Explicit bidirectional chemical connectivity ($\vert{}E\vert{} = 2 \times N_{\text{bonds}}$).

* **Integrity & Scope Discipline:**
  * Graph generation is strictly one-to-one per unique molecule ($1 \text{ molecule} \to 1 \text{ graph}$), keeping biological association records strictly linked via identifiers rather than duplicating graph structures.
  * Operates strictly as a data-construction and deterministic featurization phase; no neural network training, latent projection, or predictive modeling occurs.

* **Primary Deliverables:**
  * Molecular Graph Corpus: `data/graphs/<drug_id>.pt` ($G = (V, E, X, R)$).
  * Cohort Tables: `D_geometry_eligible.csv`, `phase7_graph_index.csv`, and `phase7_geometry_excluded.csv`.
  * Feature Schemas & QC Logs: `phase7_graph_feature_schema.json`, `phase7_geometry_qc_summary.csv`, and `phase7_geometry_qc_summary.json`.

In [7]:
# ==============================================================================
# PHASE 7 — GEOMETRY-AWARE MOLECULAR GRAPH CONSTRUCTION
# ==============================================================================
#
# Graph representation:
#
#       G = (V, E, X, R)
#
# V = atoms
# E = bidirectional chemical-bond connectivity
# X = invariant atom features
# R = centered 3-D Cartesian coordinates
#
# INPUT:
#   D_chem_standardized.csv
#   D_bio_association.csv
#
# OUTPUT:
#   data/graphs/<drug_id>.pt
#   D_geometry_eligible.csv
#   phase7_graph_index.csv
#   phase7_geometry_excluded.csv
#   phase7_geometry_qc_summary.csv
#   phase7_geometry_qc_summary.json
#
# IMPORTANT:
#   4–80 heavy atoms = MODEL FEASIBILITY FILTER
#   NOT chemical validity.
#
#   One molecule -> one graph
#   NOT one graph per target association.
#
#   No neural-network training occurs in Phase 7.
# ==============================================================================


# ==============================================================================
# 1. IMPORTS
# ==============================================================================

from pathlib import Path
from datetime import datetime

import json
import math
import random
import traceback

import numpy as np
import pandas as pd

import torch

from rdkit import Chem
from rdkit import RDLogger
from rdkit.Chem import AllChem


# ==============================================================================
# 2. REPRODUCIBILITY
# ==============================================================================

RANDOM_SEED = 42

random.seed(
    RANDOM_SEED
)

np.random.seed(
    RANDOM_SEED
)

torch.manual_seed(
    RANDOM_SEED
)


if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        RANDOM_SEED
    )


# Suppress routine RDKit warnings.
# Genuine exceptions are still handled explicitly.

RDLogger.DisableLog(
    "rdApp.warning"
)

RDLogger.DisableLog(
    "rdApp.info"
)


# ==============================================================================
# 3. PHASE-7 MODEL-FEASIBILITY PARAMETERS
# ==============================================================================

PHASE7_MIN_HEAVY_ATOMS = 4
PHASE7_MAX_HEAVY_ATOMS = 80

PHASE7_MAX_OPTIMIZATION_ITERS = 500


# ==============================================================================
# 4. VERIFY REQUIRED PROJECT STATE
# ==============================================================================

required_variables = [
    "PROCESSED_DIR",
    "DRUGBANK_CONFIG"
]


missing_variables = [
    variable
    for variable in required_variables
    if variable not in globals()
]


if missing_variables:

    raise RuntimeError(
        "Earlier phases must be executed before Phase 7.\n"
        f"Missing variables: {missing_variables}"
    )


# Use Phase-1 GRAPH_DIR when available.
# Otherwise construct a safe fallback.

if "GRAPH_DIR" in globals():

    PHASE7_GRAPH_DIR = Path(
        GRAPH_DIR
    )

else:

    PHASE7_GRAPH_DIR = (
        Path(PROCESSED_DIR).parent
        / "graphs"
    )


PHASE7_GRAPH_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("=" * 115)
print("PHASE 7 — GEOMETRY-AWARE MOLECULAR GRAPH CONSTRUCTION")
print("=" * 115)

print(
    f"\n[INPUT]  Processed directory : {PROCESSED_DIR}"
)

print(
    f"[OUTPUT] Graph directory     : {PHASE7_GRAPH_DIR}"
)

print(
    f"[CONFIG] Heavy-atom range    : "
    f"{PHASE7_MIN_HEAVY_ATOMS}–{PHASE7_MAX_HEAVY_ATOMS}"
)

print(
    f"[CONFIG] Random seed         : {RANDOM_SEED}"
)


# ==============================================================================
# 5. DEFINE INPUT FILES
# ==============================================================================

D_CHEM_STANDARDIZED_FILE = (
    Path(PROCESSED_DIR)
    / "D_chem_standardized.csv"
)


D_BIO_ASSOCIATION_FILE = (
    Path(PROCESSED_DIR)
    / "D_bio_association.csv"
)


required_input_files = [
    D_CHEM_STANDARDIZED_FILE,
    D_BIO_ASSOCIATION_FILE
]


missing_files = [
    str(file_path)
    for file_path in required_input_files
    if not file_path.exists()
]


if missing_files:

    raise FileNotFoundError(
        "Required Phase-5/6 files were not found:\n"
        + "\n".join(
            missing_files
        )
    )


print(
    "\n[PASS] D_chem_standardized found."
)

print(
    "[PASS] D_bio_association found."
)


# ==============================================================================
# 6. LOAD INPUT DATA
# ==============================================================================

D_chem_standardized = pd.read_csv(
    D_CHEM_STANDARDIZED_FILE,
    dtype=str,
    keep_default_na=False
)


D_bio_association = pd.read_csv(
    D_BIO_ASSOCIATION_FILE,
    dtype=str,
    keep_default_na=False
)


print("\n" + "=" * 115)
print("PHASE-7 INPUT DATA")
print("=" * 115)

print(
    f"Standardized molecular records : "
    f"{len(D_chem_standardized)}"
)

print(
    f"Biological association rows    : "
    f"{len(D_bio_association)}"
)


# ==============================================================================
# 7. VERIFY REQUIRED INPUT COLUMNS
# ==============================================================================

required_chem_columns = [
    "drug_id",
    "drug_name",
    "raw_smiles",
    "canonical_smiles",
    "standardized_smiles",
    "heavy_atom_count"
]


required_bio_columns = [
    "drug_id",
    "organism"
]


missing_chem_columns = [
    column
    for column in required_chem_columns
    if column not in D_chem_standardized.columns
]


missing_bio_columns = [
    column
    for column in required_bio_columns
    if column not in D_bio_association.columns
]


if missing_chem_columns:

    raise RuntimeError(
        "D_chem_standardized is missing required columns:\n"
        f"{missing_chem_columns}"
    )


if missing_bio_columns:

    raise RuntimeError(
        "D_bio_association is missing required columns:\n"
        f"{missing_bio_columns}"
    )


if not D_chem_standardized[
    "drug_id"
].is_unique:

    raise RuntimeError(
        "D_chem_standardized contains duplicate drug IDs."
    )


print(
    "\n[PASS] Required molecular columns verified."
)

print(
    "[PASS] Biological linkage columns verified."
)

print(
    "[PASS] Standardized drug IDs are unique."
)


# ==============================================================================
# 8. PRESERVE ORIGINAL DATASETS
# ==============================================================================

D_chem_standardized_original = (
    D_chem_standardized.copy(
        deep=True
    )
)


D_bio_association_original = (
    D_bio_association.copy(
        deep=True
    )
)


# ==============================================================================
# 9. HELPER FUNCTIONS
# ==============================================================================

def clean_text(value):
    """
    Safely convert a value to stripped text.
    """

    if value is None:
        return ""

    if pd.isna(value):
        return ""

    return str(
        value
    ).strip()


def safe_int(value):
    """
    Safely convert text/numeric values to integer.
    """

    try:

        return int(
            float(
                clean_text(value)
            )
        )

    except Exception:

        return None


def safe_float(value):
    """
    Safely convert a value to float.
    """

    try:

        return float(
            clean_text(value)
        )

    except Exception:

        return None


def one_hot(
    value,
    categories
):
    """
    Deterministic one-hot encoding.

    Final category should normally represent OTHER.
    """

    result = [
        0.0
        for _ in categories
    ]

    if value in categories:

        index = categories.index(
            value
        )

    else:

        index = len(
            categories
        ) - 1

    result[
        index
    ] = 1.0

    return result


# ==============================================================================
# 10. DETERMINISTIC FEATURE VOCABULARIES
# ==============================================================================

HYBRIDIZATION_TYPES = [
    "UNSPECIFIED",
    "S",
    "SP",
    "SP2",
    "SP3",
    "SP3D",
    "SP3D2",
    "OTHER"
]


CHIRALITY_TYPES = [
    "CHI_UNSPECIFIED",
    "CHI_TETRAHEDRAL_CW",
    "CHI_TETRAHEDRAL_CCW",
    "CHI_OTHER"
]


BOND_TYPES = [
    "SINGLE",
    "DOUBLE",
    "TRIPLE",
    "AROMATIC",
    "OTHER"
]


BOND_STEREO_TYPES = [
    "STEREONONE",
    "STEREOANY",
    "STEREOZ",
    "STEREOE",
    "STEREOCIS",
    "STEREOTRANS",
    "OTHER"
]


# ==============================================================================
# 11. NODE FEATURE DEFINITION
# ==============================================================================

NODE_FEATURE_NAMES = [
    "atomic_number",
    "degree",
    "formal_charge",
    "aromatic",
    "in_ring",
    "hydrogen_neighbor_count",
    "atomic_mass",
    "implicit_valence",
    "explicit_valence"
]


NODE_FEATURE_NAMES += [
    f"hybridization_{name}"
    for name in HYBRIDIZATION_TYPES
]


NODE_FEATURE_NAMES += [
    f"chirality_{name}"
    for name in CHIRALITY_TYPES
]


NODE_FEATURE_DIM = len(
    NODE_FEATURE_NAMES
)


# ==============================================================================
# 12. EDGE FEATURE DEFINITION
# ==============================================================================

EDGE_FEATURE_NAMES = [
    f"bond_type_{name}"
    for name in BOND_TYPES
]


EDGE_FEATURE_NAMES += [
    "is_conjugated",
    "is_in_ring"
]


EDGE_FEATURE_NAMES += [
    f"stereo_{name}"
    for name in BOND_STEREO_TYPES
]


EDGE_FEATURE_DIM = len(
    EDGE_FEATURE_NAMES
)


print("\n" + "=" * 115)
print("GRAPH FEATURE CONFIGURATION")
print("=" * 115)

print(
    f"Node feature dimension : {NODE_FEATURE_DIM}"
)

print(
    f"Edge feature dimension : {EDGE_FEATURE_DIM}"
)


# ==============================================================================
# 13. SAFE ATOM VALENCE HELPERS
# ==============================================================================

def get_implicit_valence(atom):

    try:

        return float(
            atom.GetImplicitValence()
        )

    except Exception:

        return 0.0


def get_explicit_valence(atom):

    try:

        return float(
            atom.GetExplicitValence()
        )

    except Exception:

        return 0.0


# ==============================================================================
# 14. ATOM FEATURE CONSTRUCTION
# ==============================================================================

def atom_feature_vector(atom):
    """
    Build invariant scalar/categorical atomic attributes.

    IMPORTANT:
    Cartesian x/y/z coordinates are NOT included here.
    """

    hybridization = str(
        atom.GetHybridization()
    )


    chirality = str(
        atom.GetChiralTag()
    )


    # Because the molecular graph is hydrogen-explicit,
    # count explicit H neighbours directly.

    hydrogen_neighbor_count = sum(
        1
        for neighbor in atom.GetNeighbors()
        if neighbor.GetAtomicNum() == 1
    )


    features = [
        float(
            atom.GetAtomicNum()
        ),

        float(
            atom.GetDegree()
        ),

        float(
            atom.GetFormalCharge()
        ),

        float(
            atom.GetIsAromatic()
        ),

        float(
            atom.IsInRing()
        ),

        float(
            hydrogen_neighbor_count
        ),

        float(
            atom.GetMass()
        ),

        get_implicit_valence(
            atom
        ),

        get_explicit_valence(
            atom
        )
    ]


    features.extend(
        one_hot(
            hybridization,
            HYBRIDIZATION_TYPES
        )
    )


    features.extend(
        one_hot(
            chirality,
            CHIRALITY_TYPES
        )
    )


    if len(
        features
    ) != NODE_FEATURE_DIM:

        raise RuntimeError(
            "Node-feature dimensionality mismatch."
        )


    return features


# ==============================================================================
# 15. BOND FEATURE CONSTRUCTION
# ==============================================================================

def bond_feature_vector(bond):

    bond_type = str(
        bond.GetBondType()
    )


    stereo = str(
        bond.GetStereo()
    )


    features = []


    features.extend(
        one_hot(
            bond_type,
            BOND_TYPES
        )
    )


    features.extend(
        [
            float(
                bond.GetIsConjugated()
            ),

            float(
                bond.IsInRing()
            )
        ]
    )


    features.extend(
        one_hot(
            stereo,
            BOND_STEREO_TYPES
        )
    )


    if len(
        features
    ) != EDGE_FEATURE_DIM:

        raise RuntimeError(
            "Edge-feature dimensionality mismatch."
        )


    return features


# ==============================================================================
# 16. BUILD X
# ==============================================================================

def build_node_features(mol):

    features = [
        atom_feature_vector(
            atom
        )
        for atom in mol.GetAtoms()
    ]


    return torch.tensor(
        features,
        dtype=torch.float32
    )


# ==============================================================================
# 17. BUILD BIDIRECTIONAL EDGE GRAPH
# ==============================================================================

def build_edges(mol):

    source_nodes = []
    target_nodes = []
    edge_features = []


    for bond in mol.GetBonds():

        begin = bond.GetBeginAtomIdx()
        end = bond.GetEndAtomIdx()


        features = bond_feature_vector(
            bond
        )


        # Forward edge

        source_nodes.append(
            begin
        )

        target_nodes.append(
            end
        )

        edge_features.append(
            features
        )


        # Reverse edge

        source_nodes.append(
            end
        )

        target_nodes.append(
            begin
        )

        edge_features.append(
            features
        )


    if len(
        source_nodes
    ) == 0:

        edge_index = torch.empty(
            (
                2,
                0
            ),
            dtype=torch.long
        )


        edge_attr = torch.empty(
            (
                0,
                EDGE_FEATURE_DIM
            ),
            dtype=torch.float32
        )


    else:

        edge_index = torch.tensor(
            [
                source_nodes,
                target_nodes
            ],
            dtype=torch.long
        )


        edge_attr = torch.tensor(
            edge_features,
            dtype=torch.float32
        )


    return (
        edge_index,
        edge_attr
    )


# ==============================================================================
# 18. PRIMARY ETKDGv3 CONFORMER GENERATION
# ==============================================================================

def generate_primary_conformer(mol):

    params = AllChem.ETKDGv3()

    params.randomSeed = RANDOM_SEED

    params.useRandomCoords = False

    params.enforceChirality = True

    params.clearConfs = True


    conformer_id = AllChem.EmbedMolecule(
        mol,
        params
    )


    return conformer_id


# ==============================================================================
# 19. DETERMINISTIC FALLBACK CONFORMER GENERATION
# ==============================================================================

def generate_fallback_conformer(mol):

    # Remove any unsuccessful/partial conformers.

    mol.RemoveAllConformers()


    params = AllChem.ETKDGv3()

    params.randomSeed = RANDOM_SEED

    params.useRandomCoords = True

    params.enforceChirality = True

    params.clearConfs = True


    conformer_id = AllChem.EmbedMolecule(
        mol,
        params
    )


    return conformer_id


# ==============================================================================
# 20. GEOMETRY OPTIMIZATION
# ==============================================================================

def optimize_geometry(
    mol,
    conformer_id
):
    """
    Optimization hierarchy:

        MMFF94
           ↓
        UFF
           ↓
        NONE

    Returns a dictionary describing optimization outcome.
    """

    result = {
        "optimization_method":
            "NONE",

        "optimization_status":
            "NOT_ATTEMPTED",

        "optimization_converged":
            False,

        "conformer_energy":
            None,

        "optimization_error":
            ""
    }


    # --------------------------------------------------------------------------
    # Try MMFF94
    # --------------------------------------------------------------------------

    try:

        has_mmff = AllChem.MMFFHasAllMoleculeParams(
            mol
        )

    except Exception:

        has_mmff = False


    if has_mmff:

        try:

            properties = AllChem.MMFFGetMoleculeProperties(
                mol,
                mmffVariant="MMFF94"
            )


            force_field = AllChem.MMFFGetMoleculeForceField(
                mol,
                properties,
                confId=conformer_id
            )


            if force_field is not None:

                force_field.Initialize()


                status = force_field.Minimize(
                    maxIts=PHASE7_MAX_OPTIMIZATION_ITERS
                )


                energy = float(
                    force_field.CalcEnergy()
                )


                result[
                    "optimization_method"
                ] = "MMFF94"


                result[
                    "optimization_converged"
                ] = (
                    status == 0
                )


                result[
                    "optimization_status"
                ] = (
                    "CONVERGED"
                    if status == 0
                    else "MAX_ITERS_REACHED"
                )


                result[
                    "conformer_energy"
                ] = energy


                return result


        except Exception as exc:

            result[
                "optimization_error"
            ] = (
                f"MMFF94: {str(exc)}"
            )


    # --------------------------------------------------------------------------
    # Try UFF
    # --------------------------------------------------------------------------

    try:

        has_uff = AllChem.UFFHasAllMoleculeParams(
            mol
        )

    except Exception:

        has_uff = False


    if has_uff:

        try:

            force_field = AllChem.UFFGetMoleculeForceField(
                mol,
                confId=conformer_id
            )


            if force_field is not None:

                force_field.Initialize()


                status = force_field.Minimize(
                    maxIts=PHASE7_MAX_OPTIMIZATION_ITERS
                )


                energy = float(
                    force_field.CalcEnergy()
                )


                result[
                    "optimization_method"
                ] = "UFF"


                result[
                    "optimization_converged"
                ] = (
                    status == 0
                )


                result[
                    "optimization_status"
                ] = (
                    "CONVERGED"
                    if status == 0
                    else "MAX_ITERS_REACHED"
                )


                result[
                    "conformer_energy"
                ] = energy


                return result


        except Exception as exc:

            previous_error = result[
                "optimization_error"
            ]


            uff_error = (
                f"UFF: {str(exc)}"
            )


            result[
                "optimization_error"
            ] = (
                previous_error
                + " | "
                + uff_error
                if previous_error
                else uff_error
            )


    # --------------------------------------------------------------------------
    # No supported force field
    # --------------------------------------------------------------------------

    if (
        not has_mmff
        and not has_uff
    ):

        result[
            "optimization_method"
        ] = "NONE"


        result[
            "optimization_status"
        ] = "NO_SUPPORTED_FORCE_FIELD"


        return result


    # --------------------------------------------------------------------------
    # Parameters existed but optimization failed unexpectedly.
    # --------------------------------------------------------------------------

    result[
        "optimization_method"
    ] = "NONE"


    result[
        "optimization_status"
    ] = "GEOMETRY_OPTIMIZATION_FAILED"


    return result


# ==============================================================================
# 21. EXTRACT AND CENTER COORDINATES
# ==============================================================================

def extract_centered_coordinates(
    mol,
    conformer_id
):

    conformer = mol.GetConformer(
        conformer_id
    )


    coordinates = np.array(
        [
            [
                conformer.GetAtomPosition(
                    atom_index
                ).x,

                conformer.GetAtomPosition(
                    atom_index
                ).y,

                conformer.GetAtomPosition(
                    atom_index
                ).z
            ]
            for atom_index
            in range(
                mol.GetNumAtoms()
            )
        ],
        dtype=np.float32
    )


    if coordinates.ndim != 2:

        raise ValueError(
            "Coordinate array is not 2-dimensional."
        )


    if coordinates.shape[1] != 3:

        raise ValueError(
            "Coordinate array does not have XYZ dimension."
        )


    if not np.isfinite(
        coordinates
    ).all():

        raise ValueError(
            "Coordinates contain NaN or infinite values."
        )


    centroid = coordinates.mean(
        axis=0,
        keepdims=True
    )


    centered_coordinates = (
        coordinates
        - centroid
    )


    if not np.isfinite(
        centered_coordinates
    ).all():

        raise ValueError(
            "Centered coordinates contain NaN or infinite values."
        )


    return (
        torch.tensor(
            centered_coordinates,
            dtype=torch.float32
        ),

        torch.tensor(
            centroid.squeeze(
                axis=0
            ),
            dtype=torch.float32
        )
    )


# ==============================================================================
# 22. GRAPH CONSISTENCY VALIDATION
# ==============================================================================

def validate_graph(
    x,
    edge_index,
    edge_attr,
    pos,
    expected_atoms
):

    errors = []


    num_nodes = int(
        x.shape[0]
    )


    # X / R / molecule atom alignment

    if num_nodes != expected_atoms:

        errors.append(
            "NODE_COUNT_MISMATCH"
        )


    if pos.shape != (
        num_nodes,
        3
    ):

        errors.append(
            "POSITION_SHAPE_MISMATCH"
        )


    if x.shape[1] != NODE_FEATURE_DIM:

        errors.append(
            "NODE_FEATURE_DIMENSION_MISMATCH"
        )


    # Edge checks

    if edge_index.ndim != 2:

        errors.append(
            "EDGE_INDEX_INVALID_DIMENSION"
        )


    elif edge_index.shape[0] != 2:

        errors.append(
            "EDGE_INDEX_INVALID_SHAPE"
        )


    if edge_attr.ndim != 2:

        errors.append(
            "EDGE_ATTRIBUTE_INVALID_DIMENSION"
        )


    elif edge_attr.shape[1] != EDGE_FEATURE_DIM:

        errors.append(
            "EDGE_ATTRIBUTE_DIMENSION_MISMATCH"
        )


    if (
        edge_index.shape[1]
        != edge_attr.shape[0]
    ):

        errors.append(
            "EDGE_ATTRIBUTE_COUNT_MISMATCH"
        )


    # Node references

    if edge_index.numel() > 0:

        minimum_index = int(
            edge_index.min().item()
        )

        maximum_index = int(
            edge_index.max().item()
        )


        if minimum_index < 0:

            errors.append(
                "NEGATIVE_EDGE_INDEX"
            )


        if maximum_index >= num_nodes:

            errors.append(
                "EDGE_INDEX_OUT_OF_RANGE"
            )


    # Numerical validity

    if not torch.isfinite(
        x
    ).all():

        errors.append(
            "NONFINITE_NODE_FEATURE"
        )


    if not torch.isfinite(
        pos
    ).all():

        errors.append(
            "NONFINITE_COORDINATE"
        )


    if not torch.isfinite(
        edge_attr
    ).all():

        errors.append(
            "NONFINITE_EDGE_FEATURE"
        )


    if num_nodes <= 0:

        errors.append(
            "EMPTY_GRAPH"
        )


    return (
        len(
            errors
        ) == 0,
        errors
    )


# ==============================================================================
# 23. MODEL-FEASIBILITY SCREEN
# ==============================================================================

D_geometry_screen = (
    D_chem_standardized.copy()
)


D_geometry_screen[
    "heavy_atom_count_numeric"
] = pd.to_numeric(
    D_geometry_screen[
        "heavy_atom_count"
    ],
    errors="coerce"
)


missing_heavy_atom_mask = (
    D_geometry_screen[
        "heavy_atom_count_numeric"
    ].isna()
)


eligible_mask = (
    D_geometry_screen[
        "heavy_atom_count_numeric"
    ].between(
        PHASE7_MIN_HEAVY_ATOMS,
        PHASE7_MAX_HEAVY_ATOMS,
        inclusive="both"
    )
)


D_geometry_eligible = (
    D_geometry_screen[
        eligible_mask
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


D_geometry_size_excluded = (
    D_geometry_screen[
        ~eligible_mask
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


print("\n" + "=" * 115)
print("MODEL-FEASIBILITY SCREEN")
print("=" * 115)


print(
    f"Input standardized molecules : "
    f"{len(D_geometry_screen)}"
)


print(
    f"Eligible (4–80 heavy atoms)  : "
    f"{len(D_geometry_eligible)}"
)


print(
    f"Outside model-size criterion : "
    f"{len(D_geometry_size_excluded)}"
)


print(
    f"Missing heavy-atom count     : "
    f"{int(missing_heavy_atom_mask.sum())}"
)


# ==============================================================================
# 24. SAVE ELIGIBLE DATASET
# ==============================================================================

D_GEOMETRY_ELIGIBLE_FILE = (
    Path(PROCESSED_DIR)
    / "D_geometry_eligible.csv"
)


D_geometry_eligible.to_csv(
    D_GEOMETRY_ELIGIBLE_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 25. INITIAL EXCLUSION AUDIT
# ==============================================================================

excluded_records = []


for _, row in D_geometry_size_excluded.iterrows():

    heavy_atom_count = safe_int(
        row[
            "heavy_atom_count"
        ]
    )


    if heavy_atom_count is None:

        reason = (
            "HEAVY_ATOM_COUNT_MISSING"
        )

    else:

        reason = (
            "MODEL_SIZE_FILTER"
        )


    excluded_records.append(
        {
            "drug_id":
                clean_text(
                    row[
                        "drug_id"
                    ]
                ),

            "drug_name":
                clean_text(
                    row[
                        "drug_name"
                    ]
                ),

            "raw_smiles":
                clean_text(
                    row[
                        "raw_smiles"
                    ]
                ),

            "canonical_smiles":
                clean_text(
                    row[
                        "canonical_smiles"
                    ]
                ),

            "standardized_smiles":
                clean_text(
                    row[
                        "standardized_smiles"
                    ]
                ),

            "heavy_atom_count":
                heavy_atom_count,

            "failure_stage":
                "MODEL_FEASIBILITY",

            "exclusion_reason":
                reason,

            "details":
                (
                    f"Heavy atom count outside "
                    f"{PHASE7_MIN_HEAVY_ATOMS}–"
                    f"{PHASE7_MAX_HEAVY_ATOMS}"
                    if reason == "MODEL_SIZE_FILTER"
                    else "Heavy atom count unavailable"
                )
        }
    )


# ==============================================================================
# 26. BIOLOGICAL LINKAGE SETS
# ==============================================================================

bio_drug_ids = set(
    D_bio_association[
        "drug_id"
    ].astype(str)
)


human_bio_drug_ids = set(
    D_bio_association.loc[
        D_bio_association[
            "organism"
        ]
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(
            [
                "humans",
                "human",
                "homo sapiens"
            ]
        ),
        "drug_id"
    ].astype(str)
)


# ==============================================================================
# 27. PHASE-7 COUNTERS
# ==============================================================================

stats = {
    "input_standardized_molecules":
        len(
            D_chem_standardized
        ),

    "model_size_eligible_molecules":
        len(
            D_geometry_eligible
        ),

    "model_size_excluded_molecules":
        len(
            D_geometry_size_excluded
        ),

    "molecule_reconstruction_success":
        0,

    "molecule_reconstruction_failed":
        0,

    "hydrogen_addition_success":
        0,

    "hydrogen_addition_failed":
        0,

    "conformer_primary_success":
        0,

    "conformer_fallback_success":
        0,

    "conformer_generation_success":
        0,

    "conformer_generation_failed":
        0,

    "mmff_optimized":
        0,

    "uff_optimized":
        0,

    "optimization_converged":
        0,

    "optimization_not_converged":
        0,

    "no_supported_force_field":
        0,

    "geometry_optimization_failed":
        0,

    "invalid_coordinates":
        0,

    "graph_construction_failed":
        0,

    "final_valid_graphs":
        0
}


# ==============================================================================
# 28. GRAPH INDEX COLLECTION
# ==============================================================================

graph_index_records = []


# ==============================================================================
# 29. START PHASE-7 PROCESSING
# ==============================================================================

phase7_started_at = datetime.now()


print("\n" + "=" * 115)
print("GENERATING 3-D GEOMETRY-AWARE MOLECULAR GRAPHS")
print("=" * 115)


total_eligible = len(
    D_geometry_eligible
)


# ==============================================================================
# 30. PROCESS EVERY ELIGIBLE MOLECULE
# ==============================================================================

for record_number, (_, row) in enumerate(
    D_geometry_eligible.iterrows(),
    start=1
):

    drug_id = clean_text(
        row[
            "drug_id"
        ]
    )


    drug_name = clean_text(
        row[
            "drug_name"
        ]
    )


    raw_smiles = clean_text(
        row[
            "raw_smiles"
        ]
    )


    canonical_smiles = clean_text(
        row[
            "canonical_smiles"
        ]
    )


    standardized_smiles = clean_text(
        row[
            "standardized_smiles"
        ]
    )


    heavy_atom_count = safe_int(
        row[
            "heavy_atom_count"
        ]
    )


    # --------------------------------------------------------------------------
    # Progress
    # --------------------------------------------------------------------------

    if (
        record_number == 1
        or record_number % 50 == 0
        or record_number == total_eligible
    ):

        print(
            f"[PROGRESS] "
            f"{record_number}/{total_eligible} "
            f"molecules processed..."
        )


    # ==========================================================================
    # 30.1 MOLECULAR RECONSTRUCTION
    # ==========================================================================

    try:

        mol = Chem.MolFromSmiles(
            standardized_smiles
        )


        if mol is None:

            raise ValueError(
                "RDKit returned None."
            )


        stats[
            "molecule_reconstruction_success"
        ] += 1


    except Exception as exc:

        stats[
            "molecule_reconstruction_failed"
        ] += 1


        excluded_records.append(
            {
                "drug_id":
                    drug_id,

                "drug_name":
                    drug_name,

                "raw_smiles":
                    raw_smiles,

                "canonical_smiles":
                    canonical_smiles,

                "standardized_smiles":
                    standardized_smiles,

                "heavy_atom_count":
                    heavy_atom_count,

                "failure_stage":
                    "MOLECULAR_RECONSTRUCTION",

                "exclusion_reason":
                    "MOLECULE_RECONSTRUCTION_FAILED",

                "details":
                    str(
                        exc
                    )
            }
        )


        continue


    # ==========================================================================
    # 30.2 ADD EXPLICIT HYDROGENS
    # ==========================================================================

    try:

        mol_h = Chem.AddHs(
            mol
        )


        if mol_h is None:

            raise ValueError(
                "Chem.AddHs returned None."
            )


        total_atom_count = int(
            mol_h.GetNumAtoms()
        )


        stats[
            "hydrogen_addition_success"
        ] += 1


    except Exception as exc:

        stats[
            "hydrogen_addition_failed"
        ] += 1


        excluded_records.append(
            {
                "drug_id":
                    drug_id,

                "drug_name":
                    drug_name,

                "raw_smiles":
                    raw_smiles,

                "canonical_smiles":
                    canonical_smiles,

                "standardized_smiles":
                    standardized_smiles,

                "heavy_atom_count":
                    heavy_atom_count,

                "failure_stage":
                    "HYDROGEN_ADDITION",

                "exclusion_reason":
                    "HYDROGEN_ADDITION_FAILED",

                "details":
                    str(
                        exc
                    )
            }
        )


        continue


    # ==========================================================================
    # 30.3 PRIMARY ETKDGv3 CONFORMER
    # ==========================================================================

    conformer_id = -1
    conformer_method = ""


    try:

        conformer_id = generate_primary_conformer(
            mol_h
        )


    except Exception:

        conformer_id = -1


    if conformer_id >= 0:

        conformer_method = (
            "ETKDGv3_PRIMARY"
        )


        stats[
            "conformer_primary_success"
        ] += 1


    # ==========================================================================
    # 30.4 DETERMINISTIC FALLBACK
    # ==========================================================================

    else:

        try:

            conformer_id = generate_fallback_conformer(
                mol_h
            )


        except Exception:

            conformer_id = -1


        if conformer_id >= 0:

            conformer_method = (
                "ETKDGv3_RANDOM_COORDS_FALLBACK"
            )


            stats[
                "conformer_fallback_success"
            ] += 1


    # ==========================================================================
    # 30.5 CONFORMER FAILURE
    # ==========================================================================

    if conformer_id < 0:

        stats[
            "conformer_generation_failed"
        ] += 1


        excluded_records.append(
            {
                "drug_id":
                    drug_id,

                "drug_name":
                    drug_name,

                "raw_smiles":
                    raw_smiles,

                "canonical_smiles":
                    canonical_smiles,

                "standardized_smiles":
                    standardized_smiles,

                "heavy_atom_count":
                    heavy_atom_count,

                "failure_stage":
                    "CONFORMER_GENERATION",

                "exclusion_reason":
                    "CONFORMER_GENERATION_FAILED",

                "details":
                    (
                        "Primary ETKDGv3 and deterministic "
                        "random-coordinate fallback both failed."
                    )
            }
        )


        continue


    stats[
        "conformer_generation_success"
    ] += 1


    # ==========================================================================
    # 30.6 GEOMETRY OPTIMIZATION
    # ==========================================================================

    optimization = optimize_geometry(
        mol_h,
        conformer_id
    )


    optimization_method = optimization[
        "optimization_method"
    ]


    optimization_status = optimization[
        "optimization_status"
    ]


    optimization_converged = optimization[
        "optimization_converged"
    ]


    conformer_energy = optimization[
        "conformer_energy"
    ]


    # --------------------------------------------------------------------------
    # Count optimizer use
    # --------------------------------------------------------------------------

    if optimization_method == "MMFF94":

        stats[
            "mmff_optimized"
        ] += 1


    elif optimization_method == "UFF":

        stats[
            "uff_optimized"
        ] += 1


    elif (
        optimization_status
        == "NO_SUPPORTED_FORCE_FIELD"
    ):

        stats[
            "no_supported_force_field"
        ] += 1


    # --------------------------------------------------------------------------
    # Optimization result
    # --------------------------------------------------------------------------

    if optimization_converged:

        stats[
            "optimization_converged"
        ] += 1


    elif optimization_status in {
        "MAX_ITERS_REACHED",
        "NO_SUPPORTED_FORCE_FIELD"
    }:

        # A valid conformer still exists.
        # Keep it, but retain explicit status.

        stats[
            "optimization_not_converged"
        ] += 1


    elif (
        optimization_status
        == "GEOMETRY_OPTIMIZATION_FAILED"
    ):

        stats[
            "geometry_optimization_failed"
        ] += 1


        excluded_records.append(
            {
                "drug_id":
                    drug_id,

                "drug_name":
                    drug_name,

                "raw_smiles":
                    raw_smiles,

                "canonical_smiles":
                    canonical_smiles,

                "standardized_smiles":
                    standardized_smiles,

                "heavy_atom_count":
                    heavy_atom_count,

                "failure_stage":
                    "GEOMETRY_OPTIMIZATION",

                "exclusion_reason":
                    "GEOMETRY_OPTIMIZATION_FAILED",

                "details":
                    optimization[
                        "optimization_error"
                    ]
            }
        )


        continue


    # ==========================================================================
    # 30.7 EXTRACT AND CENTER COORDINATES
    # ==========================================================================

    try:

        pos, original_centroid = (
            extract_centered_coordinates(
                mol_h,
                conformer_id
            )
        )


    except Exception as exc:

        stats[
            "invalid_coordinates"
        ] += 1


        excluded_records.append(
            {
                "drug_id":
                    drug_id,

                "drug_name":
                    drug_name,

                "raw_smiles":
                    raw_smiles,

                "canonical_smiles":
                    canonical_smiles,

                "standardized_smiles":
                    standardized_smiles,

                "heavy_atom_count":
                    heavy_atom_count,

                "failure_stage":
                    "COORDINATE_EXTRACTION",

                "exclusion_reason":
                    "INVALID_COORDINATES",

                "details":
                    str(
                        exc
                    )
            }
        )


        continue


    # ==========================================================================
    # 30.8 BUILD NODE FEATURES X
    # ==========================================================================

    try:

        x = build_node_features(
            mol_h
        )


        # ======================================================================
        # 30.9 BUILD EDGE GRAPH E + EDGE ATTRIBUTES
        # ======================================================================

        edge_index, edge_attr = build_edges(
            mol_h
        )


        num_nodes = int(
            mol_h.GetNumAtoms()
        )


        num_bonds = int(
            mol_h.GetNumBonds()
        )


        num_directed_edges = int(
            edge_index.shape[1]
        )


        # ======================================================================
        # 30.10 GRAPH CONSISTENCY VALIDATION
        # ======================================================================

        graph_valid, graph_errors = validate_graph(
            x=x,
            edge_index=edge_index,
            edge_attr=edge_attr,
            pos=pos,
            expected_atoms=num_nodes
        )


        if not graph_valid:

            raise ValueError(
                "; ".join(
                    graph_errors
                )
            )


        # Additional bond-direction check

        if (
            num_directed_edges
            != 2 * num_bonds
        ):

            raise ValueError(
                "Directed edge count is not exactly "
                "twice the chemical bond count."
            )


    except Exception as exc:

        stats[
            "graph_construction_failed"
        ] += 1


        excluded_records.append(
            {
                "drug_id":
                    drug_id,

                "drug_name":
                    drug_name,

                "raw_smiles":
                    raw_smiles,

                "canonical_smiles":
                    canonical_smiles,

                "standardized_smiles":
                    standardized_smiles,

                "heavy_atom_count":
                    heavy_atom_count,

                "failure_stage":
                    "GRAPH_CONSTRUCTION",

                "exclusion_reason":
                    "GRAPH_CONSTRUCTION_FAILED",

                "details":
                    str(
                        exc
                    )
            }
        )


        continue


    # ==========================================================================
    # 30.11 BIOLOGICAL LINKAGE FLAGS
    # ==========================================================================

    has_biological_association = (
        drug_id
        in bio_drug_ids
    )


    has_human_biological_association = (
        drug_id
        in human_bio_drug_ids
    )


    # ==========================================================================
    # 30.12 GRAPH IDENTIFIER
    # ==========================================================================

    graph_id = drug_id


    # ==========================================================================
    # 30.13 GRAPH FILE
    # ==========================================================================

    graph_file = (
        PHASE7_GRAPH_DIR
        / f"{graph_id}.pt"
    )


    # ==========================================================================
    # 30.14 SERIALIZABLE GRAPH DICTIONARY
    # ==========================================================================

    graph_data = {
        # ----------------------------------------------------------------------
        # Identity / lineage
        # ----------------------------------------------------------------------

        "graph_id":
            graph_id,

        "drug_id":
            drug_id,

        "drug_name":
            drug_name,

        "raw_smiles":
            raw_smiles,

        "canonical_smiles":
            canonical_smiles,

        "standardized_smiles":
            standardized_smiles,


        # ----------------------------------------------------------------------
        # Graph representation G = (V, E, X, R)
        # ----------------------------------------------------------------------

        "x":
            x,

        "edge_index":
            edge_index,

        "edge_attr":
            edge_attr,

        "pos":
            pos,


        # ----------------------------------------------------------------------
        # Coordinate provenance
        # ----------------------------------------------------------------------

        "original_centroid":
            original_centroid,

        "coordinates_centered":
            True,


        # ----------------------------------------------------------------------
        # Graph metadata
        # ----------------------------------------------------------------------

        "heavy_atom_count":
            int(
                heavy_atom_count
            ),

        "total_atom_count":
            int(
                total_atom_count
            ),

        "num_nodes":
            int(
                num_nodes
            ),

        "num_bonds":
            int(
                num_bonds
            ),

        "num_directed_edges":
            int(
                num_directed_edges
            ),


        # ----------------------------------------------------------------------
        # Feature definitions
        # ----------------------------------------------------------------------

        "node_feature_names":
            NODE_FEATURE_NAMES,

        "edge_feature_names":
            EDGE_FEATURE_NAMES,

        "node_feature_dim":
            NODE_FEATURE_DIM,

        "edge_feature_dim":
            EDGE_FEATURE_DIM,


        # ----------------------------------------------------------------------
        # Geometry generation
        # ----------------------------------------------------------------------

        "random_seed":
            RANDOM_SEED,

        "conformer_method":
            conformer_method,

        "optimization_method":
            optimization_method,

        "optimization_status":
            optimization_status,

        "optimization_converged":
            bool(
                optimization_converged
            ),

        "conformer_energy":
            (
                float(
                    conformer_energy
                )
                if conformer_energy is not None
                and math.isfinite(
                    float(
                        conformer_energy
                    )
                )
                else None
            ),


        # ----------------------------------------------------------------------
        # Biological linkage only
        # ----------------------------------------------------------------------

        "has_biological_association":
            bool(
                has_biological_association
            ),

        "has_human_biological_association":
            bool(
                has_human_biological_association
            )
    }


    # ==========================================================================
    # 30.15 SAVE GRAPH
    # ==========================================================================

    try:

        torch.save(
            graph_data,
            graph_file
        )


    except Exception as exc:

        stats[
            "graph_construction_failed"
        ] += 1


        excluded_records.append(
            {
                "drug_id":
                    drug_id,

                "drug_name":
                    drug_name,

                "raw_smiles":
                    raw_smiles,

                "canonical_smiles":
                    canonical_smiles,

                "standardized_smiles":
                    standardized_smiles,

                "heavy_atom_count":
                    heavy_atom_count,

                "failure_stage":
                    "GRAPH_SERIALIZATION",

                "exclusion_reason":
                    "GRAPH_CONSTRUCTION_FAILED",

                "details":
                    f"torch.save failed: {str(exc)}"
            }
        )


        continue


    # ==========================================================================
    # 30.16 GRAPH INDEX
    # ==========================================================================

    graph_index_records.append(
        {
            "graph_id":
                graph_id,

            "drug_id":
                drug_id,

            "drug_name":
                drug_name,

            "raw_smiles":
                raw_smiles,

            "canonical_smiles":
                canonical_smiles,

            "standardized_smiles":
                standardized_smiles,

            "heavy_atom_count":
                heavy_atom_count,

            "total_atom_count":
                total_atom_count,

            "num_nodes":
                num_nodes,

            "num_bonds":
                num_bonds,

            "num_directed_edges":
                num_directed_edges,

            "node_feature_dim":
                NODE_FEATURE_DIM,

            "edge_feature_dim":
                EDGE_FEATURE_DIM,

            "conformer_method":
                conformer_method,

            "optimization_method":
                optimization_method,

            "optimization_status":
                optimization_status,

            "optimization_converged":
                optimization_converged,

            "conformer_energy":
                conformer_energy,

            "coordinates_centered":
                True,

            "has_biological_association":
                has_biological_association,

            "has_human_biological_association":
                has_human_biological_association,

            "graph_file":
                str(
                    graph_file
                )
        }
    )


    stats[
        "final_valid_graphs"
    ] += 1


# ==============================================================================
# 31. CREATE GRAPH INDEX DATAFRAME
# ==============================================================================

graph_index_columns = [
    "graph_id",
    "drug_id",
    "drug_name",
    "raw_smiles",
    "canonical_smiles",
    "standardized_smiles",
    "heavy_atom_count",
    "total_atom_count",
    "num_nodes",
    "num_bonds",
    "num_directed_edges",
    "node_feature_dim",
    "edge_feature_dim",
    "conformer_method",
    "optimization_method",
    "optimization_status",
    "optimization_converged",
    "conformer_energy",
    "coordinates_centered",
    "has_biological_association",
    "has_human_biological_association",
    "graph_file"
]


phase7_graph_index = pd.DataFrame(
    graph_index_records,
    columns=graph_index_columns
)


# ==============================================================================
# 32. CREATE EXCLUSION DATAFRAME
# ==============================================================================

exclusion_columns = [
    "drug_id",
    "drug_name",
    "raw_smiles",
    "canonical_smiles",
    "standardized_smiles",
    "heavy_atom_count",
    "failure_stage",
    "exclusion_reason",
    "details"
]


phase7_geometry_excluded = pd.DataFrame(
    excluded_records,
    columns=exclusion_columns
)


# ==============================================================================
# 33. VERIFY INPUT ACCOUNTING
# ==============================================================================

final_graph_drug_ids = set(
    phase7_graph_index[
        "drug_id"
    ].astype(str)
)


excluded_drug_ids = set(
    phase7_geometry_excluded[
        "drug_id"
    ].astype(str)
)


input_drug_ids = set(
    D_chem_standardized[
        "drug_id"
    ].astype(str)
)


accounted_drug_ids = (
    final_graph_drug_ids
    | excluded_drug_ids
)


all_input_molecules_accounted_for = (
    input_drug_ids
    == accounted_drug_ids
)


graph_and_exclusion_overlap = (
    final_graph_drug_ids
    & excluded_drug_ids
)


no_graph_exclusion_overlap = (
    len(
        graph_and_exclusion_overlap
    ) == 0
)


# ==============================================================================
# 34. GRAPH-SIZE STATISTICS
# ==============================================================================

if not phase7_graph_index.empty:

    stats[
        "minimum_graph_nodes"
    ] = int(
        phase7_graph_index[
            "num_nodes"
        ].min()
    )


    stats[
        "median_graph_nodes"
    ] = float(
        phase7_graph_index[
            "num_nodes"
        ].median()
    )


    stats[
        "maximum_graph_nodes"
    ] = int(
        phase7_graph_index[
            "num_nodes"
        ].max()
    )


    stats[
        "minimum_graph_bonds"
    ] = int(
        phase7_graph_index[
            "num_bonds"
        ].min()
    )


    stats[
        "median_graph_bonds"
    ] = float(
        phase7_graph_index[
            "num_bonds"
        ].median()
    )


    stats[
        "maximum_graph_bonds"
    ] = int(
        phase7_graph_index[
            "num_bonds"
        ].max()
    )


    stats[
        "minimum_directed_edges"
    ] = int(
        phase7_graph_index[
            "num_directed_edges"
        ].min()
    )


    stats[
        "median_directed_edges"
    ] = float(
        phase7_graph_index[
            "num_directed_edges"
        ].median()
    )


    stats[
        "maximum_directed_edges"
    ] = int(
        phase7_graph_index[
            "num_directed_edges"
        ].max()
    )


    stats[
        "graphs_with_biological_associations"
    ] = int(
        phase7_graph_index[
            "has_biological_association"
        ].sum()
    )


    stats[
        "graphs_without_biological_associations"
    ] = int(
        (
            ~phase7_graph_index[
                "has_biological_association"
            ]
        ).sum()
    )


    stats[
        "graphs_with_human_biological_associations"
    ] = int(
        phase7_graph_index[
            "has_human_biological_association"
        ].sum()
    )


else:

    stats[
        "minimum_graph_nodes"
    ] = None

    stats[
        "median_graph_nodes"
    ] = None

    stats[
        "maximum_graph_nodes"
    ] = None

    stats[
        "minimum_graph_bonds"
    ] = None

    stats[
        "median_graph_bonds"
    ] = None

    stats[
        "maximum_graph_bonds"
    ] = None

    stats[
        "minimum_directed_edges"
    ] = None

    stats[
        "median_directed_edges"
    ] = None

    stats[
        "maximum_directed_edges"
    ] = None

    stats[
        "graphs_with_biological_associations"
    ] = 0

    stats[
        "graphs_without_biological_associations"
    ] = 0

    stats[
        "graphs_with_human_biological_associations"
    ] = 0


# ==============================================================================
# 35. OPTIMIZATION DISTRIBUTION
# ==============================================================================

if not phase7_graph_index.empty:

    optimization_distribution = (
        phase7_graph_index[
            "optimization_method"
        ]
        .value_counts()
        .to_dict()
    )


    optimization_status_distribution = (
        phase7_graph_index[
            "optimization_status"
        ]
        .value_counts()
        .to_dict()
    )


    conformer_method_distribution = (
        phase7_graph_index[
            "conformer_method"
        ]
        .value_counts()
        .to_dict()
    )


else:

    optimization_distribution = {}

    optimization_status_distribution = {}

    conformer_method_distribution = {}


# ==============================================================================
# 36. EXCLUSION DISTRIBUTION
# ==============================================================================

if not phase7_geometry_excluded.empty:

    exclusion_distribution = (
        phase7_geometry_excluded[
            "exclusion_reason"
        ]
        .value_counts()
        .to_dict()
    )


else:

    exclusion_distribution = {}


# ==============================================================================
# 37. VERIFY ORIGINAL INPUTS REMAIN UNCHANGED
# ==============================================================================

original_chem_preserved = (
    D_chem_standardized.equals(
        D_chem_standardized_original
    )
)


original_bio_preserved = (
    D_bio_association.equals(
        D_bio_association_original
    )
)


# ==============================================================================
# 38. VERIFY GRAPH-ID UNIQUENESS
# ==============================================================================

graph_ids_unique = (
    phase7_graph_index[
        "graph_id"
    ].is_unique
)


# ==============================================================================
# 39. VERIFY ALL GRAPH FILES EXIST
# ==============================================================================

if not phase7_graph_index.empty:

    all_graph_files_exist = all(
        Path(
            graph_file
        ).exists()
        for graph_file
        in phase7_graph_index[
            "graph_file"
        ]
    )


else:

    all_graph_files_exist = False


# ==============================================================================
# 40. VERIFY SAVED GRAPH STRUCTURE
# ==============================================================================

sample_graph_validation_passed = False


if not phase7_graph_index.empty:

    sample_graph_path = Path(
        phase7_graph_index.iloc[
            0
        ][
            "graph_file"
        ]
    )


    try:

        # weights_only=False is intentional because the file contains
        # tensors plus simple metadata strings/lists.

        sample_graph = torch.load(
            sample_graph_path,
            map_location="cpu",
            weights_only=False
        )


        required_graph_keys = {
            "graph_id",
            "drug_id",
            "x",
            "edge_index",
            "edge_attr",
            "pos",
            "heavy_atom_count",
            "total_atom_count",
            "conformer_method",
            "optimization_method",
            "optimization_status"
        }


        sample_graph_validation_passed = (
            required_graph_keys.issubset(
                set(
                    sample_graph.keys()
                )
            )
            and
            sample_graph[
                "x"
            ].shape[
                0
            ]
            ==
            sample_graph[
                "pos"
            ].shape[
                0
            ]
        )


    except Exception:

        sample_graph_validation_passed = False


# ==============================================================================
# 41. FINAL TIMING
# ==============================================================================

phase7_finished_at = datetime.now()


phase7_duration = (
    phase7_finished_at
    - phase7_started_at
)


stats[
    "phase7_duration_seconds"
] = round(
    phase7_duration.total_seconds(),
    3
)


stats[
    "all_input_molecules_accounted_for"
] = all_input_molecules_accounted_for


stats[
    "original_D_chem_standardized_preserved"
] = original_chem_preserved


stats[
    "original_D_bio_association_preserved"
] = original_bio_preserved


stats[
    "graph_ids_unique"
] = graph_ids_unique


stats[
    "all_graph_files_exist"
] = all_graph_files_exist


stats[
    "sample_graph_validation_passed"
] = sample_graph_validation_passed


# ==============================================================================
# 42. OUTPUT FILE PATHS
# ==============================================================================

PHASE7_GRAPH_INDEX_FILE = (
    Path(PROCESSED_DIR)
    / "phase7_graph_index.csv"
)


PHASE7_EXCLUDED_FILE = (
    Path(PROCESSED_DIR)
    / "phase7_geometry_excluded.csv"
)


PHASE7_SUMMARY_CSV_FILE = (
    Path(PROCESSED_DIR)
    / "phase7_geometry_qc_summary.csv"
)


PHASE7_SUMMARY_JSON_FILE = (
    Path(PROCESSED_DIR)
    / "phase7_geometry_qc_summary.json"
)


PHASE7_FEATURE_SCHEMA_FILE = (
    Path(PROCESSED_DIR)
    / "phase7_graph_feature_schema.json"
)


# ==============================================================================
# 43. SAVE GRAPH INDEX AND EXCLUSION REPORT
# ==============================================================================

phase7_graph_index.to_csv(
    PHASE7_GRAPH_INDEX_FILE,
    index=False,
    encoding="utf-8"
)


phase7_geometry_excluded.to_csv(
    PHASE7_EXCLUDED_FILE,
    index=False,
    encoding="utf-8"
)


print(
    "\n[PASS] Phase-7 graph index saved."
)

print(
    "[PASS] Phase-7 exclusion audit saved."
)


# ==============================================================================
# 44. SAVE FEATURE SCHEMA
# ==============================================================================

feature_schema = {
    "graph_representation":
        "G = (V, E, X, R)",

    "node_policy":
        "Hydrogen-explicit molecular graph",

    "coordinate_policy":
        (
            "Centered Cartesian coordinates; "
            "coordinates are separate from scalar node features."
        ),

    "node_feature_dimension":
        NODE_FEATURE_DIM,

    "node_feature_names":
        NODE_FEATURE_NAMES,

    "edge_feature_dimension":
        EDGE_FEATURE_DIM,

    "edge_feature_names":
        EDGE_FEATURE_NAMES,

    "edges_bidirectional":
        True,

    "coordinates_centered":
        True,

    "random_seed":
        RANDOM_SEED,

    "primary_conformer_method":
        "ETKDGv3",

    "fallback_conformer_method":
        "ETKDGv3 with deterministic random coordinates",

    "geometry_optimization_hierarchy":
        [
            "MMFF94",
            "UFF",
            "NONE"
        ],

    "model_feasibility_heavy_atom_range":
        [
            PHASE7_MIN_HEAVY_ATOMS,
            PHASE7_MAX_HEAVY_ATOMS
        ],

    "heavy_atom_filter_interpretation":
        "MODEL_FEASIBILITY_FILTER_NOT_CHEMICAL_VALIDITY"
}


with open(
    PHASE7_FEATURE_SCHEMA_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        feature_schema,
        file,
        indent=4,
        ensure_ascii=False
    )


# ==============================================================================
# 45. PHASE-7 METADATA
# ==============================================================================

phase7_metadata = {
    "phase":
        "Phase 7 — Geometry-Aware Molecular Graph Construction",

    "generated_at":
        datetime.now().isoformat(),

    "drugbank_version":
        DRUGBANK_CONFIG.get(
            "drugbank_version"
        ),

    "drugbank_exported_on":
        DRUGBANK_CONFIG.get(
            "drugbank_exported_on"
        ),

    "source_D_chem_standardized":
        str(
            D_CHEM_STANDARDIZED_FILE
        ),

    "source_D_bio_association":
        str(
            D_BIO_ASSOCIATION_FILE
        ),

    "graph_directory":
        str(
            PHASE7_GRAPH_DIR
        ),

    "graph_representation":
        "G = (V, E, X, R)",

    "node_policy":
        "hydrogen-explicit",

    "coordinate_policy":
        "centered_3D_coordinates",

    "coordinates_in_node_features":
        False,

    "single_conformer_baseline":
        True,

    "random_seed":
        RANDOM_SEED,

    "model_feasibility_rule":
        (
            f"{PHASE7_MIN_HEAVY_ATOMS}–"
            f"{PHASE7_MAX_HEAVY_ATOMS} heavy atoms"
        ),

    "model_feasibility_is_chemical_validity":
        False,

    "statistics":
        stats,

    "conformer_method_distribution":
        conformer_method_distribution,

    "optimization_method_distribution":
        optimization_distribution,

    "optimization_status_distribution":
        optimization_status_distribution,

    "exclusion_distribution":
        exclusion_distribution
}


# ==============================================================================
# 46. SAVE SUMMARY JSON
# ==============================================================================

with open(
    PHASE7_SUMMARY_JSON_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        phase7_metadata,
        file,
        indent=4,
        ensure_ascii=False,
        default=str
    )


# ==============================================================================
# 47. SAVE SUMMARY CSV
# ==============================================================================

summary_rows = []


for section_name, section_value in phase7_metadata.items():

    if isinstance(
        section_value,
        dict
    ):

        for key, value in section_value.items():

            summary_rows.append(
                {
                    "metric":
                        f"{section_name}.{key}",

                    "value":
                        value
                }
            )

    else:

        summary_rows.append(
            {
                "metric":
                    section_name,

                "value":
                    section_value
            }
        )


phase7_summary_df = pd.DataFrame(
    summary_rows
)


phase7_summary_df.to_csv(
    PHASE7_SUMMARY_CSV_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 48. DISPLAY MAIN QC RESULTS
# ==============================================================================

print("\n" + "=" * 115)
print("PHASE 7 GEOMETRY / GRAPH QC RESULTS")
print("=" * 115)


print(
    f"Input standardized molecules         : "
    f"{stats['input_standardized_molecules']}"
)

print(
    f"Model-size eligible molecules        : "
    f"{stats['model_size_eligible_molecules']}"
)

print(
    f"Model-size excluded molecules        : "
    f"{stats['model_size_excluded_molecules']}"
)

print(
    f"Molecule reconstruction success      : "
    f"{stats['molecule_reconstruction_success']}"
)

print(
    f"Molecule reconstruction failures     : "
    f"{stats['molecule_reconstruction_failed']}"
)

print(
    f"Hydrogen addition success            : "
    f"{stats['hydrogen_addition_success']}"
)

print(
    f"Hydrogen addition failures           : "
    f"{stats['hydrogen_addition_failed']}"
)

print(
    f"Primary ETKDGv3 success              : "
    f"{stats['conformer_primary_success']}"
)

print(
    f"Fallback conformer success           : "
    f"{stats['conformer_fallback_success']}"
)

print(
    f"Total conformer successes            : "
    f"{stats['conformer_generation_success']}"
)

print(
    f"Conformer-generation failures        : "
    f"{stats['conformer_generation_failed']}"
)

print(
    f"MMFF94 optimized                     : "
    f"{stats['mmff_optimized']}"
)

print(
    f"UFF optimized                        : "
    f"{stats['uff_optimized']}"
)

print(
    f"Optimization converged               : "
    f"{stats['optimization_converged']}"
)

print(
    f"Optimization non-converged/unsupported: "
    f"{stats['optimization_not_converged']}"
)

print(
    f"Geometry optimization failures       : "
    f"{stats['geometry_optimization_failed']}"
)

print(
    f"Invalid-coordinate failures          : "
    f"{stats['invalid_coordinates']}"
)

print(
    f"Graph-construction failures          : "
    f"{stats['graph_construction_failed']}"
)

print(
    f"Final valid molecular graphs         : "
    f"{stats['final_valid_graphs']}"
)


# ==============================================================================
# 49. GRAPH-SIZE SUMMARY
# ==============================================================================

print("\n" + "=" * 115)
print("GRAPH SIZE STATISTICS")
print("=" * 115)


print(
    f"Minimum nodes          : "
    f"{stats['minimum_graph_nodes']}"
)

print(
    f"Median nodes           : "
    f"{stats['median_graph_nodes']}"
)

print(
    f"Maximum nodes          : "
    f"{stats['maximum_graph_nodes']}"
)

print(
    f"Minimum bonds          : "
    f"{stats['minimum_graph_bonds']}"
)

print(
    f"Median bonds           : "
    f"{stats['median_graph_bonds']}"
)

print(
    f"Maximum bonds          : "
    f"{stats['maximum_graph_bonds']}"
)

print(
    f"Minimum directed edges : "
    f"{stats['minimum_directed_edges']}"
)

print(
    f"Median directed edges  : "
    f"{stats['median_directed_edges']}"
)

print(
    f"Maximum directed edges : "
    f"{stats['maximum_directed_edges']}"
)


# ==============================================================================
# 50. BIOLOGICAL LINKAGE SUMMARY
# ==============================================================================

print("\n" + "=" * 115)
print("GRAPH ↔ BIOLOGICAL LINKAGE")
print("=" * 115)


print(
    f"Graphs with biological associations       : "
    f"{stats['graphs_with_biological_associations']}"
)

print(
    f"Graphs without biological associations    : "
    f"{stats['graphs_without_biological_associations']}"
)

print(
    f"Graphs with human biological associations : "
    f"{stats['graphs_with_human_biological_associations']}"
)


# ==============================================================================
# 51. OPTIMIZATION METHOD DISTRIBUTION
# ==============================================================================

print("\n" + "=" * 115)
print("GEOMETRY OPTIMIZATION DISTRIBUTION")
print("=" * 115)


if optimization_distribution:

    for method, count in optimization_distribution.items():

        print(
            f"{method:<20}: {count}"
        )


else:

    print(
        "No successful graphs."
    )


# ==============================================================================
# 52. EXCLUSION BREAKDOWN
# ==============================================================================

print("\n" + "=" * 115)
print("PHASE 7 EXCLUSION BREAKDOWN")
print("=" * 115)


if exclusion_distribution:

    for reason, count in exclusion_distribution.items():

        print(
            f"{reason:<40}: {count}"
        )


else:

    print(
        "No exclusions."
    )


# ==============================================================================
# 53. GRAPH INDEX PREVIEW
# ==============================================================================

print("\n" + "=" * 115)
print("PHASE 7 GRAPH INDEX PREVIEW")
print("=" * 115)


if phase7_graph_index.empty:

    print(
        "No graph records available."
    )


else:

    preview_columns = [
        "graph_id",
        "drug_id",
        "drug_name",
        "heavy_atom_count",
        "total_atom_count",
        "num_nodes",
        "num_bonds",
        "num_directed_edges",
        "conformer_method",
        "optimization_method",
        "optimization_status",
        "has_biological_association"
    ]


    print(
        phase7_graph_index[
            preview_columns
        ]
        .head(10)
        .to_string(
            index=False
        )
    )


# ==============================================================================
# 54. PHASE-7 COMPLETION VALIDATION
# ==============================================================================

completion_checks = {
    "D_chem_standardized loaded":
        len(
            D_chem_standardized
        ) > 0,

    "Model-feasibility screen applied":
        (
            len(
                D_geometry_eligible
            )
            +
            len(
                D_geometry_size_excluded
            )
            ==
            len(
                D_chem_standardized
            )
        ),

    "4–80 criterion treated as model feasibility":
        True,

    "Eligible molecular structures processed":
        (
            stats[
                "molecule_reconstruction_success"
            ]
            +
            stats[
                "molecule_reconstruction_failed"
            ]
            ==
            len(
                D_geometry_eligible
            )
        ),

    "Explicit hydrogen addition attempted":
        (
            stats[
                "hydrogen_addition_success"
            ]
            +
            stats[
                "hydrogen_addition_failed"
            ]
            ==
            stats[
                "molecule_reconstruction_success"
            ]
        ),

    "Conformer generation accounted for":
        (
            stats[
                "conformer_generation_success"
            ]
            +
            stats[
                "conformer_generation_failed"
            ]
            ==
            stats[
                "hydrogen_addition_success"
            ]
        ),

    "Geometry-aware graphs created":
        len(
            phase7_graph_index
        ) > 0,

    "Graph IDs unique":
        graph_ids_unique,

    "All graph files exist":
        all_graph_files_exist,

    "Saved graph structure validated":
        sample_graph_validation_passed,

    "Node and coordinate ordering aligned":
        sample_graph_validation_passed,

    "Coordinates stored separately from X":
        True,

    "Coordinates centered":
        (
            phase7_graph_index[
                "coordinates_centered"
            ].all()
            if not phase7_graph_index.empty
            else False
        ),

    "Edges are bidirectional":
        (
            (
                phase7_graph_index[
                    "num_directed_edges"
                ]
                ==
                2
                * phase7_graph_index[
                    "num_bonds"
                ]
            ).all()
            if not phase7_graph_index.empty
            else False
        ),

    "Original D_chem_standardized preserved":
        original_chem_preserved,

    "Original D_bio_association preserved":
        original_bio_preserved,

    "Every input molecule accounted for":
        all_input_molecules_accounted_for,

    "No molecule both graph and excluded":
        no_graph_exclusion_overlap,

    "Graph index saved":
        PHASE7_GRAPH_INDEX_FILE.exists(),

    "Geometry exclusion report saved":
        PHASE7_EXCLUDED_FILE.exists(),

    "QC summary CSV saved":
        PHASE7_SUMMARY_CSV_FILE.exists(),

    "QC summary JSON saved":
        PHASE7_SUMMARY_JSON_FILE.exists(),

    "Feature schema saved":
        PHASE7_FEATURE_SCHEMA_FILE.exists(),

    "No neural-network training performed":
        True
}


# ==============================================================================
# 55. DISPLAY COMPLETION VALIDATION
# ==============================================================================

print("\n" + "=" * 115)
print("PHASE 7 COMPLETION VALIDATION")
print("=" * 115)


all_critical_checks_passed = True


for check_name, status in completion_checks.items():

    label = (
        "PASS"
        if status
        else "FAIL"
    )


    print(
        f"[{label:<4}] {check_name}"
    )


    if not status:

        all_critical_checks_passed = False


# ==============================================================================
# 56. PACKAGE PHASE-7 CONFIGURATION
# ==============================================================================

PHASE7_DATASETS = {
    "graph_directory":
        str(
            PHASE7_GRAPH_DIR
        ),

    "D_geometry_eligible_csv":
        str(
            D_GEOMETRY_ELIGIBLE_FILE
        ),

    "graph_index_csv":
        str(
            PHASE7_GRAPH_INDEX_FILE
        ),

    "geometry_excluded_csv":
        str(
            PHASE7_EXCLUDED_FILE
        ),

    "summary_csv":
        str(
            PHASE7_SUMMARY_CSV_FILE
        ),

    "summary_json":
        str(
            PHASE7_SUMMARY_JSON_FILE
        ),

    "feature_schema_json":
        str(
            PHASE7_FEATURE_SCHEMA_FILE
        ),

    "node_feature_dim":
        NODE_FEATURE_DIM,

    "edge_feature_dim":
        EDGE_FEATURE_DIM,

    "statistics":
        stats
}


DRUGBANK_CONFIG[
    "phase7_datasets"
] = PHASE7_DATASETS


# ==============================================================================
# 57. FINAL PHASE-7 SUMMARY
# ==============================================================================

print("\n" + "=" * 115)
print("PHASE 7 SUMMARY")
print("=" * 115)


print(
    f"Input Standardized Molecules     : "
    f"{stats['input_standardized_molecules']}"
)

print(
    f"Model-Eligible Molecules         : "
    f"{stats['model_size_eligible_molecules']}"
)

print(
    f"Model-Size Exclusions            : "
    f"{stats['model_size_excluded_molecules']}"
)

print(
    f"Final Valid Molecular Graphs     : "
    f"{stats['final_valid_graphs']}"
)

print(
    f"Node Feature Dimension           : "
    f"{NODE_FEATURE_DIM}"
)

print(
    f"Edge Feature Dimension           : "
    f"{EDGE_FEATURE_DIM}"
)

print(
    f"Coordinates Centered             : True"
)

print(
    f"Explicit Hydrogen Graphs         : True"
)

print(
    f"Single-Conformer Baseline        : True"
)

print(
    f"Graph IDs Unique                 : "
    f"{graph_ids_unique}"
)

print(
    f"Original D_chem Preserved        : "
    f"{original_chem_preserved}"
)

print(
    f"Biological Data Remains Separate : "
    f"{original_bio_preserved}"
)

print(
    f"All Molecules Accounted For      : "
    f"{all_input_molecules_accounted_for}"
)

print(
    f"Neural-Network Training          : False"
)

print(
    f"Phase Duration (seconds)         : "
    f"{stats['phase7_duration_seconds']}"
)


print("\nSaved Files:")

print(
    f"  Graph Directory       : "
    f"{PHASE7_GRAPH_DIR}"
)

print(
    f"  Geometry Eligible     : "
    f"{D_GEOMETRY_ELIGIBLE_FILE}"
)

print(
    f"  Graph Index           : "
    f"{PHASE7_GRAPH_INDEX_FILE}"
)

print(
    f"  Geometry Exclusions   : "
    f"{PHASE7_EXCLUDED_FILE}"
)

print(
    f"  Feature Schema        : "
    f"{PHASE7_FEATURE_SCHEMA_FILE}"
)

print(
    f"  QC Summary CSV        : "
    f"{PHASE7_SUMMARY_CSV_FILE}"
)

print(
    f"  QC Summary JSON       : "
    f"{PHASE7_SUMMARY_JSON_FILE}"
)


print("-" * 115)


if all_critical_checks_passed:

    print(
        "[STATUS] PHASE 7 COMPLETED SUCCESSFULLY."
    )

    print(
        "[READY]  Geometry-aware molecular graph "
        "corpus G=(V,E,X,R) has been created."
    )

    print(
        "[NEXT]   Phase 8 — Baseline Chemical Corpus Construction / "
        "Geometry-Aware Encoder Preparation"
    )


else:

    print(
        "[STATUS] PHASE 7 COMPLETED WITH ONE OR MORE "
        "FAILED VALIDATION CHECKS."
    )

    print(
        "[ACTION] Review the failed geometry / graph QC checks "
        "before proceeding to Phase 8."
    )


print("=" * 115)

PHASE 7 — GEOMETRY-AWARE MOLECULAR GRAPH CONSTRUCTION

[INPUT]  Processed directory : E:\MyPhD\My Patent-1\Implementation\data\processed
[OUTPUT] Graph directory     : E:\MyPhD\My Patent-1\Implementation\data\graphs
[CONFIG] Heavy-atom range    : 4–80
[CONFIG] Random seed         : 42

[PASS] D_chem_standardized found.
[PASS] D_bio_association found.

PHASE-7 INPUT DATA
Standardized molecular records : 789
Biological association rows    : 5058

[PASS] Required molecular columns verified.
[PASS] Biological linkage columns verified.
[PASS] Standardized drug IDs are unique.

GRAPH FEATURE CONFIGURATION
Node feature dimension : 21
Edge feature dimension : 14

MODEL-FEASIBILITY SCREEN
Input standardized molecules : 789
Eligible (4–80 heavy atoms)  : 769
Outside model-size criterion : 20
Missing heavy-atom count     : 0

GENERATING 3-D GEOMETRY-AWARE MOLECULAR GRAPHS
[PROGRESS] 1/769 molecules processed...


[14:25:05] UFFTYPER: Unrecognized charge state for atom: 1


[PROGRESS] 50/769 molecules processed...


[14:25:28] UFFTYPER: Unrecognized charge state for atom: 8


[PROGRESS] 100/769 molecules processed...
[PROGRESS] 150/769 molecules processed...
[PROGRESS] 200/769 molecules processed...


...

[PROGRESS] 500/769 molecules processed...
[PROGRESS] 550/769 molecules processed...


[14:28:16] UFFTYPER: Unrecognized charge state for atom: 8


[PROGRESS] 600/769 molecules processed...


[14:28:18] UFFTYPER: Unrecognized charge state for atom: 4


[PROGRESS] 650/769 molecules processed...
[PROGRESS] 700/769 molecules processed...
[PROGRESS] 750/769 molecules processed...
[PROGRESS] 769/769 molecules processed...

[PASS] Phase-7 graph index saved.
[PASS] Phase-7 exclusion audit saved.

PHASE 7 GEOMETRY / GRAPH QC RESULTS
Input standardized molecules         : 789
Model-size eligible molecules        : 769
Model-size excluded molecules        : 20
Molecule reconstruction success      : 769
Molecule reconstruction failures     : 0
Hydrogen addition success            : 769
Hydrogen addition failures           : 0
Primary ETKDGv3 success              : 767
Fallback conformer success           : 0
Total conformer successes            : 767
Conformer-generation failures        : 2
MMFF94 optimized                     : 765
UFF optimized                        : 2
Optimization converged               : 734
Optimization non-converged/unsupported: 33
Geometry optimization failures       : 0
Invalid-coordinate failures          : 0
Graph-

# Phase 8 — Baseline Chemical Corpus Construction

**Objective:** Partition the 767 validated geometry-aware molecular graphs into fixed, reproducible subsets for training, validation, testing, and downstream continual-learning knowledge preservation without data leakage.

* **Core Splitting & Lineage Invariants:**
  * **Lineage Preservation:** Establishes `D_baseline_graph` as a new lineage tier without mutating earlier molecular datasets (`D_chem_raw`, `D_chem`, `D_chem_standardized`).
  * **Molecule-Level Splitting:** Partitions strictly by `drug_id` / `graph_id`, preserving complete one-to-many biological target associations modularly without atom- or bond-level fragmentation.
  * **Deterministic Stratification:** Enforces an 80/10/10 random molecule-level split using `RANDOM_SEED = 42` (613 Train, 76 Validation, 78 Test).
  * **Strict Disjointness:** Guarantees zero data leakage across all partition intersections ($\text{Train} \cap \text{Val} = \emptyset$, $\text{Train} \cap \text{Test} = \emptyset$, $\text{Val} \cap \text{Test} = \emptyset$) with 100% corpus coverage.
  * **Fisher Reference Corpus:** Extracts a fixed reference subset of 128 graphs sampled strictly from `D_train` to support Module 108 (Elastic Weight Consolidation) without validation or test leakage.
  * **Unannotated Chemistry Retention:** Preserves the 12 molecules lacking biological target records to maintain an unbiased chemical structure manifold for unsupervised geometric encoder pretraining.

* **Execution Steps:**
  1. Load `phase7_graph_index.csv` and verify structural completeness, node-coordinate alignment, and finite coordinates across all 767 `.pt` graph files.
  2. Compute deterministic random permutations and partition the graph identifiers into Train, Validation, and Test subsets.
  3. Sample the Fisher reference subset deterministically from the training partition.
  4. Generate and save the master split manifest `phase8_baseline_split_manifest.csv`.
  5. Export partition tables (`phase8_train_graphs.csv`, `phase8_validation_graphs.csv`, `phase8_test_graphs.csv`, `phase8_fisher_reference.csv`).
  6. Produce comprehensive graph integrity and data-leakage audit reports in `phase8_baseline_corpus_summary.csv` and `.json`.

* **Expected Output:**
  Verified, leakage-free baseline chemical knowledge corpus partitioned into Train (613), Validation (76), Test (78), and Fisher Reference (128) datasets stored in `./data/processed/`, ready for Phase 9 geometry-aware molecular encoder modeling.

In [8]:
# ==============================================================================
# PHASE 8 — BASELINE CHEMICAL CORPUS CONSTRUCTION
# CORRECTED FINAL VERSION
# ==============================================================================

from pathlib import Path
from datetime import datetime
import json
import random

import numpy as np
import pandas as pd
import torch


# ==============================================================================
# 1. REPRODUCIBILITY
# ==============================================================================

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)


# ==============================================================================
# 2. SPLIT CONFIGURATION
# ==============================================================================

TRAIN_FRACTION = 0.80
VALIDATION_FRACTION = 0.10
TEST_FRACTION = 0.10

FISHER_REFERENCE_MAX_SIZE = 128


if not np.isclose(
    TRAIN_FRACTION + VALIDATION_FRACTION + TEST_FRACTION,
    1.0
):
    raise ValueError(
        "Train/validation/test fractions must sum to 1.0."
    )


# ==============================================================================
# 3. VERIFY PROJECT STATE
# ==============================================================================

required_variables = [
    "PROCESSED_DIR",
    "DRUGBANK_CONFIG"
]

missing_variables = [
    variable
    for variable in required_variables
    if variable not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Earlier phases must be executed before Phase 8.\n"
        f"Missing variables: {missing_variables}"
    )


print("=" * 115)
print("PHASE 8 — BASELINE CHEMICAL CORPUS CONSTRUCTION")
print("=" * 115)

print(f"\n[DIR] Processed Directory : {PROCESSED_DIR}")
print(f"[CONFIG] Random Seed      : {RANDOM_SEED}")
print(f"[CONFIG] Train Fraction   : {TRAIN_FRACTION:.2f}")
print(f"[CONFIG] Validation       : {VALIDATION_FRACTION:.2f}")
print(f"[CONFIG] Test             : {TEST_FRACTION:.2f}")
print(f"[CONFIG] Fisher Max Size  : {FISHER_REFERENCE_MAX_SIZE}")


# ==============================================================================
# 4. INPUT FILES
# ==============================================================================

PHASE7_GRAPH_INDEX_FILE = (
    Path(PROCESSED_DIR)
    / "phase7_graph_index.csv"
)

D_BIO_ASSOCIATION_FILE = (
    Path(PROCESSED_DIR)
    / "D_bio_association.csv"
)


if not PHASE7_GRAPH_INDEX_FILE.exists():
    raise FileNotFoundError(
        f"Phase-7 graph index not found:\n{PHASE7_GRAPH_INDEX_FILE}"
    )


if not D_BIO_ASSOCIATION_FILE.exists():
    raise FileNotFoundError(
        f"Phase-6 biological association file not found:\n"
        f"{D_BIO_ASSOCIATION_FILE}"
    )


print("\n[PASS] Phase-7 graph index found.")
print("[PASS] Phase-6 biological association dataset found.")


# ==============================================================================
# 5. LOAD SOURCE DATA
# ==============================================================================

D_baseline_graph_source = pd.read_csv(
    PHASE7_GRAPH_INDEX_FILE,
    dtype=str,
    keep_default_na=False
)

D_bio_association = pd.read_csv(
    D_BIO_ASSOCIATION_FILE,
    dtype=str,
    keep_default_na=False
)


print("\n" + "=" * 115)
print("BASELINE INPUT DATA")
print("=" * 115)

print(
    f"Phase-7 valid graph records : "
    f"{len(D_baseline_graph_source)}"
)

print(
    f"Biological association rows : "
    f"{len(D_bio_association)}"
)


# ==============================================================================
# 6. REQUIRED COLUMNS
# ==============================================================================

required_graph_columns = [
    "graph_id",
    "drug_id",
    "drug_name",
    "graph_file",
    "heavy_atom_count",
    "num_nodes",
    "num_bonds",
    "num_directed_edges",
    "has_biological_association",
    "has_human_biological_association"
]


missing_graph_columns = [
    column
    for column in required_graph_columns
    if column not in D_baseline_graph_source.columns
]


if missing_graph_columns:
    raise RuntimeError(
        "Phase-7 graph index missing required columns:\n"
        f"{missing_graph_columns}"
    )


if "drug_id" not in D_bio_association.columns:
    raise RuntimeError(
        "D_bio_association does not contain drug_id."
    )


print(
    "\n[PASS] Required Phase-7 graph-index columns verified."
)


# ==============================================================================
# 7. PRESERVE ORIGINAL SOURCE DATA
# ==============================================================================

D_baseline_graph_original = (
    D_baseline_graph_source.copy(deep=True)
)

D_bio_association_original = (
    D_bio_association.copy(deep=True)
)


# ==============================================================================
# 8. CREATE WORKING COPY
# ==============================================================================

D_baseline_graph = (
    D_baseline_graph_source.copy(deep=True)
)


# ==============================================================================
# 9. UNIQUE IDENTIFIER VALIDATION
# ==============================================================================

duplicate_graph_ids_mask = (
    D_baseline_graph["graph_id"]
    .duplicated(keep=False)
)

duplicate_drug_ids_mask = (
    D_baseline_graph["drug_id"]
    .duplicated(keep=False)
)


duplicate_graph_id_count = int(
    duplicate_graph_ids_mask.sum()
)

duplicate_drug_id_count = int(
    duplicate_drug_ids_mask.sum()
)


if duplicate_graph_id_count > 0:
    raise RuntimeError(
        f"Duplicate graph IDs detected: {duplicate_graph_id_count}"
    )


if duplicate_drug_id_count > 0:
    raise RuntimeError(
        f"Duplicate drug IDs detected: {duplicate_drug_id_count}"
    )


print("[PASS] Graph IDs are unique.")
print("[PASS] Drug IDs are unique.")


# ==============================================================================
# 10. BOOLEAN PARSER
# ==============================================================================

def parse_bool(value):

    value = str(value).strip().lower()

    if value in {
        "true",
        "1",
        "yes"
    }:
        return True

    if value in {
        "false",
        "0",
        "no",
        ""
    }:
        return False

    raise ValueError(
        f"Unexpected boolean value: {value}"
    )


# ==============================================================================
# 11. VERIFY PHASE-7 BIOLOGICAL FLAGS AGAINST PHASE 6
# ==============================================================================

bio_drug_ids = set(
    D_bio_association[
        "drug_id"
    ].astype(str)
)


if "is_human_target" in D_bio_association.columns:

    human_mask = (
        D_bio_association[
            "is_human_target"
        ]
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(
            [
                "true",
                "1",
                "yes"
            ]
        )
    )

else:

    human_mask = (
        D_bio_association[
            "organism"
        ]
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(
            [
                "humans",
                "human",
                "homo sapiens"
            ]
        )
    )


human_bio_drug_ids = set(
    D_bio_association.loc[
        human_mask,
        "drug_id"
    ].astype(str)
)


phase7_bio_flags = (
    D_baseline_graph[
        "has_biological_association"
    ].apply(parse_bool)
)


phase7_human_flags = (
    D_baseline_graph[
        "has_human_biological_association"
    ].apply(parse_bool)
)


expected_bio_flags = (
    D_baseline_graph[
        "drug_id"
    ]
    .astype(str)
    .isin(bio_drug_ids)
)


expected_human_flags = (
    D_baseline_graph[
        "drug_id"
    ]
    .astype(str)
    .isin(human_bio_drug_ids)
)


bio_linkage_flag_matches = bool(
    (
        phase7_bio_flags
        ==
        expected_bio_flags
    ).all()
)


human_linkage_flag_matches = bool(
    (
        phase7_human_flags
        ==
        expected_human_flags
    ).all()
)


if not bio_linkage_flag_matches:
    raise RuntimeError(
        "Phase-7 biological-association flags do not "
        "match Phase-6 data."
    )


if not human_linkage_flag_matches:
    raise RuntimeError(
        "Phase-7 human-biological-association flags do not "
        "match Phase-6 data."
    )


print(
    "[PASS] Phase-7 biological flags verified against Phase 6."
)

print(
    "[PASS] Phase-7 human-association flags verified against Phase 6."
)


# ==============================================================================
# 12. GRAPH FILE INTEGRITY VALIDATION
# ==============================================================================

print("\n" + "=" * 115)
print("GRAPH FILE INTEGRITY VALIDATION")
print("=" * 115)


graph_integrity_records = []


for _, row in D_baseline_graph.iterrows():

    graph_id = str(
        row["graph_id"]
    ).strip()

    drug_id = str(
        row["drug_id"]
    ).strip()

    graph_path = Path(
        row["graph_file"]
    )


    integrity = {
        "graph_id":
            graph_id,

        "drug_id":
            drug_id,

        "graph_file":
            str(graph_path),

        "file_exists":
            False,

        "load_success":
            False,

        "required_keys_present":
            False,

        "graph_identity_matches":
            False,

        "node_position_alignment":
            False,

        "coordinates_finite":
            False,

        "node_features_finite":
            False,

        "edge_features_finite":
            False,

        "edge_index_valid":
            False,

        "edge_attribute_count_matches":
            False,

        "integrity_pass":
            False,

        "error":
            ""
    }


    if not graph_path.exists():

        integrity["error"] = "GRAPH_FILE_MISSING"

        graph_integrity_records.append(
            integrity
        )

        continue


    integrity["file_exists"] = True


    try:

        graph = torch.load(
            graph_path,
            map_location="cpu",
            weights_only=False
        )


        integrity[
            "load_success"
        ] = True


        required_keys = {
            "graph_id",
            "drug_id",
            "x",
            "edge_index",
            "edge_attr",
            "pos"
        }


        required_keys_present = (
            required_keys.issubset(
                graph.keys()
            )
        )


        integrity[
            "required_keys_present"
        ] = required_keys_present


        if not required_keys_present:

            integrity[
                "error"
            ] = "REQUIRED_GRAPH_KEYS_MISSING"

            graph_integrity_records.append(
                integrity
            )

            continue


        graph_identity_matches = (
            str(
                graph[
                    "graph_id"
                ]
            ) == graph_id
            and
            str(
                graph[
                    "drug_id"
                ]
            ) == drug_id
        )


        integrity[
            "graph_identity_matches"
        ] = graph_identity_matches


        x = graph["x"]

        pos = graph["pos"]

        edge_index = graph["edge_index"]

        edge_attr = graph["edge_attr"]


        node_position_alignment = (
            x.ndim == 2
            and
            pos.ndim == 2
            and
            pos.shape[1] == 3
            and
            x.shape[0] == pos.shape[0]
        )


        integrity[
            "node_position_alignment"
        ] = node_position_alignment


        coordinates_finite = bool(
            torch.isfinite(
                pos
            ).all()
        )


        node_features_finite = bool(
            torch.isfinite(
                x
            ).all()
        )


        edge_features_finite = bool(
            torch.isfinite(
                edge_attr
            ).all()
        )


        integrity[
            "coordinates_finite"
        ] = coordinates_finite

        integrity[
            "node_features_finite"
        ] = node_features_finite

        integrity[
            "edge_features_finite"
        ] = edge_features_finite


        edge_index_valid = (
            edge_index.ndim == 2
            and
            edge_index.shape[0] == 2
        )


        if (
            edge_index_valid
            and
            edge_index.numel() > 0
        ):

            edge_index_valid = (
                int(
                    edge_index.min().item()
                ) >= 0
                and
                int(
                    edge_index.max().item()
                )
                <
                int(
                    x.shape[0]
                )
            )


        integrity[
            "edge_index_valid"
        ] = edge_index_valid


        edge_attribute_count_matches = (
            edge_attr.ndim == 2
            and
            edge_attr.shape[0]
            ==
            edge_index.shape[1]
        )


        integrity[
            "edge_attribute_count_matches"
        ] = edge_attribute_count_matches


        integrity_pass = (
            required_keys_present
            and
            graph_identity_matches
            and
            node_position_alignment
            and
            coordinates_finite
            and
            node_features_finite
            and
            edge_features_finite
            and
            edge_index_valid
            and
            edge_attribute_count_matches
        )


        integrity[
            "integrity_pass"
        ] = integrity_pass


        if not integrity_pass:

            integrity[
                "error"
            ] = "GRAPH_INTEGRITY_VALIDATION_FAILED"


    except Exception as exc:

        integrity[
            "error"
        ] = str(exc)


    graph_integrity_records.append(
        integrity
    )


phase8_graph_integrity = pd.DataFrame(
    graph_integrity_records
)


integrity_failure_count = int(
    (
        ~phase8_graph_integrity[
            "integrity_pass"
        ]
    ).sum()
)


print(
    f"Graph records checked : "
    f"{len(phase8_graph_integrity)}"
)

print(
    f"Integrity PASS        : "
    f"{int(phase8_graph_integrity['integrity_pass'].sum())}"
)

print(
    f"Integrity FAIL        : "
    f"{integrity_failure_count}"
)


if integrity_failure_count > 0:

    failed_graphs = (
        phase8_graph_integrity[
            ~phase8_graph_integrity[
                "integrity_pass"
            ]
        ]
    )


    print(
        failed_graphs[
            [
                "graph_id",
                "drug_id",
                "error"
            ]
        ]
        .head(20)
        .to_string(index=False)
    )


    raise RuntimeError(
        "Phase 8 cannot continue until all graphs "
        "pass integrity validation."
    )


print(
    "\n[PASS] All Phase-7 graph files passed integrity validation."
)


# ==============================================================================
# 13. VERIFY SOURCE PRESERVATION BEFORE SPLIT
# ==============================================================================

original_phase7_preserved_before_split = (
    D_baseline_graph_source.equals(
        D_baseline_graph_original
    )
)


if not original_phase7_preserved_before_split:

    raise RuntimeError(
        "Original Phase-7 graph index changed unexpectedly."
    )


print(
    "[PASS] Original Phase-7 graph index remains unchanged."
)


# ==============================================================================
# 14. DETERMINISTIC MOLECULE-LEVEL SHUFFLE
# ==============================================================================

baseline_indices = np.arange(
    len(D_baseline_graph)
)


split_rng = np.random.default_rng(
    RANDOM_SEED
)


shuffled_indices = split_rng.permutation(
    baseline_indices
)


# ==============================================================================
# 15. SPLIT COUNTS
# ==============================================================================

total_graphs = len(
    D_baseline_graph
)


train_count = int(
    np.floor(
        total_graphs
        *
        TRAIN_FRACTION
    )
)


validation_count = int(
    np.floor(
        total_graphs
        *
        VALIDATION_FRACTION
    )
)


test_count = (
    total_graphs
    -
    train_count
    -
    validation_count
)


print("\n" + "=" * 115)
print("PRIMARY SPLIT COUNTS")
print("=" * 115)

print(
    f"Total graphs : {total_graphs}"
)

print(
    f"Train        : {train_count}"
)

print(
    f"Validation   : {validation_count}"
)

print(
    f"Test         : {test_count}"
)


# ==============================================================================
# 16. SPLIT INDICES
# ==============================================================================

train_indices = shuffled_indices[
    :
    train_count
]


validation_indices = shuffled_indices[
    train_count
    :
    train_count
    +
    validation_count
]


test_indices = shuffled_indices[
    train_count
    +
    validation_count
    :
]


# ==============================================================================
# 17. CREATE PRIMARY SPLITS
# ==============================================================================

D_train = (
    D_baseline_graph
    .iloc[
        train_indices
    ]
    .copy()
    .reset_index(drop=True)
)


D_validation = (
    D_baseline_graph
    .iloc[
        validation_indices
    ]
    .copy()
    .reset_index(drop=True)
)


D_test = (
    D_baseline_graph
    .iloc[
        test_indices
    ]
    .copy()
    .reset_index(drop=True)
)


D_train[
    "split"
] = "TRAIN"


D_validation[
    "split"
] = "VALIDATION"


D_test[
    "split"
] = "TEST"


# ==============================================================================
# 18. LEAKAGE VALIDATION
# ==============================================================================

train_ids = set(
    D_train[
        "graph_id"
    ].astype(str)
)


validation_ids = set(
    D_validation[
        "graph_id"
    ].astype(str)
)


test_ids = set(
    D_test[
        "graph_id"
    ].astype(str)
)


train_validation_overlap = (
    train_ids
    &
    validation_ids
)


train_test_overlap = (
    train_ids
    &
    test_ids
)


validation_test_overlap = (
    validation_ids
    &
    test_ids
)


if train_validation_overlap:
    raise RuntimeError(
        "Train-validation leakage detected."
    )


if train_test_overlap:
    raise RuntimeError(
        "Train-test leakage detected."
    )


if validation_test_overlap:
    raise RuntimeError(
        "Validation-test leakage detected."
    )


print(
    "\n[PASS] Train/validation/test sets are mutually disjoint."
)


# ==============================================================================
# 19. COMPLETE CORPUS COVERAGE
# ==============================================================================

all_graph_ids = set(
    D_baseline_graph[
        "graph_id"
    ].astype(str)
)


primary_split_union = (
    train_ids
    |
    validation_ids
    |
    test_ids
)


complete_split_coverage = (
    primary_split_union
    ==
    all_graph_ids
)


if not complete_split_coverage:

    raise RuntimeError(
        "Split union does not equal complete graph corpus."
    )


print(
    "[PASS] Primary split union equals complete graph corpus."
)


# ==============================================================================
# 20. CREATE FISHER REFERENCE
# ==============================================================================

fisher_reference_size = min(
    FISHER_REFERENCE_MAX_SIZE,
    len(D_train)
)


fisher_rng = np.random.default_rng(
    RANDOM_SEED
)


fisher_selected_positions = (
    fisher_rng.choice(
        len(D_train),
        size=fisher_reference_size,
        replace=False
    )
)


D_fisher_reference = (
    D_train
    .iloc[
        fisher_selected_positions
    ]
    .copy()
    .reset_index(drop=True)
)


fisher_ids = set(
    D_fisher_reference[
        "graph_id"
    ].astype(str)
)


# ==============================================================================
# 21. FISHER VALIDATION
# ==============================================================================

fisher_subset_of_train = (
    fisher_ids.issubset(
        train_ids
    )
)


fisher_validation_overlap = (
    fisher_ids
    &
    validation_ids
)


fisher_test_overlap = (
    fisher_ids
    &
    test_ids
)


if not fisher_subset_of_train:

    raise RuntimeError(
        "Fisher reference contains non-training graphs."
    )


if fisher_validation_overlap:

    raise RuntimeError(
        "Fisher-validation leakage detected."
    )


if fisher_test_overlap:

    raise RuntimeError(
        "Fisher-test leakage detected."
    )


print(
    "[PASS] Fisher reference corpus sampled only from training data."
)


# ==============================================================================
# 22. ADD FISHER FLAGS ONLY TO PHASE-8 SPLIT COPIES
# ==============================================================================

D_train[
    "is_fisher_reference"
] = (
    D_train[
        "graph_id"
    ]
    .astype(str)
    .isin(fisher_ids)
)


D_validation[
    "is_fisher_reference"
] = False


D_test[
    "is_fisher_reference"
] = False


D_fisher_reference[
    "is_fisher_reference"
] = True


# ==============================================================================
# 23. MASTER MANIFEST
# ==============================================================================

phase8_baseline_split_manifest = pd.concat(
    [
        D_train,
        D_validation,
        D_test
    ],
    ignore_index=True
)


graph_order_lookup = {
    graph_id: index
    for index, graph_id
    in enumerate(
        D_baseline_graph[
            "graph_id"
        ].astype(str)
    )
}


phase8_baseline_split_manifest[
    "_phase7_order"
] = (
    phase8_baseline_split_manifest[
        "graph_id"
    ]
    .astype(str)
    .map(
        graph_order_lookup
    )
)


phase8_baseline_split_manifest = (
    phase8_baseline_split_manifest
    .sort_values(
        "_phase7_order"
    )
    .drop(
        columns=[
            "_phase7_order"
        ]
    )
    .reset_index(drop=True)
)


# ==============================================================================
# 24. EXACTLY ONE SPLIT PER GRAPH
# ==============================================================================

manifest_counts = (
    phase8_baseline_split_manifest[
        "graph_id"
    ]
    .value_counts()
)


each_graph_exactly_once = bool(
    (
        manifest_counts == 1
    ).all()
)


if not each_graph_exactly_once:

    raise RuntimeError(
        "At least one graph appears more than once "
        "in the primary split manifest."
    )


# ==============================================================================
# 25. REPRODUCIBILITY CHECK
# ==============================================================================

split_rng_check = np.random.default_rng(
    RANDOM_SEED
)


repeat_shuffle = split_rng_check.permutation(
    baseline_indices
)


split_reproducible = np.array_equal(
    shuffled_indices,
    repeat_shuffle
)


fisher_rng_check = np.random.default_rng(
    RANDOM_SEED
)


repeat_fisher_positions = (
    fisher_rng_check.choice(
        len(D_train),
        size=fisher_reference_size,
        replace=False
    )
)


fisher_reproducible = (
    set(
        repeat_fisher_positions.tolist()
    )
    ==
    set(
        fisher_selected_positions.tolist()
    )
)


# ==============================================================================
# 26. BIOLOGICAL SPLIT STATISTICS
# ==============================================================================

def bool_column_sum(
    dataframe,
    column
):

    return int(
        dataframe[
            column
        ]
        .apply(parse_bool)
        .sum()
    )


train_with_bio = bool_column_sum(
    D_train,
    "has_biological_association"
)


validation_with_bio = bool_column_sum(
    D_validation,
    "has_biological_association"
)


test_with_bio = bool_column_sum(
    D_test,
    "has_biological_association"
)


fisher_with_bio = bool_column_sum(
    D_fisher_reference,
    "has_biological_association"
)


train_with_human = bool_column_sum(
    D_train,
    "has_human_biological_association"
)


validation_with_human = bool_column_sum(
    D_validation,
    "has_human_biological_association"
)


test_with_human = bool_column_sum(
    D_test,
    "has_human_biological_association"
)


fisher_with_human = bool_column_sum(
    D_fisher_reference,
    "has_human_biological_association"
)


# ==============================================================================
# 27. GRAPH SIZE STATISTICS
# ==============================================================================

def graph_split_statistics(
    dataframe,
    prefix
):

    result = {}

    for column in [
        "num_nodes",
        "num_bonds",
        "heavy_atom_count"
    ]:

        values = pd.to_numeric(
            dataframe[
                column
            ],
            errors="coerce"
        ).dropna()


        if values.empty:

            result[
                f"{prefix}_{column}_min"
            ] = None

            result[
                f"{prefix}_{column}_median"
            ] = None

            result[
                f"{prefix}_{column}_max"
            ] = None

        else:

            result[
                f"{prefix}_{column}_min"
            ] = float(
                values.min()
            )

            result[
                f"{prefix}_{column}_median"
            ] = float(
                values.median()
            )

            result[
                f"{prefix}_{column}_max"
            ] = float(
                values.max()
            )

    return result


# ==============================================================================
# 28. FINAL SOURCE PRESERVATION CHECK
# ==============================================================================

original_phase7_graph_index_preserved = (
    D_baseline_graph_source.equals(
        D_baseline_graph_original
    )
)


original_bio_preserved = (
    D_bio_association.equals(
        D_bio_association_original
    )
)


if not original_phase7_graph_index_preserved:

    raise RuntimeError(
        "Original Phase-7 graph index was modified."
    )


if not original_bio_preserved:

    raise RuntimeError(
        "Original Phase-6 biological dataset was modified."
    )


# ==============================================================================
# 29. STATISTICS
# ==============================================================================

stats = {
    "total_graph_corpus":
        len(D_baseline_graph),

    "training_graphs":
        len(D_train),

    "validation_graphs":
        len(D_validation),

    "test_graphs":
        len(D_test),

    "fisher_reference_graphs":
        len(D_fisher_reference),

    "actual_train_percentage":
        round(
            100
            *
            len(D_train)
            /
            total_graphs,
            4
        ),

    "actual_validation_percentage":
        round(
            100
            *
            len(D_validation)
            /
            total_graphs,
            4
        ),

    "actual_test_percentage":
        round(
            100
            *
            len(D_test)
            /
            total_graphs,
            4
        ),

    "train_validation_overlap":
        len(train_validation_overlap),

    "train_test_overlap":
        len(train_test_overlap),

    "validation_test_overlap":
        len(validation_test_overlap),

    "fisher_validation_overlap":
        len(fisher_validation_overlap),

    "fisher_test_overlap":
        len(fisher_test_overlap),

    "duplicate_graph_ids":
        duplicate_graph_id_count,

    "duplicate_drug_ids":
        duplicate_drug_id_count,

    "graph_integrity_failures":
        integrity_failure_count,

    "phase7_bio_flags_match_phase6":
        bio_linkage_flag_matches,

    "phase7_human_flags_match_phase6":
        human_linkage_flag_matches,

    "training_graphs_with_bio":
        train_with_bio,

    "validation_graphs_with_bio":
        validation_with_bio,

    "test_graphs_with_bio":
        test_with_bio,

    "fisher_graphs_with_bio":
        fisher_with_bio,

    "training_graphs_with_human_bio":
        train_with_human,

    "validation_graphs_with_human_bio":
        validation_with_human,

    "test_graphs_with_human_bio":
        test_with_human,

    "fisher_graphs_with_human_bio":
        fisher_with_human,

    "split_reproducible":
        split_reproducible,

    "fisher_reproducible":
        fisher_reproducible,

    "fisher_subset_of_training":
        fisher_subset_of_train,

    "complete_split_coverage":
        complete_split_coverage,

    "each_graph_exactly_once":
        each_graph_exactly_once,

    "original_phase7_graph_index_preserved":
        original_phase7_graph_index_preserved,

    "original_biological_dataset_preserved":
        original_bio_preserved
}


stats.update(
    graph_split_statistics(
        D_train,
        "train"
    )
)


stats.update(
    graph_split_statistics(
        D_validation,
        "validation"
    )
)


stats.update(
    graph_split_statistics(
        D_test,
        "test"
    )
)


stats.update(
    graph_split_statistics(
        D_fisher_reference,
        "fisher"
    )
)


# ==============================================================================
# 30. OUTPUT FILES
# ==============================================================================

PHASE8_MANIFEST_FILE = (
    Path(PROCESSED_DIR)
    / "phase8_baseline_split_manifest.csv"
)


PHASE8_TRAIN_FILE = (
    Path(PROCESSED_DIR)
    / "phase8_train_graphs.csv"
)


PHASE8_VALIDATION_FILE = (
    Path(PROCESSED_DIR)
    / "phase8_validation_graphs.csv"
)


PHASE8_TEST_FILE = (
    Path(PROCESSED_DIR)
    / "phase8_test_graphs.csv"
)


PHASE8_FISHER_FILE = (
    Path(PROCESSED_DIR)
    / "phase8_fisher_reference.csv"
)


PHASE8_GRAPH_INTEGRITY_FILE = (
    Path(PROCESSED_DIR)
    / "phase8_graph_integrity_report.csv"
)


PHASE8_SUMMARY_CSV_FILE = (
    Path(PROCESSED_DIR)
    / "phase8_baseline_corpus_summary.csv"
)


PHASE8_SUMMARY_JSON_FILE = (
    Path(PROCESSED_DIR)
    / "phase8_baseline_corpus_summary.json"
)


# ==============================================================================
# 31. SAVE OUTPUT FILES
# ==============================================================================

phase8_baseline_split_manifest.to_csv(
    PHASE8_MANIFEST_FILE,
    index=False,
    encoding="utf-8"
)


D_train.to_csv(
    PHASE8_TRAIN_FILE,
    index=False,
    encoding="utf-8"
)


D_validation.to_csv(
    PHASE8_VALIDATION_FILE,
    index=False,
    encoding="utf-8"
)


D_test.to_csv(
    PHASE8_TEST_FILE,
    index=False,
    encoding="utf-8"
)


D_fisher_reference.to_csv(
    PHASE8_FISHER_FILE,
    index=False,
    encoding="utf-8"
)


phase8_graph_integrity.to_csv(
    PHASE8_GRAPH_INTEGRITY_FILE,
    index=False,
    encoding="utf-8"
)


print(
    "\n[PASS] Phase-8 corpus files saved."
)


# ==============================================================================
# 32. METADATA
# ==============================================================================

phase8_metadata = {
    "phase":
        "Phase 8 — Baseline Chemical Corpus Construction",

    "generated_at":
        datetime.now().isoformat(),

    "drugbank_version":
        DRUGBANK_CONFIG.get(
            "drugbank_version"
        ),

    "drugbank_exported_on":
        DRUGBANK_CONFIG.get(
            "drugbank_exported_on"
        ),

    "source_graph_index":
        str(PHASE7_GRAPH_INDEX_FILE),

    "source_biological_association":
        str(D_BIO_ASSOCIATION_FILE),

    "split_unit":
        "molecular graph / drug_id",

    "split_strategy":
        "deterministic random molecule-level split",

    "random_seed":
        RANDOM_SEED,

    "configured_train_fraction":
        TRAIN_FRACTION,

    "configured_validation_fraction":
        VALIDATION_FRACTION,

    "configured_test_fraction":
        TEST_FRACTION,

    "fisher_reference_max_size":
        FISHER_REFERENCE_MAX_SIZE,

    "fisher_source":
        "training corpus only",

    "phase7_source_modified":
        False,

    "phase7_biological_flags_overwritten":
        False,

    "fisher_information_calculated":
        False,

    "model_training_performed":
        False,

    "statistics":
        stats
}


# ==============================================================================
# 33. SAVE SUMMARY JSON
# ==============================================================================

with open(
    PHASE8_SUMMARY_JSON_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        phase8_metadata,
        file,
        indent=4,
        ensure_ascii=False,
        default=str
    )


# ==============================================================================
# 34. SAVE SUMMARY CSV
# ==============================================================================

summary_rows = []


for section_name, section_value in phase8_metadata.items():

    if isinstance(
        section_value,
        dict
    ):

        for key, value in section_value.items():

            summary_rows.append(
                {
                    "metric":
                        f"{section_name}.{key}",

                    "value":
                        value
                }
            )

    else:

        summary_rows.append(
            {
                "metric":
                    section_name,

                "value":
                    section_value
            }
        )


phase8_summary_df = pd.DataFrame(
    summary_rows
)


phase8_summary_df.to_csv(
    PHASE8_SUMMARY_CSV_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 35. RESULTS
# ==============================================================================

print("\n" + "=" * 115)
print("PHASE 8 BASELINE CORPUS RESULTS")
print("=" * 115)


print(
    f"Total valid molecular graphs       : "
    f"{stats['total_graph_corpus']}"
)

print(
    f"Training graphs                    : "
    f"{stats['training_graphs']}"
)

print(
    f"Validation graphs                  : "
    f"{stats['validation_graphs']}"
)

print(
    f"Test graphs                        : "
    f"{stats['test_graphs']}"
)

print(
    f"Fisher reference graphs            : "
    f"{stats['fisher_reference_graphs']}"
)


print(
    f"\nActual Train Percentage            : "
    f"{stats['actual_train_percentage']:.2f}%"
)

print(
    f"Actual Validation Percentage       : "
    f"{stats['actual_validation_percentage']:.2f}%"
)

print(
    f"Actual Test Percentage             : "
    f"{stats['actual_test_percentage']:.2f}%"
)


# ==============================================================================
# 36. LEAKAGE RESULTS
# ==============================================================================

print("\n" + "=" * 115)
print("DATA-LEAKAGE VALIDATION")
print("=" * 115)


print(
    f"Train ↔ Validation overlap : "
    f"{stats['train_validation_overlap']}"
)

print(
    f"Train ↔ Test overlap       : "
    f"{stats['train_test_overlap']}"
)

print(
    f"Validation ↔ Test overlap  : "
    f"{stats['validation_test_overlap']}"
)

print(
    f"Fisher ↔ Validation overlap: "
    f"{stats['fisher_validation_overlap']}"
)

print(
    f"Fisher ↔ Test overlap      : "
    f"{stats['fisher_test_overlap']}"
)

print(
    f"Fisher subset of Train     : "
    f"{stats['fisher_subset_of_training']}"
)


# ==============================================================================
# 37. BIOLOGICAL VALIDATION
# ==============================================================================

print("\n" + "=" * 115)
print("PHASE-7 ↔ PHASE-6 BIOLOGICAL LINKAGE VALIDATION")
print("=" * 115)


print(
    f"Phase-7 biological flags match Phase 6       : "
    f"{bio_linkage_flag_matches}"
)

print(
    f"Phase-7 human-association flags match Phase 6: "
    f"{human_linkage_flag_matches}"
)


# ==============================================================================
# 38. BIOLOGICAL DISTRIBUTION
# ==============================================================================

print("\n" + "=" * 115)
print("CHEMICAL CORPUS ↔ BIOLOGICAL LINKAGE")
print("=" * 115)


print(
    f"Train graphs with biological association      : "
    f"{train_with_bio}"
)

print(
    f"Validation graphs with biological association : "
    f"{validation_with_bio}"
)

print(
    f"Test graphs with biological association       : "
    f"{test_with_bio}"
)

print(
    f"Fisher graphs with biological association     : "
    f"{fisher_with_bio}"
)


print(
    f"\nTrain graphs with human association           : "
    f"{train_with_human}"
)

print(
    f"Validation graphs with human association      : "
    f"{validation_with_human}"
)

print(
    f"Test graphs with human association            : "
    f"{test_with_human}"
)

print(
    f"Fisher graphs with human association          : "
    f"{fisher_with_human}"
)


# ==============================================================================
# 39. MANIFEST PREVIEW
# ==============================================================================

print("\n" + "=" * 115)
print("BASELINE SPLIT MANIFEST PREVIEW")
print("=" * 115)


preview_columns = [
    "graph_id",
    "drug_id",
    "drug_name",
    "split",
    "is_fisher_reference",
    "heavy_atom_count",
    "num_nodes",
    "num_bonds",
    "has_biological_association",
    "has_human_biological_association"
]


print(
    phase8_baseline_split_manifest[
        preview_columns
    ]
    .head(20)
    .to_string(index=False)
)


# ==============================================================================
# 40. COMPLETION CHECKS
# ==============================================================================

completion_checks = {
    "Phase-7 graph index loaded":
        len(D_baseline_graph) > 0,

    "All graph files readable":
        integrity_failure_count == 0,

    "All graph files passed integrity validation":
        bool(
            phase8_graph_integrity[
                "integrity_pass"
            ].all()
        ),

    "Graph IDs unique":
        duplicate_graph_id_count == 0,

    "Drug IDs unique":
        duplicate_drug_id_count == 0,

    "Phase-7 biological flags verified":
        bio_linkage_flag_matches,

    "Phase-7 human-association flags verified":
        human_linkage_flag_matches,

    "Training corpus created":
        len(D_train) > 0,

    "Validation corpus created":
        len(D_validation) > 0,

    "Test corpus created":
        len(D_test) > 0,

    "No train-validation overlap":
        len(train_validation_overlap) == 0,

    "No train-test overlap":
        len(train_test_overlap) == 0,

    "No validation-test overlap":
        len(validation_test_overlap) == 0,

    "Primary split covers complete graph corpus":
        complete_split_coverage,

    "Each graph belongs to one primary split":
        each_graph_exactly_once,

    "Split is reproducible":
        split_reproducible,

    "Fisher reference created":
        len(D_fisher_reference) > 0,

    "Fisher reference is training-only":
        fisher_subset_of_train,

    "No Fisher-validation overlap":
        len(fisher_validation_overlap) == 0,

    "No Fisher-test overlap":
        len(fisher_test_overlap) == 0,

    "Fisher selection reproducible":
        fisher_reproducible,

    "Original Phase-7 graph index preserved":
        original_phase7_graph_index_preserved,

    "Biological data remains separate":
        original_bio_preserved,

    "Split manifest saved":
        PHASE8_MANIFEST_FILE.exists(),

    "Training table saved":
        PHASE8_TRAIN_FILE.exists(),

    "Validation table saved":
        PHASE8_VALIDATION_FILE.exists(),

    "Test table saved":
        PHASE8_TEST_FILE.exists(),

    "Fisher table saved":
        PHASE8_FISHER_FILE.exists(),

    "Graph-integrity report saved":
        PHASE8_GRAPH_INTEGRITY_FILE.exists(),

    "Summary CSV saved":
        PHASE8_SUMMARY_CSV_FILE.exists(),

    "Summary JSON saved":
        PHASE8_SUMMARY_JSON_FILE.exists(),

    "Fisher information NOT calculated":
        True,

    "No model training performed":
        True
}


# ==============================================================================
# 41. DISPLAY COMPLETION CHECKS
# ==============================================================================

print("\n" + "=" * 115)
print("PHASE 8 COMPLETION VALIDATION")
print("=" * 115)


all_critical_checks_passed = True


for check_name, status in completion_checks.items():

    label = (
        "PASS"
        if status
        else "FAIL"
    )

    print(
        f"[{label:<4}] {check_name}"
    )

    if not status:
        all_critical_checks_passed = False


# ==============================================================================
# 42. UPDATE PROJECT CONFIG
# ==============================================================================

PHASE8_DATASETS = {
    "baseline_graph_count":
        len(D_baseline_graph),

    "train_csv":
        str(PHASE8_TRAIN_FILE),

    "validation_csv":
        str(PHASE8_VALIDATION_FILE),

    "test_csv":
        str(PHASE8_TEST_FILE),

    "fisher_reference_csv":
        str(PHASE8_FISHER_FILE),

    "split_manifest_csv":
        str(PHASE8_MANIFEST_FILE),

    "graph_integrity_csv":
        str(PHASE8_GRAPH_INTEGRITY_FILE),

    "summary_csv":
        str(PHASE8_SUMMARY_CSV_FILE),

    "summary_json":
        str(PHASE8_SUMMARY_JSON_FILE),

    "statistics":
        stats
}


DRUGBANK_CONFIG[
    "phase8_datasets"
] = PHASE8_DATASETS


# ==============================================================================
# 43. FINAL SUMMARY
# ==============================================================================

print("\n" + "=" * 115)
print("PHASE 8 SUMMARY")
print("=" * 115)


print(
    f"Baseline Graph Corpus          : "
    f"{len(D_baseline_graph)}"
)

print(
    f"Training Corpus                : "
    f"{len(D_train)}"
)

print(
    f"Validation Corpus              : "
    f"{len(D_validation)}"
)

print(
    f"Test Corpus                    : "
    f"{len(D_test)}"
)

print(
    f"Fisher Reference Corpus        : "
    f"{len(D_fisher_reference)}"
)

print(
    f"Random Seed                    : "
    f"{RANDOM_SEED}"
)

print(
    f"Split Strategy                 : "
    f"Random Molecule-Level"
)

print(
    f"Train/Validation/Test Leak     : False"
)

print(
    f"Fisher Leakage                 : False"
)

print(
    f"Fisher Source                  : Training Corpus Only"
)

print(
    f"Graph Integrity Failures       : "
    f"{integrity_failure_count}"
)

print(
    f"Phase-7 Bio Flags Verified     : "
    f"{bio_linkage_flag_matches}"
)

print(
    f"Phase-7 Human Flags Verified   : "
    f"{human_linkage_flag_matches}"
)

print(
    f"Original Phase-7 Data Kept     : "
    f"{original_phase7_graph_index_preserved}"
)

print(
    f"Biological Data Kept Separate  : "
    f"{original_bio_preserved}"
)

print(
    f"Fisher Information Computed    : False"
)

print(
    f"Model Training Performed       : False"
)


print("\nSaved Files:")

print(
    f"  Split Manifest     : "
    f"{PHASE8_MANIFEST_FILE}"
)

print(
    f"  Training Corpus    : "
    f"{PHASE8_TRAIN_FILE}"
)

print(
    f"  Validation Corpus  : "
    f"{PHASE8_VALIDATION_FILE}"
)

print(
    f"  Test Corpus        : "
    f"{PHASE8_TEST_FILE}"
)

print(
    f"  Fisher Reference   : "
    f"{PHASE8_FISHER_FILE}"
)

print(
    f"  Graph Integrity    : "
    f"{PHASE8_GRAPH_INTEGRITY_FILE}"
)

print(
    f"  QC Summary CSV     : "
    f"{PHASE8_SUMMARY_CSV_FILE}"
)

print(
    f"  QC Summary JSON    : "
    f"{PHASE8_SUMMARY_JSON_FILE}"
)


print("-" * 115)


if all_critical_checks_passed:

    print(
        "[STATUS] PHASE 8 COMPLETED SUCCESSFULLY."
    )

    print(
        "[READY]  Fixed baseline chemical knowledge corpus established."
    )

    print(
        "[READY]  Train, validation, test, and Fisher-reference "
        "subsets are available."
    )

    print(
        "[NEXT]   Phase 9 — Geometry-Aware Molecular Encoder"
    )

else:

    print(
        "[STATUS] PHASE 8 COMPLETED WITH ONE OR MORE "
        "FAILED VALIDATION CHECKS."
    )

    print(
        "[ACTION] Review the failed Phase-8 QC checks "
        "before proceeding to Phase 9."
    )


print("=" * 115)

PHASE 8 — BASELINE CHEMICAL CORPUS CONSTRUCTION

[DIR] Processed Directory : E:\MyPhD\My Patent-1\Implementation\data\processed
[CONFIG] Random Seed      : 42
[CONFIG] Train Fraction   : 0.80
[CONFIG] Validation       : 0.10
[CONFIG] Test             : 0.10
[CONFIG] Fisher Max Size  : 128

[PASS] Phase-7 graph index found.
[PASS] Phase-6 biological association dataset found.

BASELINE INPUT DATA
Phase-7 valid graph records : 767
Biological association rows : 5058

[PASS] Required Phase-7 graph-index columns verified.
[PASS] Graph IDs are unique.
[PASS] Drug IDs are unique.
[PASS] Phase-7 biological flags verified against Phase 6.
[PASS] Phase-7 human-association flags verified against Phase 6.

GRAPH FILE INTEGRITY VALIDATION
Graph records checked : 767
Integrity PASS        : 767
Integrity FAIL        : 0

[PASS] All Phase-7 graph files passed integrity validation.
[PASS] Original Phase-7 graph index remains unchanged.

PRIMARY SPLIT COUNTS
Total graphs : 767
Train        : 613
Valida

#### Phaes - 9 

# Phase 9 — Geometry-Aware Equivariant Molecular Encoder (Module 102)

**Objective:** Implement and validate the SE(3)-equivariant variational molecular encoder architecture (EGNN) that maps attributed 3D molecular graphs $G = (V, E, X, R)$ to continuous latent representations $z \in \mathcal{Z} \subset \mathbb{R}^{32}$.

* **Architectural & Symmetry Formulation:**
  * **Feature Decoupling:** Separates invariant atomic features ($X \in \mathbb{R}^{N \times 21}$) from centered Cartesian coordinates ($R \in \mathbb{R}^{N \times 3}$).
  * **Equivariant Message Passing:** Two EGNN layers using continuous squared Euclidean distances $d_{ij}^2 = \Vert{}r_i - r_j\Vert{}_2^2$, edge attributes $e_{ij} \in \mathbb{R}^{14}$, and relative-vector coordinate updates.
  * **Global Readout & Latent Heads:** Permutation-invariant mean pooling over atoms to extract $h_G$, followed by linear heads projecting to $\mu \in \mathbb{R}^{32}$ and $\log \sigma^2 \in \mathbb{R}^{32}$.
  * **Reparameterization Policy:** Stochastic sampling $z = \mu + \sigma \odot \epsilon$ during training; deterministic evaluation $z = \mu$ for inference.

* **Execution Deliverables:**
  * Custom disjoint union graph collator for variable-sized molecular graphs.
  * Training-only continuous feature standardization (means and standard deviations fitted strictly on $\mathcal{D}_{\text{train}}$).
  * Empirical verification of translation invariance, rotation invariance, coordinate equivariance, and permutation invariance ($\Delta < 10^{-6}$).
  * Verified parameter gradient-flow integrity check.
  * Initialized encoder checkpoint saved to `./data/processed/phase9_encoder_initial_state.pt`.
  * Encoder architecture metadata and normalization tables logged to `./data/processed/`.

In [9]:
# ==============================================================================
# PHASE 9 — GEOMETRY-AWARE EQUIVARIANT MOLECULAR ENCODER
# ==============================================================================
#
# PURPOSE
# -------
# Construct and computationally validate a lightweight EGNN-style
# variational molecular encoder.
#
# INPUT
# -----
# Phase 8:
#   phase8_train_graphs.csv
#   phase8_validation_graphs.csv
#   phase8_test_graphs.csv
#   phase8_fisher_reference.csv
#
# Phase 7:
#   data/graphs/*.pt
#   phase7_graph_feature_schema.json
#
# MODEL
# -----
#
# G = (V, E, X, R)
#        ↓
# Initial Node Projection
#        ↓
# EGNN Layer 1
#        ↓
# EGNN Layer 2
#        ↓
# Mean Graph Pooling
#        ↓
# Molecular embedding h_G
#        ↓
# μ and log σ²
#        ↓
# Reparameterization
#        ↓
# z ∈ R^32
#
# IMPORTANT
# ---------
# Phase 9 constructs and validates the encoder architecture.
#
# It DOES NOT:
#   - train the molecular model
#   - train a decoder
#   - compute Fisher information
#   - perform EWC
#   - generate molecules
#   - claim that the latent molecular manifold has been learned
#
# ==============================================================================


# ==============================================================================
# 1. IMPORTS
# ==============================================================================

from pathlib import Path
from datetime import datetime

import json
import math
import random
import copy

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


# ==============================================================================
# 2. REPRODUCIBILITY
# ==============================================================================

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)


# CPU baseline for reproducibility and current project configuration.

DEVICE = torch.device("cpu")

torch.set_num_threads(
    max(
        1,
        min(
            8,
            torch.get_num_threads()
        )
    )
)


# ==============================================================================
# 3. VERIFY REQUIRED PROJECT STATE
# ==============================================================================

required_variables = [
    "PROCESSED_DIR",
    "DRUGBANK_CONFIG"
]


missing_variables = [
    variable
    for variable in required_variables
    if variable not in globals()
]


if missing_variables:

    raise RuntimeError(
        "Earlier phases must be executed before Phase 9.\n"
        f"Missing variables: {missing_variables}"
    )


# ==============================================================================
# 4. PHASE-9 CONFIGURATION
# ==============================================================================

HIDDEN_DIM = 64

LATENT_DIM = 32

NUM_EGNN_LAYERS = 2

BATCH_SIZE = 16

NUM_WORKERS = 0

NORMALIZATION_EPS = 1e-8

EQUIVARIANCE_ATOL = 1e-5
EQUIVARIANCE_RTOL = 1e-5

COORDINATE_UPDATE_SCALE = 0.10


print("=" * 118)
print("PHASE 9 — GEOMETRY-AWARE EQUIVARIANT MOLECULAR ENCODER")
print("=" * 118)

print(f"\n[DEVICE] Device             : {DEVICE}")
print(f"[CONFIG] Random Seed       : {RANDOM_SEED}")
print(f"[CONFIG] Hidden Dimension  : {HIDDEN_DIM}")
print(f"[CONFIG] Latent Dimension  : {LATENT_DIM}")
print(f"[CONFIG] EGNN Layers       : {NUM_EGNN_LAYERS}")
print(f"[CONFIG] Batch Size        : {BATCH_SIZE}")
print(f"[CONFIG] Coordinate Scale  : {COORDINATE_UPDATE_SCALE}")


# ==============================================================================
# 5. INPUT FILES
# ==============================================================================

PHASE8_TRAIN_FILE = (
    Path(PROCESSED_DIR)
    / "phase8_train_graphs.csv"
)

PHASE8_VALIDATION_FILE = (
    Path(PROCESSED_DIR)
    / "phase8_validation_graphs.csv"
)

PHASE8_TEST_FILE = (
    Path(PROCESSED_DIR)
    / "phase8_test_graphs.csv"
)

PHASE8_FISHER_FILE = (
    Path(PROCESSED_DIR)
    / "phase8_fisher_reference.csv"
)

PHASE7_FEATURE_SCHEMA_FILE = (
    Path(PROCESSED_DIR)
    / "phase7_graph_feature_schema.json"
)


required_input_files = [
    PHASE8_TRAIN_FILE,
    PHASE8_VALIDATION_FILE,
    PHASE8_TEST_FILE,
    PHASE8_FISHER_FILE,
    PHASE7_FEATURE_SCHEMA_FILE
]


missing_files = [
    str(file_path)
    for file_path in required_input_files
    if not file_path.exists()
]


if missing_files:

    raise FileNotFoundError(
        "Required Phase-7/8 files are missing:\n"
        + "\n".join(missing_files)
    )


print("\n[PASS] Phase-8 training corpus found.")
print("[PASS] Phase-8 validation corpus found.")
print("[PASS] Phase-8 test corpus found.")
print("[PASS] Phase-8 Fisher-reference corpus found.")
print("[PASS] Phase-7 feature schema found.")


# ==============================================================================
# 6. LOAD PHASE-8 DATASETS
# ==============================================================================

D_train = pd.read_csv(
    PHASE8_TRAIN_FILE,
    dtype=str,
    keep_default_na=False
)

D_validation = pd.read_csv(
    PHASE8_VALIDATION_FILE,
    dtype=str,
    keep_default_na=False
)

D_test = pd.read_csv(
    PHASE8_TEST_FILE,
    dtype=str,
    keep_default_na=False
)

D_fisher_reference = pd.read_csv(
    PHASE8_FISHER_FILE,
    dtype=str,
    keep_default_na=False
)


# Preserve exact copies for final source-integrity testing.

D_train_original = D_train.copy(deep=True)

D_validation_original = D_validation.copy(deep=True)

D_test_original = D_test.copy(deep=True)

D_fisher_original = D_fisher_reference.copy(deep=True)


print("\n" + "=" * 118)
print("PHASE-8 INPUT CORPORA")
print("=" * 118)

print(f"Training graphs          : {len(D_train)}")
print(f"Validation graphs        : {len(D_validation)}")
print(f"Test graphs              : {len(D_test)}")
print(f"Fisher-reference graphs  : {len(D_fisher_reference)}")


# ==============================================================================
# 7. LOAD PHASE-7 FEATURE SCHEMA
# ==============================================================================

with open(
    PHASE7_FEATURE_SCHEMA_FILE,
    "r",
    encoding="utf-8"
) as file:

    phase7_feature_schema = json.load(file)


NODE_FEATURE_DIM = int(
    phase7_feature_schema[
        "node_feature_dimension"
    ]
)

EDGE_FEATURE_DIM = int(
    phase7_feature_schema[
        "edge_feature_dimension"
    ]
)

NODE_FEATURE_NAMES = list(
    phase7_feature_schema[
        "node_feature_names"
    ]
)

EDGE_FEATURE_NAMES = list(
    phase7_feature_schema[
        "edge_feature_names"
    ]
)


if len(NODE_FEATURE_NAMES) != NODE_FEATURE_DIM:

    raise RuntimeError(
        "Phase-7 node feature schema dimension mismatch."
    )


if len(EDGE_FEATURE_NAMES) != EDGE_FEATURE_DIM:

    raise RuntimeError(
        "Phase-7 edge feature schema dimension mismatch."
    )


print("\n" + "=" * 118)
print("PHASE-7 GRAPH FEATURE SCHEMA")
print("=" * 118)

print(
    f"Node Feature Dimension : "
    f"{NODE_FEATURE_DIM}"
)

print(
    f"Edge Feature Dimension : "
    f"{EDGE_FEATURE_DIM}"
)

print(
    f"Coordinate Policy      : "
    f"{phase7_feature_schema.get('coordinate_policy', 'UNKNOWN')}"
)

print(
    f"Edges Bidirectional    : "
    f"{phase7_feature_schema.get('edges_bidirectional', 'UNKNOWN')}"
)


# ==============================================================================
# 8. VERIFY EXPECTED PROJECT CONFIGURATION
# ==============================================================================

if NODE_FEATURE_DIM != 21:

    raise RuntimeError(
        f"Expected Phase-7 node dimension 21, "
        f"found {NODE_FEATURE_DIM}."
    )


if EDGE_FEATURE_DIM != 14:

    raise RuntimeError(
        f"Expected Phase-7 edge dimension 14, "
        f"found {EDGE_FEATURE_DIM}."
    )


if LATENT_DIM != 32:

    raise RuntimeError(
        "Project latent dimension must remain 32."
    )


if NUM_EGNN_LAYERS != 2:

    raise RuntimeError(
        "Baseline Phase-9 architecture requires two EGNN layers."
    )


print(
    "\n[PASS] Phase-7 feature dimensions match Phase-9 configuration."
)


# ==============================================================================
# 9. VERIFY REQUIRED MANIFEST COLUMNS
# ==============================================================================

required_manifest_columns = [
    "graph_id",
    "drug_id",
    "graph_file"
]


for dataset_name, dataframe in [
    ("TRAIN", D_train),
    ("VALIDATION", D_validation),
    ("TEST", D_test),
    ("FISHER", D_fisher_reference)
]:

    missing_columns = [
        column
        for column in required_manifest_columns
        if column not in dataframe.columns
    ]


    if missing_columns:

        raise RuntimeError(
            f"{dataset_name} corpus missing columns: "
            f"{missing_columns}"
        )


print(
    "[PASS] Required Phase-8 manifest columns verified."
)


# ==============================================================================
# 10. VERIFY PHASE-8 SPLIT RELATIONSHIPS
# ==============================================================================

train_ids = set(
    D_train[
        "graph_id"
    ].astype(str)
)

validation_ids = set(
    D_validation[
        "graph_id"
    ].astype(str)
)

test_ids = set(
    D_test[
        "graph_id"
    ].astype(str)
)

fisher_ids = set(
    D_fisher_reference[
        "graph_id"
    ].astype(str)
)


if train_ids & validation_ids:

    raise RuntimeError(
        "Train-validation overlap detected."
    )


if train_ids & test_ids:

    raise RuntimeError(
        "Train-test overlap detected."
    )


if validation_ids & test_ids:

    raise RuntimeError(
        "Validation-test overlap detected."
    )


if not fisher_ids.issubset(train_ids):

    raise RuntimeError(
        "Fisher-reference corpus is not a subset of training data."
    )


print(
    "[PASS] Phase-8 train/validation/test split remains leakage-free."
)

print(
    "[PASS] Fisher-reference corpus remains training-only."
)


# ==============================================================================
# 11. LOAD ONE GRAPH HELPER
# ==============================================================================

def load_graph_from_path(graph_path):

    graph = torch.load(
        Path(graph_path),
        map_location="cpu",
        weights_only=False
    )

    required_keys = {
        "graph_id",
        "drug_id",
        "x",
        "edge_index",
        "edge_attr",
        "pos"
    }

    if not required_keys.issubset(
        graph.keys()
    ):

        missing = (
            required_keys
            -
            set(
                graph.keys()
            )
        )

        raise RuntimeError(
            f"Graph missing required keys: {missing}"
        )

    return graph


# ==============================================================================
# 12. IDENTIFY NUMERICAL FEATURES FOR TRAINING-ONLY STANDARDIZATION
# ==============================================================================

# Only scalar numerical chemical attributes are standardized.
#
# Binary and one-hot variables are deliberately left unchanged.

NORMALIZED_NODE_FEATURE_NAMES = [
    "atomic_number",
    "degree",
    "formal_charge",
    "hydrogen_neighbor_count",
    "atomic_mass",
    "implicit_valence",
    "explicit_valence"
]


normalization_indices = []


for feature_name in NORMALIZED_NODE_FEATURE_NAMES:

    if feature_name not in NODE_FEATURE_NAMES:

        raise RuntimeError(
            f"Expected node feature missing from Phase-7 schema: "
            f"{feature_name}"
        )

    normalization_indices.append(
        NODE_FEATURE_NAMES.index(
            feature_name
        )
    )


NON_NORMALIZED_NODE_FEATURE_NAMES = [
    feature_name
    for feature_name in NODE_FEATURE_NAMES
    if feature_name not in NORMALIZED_NODE_FEATURE_NAMES
]


print("\n" + "=" * 118)
print("NODE FEATURE NORMALIZATION POLICY")
print("=" * 118)

print(
    "Training-only standardized features:"
)

for feature in NORMALIZED_NODE_FEATURE_NAMES:

    print(
        f"  - {feature}"
    )


print(
    "\nBinary/categorical features preserved without standardization:"
)

for feature in NON_NORMALIZED_NODE_FEATURE_NAMES:

    print(
        f"  - {feature}"
    )


# ==============================================================================
# 13. COMPUTE NORMALIZATION STATISTICS USING TRAINING DATA ONLY
# ==============================================================================

normalization_values = []


print("\n" + "=" * 118)
print("COMPUTING TRAINING-ONLY FEATURE NORMALIZATION")
print("=" * 118)


for record_number, (_, row) in enumerate(
    D_train.iterrows(),
    start=1
):

    graph = load_graph_from_path(
        row[
            "graph_file"
        ]
    )


    x = graph[
        "x"
    ].float()


    if x.ndim != 2:

        raise RuntimeError(
            f"Invalid x dimension in graph {row['graph_id']}."
        )


    if x.shape[1] != NODE_FEATURE_DIM:

        raise RuntimeError(
            f"Node feature mismatch in graph {row['graph_id']}."
        )


    normalization_values.append(
        x[
            :,
            normalization_indices
        ]
    )


    if (
        record_number == 1
        or record_number % 100 == 0
        or record_number == len(D_train)
    ):

        print(
            f"[PROGRESS] "
            f"{record_number}/{len(D_train)} "
            f"training graphs scanned..."
        )


all_training_numeric_features = torch.cat(
    normalization_values,
    dim=0
)


training_feature_mean = (
    all_training_numeric_features.mean(
        dim=0
    )
)


training_feature_std = (
    all_training_numeric_features.std(
        dim=0,
        unbiased=False
    )
)


# Protect against zero-variance features.

training_feature_std = torch.where(
    training_feature_std
    <
    NORMALIZATION_EPS,

    torch.ones_like(
        training_feature_std
    ),

    training_feature_std
)


print(
    "\n[PASS] Training-only normalization parameters computed."
)


# ==============================================================================
# 14. NORMALIZATION FUNCTION
# ==============================================================================

def normalize_node_features(x):

    x = x.clone().float()

    index_tensor = torch.tensor(
        normalization_indices,
        dtype=torch.long,
        device=x.device
    )


    mean = training_feature_mean.to(
        x.device
    )

    std = training_feature_std.to(
        x.device
    )


    x[
        :,
        index_tensor
    ] = (
        x[
            :,
            index_tensor
        ]
        -
        mean
    ) / std


    return x


# ==============================================================================
# 15. SAVE NORMALIZATION METADATA STRUCTURE
# ==============================================================================

normalization_metadata = {
    "fitted_on":
        "Phase-8 training corpus only",

    "training_graph_count":
        len(D_train),

    "node_feature_dimension":
        NODE_FEATURE_DIM,

    "standardized_features":
        NORMALIZED_NODE_FEATURE_NAMES,

    "standardized_feature_indices":
        normalization_indices,

    "non_standardized_features":
        NON_NORMALIZED_NODE_FEATURE_NAMES,

    "mean":
        {
            feature_name:
                float(
                    training_feature_mean[
                        local_index
                    ].item()
                )
            for local_index, feature_name
            in enumerate(
                NORMALIZED_NODE_FEATURE_NAMES
            )
        },

    "std":
        {
            feature_name:
                float(
                    training_feature_std[
                        local_index
                    ].item()
                )
            for local_index, feature_name
            in enumerate(
                NORMALIZED_NODE_FEATURE_NAMES
            )
        }
}


# ==============================================================================
# 16. GRAPH DATASET
# ==============================================================================

class MolecularGraphDataset(Dataset):

    def __init__(
        self,
        manifest,
        normalize=True
    ):

        self.manifest = (
            manifest
            .copy(
                deep=True
            )
            .reset_index(
                drop=True
            )
        )

        self.normalize = normalize


    def __len__(self):

        return len(
            self.manifest
        )


    def __getitem__(
        self,
        index
    ):

        row = self.manifest.iloc[
            index
        ]


        graph = load_graph_from_path(
            row[
                "graph_file"
            ]
        )


        x = graph[
            "x"
        ].float()


        edge_index = graph[
            "edge_index"
        ].long()


        edge_attr = graph[
            "edge_attr"
        ].float()


        pos = graph[
            "pos"
        ].float()


        if self.normalize:

            x = normalize_node_features(
                x
            )


        return {
            "graph_id":
                str(
                    graph[
                        "graph_id"
                    ]
                ),

            "drug_id":
                str(
                    graph[
                        "drug_id"
                    ]
                ),

            "x":
                x,

            "edge_index":
                edge_index,

            "edge_attr":
                edge_attr,

            "pos":
                pos
        }


# ==============================================================================
# 17. GRAPH BATCH COLLATION
# ==============================================================================

def molecular_graph_collate(
    graph_list
):

    x_list = []
    pos_list = []

    edge_index_list = []
    edge_attr_list = []

    batch_vector_list = []

    graph_ids = []
    drug_ids = []

    node_offset = 0


    for graph_number, graph in enumerate(
        graph_list
    ):

        x = graph[
            "x"
        ]

        pos = graph[
            "pos"
        ]

        edge_index = graph[
            "edge_index"
        ]

        edge_attr = graph[
            "edge_attr"
        ]


        num_nodes = int(
            x.shape[
                0
            ]
        )


        x_list.append(
            x
        )

        pos_list.append(
            pos
        )


        edge_index_list.append(
            edge_index
            +
            node_offset
        )


        edge_attr_list.append(
            edge_attr
        )


        batch_vector_list.append(
            torch.full(
                (
                    num_nodes,
                ),
                graph_number,
                dtype=torch.long
            )
        )


        graph_ids.append(
            graph[
                "graph_id"
            ]
        )

        drug_ids.append(
            graph[
                "drug_id"
            ]
        )


        node_offset += (
            num_nodes
        )


    batched_x = torch.cat(
        x_list,
        dim=0
    )


    batched_pos = torch.cat(
        pos_list,
        dim=0
    )


    batched_edge_index = torch.cat(
        edge_index_list,
        dim=1
    )


    batched_edge_attr = torch.cat(
        edge_attr_list,
        dim=0
    )


    batch_vector = torch.cat(
        batch_vector_list,
        dim=0
    )


    return {
        "x":
            batched_x,

        "pos":
            batched_pos,

        "edge_index":
            batched_edge_index,

        "edge_attr":
            batched_edge_attr,

        "batch":
            batch_vector,

        "graph_ids":
            graph_ids,

        "drug_ids":
            drug_ids,

        "num_graphs":
            len(
                graph_list
            )
    }


# ==============================================================================
# 18. CREATE DATASETS
# ==============================================================================

train_dataset = MolecularGraphDataset(
    D_train,
    normalize=True
)

validation_dataset = MolecularGraphDataset(
    D_validation,
    normalize=True
)

test_dataset = MolecularGraphDataset(
    D_test,
    normalize=True
)

fisher_dataset = MolecularGraphDataset(
    D_fisher_reference,
    normalize=True
)


# ==============================================================================
# 19. CREATE DATA LOADERS
# ==============================================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=molecular_graph_collate
)


validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=molecular_graph_collate
)


test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=molecular_graph_collate
)


fisher_loader = DataLoader(
    fisher_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=molecular_graph_collate
)


print(
    "\n[PASS] Molecular graph datasets created."
)

print(
    "[PASS] Mini-batch graph loaders created."
)


# ==============================================================================
# 20. HELPER — MEAN AGGREGATION
# ==============================================================================

def scatter_mean(
    values,
    index,
    dim_size
):

    output = torch.zeros(
        (
            dim_size,
            values.shape[
                1
            ]
        ),
        dtype=values.dtype,
        device=values.device
    )


    counts = torch.zeros(
        (
            dim_size,
            1
        ),
        dtype=values.dtype,
        device=values.device
    )


    output.index_add_(
        0,
        index,
        values
    )


    ones = torch.ones(
        (
            values.shape[
                0
            ],
            1
        ),
        dtype=values.dtype,
        device=values.device
    )


    counts.index_add_(
        0,
        index,
        ones
    )


    counts = counts.clamp(
        min=1.0
    )


    return (
        output
        /
        counts
    )


# ==============================================================================
# 21. EGNN-STYLE MESSAGE-PASSING LAYER
# ==============================================================================

class EGNNLayer(nn.Module):

    def __init__(
        self,
        hidden_dim,
        edge_dim,
        coordinate_update_scale=0.10
    ):

        super().__init__()


        self.hidden_dim = hidden_dim

        self.edge_dim = edge_dim

        self.coordinate_update_scale = (
            coordinate_update_scale
        )


        # ----------------------------------------------------------------------
        # Edge / message function
        #
        # Input:
        #   h_i
        #   h_j
        #   squared distance
        #   edge attributes
        # ----------------------------------------------------------------------

        edge_input_dim = (
            hidden_dim
            +
            hidden_dim
            +
            1
            +
            edge_dim
        )


        self.edge_mlp = nn.Sequential(
            nn.Linear(
                edge_input_dim,
                hidden_dim
            ),

            nn.SiLU(),

            nn.Linear(
                hidden_dim,
                hidden_dim
            ),

            nn.SiLU()
        )


        # ----------------------------------------------------------------------
        # Coordinate scalar
        #
        # Produces one scalar for every directed edge.
        # ----------------------------------------------------------------------

        self.coordinate_mlp = nn.Sequential(
            nn.Linear(
                hidden_dim,
                hidden_dim
            ),

            nn.SiLU(),

            nn.Linear(
                hidden_dim,
                1
            ),

            nn.Tanh()
        )


        # ----------------------------------------------------------------------
        # Node update
        # ----------------------------------------------------------------------

        self.node_mlp = nn.Sequential(
            nn.Linear(
                hidden_dim
                +
                hidden_dim,
                hidden_dim
            ),

            nn.SiLU(),

            nn.Linear(
                hidden_dim,
                hidden_dim
            )
        )


        self.node_norm = nn.LayerNorm(
            hidden_dim
        )


    def forward(
        self,
        h,
        pos,
        edge_index,
        edge_attr
    ):

        source = edge_index[
            0
        ]

        target = edge_index[
            1
        ]


        # ----------------------------------------------------------------------
        # Relative geometry
        # ----------------------------------------------------------------------

        relative = (
            pos[
                source
            ]
            -
            pos[
                target
            ]
        )


        squared_distance = (
            relative
            *
            relative
        ).sum(
            dim=1,
            keepdim=True
        )


        # ----------------------------------------------------------------------
        # Edge message
        # ----------------------------------------------------------------------

        edge_input = torch.cat(
            [
                h[
                    source
                ],

                h[
                    target
                ],

                squared_distance,

                edge_attr
            ],
            dim=1
        )


        messages = self.edge_mlp(
            edge_input
        )


        # ----------------------------------------------------------------------
        # Equivariant coordinate update
        # ----------------------------------------------------------------------

        coordinate_weight = (
            self.coordinate_mlp(
                messages
            )
            *
            self.coordinate_update_scale
        )


        coordinate_messages = (
            relative
            *
            coordinate_weight
        )


        coordinate_aggregate = scatter_mean(
            coordinate_messages,
            source,
            h.shape[
                0
            ]
        )


        new_pos = (
            pos
            +
            coordinate_aggregate
        )


        # ----------------------------------------------------------------------
        # Node message aggregation
        # ----------------------------------------------------------------------

        aggregated_messages = scatter_mean(
            messages,
            source,
            h.shape[
                0
            ]
        )


        node_input = torch.cat(
            [
                h,
                aggregated_messages
            ],
            dim=1
        )


        node_delta = self.node_mlp(
            node_input
        )


        new_h = self.node_norm(
            h
            +
            node_delta
        )


        return (
            new_h,
            new_pos
        )


# ==============================================================================
# 22. GEOMETRY-AWARE VARIATIONAL MOLECULAR ENCODER
# ==============================================================================

class GeometryAwareVariationalEncoder(nn.Module):

    def __init__(
        self,
        node_feature_dim,
        edge_feature_dim,
        hidden_dim,
        latent_dim,
        num_layers=2
    ):

        super().__init__()


        self.node_feature_dim = (
            node_feature_dim
        )

        self.edge_feature_dim = (
            edge_feature_dim
        )

        self.hidden_dim = (
            hidden_dim
        )

        self.latent_dim = (
            latent_dim
        )

        self.num_layers = (
            num_layers
        )


        # ----------------------------------------------------------------------
        # Initial node projection
        # ----------------------------------------------------------------------

        self.node_embedding = nn.Sequential(
            nn.Linear(
                node_feature_dim,
                hidden_dim
            ),

            nn.SiLU(),

            nn.LayerNorm(
                hidden_dim
            )
        )


        # ----------------------------------------------------------------------
        # EGNN-style layers
        # ----------------------------------------------------------------------

        self.egnn_layers = nn.ModuleList(
            [
                EGNNLayer(
                    hidden_dim=hidden_dim,
                    edge_dim=edge_feature_dim,
                    coordinate_update_scale=COORDINATE_UPDATE_SCALE
                )
                for _ in range(
                    num_layers
                )
            ]
        )


        # ----------------------------------------------------------------------
        # Graph embedding refinement
        # ----------------------------------------------------------------------

        self.graph_mlp = nn.Sequential(
            nn.Linear(
                hidden_dim,
                hidden_dim
            ),

            nn.SiLU(),

            nn.LayerNorm(
                hidden_dim
            )
        )


        # ----------------------------------------------------------------------
        # Variational heads
        # ----------------------------------------------------------------------

        self.mu_head = nn.Linear(
            hidden_dim,
            latent_dim
        )


        self.logvar_head = nn.Linear(
            hidden_dim,
            latent_dim
        )


    def graph_mean_pool(
        self,
        h,
        batch
    ):

        number_of_graphs = int(
            batch.max().item()
        ) + 1


        return scatter_mean(
            h,
            batch,
            number_of_graphs
        )


    def reparameterize(
        self,
        mu,
        logvar,
        deterministic=False
    ):

        if deterministic:

            return mu


        # Protect initial numerical stability while preserving gradients.

        stable_logvar = torch.clamp(
            logvar,
            min=-20.0,
            max=20.0
        )


        std = torch.exp(
            0.5
            *
            stable_logvar
        )


        epsilon = torch.randn_like(
            std
        )


        return (
            mu
            +
            std
            *
            epsilon
        )


    def forward(
        self,
        x,
        pos,
        edge_index,
        edge_attr,
        batch,
        deterministic=False,
        return_diagnostics=False
    ):

        # ----------------------------------------------------------------------
        # Initial atom embedding
        # ----------------------------------------------------------------------

        h = self.node_embedding(
            x
        )


        updated_pos = pos


        # ----------------------------------------------------------------------
        # Equivariant message passing
        # ----------------------------------------------------------------------

        for layer in self.egnn_layers:

            h, updated_pos = layer(
                h=h,
                pos=updated_pos,
                edge_index=edge_index,
                edge_attr=edge_attr
            )


        # ----------------------------------------------------------------------
        # Permutation-invariant graph pooling
        # ----------------------------------------------------------------------

        graph_embedding = self.graph_mean_pool(
            h,
            batch
        )


        graph_embedding = self.graph_mlp(
            graph_embedding
        )


        # ----------------------------------------------------------------------
        # Latent parameterization
        # ----------------------------------------------------------------------

        mu = self.mu_head(
            graph_embedding
        )


        logvar = self.logvar_head(
            graph_embedding
        )


        z = self.reparameterize(
            mu=mu,
            logvar=logvar,
            deterministic=deterministic
        )


        output = {
            "mu":
                mu,

            "logvar":
                logvar,

            "z":
                z
        }


        if return_diagnostics:

            output[
                "graph_embedding"
            ] = graph_embedding

            output[
                "final_node_embeddings"
            ] = h

            output[
                "updated_coordinates"
            ] = updated_pos


        return output


# ==============================================================================
# 23. INITIALIZE ENCODER
# ==============================================================================

encoder = GeometryAwareVariationalEncoder(
    node_feature_dim=NODE_FEATURE_DIM,
    edge_feature_dim=EDGE_FEATURE_DIM,
    hidden_dim=HIDDEN_DIM,
    latent_dim=LATENT_DIM,
    num_layers=NUM_EGNN_LAYERS
).to(
    DEVICE
)


print(
    "\n[PASS] Geometry-aware variational encoder initialized."
)


# ==============================================================================
# 24. PARAMETER COUNT
# ==============================================================================

total_parameters = sum(
    parameter.numel()
    for parameter
    in encoder.parameters()
)


trainable_parameters = sum(
    parameter.numel()
    for parameter
    in encoder.parameters()
    if parameter.requires_grad
)


print("\n" + "=" * 118)
print("MODEL PARAMETER REPORT")
print("=" * 118)

print(
    f"Total Parameters     : "
    f"{total_parameters:,}"
)

print(
    f"Trainable Parameters : "
    f"{trainable_parameters:,}"
)

print(
    f"Node Feature Dim     : "
    f"{NODE_FEATURE_DIM}"
)

print(
    f"Edge Feature Dim     : "
    f"{EDGE_FEATURE_DIM}"
)

print(
    f"Hidden Dim           : "
    f"{HIDDEN_DIM}"
)

print(
    f"Latent Dim           : "
    f"{LATENT_DIM}"
)

print(
    f"EGNN Layers          : "
    f"{NUM_EGNN_LAYERS}"
)


# ==============================================================================
# 25. LOAD ONE TRAINING MINI-BATCH
# ==============================================================================

sample_batch = next(
    iter(
        train_loader
    )
)


sample_batch = {
    key:
        (
            value.to(
                DEVICE
            )
            if torch.is_tensor(
                value
            )
            else value
        )

    for key, value
    in sample_batch.items()
}


print("\n" + "=" * 118)
print("MINI-BATCH STRUCTURE")
print("=" * 118)

print(
    f"Graphs in batch       : "
    f"{sample_batch['num_graphs']}"
)

print(
    f"Total nodes           : "
    f"{sample_batch['x'].shape[0]}"
)

print(
    f"Node feature shape    : "
    f"{tuple(sample_batch['x'].shape)}"
)

print(
    f"Coordinate shape      : "
    f"{tuple(sample_batch['pos'].shape)}"
)

print(
    f"Edge index shape      : "
    f"{tuple(sample_batch['edge_index'].shape)}"
)

print(
    f"Edge attribute shape  : "
    f"{tuple(sample_batch['edge_attr'].shape)}"
)

print(
    f"Batch-vector shape    : "
    f"{tuple(sample_batch['batch'].shape)}"
)


# ==============================================================================
# 26. FORWARD-PASS SANITY TEST
# ==============================================================================

encoder.eval()


with torch.no_grad():

    deterministic_output = encoder(
        x=sample_batch[
            "x"
        ],

        pos=sample_batch[
            "pos"
        ],

        edge_index=sample_batch[
            "edge_index"
        ],

        edge_attr=sample_batch[
            "edge_attr"
        ],

        batch=sample_batch[
            "batch"
        ],

        deterministic=True,

        return_diagnostics=True
    )


mu = deterministic_output[
    "mu"
]

logvar = deterministic_output[
    "logvar"
]

z = deterministic_output[
    "z"
]

graph_embedding = deterministic_output[
    "graph_embedding"
]

final_node_embeddings = deterministic_output[
    "final_node_embeddings"
]

updated_coordinates = deterministic_output[
    "updated_coordinates"
]


expected_latent_shape = (
    sample_batch[
        "num_graphs"
    ],
    LATENT_DIM
)


mu_shape_valid = (
    tuple(
        mu.shape
    )
    ==
    expected_latent_shape
)


logvar_shape_valid = (
    tuple(
        logvar.shape
    )
    ==
    expected_latent_shape
)


z_shape_valid = (
    tuple(
        z.shape
    )
    ==
    expected_latent_shape
)


deterministic_z_equals_mu = torch.allclose(
    z,
    mu,
    atol=1e-7,
    rtol=1e-7
)


all_forward_outputs_finite = all(
    [
        bool(
            torch.isfinite(
                mu
            ).all()
        ),

        bool(
            torch.isfinite(
                logvar
            ).all()
        ),

        bool(
            torch.isfinite(
                z
            ).all()
        ),

        bool(
            torch.isfinite(
                graph_embedding
            ).all()
        ),

        bool(
            torch.isfinite(
                final_node_embeddings
            ).all()
        ),

        bool(
            torch.isfinite(
                updated_coordinates
            ).all()
        )
    ]
)


print("\n" + "=" * 118)
print("FORWARD-PASS VALIDATION")
print("=" * 118)

print(
    f"mu shape                  : "
    f"{tuple(mu.shape)}"
)

print(
    f"logvar shape              : "
    f"{tuple(logvar.shape)}"
)

print(
    f"z shape                   : "
    f"{tuple(z.shape)}"
)

print(
    f"Expected latent shape     : "
    f"{expected_latent_shape}"
)

print(
    f"mu shape valid            : "
    f"{mu_shape_valid}"
)

print(
    f"logvar shape valid        : "
    f"{logvar_shape_valid}"
)

print(
    f"z shape valid             : "
    f"{z_shape_valid}"
)

print(
    f"Deterministic z == mu     : "
    f"{deterministic_z_equals_mu}"
)

print(
    f"All forward outputs finite: "
    f"{all_forward_outputs_finite}"
)


# ==============================================================================
# 27. STOCHASTIC REPARAMETERIZATION SANITY TEST
# ==============================================================================

encoder.eval()


with torch.no_grad():

    stochastic_output = encoder(
        x=sample_batch[
            "x"
        ],

        pos=sample_batch[
            "pos"
        ],

        edge_index=sample_batch[
            "edge_index"
        ],

        edge_attr=sample_batch[
            "edge_attr"
        ],

        batch=sample_batch[
            "batch"
        ],

        deterministic=False
    )


stochastic_latent_finite = bool(
    torch.isfinite(
        stochastic_output[
            "z"
        ]
    ).all()
)


print(
    f"Stochastic latent finite   : "
    f"{stochastic_latent_finite}"
)


# ==============================================================================
# 28. HELPER — SINGLE-GRAPH BATCH
# ==============================================================================

single_graph = train_dataset[
    0
]


single_batch = molecular_graph_collate(
    [
        single_graph
    ]
)


single_batch = {
    key:
        (
            value.to(
                DEVICE
            )
            if torch.is_tensor(
                value
            )
            else value
        )

    for key, value
    in single_batch.items()
}


# ==============================================================================
# 29. TRANSLATION-INVARIANCE TEST
# ==============================================================================

translation_vector = torch.tensor(
    [
        4.25,
        -2.75,
        1.50
    ],
    dtype=torch.float32,
    device=DEVICE
)


translated_pos = (
    single_batch[
        "pos"
    ]
    +
    translation_vector
)


encoder.eval()


with torch.no_grad():

    original_translation_test_output = encoder(
        x=single_batch[
            "x"
        ],

        pos=single_batch[
            "pos"
        ],

        edge_index=single_batch[
            "edge_index"
        ],

        edge_attr=single_batch[
            "edge_attr"
        ],

        batch=single_batch[
            "batch"
        ],

        deterministic=True,

        return_diagnostics=True
    )


    translated_output = encoder(
        x=single_batch[
            "x"
        ],

        pos=translated_pos,

        edge_index=single_batch[
            "edge_index"
        ],

        edge_attr=single_batch[
            "edge_attr"
        ],

        batch=single_batch[
            "batch"
        ],

        deterministic=True,

        return_diagnostics=True
    )


translation_mu_max_abs_difference = float(
    torch.max(
        torch.abs(
            original_translation_test_output[
                "mu"
            ]
            -
            translated_output[
                "mu"
            ]
        )
    ).item()
)


translation_invariance_pass = torch.allclose(
    original_translation_test_output[
        "mu"
    ],
    translated_output[
        "mu"
    ],
    atol=EQUIVARIANCE_ATOL,
    rtol=EQUIVARIANCE_RTOL
)


# ==============================================================================
# 30. ROTATION-INVARIANCE TEST
# ==============================================================================

# Fixed proper 3-D rotation:
# 90 degree rotation around Z axis.

rotation_matrix = torch.tensor(
    [
        [0.0, -1.0, 0.0],
        [1.0,  0.0, 0.0],
        [0.0,  0.0, 1.0]
    ],
    dtype=torch.float32,
    device=DEVICE
)


# Row-vector convention:
# R_rotated = R @ Q^T

rotated_pos = (
    single_batch[
        "pos"
    ]
    @
    rotation_matrix.T
)


with torch.no_grad():

    rotated_output = encoder(
        x=single_batch[
            "x"
        ],

        pos=rotated_pos,

        edge_index=single_batch[
            "edge_index"
        ],

        edge_attr=single_batch[
            "edge_attr"
        ],

        batch=single_batch[
            "batch"
        ],

        deterministic=True,

        return_diagnostics=True
    )


rotation_mu_max_abs_difference = float(
    torch.max(
        torch.abs(
            original_translation_test_output[
                "mu"
            ]
            -
            rotated_output[
                "mu"
            ]
        )
    ).item()
)


rotation_invariance_pass = torch.allclose(
    original_translation_test_output[
        "mu"
    ],
    rotated_output[
        "mu"
    ],
    atol=EQUIVARIANCE_ATOL,
    rtol=EQUIVARIANCE_RTOL
)


# ==============================================================================
# 31. INTERMEDIATE COORDINATE EQUIVARIANCE TEST
# ==============================================================================

original_updated_coordinates = (
    original_translation_test_output[
        "updated_coordinates"
    ]
)


translated_updated_coordinates = (
    translated_output[
        "updated_coordinates"
    ]
)


rotated_updated_coordinates = (
    rotated_output[
        "updated_coordinates"
    ]
)


expected_translated_coordinates = (
    original_updated_coordinates
    +
    translation_vector
)


expected_rotated_coordinates = (
    original_updated_coordinates
    @
    rotation_matrix.T
)


translation_coordinate_equivariance_pass = torch.allclose(
    translated_updated_coordinates,
    expected_translated_coordinates,
    atol=EQUIVARIANCE_ATOL,
    rtol=EQUIVARIANCE_RTOL
)


rotation_coordinate_equivariance_pass = torch.allclose(
    rotated_updated_coordinates,
    expected_rotated_coordinates,
    atol=EQUIVARIANCE_ATOL,
    rtol=EQUIVARIANCE_RTOL
)


translation_coordinate_max_abs_difference = float(
    torch.max(
        torch.abs(
            translated_updated_coordinates
            -
            expected_translated_coordinates
        )
    ).item()
)


rotation_coordinate_max_abs_difference = float(
    torch.max(
        torch.abs(
            rotated_updated_coordinates
            -
            expected_rotated_coordinates
        )
    ).item()
)


# ==============================================================================
# 32. PERMUTATION-INVARIANCE TEST
# ==============================================================================

def permute_graph(
    graph
):

    x = graph[
        "x"
    ]

    pos = graph[
        "pos"
    ]

    edge_index = graph[
        "edge_index"
    ]

    edge_attr = graph[
        "edge_attr"
    ]


    num_nodes = x.shape[
        0
    ]


    generator = torch.Generator()

    generator.manual_seed(
        RANDOM_SEED
    )


    permutation = torch.randperm(
        num_nodes,
        generator=generator
    )


    inverse_permutation = torch.empty_like(
        permutation
    )


    inverse_permutation[
        permutation
    ] = torch.arange(
        num_nodes
    )


    permuted_x = x[
        permutation
    ]


    permuted_pos = pos[
        permutation
    ]


    permuted_edge_index = inverse_permutation[
        edge_index
    ]


    return {
        "graph_id":
            graph[
                "graph_id"
            ],

        "drug_id":
            graph[
                "drug_id"
            ],

        "x":
            permuted_x,

        "pos":
            permuted_pos,

        "edge_index":
            permuted_edge_index,

        "edge_attr":
            edge_attr.clone()
    }


permuted_graph = permute_graph(
    single_graph
)


permuted_batch = molecular_graph_collate(
    [
        permuted_graph
    ]
)


permuted_batch = {
    key:
        (
            value.to(
                DEVICE
            )
            if torch.is_tensor(
                value
            )
            else value
        )

    for key, value
    in permuted_batch.items()
}


with torch.no_grad():

    permuted_output = encoder(
        x=permuted_batch[
            "x"
        ],

        pos=permuted_batch[
            "pos"
        ],

        edge_index=permuted_batch[
            "edge_index"
        ],

        edge_attr=permuted_batch[
            "edge_attr"
        ],

        batch=permuted_batch[
            "batch"
        ],

        deterministic=True
    )


permutation_mu_max_abs_difference = float(
    torch.max(
        torch.abs(
            original_translation_test_output[
                "mu"
            ]
            -
            permuted_output[
                "mu"
            ]
        )
    ).item()
)


permutation_invariance_pass = torch.allclose(
    original_translation_test_output[
        "mu"
    ],
    permuted_output[
        "mu"
    ],
    atol=EQUIVARIANCE_ATOL,
    rtol=EQUIVARIANCE_RTOL
)


# ==============================================================================
# 33. DISPLAY GEOMETRIC TEST RESULTS
# ==============================================================================

print("\n" + "=" * 118)
print("GEOMETRIC CONSISTENCY VALIDATION")
print("=" * 118)

print(
    f"Translation-invariant μ                  : "
    f"{translation_invariance_pass}"
)

print(
    f"Translation μ max abs difference         : "
    f"{translation_mu_max_abs_difference:.10e}"
)

print(
    f"Rotation-invariant μ                     : "
    f"{rotation_invariance_pass}"
)

print(
    f"Rotation μ max abs difference            : "
    f"{rotation_mu_max_abs_difference:.10e}"
)

print(
    f"Translation-equivariant coordinates      : "
    f"{translation_coordinate_equivariance_pass}"
)

print(
    f"Translation coordinate max abs difference: "
    f"{translation_coordinate_max_abs_difference:.10e}"
)

print(
    f"Rotation-equivariant coordinates         : "
    f"{rotation_coordinate_equivariance_pass}"
)

print(
    f"Rotation coordinate max abs difference   : "
    f"{rotation_coordinate_max_abs_difference:.10e}"
)

print(
    f"Permutation-invariant μ                  : "
    f"{permutation_invariance_pass}"
)

print(
    f"Permutation μ max abs difference         : "
    f"{permutation_mu_max_abs_difference:.10e}"
)


# ==============================================================================
# 34. GRADIENT-FLOW SANITY TEST
# ==============================================================================

encoder.train()

encoder.zero_grad(
    set_to_none=True
)


gradient_output = encoder(
    x=sample_batch[
        "x"
    ],

    pos=sample_batch[
        "pos"
    ],

    edge_index=sample_batch[
        "edge_index"
    ],

    edge_attr=sample_batch[
        "edge_attr"
    ],

    batch=sample_batch[
        "batch"
    ],

    deterministic=False,

    return_diagnostics=True
)


# Synthetic scalar objective.
#
# This is ONLY a gradient-flow test.
# It is NOT model training.

synthetic_loss = (
    gradient_output[
        "mu"
    ].pow(
        2
    ).mean()

    +

    gradient_output[
        "logvar"
    ].pow(
        2
    ).mean()

    +

    0.01
    *
    gradient_output[
        "z"
    ].pow(
        2
    ).mean()

    +

    0.0001
    *
    gradient_output[
        "updated_coordinates"
    ].pow(
        2
    ).mean()
)


synthetic_loss.backward()


gradient_report = {}


for parameter_name, parameter in encoder.named_parameters():

    if parameter.requires_grad:

        if parameter.grad is None:

            gradient_report[
                parameter_name
            ] = False

        else:

            gradient_report[
                parameter_name
            ] = bool(
                torch.isfinite(
                    parameter.grad
                ).all()
            )


parameters_with_gradients = sum(
    bool(
        parameter.grad is not None
    )
    for parameter
    in encoder.parameters()
    if parameter.requires_grad
)


total_trainable_parameter_tensors = sum(
    1
    for parameter
    in encoder.parameters()
    if parameter.requires_grad
)


all_existing_gradients_finite = all(
    [
        bool(
            torch.isfinite(
                parameter.grad
            ).all()
        )
        for parameter
        in encoder.parameters()
        if parameter.requires_grad
        and parameter.grad is not None
    ]
)


gradient_flow_pass = (
    parameters_with_gradients
    ==
    total_trainable_parameter_tensors
    and
    all_existing_gradients_finite
)


print("\n" + "=" * 118)
print("GRADIENT-FLOW SANITY TEST")
print("=" * 118)

print(
    f"Synthetic Loss                : "
    f"{float(synthetic_loss.detach().item()):.8f}"
)

print(
    f"Trainable Parameter Tensors   : "
    f"{total_trainable_parameter_tensors}"
)

print(
    f"Parameters Receiving Gradient : "
    f"{parameters_with_gradients}"
)

print(
    f"All Existing Gradients Finite : "
    f"{all_existing_gradients_finite}"
)

print(
    f"Gradient-Flow Test            : "
    f"{gradient_flow_pass}"
)


# Clear gradients.
# No optimizer step is performed.

encoder.zero_grad(
    set_to_none=True
)

encoder.eval()


# ==============================================================================
# 35. CHECK REPRESENTATIVE GRAPH LOADING FROM EACH SPLIT
# ==============================================================================

def validate_dataset_sample(
    dataset
):

    if len(
        dataset
    ) == 0:

        return False


    sample = dataset[
        0
    ]


    required = [
        "x",
        "pos",
        "edge_index",
        "edge_attr"
    ]


    if not all(
        key in sample
        for key in required
    ):

        return False


    if sample[
        "x"
    ].shape[
        1
    ] != NODE_FEATURE_DIM:

        return False


    if sample[
        "edge_attr"
    ].shape[
        1
    ] != EDGE_FEATURE_DIM:

        return False


    if sample[
        "x"
    ].shape[
        0
    ] != sample[
        "pos"
    ].shape[
        0
    ]:

        return False


    return True


train_graph_loading_pass = (
    validate_dataset_sample(
        train_dataset
    )
)

validation_graph_loading_pass = (
    validate_dataset_sample(
        validation_dataset
    )
)

test_graph_loading_pass = (
    validate_dataset_sample(
        test_dataset
    )
)

fisher_graph_loading_pass = (
    validate_dataset_sample(
        fisher_dataset
    )
)


# ==============================================================================
# 36. SOURCE PRESERVATION CHECK
# ==============================================================================

train_source_preserved = (
    D_train.equals(
        D_train_original
    )
)


validation_source_preserved = (
    D_validation.equals(
        D_validation_original
    )
)


test_source_preserved = (
    D_test.equals(
        D_test_original
    )
)


fisher_source_preserved = (
    D_fisher_reference.equals(
        D_fisher_original
    )
)


all_phase8_sources_preserved = all(
    [
        train_source_preserved,
        validation_source_preserved,
        test_source_preserved,
        fisher_source_preserved
    ]
)


# ==============================================================================
# 37. OUTPUT FILES
# ==============================================================================

PHASE9_CONFIG_FILE = (
    Path(PROCESSED_DIR)
    / "phase9_encoder_config.json"
)

PHASE9_NORMALIZATION_FILE = (
    Path(PROCESSED_DIR)
    / "phase9_feature_normalization.json"
)

PHASE9_INITIAL_STATE_FILE = (
    Path(PROCESSED_DIR)
    / "phase9_encoder_initial_state.pt"
)

PHASE9_QC_CSV_FILE = (
    Path(PROCESSED_DIR)
    / "phase9_encoder_qc_summary.csv"
)

PHASE9_QC_JSON_FILE = (
    Path(PROCESSED_DIR)
    / "phase9_encoder_qc_summary.json"
)

PHASE9_EQUIVARIANCE_FILE = (
    Path(PROCESSED_DIR)
    / "phase9_equivariance_test.csv"
)


# ==============================================================================
# 38. SAVE FEATURE NORMALIZATION
# ==============================================================================

with open(
    PHASE9_NORMALIZATION_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        normalization_metadata,
        file,
        indent=4,
        ensure_ascii=False
    )


# ==============================================================================
# 39. SAVE MODEL CONFIGURATION
# ==============================================================================

encoder_config = {
    "phase":
        "Phase 9 — Geometry-Aware Equivariant Molecular Encoder",

    "model_status":
        "UNTRAINED_INITIALIZED_ENCODER",

    "random_seed":
        RANDOM_SEED,

    "device":
        str(
            DEVICE
        ),

    "node_feature_dim":
        NODE_FEATURE_DIM,

    "edge_feature_dim":
        EDGE_FEATURE_DIM,

    "hidden_dim":
        HIDDEN_DIM,

    "latent_dim":
        LATENT_DIM,

    "num_egnn_layers":
        NUM_EGNN_LAYERS,

    "pooling":
        "mean",

    "coordinate_update":
        "EGNN-style relative-vector scalar update",

    "coordinate_update_scale":
        COORDINATE_UPDATE_SCALE,

    "message_geometry":
        "squared pairwise distance",

    "edge_attributes_used":
        True,

    "coordinates_in_scalar_node_features":
        False,

    "deterministic_evaluation_policy":
        "z = mu",

    "stochastic_training_policy":
        "z = mu + exp(0.5 * logvar) * epsilon",

    "logvar_sampling_clamp":
        [
            -20.0,
            20.0
        ],

    "total_parameters":
        total_parameters,

    "trainable_parameters":
        trainable_parameters,

    "training_graph_count":
        len(
            D_train
        ),

    "validation_graph_count":
        len(
            D_validation
        ),

    "test_graph_count":
        len(
            D_test
        ),

    "fisher_reference_graph_count":
        len(
            D_fisher_reference
        ),

    "fisher_information_calculated":
        False,

    "ewc_performed":
        False,

    "decoder_training_performed":
        False,

    "model_training_performed":
        False
}


with open(
    PHASE9_CONFIG_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        encoder_config,
        file,
        indent=4,
        ensure_ascii=False
    )


# ==============================================================================
# 40. SAVE INITIAL UNTRAINED ENCODER CHECKPOINT
# ==============================================================================

initial_encoder_checkpoint = {
    "model_status":
        "UNTRAINED_INITIALIZED_ENCODER",

    "state_dict":
        encoder.state_dict(),

    "model_config":
        encoder_config,

    "normalization_metadata":
        normalization_metadata,

    "created_at":
        datetime.now().isoformat()
}


torch.save(
    initial_encoder_checkpoint,
    PHASE9_INITIAL_STATE_FILE
)


# ==============================================================================
# 41. EQUIVARIANCE DIAGNOSTIC TABLE
# ==============================================================================

equivariance_records = [
    {
        "test":
            "translation_mu_invariance",

        "pass":
            bool(
                translation_invariance_pass
            ),

        "max_abs_difference":
            translation_mu_max_abs_difference,

        "atol":
            EQUIVARIANCE_ATOL,

        "rtol":
            EQUIVARIANCE_RTOL
    },

    {
        "test":
            "rotation_mu_invariance",

        "pass":
            bool(
                rotation_invariance_pass
            ),

        "max_abs_difference":
            rotation_mu_max_abs_difference,

        "atol":
            EQUIVARIANCE_ATOL,

        "rtol":
            EQUIVARIANCE_RTOL
    },

    {
        "test":
            "translation_coordinate_equivariance",

        "pass":
            bool(
                translation_coordinate_equivariance_pass
            ),

        "max_abs_difference":
            translation_coordinate_max_abs_difference,

        "atol":
            EQUIVARIANCE_ATOL,

        "rtol":
            EQUIVARIANCE_RTOL
    },

    {
        "test":
            "rotation_coordinate_equivariance",

        "pass":
            bool(
                rotation_coordinate_equivariance_pass
            ),

        "max_abs_difference":
            rotation_coordinate_max_abs_difference,

        "atol":
            EQUIVARIANCE_ATOL,

        "rtol":
            EQUIVARIANCE_RTOL
    },

    {
        "test":
            "node_permutation_mu_invariance",

        "pass":
            bool(
                permutation_invariance_pass
            ),

        "max_abs_difference":
            permutation_mu_max_abs_difference,

        "atol":
            EQUIVARIANCE_ATOL,

        "rtol":
            EQUIVARIANCE_RTOL
    }
]


phase9_equivariance_df = pd.DataFrame(
    equivariance_records
)


phase9_equivariance_df.to_csv(
    PHASE9_EQUIVARIANCE_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 42. PHASE-9 QC STATISTICS
# ==============================================================================

phase9_stats = {
    "training_graphs_available":
        len(
            D_train
        ),

    "validation_graphs_available":
        len(
            D_validation
        ),

    "test_graphs_available":
        len(
            D_test
        ),

    "fisher_reference_graphs_available":
        len(
            D_fisher_reference
        ),

    "node_feature_dimension":
        NODE_FEATURE_DIM,

    "edge_feature_dimension":
        EDGE_FEATURE_DIM,

    "hidden_dimension":
        HIDDEN_DIM,

    "latent_dimension":
        LATENT_DIM,

    "message_passing_layers":
        NUM_EGNN_LAYERS,

    "total_model_parameters":
        total_parameters,

    "trainable_model_parameters":
        trainable_parameters,

    "training_sample_load_success":
        train_graph_loading_pass,

    "validation_sample_load_success":
        validation_graph_loading_pass,

    "test_sample_load_success":
        test_graph_loading_pass,

    "fisher_sample_load_success":
        fisher_graph_loading_pass,

    "mini_batch_forward_pass_success":
        bool(
            mu_shape_valid
            and
            logvar_shape_valid
            and
            z_shape_valid
            and
            all_forward_outputs_finite
        ),

    "mu_output_shape":
        list(
            mu.shape
        ),

    "logvar_output_shape":
        list(
            logvar.shape
        ),

    "z_output_shape":
        list(
            z.shape
        ),

    "finite_output_status":
        all_forward_outputs_finite,

    "deterministic_z_equals_mu":
        bool(
            deterministic_z_equals_mu
        ),

    "stochastic_latent_finite":
        stochastic_latent_finite,

    "gradient_flow_status":
        gradient_flow_pass,

    "translation_invariance_pass":
        bool(
            translation_invariance_pass
        ),

    "rotation_invariance_pass":
        bool(
            rotation_invariance_pass
        ),

    "translation_coordinate_equivariance_pass":
        bool(
            translation_coordinate_equivariance_pass
        ),

    "rotation_coordinate_equivariance_pass":
        bool(
            rotation_coordinate_equivariance_pass
        ),

    "permutation_invariance_pass":
        bool(
            permutation_invariance_pass
        ),

    "phase8_sources_preserved":
        all_phase8_sources_preserved,

    "fisher_information_calculated":
        False,

    "ewc_performed":
        False,

    "decoder_training_performed":
        False,

    "candidate_generation_performed":
        False,

    "model_training_performed":
        False
}


# ==============================================================================
# 43. COMPLETION VALIDATION
# ==============================================================================

completion_checks = {
    "Phase-8 training corpus loaded":
        len(
            D_train
        ) > 0,

    "Phase-8 validation corpus loaded":
        len(
            D_validation
        ) > 0,

    "Phase-8 test corpus loaded":
        len(
            D_test
        ) > 0,

    "Fisher-reference corpus loaded":
        len(
            D_fisher_reference
        ) > 0,

    "Fisher reference remains training-only":
        fisher_ids.issubset(
            train_ids
        ),

    "Training graphs load successfully":
        train_graph_loading_pass,

    "Validation graphs load successfully":
        validation_graph_loading_pass,

    "Test graphs load successfully":
        test_graph_loading_pass,

    "Fisher graphs load successfully":
        fisher_graph_loading_pass,

    "Node feature dimension matches Phase 7":
        NODE_FEATURE_DIM == 21,

    "Edge feature dimension matches Phase 7":
        EDGE_FEATURE_DIM == 14,

    "Training-only normalization established":
        len(
            normalization_metadata[
                "mean"
            ]
        ) > 0,

    "Initial atom embedding implemented":
        hasattr(
            encoder,
            "node_embedding"
        ),

    "Edge attributes used in messages":
        True,

    "Relative geometry used in messages":
        True,

    "Two EGNN-style layers implemented":
        len(
            encoder.egnn_layers
        ) == 2,

    "Equivariant coordinate updates implemented":
        True,

    "Permutation-invariant mean pooling implemented":
        True,

    "mu head implemented":
        hasattr(
            encoder,
            "mu_head"
        ),

    "logvar head implemented":
        hasattr(
            encoder,
            "logvar_head"
        ),

    "Reparameterization implemented":
        hasattr(
            encoder,
            "reparameterize"
        ),

    "Latent dimension equals 32":
        LATENT_DIM == 32,

    "mu output shape valid":
        mu_shape_valid,

    "logvar output shape valid":
        logvar_shape_valid,

    "z output shape valid":
        z_shape_valid,

    "Forward-pass outputs finite":
        all_forward_outputs_finite,

    "Deterministic evaluation uses z = mu":
        deterministic_z_equals_mu,

    "Stochastic latent sampling finite":
        stochastic_latent_finite,

    "Translation invariance passes":
        bool(
            translation_invariance_pass
        ),

    "Rotation invariance passes":
        bool(
            rotation_invariance_pass
        ),

    "Translation coordinate equivariance passes":
        bool(
            translation_coordinate_equivariance_pass
        ),

    "Rotation coordinate equivariance passes":
        bool(
            rotation_coordinate_equivariance_pass
        ),

    "Permutation invariance passes":
        bool(
            permutation_invariance_pass
        ),

    "Gradient-flow sanity test passes":
        gradient_flow_pass,

    "Model parameter counts recorded":
        (
            total_parameters > 0
            and
            trainable_parameters > 0
        ),

    "Phase-8 source datasets preserved":
        all_phase8_sources_preserved,

    "Fisher reference untouched":
        fisher_source_preserved,

    "Encoder config saved":
        PHASE9_CONFIG_FILE.exists(),

    "Feature normalization saved":
        PHASE9_NORMALIZATION_FILE.exists(),

    "Initial encoder checkpoint saved":
        PHASE9_INITIAL_STATE_FILE.exists(),

    "Equivariance diagnostics saved":
        PHASE9_EQUIVARIANCE_FILE.exists(),

    "No Fisher information calculated":
        True,

    "No EWC performed":
        True,

    "No decoder training performed":
        True,

    "No candidate generation performed":
        True,

    "No model training performed":
        True
}


# ==============================================================================
# 44. SAVE QC SUMMARY
# ==============================================================================

phase9_metadata = {
    "phase":
        "Phase 9 — Geometry-Aware Equivariant Molecular Encoder",

    "generated_at":
        datetime.now().isoformat(),

    "model_status":
        "UNTRAINED_INITIALIZED_ENCODER",

    "encoder_config":
        encoder_config,

    "statistics":
        phase9_stats,

    "completion_checks":
        completion_checks,

    "scientific_interpretation":
        (
            "Geometry-aware equivariant variational molecular "
            "encoder architecture constructed and computationally validated."
        ),

    "learned_latent_manifold_claim":
        False
}


with open(
    PHASE9_QC_JSON_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        phase9_metadata,
        file,
        indent=4,
        ensure_ascii=False,
        default=str
    )


qc_rows = []


for metric_name, metric_value in phase9_stats.items():

    qc_rows.append(
        {
            "metric":
                metric_name,

            "value":
                metric_value
        }
    )


for check_name, check_value in completion_checks.items():

    qc_rows.append(
        {
            "metric":
                f"completion.{check_name}",

            "value":
                check_value
        }
    )


phase9_qc_df = pd.DataFrame(
    qc_rows
)


phase9_qc_df.to_csv(
    PHASE9_QC_CSV_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 45. DISPLAY COMPLETION VALIDATION
# ==============================================================================

print("\n" + "=" * 118)
print("PHASE 9 COMPLETION VALIDATION")
print("=" * 118)


all_critical_checks_passed = True


for check_name, status in completion_checks.items():

    label = (
        "PASS"
        if bool(
            status
        )
        else "FAIL"
    )


    print(
        f"[{label:<4}] {check_name}"
    )


    if not bool(
        status
    ):

        all_critical_checks_passed = False


# ==============================================================================
# 46. UPDATE PROJECT CONFIG
# ==============================================================================

PHASE9_DATASETS = {
    "encoder_config_json":
        str(
            PHASE9_CONFIG_FILE
        ),

    "feature_normalization_json":
        str(
            PHASE9_NORMALIZATION_FILE
        ),

    "initial_encoder_state":
        str(
            PHASE9_INITIAL_STATE_FILE
        ),

    "equivariance_test_csv":
        str(
            PHASE9_EQUIVARIANCE_FILE
        ),

    "qc_summary_csv":
        str(
            PHASE9_QC_CSV_FILE
        ),

    "qc_summary_json":
        str(
            PHASE9_QC_JSON_FILE
        ),

    "model_status":
        "UNTRAINED_INITIALIZED_ENCODER",

    "node_feature_dim":
        NODE_FEATURE_DIM,

    "edge_feature_dim":
        EDGE_FEATURE_DIM,

    "hidden_dim":
        HIDDEN_DIM,

    "latent_dim":
        LATENT_DIM,

    "num_egnn_layers":
        NUM_EGNN_LAYERS,

    "total_parameters":
        total_parameters,

    "trainable_parameters":
        trainable_parameters
}


DRUGBANK_CONFIG[
    "phase9_datasets"
] = PHASE9_DATASETS


# ==============================================================================
# 47. FINAL PHASE-9 SUMMARY
# ==============================================================================

print("\n" + "=" * 118)
print("PHASE 9 SUMMARY")
print("=" * 118)


print(
    f"Training Graphs                 : "
    f"{len(D_train)}"
)

print(
    f"Validation Graphs               : "
    f"{len(D_validation)}"
)

print(
    f"Test Graphs                     : "
    f"{len(D_test)}"
)

print(
    f"Fisher Reference Graphs         : "
    f"{len(D_fisher_reference)}"
)

print(
    f"Node Feature Dimension          : "
    f"{NODE_FEATURE_DIM}"
)

print(
    f"Edge Feature Dimension          : "
    f"{EDGE_FEATURE_DIM}"
)

print(
    f"Hidden Dimension                : "
    f"{HIDDEN_DIM}"
)

print(
    f"Latent Dimension                : "
    f"{LATENT_DIM}"
)

print(
    f"EGNN Layers                     : "
    f"{NUM_EGNN_LAYERS}"
)

print(
    f"Total Parameters                : "
    f"{total_parameters:,}"
)

print(
    f"Trainable Parameters            : "
    f"{trainable_parameters:,}"
)

print(
    f"μ Shape                         : "
    f"{tuple(mu.shape)}"
)

print(
    f"log σ² Shape                    : "
    f"{tuple(logvar.shape)}"
)

print(
    f"z Shape                         : "
    f"{tuple(z.shape)}"
)

print(
    f"Finite Forward Outputs          : "
    f"{all_forward_outputs_finite}"
)

print(
    f"Translation Invariance          : "
    f"{translation_invariance_pass}"
)

print(
    f"Rotation Invariance             : "
    f"{rotation_invariance_pass}"
)

print(
    f"Coordinate Translation Equiv.   : "
    f"{translation_coordinate_equivariance_pass}"
)

print(
    f"Coordinate Rotation Equiv.      : "
    f"{rotation_coordinate_equivariance_pass}"
)

print(
    f"Permutation Invariance          : "
    f"{permutation_invariance_pass}"
)

print(
    f"Gradient Flow                   : "
    f"{gradient_flow_pass}"
)

print(
    f"Phase-8 Sources Preserved       : "
    f"{all_phase8_sources_preserved}"
)

print(
    f"Encoder Status                  : "
    f"UNTRAINED / INITIALIZED"
)

print(
    f"Fisher Information Computed     : False"
)

print(
    f"EWC Performed                   : False"
)

print(
    f"Decoder Training Performed      : False"
)

print(
    f"Model Training Performed        : False"
)


print("\nSaved Files:")

print(
    f"  Encoder Config         : "
    f"{PHASE9_CONFIG_FILE}"
)

print(
    f"  Feature Normalization  : "
    f"{PHASE9_NORMALIZATION_FILE}"
)

print(
    f"  Initial Encoder State  : "
    f"{PHASE9_INITIAL_STATE_FILE}"
)

print(
    f"  Equivariance Tests     : "
    f"{PHASE9_EQUIVARIANCE_FILE}"
)

print(
    f"  QC Summary CSV         : "
    f"{PHASE9_QC_CSV_FILE}"
)

print(
    f"  QC Summary JSON        : "
    f"{PHASE9_QC_JSON_FILE}"
)


print("-" * 118)


if all_critical_checks_passed:

    print(
        "[STATUS] PHASE 9 COMPLETED SUCCESSFULLY."
    )

    print(
        "[READY]  Geometry-aware equivariant variational "
        "molecular encoder architecture constructed."
    )

    print(
        "[READY]  Equivariance, invariance, batching, "
        "numerical stability and gradient flow validated."
    )

    print(
        "[IMPORTANT] Encoder is INITIALIZED but NOT TRAINED."
    )

    print(
        "[IMPORTANT] No learned molecular latent manifold "
        "is claimed at this stage."
    )

    print(
        "[NEXT]   Phase 10 — Molecular Decoder and "
        "Baseline Variational Learning Objective"
    )

else:

    print(
        "[STATUS] PHASE 9 COMPLETED WITH ONE OR MORE "
        "FAILED VALIDATION CHECKS."
    )

    print(
        "[ACTION] Review the failed encoder / equivariance "
        "checks before proceeding to Phase 10."
    )


print("=" * 118)

PHASE 9 — GEOMETRY-AWARE EQUIVARIANT MOLECULAR ENCODER

[DEVICE] Device             : cpu
[CONFIG] Random Seed       : 42
[CONFIG] Hidden Dimension  : 64
[CONFIG] Latent Dimension  : 32
[CONFIG] EGNN Layers       : 2
[CONFIG] Batch Size        : 16
[CONFIG] Coordinate Scale  : 0.1

[PASS] Phase-8 training corpus found.
[PASS] Phase-8 validation corpus found.
[PASS] Phase-8 test corpus found.
[PASS] Phase-8 Fisher-reference corpus found.
[PASS] Phase-7 feature schema found.

PHASE-8 INPUT CORPORA
Training graphs          : 613
Validation graphs        : 76
Test graphs              : 78
Fisher-reference graphs  : 128

PHASE-7 GRAPH FEATURE SCHEMA
Node Feature Dimension : 21
Edge Feature Dimension : 14
Coordinate Policy      : Centered Cartesian coordinates; coordinates are separate from scalar node features.
Edges Bidirectional    : True

[PASS] Phase-7 feature dimensions match Phase-9 configuration.
[PASS] Required Phase-8 manifest columns verified.
[PASS] Phase-8 train/validation/test 

# Phase 10 — Baseline Variational Latent Representation Learning

**Objective:** Jointly train the Phase 9 geometry-aware equivariant encoder (EGNN) and a lightweight graph-level reconstruction decoder on $\mathcal{D}_{\text{train}}$ to structure the continuous latent manifold $\mathcal{Z} \subset \mathbb{R}^{32}$ via a variational loss (ELBO).

* **Variational Formulation & Objective:**
  * **Encoder Output:** Maps $G = (V, E, X, R) \to \boldsymbol{\mu} \in \mathbb{R}^{32}, \log \boldsymbol{\sigma}^2 \in \mathbb{R}^{32}$.
  * **Stochastic Reparameterization:** $\mathbf{z} = \boldsymbol{\mu} + \boldsymbol{\sigma} \odot \boldsymbol{\epsilon}$ (where $\boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$) during training; deterministic $\mathbf{z} = \boldsymbol{\mu}$ for validation/testing.
  * **Graph Structural Target ($\mathbf{y}_G \in \mathbb{R}^{38}$):** Concatenation of mean atomic attributes ($21$), mean bond attributes ($14$), and molecular size descriptors ($3$).
  * **Objective:** $\mathcal{L}_{\text{total}} = \text{MSE}(\hat{\mathbf{y}}_G, \mathbf{y}_G) + \beta_t \, \mathcal{L}_{\mathrm{KL}}$, with training-only target standardization and linear $\beta$-annealing to prevent posterior collapse.

* **Experimental Boundaries:**
  * **No Target Overfitting:** $\mathcal{D}_{\text{train}}$ ($613$ graphs) updates parameters; $\mathcal{D}_{\text{validation}}$ ($76$ graphs) guides early stopping; $\mathcal{D}_{\text{test}}$ ($78$ graphs) measures held-out generalization.
  * **Phase Separation:** Establishes a trained structural latent manifold without yet calculating Fisher information (Module 108) or running chemical sequence generation.

* **Outputs:**
  * Trained checkpoints: `phase10_best_encoder.pt`, `phase10_best_latent_decoder.pt`.
  * Latent embeddings: `phase10_train_latent.csv`, `phase10_validation_latent.csv`, `phase10_test_latent.csv`.
  * Training audit & QC: `phase10_training_history.csv`, `phase10_training_summary.json`.

In [10]:
# ==============================================================================
# PHASE 10 — BASELINE VARIATIONAL LATENT REPRESENTATION LEARNING
# ==============================================================================
#
# PURPOSE
# -------
# Train the Phase-9 geometry-aware equivariant variational encoder using a
# lightweight structural reconstruction objective.
#
# Training objective:
#
#       L_total = L_reconstruction + beta * L_KL
#
# Reconstruction target:
#
#       mean node features       = 21 dimensions
#       mean edge features       = 14 dimensions
#       graph-size descriptors   =  3 dimensions
#                                      ----------
#       TOTAL                    = 38 dimensions
#
# IMPORTANT
# ---------
# - TRAIN split updates parameters.
# - VALIDATION selects the best checkpoint.
# - TEST is evaluated only AFTER best checkpoint selection.
# - Fisher-reference corpus is NOT used for training or Fisher calculation.
# - No full molecular generation occurs in Phase 10.
# ==============================================================================


# ==============================================================================
# 1. IMPORTS
# ==============================================================================

from pathlib import Path
from datetime import datetime

import copy
import json
import math
import random
import time

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


# ==============================================================================
# 2. REPRODUCIBILITY
# ==============================================================================

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)


DEVICE = torch.device("cpu")


# ==============================================================================
# 3. VERIFY PROJECT STATE
# ==============================================================================

required_variables = [
    "PROCESSED_DIR",
    "DRUGBANK_CONFIG"
]


missing_variables = [
    variable
    for variable in required_variables
    if variable not in globals()
]


if missing_variables:

    raise RuntimeError(
        "Earlier phases must be executed before Phase 10.\n"
        f"Missing variables: {missing_variables}"
    )


# ==============================================================================
# 4. PHASE-10 CONFIGURATION
# ==============================================================================

HIDDEN_DIM = 64
LATENT_DIM = 32
NUM_EGNN_LAYERS = 2

BATCH_SIZE = 16

LEARNING_RATE = 1e-3

MAX_EPOCHS = 100

EARLY_STOPPING_PATIENCE = 12

BETA_KL_MAX = 0.001

KL_WARMUP_EPOCHS = 20

COORDINATE_UPDATE_SCALE = 0.10

NORMALIZATION_EPS = 1e-8

LATENT_VARIANCE_THRESHOLD = 1e-6

POSTERIOR_KL_WARNING_THRESHOLD = 1e-4


print("=" * 120)
print("PHASE 10 — BASELINE VARIATIONAL LATENT REPRESENTATION LEARNING")
print("=" * 120)

print(f"\n[DEVICE] Device                : {DEVICE}")
print(f"[CONFIG] Random Seed          : {RANDOM_SEED}")
print(f"[CONFIG] Hidden Dimension     : {HIDDEN_DIM}")
print(f"[CONFIG] Latent Dimension     : {LATENT_DIM}")
print(f"[CONFIG] EGNN Layers          : {NUM_EGNN_LAYERS}")
print(f"[CONFIG] Batch Size           : {BATCH_SIZE}")
print(f"[CONFIG] Learning Rate        : {LEARNING_RATE}")
print(f"[CONFIG] Maximum Epochs       : {MAX_EPOCHS}")
print(f"[CONFIG] Early Stop Patience  : {EARLY_STOPPING_PATIENCE}")
print(f"[CONFIG] Maximum KL Beta      : {BETA_KL_MAX}")
print(f"[CONFIG] KL Warm-up Epochs    : {KL_WARMUP_EPOCHS}")


# ==============================================================================
# 5. INPUT FILES
# ==============================================================================

PHASE8_TRAIN_FILE = (
    Path(PROCESSED_DIR)
    / "phase8_train_graphs.csv"
)

PHASE8_VALIDATION_FILE = (
    Path(PROCESSED_DIR)
    / "phase8_validation_graphs.csv"
)

PHASE8_TEST_FILE = (
    Path(PROCESSED_DIR)
    / "phase8_test_graphs.csv"
)

PHASE8_FISHER_FILE = (
    Path(PROCESSED_DIR)
    / "phase8_fisher_reference.csv"
)


PHASE7_FEATURE_SCHEMA_FILE = (
    Path(PROCESSED_DIR)
    / "phase7_graph_feature_schema.json"
)


PHASE9_NORMALIZATION_FILE = (
    Path(PROCESSED_DIR)
    / "phase9_feature_normalization.json"
)


PHASE9_CONFIG_FILE = (
    Path(PROCESSED_DIR)
    / "phase9_encoder_config.json"
)


PHASE9_INITIAL_STATE_FILE = (
    Path(PROCESSED_DIR)
    / "phase9_encoder_initial_state.pt"
)


required_files = [
    PHASE8_TRAIN_FILE,
    PHASE8_VALIDATION_FILE,
    PHASE8_TEST_FILE,
    PHASE8_FISHER_FILE,
    PHASE7_FEATURE_SCHEMA_FILE,
    PHASE9_NORMALIZATION_FILE,
    PHASE9_CONFIG_FILE,
    PHASE9_INITIAL_STATE_FILE
]


missing_files = [
    str(path)
    for path in required_files
    if not path.exists()
]


if missing_files:

    raise FileNotFoundError(
        "Required Phase-7/8/9 files are missing:\n"
        + "\n".join(missing_files)
    )


print("\n[PASS] Required Phase-8 and Phase-9 files found.")


# ==============================================================================
# 6. LOAD PHASE-8 SPLITS
# ==============================================================================

D_train = pd.read_csv(
    PHASE8_TRAIN_FILE,
    dtype=str,
    keep_default_na=False
)


D_validation = pd.read_csv(
    PHASE8_VALIDATION_FILE,
    dtype=str,
    keep_default_na=False
)


D_test = pd.read_csv(
    PHASE8_TEST_FILE,
    dtype=str,
    keep_default_na=False
)


D_fisher_reference = pd.read_csv(
    PHASE8_FISHER_FILE,
    dtype=str,
    keep_default_na=False
)


# Preserve exact copies.

D_train_original = D_train.copy(deep=True)

D_validation_original = D_validation.copy(deep=True)

D_test_original = D_test.copy(deep=True)

D_fisher_original = D_fisher_reference.copy(deep=True)


print("\n" + "=" * 120)
print("PHASE-8 DATASET INPUT")
print("=" * 120)

print(f"Training graphs         : {len(D_train)}")
print(f"Validation graphs       : {len(D_validation)}")
print(f"Test graphs             : {len(D_test)}")
print(f"Fisher reference graphs : {len(D_fisher_reference)}")


# ==============================================================================
# 7. VERIFY SPLIT RELATIONSHIPS
# ==============================================================================

train_ids = set(
    D_train["graph_id"].astype(str)
)

validation_ids = set(
    D_validation["graph_id"].astype(str)
)

test_ids = set(
    D_test["graph_id"].astype(str)
)

fisher_ids = set(
    D_fisher_reference["graph_id"].astype(str)
)


if train_ids & validation_ids:
    raise RuntimeError(
        "Train-validation overlap detected."
    )


if train_ids & test_ids:
    raise RuntimeError(
        "Train-test overlap detected."
    )


if validation_ids & test_ids:
    raise RuntimeError(
        "Validation-test overlap detected."
    )


if not fisher_ids.issubset(train_ids):
    raise RuntimeError(
        "Fisher corpus is not a subset of training data."
    )


print(
    "\n[PASS] Train/validation/test split remains leakage-free."
)

print(
    "[PASS] Fisher reference remains training-only."
)


# ==============================================================================
# 8. LOAD PHASE-7 FEATURE SCHEMA
# ==============================================================================

with open(
    PHASE7_FEATURE_SCHEMA_FILE,
    "r",
    encoding="utf-8"
) as file:

    phase7_schema = json.load(file)


NODE_FEATURE_DIM = int(
    phase7_schema[
        "node_feature_dimension"
    ]
)


EDGE_FEATURE_DIM = int(
    phase7_schema[
        "edge_feature_dimension"
    ]
)


NODE_FEATURE_NAMES = list(
    phase7_schema[
        "node_feature_names"
    ]
)


EDGE_FEATURE_NAMES = list(
    phase7_schema[
        "edge_feature_names"
    ]
)


if NODE_FEATURE_DIM != 21:

    raise RuntimeError(
        f"Expected node feature dimension 21; "
        f"found {NODE_FEATURE_DIM}."
    )


if EDGE_FEATURE_DIM != 14:

    raise RuntimeError(
        f"Expected edge feature dimension 14; "
        f"found {EDGE_FEATURE_DIM}."
    )


print(
    "\n[PASS] Phase-7 graph schema loaded."
)

print(
    f"[INFO] Node feature dimension : {NODE_FEATURE_DIM}"
)

print(
    f"[INFO] Edge feature dimension : {EDGE_FEATURE_DIM}"
)


# ==============================================================================
# 9. LOAD PHASE-9 NORMALIZATION
# ==============================================================================

with open(
    PHASE9_NORMALIZATION_FILE,
    "r",
    encoding="utf-8"
) as file:

    phase9_normalization = json.load(file)


NORMALIZED_NODE_FEATURE_NAMES = list(
    phase9_normalization[
        "standardized_features"
    ]
)


normalization_indices = list(
    phase9_normalization[
        "standardized_feature_indices"
    ]
)


phase9_feature_mean = torch.tensor(
    [
        phase9_normalization[
            "mean"
        ][feature]
        for feature in NORMALIZED_NODE_FEATURE_NAMES
    ],
    dtype=torch.float32
)


phase9_feature_std = torch.tensor(
    [
        phase9_normalization[
            "std"
        ][feature]
        for feature in NORMALIZED_NODE_FEATURE_NAMES
    ],
    dtype=torch.float32
)


print(
    "[PASS] Phase-9 training-only feature normalization loaded."
)


# ==============================================================================
# 10. LOAD PHASE-9 MODEL CONFIG
# ==============================================================================

with open(
    PHASE9_CONFIG_FILE,
    "r",
    encoding="utf-8"
) as file:

    phase9_config = json.load(file)


if int(
    phase9_config[
        "latent_dim"
    ]
) != LATENT_DIM:

    raise RuntimeError(
        "Phase-9 latent dimension does not match Phase 10."
    )


if int(
    phase9_config[
        "hidden_dim"
    ]
) != HIDDEN_DIM:

    raise RuntimeError(
        "Phase-9 hidden dimension does not match Phase 10."
    )


print(
    "[PASS] Phase-9 encoder configuration verified."
)


# ==============================================================================
# 11. GRAPH LOADING HELPER
# ==============================================================================

def load_graph(
    graph_path
):

    graph = torch.load(
        Path(graph_path),
        map_location="cpu",
        weights_only=False
    )


    required_keys = {
        "graph_id",
        "drug_id",
        "x",
        "edge_index",
        "edge_attr",
        "pos"
    }


    if not required_keys.issubset(
        graph.keys()
    ):

        raise RuntimeError(
            f"Graph missing keys: "
            f"{required_keys - set(graph.keys())}"
        )


    return graph


# ==============================================================================
# 12. PHASE-9 NODE NORMALIZATION FUNCTION
# ==============================================================================

def normalize_node_features(
    x
):

    x = x.clone().float()


    index_tensor = torch.tensor(
        normalization_indices,
        dtype=torch.long,
        device=x.device
    )


    mean = phase9_feature_mean.to(
        x.device
    )


    std = phase9_feature_std.to(
        x.device
    )


    x[
        :,
        index_tensor
    ] = (
        x[
            :,
            index_tensor
        ]
        -
        mean
    ) / std


    return x


# ==============================================================================
# 13. RECONSTRUCTION TARGET DEFINITION
# ==============================================================================

RECONSTRUCTION_TARGET_NAMES = (
    [
        f"mean_node::{name}"
        for name in NODE_FEATURE_NAMES
    ]
    +
    [
        f"mean_edge::{name}"
        for name in EDGE_FEATURE_NAMES
    ]
    +
    [
        "graph_size::num_nodes",
        "graph_size::num_bonds",
        "graph_size::heavy_atom_count"
    ]
)


RECONSTRUCTION_DIM = len(
    RECONSTRUCTION_TARGET_NAMES
)


expected_reconstruction_dim = (
    NODE_FEATURE_DIM
    +
    EDGE_FEATURE_DIM
    +
    3
)


if RECONSTRUCTION_DIM != expected_reconstruction_dim:

    raise RuntimeError(
        "Reconstruction target dimension mismatch."
    )


if RECONSTRUCTION_DIM != 38:

    raise RuntimeError(
        f"Expected reconstruction dimension 38; "
        f"found {RECONSTRUCTION_DIM}."
    )


print("\n" + "=" * 120)
print("RECONSTRUCTION TARGET DEFINITION")
print("=" * 120)

print(
    f"Mean node features : {NODE_FEATURE_DIM}"
)

print(
    f"Mean edge features : {EDGE_FEATURE_DIM}"
)

print(
    "Graph-size values : 3"
)

print(
    f"Total target dim   : {RECONSTRUCTION_DIM}"
)


# ==============================================================================
# 14. RAW RECONSTRUCTION TARGET FUNCTION
# ==============================================================================

def construct_raw_reconstruction_target(
    graph,
    manifest_row=None
):

    # --------------------------------------------------------------------------
    # Raw Phase-7 graph features are used to define the structural target.
    #
    # The target itself is subsequently standardized using TRAINING-ONLY
    # target statistics.
    # --------------------------------------------------------------------------

    raw_x = graph[
        "x"
    ].float()


    raw_edge_attr = graph[
        "edge_attr"
    ].float()


    mean_node = raw_x.mean(
        dim=0
    )


    if raw_edge_attr.shape[0] > 0:

        mean_edge = raw_edge_attr.mean(
            dim=0
        )

    else:

        mean_edge = torch.zeros(
            EDGE_FEATURE_DIM,
            dtype=torch.float32
        )


    num_nodes = float(
        raw_x.shape[
            0
        ]
    )


    # Phase-7 edge_index is bidirectional.
    # Therefore undirected bond count is half the directed edge count.

    num_directed_edges = int(
        graph[
            "edge_index"
        ].shape[
            1
        ]
    )


    num_bonds = float(
        num_directed_edges
        /
        2
    )


    if "heavy_atom_count" in graph:

        heavy_atom_count = float(
            graph[
                "heavy_atom_count"
            ]
        )

    elif manifest_row is not None:

        heavy_atom_count = float(
            manifest_row[
                "heavy_atom_count"
            ]
        )

    else:

        raise RuntimeError(
            "Heavy atom count unavailable."
        )


    graph_size = torch.tensor(
        [
            num_nodes,
            num_bonds,
            heavy_atom_count
        ],
        dtype=torch.float32
    )


    target = torch.cat(
        [
            mean_node,
            mean_edge,
            graph_size
        ],
        dim=0
    )


    if target.shape[0] != RECONSTRUCTION_DIM:

        raise RuntimeError(
            "Unexpected reconstruction target dimension."
        )


    if not torch.isfinite(
        target
    ).all():

        raise RuntimeError(
            "Non-finite reconstruction target detected."
        )


    return target


# ==============================================================================
# 15. FIT RECONSTRUCTION TARGET NORMALIZATION ON TRAINING DATA ONLY
# ==============================================================================

print("\n" + "=" * 120)
print("FITTING TRAINING-ONLY RECONSTRUCTION TARGET NORMALIZATION")
print("=" * 120)


training_raw_targets = []


for i, row in D_train.iterrows():

    graph = load_graph(
        row[
            "graph_file"
        ]
    )


    target = construct_raw_reconstruction_target(
        graph,
        row
    )


    training_raw_targets.append(
        target
    )


    count = i + 1


    if (
        count == 1
        or
        count % 100 == 0
        or
        count == len(D_train)
    ):

        print(
            f"[PROGRESS] "
            f"{count}/{len(D_train)} training targets constructed..."
        )


training_raw_targets_tensor = torch.stack(
    training_raw_targets,
    dim=0
)


target_mean = training_raw_targets_tensor.mean(
    dim=0
)


target_std = training_raw_targets_tensor.std(
    dim=0,
    unbiased=False
)


target_zero_variance_mask = (
    target_std
    <
    NORMALIZATION_EPS
)


target_zero_variance_dimensions = int(
    target_zero_variance_mask.sum().item()
)


target_std_safe = torch.where(
    target_zero_variance_mask,
    torch.ones_like(
        target_std
    ),
    target_std
)


print(
    "\n[PASS] Reconstruction-target normalization fitted "
    "using training corpus only."
)

print(
    f"[INFO] Zero-variance target dimensions : "
    f"{target_zero_variance_dimensions}"
)


# ==============================================================================
# 16. TARGET NORMALIZATION FUNCTIONS
# ==============================================================================

def normalize_reconstruction_target(
    target
):

    return (
        target
        -
        target_mean
    ) / target_std_safe


def denormalize_reconstruction_target(
    normalized_target
):

    return (
        normalized_target
        *
        target_std_safe.to(
            normalized_target.device
        )
        +
        target_mean.to(
            normalized_target.device
        )
    )


# ==============================================================================
# 17. TARGET NORMALIZATION METADATA
# ==============================================================================

target_normalization_metadata = {
    "fitted_on":
        "Phase-8 training corpus only",

    "training_graph_count":
        len(D_train),

    "reconstruction_dimension":
        RECONSTRUCTION_DIM,

    "target_names":
        RECONSTRUCTION_TARGET_NAMES,

    "mean":
        {
            name:
                float(
                    target_mean[
                        index
                    ].item()
                )
            for index, name
            in enumerate(
                RECONSTRUCTION_TARGET_NAMES
            )
        },

    "std":
        {
            name:
                float(
                    target_std_safe[
                        index
                    ].item()
                )
            for index, name
            in enumerate(
                RECONSTRUCTION_TARGET_NAMES
            )
        },

    "original_zero_variance_dimensions":
        target_zero_variance_dimensions
}


# ==============================================================================
# 18. GRAPH DATASET
# ==============================================================================

class Phase10GraphDataset(Dataset):

    def __init__(
        self,
        manifest
    ):

        self.manifest = (
            manifest
            .copy(
                deep=True
            )
            .reset_index(
                drop=True
            )
        )


    def __len__(
        self
    ):

        return len(
            self.manifest
        )


    def __getitem__(
        self,
        index
    ):

        row = self.manifest.iloc[
            index
        ]


        graph = load_graph(
            row[
                "graph_file"
            ]
        )


        raw_target = construct_raw_reconstruction_target(
            graph,
            row
        )


        normalized_target = normalize_reconstruction_target(
            raw_target
        )


        x = normalize_node_features(
            graph[
                "x"
            ].float()
        )


        return {
            "graph_id":
                str(
                    graph[
                        "graph_id"
                    ]
                ),

            "drug_id":
                str(
                    graph[
                        "drug_id"
                    ]
                ),

            "drug_name":
                str(
                    row.get(
                        "drug_name",
                        ""
                    )
                ),

            "x":
                x,

            "edge_index":
                graph[
                    "edge_index"
                ].long(),

            "edge_attr":
                graph[
                    "edge_attr"
                ].float(),

            "pos":
                graph[
                    "pos"
                ].float(),

            "target":
                normalized_target.float(),

            "raw_target":
                raw_target.float()
        }


# ==============================================================================
# 19. GRAPH COLLATE FUNCTION
# ==============================================================================

def phase10_collate(
    graph_list
):

    x_list = []

    pos_list = []

    edge_index_list = []

    edge_attr_list = []

    batch_list = []

    target_list = []

    raw_target_list = []

    graph_ids = []

    drug_ids = []

    drug_names = []

    node_offset = 0


    for graph_number, graph in enumerate(
        graph_list
    ):

        x = graph[
            "x"
        ]


        num_nodes = int(
            x.shape[
                0
            ]
        )


        x_list.append(
            x
        )


        pos_list.append(
            graph[
                "pos"
            ]
        )


        edge_index_list.append(
            graph[
                "edge_index"
            ]
            +
            node_offset
        )


        edge_attr_list.append(
            graph[
                "edge_attr"
            ]
        )


        batch_list.append(
            torch.full(
                (
                    num_nodes,
                ),
                graph_number,
                dtype=torch.long
            )
        )


        target_list.append(
            graph[
                "target"
            ]
        )


        raw_target_list.append(
            graph[
                "raw_target"
            ]
        )


        graph_ids.append(
            graph[
                "graph_id"
            ]
        )


        drug_ids.append(
            graph[
                "drug_id"
            ]
        )


        drug_names.append(
            graph[
                "drug_name"
            ]
        )


        node_offset += (
            num_nodes
        )


    return {
        "x":
            torch.cat(
                x_list,
                dim=0
            ),

        "pos":
            torch.cat(
                pos_list,
                dim=0
            ),

        "edge_index":
            torch.cat(
                edge_index_list,
                dim=1
            ),

        "edge_attr":
            torch.cat(
                edge_attr_list,
                dim=0
            ),

        "batch":
            torch.cat(
                batch_list,
                dim=0
            ),

        "target":
            torch.stack(
                target_list,
                dim=0
            ),

        "raw_target":
            torch.stack(
                raw_target_list,
                dim=0
            ),

        "graph_ids":
            graph_ids,

        "drug_ids":
            drug_ids,

        "drug_names":
            drug_names,

        "num_graphs":
            len(
                graph_list
            )
    }


# ==============================================================================
# 20. CREATE ONLY TRAIN + VALIDATION DATASETS BEFORE MODEL SELECTION
# ==============================================================================

train_dataset = Phase10GraphDataset(
    D_train
)


validation_dataset = Phase10GraphDataset(
    D_validation
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=phase10_collate,
    generator=torch.Generator().manual_seed(
        RANDOM_SEED
    )
)


validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=phase10_collate
)


print(
    "\n[PASS] Training and validation datasets prepared."
)

print(
    "[INFO] Test graphs have not been used for model selection."
)

print(
    "[INFO] Fisher-reference graphs have not been used for training."
)


# ==============================================================================
# 21. SCATTER MEAN
# ==============================================================================

def scatter_mean(
    values,
    index,
    dim_size
):

    output = torch.zeros(
        (
            dim_size,
            values.shape[
                1
            ]
        ),
        dtype=values.dtype,
        device=values.device
    )


    counts = torch.zeros(
        (
            dim_size,
            1
        ),
        dtype=values.dtype,
        device=values.device
    )


    output.index_add_(
        0,
        index,
        values
    )


    ones = torch.ones(
        (
            values.shape[
                0
            ],
            1
        ),
        dtype=values.dtype,
        device=values.device
    )


    counts.index_add_(
        0,
        index,
        ones
    )


    return (
        output
        /
        counts.clamp(
            min=1.0
        )
    )


# ==============================================================================
# 22. EGNN LAYER — SAME ARCHITECTURE AS PHASE 9
# ==============================================================================

class EGNNLayer(nn.Module):

    def __init__(
        self,
        hidden_dim,
        edge_dim,
        coordinate_update_scale=0.10
    ):

        super().__init__()


        self.coordinate_update_scale = (
            coordinate_update_scale
        )


        edge_input_dim = (
            2
            *
            hidden_dim
            +
            1
            +
            edge_dim
        )


        self.edge_mlp = nn.Sequential(

            nn.Linear(
                edge_input_dim,
                hidden_dim
            ),

            nn.SiLU(),

            nn.Linear(
                hidden_dim,
                hidden_dim
            ),

            nn.SiLU()
        )


        self.coordinate_mlp = nn.Sequential(

            nn.Linear(
                hidden_dim,
                hidden_dim
            ),

            nn.SiLU(),

            nn.Linear(
                hidden_dim,
                1
            ),

            nn.Tanh()
        )


        self.node_mlp = nn.Sequential(

            nn.Linear(
                2
                *
                hidden_dim,
                hidden_dim
            ),

            nn.SiLU(),

            nn.Linear(
                hidden_dim,
                hidden_dim
            )
        )


        self.node_norm = nn.LayerNorm(
            hidden_dim
        )


    def forward(
        self,
        h,
        pos,
        edge_index,
        edge_attr
    ):

        source = edge_index[
            0
        ]


        target = edge_index[
            1
        ]


        relative = (
            pos[
                source
            ]
            -
            pos[
                target
            ]
        )


        squared_distance = (
            relative
            *
            relative
        ).sum(
            dim=1,
            keepdim=True
        )


        edge_input = torch.cat(
            [
                h[
                    source
                ],

                h[
                    target
                ],

                squared_distance,

                edge_attr
            ],
            dim=1
        )


        messages = self.edge_mlp(
            edge_input
        )


        coordinate_weight = (
            self.coordinate_mlp(
                messages
            )
            *
            self.coordinate_update_scale
        )


        coordinate_messages = (
            relative
            *
            coordinate_weight
        )


        coordinate_aggregate = scatter_mean(
            coordinate_messages,
            source,
            h.shape[
                0
            ]
        )


        updated_pos = (
            pos
            +
            coordinate_aggregate
        )


        aggregated_messages = scatter_mean(
            messages,
            source,
            h.shape[
                0
            ]
        )


        node_input = torch.cat(
            [
                h,
                aggregated_messages
            ],
            dim=1
        )


        node_delta = self.node_mlp(
            node_input
        )


        updated_h = self.node_norm(
            h
            +
            node_delta
        )


        return (
            updated_h,
            updated_pos
        )


# ==============================================================================
# 23. PHASE-9 ENCODER ARCHITECTURE
# ==============================================================================

class GeometryAwareVariationalEncoder(nn.Module):

    def __init__(
        self,
        node_feature_dim,
        edge_feature_dim,
        hidden_dim,
        latent_dim,
        num_layers=2
    ):

        super().__init__()


        self.node_feature_dim = node_feature_dim

        self.edge_feature_dim = edge_feature_dim

        self.hidden_dim = hidden_dim

        self.latent_dim = latent_dim

        self.num_layers = num_layers


        self.node_embedding = nn.Sequential(

            nn.Linear(
                node_feature_dim,
                hidden_dim
            ),

            nn.SiLU(),

            nn.LayerNorm(
                hidden_dim
            )
        )


        self.egnn_layers = nn.ModuleList(
            [
                EGNNLayer(
                    hidden_dim=hidden_dim,
                    edge_dim=edge_feature_dim,
                    coordinate_update_scale=COORDINATE_UPDATE_SCALE
                )
                for _ in range(
                    num_layers
                )
            ]
        )


        self.graph_mlp = nn.Sequential(

            nn.Linear(
                hidden_dim,
                hidden_dim
            ),

            nn.SiLU(),

            nn.LayerNorm(
                hidden_dim
            )
        )


        self.mu_head = nn.Linear(
            hidden_dim,
            latent_dim
        )


        self.logvar_head = nn.Linear(
            hidden_dim,
            latent_dim
        )


    def graph_mean_pool(
        self,
        h,
        batch
    ):

        number_of_graphs = (
            int(
                batch.max().item()
            )
            +
            1
        )


        return scatter_mean(
            h,
            batch,
            number_of_graphs
        )


    def reparameterize(
        self,
        mu,
        logvar,
        deterministic=False
    ):

        if deterministic:

            return mu


        stable_logvar = torch.clamp(
            logvar,
            min=-20.0,
            max=20.0
        )


        std = torch.exp(
            0.5
            *
            stable_logvar
        )


        epsilon = torch.randn_like(
            std
        )


        return (
            mu
            +
            std
            *
            epsilon
        )


    def forward(
        self,
        x,
        pos,
        edge_index,
        edge_attr,
        batch,
        deterministic=False
    ):

        h = self.node_embedding(
            x
        )


        updated_pos = pos


        for layer in self.egnn_layers:

            h, updated_pos = layer(
                h=h,
                pos=updated_pos,
                edge_index=edge_index,
                edge_attr=edge_attr
            )


        graph_embedding = self.graph_mean_pool(
            h,
            batch
        )


        graph_embedding = self.graph_mlp(
            graph_embedding
        )


        mu = self.mu_head(
            graph_embedding
        )


        logvar = self.logvar_head(
            graph_embedding
        )


        z = self.reparameterize(
            mu,
            logvar,
            deterministic=deterministic
        )


        return {
            "mu":
                mu,

            "logvar":
                logvar,

            "z":
                z,

            "graph_embedding":
                graph_embedding,

            "updated_coordinates":
                updated_pos
        }


# ==============================================================================
# 24. LATENT RECONSTRUCTION DECODER
# ==============================================================================

class LatentReconstructionDecoder(nn.Module):

    def __init__(
        self,
        latent_dim,
        hidden_dim,
        output_dim
    ):

        super().__init__()


        self.network = nn.Sequential(

            nn.Linear(
                latent_dim,
                hidden_dim
            ),

            nn.SiLU(),

            nn.Linear(
                hidden_dim,
                output_dim
            )
        )


    def forward(
        self,
        z
    ):

        return self.network(
            z
        )


# ==============================================================================
# 25. INITIALIZE ENCODER AND LOAD PHASE-9 CHECKPOINT
# ==============================================================================

encoder = GeometryAwareVariationalEncoder(
    node_feature_dim=NODE_FEATURE_DIM,
    edge_feature_dim=EDGE_FEATURE_DIM,
    hidden_dim=HIDDEN_DIM,
    latent_dim=LATENT_DIM,
    num_layers=NUM_EGNN_LAYERS
).to(
    DEVICE
)


phase9_checkpoint = torch.load(
    PHASE9_INITIAL_STATE_FILE,
    map_location=DEVICE,
    weights_only=False
)


if "state_dict" not in phase9_checkpoint:

    raise RuntimeError(
        "Phase-9 checkpoint does not contain state_dict."
    )


encoder.load_state_dict(
    phase9_checkpoint[
        "state_dict"
    ]
)


print(
    "\n[PASS] Phase-9 initialized encoder loaded successfully."
)


# Keep a copy for later confirmation that training actually modified parameters.

initial_encoder_state = {
    name:
        tensor.detach().clone()
    for name, tensor
    in encoder.state_dict().items()
}


# ==============================================================================
# 26. INITIALIZE LATENT RECONSTRUCTION DECODER
# ==============================================================================

torch.manual_seed(
    RANDOM_SEED
)


latent_decoder = LatentReconstructionDecoder(
    latent_dim=LATENT_DIM,
    hidden_dim=HIDDEN_DIM,
    output_dim=RECONSTRUCTION_DIM
).to(
    DEVICE
)


initial_decoder_state = {
    name:
        tensor.detach().clone()
    for name, tensor
    in latent_decoder.state_dict().items()
}


encoder_parameter_count = sum(
    p.numel()
    for p in encoder.parameters()
)


decoder_parameter_count = sum(
    p.numel()
    for p in latent_decoder.parameters()
)


total_trainable_parameters = (
    encoder_parameter_count
    +
    decoder_parameter_count
)


print("\n" + "=" * 120)
print("PHASE-10 MODEL CONFIGURATION")
print("=" * 120)

print(
    f"Encoder parameters       : "
    f"{encoder_parameter_count:,}"
)

print(
    f"Latent decoder parameters: "
    f"{decoder_parameter_count:,}"
)

print(
    f"Combined parameters      : "
    f"{total_trainable_parameters:,}"
)

print(
    f"Reconstruction dimension : "
    f"{RECONSTRUCTION_DIM}"
)


# ==============================================================================
# 27. OPTIMIZER
# ==============================================================================

optimizer = torch.optim.Adam(
    list(
        encoder.parameters()
    )
    +
    list(
        latent_decoder.parameters()
    ),
    lr=LEARNING_RATE
)


reconstruction_loss_function = nn.MSELoss(
    reduction="mean"
)


# ==============================================================================
# 28. KL-DIVERGENCE
# ==============================================================================

def kl_divergence(
    mu,
    logvar
):

    # Mean KL divergence per molecular graph.

    kl_per_graph = (
        -0.5
        *
        torch.sum(
            1.0
            +
            logvar
            -
            mu.pow(
                2
            )
            -
            torch.exp(
                torch.clamp(
                    logvar,
                    min=-20.0,
                    max=20.0
                )
            ),
            dim=1
        )
    )


    return kl_per_graph.mean()


# ==============================================================================
# 29. KL WARM-UP
# ==============================================================================

def get_kl_beta(
    epoch
):

    warmup_fraction = min(
        1.0,
        epoch
        /
        max(
            1,
            KL_WARMUP_EPOCHS
        )
    )


    return (
        BETA_KL_MAX
        *
        warmup_fraction
    )


# ==============================================================================
# 30. MOVE BATCH TO DEVICE
# ==============================================================================

def move_batch_to_device(
    batch
):

    result = {}


    for key, value in batch.items():

        if torch.is_tensor(
            value
        ):

            result[
                key
            ] = value.to(
                DEVICE
            )

        else:

            result[
                key
            ] = value


    return result


# ==============================================================================
# 31. TRAIN ONE EPOCH
# ==============================================================================

def train_one_epoch(
    encoder,
    decoder,
    loader,
    optimizer,
    beta
):

    encoder.train()

    decoder.train()


    total_loss_sum = 0.0

    reconstruction_loss_sum = 0.0

    kl_loss_sum = 0.0

    molecule_count = 0


    for batch in loader:

        batch = move_batch_to_device(
            batch
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        output = encoder(
            x=batch[
                "x"
            ],
            pos=batch[
                "pos"
            ],
            edge_index=batch[
                "edge_index"
            ],
            edge_attr=batch[
                "edge_attr"
            ],
            batch=batch[
                "batch"
            ],
            deterministic=False
        )


        prediction = decoder(
            output[
                "z"
            ]
        )


        reconstruction_loss = (
            reconstruction_loss_function(
                prediction,
                batch[
                    "target"
                ]
            )
        )


        kl_loss = kl_divergence(
            output[
                "mu"
            ],
            output[
                "logvar"
            ]
        )


        total_loss = (
            reconstruction_loss
            +
            beta
            *
            kl_loss
        )


        if not torch.isfinite(
            total_loss
        ):

            raise RuntimeError(
                "Non-finite training loss encountered."
            )


        total_loss.backward()


        torch.nn.utils.clip_grad_norm_(
            list(
                encoder.parameters()
            )
            +
            list(
                decoder.parameters()
            ),
            max_norm=5.0
        )


        optimizer.step()


        batch_size_actual = int(
            batch[
                "num_graphs"
            ]
        )


        total_loss_sum += (
            float(
                total_loss.detach().item()
            )
            *
            batch_size_actual
        )


        reconstruction_loss_sum += (
            float(
                reconstruction_loss.detach().item()
            )
            *
            batch_size_actual
        )


        kl_loss_sum += (
            float(
                kl_loss.detach().item()
            )
            *
            batch_size_actual
        )


        molecule_count += (
            batch_size_actual
        )


    return {
        "total_loss":
            total_loss_sum
            /
            molecule_count,

        "reconstruction_loss":
            reconstruction_loss_sum
            /
            molecule_count,

        "kl_loss":
            kl_loss_sum
            /
            molecule_count
    }


# ==============================================================================
# 32. VALIDATE MODEL
# ==============================================================================

def evaluate_loss(
    encoder,
    decoder,
    loader,
    beta
):

    encoder.eval()

    decoder.eval()


    total_loss_sum = 0.0

    reconstruction_loss_sum = 0.0

    kl_loss_sum = 0.0

    molecule_count = 0


    with torch.no_grad():

        for batch in loader:

            batch = move_batch_to_device(
                batch
            )


            output = encoder(
                x=batch[
                    "x"
                ],
                pos=batch[
                    "pos"
                ],
                edge_index=batch[
                    "edge_index"
                ],
                edge_attr=batch[
                    "edge_attr"
                ],
                batch=batch[
                    "batch"
                ],

                # deterministic validation for stable model selection
                deterministic=True
            )


            prediction = decoder(
                output[
                    "z"
                ]
            )


            reconstruction_loss = (
                reconstruction_loss_function(
                    prediction,
                    batch[
                        "target"
                    ]
                )
            )


            kl_loss = kl_divergence(
                output[
                    "mu"
                ],
                output[
                    "logvar"
                ]
            )


            total_loss = (
                reconstruction_loss
                +
                beta
                *
                kl_loss
            )


            batch_size_actual = int(
                batch[
                    "num_graphs"
                ]
            )


            total_loss_sum += (
                float(
                    total_loss.item()
                )
                *
                batch_size_actual
            )


            reconstruction_loss_sum += (
                float(
                    reconstruction_loss.item()
                )
                *
                batch_size_actual
            )


            kl_loss_sum += (
                float(
                    kl_loss.item()
                )
                *
                batch_size_actual
            )


            molecule_count += (
                batch_size_actual
            )


    return {
        "total_loss":
            total_loss_sum
            /
            molecule_count,

        "reconstruction_loss":
            reconstruction_loss_sum
            /
            molecule_count,

        "kl_loss":
            kl_loss_sum
            /
            molecule_count
    }


# ==============================================================================
# 33. OUTPUT FILE PATHS
# ==============================================================================

PHASE10_TRAINING_CONFIG_FILE = (
    Path(PROCESSED_DIR)
    / "phase10_training_config.json"
)


PHASE10_TRAINING_HISTORY_FILE = (
    Path(PROCESSED_DIR)
    / "phase10_training_history.csv"
)


PHASE10_BEST_ENCODER_FILE = (
    Path(PROCESSED_DIR)
    / "phase10_best_encoder.pt"
)


PHASE10_BEST_DECODER_FILE = (
    Path(PROCESSED_DIR)
    / "phase10_best_latent_decoder.pt"
)


PHASE10_TARGET_SCHEMA_FILE = (
    Path(PROCESSED_DIR)
    / "phase10_reconstruction_target_schema.json"
)


PHASE10_TARGET_NORMALIZATION_FILE = (
    Path(PROCESSED_DIR)
    / "phase10_reconstruction_normalization.json"
)


PHASE10_RECONSTRUCTION_METRICS_FILE = (
    Path(PROCESSED_DIR)
    / "phase10_reconstruction_metrics.csv"
)


PHASE10_TRAIN_LATENT_FILE = (
    Path(PROCESSED_DIR)
    / "phase10_train_latent.csv"
)


PHASE10_VALIDATION_LATENT_FILE = (
    Path(PROCESSED_DIR)
    / "phase10_validation_latent.csv"
)


PHASE10_TEST_LATENT_FILE = (
    Path(PROCESSED_DIR)
    / "phase10_test_latent.csv"
)


PHASE10_LATENT_QC_FILE = (
    Path(PROCESSED_DIR)
    / "phase10_latent_qc.csv"
)


PHASE10_SUMMARY_CSV_FILE = (
    Path(PROCESSED_DIR)
    / "phase10_training_summary.csv"
)


PHASE10_SUMMARY_JSON_FILE = (
    Path(PROCESSED_DIR)
    / "phase10_training_summary.json"
)


# ==============================================================================
# 34. TRAINING CONFIGURATION
# ==============================================================================

phase10_training_config = {
    "phase":
        "Phase 10 — Baseline Variational Latent Representation Learning",

    "random_seed":
        RANDOM_SEED,

    "device":
        str(
            DEVICE
        ),

    "training_graphs":
        len(
            D_train
        ),

    "validation_graphs":
        len(
            D_validation
        ),

    "test_graphs":
        len(
            D_test
        ),

    "fisher_reference_graphs":
        len(
            D_fisher_reference
        ),

    "node_feature_dim":
        NODE_FEATURE_DIM,

    "edge_feature_dim":
        EDGE_FEATURE_DIM,

    "hidden_dim":
        HIDDEN_DIM,

    "latent_dim":
        LATENT_DIM,

    "egnn_layers":
        NUM_EGNN_LAYERS,

    "reconstruction_dimension":
        RECONSTRUCTION_DIM,

    "reconstruction_target":
        (
            "mean raw Phase-7 node features + "
            "mean raw Phase-7 edge features + "
            "num_nodes + num_bonds + heavy_atom_count"
        ),

    "optimizer":
        "Adam",

    "learning_rate":
        LEARNING_RATE,

    "batch_size":
        BATCH_SIZE,

    "maximum_epochs":
        MAX_EPOCHS,

    "early_stopping_patience":
        EARLY_STOPPING_PATIENCE,

    "kl_beta_max":
        BETA_KL_MAX,

    "kl_schedule":
        "linear warm-up",

    "kl_warmup_epochs":
        KL_WARMUP_EPOCHS,

    "training_sampling":
        "stochastic reparameterization",

    "validation_encoding":
        "deterministic z = mu",

    "test_encoding":
        "deterministic z = mu",

    "test_used_for_model_selection":
        False,

    "fisher_used_for_training":
        False,

    "fisher_information_calculated":
        False,

    "ewc_performed":
        False,

    "full_molecular_generation_performed":
        False
}


with open(
    PHASE10_TRAINING_CONFIG_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        phase10_training_config,
        file,
        indent=4,
        ensure_ascii=False
    )


# ==============================================================================
# 35. SAVE RECONSTRUCTION SCHEMA + NORMALIZATION
# ==============================================================================

reconstruction_schema = {
    "reconstruction_dimension":
        RECONSTRUCTION_DIM,

    "node_summary_dimensions":
        NODE_FEATURE_DIM,

    "edge_summary_dimensions":
        EDGE_FEATURE_DIM,

    "graph_size_dimensions":
        3,

    "target_names":
        RECONSTRUCTION_TARGET_NAMES,

    "target_source":
        "Phase-7 graph representation only",

    "biological_information_used":
        False
}


with open(
    PHASE10_TARGET_SCHEMA_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        reconstruction_schema,
        file,
        indent=4,
        ensure_ascii=False
    )


with open(
    PHASE10_TARGET_NORMALIZATION_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        target_normalization_metadata,
        file,
        indent=4,
        ensure_ascii=False
    )


# ==============================================================================
# 36. VARIATIONAL TRAINING LOOP
# ==============================================================================

print("\n" + "=" * 120)
print("BASELINE VARIATIONAL TRAINING")
print("=" * 120)


training_history = []


best_validation_loss = float(
    "inf"
)


best_epoch = 0


epochs_without_improvement = 0


training_start_time = time.time()


for epoch in range(
    1,
    MAX_EPOCHS
    +
    1
):

    beta = get_kl_beta(
        epoch
    )


    train_metrics = train_one_epoch(
        encoder=encoder,
        decoder=latent_decoder,
        loader=train_loader,
        optimizer=optimizer,
        beta=beta
    )


    validation_metrics = evaluate_loss(
        encoder=encoder,
        decoder=latent_decoder,
        loader=validation_loader,
        beta=beta
    )


    improved = (
        validation_metrics[
            "total_loss"
        ]
        <
        best_validation_loss
    )


    if improved:

        best_validation_loss = (
            validation_metrics[
                "total_loss"
            ]
        )


        best_epoch = epoch


        epochs_without_improvement = 0


        torch.save(
            {
                "model_status":
                    "TRAINED_BASELINE_ENCODER",

                "epoch":
                    epoch,

                "validation_total_loss":
                    validation_metrics[
                        "total_loss"
                    ],

                "state_dict":
                    encoder.state_dict(),

                "model_config":
                    phase10_training_config,

                "phase9_normalization":
                    phase9_normalization,

                "target_normalization":
                    target_normalization_metadata
            },
            PHASE10_BEST_ENCODER_FILE
        )


        torch.save(
            {
                "model_status":
                    "TRAINED_BASELINE_LATENT_RECONSTRUCTION_DECODER",

                "epoch":
                    epoch,

                "validation_total_loss":
                    validation_metrics[
                        "total_loss"
                    ],

                "state_dict":
                    latent_decoder.state_dict(),

                "input_dim":
                    LATENT_DIM,

                "hidden_dim":
                    HIDDEN_DIM,

                "output_dim":
                    RECONSTRUCTION_DIM
            },
            PHASE10_BEST_DECODER_FILE
        )


    else:

        epochs_without_improvement += 1


    training_history.append(
        {
            "epoch":
                epoch,

            "kl_beta":
                beta,

            "train_total_loss":
                train_metrics[
                    "total_loss"
                ],

            "train_reconstruction_loss":
                train_metrics[
                    "reconstruction_loss"
                ],

            "train_kl_loss":
                train_metrics[
                    "kl_loss"
                ],

            "validation_total_loss":
                validation_metrics[
                    "total_loss"
                ],

            "validation_reconstruction_loss":
                validation_metrics[
                    "reconstruction_loss"
                ],

            "validation_kl_loss":
                validation_metrics[
                    "kl_loss"
                ],

            "best_checkpoint":
                bool(
                    improved
                )
        }
    )


    print(
        f"[EPOCH {epoch:03d}] "
        f"beta={beta:.6f} | "
        f"train_total={train_metrics['total_loss']:.6f} | "
        f"train_recon={train_metrics['reconstruction_loss']:.6f} | "
        f"train_KL={train_metrics['kl_loss']:.6f} | "
        f"val_total={validation_metrics['total_loss']:.6f} | "
        f"val_recon={validation_metrics['reconstruction_loss']:.6f} | "
        f"val_KL={validation_metrics['kl_loss']:.6f}"
        +
        (
            " | BEST"
            if improved
            else ""
        )
    )


    if (
        epochs_without_improvement
        >=
        EARLY_STOPPING_PATIENCE
    ):

        print(
            f"\n[EARLY STOP] No validation improvement for "
            f"{EARLY_STOPPING_PATIENCE} epochs."
        )

        break


training_duration_seconds = (
    time.time()
    -
    training_start_time
)


epochs_completed = len(
    training_history
)


training_history_df = pd.DataFrame(
    training_history
)


training_history_df.to_csv(
    PHASE10_TRAINING_HISTORY_FILE,
    index=False,
    encoding="utf-8"
)


print(
    f"\n[PASS] Training completed after "
    f"{epochs_completed} epochs."
)

print(
    f"[BEST] Best validation epoch : {best_epoch}"
)

print(
    f"[BEST] Best validation loss  : "
    f"{best_validation_loss:.8f}"
)


# ==============================================================================
# 37. VERIFY PARAMETERS WERE ACTUALLY UPDATED
# ==============================================================================

encoder_parameters_changed = any(
    not torch.equal(
        initial_encoder_state[
            name
        ],
        tensor.detach().cpu()
    )
    for name, tensor
    in encoder.state_dict().items()
)


decoder_parameters_changed = any(
    not torch.equal(
        initial_decoder_state[
            name
        ],
        tensor.detach().cpu()
    )
    for name, tensor
    in latent_decoder.state_dict().items()
)


print("\n" + "=" * 120)
print("PARAMETER-UPDATE VALIDATION")
print("=" * 120)

print(
    f"Encoder parameters changed : "
    f"{encoder_parameters_changed}"
)

print(
    f"Decoder parameters changed : "
    f"{decoder_parameters_changed}"
)


# ==============================================================================
# 38. RELOAD BEST CHECKPOINT
# ==============================================================================

best_encoder_checkpoint = torch.load(
    PHASE10_BEST_ENCODER_FILE,
    map_location=DEVICE,
    weights_only=False
)


best_decoder_checkpoint = torch.load(
    PHASE10_BEST_DECODER_FILE,
    map_location=DEVICE,
    weights_only=False
)


best_encoder = GeometryAwareVariationalEncoder(
    node_feature_dim=NODE_FEATURE_DIM,
    edge_feature_dim=EDGE_FEATURE_DIM,
    hidden_dim=HIDDEN_DIM,
    latent_dim=LATENT_DIM,
    num_layers=NUM_EGNN_LAYERS
).to(
    DEVICE
)


best_decoder = LatentReconstructionDecoder(
    latent_dim=LATENT_DIM,
    hidden_dim=HIDDEN_DIM,
    output_dim=RECONSTRUCTION_DIM
).to(
    DEVICE
)


best_encoder.load_state_dict(
    best_encoder_checkpoint[
        "state_dict"
    ]
)


best_decoder.load_state_dict(
    best_decoder_checkpoint[
        "state_dict"
    ]
)


best_encoder.eval()

best_decoder.eval()


best_checkpoint_reload_success = True


print(
    "\n[PASS] Best validation checkpoint reloaded successfully."
)


# ==============================================================================
# 39. EVALUATE TRAIN + VALIDATION USING BEST MODEL
# ==============================================================================

FINAL_BETA = BETA_KL_MAX


best_train_metrics = evaluate_loss(
    encoder=best_encoder,
    decoder=best_decoder,
    loader=DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        collate_fn=phase10_collate
    ),
    beta=FINAL_BETA
)


best_validation_metrics = evaluate_loss(
    encoder=best_encoder,
    decoder=best_decoder,
    loader=validation_loader,
    beta=FINAL_BETA
)


# ==============================================================================
# 40. ONLY NOW CREATE TEST DATASET + TEST LOADER
# ==============================================================================

test_dataset = Phase10GraphDataset(
    D_test
)


test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=phase10_collate
)


print(
    "\n[PASS] Best model selected using validation data."
)

print(
    "[INFO] Test corpus is now being evaluated for the first time "
    "in Phase 10."
)


best_test_metrics = evaluate_loss(
    encoder=best_encoder,
    decoder=best_decoder,
    loader=test_loader,
    beta=FINAL_BETA
)


# ==============================================================================
# 41. DETERMINISTIC LATENT + RECONSTRUCTION EXTRACTION
# ==============================================================================

def extract_latents_and_predictions(
    encoder,
    decoder,
    dataset
):

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        collate_fn=phase10_collate
    )


    latent_rows = []

    normalized_prediction_batches = []

    normalized_target_batches = []

    raw_prediction_batches = []

    raw_target_batches = []

    mu_batches = []

    logvar_batches = []


    encoder.eval()

    decoder.eval()


    with torch.no_grad():

        for batch in loader:

            batch = move_batch_to_device(
                batch
            )


            output = encoder(
                x=batch[
                    "x"
                ],
                pos=batch[
                    "pos"
                ],
                edge_index=batch[
                    "edge_index"
                ],
                edge_attr=batch[
                    "edge_attr"
                ],
                batch=batch[
                    "batch"
                ],
                deterministic=True
            )


            # Deterministic representation.
            # z = mu

            mu = output[
                "mu"
            ]


            logvar = output[
                "logvar"
            ]


            z = output[
                "z"
            ]


            prediction_normalized = decoder(
                z
            )


            prediction_raw = (
                prediction_normalized
                *
                target_std_safe.to(
                    DEVICE
                )
                +
                target_mean.to(
                    DEVICE
                )
            )


            for row_index in range(
                z.shape[
                    0
                ]
            ):

                latent_row = {
                    "graph_id":
                        batch[
                            "graph_ids"
                        ][row_index],

                    "drug_id":
                        batch[
                            "drug_ids"
                        ][row_index],

                    "drug_name":
                        batch[
                            "drug_names"
                        ][row_index]
                }


                for latent_index in range(
                    LATENT_DIM
                ):

                    latent_row[
                        f"z_{latent_index}"
                    ] = float(
                        z[
                            row_index,
                            latent_index
                        ].item()
                    )


                latent_rows.append(
                    latent_row
                )


            normalized_prediction_batches.append(
                prediction_normalized.cpu()
            )


            normalized_target_batches.append(
                batch[
                    "target"
                ].cpu()
            )


            raw_prediction_batches.append(
                prediction_raw.cpu()
            )


            raw_target_batches.append(
                batch[
                    "raw_target"
                ].cpu()
            )


            mu_batches.append(
                mu.cpu()
            )


            logvar_batches.append(
                logvar.cpu()
            )


    return {
        "latent_dataframe":
            pd.DataFrame(
                latent_rows
            ),

        "normalized_prediction":
            torch.cat(
                normalized_prediction_batches,
                dim=0
            ),

        "normalized_target":
            torch.cat(
                normalized_target_batches,
                dim=0
            ),

        "raw_prediction":
            torch.cat(
                raw_prediction_batches,
                dim=0
            ),

        "raw_target":
            torch.cat(
                raw_target_batches,
                dim=0
            ),

        "mu":
            torch.cat(
                mu_batches,
                dim=0
            ),

        "logvar":
            torch.cat(
                logvar_batches,
                dim=0
            )
    }


train_results = extract_latents_and_predictions(
    best_encoder,
    best_decoder,
    train_dataset
)


validation_results = extract_latents_and_predictions(
    best_encoder,
    best_decoder,
    validation_dataset
)


test_results = extract_latents_and_predictions(
    best_encoder,
    best_decoder,
    test_dataset
)


# ==============================================================================
# 42. SAVE DETERMINISTIC LATENT VECTORS
# ==============================================================================

train_results[
    "latent_dataframe"
].to_csv(
    PHASE10_TRAIN_LATENT_FILE,
    index=False,
    encoding="utf-8"
)


validation_results[
    "latent_dataframe"
].to_csv(
    PHASE10_VALIDATION_LATENT_FILE,
    index=False,
    encoding="utf-8"
)


test_results[
    "latent_dataframe"
].to_csv(
    PHASE10_TEST_LATENT_FILE,
    index=False,
    encoding="utf-8"
)


print(
    "\n[PASS] Deterministic μ-based latent representations saved."
)


# ==============================================================================
# 43. RECONSTRUCTION METRICS
# ==============================================================================

def reconstruction_metrics(
    split_name,
    results
):

    normalized_prediction = results[
        "normalized_prediction"
    ]


    normalized_target = results[
        "normalized_target"
    ]


    raw_prediction = results[
        "raw_prediction"
    ]


    raw_target = results[
        "raw_target"
    ]


    normalized_mse = float(
        torch.mean(
            (
                normalized_prediction
                -
                normalized_target
            )
            **
            2
        ).item()
    )


    normalized_mae = float(
        torch.mean(
            torch.abs(
                normalized_prediction
                -
                normalized_target
            )
        ).item()
    )


    raw_mse = float(
        torch.mean(
            (
                raw_prediction
                -
                raw_target
            )
            **
            2
        ).item()
    )


    raw_mae = float(
        torch.mean(
            torch.abs(
                raw_prediction
                -
                raw_target
            )
        ).item()
    )


    return {
        "split":
            split_name,

        "graphs":
            int(
                raw_target.shape[
                    0
                ]
            ),

        "normalized_target_mse":
            normalized_mse,

        "normalized_target_mae":
            normalized_mae,

        "raw_target_mse":
            raw_mse,

        "raw_target_mae":
            raw_mae
    }


reconstruction_metrics_rows = [
    reconstruction_metrics(
        "TRAIN",
        train_results
    ),

    reconstruction_metrics(
        "VALIDATION",
        validation_results
    ),

    reconstruction_metrics(
        "TEST",
        test_results
    )
]


reconstruction_metrics_df = pd.DataFrame(
    reconstruction_metrics_rows
)


reconstruction_metrics_df.to_csv(
    PHASE10_RECONSTRUCTION_METRICS_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 44. LATENT QC
# ==============================================================================

def latent_qc(
    split_name,
    results
):

    mu = results[
        "mu"
    ].float()


    logvar = results[
        "logvar"
    ].float()


    finite = bool(
        torch.isfinite(
            mu
        ).all()
    )


    latent_variance = torch.var(
        mu,
        dim=0,
        unbiased=False
    )


    near_zero_dimensions = int(
        (
            latent_variance
            <
            LATENT_VARIANCE_THRESHOLD
        ).sum().item()
    )


    mu_numpy = mu.numpy()


    unique_latents = np.unique(
        mu_numpy,
        axis=0
    )


    duplicate_latent_vectors = (
        mu_numpy.shape[
            0
        ]
        -
        unique_latents.shape[
            0
        ]
    )


    mean_kl = float(
        kl_divergence(
            mu,
            logvar
        ).item()
    )


    return {
        "split":
            split_name,

        "latent_vectors":
            int(
                mu.shape[
                    0
                ]
            ),

        "latent_dimension":
            int(
                mu.shape[
                    1
                ]
            ),

        "finite_latents":
            finite,

        "latent_mean":
            float(
                mu.mean().item()
            ),

        "latent_std":
            float(
                mu.std(
                    unbiased=False
                ).item()
            ),

        "minimum_dimension_variance":
            float(
                latent_variance.min().item()
            ),

        "maximum_dimension_variance":
            float(
                latent_variance.max().item()
            ),

        "mean_dimension_variance":
            float(
                latent_variance.mean().item()
            ),

        "near_zero_variance_dimensions":
            near_zero_dimensions,

        "duplicate_latent_vectors":
            int(
                duplicate_latent_vectors
            ),

        "max_absolute_latent_value":
            float(
                torch.abs(
                    mu
                ).max().item()
            ),

        "mean_kl_divergence":
            mean_kl
    }


latent_qc_rows = [
    latent_qc(
        "TRAIN",
        train_results
    ),

    latent_qc(
        "VALIDATION",
        validation_results
    ),

    latent_qc(
        "TEST",
        test_results
    )
]


latent_qc_df = pd.DataFrame(
    latent_qc_rows
)


latent_qc_df.to_csv(
    PHASE10_LATENT_QC_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 45. POSTERIOR-COLLAPSE DIAGNOSTIC
# ==============================================================================

train_latent_qc = latent_qc_rows[
    0
]


posterior_collapse_warning = bool(

    train_latent_qc[
        "near_zero_variance_dimensions"
    ]
    >=
    (
        LATENT_DIM
        /
        2
    )

    or

    train_latent_qc[
        "mean_kl_divergence"
    ]
    <
    POSTERIOR_KL_WARNING_THRESHOLD
)


# Diagnostic only — not proof of collapse.

print("\n" + "=" * 120)
print("LATENT REPRESENTATION QC")
print("=" * 120)


print(
    f"Train latent vectors             : "
    f"{train_latent_qc['latent_vectors']}"
)

print(
    f"Latent dimension                 : "
    f"{train_latent_qc['latent_dimension']}"
)

print(
    f"Finite latent values             : "
    f"{train_latent_qc['finite_latents']}"
)

print(
    f"Latent mean                      : "
    f"{train_latent_qc['latent_mean']:.6f}"
)

print(
    f"Latent standard deviation        : "
    f"{train_latent_qc['latent_std']:.6f}"
)

print(
    f"Minimum dimension variance       : "
    f"{train_latent_qc['minimum_dimension_variance']:.8e}"
)

print(
    f"Maximum dimension variance       : "
    f"{train_latent_qc['maximum_dimension_variance']:.8e}"
)

print(
    f"Near-zero variance dimensions    : "
    f"{train_latent_qc['near_zero_variance_dimensions']}"
)

print(
    f"Duplicate latent vectors         : "
    f"{train_latent_qc['duplicate_latent_vectors']}"
)

print(
    f"Mean KL divergence               : "
    f"{train_latent_qc['mean_kl_divergence']:.8f}"
)

print(
    f"Posterior-collapse warning       : "
    f"{posterior_collapse_warning}"
)


# ==============================================================================
# 46. RECONSTRUCTION RESULTS
# ==============================================================================

print("\n" + "=" * 120)
print("RECONSTRUCTION PERFORMANCE")
print("=" * 120)


print(
    reconstruction_metrics_df.to_string(
        index=False
    )
)


# ==============================================================================
# 47. SOURCE PRESERVATION
# ==============================================================================

phase8_sources_preserved = all(
    [
        D_train.equals(
            D_train_original
        ),

        D_validation.equals(
            D_validation_original
        ),

        D_test.equals(
            D_test_original
        ),

        D_fisher_reference.equals(
            D_fisher_original
        )
    ]
)


# ==============================================================================
# 48. LATENT OUTPUT VALIDATION
# ==============================================================================

train_latent_count_correct = (
    len(
        train_results[
            "latent_dataframe"
        ]
    )
    ==
    len(
        D_train
    )
)


validation_latent_count_correct = (
    len(
        validation_results[
            "latent_dataframe"
        ]
    )
    ==
    len(
        D_validation
    )
)


test_latent_count_correct = (
    len(
        test_results[
            "latent_dataframe"
        ]
    )
    ==
    len(
        D_test
    )
)


all_latents_finite = all(
    row[
        "finite_latents"
    ]
    for row in latent_qc_rows
)


latent_dimension_correct = all(
    row[
        "latent_dimension"
    ]
    ==
    LATENT_DIM
    for row in latent_qc_rows
)


# ==============================================================================
# 49. TRAINING SUMMARY
# ==============================================================================

best_history_row = (
    training_history_df.loc[
        training_history_df[
            "epoch"
        ]
        ==
        best_epoch
    ]
    .iloc[
        0
    ]
)


test_reconstruction_row = (
    reconstruction_metrics_df[
        reconstruction_metrics_df[
            "split"
        ]
        ==
        "TEST"
    ]
    .iloc[
        0
    ]
)


summary = {
    "phase":
        "Phase 10 — Baseline Variational Latent Representation Learning",

    "generated_at":
        datetime.now().isoformat(),

    "model_status":
        "TRAINED_BASELINE_GEOMETRY_AWARE_VARIATIONAL_ENCODER",

    "training_graphs":
        len(
            D_train
        ),

    "validation_graphs":
        len(
            D_validation
        ),

    "test_graphs":
        len(
            D_test
        ),

    "fisher_reference_graphs":
        len(
            D_fisher_reference
        ),

    "epochs_completed":
        epochs_completed,

    "best_epoch":
        best_epoch,

    "best_train_reconstruction_loss":
        float(
            best_history_row[
                "train_reconstruction_loss"
            ]
        ),

    "best_validation_reconstruction_loss":
        float(
            best_history_row[
                "validation_reconstruction_loss"
            ]
        ),

    "best_validation_total_loss":
        float(
            best_history_row[
                "validation_total_loss"
            ]
        ),

    "best_validation_kl_loss":
        float(
            best_history_row[
                "validation_kl_loss"
            ]
        ),

    "test_total_loss":
        float(
            best_test_metrics[
                "total_loss"
            ]
        ),

    "test_kl_loss":
        float(
            best_test_metrics[
                "kl_loss"
            ]
        ),

    "test_reconstruction_mse_normalized":
        float(
            test_reconstruction_row[
                "normalized_target_mse"
            ]
        ),

    "test_reconstruction_mae_normalized":
        float(
            test_reconstruction_row[
                "normalized_target_mae"
            ]
        ),

    "test_reconstruction_mse_raw":
        float(
            test_reconstruction_row[
                "raw_target_mse"
            ]
        ),

    "test_reconstruction_mae_raw":
        float(
            test_reconstruction_row[
                "raw_target_mae"
            ]
        ),

    "latent_dimension":
        LATENT_DIM,

    "train_latent_vectors":
        len(
            D_train
        ),

    "validation_latent_vectors":
        len(
            D_validation
        ),

    "test_latent_vectors":
        len(
            D_test
        ),

    "finite_latent_vectors":
        all_latents_finite,

    "train_latent_mean":
        train_latent_qc[
            "latent_mean"
        ],

    "train_latent_std":
        train_latent_qc[
            "latent_std"
        ],

    "minimum_latent_variance":
        train_latent_qc[
            "minimum_dimension_variance"
        ],

    "maximum_latent_variance":
        train_latent_qc[
            "maximum_dimension_variance"
        ],

    "near_zero_variance_dimensions":
        train_latent_qc[
            "near_zero_variance_dimensions"
        ],

    "duplicate_train_latent_vectors":
        train_latent_qc[
            "duplicate_latent_vectors"
        ],

    "posterior_collapse_warning":
        posterior_collapse_warning,

    "encoder_parameters_changed":
        encoder_parameters_changed,

    "decoder_parameters_changed":
        decoder_parameters_changed,

    "best_checkpoint_reloadable":
        best_checkpoint_reload_success,

    "phase8_sources_preserved":
        phase8_sources_preserved,

    "test_used_for_model_selection":
        False,

    "fisher_used_for_parameter_updates":
        False,

    "fisher_information_calculated":
        False,

    "ewc_applied":
        False,

    "candidate_generation_performed":
        False,

    "training_duration_seconds":
        training_duration_seconds
}


# ==============================================================================
# 50. COMPLETION VALIDATION
# ==============================================================================

completion_checks = {
    "Phase-9 initialized encoder loaded":
        True,

    "Training corpus used for optimization":
        len(
            D_train
        ) > 0,

    "Validation corpus used for model selection":
        len(
            D_validation
        ) > 0,

    "Test excluded from parameter optimization":
        True,

    "Test excluded from checkpoint selection":
        True,

    "Fisher reference excluded from training":
        True,

    "Phase-9 normalization reused":
        True,

    "Reconstruction target defined deterministically":
        RECONSTRUCTION_DIM
        ==
        38,

    "Target normalization fitted on training only":
        True,

    "Latent reconstruction decoder implemented":
        isinstance(
            latent_decoder,
            LatentReconstructionDecoder
        ),

    "Stochastic reparameterization used during training":
        True,

    "Reconstruction loss calculated":
        True,

    "KL-divergence calculated":
        True,

    "Total variational loss calculated":
        True,

    "Encoder parameters updated":
        encoder_parameters_changed,

    "Decoder parameters updated":
        decoder_parameters_changed,

    "Validation monitored":
        len(
            training_history
        ) > 0,

    "Early stopping/best-checkpoint logic implemented":
        True,

    "Best encoder checkpoint saved":
        PHASE10_BEST_ENCODER_FILE.exists(),

    "Best decoder checkpoint saved":
        PHASE10_BEST_DECODER_FILE.exists(),

    "Best checkpoints reload successfully":
        best_checkpoint_reload_success,

    "Deterministic latent extraction uses z = mu":
        True,

    "Latent dimension equals 32":
        latent_dimension_correct,

    "Training latent count correct":
        train_latent_count_correct,

    "Validation latent count correct":
        validation_latent_count_correct,

    "Test latent count correct":
        test_latent_count_correct,

    "Train latent file saved":
        PHASE10_TRAIN_LATENT_FILE.exists(),

    "Validation latent file saved":
        PHASE10_VALIDATION_LATENT_FILE.exists(),

    "Test latent file saved":
        PHASE10_TEST_LATENT_FILE.exists(),

    "All latent vectors finite":
        all_latents_finite,

    "Latent variance evaluated":
        True,

    "Posterior-collapse diagnostic performed":
        True,

    "Reconstruction metrics reported":
        PHASE10_RECONSTRUCTION_METRICS_FILE.exists(),

    "Phase-8 split datasets preserved":
        phase8_sources_preserved,

    "No Fisher information calculated":
        True,

    "No EWC applied":
        True,

    "No candidate molecules generated":
        True
}


# ==============================================================================
# 51. SAVE SUMMARY FILES
# ==============================================================================

summary[
    "completion_checks"
] = completion_checks


with open(
    PHASE10_SUMMARY_JSON_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        summary,
        file,
        indent=4,
        ensure_ascii=False,
        default=str
    )


summary_rows = []


for key, value in summary.items():

    if key == "completion_checks":

        for check_name, status in value.items():

            summary_rows.append(
                {
                    "metric":
                        f"completion.{check_name}",

                    "value":
                        status
                }
            )

    else:

        summary_rows.append(
            {
                "metric":
                    key,

                "value":
                    value
            }
        )


phase10_summary_df = pd.DataFrame(
    summary_rows
)


phase10_summary_df.to_csv(
    PHASE10_SUMMARY_CSV_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 52. DISPLAY COMPLETION VALIDATION
# ==============================================================================

print("\n" + "=" * 120)
print("PHASE 10 COMPLETION VALIDATION")
print("=" * 120)


all_completion_checks_passed = True


for check_name, status in completion_checks.items():

    label = (
        "PASS"
        if bool(
            status
        )
        else "FAIL"
    )


    print(
        f"[{label:<4}] {check_name}"
    )


    if not bool(
        status
    ):

        all_completion_checks_passed = False


# ==============================================================================
# 53. UPDATE PROJECT CONFIG
# ==============================================================================

PHASE10_DATASETS = {
    "training_config":
        str(
            PHASE10_TRAINING_CONFIG_FILE
        ),

    "training_history":
        str(
            PHASE10_TRAINING_HISTORY_FILE
        ),

    "best_encoder":
        str(
            PHASE10_BEST_ENCODER_FILE
        ),

    "best_latent_decoder":
        str(
            PHASE10_BEST_DECODER_FILE
        ),

    "reconstruction_target_schema":
        str(
            PHASE10_TARGET_SCHEMA_FILE
        ),

    "reconstruction_normalization":
        str(
            PHASE10_TARGET_NORMALIZATION_FILE
        ),

    "reconstruction_metrics":
        str(
            PHASE10_RECONSTRUCTION_METRICS_FILE
        ),

    "train_latent":
        str(
            PHASE10_TRAIN_LATENT_FILE
        ),

    "validation_latent":
        str(
            PHASE10_VALIDATION_LATENT_FILE
        ),

    "test_latent":
        str(
            PHASE10_TEST_LATENT_FILE
        ),

    "latent_qc":
        str(
            PHASE10_LATENT_QC_FILE
        ),

    "summary_csv":
        str(
            PHASE10_SUMMARY_CSV_FILE
        ),

    "summary_json":
        str(
            PHASE10_SUMMARY_JSON_FILE
        ),

    "best_epoch":
        best_epoch,

    "model_status":
        "TRAINED_BASELINE_GEOMETRY_AWARE_VARIATIONAL_ENCODER"
}


DRUGBANK_CONFIG[
    "phase10_datasets"
] = PHASE10_DATASETS


# ==============================================================================
# 54. FINAL SUMMARY
# ==============================================================================

print("\n" + "=" * 120)
print("PHASE 10 SUMMARY")
print("=" * 120)


print(
    f"Training Graphs                 : "
    f"{len(D_train)}"
)

print(
    f"Validation Graphs               : "
    f"{len(D_validation)}"
)

print(
    f"Test Graphs                     : "
    f"{len(D_test)}"
)

print(
    f"Fisher Reference Graphs         : "
    f"{len(D_fisher_reference)}"
)

print(
    f"Epochs Completed                : "
    f"{epochs_completed}"
)

print(
    f"Best Epoch                      : "
    f"{best_epoch}"
)

print(
    f"Best Validation Total Loss      : "
    f"{best_validation_loss:.8f}"
)

print(
    f"Best Validation Recon Loss      : "
    f"{best_validation_metrics['reconstruction_loss']:.8f}"
)

print(
    f"Test Reconstruction MSE (norm.) : "
    f"{test_reconstruction_row['normalized_target_mse']:.8f}"
)

print(
    f"Test Reconstruction MAE (norm.) : "
    f"{test_reconstruction_row['normalized_target_mae']:.8f}"
)

print(
    f"Test KL Loss                    : "
    f"{best_test_metrics['kl_loss']:.8f}"
)

print(
    f"Latent Dimension                : "
    f"{LATENT_DIM}"
)

print(
    f"Training Latent Vectors         : "
    f"{len(D_train)}"
)

print(
    f"Validation Latent Vectors       : "
    f"{len(D_validation)}"
)

print(
    f"Test Latent Vectors             : "
    f"{len(D_test)}"
)

print(
    f"All Latents Finite              : "
    f"{all_latents_finite}"
)

print(
    f"Near-Zero Latent Dimensions     : "
    f"{train_latent_qc['near_zero_variance_dimensions']}"
)

print(
    f"Duplicate Train Latents         : "
    f"{train_latent_qc['duplicate_latent_vectors']}"
)

print(
    f"Posterior Collapse Warning      : "
    f"{posterior_collapse_warning}"
)

print(
    f"Encoder Parameters Updated      : "
    f"{encoder_parameters_changed}"
)

print(
    f"Decoder Parameters Updated      : "
    f"{decoder_parameters_changed}"
)

print(
    f"Best Checkpoint Reloadable      : "
    f"{best_checkpoint_reload_success}"
)

print(
    f"Phase-8 Sources Preserved       : "
    f"{phase8_sources_preserved}"
)

print(
    f"Fisher Information Computed     : False"
)

print(
    f"EWC Applied                     : False"
)

print(
    f"Candidate Generation Performed  : False"
)

print(
    f"Model Status                    : "
    f"TRAINED BASELINE"
)


print("\nSaved Files:")

print(
    f"  Training Config          : "
    f"{PHASE10_TRAINING_CONFIG_FILE}"
)

print(
    f"  Training History         : "
    f"{PHASE10_TRAINING_HISTORY_FILE}"
)

print(
    f"  Best Encoder             : "
    f"{PHASE10_BEST_ENCODER_FILE}"
)

print(
    f"  Best Latent Decoder      : "
    f"{PHASE10_BEST_DECODER_FILE}"
)

print(
    f"  Reconstruction Schema    : "
    f"{PHASE10_TARGET_SCHEMA_FILE}"
)

print(
    f"  Reconstruction Norm      : "
    f"{PHASE10_TARGET_NORMALIZATION_FILE}"
)

print(
    f"  Reconstruction Metrics   : "
    f"{PHASE10_RECONSTRUCTION_METRICS_FILE}"
)

print(
    f"  Train Latent             : "
    f"{PHASE10_TRAIN_LATENT_FILE}"
)

print(
    f"  Validation Latent        : "
    f"{PHASE10_VALIDATION_LATENT_FILE}"
)

print(
    f"  Test Latent              : "
    f"{PHASE10_TEST_LATENT_FILE}"
)

print(
    f"  Latent QC                : "
    f"{PHASE10_LATENT_QC_FILE}"
)

print(
    f"  Summary CSV              : "
    f"{PHASE10_SUMMARY_CSV_FILE}"
)

print(
    f"  Summary JSON             : "
    f"{PHASE10_SUMMARY_JSON_FILE}"
)


print("-" * 120)


if all_completion_checks_passed:

    print(
        "[STATUS] PHASE 10 COMPLETED SUCCESSFULLY."
    )

    print(
        "[READY]  Geometry-aware variational encoder has been "
        "trained on the baseline DrugBank molecular graph corpus."
    )

    print(
        "[READY]  A continuous 32-dimensional baseline structural "
        "molecular representation has been established."
    )

    if posterior_collapse_warning:

        print(
            "[WARNING] Posterior-collapse diagnostic triggered. "
            "Review KL and latent-variance results before freezing Phase 10."
        )

    else:

        print(
            "[PASS] No posterior-collapse warning triggered."
        )

    print(
        "[IMPORTANT] No binding-affinity, therapeutic-efficacy, "
        "or biological-optimality claim is made."
    )

    print(
        "[NEXT]   Phase 11 — Molecular Decoder / "
        "Generative Representation Development"
    )


else:

    print(
        "[STATUS] PHASE 10 COMPLETED WITH ONE OR MORE "
        "FAILED VALIDATION CHECKS."
    )

    print(
        "[ACTION] Review failed checks before proceeding "
        "to Phase 11."
    )


print("=" * 120)

PHASE 10 — BASELINE VARIATIONAL LATENT REPRESENTATION LEARNING

[DEVICE] Device                : cpu
[CONFIG] Random Seed          : 42
[CONFIG] Hidden Dimension     : 64
[CONFIG] Latent Dimension     : 32
[CONFIG] EGNN Layers          : 2
[CONFIG] Batch Size           : 16
[CONFIG] Learning Rate        : 0.001
[CONFIG] Maximum Epochs       : 100
[CONFIG] Early Stop Patience  : 12
[CONFIG] Maximum KL Beta      : 0.001
[CONFIG] KL Warm-up Epochs    : 20

[PASS] Required Phase-8 and Phase-9 files found.

PHASE-8 DATASET INPUT
Training graphs         : 613
Validation graphs       : 76
Test graphs             : 78
Fisher reference graphs : 128

[PASS] Train/validation/test split remains leakage-free.
[PASS] Fisher reference remains training-only.

[PASS] Phase-7 graph schema loaded.
[INFO] Node feature dimension : 21
[INFO] Edge feature dimension : 14
[PASS] Phase-9 training-only feature normalization loaded.
[PASS] Phase-9 encoder configuration verified.

RECONSTRUCTION TARGET DEFINITION


# Phase 11 — Molecular Decoder / Candidate Generation Module

**Objective:** Train a lightweight, latent-conditioned SELFIES autoregressive sequence decoder using the frozen 32-dimensional Phase 10 molecular representations ($z = \boldsymbol{\mu}$) to enable candidate molecule generation, reconstruction, and chemical quality control.

* **Architecture & Generative Formulation:**
  * **Conditioning Source:** Uses deterministic Phase 10 representations ($z = \boldsymbol{\mu} \in \mathbb{R}^{32}$) without modifying or fine-tuning the Phase 10 EGNN encoder.
  * **Sequence Modeling:** A 2-layer GRU decoder (`hidden_dim = 128`, `embed_dim = 64`) initialized via a linear projection of latent vector $z$ ($h_0 = \tanh(W_z z)$).
  * **Representation:** SELFIES sequence decoding to eliminate unclosed ring and branch syntax errors common to raw SMILES.
  * **Supervision:** Teacher-forced cross-entropy loss with padding tokens (`<PAD>`) masked out during optimization.

* **Experimental & Data Isolation Rules:**
  * **Vocabulary Isolation:** Derived strictly from `D_train_seq` molecules; unseen validation/test tokens map to `<UNK>` to prevent vocabulary leakage.
  * **Dataset Partitioning:** Parameter updates occur strictly on the 613 training molecules; checkpoint selection uses the 76 validation molecules; the 78 test molecules are evaluated only after freezing the best checkpoint.
  * **Scope Discipline:** Focuses strictly on latent-to-structure decoding; no Fisher information calculation, Elastic Weight Consolidation (EWC), or biological activity prediction occurs in this phase.

* **Sampling Modes & Validation Pipeline:**
  * **Reconstruction Testing:** Deterministic greedy decoding ($z = \boldsymbol{\mu}$) to benchmark exact SELFIES and canonical SMILES recovery rates.
  * **Mode A (Latent Perturbation):** Generates candidates around training seeds via localized Gaussian perturbations ($z' = z + \delta$, where $\delta \sim \mathcal{N}(0, \sigma^2 I)$ with $\sigma = 0.25$).
  * **Mode B (Prior Exploration):** Generates exploratory samples directly from standard Gaussian latent space ($z \sim \mathcal{N}(0, I)$).
  * **Chemical QC Pipeline:** Decoded sequences are verified through RDKit parsing, sanitization, explicit valence checks, standardization (cleanup, fragment parent, uncharging), and heavy-atom feasibility gates ($4 \le N_{\text{heavy}} \le 80$).
  * **Novelty & Diversity Assessment:** Evaluates candidate deduplication, Morgan fingerprint diversity ($1 - \text{Tanimoto}$), and novelty relative to the baseline DrugBank standardized corpus.

* **Primary Deliverables:**
  * Trained decoder checkpoint: `phase11_best_generative_decoder.pt`.
  * Token configuration & history: `phase11_selfies_vocabulary.json`, `phase11_decoder_training_history.csv`.
  * Reconstruction benchmarks: `phase11_validation_reconstruction.csv`, `phase11_test_reconstruction.csv`, `phase11_reconstruction_metrics.csv`.
  * Candidate molecule sets: `phase11_generated_candidates_raw.csv`, `phase11_generated_candidates_validated.csv`, `phase11_generated_candidates_unique.csv`, `phase11_generated_candidates_novel.csv`.
  * Audit summaries: `phase11_generation_qc_summary.csv` and `.json`.

In [11]:
# ==============================================================================
# PHASE 11 — REVISED MOLECULAR DECODER / CANDIDATE GENERATION MODULE
# STRONG LATENT CONDITIONING + LATENT-SENSITIVITY VALIDATION
# ==============================================================================
#
# PURPOSE
# -------
# Train a lightweight SELFIES molecular decoder conditioned on the learned
# Phase-10 latent representation.
#
# MAJOR REVISION FROM PREVIOUS PHASE-11 VERSION
# ---------------------------------------------
#
# Previous decoder:
#
#       z -> initial GRU hidden state only
#
# Revised decoder:
#
#       z -> initial GRU hidden state
#       +
#       z -> latent context concatenated with EVERY token embedding
#
# This makes the molecular sequence decoder depend more directly on z
# throughout generation.
#
#
# WORKFLOW
# --------
#
# Phase-10 deterministic latent representation
#
#               z = mu ∈ R^32
#                     |
#            +--------+--------+
#            |                 |
#            v                 v
#      Initial GRU State   Latent Context
#                              |
#                              +-------------------+
#                                                  |
# Token_t -> Token Embedding ----------------------+ 
#                                                  |
#                                                  v
#                                      [Token Embedding || z-context]
#                                                  |
#                                                  v
#                                               GRU
#                                                  |
#                                                  v
#                                         Vocabulary Logits
#                                                  |
#                                                  v
#                                         SELFIES Sequence
#                                                  |
#                                                  v
#                                         SELFIES -> SMILES
#                                                  |
#                                                  v
#                                             RDKit QC
#                                                  |
#                                                  v
#                                      Validated Candidate Set
#
#
# ADDITIONAL SCIENTIFIC VALIDATION
# --------------------------------
#
# 1. Different latent vectors must influence generated sequences.
#
# 2. Latent-vector swapping:
#
#       z_A -> molecule_A
#       z_B -> molecule_B
#
#    If swapping z does not alter output, conditioning may be weak.
#
# 3. Controlled perturbation:
#
#       z
#       z + small noise
#       z + larger noise
#
#    Output changes and molecular similarities are recorded.
#
# 4. Greedy-output diversity across different latent vectors is measured.
#
#
# IMPORTANT
# ---------
# Phase-10 encoder remains frozen.
#
# No:
#   - biological activity prediction
#   - binding-affinity prediction
#   - Fisher information
#   - EWC
#   - UCB
#   - closed-loop optimization
#
# ==============================================================================


# ==============================================================================
# 1. IMPORTS
# ==============================================================================

from pathlib import Path
from datetime import datetime

import json
import random
import time
import sys
import subprocess
import hashlib

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.MolStandardize import rdMolStandardize


# ==============================================================================
# 2. SELFIES DEPENDENCY
# ==============================================================================

try:

    import selfies as sf

except ImportError:

    print(
        "[INFO] SELFIES package not found. Installing selfies..."
    )

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "selfies"
        ]
    )

    import selfies as sf


# ==============================================================================
# 3. REPRODUCIBILITY
# ==============================================================================

RANDOM_SEED = 42

random.seed(
    RANDOM_SEED
)

np.random.seed(
    RANDOM_SEED
)

torch.manual_seed(
    RANDOM_SEED
)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        RANDOM_SEED
    )


DEVICE = torch.device(
    "cpu"
)


# ==============================================================================
# 4. VERIFY PROJECT STATE
# ==============================================================================

required_variables = [
    "PROCESSED_DIR",
    "DRUGBANK_CONFIG"
]


missing_variables = [
    variable
    for variable in required_variables
    if variable not in globals()
]


if missing_variables:

    raise RuntimeError(
        "Earlier phases must be executed before revised Phase 11.\n"
        f"Missing variables: {missing_variables}"
    )


# ==============================================================================
# 5. CONFIGURATION
# ==============================================================================

LATENT_DIM = 32

TOKEN_EMBED_DIM = 64

LATENT_CONTEXT_DIM = 64

DECODER_HIDDEN_DIM = 128

NUM_GRU_LAYERS = 2


BATCH_SIZE = 32

LEARNING_RATE = 1e-3

MAX_EPOCHS = 100

EARLY_STOPPING_PATIENCE = 12


MIN_HEAVY_ATOMS = 4

MAX_HEAVY_ATOMS = 80


# ------------------------------------------------------------------------------
# Generation configuration
# ------------------------------------------------------------------------------

NUM_GENERATION_SEEDS = 100

CANDIDATES_PER_SEED = 5

NUM_PRIOR_SAMPLES = 100


LATENT_PERTURBATION_SCALE = 0.25

GENERATION_TEMPERATURE = 1.0

TOP_K = 10


# ------------------------------------------------------------------------------
# Latent-conditioning diagnostic configuration
# ------------------------------------------------------------------------------

NUM_LATENT_DIAGNOSTIC_MOLECULES = 50

SMALL_PERTURBATION_SCALE = 0.10

LARGE_PERTURBATION_SCALE = 0.50


# Diagnostic thresholds.
#
# These are operational QC thresholds, NOT universal scientific constants.

MIN_LATENT_SWAP_CHANGE_RATE = 0.20

MIN_GREEDY_UNIQUE_FRACTION = 0.10


print("=" * 124)
print("PHASE 11 — REVISED LATENT-CONDITIONED MOLECULAR GENERATION")
print("=" * 124)

print(f"\n[DEVICE] Device                    : {DEVICE}")
print(f"[CONFIG] Random Seed              : {RANDOM_SEED}")
print(f"[CONFIG] Latent Dimension         : {LATENT_DIM}")
print(f"[CONFIG] Token Embedding Dim      : {TOKEN_EMBED_DIM}")
print(f"[CONFIG] Latent Context Dim       : {LATENT_CONTEXT_DIM}")
print(f"[CONFIG] Decoder Hidden Dim       : {DECODER_HIDDEN_DIM}")
print(f"[CONFIG] GRU Layers               : {NUM_GRU_LAYERS}")
print(f"[CONFIG] Batch Size               : {BATCH_SIZE}")
print(f"[CONFIG] Learning Rate            : {LEARNING_RATE}")
print(f"[CONFIG] Maximum Epochs           : {MAX_EPOCHS}")
print(f"[CONFIG] Early Stop Patience      : {EARLY_STOPPING_PATIENCE}")

print(
    "[CONFIG] Conditioning Policy       : "
    "z -> initial hidden state + z-context at every token step"
)


# ==============================================================================
# 6. INPUT FILES
# ==============================================================================

PHASE8_TRAIN_FILE = (
    Path(PROCESSED_DIR)
    / "phase8_train_graphs.csv"
)


PHASE8_VALIDATION_FILE = (
    Path(PROCESSED_DIR)
    / "phase8_validation_graphs.csv"
)


PHASE8_TEST_FILE = (
    Path(PROCESSED_DIR)
    / "phase8_test_graphs.csv"
)


PHASE10_TRAIN_LATENT_FILE = (
    Path(PROCESSED_DIR)
    / "phase10_train_latent.csv"
)


PHASE10_VALIDATION_LATENT_FILE = (
    Path(PROCESSED_DIR)
    / "phase10_validation_latent.csv"
)


PHASE10_TEST_LATENT_FILE = (
    Path(PROCESSED_DIR)
    / "phase10_test_latent.csv"
)


PHASE10_BEST_ENCODER_FILE = (
    Path(PROCESSED_DIR)
    / "phase10_best_encoder.pt"
)


D_CHEM_STANDARDIZED_FILE = (
    Path(PROCESSED_DIR)
    / "D_chem_standardized.csv"
)


required_files = [
    PHASE8_TRAIN_FILE,
    PHASE8_VALIDATION_FILE,
    PHASE8_TEST_FILE,
    PHASE10_TRAIN_LATENT_FILE,
    PHASE10_VALIDATION_LATENT_FILE,
    PHASE10_TEST_LATENT_FILE,
    PHASE10_BEST_ENCODER_FILE,
    D_CHEM_STANDARDIZED_FILE
]


missing_files = [
    str(
        file_path
    )
    for file_path
    in required_files
    if not file_path.exists()
]


if missing_files:

    raise FileNotFoundError(
        "Required Phase-8/10 files are missing:\n"
        +
        "\n".join(
            missing_files
        )
    )


print(
    "\n[PASS] Required Phase-8 and Phase-10 files found."
)


# ==============================================================================
# 7. LOAD PHASE-8 SPLITS
# ==============================================================================

D_train_manifest = pd.read_csv(
    PHASE8_TRAIN_FILE,
    dtype=str,
    keep_default_na=False
)


D_validation_manifest = pd.read_csv(
    PHASE8_VALIDATION_FILE,
    dtype=str,
    keep_default_na=False
)


D_test_manifest = pd.read_csv(
    PHASE8_TEST_FILE,
    dtype=str,
    keep_default_na=False
)


D_train_manifest_original = (
    D_train_manifest.copy(
        deep=True
    )
)


D_validation_manifest_original = (
    D_validation_manifest.copy(
        deep=True
    )
)


D_test_manifest_original = (
    D_test_manifest.copy(
        deep=True
    )
)


# ==============================================================================
# 8. VERIFY SPLIT RELATIONSHIPS
# ==============================================================================

train_ids = set(
    D_train_manifest[
        "graph_id"
    ].astype(str)
)


validation_ids = set(
    D_validation_manifest[
        "graph_id"
    ].astype(str)
)


test_ids = set(
    D_test_manifest[
        "graph_id"
    ].astype(str)
)


if train_ids & validation_ids:

    raise RuntimeError(
        "Train-validation overlap detected."
    )


if train_ids & test_ids:

    raise RuntimeError(
        "Train-test overlap detected."
    )


if validation_ids & test_ids:

    raise RuntimeError(
        "Validation-test overlap detected."
    )


print(
    "[PASS] Phase-8 molecule-level split remains leakage-free."
)


# ==============================================================================
# 9. LOAD PHASE-10 LATENT VECTORS
# ==============================================================================

D_train_latent = pd.read_csv(
    PHASE10_TRAIN_LATENT_FILE,
    dtype={
        "graph_id": str,
        "drug_id": str
    }
)


D_validation_latent = pd.read_csv(
    PHASE10_VALIDATION_LATENT_FILE,
    dtype={
        "graph_id": str,
        "drug_id": str
    }
)


D_test_latent = pd.read_csv(
    PHASE10_TEST_LATENT_FILE,
    dtype={
        "graph_id": str,
        "drug_id": str
    }
)


latent_columns = [
    f"z_{index}"
    for index in range(
        LATENT_DIM
    )
]


for dataset_name, dataframe in [
    (
        "TRAIN",
        D_train_latent
    ),
    (
        "VALIDATION",
        D_validation_latent
    ),
    (
        "TEST",
        D_test_latent
    )
]:

    missing_columns = [
        column
        for column in latent_columns
        if column not in dataframe.columns
    ]


    if missing_columns:

        raise RuntimeError(
            f"{dataset_name} latent file missing columns: "
            f"{missing_columns}"
        )


    latent_values = (
        dataframe[
            latent_columns
        ]
        .astype(float)
        .to_numpy()
    )


    if not np.isfinite(
        latent_values
    ).all():

        raise RuntimeError(
            f"{dataset_name} contains non-finite latent values."
        )


print(
    "[PASS] Phase-10 finite 32-dimensional latent vectors loaded."
)


# ==============================================================================
# 10. LOAD STANDARDIZED CHEMICAL CORPUS
# ==============================================================================

D_chem_standardized = pd.read_csv(
    D_CHEM_STANDARDIZED_FILE,
    dtype=str,
    keep_default_na=False
)


required_chem_columns = [
    "drug_id",
    "standardized_smiles"
]


for column in required_chem_columns:

    if column not in D_chem_standardized.columns:

        raise RuntimeError(
            f"D_chem_standardized missing {column}."
        )


chemical_lookup = (
    D_chem_standardized[
        [
            "drug_id",
            "standardized_smiles"
        ]
    ]
    .drop_duplicates(
        subset=[
            "drug_id"
        ]
    )
)


print(
    "[PASS] Standardized molecular structures loaded."
)


# ==============================================================================
# 11. LINK LATENT VECTORS TO STRUCTURES
# ==============================================================================

def build_sequence_dataframe(
    manifest,
    latent_dataframe,
    split_name
):

    manifest_columns = [
        "graph_id",
        "drug_id"
    ]


    if "drug_name" in manifest.columns:

        manifest_columns.append(
            "drug_name"
        )


    joined = latent_dataframe.merge(
        manifest[
            manifest_columns
        ],
        on=[
            "graph_id",
            "drug_id"
        ],
        how="inner",
        validate="one_to_one",
        suffixes=(
            "_latent",
            "_manifest"
        )
    )


    joined = joined.merge(
        chemical_lookup,
        on="drug_id",
        how="left",
        validate="many_to_one"
    )


    if len(
        joined
    ) != len(
        manifest
    ):

        raise RuntimeError(
            f"{split_name} linkage failed. "
            f"Expected {len(manifest)}, obtained {len(joined)}."
        )


    missing_smiles = (
        joined[
            "standardized_smiles"
        ]
        .astype(str)
        .str.strip()
        ==
        ""
    )


    if missing_smiles.any():

        raise RuntimeError(
            f"{split_name} contains "
            f"{int(missing_smiles.sum())} missing standardized SMILES."
        )


    joined[
        "split"
    ] = split_name


    return joined


D_train_seq = build_sequence_dataframe(
    D_train_manifest,
    D_train_latent,
    "TRAIN"
)


D_validation_seq = build_sequence_dataframe(
    D_validation_manifest,
    D_validation_latent,
    "VALIDATION"
)


D_test_seq = build_sequence_dataframe(
    D_test_manifest,
    D_test_latent,
    "TEST"
)


print("\n" + "=" * 124)
print("LATENT ↔ MOLECULAR STRUCTURE LINKAGE")
print("=" * 124)

print(
    f"Training pairs   : {len(D_train_seq)}"
)

print(
    f"Validation pairs : {len(D_validation_seq)}"
)

print(
    f"Test pairs       : {len(D_test_seq)}"
)


# ==============================================================================
# 12. CANONICALIZE REFERENCE STRUCTURES
# ==============================================================================

def canonicalize_smiles(
    smiles
):

    try:

        molecule = Chem.MolFromSmiles(
            smiles
        )


        if molecule is None:

            return None


        return Chem.MolToSmiles(
            molecule,
            canonical=True,
            isomericSmiles=True
        )


    except Exception:

        return None


for dataframe in [
    D_train_seq,
    D_validation_seq,
    D_test_seq
]:

    dataframe[
        "reference_canonical_smiles"
    ] = dataframe[
        "standardized_smiles"
    ].apply(
        canonicalize_smiles
    )


if D_train_seq[
    "reference_canonical_smiles"
].isna().any():

    raise RuntimeError(
        "Training molecules contain invalid canonical reference SMILES."
    )


# ==============================================================================
# 13. SMILES -> SELFIES
# ==============================================================================

def smiles_to_selfies(
    smiles
):

    try:

        return sf.encoder(
            smiles
        )

    except Exception:

        return None


sequence_preparation_failures = []


def prepare_selfies_dataframe(
    dataframe
):

    dataframe = dataframe.copy(
        deep=True
    )


    selfies_list = []


    for _, row in dataframe.iterrows():

        encoded = smiles_to_selfies(
            row[
                "reference_canonical_smiles"
            ]
        )


        selfies_list.append(
            encoded
        )


        if encoded is None:

            sequence_preparation_failures.append(
                {
                    "split":
                        row[
                            "split"
                        ],

                    "graph_id":
                        row[
                            "graph_id"
                        ],

                    "drug_id":
                        row[
                            "drug_id"
                        ],

                    "failure_reason":
                        "SELFIES_ENCODING_FAILURE"
                }
            )


    dataframe[
        "reference_selfies"
    ] = selfies_list


    return dataframe


D_train_seq = prepare_selfies_dataframe(
    D_train_seq
)


D_validation_seq = prepare_selfies_dataframe(
    D_validation_seq
)


D_test_seq = prepare_selfies_dataframe(
    D_test_seq
)


if D_train_seq[
    "reference_selfies"
].isna().any():

    raise RuntimeError(
        "Training SELFIES preparation failure detected."
    )


# Validation/test failures remain auditable.

D_validation_seq = (
    D_validation_seq[
        D_validation_seq[
            "reference_selfies"
        ].notna()
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


D_test_seq = (
    D_test_seq[
        D_test_seq[
            "reference_selfies"
        ].notna()
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


print(
    "\n[PASS] Training molecular structures converted to SELFIES."
)

print(
    f"[INFO] Sequence preparation failures : "
    f"{len(sequence_preparation_failures)}"
)


# ==============================================================================
# 14. TOKENIZATION
# ==============================================================================

def tokenize_selfies(
    selfies_string
):

    return list(
        sf.split_selfies(
            selfies_string
        )
    )


for dataframe in [
    D_train_seq,
    D_validation_seq,
    D_test_seq
]:

    dataframe[
        "selfies_tokens"
    ] = dataframe[
        "reference_selfies"
    ].apply(
        tokenize_selfies
    )


# ==============================================================================
# 15. TRAINING-ONLY VOCABULARY
# ==============================================================================

PAD_TOKEN = "<PAD>"

BOS_TOKEN = "<BOS>"

EOS_TOKEN = "<EOS>"

UNK_TOKEN = "<UNK>"


SPECIAL_TOKENS = [
    PAD_TOKEN,
    BOS_TOKEN,
    EOS_TOKEN,
    UNK_TOKEN
]


training_token_set = set()


for token_list in D_train_seq[
    "selfies_tokens"
]:

    training_token_set.update(
        token_list
    )


vocabulary = (
    SPECIAL_TOKENS
    +
    sorted(
        training_token_set
    )
)


token_to_id = {
    token:
        index
    for index, token
    in enumerate(
        vocabulary
    )
}


id_to_token = {
    index:
        token
    for token, index
    in token_to_id.items()
}


PAD_ID = token_to_id[
    PAD_TOKEN
]

BOS_ID = token_to_id[
    BOS_TOKEN
]

EOS_ID = token_to_id[
    EOS_TOKEN
]

UNK_ID = token_to_id[
    UNK_TOKEN
]


VOCAB_SIZE = len(
    vocabulary
)


# ==============================================================================
# 16. SEQUENCE-LENGTH POLICY
# ==============================================================================

training_lengths = (
    D_train_seq[
        "selfies_tokens"
    ]
    .apply(
        len
    )
)


MAX_SEQUENCE_LENGTH = (
    int(
        training_lengths.max()
    )
    +
    2
)


print("\n" + "=" * 124)
print("SELFIES VOCABULARY / SEQUENCE CONFIGURATION")
print("=" * 124)

print(
    f"Vocabulary size                 : {VOCAB_SIZE}"
)

print(
    f"Training token minimum          : "
    f"{int(training_lengths.min())}"
)

print(
    f"Training token median           : "
    f"{float(training_lengths.median()):.1f}"
)

print(
    f"Training token maximum          : "
    f"{int(training_lengths.max())}"
)

print(
    f"Decoder maximum sequence length : "
    f"{MAX_SEQUENCE_LENGTH}"
)


# ==============================================================================
# 17. ENCODE SEQUENCES
# ==============================================================================

def encode_sequence(
    tokens
):

    ids = [
        BOS_ID
    ]


    unknown_count = 0


    for token in tokens:

        if token in token_to_id:

            ids.append(
                token_to_id[
                    token
                ]
            )

        else:

            ids.append(
                UNK_ID
            )

            unknown_count += 1


    ids.append(
        EOS_ID
    )


    truncated = False


    if len(
        ids
    ) > MAX_SEQUENCE_LENGTH:

        ids = ids[
            :
            MAX_SEQUENCE_LENGTH
        ]


        ids[
            -1
        ] = EOS_ID


        truncated = True


    return (
        ids,
        truncated,
        unknown_count
    )


def attach_encoded_sequences(
    dataframe
):

    dataframe = dataframe.copy(
        deep=True
    )


    encoded_sequences = []

    truncations = []

    unknown_counts = []


    for tokens in dataframe[
        "selfies_tokens"
    ]:

        ids, truncated, unknown_count = encode_sequence(
            tokens
        )


        encoded_sequences.append(
            ids
        )


        truncations.append(
            truncated
        )


        unknown_counts.append(
            unknown_count
        )


    dataframe[
        "sequence_ids"
    ] = encoded_sequences


    dataframe[
        "sequence_truncated"
    ] = truncations


    dataframe[
        "unknown_token_count"
    ] = unknown_counts


    return dataframe


D_train_seq = attach_encoded_sequences(
    D_train_seq
)


D_validation_seq = attach_encoded_sequences(
    D_validation_seq
)


D_test_seq = attach_encoded_sequences(
    D_test_seq
)


training_truncated = int(
    D_train_seq[
        "sequence_truncated"
    ].sum()
)


validation_truncated = int(
    D_validation_seq[
        "sequence_truncated"
    ].sum()
)


test_truncated = int(
    D_test_seq[
        "sequence_truncated"
    ].sum()
)


validation_unknown_tokens = int(
    D_validation_seq[
        "unknown_token_count"
    ].sum()
)


test_unknown_tokens = int(
    D_test_seq[
        "unknown_token_count"
    ].sum()
)


print(
    f"\nTraining sequence truncations   : {training_truncated}"
)

print(
    f"Validation sequence truncations : {validation_truncated}"
)

print(
    f"Test sequence truncations       : {test_truncated}"
)

print(
    f"Validation unknown tokens       : {validation_unknown_tokens}"
)

print(
    f"Test unknown tokens             : {test_unknown_tokens}"
)


# ==============================================================================
# 18. LATENT TENSOR HELPER
# ==============================================================================

def row_to_latent(
    row
):

    return torch.tensor(
        [
            float(
                row[
                    column
                ]
            )
            for column
            in latent_columns
        ],
        dtype=torch.float32
    )


# ==============================================================================
# 19. DATASET
# ==============================================================================

class LatentSelfiesDataset(Dataset):

    def __init__(
        self,
        dataframe
    ):

        self.dataframe = (
            dataframe
            .copy(
                deep=True
            )
            .reset_index(
                drop=True
            )
        )


    def __len__(
        self
    ):

        return len(
            self.dataframe
        )


    def __getitem__(
        self,
        index
    ):

        row = self.dataframe.iloc[
            index
        ]


        return {
            "graph_id":
                str(
                    row[
                        "graph_id"
                    ]
                ),

            "drug_id":
                str(
                    row[
                        "drug_id"
                    ]
                ),

            "drug_name":
                str(
                    row.get(
                        "drug_name",
                        ""
                    )
                ),

            "latent":
                row_to_latent(
                    row
                ),

            "sequence":
                torch.tensor(
                    row[
                        "sequence_ids"
                    ],
                    dtype=torch.long
                ),

            "reference_selfies":
                row[
                    "reference_selfies"
                ],

            "reference_smiles":
                row[
                    "reference_canonical_smiles"
                ]
        }


# ==============================================================================
# 20. COLLATE FUNCTION
# ==============================================================================

def sequence_collate(
    records
):

    batch_size = len(
        records
    )


    max_length = max(
        len(
            record[
                "sequence"
            ]
        )
        for record
        in records
    )


    sequences = torch.full(
        (
            batch_size,
            max_length
        ),
        PAD_ID,
        dtype=torch.long
    )


    latents = torch.stack(
        [
            record[
                "latent"
            ]
            for record
            in records
        ]
    )


    graph_ids = []

    drug_ids = []

    drug_names = []

    reference_selfies = []

    reference_smiles = []


    for index, record in enumerate(
        records
    ):

        sequence = record[
            "sequence"
        ]


        sequences[
            index,
            :
            len(
                sequence
            )
        ] = sequence


        graph_ids.append(
            record[
                "graph_id"
            ]
        )


        drug_ids.append(
            record[
                "drug_id"
            ]
        )


        drug_names.append(
            record[
                "drug_name"
            ]
        )


        reference_selfies.append(
            record[
                "reference_selfies"
            ]
        )


        reference_smiles.append(
            record[
                "reference_smiles"
            ]
        )


    return {
        "latent":
            latents,

        "sequence":
            sequences,

        "graph_ids":
            graph_ids,

        "drug_ids":
            drug_ids,

        "drug_names":
            drug_names,

        "reference_selfies":
            reference_selfies,

        "reference_smiles":
            reference_smiles
    }


# ==============================================================================
# 21. TRAIN / VALIDATION DATASETS
# ==============================================================================

train_dataset = LatentSelfiesDataset(
    D_train_seq
)


validation_dataset = LatentSelfiesDataset(
    D_validation_seq
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=sequence_collate,
    generator=torch.Generator().manual_seed(
        RANDOM_SEED
    )
)


validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=sequence_collate
)


# ==============================================================================
# 22. REVISED STRONGLY LATENT-CONDITIONED GRU DECODER
# ==============================================================================

class StrongLatentConditionalGRUDecoder(nn.Module):

    def __init__(
        self,
        latent_dim,
        vocab_size,
        token_embed_dim,
        latent_context_dim,
        hidden_dim,
        num_layers,
        pad_id
    ):

        super().__init__()


        self.latent_dim = latent_dim

        self.vocab_size = vocab_size

        self.token_embed_dim = token_embed_dim

        self.latent_context_dim = latent_context_dim

        self.hidden_dim = hidden_dim

        self.num_layers = num_layers


        # ----------------------------------------------------------------------
        # Token representation
        # ----------------------------------------------------------------------

        self.token_embedding = nn.Embedding(
            vocab_size,
            token_embed_dim,
            padding_idx=pad_id
        )


        # ----------------------------------------------------------------------
        # Latent vector -> recurrent initial state
        # ----------------------------------------------------------------------

        self.latent_to_hidden = nn.Sequential(

            nn.Linear(
                latent_dim,
                hidden_dim
                *
                num_layers
            ),

            nn.Tanh()
        )


        # ----------------------------------------------------------------------
        # Latent vector -> per-token context
        #
        # This context is concatenated to every token embedding.
        # ----------------------------------------------------------------------

        self.latent_to_context = nn.Sequential(

            nn.Linear(
                latent_dim,
                latent_context_dim
            ),

            nn.SiLU(),

            nn.LayerNorm(
                latent_context_dim
            )
        )


        # ----------------------------------------------------------------------
        # GRU
        # ----------------------------------------------------------------------

        self.gru = nn.GRU(
            input_size=(
                token_embed_dim
                +
                latent_context_dim
            ),
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )


        # ----------------------------------------------------------------------
        # Vocabulary output
        # ----------------------------------------------------------------------

        self.output_projection = nn.Linear(
            hidden_dim,
            vocab_size
        )


    def initialize_hidden(
        self,
        latent
    ):

        batch_size = latent.shape[
            0
        ]


        hidden = self.latent_to_hidden(
            latent
        )


        hidden = hidden.view(
            batch_size,
            self.num_layers,
            self.hidden_dim
        )


        hidden = hidden.transpose(
            0,
            1
        ).contiguous()


        return hidden


    def latent_context(
        self,
        latent,
        sequence_length
    ):

        context = self.latent_to_context(
            latent
        )


        context = context.unsqueeze(
            1
        )


        context = context.expand(
            -1,
            sequence_length,
            -1
        )


        return context


    def forward(
        self,
        latent,
        input_tokens
    ):

        hidden = self.initialize_hidden(
            latent
        )


        token_embeddings = self.token_embedding(
            input_tokens
        )


        context = self.latent_context(
            latent,
            input_tokens.shape[
                1
            ]
        )


        conditioned_input = torch.cat(
            [
                token_embeddings,
                context
            ],
            dim=-1
        )


        output, hidden = self.gru(
            conditioned_input,
            hidden
        )


        logits = self.output_projection(
            output
        )


        return logits


# ==============================================================================
# 23. INITIALIZE MODEL
# ==============================================================================

torch.manual_seed(
    RANDOM_SEED
)


decoder = StrongLatentConditionalGRUDecoder(
    latent_dim=LATENT_DIM,
    vocab_size=VOCAB_SIZE,
    token_embed_dim=TOKEN_EMBED_DIM,
    latent_context_dim=LATENT_CONTEXT_DIM,
    hidden_dim=DECODER_HIDDEN_DIM,
    num_layers=NUM_GRU_LAYERS,
    pad_id=PAD_ID
).to(
    DEVICE
)


decoder_parameter_count = sum(
    parameter.numel()
    for parameter
    in decoder.parameters()
)


trainable_decoder_parameter_count = sum(
    parameter.numel()
    for parameter
    in decoder.parameters()
    if parameter.requires_grad
)


print("\n" + "=" * 124)
print("REVISED GENERATIVE DECODER CONFIGURATION")
print("=" * 124)

print(
    f"Vocabulary Size                 : {VOCAB_SIZE}"
)

print(
    f"Latent Dimension                : {LATENT_DIM}"
)

print(
    f"Token Embedding Dimension       : {TOKEN_EMBED_DIM}"
)

print(
    f"Latent Context Dimension        : {LATENT_CONTEXT_DIM}"
)

print(
    f"GRU Hidden Dimension            : {DECODER_HIDDEN_DIM}"
)

print(
    f"GRU Layers                      : {NUM_GRU_LAYERS}"
)

print(
    f"Decoder Parameters              : {decoder_parameter_count:,}"
)

print(
    f"Trainable Parameters            : {trainable_decoder_parameter_count:,}"
)

print(
    "Latent Conditioning            : "
    "initial hidden state + every decoding step"
)


# ==============================================================================
# 24. LOSS + OPTIMIZER
# ==============================================================================

criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_ID
)


optimizer = torch.optim.Adam(
    decoder.parameters(),
    lr=LEARNING_RATE
)


# ==============================================================================
# 25. TRAIN ONE EPOCH
# ==============================================================================

def train_epoch(
    model,
    loader
):

    model.train()


    weighted_loss_sum = 0.0

    total_valid_tokens = 0

    correct_tokens = 0


    for batch in loader:

        latent = batch[
            "latent"
        ].to(
            DEVICE
        )


        sequence = batch[
            "sequence"
        ].to(
            DEVICE
        )


        decoder_input = sequence[
            :,
            :-1
        ]


        targets = sequence[
            :,
            1:
        ]


        optimizer.zero_grad(
            set_to_none=True
        )


        logits = model(
            latent,
            decoder_input
        )


        loss = criterion(
            logits.reshape(
                -1,
                VOCAB_SIZE
            ),
            targets.reshape(
                -1
            )
        )


        if not torch.isfinite(
            loss
        ):

            raise RuntimeError(
                "Non-finite decoder training loss detected."
            )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0
        )


        optimizer.step()


        valid_mask = (
            targets
            !=
            PAD_ID
        )


        predictions = logits.argmax(
            dim=-1
        )


        valid_count = int(
            valid_mask.sum().item()
        )


        correct_tokens += int(
            (
                (
                    predictions
                    ==
                    targets
                )
                &
                valid_mask
            ).sum().item()
        )


        total_valid_tokens += (
            valid_count
        )


        weighted_loss_sum += (
            float(
                loss.item()
            )
            *
            valid_count
        )


    return {
        "loss":
            (
                weighted_loss_sum
                /
                max(
                    total_valid_tokens,
                    1
                )
            ),

        "token_accuracy":
            (
                correct_tokens
                /
                max(
                    total_valid_tokens,
                    1
                )
            )
    }


# ==============================================================================
# 26. EVALUATE ONE EPOCH
# ==============================================================================

def evaluate_epoch(
    model,
    loader
):

    model.eval()


    weighted_loss_sum = 0.0

    total_valid_tokens = 0

    correct_tokens = 0


    with torch.no_grad():

        for batch in loader:

            latent = batch[
                "latent"
            ].to(
                DEVICE
            )


            sequence = batch[
                "sequence"
            ].to(
                DEVICE
            )


            decoder_input = sequence[
                :,
                :-1
            ]


            targets = sequence[
                :,
                1:
            ]


            logits = model(
                latent,
                decoder_input
            )


            loss = criterion(
                logits.reshape(
                    -1,
                    VOCAB_SIZE
                ),
                targets.reshape(
                    -1
                )
            )


            valid_mask = (
                targets
                !=
                PAD_ID
            )


            predictions = logits.argmax(
                dim=-1
            )


            valid_count = int(
                valid_mask.sum().item()
            )


            correct_tokens += int(
                (
                    (
                        predictions
                        ==
                        targets
                    )
                    &
                    valid_mask
                ).sum().item()
            )


            total_valid_tokens += (
                valid_count
            )


            weighted_loss_sum += (
                float(
                    loss.item()
                )
                *
                valid_count
            )


    return {
        "loss":
            (
                weighted_loss_sum
                /
                max(
                    total_valid_tokens,
                    1
                )
            ),

        "token_accuracy":
            (
                correct_tokens
                /
                max(
                    total_valid_tokens,
                    1
                )
            )
    }


# ==============================================================================
# 27. OUTPUT PATHS
# ==============================================================================

PHASE11_CONFIG_FILE = (
    Path(PROCESSED_DIR)
    / "phase11_decoder_config.json"
)


PHASE11_VOCAB_FILE = (
    Path(PROCESSED_DIR)
    / "phase11_selfies_vocabulary.json"
)


PHASE11_HISTORY_FILE = (
    Path(PROCESSED_DIR)
    / "phase11_decoder_training_history.csv"
)


PHASE11_BEST_DECODER_FILE = (
    Path(PROCESSED_DIR)
    / "phase11_best_generative_decoder.pt"
)


PHASE11_VALIDATION_RECON_FILE = (
    Path(PROCESSED_DIR)
    / "phase11_validation_reconstruction.csv"
)


PHASE11_TEST_RECON_FILE = (
    Path(PROCESSED_DIR)
    / "phase11_test_reconstruction.csv"
)


PHASE11_RECON_METRICS_FILE = (
    Path(PROCESSED_DIR)
    / "phase11_reconstruction_metrics.csv"
)


PHASE11_LATENT_DIAGNOSTIC_FILE = (
    Path(PROCESSED_DIR)
    / "phase11_latent_conditioning_diagnostics.csv"
)


PHASE11_RAW_CANDIDATES_FILE = (
    Path(PROCESSED_DIR)
    / "phase11_generated_candidates_raw.csv"
)


PHASE11_VALIDATED_CANDIDATES_FILE = (
    Path(PROCESSED_DIR)
    / "phase11_generated_candidates_validated.csv"
)


PHASE11_UNIQUE_CANDIDATES_FILE = (
    Path(PROCESSED_DIR)
    / "phase11_generated_candidates_unique.csv"
)


PHASE11_NOVEL_CANDIDATES_FILE = (
    Path(PROCESSED_DIR)
    / "phase11_generated_candidates_novel.csv"
)


PHASE11_PREPARATION_FAILURES_FILE = (
    Path(PROCESSED_DIR)
    / "phase11_sequence_preparation_failures.csv"
)


PHASE11_QC_CSV_FILE = (
    Path(PROCESSED_DIR)
    / "phase11_generation_qc_summary.csv"
)


PHASE11_QC_JSON_FILE = (
    Path(PROCESSED_DIR)
    / "phase11_generation_qc_summary.json"
)


# ==============================================================================
# 28. SAVE VOCABULARY
# ==============================================================================

vocabulary_metadata = {
    "source":
        "Phase-8 training corpus only",

    "vocabulary_size":
        VOCAB_SIZE,

    "special_tokens":
        SPECIAL_TOKENS,

    "pad_id":
        PAD_ID,

    "bos_id":
        BOS_ID,

    "eos_id":
        EOS_ID,

    "unk_id":
        UNK_ID,

    "vocabulary":
        vocabulary,

    "token_to_id":
        token_to_id,

    "maximum_sequence_length":
        MAX_SEQUENCE_LENGTH,

    "training_token_length_minimum":
        int(
            training_lengths.min()
        ),

    "training_token_length_median":
        float(
            training_lengths.median()
        ),

    "training_token_length_maximum":
        int(
            training_lengths.max()
        )
}


with open(
    PHASE11_VOCAB_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        vocabulary_metadata,
        file,
        indent=4,
        ensure_ascii=False
    )


# ==============================================================================
# 29. SAVE CONFIGURATION
# ==============================================================================

phase11_config = {
    "phase":
        "Phase 11 — Revised Molecular Decoder / Candidate Generation",

    "random_seed":
        RANDOM_SEED,

    "device":
        str(
            DEVICE
        ),

    "molecular_sequence_representation":
        "SELFIES",

    "latent_source":
        "Phase-10 deterministic mu",

    "phase10_encoder_frozen":
        True,

    "latent_dimension":
        LATENT_DIM,

    "token_embedding_dimension":
        TOKEN_EMBED_DIM,

    "latent_context_dimension":
        LATENT_CONTEXT_DIM,

    "decoder_hidden_dimension":
        DECODER_HIDDEN_DIM,

    "gru_layers":
        NUM_GRU_LAYERS,

    "latent_conditioning":
        (
            "Latent vector initializes all GRU hidden layers and "
            "projected latent context is concatenated at every decoding step."
        ),

    "vocabulary_size":
        VOCAB_SIZE,

    "maximum_sequence_length":
        MAX_SEQUENCE_LENGTH,

    "batch_size":
        BATCH_SIZE,

    "learning_rate":
        LEARNING_RATE,

    "maximum_epochs":
        MAX_EPOCHS,

    "early_stopping_patience":
        EARLY_STOPPING_PATIENCE,

    "candidate_generation_temperature":
        GENERATION_TEMPERATURE,

    "candidate_generation_top_k":
        TOP_K,

    "candidate_latent_perturbation":
        LATENT_PERTURBATION_SCALE,

    "minimum_heavy_atoms":
        MIN_HEAVY_ATOMS,

    "maximum_heavy_atoms":
        MAX_HEAVY_ATOMS,

    "latent_diagnostic_sample_size":
        NUM_LATENT_DIAGNOSTIC_MOLECULES,

    "small_latent_perturbation":
        SMALL_PERTURBATION_SCALE,

    "large_latent_perturbation":
        LARGE_PERTURBATION_SCALE,

    "fisher_information_calculated":
        False,

    "ewc_performed":
        False,

    "closed_loop_optimization":
        False
}


with open(
    PHASE11_CONFIG_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        phase11_config,
        file,
        indent=4,
        ensure_ascii=False
    )


# ==============================================================================
# 30. TRAIN DECODER
# ==============================================================================

print("\n" + "=" * 124)
print("REVISED GENERATIVE DECODER TRAINING")
print("=" * 124)


training_history = []


best_validation_loss = float(
    "inf"
)


best_epoch = 0


epochs_without_improvement = 0


training_start_time = time.time()


for epoch in range(
    1,
    MAX_EPOCHS
    +
    1
):

    train_metrics = train_epoch(
        decoder,
        train_loader
    )


    validation_metrics = evaluate_epoch(
        decoder,
        validation_loader
    )


    improved = (
        validation_metrics[
            "loss"
        ]
        <
        best_validation_loss
    )


    if improved:

        best_validation_loss = validation_metrics[
            "loss"
        ]


        best_epoch = epoch


        epochs_without_improvement = 0


        torch.save(
            {
                "model_status":
                    "TRAINED_STRONGLY_LATENT_CONDITIONED_SELFIES_DECODER",

                "epoch":
                    epoch,

                "validation_loss":
                    best_validation_loss,

                "state_dict":
                    decoder.state_dict(),

                "config":
                    phase11_config,

                "vocabulary_metadata":
                    vocabulary_metadata
            },
            PHASE11_BEST_DECODER_FILE
        )


    else:

        epochs_without_improvement += 1


    training_history.append(
        {
            "epoch":
                epoch,

            "train_loss":
                train_metrics[
                    "loss"
                ],

            "train_token_accuracy":
                train_metrics[
                    "token_accuracy"
                ],

            "validation_loss":
                validation_metrics[
                    "loss"
                ],

            "validation_token_accuracy":
                validation_metrics[
                    "token_accuracy"
                ],

            "best_checkpoint":
                bool(
                    improved
                )
        }
    )


    print(
        f"[EPOCH {epoch:03d}] "
        f"train_loss={train_metrics['loss']:.6f} | "
        f"train_acc={train_metrics['token_accuracy']:.4f} | "
        f"val_loss={validation_metrics['loss']:.6f} | "
        f"val_acc={validation_metrics['token_accuracy']:.4f}"
        +
        (
            " | BEST"
            if improved
            else ""
        )
    )


    if (
        epochs_without_improvement
        >=
        EARLY_STOPPING_PATIENCE
    ):

        print(
            f"\n[EARLY STOP] No validation improvement for "
            f"{EARLY_STOPPING_PATIENCE} epochs."
        )

        break


training_duration_seconds = (
    time.time()
    -
    training_start_time
)


epochs_completed = len(
    training_history
)


history_df = pd.DataFrame(
    training_history
)


history_df.to_csv(
    PHASE11_HISTORY_FILE,
    index=False,
    encoding="utf-8"
)


print(
    f"\n[PASS] Training completed after "
    f"{epochs_completed} epochs."
)

print(
    f"[BEST] Best epoch           : {best_epoch}"
)

print(
    f"[BEST] Best validation loss : "
    f"{best_validation_loss:.8f}"
)


# ==============================================================================
# 31. RELOAD BEST DECODER
# ==============================================================================

best_checkpoint = torch.load(
    PHASE11_BEST_DECODER_FILE,
    map_location=DEVICE,
    weights_only=False
)


best_decoder = StrongLatentConditionalGRUDecoder(
    latent_dim=LATENT_DIM,
    vocab_size=VOCAB_SIZE,
    token_embed_dim=TOKEN_EMBED_DIM,
    latent_context_dim=LATENT_CONTEXT_DIM,
    hidden_dim=DECODER_HIDDEN_DIM,
    num_layers=NUM_GRU_LAYERS,
    pad_id=PAD_ID
).to(
    DEVICE
)


best_decoder.load_state_dict(
    best_checkpoint[
        "state_dict"
    ]
)


best_decoder.eval()


best_checkpoint_reloadable = True


print(
    "\n[PASS] Best revised generative decoder reloaded successfully."
)


# ==============================================================================
# 32. AUTOREGRESSIVE GENERATION
# ==============================================================================

def generate_token_ids(
    model,
    latent,
    strategy="greedy",
    temperature=1.0,
    top_k=None,
    max_length=None
):

    if max_length is None:

        max_length = MAX_SEQUENCE_LENGTH


    if latent.ndim == 1:

        latent = latent.unsqueeze(
            0
        )


    latent = latent.to(
        DEVICE
    )


    model.eval()


    hidden = model.initialize_hidden(
        latent
    )


    latent_context = model.latent_to_context(
        latent
    ).unsqueeze(
        1
    )


    current_token = torch.tensor(
        [
            [
                BOS_ID
            ]
        ],
        dtype=torch.long,
        device=DEVICE
    )


    generated_ids = []


    with torch.no_grad():

        for _ in range(
            max_length
            -
            1
        ):

            token_embedding = model.token_embedding(
                current_token
            )


            conditioned_input = torch.cat(
                [
                    token_embedding,
                    latent_context
                ],
                dim=-1
            )


            output, hidden = model.gru(
                conditioned_input,
                hidden
            )


            logits = model.output_projection(
                output[
                    :,
                    -1,
                    :
                ]
            )


            # Prevent control tokens other than EOS from being sampled.

            logits[
                :,
                PAD_ID
            ] = -float(
                "inf"
            )


            logits[
                :,
                BOS_ID
            ] = -float(
                "inf"
            )


            logits[
                :,
                UNK_ID
            ] = -float(
                "inf"
            )


            if strategy == "greedy":

                next_token = logits.argmax(
                    dim=-1
                )


            elif strategy == "sample":

                temperature = max(
                    float(
                        temperature
                    ),
                    1e-6
                )


                sampling_logits = (
                    logits
                    /
                    temperature
                )


                if (
                    top_k is not None
                    and
                    top_k > 0
                    and
                    top_k < VOCAB_SIZE
                ):

                    top_values, top_indices = torch.topk(
                        sampling_logits,
                        k=top_k,
                        dim=-1
                    )


                    probabilities = torch.softmax(
                        top_values,
                        dim=-1
                    )


                    sampled_position = torch.multinomial(
                        probabilities,
                        num_samples=1
                    )


                    next_token = top_indices.gather(
                        1,
                        sampled_position
                    ).squeeze(
                        1
                    )


                else:

                    probabilities = torch.softmax(
                        sampling_logits,
                        dim=-1
                    )


                    next_token = torch.multinomial(
                        probabilities,
                        num_samples=1
                    ).squeeze(
                        1
                    )


            else:

                raise ValueError(
                    "strategy must be 'greedy' or 'sample'."
                )


            next_id = int(
                next_token.item()
            )


            generated_ids.append(
                next_id
            )


            if next_id == EOS_ID:

                break


            current_token = next_token.view(
                1,
                1
            )


    return generated_ids


# ==============================================================================
# 33. TOKEN IDS -> SELFIES
# ==============================================================================

def token_ids_to_selfies(
    token_ids
):

    tokens = []


    for token_id in token_ids:

        token_id = int(
            token_id
        )


        if token_id == EOS_ID:

            break


        if token_id in {
            PAD_ID,
            BOS_ID,
            UNK_ID
        }:

            continue


        token = id_to_token.get(
            token_id
        )


        if token is not None:

            tokens.append(
                token
            )


    return "".join(
        tokens
    )


# ==============================================================================
# 34. SELFIES -> SMILES
# ==============================================================================

def decode_selfies(
    selfies_string
):

    if (
        selfies_string is None
        or
        selfies_string == ""
    ):

        return None


    try:

        return sf.decoder(
            selfies_string
        )

    except Exception:

        return None


# ==============================================================================
# 35. MOLECULAR STANDARDIZATION
# ==============================================================================

cleanup_function = (
    rdMolStandardize.Cleanup
)


fragment_parent_function = (
    rdMolStandardize.FragmentParent
)


uncharger = (
    rdMolStandardize.Uncharger()
)


def validate_generated_molecule(
    smiles
):

    result = {
        "selfies_decode_success":
            False,

        "rdkit_parse_success":
            False,

        "sanitization_success":
            False,

        "valence_validation_success":
            False,

        "standardization_success":
            False,

        "canonical_smiles":
            None,

        "heavy_atom_count":
            None,

        "within_model_feasibility":
            False,

        "validation_status":
            "REJECTED",

        "failure_reason":
            None
    }


    if (
        smiles is None
        or
        smiles == ""
    ):

        result[
            "failure_reason"
        ] = "SELFIES_DECODE_FAILURE"

        return result


    result[
        "selfies_decode_success"
    ] = True


    try:

        molecule = Chem.MolFromSmiles(
            smiles,
            sanitize=False
        )

    except Exception:

        molecule = None


    if molecule is None:

        result[
            "failure_reason"
        ] = "RDKIT_PARSE_FAILURE"

        return result


    result[
        "rdkit_parse_success"
    ] = True


    try:

        Chem.SanitizeMol(
            molecule
        )

    except Exception:

        result[
            "failure_reason"
        ] = "SANITIZATION_FAILURE"

        return result


    result[
        "sanitization_success"
    ] = True


    try:

        molecule.UpdatePropertyCache(
            strict=True
        )

    except Exception:

        result[
            "failure_reason"
        ] = "VALENCE_VALIDATION_FAILURE"

        return result


    result[
        "valence_validation_success"
    ] = True


    try:

        standardized = cleanup_function(
            molecule
        )


        standardized = fragment_parent_function(
            standardized
        )


        standardized = uncharger.uncharge(
            standardized
        )


        Chem.SanitizeMol(
            standardized
        )


        standardized.UpdatePropertyCache(
            strict=True
        )


        canonical_smiles = Chem.MolToSmiles(
            standardized,
            canonical=True,
            isomericSmiles=True
        )


        heavy_atoms = int(
            standardized.GetNumHeavyAtoms()
        )


    except Exception:

        result[
            "failure_reason"
        ] = "STANDARDIZATION_FAILURE"

        return result


    result[
        "standardization_success"
    ] = True


    result[
        "canonical_smiles"
    ] = canonical_smiles


    result[
        "heavy_atom_count"
    ] = heavy_atoms


    result[
        "within_model_feasibility"
    ] = bool(
        MIN_HEAVY_ATOMS
        <=
        heavy_atoms
        <=
        MAX_HEAVY_ATOMS
    )


    if not result[
        "within_model_feasibility"
    ]:

        result[
            "failure_reason"
        ] = "OUTSIDE_MODEL_FEASIBILITY_RANGE"

        return result


    result[
        "validation_status"
    ] = "VALID"


    result[
        "failure_reason"
    ] = ""


    return result


# ==============================================================================
# 36. DECODE ONE LATENT VECTOR
# ==============================================================================

def decode_latent(
    latent,
    strategy="greedy",
    temperature=1.0,
    top_k=None
):

    generated_ids = generate_token_ids(
        best_decoder,
        latent,
        strategy=strategy,
        temperature=temperature,
        top_k=top_k
    )


    generated_selfies = token_ids_to_selfies(
        generated_ids
    )


    generated_smiles = decode_selfies(
        generated_selfies
    )


    qc = validate_generated_molecule(
        generated_smiles
    )


    return {
        "generated_ids":
            generated_ids,

        "generated_selfies":
            generated_selfies,

        "generated_smiles":
            generated_smiles,

        "qc":
            qc
    }


# ==============================================================================
# 37. VALIDATION / TEST GREEDY RECONSTRUCTION
# ==============================================================================

def evaluate_greedy_reconstruction(
    dataframe,
    split_name
):

    records = []


    for _, row in dataframe.iterrows():

        latent = row_to_latent(
            row
        )


        decoded = decode_latent(
            latent,
            strategy="greedy"
        )


        qc = decoded[
            "qc"
        ]


        exact_selfies_match = (
            decoded[
                "generated_selfies"
            ]
            ==
            row[
                "reference_selfies"
            ]
        )


        canonical_match = (
            qc[
                "canonical_smiles"
            ]
            is not None
            and
            qc[
                "canonical_smiles"
            ]
            ==
            row[
                "reference_canonical_smiles"
            ]
        )


        records.append(
            {
                "split":
                    split_name,

                "graph_id":
                    row[
                        "graph_id"
                    ],

                "drug_id":
                    row[
                        "drug_id"
                    ],

                "reference_smiles":
                    row[
                        "reference_canonical_smiles"
                    ],

                "reference_selfies":
                    row[
                        "reference_selfies"
                    ],

                "generated_selfies":
                    decoded[
                        "generated_selfies"
                    ],

                "generated_smiles":
                    decoded[
                        "generated_smiles"
                    ],

                "generated_canonical_smiles":
                    qc[
                        "canonical_smiles"
                    ],

                "chemical_validity":
                    (
                        qc[
                            "validation_status"
                        ]
                        ==
                        "VALID"
                    ),

                "exact_selfies_match":
                    exact_selfies_match,

                "canonical_molecular_reconstruction":
                    canonical_match,

                "failure_reason":
                    qc[
                        "failure_reason"
                    ]
            }
        )


    return pd.DataFrame(
        records
    )


validation_reconstruction = evaluate_greedy_reconstruction(
    D_validation_seq,
    "VALIDATION"
)


validation_reconstruction.to_csv(
    PHASE11_VALIDATION_RECON_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 38. TEST DATASET — AFTER MODEL SELECTION ONLY
# ==============================================================================

test_dataset = LatentSelfiesDataset(
    D_test_seq
)


test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=sequence_collate
)


test_epoch_metrics = evaluate_epoch(
    best_decoder,
    test_loader
)


test_reconstruction = evaluate_greedy_reconstruction(
    D_test_seq,
    "TEST"
)


test_reconstruction.to_csv(
    PHASE11_TEST_RECON_FILE,
    index=False,
    encoding="utf-8"
)


print(
    "\n[PASS] Test corpus evaluated after decoder model selection."
)


# ==============================================================================
# 39. RECONSTRUCTION METRICS
# ==============================================================================

best_history_row = (
    history_df.loc[
        history_df[
            "epoch"
        ]
        ==
        best_epoch
    ]
    .iloc[
        0
    ]
)


def reconstruction_metrics(
    split_name,
    reconstruction_df,
    token_loss,
    token_accuracy
):

    return {
        "split":
            split_name,

        "molecules":
            len(
                reconstruction_df
            ),

        "token_loss":
            float(
                token_loss
            ),

        "token_accuracy":
            float(
                token_accuracy
            ),

        "chemical_validity_rate":
            float(
                reconstruction_df[
                    "chemical_validity"
                ].mean()
            ),

        "exact_selfies_match_rate":
            float(
                reconstruction_df[
                    "exact_selfies_match"
                ].mean()
            ),

        "canonical_molecular_reconstruction_rate":
            float(
                reconstruction_df[
                    "canonical_molecular_reconstruction"
                ].mean()
            )
    }


validation_metrics = reconstruction_metrics(
    "VALIDATION",
    validation_reconstruction,
    float(
        best_history_row[
            "validation_loss"
        ]
    ),
    float(
        best_history_row[
            "validation_token_accuracy"
        ]
    )
)


test_metrics = reconstruction_metrics(
    "TEST",
    test_reconstruction,
    test_epoch_metrics[
        "loss"
    ],
    test_epoch_metrics[
        "token_accuracy"
    ]
)


reconstruction_metrics_df = pd.DataFrame(
    [
        validation_metrics,
        test_metrics
    ]
)


reconstruction_metrics_df.to_csv(
    PHASE11_RECON_METRICS_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 40. FINGERPRINT HELPERS
# ==============================================================================

fingerprint_generator = (
    rdFingerprintGenerator.GetMorganGenerator(
        radius=2,
        fpSize=2048
    )
)


def smiles_fingerprint(
    smiles
):

    if (
        smiles is None
        or
        smiles == ""
    ):

        return None


    molecule = Chem.MolFromSmiles(
        smiles
    )


    if molecule is None:

        return None


    return fingerprint_generator.GetFingerprint(
        molecule
    )


def fingerprint_similarity(
    smiles_a,
    smiles_b
):

    fp_a = smiles_fingerprint(
        smiles_a
    )


    fp_b = smiles_fingerprint(
        smiles_b
    )


    if (
        fp_a is None
        or
        fp_b is None
    ):

        return None


    return float(
        DataStructs.TanimotoSimilarity(
            fp_a,
            fp_b
        )
    )


# ==============================================================================
# 41. LATENT-CONDITIONING DIAGNOSTIC
# ==============================================================================

print("\n" + "=" * 124)
print("LATENT-CONDITIONING DIAGNOSTIC")
print("=" * 124)


diagnostic_rng = np.random.default_rng(
    RANDOM_SEED
)


diagnostic_count = min(
    NUM_LATENT_DIAGNOSTIC_MOLECULES,
    len(
        D_validation_seq
    )
)


diagnostic_indices = diagnostic_rng.choice(
    len(
        D_validation_seq
    ),
    size=diagnostic_count,
    replace=False
)


diagnostic_subset = (
    D_validation_seq.iloc[
        diagnostic_indices
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# Create deterministic permutation for latent swapping.

permuted_indices = np.roll(
    np.arange(
        diagnostic_count
    ),
    1
)


diagnostic_records = []


greedy_base_selfies = []


greedy_base_canonical = []


swap_changed_count = 0


small_perturbation_changed_count = 0


large_perturbation_changed_count = 0


small_similarity_values = []

large_similarity_values = []


for index in range(
    diagnostic_count
):

    row = diagnostic_subset.iloc[
        index
    ]


    swapped_row = diagnostic_subset.iloc[
        int(
            permuted_indices[
                index
            ]
        )
    ]


    base_latent = row_to_latent(
        row
    )


    swapped_latent = row_to_latent(
        swapped_row
    )


    # --------------------------------------------------------------------------
    # Base greedy decoding
    # --------------------------------------------------------------------------

    base_decoded = decode_latent(
        base_latent,
        strategy="greedy"
    )


    base_selfies = base_decoded[
        "generated_selfies"
    ]


    base_canonical = base_decoded[
        "qc"
    ][
        "canonical_smiles"
    ]


    greedy_base_selfies.append(
        base_selfies
    )


    greedy_base_canonical.append(
        base_canonical
    )


    # --------------------------------------------------------------------------
    # Swapped latent
    # --------------------------------------------------------------------------

    swapped_decoded = decode_latent(
        swapped_latent,
        strategy="greedy"
    )


    swap_changed = (
        swapped_decoded[
            "generated_selfies"
        ]
        !=
        base_selfies
    )


    swap_changed_count += int(
        swap_changed
    )


    # --------------------------------------------------------------------------
    # Small latent perturbation
    # --------------------------------------------------------------------------

    small_noise = torch.tensor(
        diagnostic_rng.normal(
            0.0,
            SMALL_PERTURBATION_SCALE,
            LATENT_DIM
        ),
        dtype=torch.float32
    )


    small_latent = (
        base_latent
        +
        small_noise
    )


    small_decoded = decode_latent(
        small_latent,
        strategy="greedy"
    )


    small_changed = (
        small_decoded[
            "generated_selfies"
        ]
        !=
        base_selfies
    )


    small_perturbation_changed_count += int(
        small_changed
    )


    small_similarity = fingerprint_similarity(
        base_canonical,
        small_decoded[
            "qc"
        ][
            "canonical_smiles"
        ]
    )


    if small_similarity is not None:

        small_similarity_values.append(
            small_similarity
        )


    # --------------------------------------------------------------------------
    # Large latent perturbation
    # --------------------------------------------------------------------------

    large_noise = torch.tensor(
        diagnostic_rng.normal(
            0.0,
            LARGE_PERTURBATION_SCALE,
            LATENT_DIM
        ),
        dtype=torch.float32
    )


    large_latent = (
        base_latent
        +
        large_noise
    )


    large_decoded = decode_latent(
        large_latent,
        strategy="greedy"
    )


    large_changed = (
        large_decoded[
            "generated_selfies"
        ]
        !=
        base_selfies
    )


    large_perturbation_changed_count += int(
        large_changed
    )


    large_similarity = fingerprint_similarity(
        base_canonical,
        large_decoded[
            "qc"
        ][
            "canonical_smiles"
        ]
    )


    if large_similarity is not None:

        large_similarity_values.append(
            large_similarity
        )


    diagnostic_records.append(
        {
            "graph_id":
                row[
                    "graph_id"
                ],

            "drug_id":
                row[
                    "drug_id"
                ],

            "base_generated_selfies":
                base_selfies,

            "base_canonical_smiles":
                base_canonical,

            "swapped_source_drug_id":
                swapped_row[
                    "drug_id"
                ],

            "swapped_generated_selfies":
                swapped_decoded[
                    "generated_selfies"
                ],

            "latent_swap_changed_output":
                swap_changed,

            "small_perturbation_scale":
                SMALL_PERTURBATION_SCALE,

            "small_perturbed_selfies":
                small_decoded[
                    "generated_selfies"
                ],

            "small_perturbation_changed_output":
                small_changed,

            "small_perturbation_similarity":
                small_similarity,

            "large_perturbation_scale":
                LARGE_PERTURBATION_SCALE,

            "large_perturbed_selfies":
                large_decoded[
                    "generated_selfies"
                ],

            "large_perturbation_changed_output":
                large_changed,

            "large_perturbation_similarity":
                large_similarity
        }
    )


latent_diagnostic_df = pd.DataFrame(
    diagnostic_records
)


latent_diagnostic_df.to_csv(
    PHASE11_LATENT_DIAGNOSTIC_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 42. LATENT-CONDITIONING METRICS
# ==============================================================================

latent_swap_change_rate = (
    swap_changed_count
    /
    max(
        diagnostic_count,
        1
    )
)


small_perturbation_change_rate = (
    small_perturbation_changed_count
    /
    max(
        diagnostic_count,
        1
    )
)


large_perturbation_change_rate = (
    large_perturbation_changed_count
    /
    max(
        diagnostic_count,
        1
    )
)


unique_greedy_outputs = len(
    set(
        greedy_base_selfies
    )
)


greedy_unique_fraction = (
    unique_greedy_outputs
    /
    max(
        diagnostic_count,
        1
    )
)


mean_small_similarity = (
    float(
        np.mean(
            small_similarity_values
        )
    )
    if small_similarity_values
    else None
)


mean_large_similarity = (
    float(
        np.mean(
            large_similarity_values
        )
    )
    if large_similarity_values
    else None
)


# Evidence that the decoder responds to z.
#
# These thresholds are QC heuristics, not universal model-quality standards.

latent_conditioning_evidence_pass = bool(

    latent_swap_change_rate
    >=
    MIN_LATENT_SWAP_CHANGE_RATE

    and

    greedy_unique_fraction
    >=
    MIN_GREEDY_UNIQUE_FRACTION
)


print(
    f"Diagnostic molecules                    : "
    f"{diagnostic_count}"
)

print(
    f"Unique greedy decoded sequences         : "
    f"{unique_greedy_outputs}"
)

print(
    f"Greedy-output unique fraction           : "
    f"{greedy_unique_fraction:.4f}"
)

print(
    f"Latent-swap output change rate          : "
    f"{latent_swap_change_rate:.4f}"
)

print(
    f"Small perturbation output change rate   : "
    f"{small_perturbation_change_rate:.4f}"
)

print(
    f"Large perturbation output change rate   : "
    f"{large_perturbation_change_rate:.4f}"
)


if mean_small_similarity is not None:

    print(
        f"Mean small-perturbation similarity      : "
        f"{mean_small_similarity:.4f}"
    )


if mean_large_similarity is not None:

    print(
        f"Mean large-perturbation similarity      : "
        f"{mean_large_similarity:.4f}"
    )


print(
    f"Latent-conditioning evidence            : "
    f"{latent_conditioning_evidence_pass}"
)


# ==============================================================================
# 43. BUILD BASELINE NOVELTY REFERENCE
# ==============================================================================

baseline_canonical_set = set()


for smiles in D_chem_standardized[
    "standardized_smiles"
].astype(str):

    if smiles.strip() == "":

        continue


    canonical = canonicalize_smiles(
        smiles
    )


    if canonical is None:

        continue


    qc = validate_generated_molecule(
        canonical
    )


    if qc[
        "canonical_smiles"
    ] is not None:

        baseline_canonical_set.add(
            qc[
                "canonical_smiles"
            ]
        )


print(
    f"\n[INFO] Baseline novelty-reference molecules : "
    f"{len(baseline_canonical_set)}"
)


# ==============================================================================
# 44. GENERATE CANDIDATE MOLECULES
# ==============================================================================

print("\n" + "=" * 124)
print("CANDIDATE GENERATION")
print("=" * 124)


generation_rng = np.random.default_rng(
    RANDOM_SEED
)


candidate_records = []


candidate_counter = 1


seed_count = min(
    NUM_GENERATION_SEEDS,
    len(
        D_train_seq
    )
)


seed_indices = generation_rng.choice(
    len(
        D_train_seq
    ),
    size=seed_count,
    replace=False
)


# ------------------------------------------------------------------------------
# MODE 1 — LOCAL LATENT PERTURBATION
# ------------------------------------------------------------------------------

for seed_index in seed_indices:

    source_row = D_train_seq.iloc[
        int(
            seed_index
        )
    ]


    source_latent = row_to_latent(
        source_row
    )


    for _ in range(
        CANDIDATES_PER_SEED
    ):

        noise = torch.tensor(
            generation_rng.normal(
                loc=0.0,
                scale=LATENT_PERTURBATION_SCALE,
                size=LATENT_DIM
            ),
            dtype=torch.float32
        )


        candidate_latent = (
            source_latent
            +
            noise
        )


        decoded = decode_latent(
            candidate_latent,
            strategy="sample",
            temperature=GENERATION_TEMPERATURE,
            top_k=TOP_K
        )


        candidate_records.append(
            {
                "candidate_id":
                    f"CAND{candidate_counter:06d}",

                "source_mode":
                    "LATENT_PERTURBATION",

                "source_graph_id":
                    source_row[
                        "graph_id"
                    ],

                "source_drug_id":
                    source_row[
                        "drug_id"
                    ],

                "source_latent_type":
                    "PHASE10_MU",

                "latent_perturbation_scale":
                    LATENT_PERTURBATION_SCALE,

                "sampling_method":
                    "temperature_top_k",

                "temperature":
                    GENERATION_TEMPERATURE,

                "top_k":
                    TOP_K,

                "generated_selfies":
                    decoded[
                        "generated_selfies"
                    ],

                "generated_smiles":
                    decoded[
                        "generated_smiles"
                    ]
            }
        )


        candidate_counter += 1


# ------------------------------------------------------------------------------
# MODE 2 — EXPLORATORY PRIOR SAMPLING
# ------------------------------------------------------------------------------

for _ in range(
    NUM_PRIOR_SAMPLES
):

    random_latent = torch.tensor(
        generation_rng.normal(
            loc=0.0,
            scale=1.0,
            size=LATENT_DIM
        ),
        dtype=torch.float32
    )


    decoded = decode_latent(
        random_latent,
        strategy="sample",
        temperature=GENERATION_TEMPERATURE,
        top_k=TOP_K
    )


    candidate_records.append(
        {
            "candidate_id":
                f"CAND{candidate_counter:06d}",

            "source_mode":
                "PRIOR_SAMPLE",

            "source_graph_id":
                "",

            "source_drug_id":
                "",

            "source_latent_type":
                "N_0_I_EXPLORATORY",

            "latent_perturbation_scale":
                "",

            "sampling_method":
                "temperature_top_k",

            "temperature":
                GENERATION_TEMPERATURE,

            "top_k":
                TOP_K,

            "generated_selfies":
                decoded[
                    "generated_selfies"
                ],

            "generated_smiles":
                decoded[
                    "generated_smiles"
                ]
        }
    )


    candidate_counter += 1


generated_candidates_raw = pd.DataFrame(
    candidate_records
)


generated_candidates_raw.to_csv(
    PHASE11_RAW_CANDIDATES_FILE,
    index=False,
    encoding="utf-8"
)


print(
    f"Generated sequences : {len(generated_candidates_raw)}"
)


# ==============================================================================
# 45. VALIDATE GENERATED CANDIDATES
# ==============================================================================

validated_records = []


for _, candidate in generated_candidates_raw.iterrows():

    qc = validate_generated_molecule(
        candidate[
            "generated_smiles"
        ]
    )


    record = candidate.to_dict()


    record.update(
        qc
    )


    validated_records.append(
        record
    )


generated_candidates_validated = pd.DataFrame(
    validated_records
)


# ==============================================================================
# 46. UNIQUENESS + BASELINE NOVELTY
# ==============================================================================

generated_candidates_validated[
    "is_unique"
] = False


generated_candidates_validated[
    "is_novel_relative_to_baseline"
] = False


valid_indices = generated_candidates_validated.index[
    generated_candidates_validated[
        "validation_status"
    ]
    ==
    "VALID"
]


valid_candidates = generated_candidates_validated.loc[
    valid_indices
].copy()


if len(
    valid_candidates
) > 0:

    valid_candidates[
        "is_unique"
    ] = (
        ~valid_candidates[
            "canonical_smiles"
        ].duplicated(
            keep="first"
        )
    )


    valid_candidates[
        "is_novel_relative_to_baseline"
    ] = (
        ~valid_candidates[
            "canonical_smiles"
        ].isin(
            baseline_canonical_set
        )
    )


    generated_candidates_validated.loc[
        valid_candidates.index,
        "is_unique"
    ] = valid_candidates[
        "is_unique"
    ]


    generated_candidates_validated.loc[
        valid_candidates.index,
        "is_novel_relative_to_baseline"
    ] = valid_candidates[
        "is_novel_relative_to_baseline"
    ]


generated_candidates_unique = (
    generated_candidates_validated[
        (
            generated_candidates_validated[
                "validation_status"
            ]
            ==
            "VALID"
        )
        &
        (
            generated_candidates_validated[
                "is_unique"
            ]
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


generated_candidates_novel = (
    generated_candidates_unique[
        generated_candidates_unique[
            "is_novel_relative_to_baseline"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# ==============================================================================
# 47. SAVE CANDIDATES
# ==============================================================================

generated_candidates_validated.to_csv(
    PHASE11_VALIDATED_CANDIDATES_FILE,
    index=False,
    encoding="utf-8"
)


generated_candidates_unique.to_csv(
    PHASE11_UNIQUE_CANDIDATES_FILE,
    index=False,
    encoding="utf-8"
)


generated_candidates_novel.to_csv(
    PHASE11_NOVEL_CANDIDATES_FILE,
    index=False,
    encoding="utf-8"
)


pd.DataFrame(
    sequence_preparation_failures
).to_csv(
    PHASE11_PREPARATION_FAILURES_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 48. CANDIDATE DIVERSITY
# ==============================================================================

candidate_fingerprints = []


for smiles in generated_candidates_unique[
    "canonical_smiles"
].astype(str):

    fingerprint = smiles_fingerprint(
        smiles
    )


    if fingerprint is not None:

        candidate_fingerprints.append(
            fingerprint
        )


pairwise_similarities = []


for index in range(
    len(
        candidate_fingerprints
    )
):

    if (
        index
        +
        1
        >=
        len(
            candidate_fingerprints
        )
    ):

        continue


    similarities = DataStructs.BulkTanimotoSimilarity(
        candidate_fingerprints[
            index
        ],
        candidate_fingerprints[
            index
            +
            1:
        ]
    )


    pairwise_similarities.extend(
        similarities
    )


if pairwise_similarities:

    mean_pairwise_similarity = float(
        np.mean(
            pairwise_similarities
        )
    )


    mean_pairwise_diversity = float(
        1.0
        -
        mean_pairwise_similarity
    )


else:

    mean_pairwise_similarity = None

    mean_pairwise_diversity = None


# ==============================================================================
# 49. CANDIDATE QC COUNTS
# ==============================================================================

total_generated = len(
    generated_candidates_validated
)


selfies_decode_success_count = int(
    generated_candidates_validated[
        "selfies_decode_success"
    ].sum()
)


rdkit_parse_success_count = int(
    generated_candidates_validated[
        "rdkit_parse_success"
    ].sum()
)


sanitization_success_count = int(
    generated_candidates_validated[
        "sanitization_success"
    ].sum()
)


valence_success_count = int(
    generated_candidates_validated[
        "valence_validation_success"
    ].sum()
)


standardization_success_count = int(
    generated_candidates_validated[
        "standardization_success"
    ].sum()
)


feasibility_success_count = int(
    generated_candidates_validated[
        "within_model_feasibility"
    ].sum()
)


valid_candidate_count = int(
    (
        generated_candidates_validated[
            "validation_status"
        ]
        ==
        "VALID"
    ).sum()
)


unique_candidate_count = len(
    generated_candidates_unique
)


novel_candidate_count = len(
    generated_candidates_novel
)


validity_percentage = (
    100.0
    *
    valid_candidate_count
    /
    max(
        total_generated,
        1
    )
)


uniqueness_percentage = (
    100.0
    *
    unique_candidate_count
    /
    max(
        valid_candidate_count,
        1
    )
)


novelty_percentage = (
    100.0
    *
    novel_candidate_count
    /
    max(
        unique_candidate_count,
        1
    )
)


# ==============================================================================
# 50. SOURCE PRESERVATION
# ==============================================================================

phase8_sources_preserved = all(
    [
        D_train_manifest.equals(
            D_train_manifest_original
        ),

        D_validation_manifest.equals(
            D_validation_manifest_original
        ),

        D_test_manifest.equals(
            D_test_manifest_original
        )
    ]
)


# ==============================================================================
# 51. COMPLETION CHECKS
# ==============================================================================

completion_checks = {
    "Phase-10 encoder checkpoint preserved":
        PHASE10_BEST_ENCODER_FILE.exists(),

    "Phase-10 training latents loaded":
        len(
            D_train_seq
        ) > 0,

    "Phase-10 validation latents loaded":
        len(
            D_validation_seq
        ) > 0,

    "Phase-10 test latents loaded":
        len(
            D_test_seq
        ) > 0,

    "Latent vectors linked to molecular structures":
        len(
            D_train_seq
        )
        ==
        len(
            D_train_manifest
        ),

    "Phase-8 split assignments preserved":
        phase8_sources_preserved,

    "Training SELFIES conversion successful":
        bool(
            D_train_seq[
                "reference_selfies"
            ].notna().all()
        ),

    "Vocabulary derived from training only":
        True,

    "PAD token defined":
        PAD_TOKEN
        in
        token_to_id,

    "BOS token defined":
        BOS_TOKEN
        in
        token_to_id,

    "EOS token defined":
        EOS_TOKEN
        in
        token_to_id,

    "UNK token defined":
        UNK_TOKEN
        in
        token_to_id,

    "No training sequence truncation":
        training_truncated == 0,

    "Latent dimension equals 32":
        LATENT_DIM == 32,

    "Latent initializes recurrent hidden state":
        True,

    "Latent context enters every decoding step":
        True,

    "Teacher forcing implemented":
        True,

    "PAD ignored in sequence loss":
        criterion.ignore_index
        ==
        PAD_ID,

    "Decoder optimized using training molecules only":
        epochs_completed > 0,

    "Validation used for checkpoint selection":
        best_epoch > 0,

    "Test excluded from checkpoint selection":
        True,

    "Best decoder checkpoint saved":
        PHASE11_BEST_DECODER_FILE.exists(),

    "Best decoder reloadable":
        best_checkpoint_reloadable,

    "Validation reconstruction evaluated":
        len(
            validation_reconstruction
        ) > 0,

    "Test reconstruction evaluated":
        len(
            test_reconstruction
        ) > 0,

    "Greedy generation supported":
        True,

    "Stochastic top-k generation supported":
        True,

    "Latent-swap diagnostic performed":
        diagnostic_count > 0,

    "Small latent perturbation diagnostic performed":
        diagnostic_count > 0,

    "Large latent perturbation diagnostic performed":
        diagnostic_count > 0,

    "Latent-conditioning evidence passes QC threshold":
        latent_conditioning_evidence_pass,

    "SELFIES-to-SMILES conversion performed":
        True,

    "RDKit parsing performed":
        True,

    "Sanitization performed":
        True,

    "Valence validation performed":
        True,

    "Generated molecule standardization performed":
        True,

    "Heavy-atom feasibility filtering performed":
        True,

    "Candidate deduplication performed":
        True,

    "Baseline novelty evaluation performed":
        True,

    "Candidate provenance preserved":
        True,

    "No biological activity assigned":
        True,

    "No Fisher information calculated":
        True,

    "No EWC performed":
        True,

    "No closed-loop optimization performed":
        True
}


# ==============================================================================
# 52. QC SUMMARY
# ==============================================================================

qc_summary = {
    "phase":
        "Phase 11 — Revised Molecular Decoder / Candidate Generation",

    "generated_at":
        datetime.now().isoformat(),

    "model_status":
        (
            "TRAINED_STRONGLY_LATENT_CONDITIONED_SELFIES_DECODER"
        ),

    "training_sequences":
        len(
            D_train_seq
        ),

    "validation_sequences":
        len(
            D_validation_seq
        ),

    "test_sequences":
        len(
            D_test_seq
        ),

    "vocabulary_size":
        VOCAB_SIZE,

    "maximum_sequence_length":
        MAX_SEQUENCE_LENGTH,

    "training_sequence_truncations":
        training_truncated,

    "validation_sequence_truncations":
        validation_truncated,

    "test_sequence_truncations":
        test_truncated,

    "validation_unknown_tokens":
        validation_unknown_tokens,

    "test_unknown_tokens":
        test_unknown_tokens,

    "decoder_parameters":
        decoder_parameter_count,

    "epochs_completed":
        epochs_completed,

    "best_epoch":
        best_epoch,

    "best_validation_loss":
        best_validation_loss,

    "best_validation_token_accuracy":
        float(
            best_history_row[
                "validation_token_accuracy"
            ]
        ),

    "test_token_loss":
        test_epoch_metrics[
            "loss"
        ],

    "test_token_accuracy":
        test_epoch_metrics[
            "token_accuracy"
        ],

    "validation_canonical_reconstruction_rate":
        validation_metrics[
            "canonical_molecular_reconstruction_rate"
        ],

    "test_canonical_reconstruction_rate":
        test_metrics[
            "canonical_molecular_reconstruction_rate"
        ],

    # --------------------------------------------------------------------------
    # New latent-conditioning diagnostics
    # --------------------------------------------------------------------------

    "latent_diagnostic_molecules":
        diagnostic_count,

    "greedy_unique_outputs":
        unique_greedy_outputs,

    "greedy_unique_fraction":
        greedy_unique_fraction,

    "latent_swap_change_rate":
        latent_swap_change_rate,

    "small_perturbation_change_rate":
        small_perturbation_change_rate,

    "large_perturbation_change_rate":
        large_perturbation_change_rate,

    "mean_small_perturbation_similarity":
        mean_small_similarity,

    "mean_large_perturbation_similarity":
        mean_large_similarity,

    "latent_conditioning_evidence_pass":
        latent_conditioning_evidence_pass,

    # --------------------------------------------------------------------------
    # Candidate-generation QC
    # --------------------------------------------------------------------------

    "generated_sequences":
        total_generated,

    "selfies_decode_success_count":
        selfies_decode_success_count,

    "rdkit_parse_success_count":
        rdkit_parse_success_count,

    "sanitization_success_count":
        sanitization_success_count,

    "valence_validation_success_count":
        valence_success_count,

    "standardization_success_count":
        standardization_success_count,

    "heavy_atom_feasibility_count":
        feasibility_success_count,

    "valid_candidate_count":
        valid_candidate_count,

    "unique_candidate_count":
        unique_candidate_count,

    "novel_relative_to_baseline_count":
        novel_candidate_count,

    "validity_percentage":
        validity_percentage,

    "uniqueness_percentage":
        uniqueness_percentage,

    "novelty_percentage":
        novelty_percentage,

    "mean_pairwise_fingerprint_similarity":
        mean_pairwise_similarity,

    "mean_pairwise_fingerprint_diversity":
        mean_pairwise_diversity,

    "baseline_novelty_reference_count":
        len(
            baseline_canonical_set
        ),

    "phase10_encoder_fine_tuned":
        False,

    "phase8_sources_preserved":
        phase8_sources_preserved,

    "biological_activity_prediction":
        False,

    "fisher_information_calculated":
        False,

    "ewc_performed":
        False,

    "closed_loop_optimization":
        False,

    "training_duration_seconds":
        training_duration_seconds,

    "completion_checks":
        completion_checks
}


with open(
    PHASE11_QC_JSON_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        qc_summary,
        file,
        indent=4,
        ensure_ascii=False,
        default=str
    )


qc_rows = []


for key, value in qc_summary.items():

    if key == "completion_checks":

        for check_name, status in value.items():

            qc_rows.append(
                {
                    "metric":
                        f"completion.{check_name}",

                    "value":
                        status
                }
            )

    else:

        qc_rows.append(
            {
                "metric":
                    key,

                "value":
                    value
            }
        )


pd.DataFrame(
    qc_rows
).to_csv(
    PHASE11_QC_CSV_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 53. DISPLAY RECONSTRUCTION PERFORMANCE
# ==============================================================================

print("\n" + "=" * 124)
print("MOLECULAR RECONSTRUCTION PERFORMANCE")
print("=" * 124)

print(
    reconstruction_metrics_df.to_string(
        index=False
    )
)


# ==============================================================================
# 54. DISPLAY LATENT-CONDITIONING EVIDENCE
# ==============================================================================

print("\n" + "=" * 124)
print("LATENT-CONDITIONING EVIDENCE")
print("=" * 124)

print(
    f"Diagnostic Molecules              : "
    f"{diagnostic_count}"
)

print(
    f"Unique Greedy Outputs             : "
    f"{unique_greedy_outputs}"
)

print(
    f"Greedy Unique Fraction            : "
    f"{greedy_unique_fraction:.2%}"
)

print(
    f"Latent Swap Change Rate           : "
    f"{latent_swap_change_rate:.2%}"
)

print(
    f"Small Perturbation Change Rate    : "
    f"{small_perturbation_change_rate:.2%}"
)

print(
    f"Large Perturbation Change Rate    : "
    f"{large_perturbation_change_rate:.2%}"
)


if mean_small_similarity is not None:

    print(
        f"Mean Small Perturbation Similarity: "
        f"{mean_small_similarity:.4f}"
    )


if mean_large_similarity is not None:

    print(
        f"Mean Large Perturbation Similarity: "
        f"{mean_large_similarity:.4f}"
    )


print(
    f"Latent Conditioning Evidence PASS : "
    f"{latent_conditioning_evidence_pass}"
)


# ==============================================================================
# 55. DISPLAY GENERATED-CANDIDATE QC
# ==============================================================================

print("\n" + "=" * 124)
print("GENERATED CANDIDATE QC")
print("=" * 124)

print(
    f"Total Generated Sequences         : "
    f"{total_generated}"
)

print(
    f"SELFIES Decode Success            : "
    f"{selfies_decode_success_count}"
)

print(
    f"RDKit Parse Success               : "
    f"{rdkit_parse_success_count}"
)

print(
    f"Sanitization Success              : "
    f"{sanitization_success_count}"
)

print(
    f"Valence Validation Success        : "
    f"{valence_success_count}"
)

print(
    f"Standardization Success           : "
    f"{standardization_success_count}"
)

print(
    f"Heavy-Atom Feasibility Pass       : "
    f"{feasibility_success_count}"
)

print(
    f"Valid Candidate Molecules         : "
    f"{valid_candidate_count}"
)

print(
    f"Unique Valid Candidates           : "
    f"{unique_candidate_count}"
)

print(
    f"Novel Relative to Baseline        : "
    f"{novel_candidate_count}"
)

print(
    f"Validity Percentage               : "
    f"{validity_percentage:.2f}%"
)

print(
    f"Uniqueness Percentage             : "
    f"{uniqueness_percentage:.2f}%"
)

print(
    f"Novelty Percentage                : "
    f"{novelty_percentage:.2f}%"
)


if mean_pairwise_diversity is not None:

    print(
        f"Mean Fingerprint Diversity        : "
        f"{mean_pairwise_diversity:.4f}"
    )


# ==============================================================================
# 56. COMPLETION VALIDATION
# ==============================================================================

print("\n" + "=" * 124)
print("PHASE 11 COMPLETION VALIDATION")
print("=" * 124)


all_completion_checks_passed = True


for check_name, status in completion_checks.items():

    label = (
        "PASS"
        if bool(
            status
        )
        else "FAIL"
    )


    print(
        f"[{label:<4}] {check_name}"
    )


    if not bool(
        status
    ):

        all_completion_checks_passed = False


# ==============================================================================
# 57. UPDATE PROJECT CONFIGURATION
# ==============================================================================

PHASE11_DATASETS = {
    "decoder_config":
        str(
            PHASE11_CONFIG_FILE
        ),

    "selfies_vocabulary":
        str(
            PHASE11_VOCAB_FILE
        ),

    "training_history":
        str(
            PHASE11_HISTORY_FILE
        ),

    "best_generative_decoder":
        str(
            PHASE11_BEST_DECODER_FILE
        ),

    "validation_reconstruction":
        str(
            PHASE11_VALIDATION_RECON_FILE
        ),

    "test_reconstruction":
        str(
            PHASE11_TEST_RECON_FILE
        ),

    "reconstruction_metrics":
        str(
            PHASE11_RECON_METRICS_FILE
        ),

    "latent_conditioning_diagnostics":
        str(
            PHASE11_LATENT_DIAGNOSTIC_FILE
        ),

    "generated_candidates_raw":
        str(
            PHASE11_RAW_CANDIDATES_FILE
        ),

    "generated_candidates_validated":
        str(
            PHASE11_VALIDATED_CANDIDATES_FILE
        ),

    "generated_candidates_unique":
        str(
            PHASE11_UNIQUE_CANDIDATES_FILE
        ),

    "generated_candidates_novel":
        str(
            PHASE11_NOVEL_CANDIDATES_FILE
        ),

    "qc_summary_csv":
        str(
            PHASE11_QC_CSV_FILE
        ),

    "qc_summary_json":
        str(
            PHASE11_QC_JSON_FILE
        ),

    "best_epoch":
        best_epoch,

    "latent_conditioning_evidence_pass":
        latent_conditioning_evidence_pass,

    "valid_candidates":
        valid_candidate_count,

    "unique_candidates":
        unique_candidate_count,

    "novel_relative_to_baseline":
        novel_candidate_count
}


DRUGBANK_CONFIG[
    "phase11_datasets"
] = PHASE11_DATASETS


# ==============================================================================
# 58. FINAL SUMMARY
# ==============================================================================

print("\n" + "=" * 124)
print("PHASE 11 SUMMARY")
print("=" * 124)

print(
    f"Training Molecules                 : "
    f"{len(D_train_seq)}"
)

print(
    f"Validation Molecules               : "
    f"{len(D_validation_seq)}"
)

print(
    f"Test Molecules                     : "
    f"{len(D_test_seq)}"
)

print(
    f"Vocabulary Size                    : "
    f"{VOCAB_SIZE}"
)

print(
    f"Maximum Sequence Length            : "
    f"{MAX_SEQUENCE_LENGTH}"
)

print(
    f"Decoder Parameters                 : "
    f"{decoder_parameter_count:,}"
)

print(
    f"Epochs Completed                   : "
    f"{epochs_completed}"
)

print(
    f"Best Epoch                         : "
    f"{best_epoch}"
)

print(
    f"Best Validation Loss               : "
    f"{best_validation_loss:.8f}"
)

print(
    f"Best Validation Token Accuracy     : "
    f"{float(best_history_row['validation_token_accuracy']):.4f}"
)

print(
    f"Test Token Accuracy                : "
    f"{test_epoch_metrics['token_accuracy']:.4f}"
)

print(
    f"Validation Molecular Recon. Rate   : "
    f"{validation_metrics['canonical_molecular_reconstruction_rate']:.4f}"
)

print(
    f"Test Molecular Recon. Rate         : "
    f"{test_metrics['canonical_molecular_reconstruction_rate']:.4f}"
)


print(
    f"Latent Swap Change Rate            : "
    f"{latent_swap_change_rate:.2%}"
)

print(
    f"Greedy Latent Output Uniqueness    : "
    f"{greedy_unique_fraction:.2%}"
)

print(
    f"Small Perturbation Change Rate     : "
    f"{small_perturbation_change_rate:.2%}"
)

print(
    f"Large Perturbation Change Rate     : "
    f"{large_perturbation_change_rate:.2%}"
)

print(
    f"Latent Conditioning Evidence       : "
    f"{latent_conditioning_evidence_pass}"
)


print(
    f"Generated Sequences                : "
    f"{total_generated}"
)

print(
    f"Valid Candidates                   : "
    f"{valid_candidate_count}"
)

print(
    f"Unique Valid Candidates            : "
    f"{unique_candidate_count}"
)

print(
    f"Novel Relative to Baseline         : "
    f"{novel_candidate_count}"
)

print(
    f"Validity Percentage                : "
    f"{validity_percentage:.2f}%"
)

print(
    f"Uniqueness Percentage              : "
    f"{uniqueness_percentage:.2f}%"
)

print(
    f"Novelty Percentage                 : "
    f"{novelty_percentage:.2f}%"
)

print(
    f"Phase-10 Encoder Fine-Tuned        : False"
)

print(
    f"Biological Activity Assigned       : False"
)

print(
    f"Fisher Information Computed        : False"
)

print(
    f"EWC Performed                      : False"
)

print(
    f"Closed-Loop Optimization           : False"
)


print("\nSaved Files:")

print(
    f"  Decoder Config              : "
    f"{PHASE11_CONFIG_FILE}"
)

print(
    f"  SELFIES Vocabulary          : "
    f"{PHASE11_VOCAB_FILE}"
)

print(
    f"  Training History            : "
    f"{PHASE11_HISTORY_FILE}"
)

print(
    f"  Best Decoder                : "
    f"{PHASE11_BEST_DECODER_FILE}"
)

print(
    f"  Validation Reconstruction   : "
    f"{PHASE11_VALIDATION_RECON_FILE}"
)

print(
    f"  Test Reconstruction         : "
    f"{PHASE11_TEST_RECON_FILE}"
)

print(
    f"  Reconstruction Metrics      : "
    f"{PHASE11_RECON_METRICS_FILE}"
)

print(
    f"  Latent Conditioning Tests   : "
    f"{PHASE11_LATENT_DIAGNOSTIC_FILE}"
)

print(
    f"  Raw Candidates              : "
    f"{PHASE11_RAW_CANDIDATES_FILE}"
)

print(
    f"  Validated Candidates        : "
    f"{PHASE11_VALIDATED_CANDIDATES_FILE}"
)

print(
    f"  Unique Candidates           : "
    f"{PHASE11_UNIQUE_CANDIDATES_FILE}"
)

print(
    f"  Novel Candidates            : "
    f"{PHASE11_NOVEL_CANDIDATES_FILE}"
)

print(
    f"  QC Summary CSV              : "
    f"{PHASE11_QC_CSV_FILE}"
)

print(
    f"  QC Summary JSON             : "
    f"{PHASE11_QC_JSON_FILE}"
)


print("-" * 124)


if all_completion_checks_passed:

    print(
        "[STATUS] PHASE 11 COMPLETED SUCCESSFULLY."
    )

    print(
        "[PASS] Strong latent-conditioning evidence was observed."
    )

    print(
        "[READY] The decoder responds measurably to changes "
        "in the Phase-10 latent molecular representation."
    )

    print(
        "[READY] Candidate molecules were generated and passed "
        "defined computational chemical-quality controls."
    )

    print(
        "[IMPORTANT] 'Novel' means novel relative to the "
        "baseline DrugBank molecular corpus only."
    )

    print(
        "[IMPORTANT] Generated candidates have NOT yet been assigned "
        "biological activity, binding affinity, efficacy, or "
        "therapeutic superiority."
    )

    print(
        "[NEXT] Phase 12 — Baseline System Consolidation, "
        "Predictor Training, and Reference-State Establishment"
    )


else:

    print(
        "[STATUS] PHASE 11 REQUIRES REVIEW BEFORE FREEZING."
    )


    if not latent_conditioning_evidence_pass:

        print(
            "[WARNING] Latent-conditioning diagnostic did not meet "
            "the operational QC threshold."
        )

        print(
            "[ACTION] Do NOT freeze the Phase-11 decoder yet."
        )

    else:

        print(
            "[ACTION] Review the failed completion checks before "
            "proceeding."
        )


print("=" * 124)

PHASE 11 — REVISED LATENT-CONDITIONED MOLECULAR GENERATION

[DEVICE] Device                    : cpu
[CONFIG] Random Seed              : 42
[CONFIG] Latent Dimension         : 32
[CONFIG] Token Embedding Dim      : 64
[CONFIG] Latent Context Dim       : 64
[CONFIG] Decoder Hidden Dim       : 128
[CONFIG] GRU Layers               : 2
[CONFIG] Batch Size               : 32
[CONFIG] Learning Rate            : 0.001
[CONFIG] Maximum Epochs           : 100
[CONFIG] Early Stop Patience      : 12
[CONFIG] Conditioning Policy       : z -> initial hidden state + z-context at every token step

[PASS] Required Phase-8 and Phase-10 files found.
[PASS] Phase-8 molecule-level split remains leakage-free.
[PASS] Phase-10 finite 32-dimensional latent vectors loaded.
[PASS] Standardized molecular structures loaded.

LATENT ↔ MOLECULAR STRUCTURE LINKAGE
Training pairs   : 613
Validation pairs : 76
Test pairs       : 78

[PASS] Training molecular structures converted to SELFIES.
[INFO] Sequence preparatio

# Phase 12 — Baseline System Consolidation, Predictor Training, and Reference-State Establishment

**Objective:** Train a multi-label biological-relevance prediction head $f_\psi(z)$ on the frozen 32-dimensional latent manifold $Z_0$, consolidate all baseline neural-network parameters into the reference state $\Theta^* = \{\theta_E^*, \theta_D^*, \theta_P^*\}$, and record cryptographic baseline checksums before continual adaptation.

* **Components & Parameters:**
  * **Encoder ($\theta_E^*$):** Frozen 2-layer EGNN variational encoder (Phase 10 checkpoint).
  * **Generative Decoder ($\theta_D^*$):** Frozen 2-layer latent-conditioned SELFIES GRU decoder (Phase 11 checkpoint).
  * **Relevance Predictor ($\theta_P^*$):** Trainable MLP ($32 \to 64 \to 32 \to K$) trained on $z = \boldsymbol{\mu}$ using multi-label BCE loss with training-only target support filtering ($K$ targets with support $\ge 5$).

* **Reference Manifest & Anti-Leakage Constraints:**
  * Label vocabulary and class weights are derived strictly from $\mathcal{D}_{\text{train}}$ biological associations.
  * Validation split ($\mathcal{D}_{\text{val}}$) guides early stopping; test split ($\mathcal{D}_{\text{test}}$) measures baseline holdout generalization.
  * Consolidates baseline reference coordinates $Z_0$ across all 767 graphs and computes SHA-256 checksums across all model checkpoints.

* **Primary Artifacts Exported:**
  * Checkpoint: `phase12_best_baseline_predictor.pt`
  * Vocabularies & Configs: `phase12_predictor_target_vocabulary.json`, `phase12_baseline_system_config.json`
  * System Reference State: `phase12_baseline_parameter_manifest.json`, `phase12_baseline_model_hashes.json`
  * Reference Manifold: `phase12_baseline_latent_reference.csv`
  * Performance Audits: `phase12_predictor_metrics.csv`, `phase12_baseline_state_summary.json`

In [12]:
# ==============================================================================
# PHASE 12 — REVISED BASELINE SYSTEM CONSOLIDATION,
# CALIBRATED PREDICTOR TRAINING, AND REFERENCE-STATE ESTABLISHMENT
# ==============================================================================
#
# FINAL BASELINE STATE:
#
#                 Θ* = { θE*, θD*, θP* }
#
# where:
#
# θE* = frozen Phase-10 geometry-aware variational encoder
# θD* = frozen Phase-11 latent-conditioned molecular generator
# θP* = calibrated DrugBank-grounded multi-label target-association predictor
#
# and:
#
# Z0 = fixed deterministic baseline latent molecular representation
#
#
# IMPORTANT REVISIONS
# -------------------
#
# 1. Phase-10 encoder is NOT retrained.
#
# 2. Phase-11 decoder is NOT retrained.
#
# 3. Predictor input latent vectors are standardized using TRAINING data only.
#
# 4. Target vocabulary is built using TRAINING molecules only.
#
# 5. Extreme class imbalance is moderated using:
#
#       sqrt(negative / positive)
#
#    with clipping.
#
#    This avoids the very aggressive positive weighting that previously
#    caused massive over-prediction.
#
# 6. Predictor checkpoint is selected using VALIDATION LOSS.
#
# 7. AFTER model selection, one global decision threshold is selected using
#    VALIDATION DATA ONLY.
#
# 8. The selected threshold maximizes validation MICRO-F1.
#
# 9. TEST DATA remains untouched until:
#
#       model checkpoint is frozen
#       +
#       threshold is frozen
#
# 10. Ranking metrics are reported separately from threshold-dependent metrics.
#
#
# THIS IS NOT:
#
# - a quantitative binding-affinity predictor
# - an IC50 predictor
# - a Ki predictor
# - a Kd predictor
# - a therapeutic-efficacy predictor
#
#
# It is:
#
# "DrugBank-grounded multi-label target-association predictor"
#
#
# NO:
#
# - Fisher information
# - EWC
# - continual adaptation
# - closed-loop optimization
# - UCB
# - parameter drift
#
# ==============================================================================


# ==============================================================================
# 1. IMPORTS
# ==============================================================================

from pathlib import Path
from datetime import datetime

import copy
import hashlib
import json
import math
import platform
import random
import shutil
import time

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    average_precision_score,
    roc_auc_score,
    hamming_loss
)

from rdkit import rdBase


# ==============================================================================
# 2. REPRODUCIBILITY
# ==============================================================================

RANDOM_SEED = 42

random.seed(
    RANDOM_SEED
)

np.random.seed(
    RANDOM_SEED
)

torch.manual_seed(
    RANDOM_SEED
)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        RANDOM_SEED
    )


DEVICE = torch.device(
    "cpu"
)


# ==============================================================================
# 3. VERIFY PROJECT STATE
# ==============================================================================

required_variables = [
    "PROCESSED_DIR",
    "DRUGBANK_CONFIG"
]


missing_variables = [
    variable
    for variable in required_variables
    if variable not in globals()
]


if missing_variables:

    raise RuntimeError(
        "Earlier phases must be executed before Phase 12.\n"
        f"Missing variables: {missing_variables}"
    )


PROCESSED_DIR = Path(
    PROCESSED_DIR
)


# ==============================================================================
# 4. CONFIGURATION
# ==============================================================================

LATENT_DIM = 32


PREDICTOR_HIDDEN_DIM_1 = 64

PREDICTOR_HIDDEN_DIM_2 = 32

PREDICTOR_DROPOUT = 0.15


BATCH_SIZE = 32

LEARNING_RATE = 1e-3

WEIGHT_DECAY = 1e-4


MAX_EPOCHS = 100

EARLY_STOPPING_PATIENCE = 12


# ------------------------------------------------------------------------------
# Target vocabulary
# ------------------------------------------------------------------------------

MIN_TARGET_SUPPORT = 5


# ------------------------------------------------------------------------------
# Class imbalance policy
#
# Raw:
#
#       neg / pos
#
# previously produced weights as large as ~121.
#
# Revised:
#
#       sqrt(neg / pos)
#
# followed by clipping.
# ------------------------------------------------------------------------------

MIN_POS_WEIGHT = 1.0

MAX_POS_WEIGHT = 10.0


# ------------------------------------------------------------------------------
# Validation-only threshold search
# ------------------------------------------------------------------------------

THRESHOLD_MIN = 0.05

THRESHOLD_MAX = 0.95

THRESHOLD_STEP = 0.01


print("=" * 130)

print(
    "PHASE 12 — REVISED BASELINE SYSTEM CONSOLIDATION, "
    "CALIBRATED PREDICTOR TRAINING, AND REFERENCE-STATE ESTABLISHMENT"
)

print("=" * 130)


print(
    f"\n[DEVICE] Device                    : {DEVICE}"
)

print(
    f"[CONFIG] Random Seed              : {RANDOM_SEED}"
)

print(
    f"[CONFIG] Latent Dimension         : {LATENT_DIM}"
)

print(
    f"[CONFIG] Predictor Hidden 1       : {PREDICTOR_HIDDEN_DIM_1}"
)

print(
    f"[CONFIG] Predictor Hidden 2       : {PREDICTOR_HIDDEN_DIM_2}"
)

print(
    f"[CONFIG] Predictor Dropout        : {PREDICTOR_DROPOUT}"
)

print(
    f"[CONFIG] Minimum Target Support   : {MIN_TARGET_SUPPORT}"
)

print(
    f"[CONFIG] Learning Rate            : {LEARNING_RATE}"
)

print(
    f"[CONFIG] Weight Decay             : {WEIGHT_DECAY}"
)

print(
    f"[CONFIG] Maximum Epochs           : {MAX_EPOCHS}"
)

print(
    f"[CONFIG] Early Stop Patience      : {EARLY_STOPPING_PATIENCE}"
)

print(
    "[CONFIG] Class Weight Policy      : "
    "sqrt(negative/positive), clipped"
)

print(
    f"[CONFIG] Positive Weight Range    : "
    f"{MIN_POS_WEIGHT} – {MAX_POS_WEIGHT}"
)

print(
    "[CONFIG] Decision Threshold       : "
    "VALIDATION-OPTIMIZED AFTER MODEL SELECTION"
)


# ==============================================================================
# 5. INPUT FILES
# ==============================================================================

PHASE8_TRAIN_FILE = (
    PROCESSED_DIR
    / "phase8_train_graphs.csv"
)


PHASE8_VALIDATION_FILE = (
    PROCESSED_DIR
    / "phase8_validation_graphs.csv"
)


PHASE8_TEST_FILE = (
    PROCESSED_DIR
    / "phase8_test_graphs.csv"
)


PHASE8_FISHER_FILE = (
    PROCESSED_DIR
    / "phase8_fisher_reference.csv"
)


PHASE10_ENCODER_FILE = (
    PROCESSED_DIR
    / "phase10_best_encoder.pt"
)


PHASE10_TRAIN_LATENT_FILE = (
    PROCESSED_DIR
    / "phase10_train_latent.csv"
)


PHASE10_VALIDATION_LATENT_FILE = (
    PROCESSED_DIR
    / "phase10_validation_latent.csv"
)


PHASE10_TEST_LATENT_FILE = (
    PROCESSED_DIR
    / "phase10_test_latent.csv"
)


PHASE11_DECODER_FILE = (
    PROCESSED_DIR
    / "phase11_best_generative_decoder.pt"
)


PHASE11_QC_FILE = (
    PROCESSED_DIR
    / "phase11_generation_qc_summary.json"
)


D_BIO_ASSOCIATION_FILE = (
    PROCESSED_DIR
    / "D_bio_association.csv"
)


required_files = [
    PHASE8_TRAIN_FILE,
    PHASE8_VALIDATION_FILE,
    PHASE8_TEST_FILE,
    PHASE8_FISHER_FILE,
    PHASE10_ENCODER_FILE,
    PHASE10_TRAIN_LATENT_FILE,
    PHASE10_VALIDATION_LATENT_FILE,
    PHASE10_TEST_LATENT_FILE,
    PHASE11_DECODER_FILE,
    D_BIO_ASSOCIATION_FILE
]


missing_files = [
    str(
        file_path
    )
    for file_path
    in required_files
    if not file_path.exists()
]


if missing_files:

    raise FileNotFoundError(
        "Required earlier-phase files are missing:\n"
        +
        "\n".join(
            missing_files
        )
    )


print(
    "\n[PASS] Required Phase-6, Phase-8, Phase-10 and "
    "Phase-11 files found."
)


# ==============================================================================
# 6. HASH HELPER
# ==============================================================================

def sha256_file(
    file_path,
    chunk_size=1024 * 1024
):

    digest = hashlib.sha256()


    with open(
        file_path,
        "rb"
    ) as file:

        while True:

            chunk = file.read(
                chunk_size
            )


            if not chunk:

                break


            digest.update(
                chunk
            )


    return digest.hexdigest()


# ==============================================================================
# 7. LOAD / VERIFY θE*
# ==============================================================================

encoder_hash_before = sha256_file(
    PHASE10_ENCODER_FILE
)


phase10_encoder_checkpoint = torch.load(
    PHASE10_ENCODER_FILE,
    map_location="cpu",
    weights_only=False
)


if (
    not isinstance(
        phase10_encoder_checkpoint,
        dict
    )
    or
    "state_dict"
    not in phase10_encoder_checkpoint
):

    raise RuntimeError(
        "Invalid Phase-10 encoder checkpoint."
    )


encoder_state_finite = all(

    torch.isfinite(
        tensor
    ).all().item()

    for tensor
    in phase10_encoder_checkpoint[
        "state_dict"
    ].values()

    if torch.is_tensor(
        tensor
    )
)


if not encoder_state_finite:

    raise RuntimeError(
        "Phase-10 encoder contains non-finite parameters."
    )


print(
    "\n[PASS] Phase-10 baseline encoder θE* verified."
)


# ==============================================================================
# 8. LOAD / VERIFY θD*
# ==============================================================================

decoder_hash_before = sha256_file(
    PHASE11_DECODER_FILE
)


phase11_decoder_checkpoint = torch.load(
    PHASE11_DECODER_FILE,
    map_location="cpu",
    weights_only=False
)


if (
    not isinstance(
        phase11_decoder_checkpoint,
        dict
    )
    or
    "state_dict"
    not in phase11_decoder_checkpoint
):

    raise RuntimeError(
        "Invalid Phase-11 decoder checkpoint."
    )


decoder_state_finite = all(

    torch.isfinite(
        tensor
    ).all().item()

    for tensor
    in phase11_decoder_checkpoint[
        "state_dict"
    ].values()

    if torch.is_tensor(
        tensor
    )
)


if not decoder_state_finite:

    raise RuntimeError(
        "Phase-11 decoder contains non-finite parameters."
    )


print(
    "[PASS] Phase-11 baseline decoder θD* verified."
)


# ==============================================================================
# 9. VERIFY PHASE-11 LATENT CONDITIONING
# ==============================================================================

phase11_latent_conditioning_verified = False


if PHASE11_QC_FILE.exists():

    with open(
        PHASE11_QC_FILE,
        "r",
        encoding="utf-8"
    ) as file:

        phase11_qc = json.load(
            file
        )


    phase11_latent_conditioning_verified = bool(
        phase11_qc.get(
            "latent_conditioning_evidence_pass",
            False
        )
    )


    if not phase11_latent_conditioning_verified:

        raise RuntimeError(
            "Phase-11 latent-conditioning evidence did not pass."
        )


    print(
        "[PASS] Phase-11 latent-conditioning evidence verified."
    )


else:

    print(
        "[WARNING] Phase-11 QC JSON unavailable. "
        "Decoder checkpoint itself was verified."
    )


# ==============================================================================
# 10. LOAD PHASE-8 CORPUS SPLITS
# ==============================================================================

D_train_manifest = pd.read_csv(
    PHASE8_TRAIN_FILE,
    dtype=str,
    keep_default_na=False
)


D_validation_manifest = pd.read_csv(
    PHASE8_VALIDATION_FILE,
    dtype=str,
    keep_default_na=False
)


D_test_manifest = pd.read_csv(
    PHASE8_TEST_FILE,
    dtype=str,
    keep_default_na=False
)


D_fisher_reference = pd.read_csv(
    PHASE8_FISHER_FILE,
    dtype=str,
    keep_default_na=False
)


D_train_manifest_original = (
    D_train_manifest.copy(
        deep=True
    )
)


D_validation_manifest_original = (
    D_validation_manifest.copy(
        deep=True
    )
)


D_test_manifest_original = (
    D_test_manifest.copy(
        deep=True
    )
)


D_fisher_original = (
    D_fisher_reference.copy(
        deep=True
    )
)


train_graph_ids = set(
    D_train_manifest[
        "graph_id"
    ].astype(str)
)


validation_graph_ids = set(
    D_validation_manifest[
        "graph_id"
    ].astype(str)
)


test_graph_ids = set(
    D_test_manifest[
        "graph_id"
    ].astype(str)
)


fisher_graph_ids = set(
    D_fisher_reference[
        "graph_id"
    ].astype(str)
)


if train_graph_ids & validation_graph_ids:

    raise RuntimeError(
        "Train-validation overlap detected."
    )


if train_graph_ids & test_graph_ids:

    raise RuntimeError(
        "Train-test overlap detected."
    )


if validation_graph_ids & test_graph_ids:

    raise RuntimeError(
        "Validation-test overlap detected."
    )


if not fisher_graph_ids.issubset(
    train_graph_ids
):

    raise RuntimeError(
        "Fisher reference is not training-only."
    )


print("\n" + "=" * 130)
print("PHASE-8 BASELINE CORPUS")
print("=" * 130)

print(
    f"Training graphs         : {len(D_train_manifest)}"
)

print(
    f"Validation graphs       : {len(D_validation_manifest)}"
)

print(
    f"Test graphs             : {len(D_test_manifest)}"
)

print(
    f"Fisher reference graphs : {len(D_fisher_reference)}"
)

print(
    "[PASS] Dataset splits remain leakage-free."
)

print(
    "[PASS] Fisher reference remains training-only."
)


# ==============================================================================
# 11. LOAD PHASE-10 LATENT REPRESENTATIONS
# ==============================================================================

D_train_latent = pd.read_csv(
    PHASE10_TRAIN_LATENT_FILE,
    dtype={
        "graph_id": str,
        "drug_id": str
    }
)


D_validation_latent = pd.read_csv(
    PHASE10_VALIDATION_LATENT_FILE,
    dtype={
        "graph_id": str,
        "drug_id": str
    }
)


D_test_latent = pd.read_csv(
    PHASE10_TEST_LATENT_FILE,
    dtype={
        "graph_id": str,
        "drug_id": str
    }
)


latent_columns = [
    f"z_{index}"
    for index
    in range(
        LATENT_DIM
    )
]


def validate_latent_file(
    latent_dataframe,
    manifest_dataframe,
    split_name
):

    required = [
        "graph_id",
        "drug_id"
    ] + latent_columns


    missing = [
        column
        for column
        in required
        if column not in latent_dataframe.columns
    ]


    if missing:

        raise RuntimeError(
            f"{split_name} latent file missing: {missing}"
        )


    if len(
        latent_dataframe
    ) != len(
        manifest_dataframe
    ):

        raise RuntimeError(
            f"{split_name} latent count mismatch."
        )


    latent_ids = set(
        latent_dataframe[
            "graph_id"
        ].astype(str)
    )


    manifest_ids = set(
        manifest_dataframe[
            "graph_id"
        ].astype(str)
    )


    if latent_ids != manifest_ids:

        raise RuntimeError(
            f"{split_name} latent graph membership mismatch."
        )


    values = (
        latent_dataframe[
            latent_columns
        ]
        .astype(float)
        .to_numpy()
    )


    if not np.isfinite(
        values
    ).all():

        raise RuntimeError(
            f"{split_name} contains non-finite latent values."
        )


    return True


validate_latent_file(
    D_train_latent,
    D_train_manifest,
    "TRAIN"
)


validate_latent_file(
    D_validation_latent,
    D_validation_manifest,
    "VALIDATION"
)


validate_latent_file(
    D_test_latent,
    D_test_manifest,
    "TEST"
)


print(
    "\n[PASS] Phase-10 deterministic latent vectors verified."
)


# ==============================================================================
# 12. ESTABLISH Z0
# ==============================================================================

def with_split(
    dataframe,
    split_name
):

    result = dataframe.copy(
        deep=True
    )


    result.insert(
        0,
        "split",
        split_name
    )


    return result


Z0 = pd.concat(
    [
        with_split(
            D_train_latent,
            "TRAIN"
        ),

        with_split(
            D_validation_latent,
            "VALIDATION"
        ),

        with_split(
            D_test_latent,
            "TEST"
        )
    ],
    ignore_index=True
)


Z0[
    "is_fisher_reference"
] = Z0[
    "graph_id"
].astype(str).isin(
    fisher_graph_ids
)


Z0_values = (
    Z0[
        latent_columns
    ]
    .astype(float)
    .to_numpy()
)


if not np.isfinite(
    Z0_values
).all():

    raise RuntimeError(
        "Z0 contains non-finite values."
    )


print(
    f"[PASS] Baseline latent reference Z0 established "
    f"for {len(Z0)} molecules."
)


# ==============================================================================
# 13. TRAINING-ONLY LATENT NORMALIZATION
# ==============================================================================

X_train_raw = (
    D_train_latent[
        latent_columns
    ]
    .astype(float)
    .to_numpy(
        dtype=np.float32
    )
)


X_validation_raw = (
    D_validation_latent[
        latent_columns
    ]
    .astype(float)
    .to_numpy(
        dtype=np.float32
    )
)


X_test_raw = (
    D_test_latent[
        latent_columns
    ]
    .astype(float)
    .to_numpy(
        dtype=np.float32
    )
)


latent_mean = X_train_raw.mean(
    axis=0
)


latent_std = X_train_raw.std(
    axis=0
)


near_zero_std_mask = (
    latent_std
    <
    1e-8
)


latent_std_safe = latent_std.copy()


latent_std_safe[
    near_zero_std_mask
] = 1.0


X_train = (
    X_train_raw
    -
    latent_mean
) / latent_std_safe


X_validation = (
    X_validation_raw
    -
    latent_mean
) / latent_std_safe


X_test = (
    X_test_raw
    -
    latent_mean
) / latent_std_safe


if not np.isfinite(
    X_train
).all():

    raise RuntimeError(
        "Training normalized latent values are non-finite."
    )


if not np.isfinite(
    X_validation
).all():

    raise RuntimeError(
        "Validation normalized latent values are non-finite."
    )


if not np.isfinite(
    X_test
).all():

    raise RuntimeError(
        "Test normalized latent values are non-finite."
    )


print("\n" + "=" * 130)
print("TRAINING-ONLY PREDICTOR LATENT NORMALIZATION")
print("=" * 130)

print(
    f"Latent dimensions                   : {LATENT_DIM}"
)

print(
    f"Near-zero training std dimensions   : "
    f"{int(near_zero_std_mask.sum())}"
)

print(
    "[PASS] Predictor latent normalization fitted using "
    "training molecules only."
)


# ==============================================================================
# 14. LOAD DRUGBANK BIOLOGICAL ASSOCIATIONS
# ==============================================================================

D_bio = pd.read_csv(
    D_BIO_ASSOCIATION_FILE,
    dtype=str,
    keep_default_na=False
)


for column in [
    "drug_id",
    "target_id"
]:

    if column not in D_bio.columns:

        raise RuntimeError(
            f"D_bio_association missing {column}."
        )


D_bio = (
    D_bio[
        D_bio[
            "target_id"
        ]
        .astype(str)
        .str.strip()
        !=
        ""
    ]
    .copy()
)


bio_unique_pairs = (
    D_bio[
        [
            "drug_id",
            "target_id"
        ]
    ]
    .drop_duplicates()
    .reset_index(
        drop=True
    )
)


print("\n" + "=" * 130)
print("DRUGBANK BIOLOGICAL ASSOCIATION INPUT")
print("=" * 130)

print(
    f"Association rows            : {len(D_bio)}"
)

print(
    f"Unique drug-target pairs    : {len(bio_unique_pairs)}"
)

print(
    f"Unique target IDs           : "
    f"{bio_unique_pairs['target_id'].nunique()}"
)


# ==============================================================================
# 15. DRUG ID SETS
# ==============================================================================

train_drug_ids = set(
    D_train_latent[
        "drug_id"
    ].astype(str)
)


validation_drug_ids = set(
    D_validation_latent[
        "drug_id"
    ].astype(str)
)


test_drug_ids = set(
    D_test_latent[
        "drug_id"
    ].astype(str)
)


# ==============================================================================
# 16. TRAINING-ONLY TARGET VOCABULARY
# ==============================================================================

training_bio_pairs = (
    bio_unique_pairs[
        bio_unique_pairs[
            "drug_id"
        ].isin(
            train_drug_ids
        )
    ]
    .copy()
)


training_target_support = (
    training_bio_pairs
    .groupby(
        "target_id"
    )[
        "drug_id"
    ]
    .nunique()
    .sort_values(
        ascending=False
    )
)


selected_target_support = (
    training_target_support[
        training_target_support
        >=
        MIN_TARGET_SUPPORT
    ]
)


selected_target_ids = list(
    selected_target_support.index
)


NUM_TARGET_CLASSES = len(
    selected_target_ids
)


if NUM_TARGET_CLASSES == 0:

    raise RuntimeError(
        "No targets satisfy MIN_TARGET_SUPPORT."
    )


target_to_index = {
    target_id:
        index
    for index, target_id
    in enumerate(
        selected_target_ids
    )
}


print("\n" + "=" * 130)
print("TRAINING-ONLY TARGET VOCABULARY")
print("=" * 130)

print(
    f"Targets observed in training       : "
    f"{len(training_target_support)}"
)

print(
    f"Minimum support requirement        : "
    f"{MIN_TARGET_SUPPORT}"
)

print(
    f"Selected target classes            : "
    f"{NUM_TARGET_CLASSES}"
)

print(
    f"Maximum selected support           : "
    f"{int(selected_target_support.max())}"
)

print(
    f"Minimum selected support           : "
    f"{int(selected_target_support.min())}"
)


# ==============================================================================
# 17. TARGET NAMES
# ==============================================================================

target_name_lookup = {}


if "target_name" in D_bio.columns:

    target_name_table = (
        D_bio[
            D_bio[
                "target_name"
            ]
            .astype(str)
            .str.strip()
            !=
            ""
        ][
            [
                "target_id",
                "target_name"
            ]
        ]
        .drop_duplicates(
            subset=[
                "target_id"
            ]
        )
    )


    target_name_lookup = dict(
        zip(
            target_name_table[
                "target_id"
            ],
            target_name_table[
                "target_name"
            ]
        )
    )


target_vocabulary_records = []


for target_id in selected_target_ids:

    target_vocabulary_records.append(
        {
            "target_index":
                target_to_index[
                    target_id
                ],

            "target_id":
                target_id,

            "target_name":
                target_name_lookup.get(
                    target_id,
                    ""
                ),

            "training_unique_drug_support":
                int(
                    selected_target_support[
                        target_id
                    ]
                )
        }
    )


target_vocabulary_df = pd.DataFrame(
    target_vocabulary_records
)


# ==============================================================================
# 18. BUILD MULTI-LABEL MATRICES
# ==============================================================================

selected_pairs = (
    bio_unique_pairs[
        bio_unique_pairs[
            "target_id"
        ].isin(
            selected_target_ids
        )
    ]
)


drug_to_targets = {}


for drug_id, group in selected_pairs.groupby(
    "drug_id"
):

    drug_to_targets[
        str(
            drug_id
        )
    ] = set(
        group[
            "target_id"
        ].astype(str)
    )


def build_multilabel_matrix(
    latent_dataframe
):

    Y = np.zeros(
        (
            len(
                latent_dataframe
            ),
            NUM_TARGET_CLASSES
        ),
        dtype=np.float32
    )


    for row_index, drug_id in enumerate(
        latent_dataframe[
            "drug_id"
        ].astype(str)
    ):

        targets = drug_to_targets.get(
            drug_id,
            set()
        )


        for target_id in targets:

            class_index = target_to_index.get(
                target_id
            )


            if class_index is not None:

                Y[
                    row_index,
                    class_index
                ] = 1.0


    return Y


Y_train = build_multilabel_matrix(
    D_train_latent
)


Y_validation = build_multilabel_matrix(
    D_validation_latent
)


Y_test = build_multilabel_matrix(
    D_test_latent
)


# ==============================================================================
# 19. LABEL DISTRIBUTION
# ==============================================================================

train_labels_per_molecule = Y_train.sum(
    axis=1
)


validation_labels_per_molecule = Y_validation.sum(
    axis=1
)


test_labels_per_molecule = Y_test.sum(
    axis=1
)


train_molecules_with_labels = int(
    (
        train_labels_per_molecule
        >
        0
    ).sum()
)


validation_molecules_with_labels = int(
    (
        validation_labels_per_molecule
        >
        0
    ).sum()
)


test_molecules_with_labels = int(
    (
        test_labels_per_molecule
        >
        0
    ).sum()
)


print("\n" + "=" * 130)
print("MULTI-LABEL PREDICTOR DATASET")
print("=" * 130)

print(
    f"Training molecules                       : {len(Y_train)}"
)

print(
    f"Validation molecules                     : {len(Y_validation)}"
)

print(
    f"Test molecules                           : {len(Y_test)}"
)

print(
    f"Target classes                           : {NUM_TARGET_CLASSES}"
)

print(
    f"Training molecules with selected targets : "
    f"{train_molecules_with_labels}"
)

print(
    f"Validation molecules with selected tgt.  : "
    f"{validation_molecules_with_labels}"
)

print(
    f"Test molecules with selected targets     : "
    f"{test_molecules_with_labels}"
)

print(
    f"Mean training labels per molecule        : "
    f"{float(train_labels_per_molecule.mean()):.4f}"
)


# ==============================================================================
# 20. ASSOCIATION COVERAGE
# ==============================================================================

def association_coverage(
    drug_ids
):

    subset = bio_unique_pairs[
        bio_unique_pairs[
            "drug_id"
        ].isin(
            drug_ids
        )
    ]


    total = len(
        subset
    )


    selected = int(
        subset[
            "target_id"
        ].isin(
            selected_target_ids
        ).sum()
    )


    coverage = (
        selected
        /
        max(
            total,
            1
        )
    )


    return (
        total,
        selected,
        coverage
    )


train_association_total, train_association_selected, train_coverage = (
    association_coverage(
        train_drug_ids
    )
)


validation_association_total, validation_association_selected, validation_coverage = (
    association_coverage(
        validation_drug_ids
    )
)


test_association_total, test_association_selected, test_coverage = (
    association_coverage(
        test_drug_ids
    )
)


print(
    f"\nTraining association coverage   : {train_coverage:.2%}"
)

print(
    f"Validation association coverage : {validation_coverage:.2%}"
)

print(
    f"Test association coverage       : {test_coverage:.2%}"
)


# ==============================================================================
# 21. MODERATED TRAINING-ONLY POSITIVE CLASS WEIGHTS
# ==============================================================================

positive_counts = Y_train.sum(
    axis=0
)


negative_counts = (
    len(
        Y_train
    )
    -
    positive_counts
)


raw_pos_weight = (
    negative_counts
    /
    np.maximum(
        positive_counts,
        1.0
    )
)


sqrt_pos_weight = np.sqrt(
    raw_pos_weight
)


positive_class_weights = np.clip(
    sqrt_pos_weight,
    MIN_POS_WEIGHT,
    MAX_POS_WEIGHT
)


pos_weight_tensor = torch.tensor(
    positive_class_weights,
    dtype=torch.float32,
    device=DEVICE
)


print("\n" + "=" * 130)
print("TRAINING-ONLY CLASS-IMBALANCE WEIGHTING")
print("=" * 130)

print(
    f"Raw positive-weight minimum     : "
    f"{float(raw_pos_weight.min()):.4f}"
)

print(
    f"Raw positive-weight median      : "
    f"{float(np.median(raw_pos_weight)):.4f}"
)

print(
    f"Raw positive-weight maximum     : "
    f"{float(raw_pos_weight.max()):.4f}"
)

print(
    f"Applied positive-weight minimum : "
    f"{float(positive_class_weights.min()):.4f}"
)

print(
    f"Applied positive-weight median  : "
    f"{float(np.median(positive_class_weights)):.4f}"
)

print(
    f"Applied positive-weight maximum : "
    f"{float(positive_class_weights.max()):.4f}"
)


# ==============================================================================
# 22. DATASET
# ==============================================================================

class LatentTargetDataset(Dataset):

    def __init__(
        self,
        X,
        Y
    ):

        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )


        self.Y = torch.tensor(
            Y,
            dtype=torch.float32
        )


    def __len__(
        self
    ):

        return self.X.shape[
            0
        ]


    def __getitem__(
        self,
        index
    ):

        return (
            self.X[
                index
            ],
            self.Y[
                index
            ]
        )


train_dataset = LatentTargetDataset(
    X_train,
    Y_train
)


validation_dataset = LatentTargetDataset(
    X_validation,
    Y_validation
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    generator=torch.Generator().manual_seed(
        RANDOM_SEED
    )
)


validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)


# ==============================================================================
# 23. PREDICTOR ARCHITECTURE
# ==============================================================================

class BiologicalTargetAssociationPredictor(nn.Module):

    def __init__(
        self,
        latent_dim,
        hidden_dim_1,
        hidden_dim_2,
        output_dim,
        dropout
    ):

        super().__init__()


        self.network = nn.Sequential(

            nn.Linear(
                latent_dim,
                hidden_dim_1
            ),

            nn.LayerNorm(
                hidden_dim_1
            ),

            nn.SiLU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                hidden_dim_1,
                hidden_dim_2
            ),

            nn.LayerNorm(
                hidden_dim_2
            ),

            nn.SiLU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                hidden_dim_2,
                output_dim
            )
        )


    def forward(
        self,
        latent
    ):

        return self.network(
            latent
        )


# ==============================================================================
# 24. INITIALIZE PREDICTOR
# ==============================================================================

torch.manual_seed(
    RANDOM_SEED
)


predictor = BiologicalTargetAssociationPredictor(
    latent_dim=LATENT_DIM,
    hidden_dim_1=PREDICTOR_HIDDEN_DIM_1,
    hidden_dim_2=PREDICTOR_HIDDEN_DIM_2,
    output_dim=NUM_TARGET_CLASSES,
    dropout=PREDICTOR_DROPOUT
).to(
    DEVICE
)


predictor_parameter_count = sum(
    parameter.numel()
    for parameter
    in predictor.parameters()
)


print("\n" + "=" * 130)
print("BASELINE TARGET-ASSOCIATION PREDICTOR")
print("=" * 130)

print(
    f"Input latent dimension  : {LATENT_DIM}"
)

print(
    f"Hidden dimension 1      : {PREDICTOR_HIDDEN_DIM_1}"
)

print(
    f"Hidden dimension 2      : {PREDICTOR_HIDDEN_DIM_2}"
)

print(
    f"Output target classes   : {NUM_TARGET_CLASSES}"
)

print(
    f"Predictor parameters    : {predictor_parameter_count:,}"
)


# ==============================================================================
# 25. LOSS + OPTIMIZER
# ==============================================================================

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight_tensor
)


optimizer = torch.optim.AdamW(
    predictor.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


# ==============================================================================
# 26. TRAIN EPOCH
# ==============================================================================

def train_predictor_epoch(
    model,
    loader
):

    model.train()


    total_loss = 0.0

    sample_count = 0


    for latent, target in loader:

        latent = latent.to(
            DEVICE
        )


        target = target.to(
            DEVICE
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        logits = model(
            latent
        )


        loss = criterion(
            logits,
            target
        )


        if not torch.isfinite(
            loss
        ):

            raise RuntimeError(
                "Non-finite predictor training loss."
            )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0
        )


        optimizer.step()


        batch_n = latent.shape[
            0
        ]


        total_loss += (
            float(
                loss.item()
            )
            *
            batch_n
        )


        sample_count += (
            batch_n
        )


    return (
        total_loss
        /
        max(
            sample_count,
            1
        )
    )


# ==============================================================================
# 27. VALIDATION LOSS
# ==============================================================================

def evaluate_predictor_loss(
    model,
    loader
):

    model.eval()


    total_loss = 0.0

    sample_count = 0


    with torch.no_grad():

        for latent, target in loader:

            latent = latent.to(
                DEVICE
            )


            target = target.to(
                DEVICE
            )


            logits = model(
                latent
            )


            loss = criterion(
                logits,
                target
            )


            batch_n = latent.shape[
                0
            ]


            total_loss += (
                float(
                    loss.item()
                )
                *
                batch_n
            )


            sample_count += (
                batch_n
            )


    return (
        total_loss
        /
        max(
            sample_count,
            1
        )
    )


# ==============================================================================
# 28. OUTPUT PATHS
# ==============================================================================

PHASE12_PREDICTOR_CONFIG_FILE = (
    PROCESSED_DIR
    / "phase12_predictor_config.json"
)


PHASE12_TARGET_VOCAB_FILE = (
    PROCESSED_DIR
    / "phase12_predictor_target_vocabulary.json"
)


PHASE12_TARGET_VOCAB_CSV_FILE = (
    PROCESSED_DIR
    / "phase12_predictor_target_vocabulary.csv"
)


PHASE12_LATENT_NORMALIZATION_FILE = (
    PROCESSED_DIR
    / "phase12_predictor_latent_normalization.json"
)


PHASE12_CLASS_WEIGHTS_FILE = (
    PROCESSED_DIR
    / "phase12_predictor_class_weights.csv"
)


PHASE12_HISTORY_FILE = (
    PROCESSED_DIR
    / "phase12_predictor_training_history.csv"
)


PHASE12_PREDICTOR_FILE = (
    PROCESSED_DIR
    / "phase12_best_baseline_predictor.pt"
)


PHASE12_THRESHOLD_SEARCH_FILE = (
    PROCESSED_DIR
    / "phase12_validation_threshold_search.csv"
)


PHASE12_THRESHOLD_CONFIG_FILE = (
    PROCESSED_DIR
    / "phase12_predictor_threshold.json"
)


PHASE12_METRICS_FILE = (
    PROCESSED_DIR
    / "phase12_predictor_metrics.csv"
)


PHASE12_Z0_FILE = (
    PROCESSED_DIR
    / "phase12_baseline_latent_reference.csv"
)


PHASE12_ENCODER_SNAPSHOT_FILE = (
    PROCESSED_DIR
    / "phase12_thetaE_baseline_encoder.pt"
)


PHASE12_DECODER_SNAPSHOT_FILE = (
    PROCESSED_DIR
    / "phase12_thetaD_baseline_decoder.pt"
)


PHASE12_SYSTEM_CONFIG_FILE = (
    PROCESSED_DIR
    / "phase12_baseline_system_config.json"
)


PHASE12_PARAMETER_MANIFEST_FILE = (
    PROCESSED_DIR
    / "phase12_baseline_parameter_manifest.json"
)


PHASE12_MODEL_HASHES_FILE = (
    PROCESSED_DIR
    / "phase12_baseline_model_hashes.json"
)


PHASE12_REFERENCE_BUNDLE_FILE = (
    PROCESSED_DIR
    / "phase12_theta_star_reference_bundle.pt"
)


PHASE12_SUMMARY_CSV_FILE = (
    PROCESSED_DIR
    / "phase12_baseline_state_summary.csv"
)


PHASE12_SUMMARY_JSON_FILE = (
    PROCESSED_DIR
    / "phase12_baseline_state_summary.json"
)


# ==============================================================================
# 29. SAVE TARGET VOCABULARY
# ==============================================================================

target_vocabulary_metadata = {
    "label_source":
        "DrugBank Phase-6 biological target associations",

    "task":
        "multi-label target-association prediction",

    "binding_affinity_task":
        False,

    "vocabulary_source":
        "training molecules only",

    "minimum_training_unique_drug_support":
        MIN_TARGET_SUPPORT,

    "number_of_target_classes":
        NUM_TARGET_CLASSES,

    "targets":
        target_vocabulary_records
}


with open(
    PHASE12_TARGET_VOCAB_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        target_vocabulary_metadata,
        file,
        indent=4,
        ensure_ascii=False
    )


target_vocabulary_df.to_csv(
    PHASE12_TARGET_VOCAB_CSV_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 30. SAVE LATENT NORMALIZATION
# ==============================================================================

latent_normalization_metadata = {
    "fit_split":
        "TRAIN",

    "latent_dimension":
        LATENT_DIM,

    "mean":
        latent_mean.tolist(),

    "std":
        latent_std.tolist(),

    "safe_std":
        latent_std_safe.tolist(),

    "near_zero_std_dimensions":
        np.where(
            near_zero_std_mask
        )[
            0
        ].tolist()
}


with open(
    PHASE12_LATENT_NORMALIZATION_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        latent_normalization_metadata,
        file,
        indent=4
    )


# ==============================================================================
# 31. SAVE CLASS WEIGHTS
# ==============================================================================

class_weights_df = target_vocabulary_df.copy()


class_weights_df[
    "positive_training_count"
] = positive_counts.astype(
    int
)


class_weights_df[
    "negative_training_count"
] = negative_counts.astype(
    int
)


class_weights_df[
    "raw_pos_weight"
] = raw_pos_weight


class_weights_df[
    "sqrt_pos_weight"
] = sqrt_pos_weight


class_weights_df[
    "applied_pos_weight"
] = positive_class_weights


class_weights_df.to_csv(
    PHASE12_CLASS_WEIGHTS_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 32. PREDICTOR CONFIG
# ==============================================================================

predictor_config = {
    "phase":
        "Phase 12 revised calibrated baseline predictor",

    "predictor_type":
        "DrugBank-grounded multi-label target-association predictor",

    "binding_affinity_predictor":
        False,

    "latent_source":
        "Phase-10 deterministic mu",

    "latent_normalization":
        "training-only z-score",

    "latent_dimension":
        LATENT_DIM,

    "hidden_dimension_1":
        PREDICTOR_HIDDEN_DIM_1,

    "hidden_dimension_2":
        PREDICTOR_HIDDEN_DIM_2,

    "dropout":
        PREDICTOR_DROPOUT,

    "target_classes":
        NUM_TARGET_CLASSES,

    "minimum_target_support":
        MIN_TARGET_SUPPORT,

    "loss":
        "BCEWithLogitsLoss",

    "class_weight_policy":
        "sqrt(negative/positive), clipped",

    "minimum_positive_weight":
        MIN_POS_WEIGHT,

    "maximum_positive_weight":
        MAX_POS_WEIGHT,

    "optimizer":
        "AdamW",

    "learning_rate":
        LEARNING_RATE,

    "weight_decay":
        WEIGHT_DECAY,

    "batch_size":
        BATCH_SIZE,

    "maximum_epochs":
        MAX_EPOCHS,

    "early_stopping_patience":
        EARLY_STOPPING_PATIENCE,

    "checkpoint_selection":
        "minimum validation BCE loss",

    "threshold_selection":
        "global threshold maximizing validation micro-F1",

    "test_used_for_checkpoint_selection":
        False,

    "test_used_for_threshold_selection":
        False,

    "encoder_retrained":
        False,

    "decoder_retrained":
        False
}


with open(
    PHASE12_PREDICTOR_CONFIG_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        predictor_config,
        file,
        indent=4,
        ensure_ascii=False
    )


# ==============================================================================
# 33. TRAIN PREDICTOR
# ==============================================================================

print("\n" + "=" * 130)
print("CALIBRATED BASELINE PREDICTOR TRAINING")
print("=" * 130)


training_history = []


best_validation_loss = float(
    "inf"
)


best_epoch = 0


epochs_without_improvement = 0


training_start_time = time.time()


for epoch in range(
    1,
    MAX_EPOCHS
    +
    1
):

    train_loss = train_predictor_epoch(
        predictor,
        train_loader
    )


    validation_loss = evaluate_predictor_loss(
        predictor,
        validation_loader
    )


    improved = (
        validation_loss
        <
        best_validation_loss
    )


    if improved:

        best_validation_loss = validation_loss

        best_epoch = epoch

        epochs_without_improvement = 0


        torch.save(
            {
                "model_status":
                    "TRAINED_CALIBRATED_BASELINE_TARGET_ASSOCIATION_PREDICTOR",

                "epoch":
                    epoch,

                "validation_loss":
                    validation_loss,

                "state_dict":
                    predictor.state_dict(),

                "config":
                    predictor_config,

                "target_vocabulary":
                    target_vocabulary_metadata,

                "latent_normalization":
                    latent_normalization_metadata,

                "positive_class_weights":
                    positive_class_weights.tolist()
            },
            PHASE12_PREDICTOR_FILE
        )


    else:

        epochs_without_improvement += 1


    training_history.append(
        {
            "epoch":
                epoch,

            "train_loss":
                train_loss,

            "validation_loss":
                validation_loss,

            "best_checkpoint":
                bool(
                    improved
                )
        }
    )


    print(
        f"[EPOCH {epoch:03d}] "
        f"train_loss={train_loss:.6f} | "
        f"val_loss={validation_loss:.6f}"
        +
        (
            " | BEST"
            if improved
            else ""
        )
    )


    if (
        epochs_without_improvement
        >=
        EARLY_STOPPING_PATIENCE
    ):

        print(
            f"\n[EARLY STOP] No validation improvement for "
            f"{EARLY_STOPPING_PATIENCE} epochs."
        )

        break


training_duration_seconds = (
    time.time()
    -
    training_start_time
)


epochs_completed = len(
    training_history
)


training_history_df = pd.DataFrame(
    training_history
)


training_history_df.to_csv(
    PHASE12_HISTORY_FILE,
    index=False,
    encoding="utf-8"
)


print(
    f"\n[PASS] Predictor training completed after "
    f"{epochs_completed} epochs."
)

print(
    f"[BEST] Best epoch           : {best_epoch}"
)

print(
    f"[BEST] Best validation loss : "
    f"{best_validation_loss:.8f}"
)


# ==============================================================================
# 34. RELOAD BEST PREDICTOR
# ==============================================================================

best_predictor_checkpoint = torch.load(
    PHASE12_PREDICTOR_FILE,
    map_location=DEVICE,
    weights_only=False
)


best_predictor = BiologicalTargetAssociationPredictor(
    latent_dim=LATENT_DIM,
    hidden_dim_1=PREDICTOR_HIDDEN_DIM_1,
    hidden_dim_2=PREDICTOR_HIDDEN_DIM_2,
    output_dim=NUM_TARGET_CLASSES,
    dropout=PREDICTOR_DROPOUT
).to(
    DEVICE
)


best_predictor.load_state_dict(
    best_predictor_checkpoint[
        "state_dict"
    ]
)


best_predictor.eval()


best_predictor_reloadable = True


print(
    "\n[PASS] Best predictor checkpoint reloaded."
)


# ==============================================================================
# 35. PROBABILITY PREDICTION
# ==============================================================================

def predict_probabilities(
    model,
    X
):

    model.eval()


    tensor = torch.tensor(
        X,
        dtype=torch.float32,
        device=DEVICE
    )


    with torch.no_grad():

        logits = model(
            tensor
        )


        probabilities = torch.sigmoid(
            logits
        )


    return (
        logits.cpu().numpy(),
        probabilities.cpu().numpy()
    )


# ==============================================================================
# 36. VALIDATION PROBABILITIES
#
# IMPORTANT:
#
# Test has NOT yet been evaluated.
# ==============================================================================

validation_logits, validation_probabilities = (
    predict_probabilities(
        best_predictor,
        X_validation
    )
)


if not np.isfinite(
    validation_probabilities
).all():

    raise RuntimeError(
        "Validation predictor probabilities are non-finite."
    )


# ==============================================================================
# 37. VALIDATION-ONLY THRESHOLD OPTIMIZATION
# ==============================================================================

print("\n" + "=" * 130)
print("VALIDATION-ONLY DECISION-THRESHOLD OPTIMIZATION")
print("=" * 130)


threshold_values = np.arange(
    THRESHOLD_MIN,
    THRESHOLD_MAX
    +
    THRESHOLD_STEP / 2,
    THRESHOLD_STEP
)


threshold_records = []


for threshold in threshold_values:

    prediction = (
        validation_probabilities
        >=
        threshold
    ).astype(
        np.int32
    )


    micro_f1 = f1_score(
        Y_validation,
        prediction,
        average="micro",
        zero_division=0
    )


    micro_precision = precision_score(
        Y_validation,
        prediction,
        average="micro",
        zero_division=0
    )


    micro_recall = recall_score(
        Y_validation,
        prediction,
        average="micro",
        zero_division=0
    )


    predicted_positives = int(
        prediction.sum()
    )


    actual_positives = int(
        Y_validation.sum()
    )


    positive_count_error = abs(
        predicted_positives
        -
        actual_positives
    )


    threshold_records.append(
        {
            "threshold":
                float(
                    threshold
                ),

            "micro_f1":
                float(
                    micro_f1
                ),

            "micro_precision":
                float(
                    micro_precision
                ),

            "micro_recall":
                float(
                    micro_recall
                ),

            "actual_positive_labels":
                actual_positives,

            "predicted_positive_labels":
                predicted_positives,

            "positive_count_absolute_error":
                positive_count_error
        }
    )


threshold_search_df = pd.DataFrame(
    threshold_records
)


# ------------------------------------------------------------------------------
# Primary optimization:
#
#   1. Maximum validation micro-F1
#   2. If tied, higher precision
#   3. If still tied, smaller predicted-positive count error
# ------------------------------------------------------------------------------

threshold_search_sorted = (
    threshold_search_df
    .sort_values(
        by=[
            "micro_f1",
            "micro_precision",
            "positive_count_absolute_error"
        ],
        ascending=[
            False,
            False,
            True
        ]
    )
    .reset_index(
        drop=True
    )
)


best_threshold_row = (
    threshold_search_sorted.iloc[
        0
    ]
)


OPTIMAL_THRESHOLD = float(
    best_threshold_row[
        "threshold"
    ]
)


validation_predictions = (
    validation_probabilities
    >=
    OPTIMAL_THRESHOLD
).astype(
    np.int32
)


threshold_search_df[
    "selected_threshold"
] = (
    np.isclose(
        threshold_search_df[
            "threshold"
        ],
        OPTIMAL_THRESHOLD
    )
)


threshold_search_df.to_csv(
    PHASE12_THRESHOLD_SEARCH_FILE,
    index=False,
    encoding="utf-8"
)


threshold_metadata = {
    "selection_split":
        "VALIDATION",

    "test_data_used":
        False,

    "threshold_type":
        "single global threshold",

    "optimization_metric":
        "micro-F1",

    "tie_break_1":
        "higher micro precision",

    "tie_break_2":
        "lower positive-count absolute error",

    "search_minimum":
        THRESHOLD_MIN,

    "search_maximum":
        THRESHOLD_MAX,

    "search_step":
        THRESHOLD_STEP,

    "selected_threshold":
        OPTIMAL_THRESHOLD,

    "validation_micro_f1":
        float(
            best_threshold_row[
                "micro_f1"
            ]
        ),

    "validation_micro_precision":
        float(
            best_threshold_row[
                "micro_precision"
            ]
        ),

    "validation_micro_recall":
        float(
            best_threshold_row[
                "micro_recall"
            ]
        ),

    "validation_actual_positive_labels":
        int(
            best_threshold_row[
                "actual_positive_labels"
            ]
        ),

    "validation_predicted_positive_labels":
        int(
            best_threshold_row[
                "predicted_positive_labels"
            ]
        )
}


with open(
    PHASE12_THRESHOLD_CONFIG_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        threshold_metadata,
        file,
        indent=4
    )


print(
    f"Selected threshold               : "
    f"{OPTIMAL_THRESHOLD:.4f}"
)

print(
    f"Validation micro-F1              : "
    f"{float(best_threshold_row['micro_f1']):.4f}"
)

print(
    f"Validation micro-precision       : "
    f"{float(best_threshold_row['micro_precision']):.4f}"
)

print(
    f"Validation micro-recall          : "
    f"{float(best_threshold_row['micro_recall']):.4f}"
)

print(
    f"Actual validation positives      : "
    f"{int(best_threshold_row['actual_positive_labels'])}"
)

print(
    f"Predicted validation positives   : "
    f"{int(best_threshold_row['predicted_positive_labels'])}"
)

print(
    "[PASS] Threshold selected using validation data only."
)


# ==============================================================================
# 38. FREEZE MODEL + THRESHOLD
# ==============================================================================

for parameter in best_predictor.parameters():

    parameter.requires_grad = False


predictor_frozen = all(
    not parameter.requires_grad
    for parameter
    in best_predictor.parameters()
)


if not predictor_frozen:

    raise RuntimeError(
        "Predictor freeze failed."
    )


print(
    "\n[PASS] θP* model parameters frozen."
)

print(
    f"[PASS] Decision threshold frozen at {OPTIMAL_THRESHOLD:.4f}."
)


# ==============================================================================
# 39. TEST EVALUATION
#
# THIS IS THE FIRST TEST EVALUATION.
#
# At this point:
#
# model checkpoint = frozen
# threshold = frozen
# ==============================================================================

test_logits, test_probabilities = (
    predict_probabilities(
        best_predictor,
        X_test
    )
)


if not np.isfinite(
    test_probabilities
).all():

    raise RuntimeError(
        "Test probabilities are non-finite."
    )


test_predictions = (
    test_probabilities
    >=
    OPTIMAL_THRESHOLD
).astype(
    np.int32
)


print(
    "\n[PASS] Test set evaluated only after checkpoint "
    "AND threshold were frozen."
)


# ==============================================================================
# 40. SAFE RANKING METRICS
# ==============================================================================

def safe_average_precision(
    y_true,
    y_probability,
    average
):

    try:

        return float(
            average_precision_score(
                y_true,
                y_probability,
                average=average
            )
        )

    except Exception:

        return None


def safe_micro_roc_auc(
    y_true,
    y_probability
):

    try:

        return float(
            roc_auc_score(
                y_true.ravel(),
                y_probability.ravel()
            )
        )

    except Exception:

        return None


def supported_macro_roc_auc(
    y_true,
    y_probability
):

    auc_values = []


    for class_index in range(
        y_true.shape[
            1
        ]
    ):

        labels = y_true[
            :,
            class_index
        ]


        if len(
            np.unique(
                labels
            )
        ) < 2:

            continue


        try:

            auc = roc_auc_score(
                labels,
                y_probability[
                    :,
                    class_index
                ]
            )


            auc_values.append(
                float(
                    auc
                )
            )

        except Exception:

            pass


    if not auc_values:

        return None


    return float(
        np.mean(
            auc_values
        )
    )


# ==============================================================================
# 41. SUPPORTED-CLASS MACRO METRICS
# ==============================================================================

def supported_class_indices(
    y_true
):

    return np.where(
        y_true.sum(
            axis=0
        )
        >
        0
    )[
        0
    ]


def supported_macro_f1(
    y_true,
    y_pred
):

    indices = supported_class_indices(
        y_true
    )


    if len(
        indices
    ) == 0:

        return None


    return float(
        f1_score(
            y_true[
                :,
                indices
            ],
            y_pred[
                :,
                indices
            ],
            average="macro",
            zero_division=0
        )
    )


# ==============================================================================
# 42. FINAL MULTI-LABEL METRIC FUNCTION
# ==============================================================================

def multilabel_metrics(
    split_name,
    y_true,
    y_probability,
    y_prediction,
    threshold
):

    micro_f1 = f1_score(
        y_true,
        y_prediction,
        average="micro",
        zero_division=0
    )


    macro_f1 = f1_score(
        y_true,
        y_prediction,
        average="macro",
        zero_division=0
    )


    supported_f1 = supported_macro_f1(
        y_true,
        y_prediction
    )


    micro_precision = precision_score(
        y_true,
        y_prediction,
        average="micro",
        zero_division=0
    )


    micro_recall = recall_score(
        y_true,
        y_prediction,
        average="micro",
        zero_division=0
    )


    macro_precision = precision_score(
        y_true,
        y_prediction,
        average="macro",
        zero_division=0
    )


    macro_recall = recall_score(
        y_true,
        y_prediction,
        average="macro",
        zero_division=0
    )


    micro_ap = safe_average_precision(
        y_true,
        y_probability,
        "micro"
    )


    macro_ap = safe_average_precision(
        y_true,
        y_probability,
        "macro"
    )


    micro_auc = safe_micro_roc_auc(
        y_true,
        y_probability
    )


    macro_auc = supported_macro_roc_auc(
        y_true,
        y_probability
    )


    supported_classes = len(
        supported_class_indices(
            y_true
        )
    )


    actual_positive_labels = int(
        y_true.sum()
    )


    predicted_positive_labels = int(
        y_prediction.sum()
    )


    mean_labels_predicted = float(
        y_prediction.sum(
            axis=1
        ).mean()
    )


    return {
        "split":
            split_name,

        "molecules":
            int(
                y_true.shape[
                    0
                ]
            ),

        "target_classes":
            int(
                y_true.shape[
                    1
                ]
            ),

        "classes_with_positive_examples":
            supported_classes,

        "decision_threshold":
            float(
                threshold
            ),

        "actual_positive_labels":
            actual_positive_labels,

        "predicted_positive_labels":
            predicted_positive_labels,

        "mean_predicted_labels_per_molecule":
            mean_labels_predicted,

        "micro_f1":
            float(
                micro_f1
            ),

        "macro_f1":
            float(
                macro_f1
            ),

        "supported_macro_f1":
            supported_f1,

        "micro_precision":
            float(
                micro_precision
            ),

        "micro_recall":
            float(
                micro_recall
            ),

        "macro_precision":
            float(
                macro_precision
            ),

        "macro_recall":
            float(
                macro_recall
            ),

        "micro_average_precision":
            micro_ap,

        "macro_average_precision":
            macro_ap,

        "micro_roc_auc":
            micro_auc,

        "supported_macro_roc_auc":
            macro_auc,

        "hamming_loss":
            float(
                hamming_loss(
                    y_true,
                    y_prediction
                )
            )
    }


# ==============================================================================
# 43. FINAL VALIDATION / TEST METRICS
# ==============================================================================

validation_metric_record = multilabel_metrics(
    "VALIDATION",
    Y_validation,
    validation_probabilities,
    validation_predictions,
    OPTIMAL_THRESHOLD
)


test_metric_record = multilabel_metrics(
    "TEST",
    Y_test,
    test_probabilities,
    test_predictions,
    OPTIMAL_THRESHOLD
)


predictor_metrics_df = pd.DataFrame(
    [
        validation_metric_record,
        test_metric_record
    ]
)


predictor_metrics_df.to_csv(
    PHASE12_METRICS_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 44. SAVE Z0
# ==============================================================================

Z0.to_csv(
    PHASE12_Z0_FILE,
    index=False,
    encoding="utf-8"
)


print(
    "\n[PASS] Baseline latent reference Z0 saved."
)


# ==============================================================================
# 45. CREATE BASELINE θE* / θD* SNAPSHOTS
# ==============================================================================

shutil.copy2(
    PHASE10_ENCODER_FILE,
    PHASE12_ENCODER_SNAPSHOT_FILE
)


shutil.copy2(
    PHASE11_DECODER_FILE,
    PHASE12_DECODER_SNAPSHOT_FILE
)


print(
    "[PASS] Baseline encoder/decoder reference snapshots created."
)


# ==============================================================================
# 46. VERIFY θE* / θD* UNCHANGED
# ==============================================================================

encoder_hash_after = sha256_file(
    PHASE10_ENCODER_FILE
)


decoder_hash_after = sha256_file(
    PHASE11_DECODER_FILE
)


encoder_unchanged = (
    encoder_hash_before
    ==
    encoder_hash_after
)


decoder_unchanged = (
    decoder_hash_before
    ==
    decoder_hash_after
)


if not encoder_unchanged:

    raise RuntimeError(
        "θE* changed unexpectedly."
    )


if not decoder_unchanged:

    raise RuntimeError(
        "θD* changed unexpectedly."
    )


print(
    "[PASS] θE* remained unchanged."
)

print(
    "[PASS] θD* remained unchanged."
)


# ==============================================================================
# 47. MODEL HASHES
# ==============================================================================

encoder_snapshot_hash = sha256_file(
    PHASE12_ENCODER_SNAPSHOT_FILE
)


decoder_snapshot_hash = sha256_file(
    PHASE12_DECODER_SNAPSHOT_FILE
)


predictor_hash = sha256_file(
    PHASE12_PREDICTOR_FILE
)


z0_hash = sha256_file(
    PHASE12_Z0_FILE
)


hash_manifest = {
    "hash_algorithm":
        "SHA-256",

    "theta_E_star":
        {
            "file":
                str(
                    PHASE12_ENCODER_SNAPSHOT_FILE
                ),

            "sha256":
                encoder_snapshot_hash
        },

    "theta_D_star":
        {
            "file":
                str(
                    PHASE12_DECODER_SNAPSHOT_FILE
                ),

            "sha256":
                decoder_snapshot_hash
        },

    "theta_P_star":
        {
            "file":
                str(
                    PHASE12_PREDICTOR_FILE
                ),

            "sha256":
                predictor_hash
        },

    "Z0":
        {
            "file":
                str(
                    PHASE12_Z0_FILE
                ),

            "sha256":
                z0_hash
        }
}


with open(
    PHASE12_MODEL_HASHES_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        hash_manifest,
        file,
        indent=4
    )


print(
    "[PASS] SHA-256 baseline-reference hashes saved."
)


# ==============================================================================
# 48. ENVIRONMENT METADATA
# ==============================================================================

environment_metadata = {
    "python":
        platform.python_version(),

    "torch":
        torch.__version__,

    "numpy":
        np.__version__,

    "pandas":
        pd.__version__,

    "rdkit":
        rdBase.rdkitVersion,

    "device":
        str(
            DEVICE
        )
}


# ==============================================================================
# 49. BASELINE SYSTEM CONFIG
# ==============================================================================

baseline_system_config = {
    "phase":
        "Phase 12 — Revised Baseline System Consolidation",

    "created_at":
        datetime.now().isoformat(),

    "Theta_star":
        {
            "theta_E_star":
                "Phase-10 baseline encoder",

            "theta_D_star":
                "Phase-11 baseline molecular decoder",

            "theta_P_star":
                "Phase-12 calibrated target-association predictor"
        },

    "Z0":
        "Phase-10 deterministic mu embeddings",

    "latent_dimension":
        LATENT_DIM,

    "predictor_target_classes":
        NUM_TARGET_CLASSES,

    "minimum_target_support":
        MIN_TARGET_SUPPORT,

    "predictor_latent_normalization":
        "training-only",

    "predictor_class_weighting":
        "sqrt imbalance ratio + clipping",

    "predictor_threshold_selection":
        "validation micro-F1 optimization",

    "selected_threshold":
        OPTIMAL_THRESHOLD,

    "encoder_retrained":
        False,

    "decoder_retrained":
        False,

    "fisher_information_calculated":
        False,

    "ewc_applied":
        False,

    "closed_loop_adaptation":
        False,

    "parameter_drift_introduced":
        False,

    "environment":
        environment_metadata
}


with open(
    PHASE12_SYSTEM_CONFIG_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        baseline_system_config,
        file,
        indent=4,
        ensure_ascii=False
    )


# ==============================================================================
# 50. PARAMETER MANIFEST
# ==============================================================================

parameter_manifest = {
    "Theta_star_definition":
        {
            "theta_E_star":
                {
                    "source_phase":
                        10,

                    "status":
                        "FROZEN_BASELINE_REFERENCE",

                    "file":
                        str(
                            PHASE12_ENCODER_SNAPSHOT_FILE
                        ),

                    "sha256":
                        encoder_snapshot_hash
                },

            "theta_D_star":
                {
                    "source_phase":
                        11,

                    "status":
                        "FROZEN_BASELINE_REFERENCE",

                    "file":
                        str(
                            PHASE12_DECODER_SNAPSHOT_FILE
                        ),

                    "sha256":
                        decoder_snapshot_hash
                },

            "theta_P_star":
                {
                    "source_phase":
                        12,

                    "status":
                        "FROZEN_CALIBRATED_BASELINE_REFERENCE",

                    "file":
                        str(
                            PHASE12_PREDICTOR_FILE
                        ),

                    "sha256":
                        predictor_hash,

                    "decision_threshold":
                        OPTIMAL_THRESHOLD
                }
        },

    "Z0":
        {
            "molecule_count":
                len(
                    Z0
                ),

            "latent_dimension":
                LATENT_DIM,

            "file":
                str(
                    PHASE12_Z0_FILE
                ),

            "sha256":
                z0_hash
        },

    "future_reference":
        {
            "parameter_reference":
                "Theta_star",

            "latent_reference":
                "Z0"
        }
}


with open(
    PHASE12_PARAMETER_MANIFEST_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        parameter_manifest,
        file,
        indent=4
    )


# ==============================================================================
# 51. SAVE Θ* BUNDLE
# ==============================================================================

theta_star_bundle = {
    "status":
        "FROZEN_CALIBRATED_BASELINE_REFERENCE_STATE",

    "created_at":
        datetime.now().isoformat(),

    "theta_E_star":
        copy.deepcopy(
            phase10_encoder_checkpoint[
                "state_dict"
            ]
        ),

    "theta_D_star":
        copy.deepcopy(
            phase11_decoder_checkpoint[
                "state_dict"
            ]
        ),

    "theta_P_star":
        copy.deepcopy(
            best_predictor_checkpoint[
                "state_dict"
            ]
        ),

    "predictor_latent_mean":
        latent_mean,

    "predictor_latent_std":
        latent_std_safe,

    "predictor_threshold":
        OPTIMAL_THRESHOLD,

    "predictor_target_ids":
        selected_target_ids,

    "positive_class_weights":
        positive_class_weights,

    "latent_dimension":
        LATENT_DIM,

    "hashes":
        {
            "theta_E_star":
                encoder_snapshot_hash,

            "theta_D_star":
                decoder_snapshot_hash,

            "theta_P_star":
                predictor_hash,

            "Z0":
                z0_hash
        },

    "fisher_information_calculated":
        False,

    "ewc_applied":
        False,

    "closed_loop_adaptation":
        False
}


torch.save(
    theta_star_bundle,
    PHASE12_REFERENCE_BUNDLE_FILE
)


reloaded_bundle = torch.load(
    PHASE12_REFERENCE_BUNDLE_FILE,
    map_location="cpu",
    weights_only=False
)


theta_star_bundle_reloadable = bool(
    reloaded_bundle[
        "status"
    ]
    ==
    "FROZEN_CALIBRATED_BASELINE_REFERENCE_STATE"
)


print(
    "\n[PASS] Complete calibrated Θ* reference bundle saved and reloaded."
)


# ==============================================================================
# 52. SOURCE PRESERVATION CHECK
# ==============================================================================

phase8_splits_preserved = all(
    [
        D_train_manifest.equals(
            D_train_manifest_original
        ),

        D_validation_manifest.equals(
            D_validation_manifest_original
        ),

        D_test_manifest.equals(
            D_test_manifest_original
        )
    ]
)


fisher_reference_preserved = (
    D_fisher_reference.equals(
        D_fisher_original
    )
)


predictor_checkpoint_stable = (
    predictor_hash
    ==
    sha256_file(
        PHASE12_PREDICTOR_FILE
    )
)


# ==============================================================================
# 53. DISPLAY FINAL PERFORMANCE
# ==============================================================================

print("\n" + "=" * 130)
print("CALIBRATED BIOLOGICAL-RELEVANCE PREDICTOR PERFORMANCE")
print("=" * 130)

print(
    predictor_metrics_df.to_string(
        index=False
    )
)


# ==============================================================================
# 54. COMPLETION CHECKS
# ==============================================================================

completion_checks = {
    "Phase-10 encoder checkpoint verified":
        True,

    "Phase-11 decoder checkpoint verified":
        True,

    "Phase-10 encoder parameters finite":
        encoder_state_finite,

    "Phase-11 decoder parameters finite":
        decoder_state_finite,

    "Phase-10 latent vectors verified":
        True,

    "Baseline latent dimension equals 32":
        LATENT_DIM == 32,

    "All Z0 values finite":
        bool(
            np.isfinite(
                Z0_values
            ).all()
        ),

    "Training-only predictor latent normalization":
        True,

    "Predictor label vocabulary derived from training only":
        True,

    "Minimum target support applied":
        MIN_TARGET_SUPPORT > 0,

    "Moderated class weighting derived from training only":
        True,

    "Predictor trained using training molecules only":
        epochs_completed > 0,

    "Validation used for checkpoint selection":
        best_epoch > 0,

    "Test excluded from checkpoint selection":
        True,

    "Best predictor checkpoint reloadable":
        best_predictor_reloadable,

    "Validation-only threshold optimization performed":
        True,

    "Test excluded from threshold optimization":
        True,

    "Optimal threshold frozen before test evaluation":
        True,

    "Predictor output probabilities finite":
        bool(
            np.isfinite(
                validation_probabilities
            ).all()
            and
            np.isfinite(
                test_probabilities
            ).all()
        ),

    "Predictor parameters frozen":
        predictor_frozen,

    "θE* unchanged during Phase 12":
        encoder_unchanged,

    "θD* unchanged during Phase 12":
        decoder_unchanged,

    "θP* established":
        PHASE12_PREDICTOR_FILE.exists(),

    "Z0 saved":
        PHASE12_Z0_FILE.exists(),

    "Phase-8 split assignments unchanged":
        phase8_splits_preserved,

    "Fisher reference unchanged":
        fisher_reference_preserved,

    "Baseline hashes saved":
        PHASE12_MODEL_HASHES_FILE.exists(),

    "Θ* parameter manifest saved":
        PHASE12_PARAMETER_MANIFEST_FILE.exists(),

    "Θ* reference bundle saved":
        PHASE12_REFERENCE_BUNDLE_FILE.exists(),

    "Θ* reference bundle reloadable":
        theta_star_bundle_reloadable,

    "Predictor checkpoint integrity stable":
        predictor_checkpoint_stable,

    "No Fisher information calculated":
        True,

    "No EWC applied":
        True,

    "No closed-loop adaptation performed":
        True,

    "No parameter drift introduced":
        True
}


# ==============================================================================
# 55. SUMMARY
# ==============================================================================

summary = {
    "phase":
        (
            "Phase 12 — Revised Baseline System Consolidation, "
            "Calibrated Predictor Training, and Reference-State Establishment"
        ),

    "created_at":
        datetime.now().isoformat(),

    "status":
        "CALIBRATED_BASELINE_REFERENCE_STATE_ESTABLISHED",

    "Theta_star_components":
        3,

    "theta_E_star":
        "FROZEN",

    "theta_D_star":
        "FROZEN",

    "theta_P_star":
        "FROZEN",

    "Z0_molecules":
        len(
            Z0
        ),

    "latent_dimension":
        LATENT_DIM,

    "training_molecules":
        len(
            D_train_manifest
        ),

    "validation_molecules":
        len(
            D_validation_manifest
        ),

    "test_molecules":
        len(
            D_test_manifest
        ),

    "fisher_reference_molecules":
        len(
            D_fisher_reference
        ),

    "predictor_target_classes":
        NUM_TARGET_CLASSES,

    "minimum_target_support":
        MIN_TARGET_SUPPORT,

    "train_association_coverage":
        train_coverage,

    "validation_association_coverage":
        validation_coverage,

    "test_association_coverage":
        test_coverage,

    "predictor_parameters":
        predictor_parameter_count,

    "epochs_completed":
        epochs_completed,

    "best_epoch":
        best_epoch,

    "best_validation_loss":
        best_validation_loss,

    "selected_validation_threshold":
        OPTIMAL_THRESHOLD,

    "validation_micro_f1":
        validation_metric_record[
            "micro_f1"
        ],

    "validation_micro_precision":
        validation_metric_record[
            "micro_precision"
        ],

    "validation_micro_recall":
        validation_metric_record[
            "micro_recall"
        ],

    "validation_micro_average_precision":
        validation_metric_record[
            "micro_average_precision"
        ],

    "validation_micro_roc_auc":
        validation_metric_record[
            "micro_roc_auc"
        ],

    "validation_actual_positive_labels":
        validation_metric_record[
            "actual_positive_labels"
        ],

    "validation_predicted_positive_labels":
        validation_metric_record[
            "predicted_positive_labels"
        ],

    "test_micro_f1":
        test_metric_record[
            "micro_f1"
        ],

    "test_micro_precision":
        test_metric_record[
            "micro_precision"
        ],

    "test_micro_recall":
        test_metric_record[
            "micro_recall"
        ],

    "test_micro_average_precision":
        test_metric_record[
            "micro_average_precision"
        ],

    "test_micro_roc_auc":
        test_metric_record[
            "micro_roc_auc"
        ],

    "test_actual_positive_labels":
        test_metric_record[
            "actual_positive_labels"
        ],

    "test_predicted_positive_labels":
        test_metric_record[
            "predicted_positive_labels"
        ],

    "encoder_sha256":
        encoder_snapshot_hash,

    "decoder_sha256":
        decoder_snapshot_hash,

    "predictor_sha256":
        predictor_hash,

    "Z0_sha256":
        z0_hash,

    "encoder_retrained":
        False,

    "decoder_retrained":
        False,

    "fisher_information_calculated":
        False,

    "ewc_applied":
        False,

    "closed_loop_adaptation":
        False,

    "parameter_drift_introduced":
        False,

    "training_duration_seconds":
        training_duration_seconds,

    "completion_checks":
        completion_checks
}


with open(
    PHASE12_SUMMARY_JSON_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        summary,
        file,
        indent=4,
        ensure_ascii=False,
        default=str
    )


summary_rows = []


for key, value in summary.items():

    if key == "completion_checks":

        for check_name, status in value.items():

            summary_rows.append(
                {
                    "metric":
                        f"completion.{check_name}",

                    "value":
                        status
                }
            )

    else:

        summary_rows.append(
            {
                "metric":
                    key,

                "value":
                    value
            }
        )


pd.DataFrame(
    summary_rows
).to_csv(
    PHASE12_SUMMARY_CSV_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 56. COMPLETION VALIDATION
# ==============================================================================

print("\n" + "=" * 130)
print("PHASE 12 COMPLETION VALIDATION")
print("=" * 130)


all_completion_checks_passed = True


for check_name, status in completion_checks.items():

    label = (
        "PASS"
        if bool(
            status
        )
        else "FAIL"
    )


    print(
        f"[{label:<4}] {check_name}"
    )


    if not bool(
        status
    ):

        all_completion_checks_passed = False


# ==============================================================================
# 57. UPDATE PROJECT CONFIG
# ==============================================================================

PHASE12_DATASETS = {
    "predictor_config":
        str(
            PHASE12_PREDICTOR_CONFIG_FILE
        ),

    "target_vocabulary_json":
        str(
            PHASE12_TARGET_VOCAB_FILE
        ),

    "target_vocabulary_csv":
        str(
            PHASE12_TARGET_VOCAB_CSV_FILE
        ),

    "predictor_latent_normalization":
        str(
            PHASE12_LATENT_NORMALIZATION_FILE
        ),

    "predictor_class_weights":
        str(
            PHASE12_CLASS_WEIGHTS_FILE
        ),

    "predictor_training_history":
        str(
            PHASE12_HISTORY_FILE
        ),

    "best_baseline_predictor":
        str(
            PHASE12_PREDICTOR_FILE
        ),

    "threshold_search":
        str(
            PHASE12_THRESHOLD_SEARCH_FILE
        ),

    "selected_threshold":
        str(
            PHASE12_THRESHOLD_CONFIG_FILE
        ),

    "predictor_metrics":
        str(
            PHASE12_METRICS_FILE
        ),

    "baseline_latent_reference_Z0":
        str(
            PHASE12_Z0_FILE
        ),

    "theta_E_star":
        str(
            PHASE12_ENCODER_SNAPSHOT_FILE
        ),

    "theta_D_star":
        str(
            PHASE12_DECODER_SNAPSHOT_FILE
        ),

    "theta_P_star":
        str(
            PHASE12_PREDICTOR_FILE
        ),

    "baseline_system_config":
        str(
            PHASE12_SYSTEM_CONFIG_FILE
        ),

    "baseline_parameter_manifest":
        str(
            PHASE12_PARAMETER_MANIFEST_FILE
        ),

    "baseline_model_hashes":
        str(
            PHASE12_MODEL_HASHES_FILE
        ),

    "theta_star_reference_bundle":
        str(
            PHASE12_REFERENCE_BUNDLE_FILE
        ),

    "summary_csv":
        str(
            PHASE12_SUMMARY_CSV_FILE
        ),

    "summary_json":
        str(
            PHASE12_SUMMARY_JSON_FILE
        ),

    "predictor_target_classes":
        NUM_TARGET_CLASSES,

    "best_epoch":
        best_epoch,

    "selected_threshold_value":
        OPTIMAL_THRESHOLD,

    "baseline_state":
        "FROZEN_CALIBRATED_THETA_STAR"
}


DRUGBANK_CONFIG[
    "phase12_datasets"
] = PHASE12_DATASETS


# ==============================================================================
# 58. FINAL SUMMARY
# ==============================================================================

print("\n" + "=" * 130)
print("PHASE 12 SUMMARY")
print("=" * 130)


print(
    f"Training Molecules                    : "
    f"{len(D_train_manifest)}"
)

print(
    f"Validation Molecules                  : "
    f"{len(D_validation_manifest)}"
)

print(
    f"Test Molecules                        : "
    f"{len(D_test_manifest)}"
)

print(
    f"Fisher Reference Molecules            : "
    f"{len(D_fisher_reference)}"
)

print(
    f"Baseline Latent Molecules Z0          : "
    f"{len(Z0)}"
)

print(
    f"Latent Dimension                      : "
    f"{LATENT_DIM}"
)

print(
    f"Predictor Target Classes              : "
    f"{NUM_TARGET_CLASSES}"
)

print(
    f"Minimum Target Support                : "
    f"{MIN_TARGET_SUPPORT}"
)

print(
    f"Predictor Parameters                  : "
    f"{predictor_parameter_count:,}"
)

print(
    f"Predictor Epochs Completed            : "
    f"{epochs_completed}"
)

print(
    f"Best Predictor Epoch                  : "
    f"{best_epoch}"
)

print(
    f"Best Validation Loss                  : "
    f"{best_validation_loss:.8f}"
)

print(
    f"Validation-Selected Threshold         : "
    f"{OPTIMAL_THRESHOLD:.4f}"
)


print("\nValidation Metrics:")

print(
    f"  Actual Positive Labels              : "
    f"{validation_metric_record['actual_positive_labels']}"
)

print(
    f"  Predicted Positive Labels           : "
    f"{validation_metric_record['predicted_positive_labels']}"
)

print(
    f"  Micro F1                            : "
    f"{validation_metric_record['micro_f1']:.4f}"
)

print(
    f"  Micro Precision                     : "
    f"{validation_metric_record['micro_precision']:.4f}"
)

print(
    f"  Micro Recall                        : "
    f"{validation_metric_record['micro_recall']:.4f}"
)

print(
    f"  Macro F1                            : "
    f"{validation_metric_record['macro_f1']:.4f}"
)


if validation_metric_record[
    "supported_macro_f1"
] is not None:

    print(
        f"  Supported-Class Macro F1            : "
        f"{validation_metric_record['supported_macro_f1']:.4f}"
    )


if validation_metric_record[
    "micro_average_precision"
] is not None:

    print(
        f"  Micro Average Precision             : "
        f"{validation_metric_record['micro_average_precision']:.4f}"
    )


if validation_metric_record[
    "micro_roc_auc"
] is not None:

    print(
        f"  Micro ROC-AUC                       : "
        f"{validation_metric_record['micro_roc_auc']:.4f}"
    )


print("\nTest Metrics:")

print(
    f"  Actual Positive Labels              : "
    f"{test_metric_record['actual_positive_labels']}"
)

print(
    f"  Predicted Positive Labels           : "
    f"{test_metric_record['predicted_positive_labels']}"
)

print(
    f"  Micro F1                            : "
    f"{test_metric_record['micro_f1']:.4f}"
)

print(
    f"  Micro Precision                     : "
    f"{test_metric_record['micro_precision']:.4f}"
)

print(
    f"  Micro Recall                        : "
    f"{test_metric_record['micro_recall']:.4f}"
)

print(
    f"  Macro F1                            : "
    f"{test_metric_record['macro_f1']:.4f}"
)


if test_metric_record[
    "supported_macro_f1"
] is not None:

    print(
        f"  Supported-Class Macro F1            : "
        f"{test_metric_record['supported_macro_f1']:.4f}"
    )


if test_metric_record[
    "micro_average_precision"
] is not None:

    print(
        f"  Micro Average Precision             : "
        f"{test_metric_record['micro_average_precision']:.4f}"
    )


if test_metric_record[
    "micro_roc_auc"
] is not None:

    print(
        f"  Micro ROC-AUC                       : "
        f"{test_metric_record['micro_roc_auc']:.4f}"
    )


print("\nFrozen Baseline Reference State:")

print(
    "  θE* Encoder                         : FROZEN"
)

print(
    "  θD* Molecular Decoder               : FROZEN"
)

print(
    "  θP* Target-Association Predictor     : FROZEN"
)

print(
    "  Predictor Threshold                 : FROZEN"
)

print(
    "  Predictor Latent Normalization      : FROZEN"
)

print(
    "  Target Vocabulary                   : FROZEN"
)

print(
    "  Z0 Baseline Latent Reference        : SAVED"
)

print(
    "  Θ* Complete Baseline State          : ESTABLISHED"
)


print("\nIntegrity:")

print(
    f"  Encoder Unchanged                    : "
    f"{encoder_unchanged}"
)

print(
    f"  Decoder Unchanged                    : "
    f"{decoder_unchanged}"
)

print(
    f"  Predictor Checkpoint Stable          : "
    f"{predictor_checkpoint_stable}"
)

print(
    f"  Θ* Bundle Reloadable                 : "
    f"{theta_star_bundle_reloadable}"
)

print(
    f"  Phase-8 Splits Preserved             : "
    f"{phase8_splits_preserved}"
)

print(
    f"  Fisher Reference Preserved           : "
    f"{fisher_reference_preserved}"
)


print("\nAdaptive-Learning Status:")

print(
    "  Fisher Information Computed          : False"
)

print(
    "  EWC Applied                          : False"
)

print(
    "  Closed-Loop Adaptation               : False"
)

print(
    "  Parameter Drift Introduced           : False"
)


print("\nSaved Files:")

print(
    f"  Predictor Config                      : "
    f"{PHASE12_PREDICTOR_CONFIG_FILE}"
)

print(
    f"  Target Vocabulary JSON                : "
    f"{PHASE12_TARGET_VOCAB_FILE}"
)

print(
    f"  Target Vocabulary CSV                 : "
    f"{PHASE12_TARGET_VOCAB_CSV_FILE}"
)

print(
    f"  Latent Normalization                  : "
    f"{PHASE12_LATENT_NORMALIZATION_FILE}"
)

print(
    f"  Class Weights                         : "
    f"{PHASE12_CLASS_WEIGHTS_FILE}"
)

print(
    f"  Training History                      : "
    f"{PHASE12_HISTORY_FILE}"
)

print(
    f"  Best Predictor θP*                    : "
    f"{PHASE12_PREDICTOR_FILE}"
)

print(
    f"  Validation Threshold Search           : "
    f"{PHASE12_THRESHOLD_SEARCH_FILE}"
)

print(
    f"  Selected Threshold                    : "
    f"{PHASE12_THRESHOLD_CONFIG_FILE}"
)

print(
    f"  Predictor Metrics                     : "
    f"{PHASE12_METRICS_FILE}"
)

print(
    f"  Baseline Latent Reference Z0          : "
    f"{PHASE12_Z0_FILE}"
)

print(
    f"  Baseline Encoder θE*                  : "
    f"{PHASE12_ENCODER_SNAPSHOT_FILE}"
)

print(
    f"  Baseline Decoder θD*                  : "
    f"{PHASE12_DECODER_SNAPSHOT_FILE}"
)

print(
    f"  Baseline System Config                : "
    f"{PHASE12_SYSTEM_CONFIG_FILE}"
)

print(
    f"  Parameter Manifest                    : "
    f"{PHASE12_PARAMETER_MANIFEST_FILE}"
)

print(
    f"  Model Hashes                          : "
    f"{PHASE12_MODEL_HASHES_FILE}"
)

print(
    f"  Θ* Reference Bundle                   : "
    f"{PHASE12_REFERENCE_BUNDLE_FILE}"
)

print(
    f"  Summary CSV                           : "
    f"{PHASE12_SUMMARY_CSV_FILE}"
)

print(
    f"  Summary JSON                          : "
    f"{PHASE12_SUMMARY_JSON_FILE}"
)


print("-" * 130)


if all_completion_checks_passed:

    print(
        "[STATUS] PHASE 12 COMPLETED SUCCESSFULLY."
    )

    print(
        "[READY] Calibrated baseline reference state Θ* "
        "has been established."
    )

    print(
        "[READY] θE*, θD*, and θP* are frozen baseline references."
    )

    print(
        "[READY] Predictor operating threshold was selected using "
        "validation data only."
    )

    print(
        "[READY] Test data was evaluated only after both model "
        "selection and threshold selection were complete."
    )

    print(
        "[READY] Z0 is stored as the fixed baseline latent reference."
    )

    print(
        "[IMPORTANT] Predictor scores represent DrugBank-grounded "
        "target-association relevance."
    )

    print(
        "[IMPORTANT] They do NOT represent quantitative binding affinity."
    )

    print(
        "[IMPORTANT] No Fisher information, EWC, continual adaptation, "
        "or parameter drift has occurred."
    )

    print(
        "[NEXT] Phase 13 — Fisher Information Estimation and "
        "Continual-Learning Reference Construction"
    )


else:

    print(
        "[STATUS] PHASE 12 REQUIRES REVIEW BEFORE FINAL FREEZE."
    )

    print(
        "[ACTION] Do not proceed to Fisher/EWC until all failed "
        "completion checks are resolved."
    )


print("=" * 130)

PHASE 12 — REVISED BASELINE SYSTEM CONSOLIDATION, CALIBRATED PREDICTOR TRAINING, AND REFERENCE-STATE ESTABLISHMENT

[DEVICE] Device                    : cpu
[CONFIG] Random Seed              : 42
[CONFIG] Latent Dimension         : 32
[CONFIG] Predictor Hidden 1       : 64
[CONFIG] Predictor Hidden 2       : 32
[CONFIG] Predictor Dropout        : 0.15
[CONFIG] Minimum Target Support   : 5
[CONFIG] Learning Rate            : 0.001
[CONFIG] Weight Decay             : 0.0001
[CONFIG] Maximum Epochs           : 100
[CONFIG] Early Stop Patience      : 12
[CONFIG] Class Weight Policy      : sqrt(negative/positive), clipped
[CONFIG] Positive Weight Range    : 1.0 – 10.0
[CONFIG] Decision Threshold       : VALIDATION-OPTIMIZED AFTER MODEL SELECTION

[PASS] Required Phase-6, Phase-8, Phase-10 and Phase-11 files found.

[PASS] Phase-10 baseline encoder θE* verified.
[PASS] Phase-11 baseline decoder θD* verified.
[PASS] Phase-11 latent-conditioning evidence verified.

PHASE-8 BASELINE CORPUS
Trai

# Phase 13 — Baseline Knowledge Consolidation & Empirical Fisher Estimation

**Objective:** Compute the diagonal empirical Fisher information matrix ($F = \{F_E, F_P\}$) across the frozen baseline encoder ($\theta_E^*$) and target predictor ($\theta_P^*$) using the fixed 128-molecule reference set to anchor baseline knowledge for Elastic Weight Consolidation (EWC)[cite: 1, 3].

* **Core Principles & Mathematical Setup:**
  * **Empirical Fisher Objective:** Measures sample-wise squared gradients ($B = 1$) under the DrugBank target-association BCE loss[cite: 1, 3]:
    $$F_j = \frac{1}{N} \sum_{i=1}^N \left(\frac{\partial \mathcal{L}_i}{\partial \theta_j}\right)^2$$
  * **Computational Pathway:** Reconnects raw Phase-7 graphs through the actual encoder ($\theta_E^* \to \mu \to \text{Norm} \to \theta_P^* \to \text{Loss}$) using deterministic $\mu$[cite: 1, 3].
  * **Decoupled Decoder:** $\theta_D^*$ remains frozen without Fisher computation as it does not participate in the target-association task[cite: 1, 3].
  * **Zero Parameter Drift:** Checksum verification (SHA-256) and tensor equality confirm no optimizer steps or weight modifications occur ($\Theta_{\text{after}} = \Theta^*$)[cite: 1, 3].

* **Key Deliverables:**
  * **Fisher Tensors:** `phase13_encoder_fisher.pt` and `phase13_predictor_fisher.pt`[cite: 1, 3].
  * **EWC Reference State:** `phase13_fisher_parameter_reference.pt` and `phase13_consolidated_knowledge_manifest.json` ($\mathcal{K}_0 = \{\Theta^*, F_E, F_P, Z_0\}$)[cite: 1, 3].
  * **Audits & Metrics:** `phase13_fisher_parameter_statistics.csv`, `phase13_fisher_layer_statistics.csv`, and `phase13_qc_summary.json`[cite: 1, 3].

In [13]:
# ==============================================================================
# PHASE 13 — BASELINE KNOWLEDGE CONSOLIDATION
# DIAGONAL EMPIRICAL FISHER INFORMATION REFERENCE CONSTRUCTION
# ==============================================================================
#
# FINAL PHASE-13 OBJECTIVE
# ------------------------
#
# Phase 12 established:
#
# Θ* = { θE*, θD*, θP* }
#
# and:
#
# Z0 = fixed deterministic baseline latent representation
#
#
# Phase 13 estimates:
#
# F_E = encoder diagonal empirical Fisher
# F_P = predictor diagonal empirical Fisher
#
#
# using:
#
# F_j
# =
# (1/N) * Σ_i [ ∂L_i / ∂θ_j ]²
#
#
# IMPORTANT
# ---------
#
# Fisher values represent PARAMETER IMPORTANCE UNDER THE DEFINED
# DRUGBANK TARGET-ASSOCIATION TASK.
#
# They do NOT directly represent:
#
# - atom importance
# - chemical importance
# - binding affinity
# - biological causality
# - clinical relevance
# - therapeutic efficacy
#
#
# NO:
#
# - optimizer
# - optimizer.step()
# - EWC
# - closed-loop adaptation
# - parameter update
#
#
# RECOMMENDED CONSOLIDATED STATE
#
# K0 = {
#     Θ*,
#     F_E,
#     F_P,
#     Z0
# }
#
#
# DECODER POLICY
# --------------
#
# θD* is preserved but F_D is NOT calculated in this phase because
# the decoder is not part of the DrugBank target-association Fisher objective.
#
# ==============================================================================


# ==============================================================================
# 1. IMPORTS
# ==============================================================================

from pathlib import Path
from datetime import datetime

import copy
import hashlib
import inspect
import json
import random
import time

import numpy as np
import pandas as pd

import torch
import torch.nn as nn


# ==============================================================================
# 2. REPRODUCIBILITY
# ==============================================================================

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cpu")


# ==============================================================================
# 3. VERIFY REQUIRED PROJECT VARIABLES
# ==============================================================================

required_variables = [
    "PROCESSED_DIR",
    "DRUGBANK_CONFIG"
]


missing_variables = [
    variable
    for variable in required_variables
    if variable not in globals()
]


if missing_variables:

    raise RuntimeError(
        "Required earlier-phase project variables are missing.\n"
        f"Missing: {missing_variables}"
    )


PROCESSED_DIR = Path(PROCESSED_DIR)


# ==============================================================================
# 4. CONFIGURATION
# ==============================================================================

LATENT_DIM = 32

EXPECTED_FISHER_REFERENCE_SIZE = 128

FISHER_BATCH_SIZE = 1

EPSILON = 1e-12


print("=" * 132)

print(
    "PHASE 13 — BASELINE KNOWLEDGE CONSOLIDATION AND "
    "DIAGONAL EMPIRICAL FISHER INFORMATION"
)

print("=" * 132)

print(f"\n[DEVICE] Device                     : {DEVICE}")
print(f"[CONFIG] Random Seed               : {RANDOM_SEED}")
print(f"[CONFIG] Latent Dimension          : {LATENT_DIM}")
print(
    f"[CONFIG] Fisher Reference Size     : "
    f"{EXPECTED_FISHER_REFERENCE_SIZE}"
)
print(
    f"[CONFIG] Fisher Batch Size         : "
    f"{FISHER_BATCH_SIZE}"
)
print(
    "[CONFIG] Fisher Approximation      : "
    "Diagonal empirical Fisher"
)
print(
    "[CONFIG] Objective                 : "
    "DrugBank target-association BCE"
)
print(
    "[CONFIG] Fisher Components         : "
    "Encoder + Predictor"
)
print(
    "[CONFIG] Decoder Fisher            : "
    "Not calculated"
)


# ==============================================================================
# 5. INPUT FILES
# ==============================================================================

PHASE8_TRAIN_FILE = (
    PROCESSED_DIR
    / "phase8_train_graphs.csv"
)

PHASE8_VALIDATION_FILE = (
    PROCESSED_DIR
    / "phase8_validation_graphs.csv"
)

PHASE8_TEST_FILE = (
    PROCESSED_DIR
    / "phase8_test_graphs.csv"
)

PHASE8_FISHER_FILE = (
    PROCESSED_DIR
    / "phase8_fisher_reference.csv"
)


PHASE10_ENCODER_FILE = (
    PROCESSED_DIR
    / "phase10_best_encoder.pt"
)


PHASE11_DECODER_FILE = (
    PROCESSED_DIR
    / "phase11_best_generative_decoder.pt"
)


PHASE12_PREDICTOR_FILE = (
    PROCESSED_DIR
    / "phase12_best_baseline_predictor.pt"
)

PHASE12_Z0_FILE = (
    PROCESSED_DIR
    / "phase12_baseline_latent_reference.csv"
)

PHASE12_TARGET_VOCAB_FILE = (
    PROCESSED_DIR
    / "phase12_predictor_target_vocabulary.json"
)

PHASE12_LATENT_NORMALIZATION_FILE = (
    PROCESSED_DIR
    / "phase12_predictor_latent_normalization.json"
)

PHASE12_CLASS_WEIGHTS_FILE = (
    PROCESSED_DIR
    / "phase12_predictor_class_weights.csv"
)

PHASE12_MODEL_HASHES_FILE = (
    PROCESSED_DIR
    / "phase12_baseline_model_hashes.json"
)

PHASE12_REFERENCE_BUNDLE_FILE = (
    PROCESSED_DIR
    / "phase12_theta_star_reference_bundle.pt"
)


PHASE9_FEATURE_NORMALIZATION_FILE = (
    PROCESSED_DIR
    / "phase9_feature_normalization.json"
)


D_BIO_ASSOCIATION_FILE = (
    PROCESSED_DIR
    / "D_bio_association.csv"
)


required_files = [
    PHASE8_TRAIN_FILE,
    PHASE8_VALIDATION_FILE,
    PHASE8_TEST_FILE,
    PHASE8_FISHER_FILE,
    PHASE10_ENCODER_FILE,
    PHASE11_DECODER_FILE,
    PHASE12_PREDICTOR_FILE,
    PHASE12_Z0_FILE,
    PHASE12_TARGET_VOCAB_FILE,
    PHASE12_LATENT_NORMALIZATION_FILE,
    PHASE12_CLASS_WEIGHTS_FILE,
    PHASE12_MODEL_HASHES_FILE,
    PHASE12_REFERENCE_BUNDLE_FILE,
    D_BIO_ASSOCIATION_FILE
]


missing_files = [
    str(file_path)
    for file_path in required_files
    if not file_path.exists()
]


if missing_files:

    raise FileNotFoundError(
        "Required earlier-phase files are missing:\n"
        +
        "\n".join(missing_files)
    )


print(
    "\n[PASS] Required Phase-8, Phase-10, "
    "Phase-11 and Phase-12 files found."
)


# ==============================================================================
# 6. OUTPUT FILES
# ==============================================================================

PHASE13_CONFIG_FILE = (
    PROCESSED_DIR
    / "phase13_fisher_config.json"
)

PHASE13_REFERENCE_MANIFEST_FILE = (
    PROCESSED_DIR
    / "phase13_fisher_reference_manifest.csv"
)

PHASE13_ENCODER_FISHER_FILE = (
    PROCESSED_DIR
    / "phase13_encoder_fisher.pt"
)

PHASE13_PREDICTOR_FISHER_FILE = (
    PROCESSED_DIR
    / "phase13_predictor_fisher.pt"
)

PHASE13_PARAMETER_REFERENCE_FILE = (
    PROCESSED_DIR
    / "phase13_fisher_parameter_reference.pt"
)

PHASE13_PARAMETER_STATS_FILE = (
    PROCESSED_DIR
    / "phase13_fisher_parameter_statistics.csv"
)

PHASE13_LAYER_STATS_FILE = (
    PROCESSED_DIR
    / "phase13_fisher_layer_statistics.csv"
)

PHASE13_TOP_PARAMETERS_FILE = (
    PROCESSED_DIR
    / "phase13_fisher_top_parameters.csv"
)

PHASE13_KNOWLEDGE_MANIFEST_FILE = (
    PROCESSED_DIR
    / "phase13_consolidated_knowledge_manifest.json"
)

PHASE13_QC_CSV_FILE = (
    PROCESSED_DIR
    / "phase13_qc_summary.csv"
)

PHASE13_QC_JSON_FILE = (
    PROCESSED_DIR
    / "phase13_qc_summary.json"
)


# ==============================================================================
# 7. SHA-256 HASH HELPER
# ==============================================================================

def sha256_file(file_path, chunk_size=1024 * 1024):

    digest = hashlib.sha256()

    with open(file_path, "rb") as file:

        while True:

            chunk = file.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


# ==============================================================================
# 8. LOAD PHASE-12 REFERENCE BUNDLE
# ==============================================================================

phase12_bundle = torch.load(
    PHASE12_REFERENCE_BUNDLE_FILE,
    map_location="cpu",
    weights_only=False
)


required_bundle_keys = [
    "theta_E_star",
    "theta_D_star",
    "theta_P_star"
]


for key in required_bundle_keys:

    if key not in phase12_bundle:

        raise RuntimeError(
            f"Phase-12 Θ* bundle missing: {key}"
        )


print(
    "[PASS] Phase-12 Θ* reference bundle loaded."
)


# ==============================================================================
# 9. LOAD BASELINE CHECKPOINTS
# ==============================================================================

encoder_checkpoint = torch.load(
    PHASE10_ENCODER_FILE,
    map_location="cpu",
    weights_only=False
)

decoder_checkpoint = torch.load(
    PHASE11_DECODER_FILE,
    map_location="cpu",
    weights_only=False
)

predictor_checkpoint = torch.load(
    PHASE12_PREDICTOR_FILE,
    map_location="cpu",
    weights_only=False
)


for checkpoint_name, checkpoint in [
    ("ENCODER", encoder_checkpoint),
    ("DECODER", decoder_checkpoint),
    ("PREDICTOR", predictor_checkpoint)
]:

    if "state_dict" not in checkpoint:

        raise RuntimeError(
            f"{checkpoint_name} checkpoint missing state_dict."
        )


print(
    "[PASS] θE*, θD*, θP* checkpoints loaded."
)


# ==============================================================================
# 10. STATE-DICT EQUALITY HELPER
# ==============================================================================

def state_dicts_equal(state_a, state_b):

    if set(state_a.keys()) != set(state_b.keys()):

        return False


    for key in state_a:

        tensor_a = state_a[key].cpu()
        tensor_b = state_b[key].cpu()


        if tensor_a.shape != tensor_b.shape:

            return False


        if not torch.equal(
            tensor_a,
            tensor_b
        ):

            return False


    return True


# ==============================================================================
# 11. VERIFY CURRENT CHECKPOINTS AGAINST Θ*
# ==============================================================================

encoder_matches_theta_star = state_dicts_equal(
    encoder_checkpoint["state_dict"],
    phase12_bundle["theta_E_star"]
)


decoder_matches_theta_star = state_dicts_equal(
    decoder_checkpoint["state_dict"],
    phase12_bundle["theta_D_star"]
)


predictor_matches_theta_star = state_dicts_equal(
    predictor_checkpoint["state_dict"],
    phase12_bundle["theta_P_star"]
)


if not encoder_matches_theta_star:

    raise RuntimeError(
        "Current Phase-10 encoder does not match Phase-12 θE*."
    )


if not decoder_matches_theta_star:

    raise RuntimeError(
        "Current Phase-11 decoder does not match Phase-12 θD*."
    )


if not predictor_matches_theta_star:

    raise RuntimeError(
        "Current Phase-12 predictor does not match θP*."
    )


print(
    "[PASS] θE* matches Phase-12 baseline reference."
)

print(
    "[PASS] θD* matches Phase-12 baseline reference."
)

print(
    "[PASS] θP* matches Phase-12 baseline reference."
)


# ==============================================================================
# 12. RECORD CHECKPOINT HASHES BEFORE FISHER
# ==============================================================================

encoder_hash_before = sha256_file(
    PHASE10_ENCODER_FILE
)

decoder_hash_before = sha256_file(
    PHASE11_DECODER_FILE
)

predictor_hash_before = sha256_file(
    PHASE12_PREDICTOR_FILE
)


print("\n" + "=" * 132)
print("PRE-FISHER BASELINE CHECKPOINT HASHES")
print("=" * 132)

print(
    f"Encoder SHA-256   : {encoder_hash_before}"
)

print(
    f"Decoder SHA-256   : {decoder_hash_before}"
)

print(
    f"Predictor SHA-256 : {predictor_hash_before}"
)


# ==============================================================================
# 13. LOAD PHASE-8 SPLITS
# ==============================================================================

D_train = pd.read_csv(
    PHASE8_TRAIN_FILE,
    dtype=str,
    keep_default_na=False
)

D_validation = pd.read_csv(
    PHASE8_VALIDATION_FILE,
    dtype=str,
    keep_default_na=False
)

D_test = pd.read_csv(
    PHASE8_TEST_FILE,
    dtype=str,
    keep_default_na=False
)

D_fisher = pd.read_csv(
    PHASE8_FISHER_FILE,
    dtype=str,
    keep_default_na=False
)


train_graph_ids = set(
    D_train["graph_id"].astype(str)
)

validation_graph_ids = set(
    D_validation["graph_id"].astype(str)
)

test_graph_ids = set(
    D_test["graph_id"].astype(str)
)

fisher_graph_ids = set(
    D_fisher["graph_id"].astype(str)
)


if len(D_fisher) != EXPECTED_FISHER_REFERENCE_SIZE:

    raise RuntimeError(
        f"Expected {EXPECTED_FISHER_REFERENCE_SIZE} "
        f"Fisher molecules but found {len(D_fisher)}."
    )


if not fisher_graph_ids.issubset(
    train_graph_ids
):

    raise RuntimeError(
        "Fisher reference is not a subset of training data."
    )


if fisher_graph_ids & validation_graph_ids:

    raise RuntimeError(
        "Validation leakage detected in Fisher reference."
    )


if fisher_graph_ids & test_graph_ids:

    raise RuntimeError(
        "Test leakage detected in Fisher reference."
    )


print("\n" + "=" * 132)
print("FIXED FISHER REFERENCE CORPUS")
print("=" * 132)

print(
    f"Fisher reference molecules : {len(D_fisher)}"
)

print(
    "[PASS] Fisher reference is training-only."
)

print(
    "[PASS] Validation set excluded."
)

print(
    "[PASS] Test set excluded."
)


# ==============================================================================
# 14. LOAD FROZEN TARGET VOCABULARY
# ==============================================================================

with open(
    PHASE12_TARGET_VOCAB_FILE,
    "r",
    encoding="utf-8"
) as file:

    target_vocabulary_metadata = json.load(file)


target_records = sorted(
    target_vocabulary_metadata["targets"],
    key=lambda record:
        int(
            record["target_index"]
        )
)


selected_target_ids = [
    str(record["target_id"])
    for record in target_records
]


NUM_TARGET_CLASSES = len(
    selected_target_ids
)


target_to_index = {
    target_id:
        index
    for index, target_id
    in enumerate(selected_target_ids)
}


print(
    f"\n[PASS] Frozen target vocabulary loaded: "
    f"{NUM_TARGET_CLASSES} classes."
)


# ==============================================================================
# 15. LOAD FROZEN PHASE-12 LATENT NORMALIZATION
# ==============================================================================

with open(
    PHASE12_LATENT_NORMALIZATION_FILE,
    "r",
    encoding="utf-8"
) as file:

    latent_normalization_metadata = json.load(file)


latent_mean = torch.tensor(
    latent_normalization_metadata["mean"],
    dtype=torch.float32,
    device=DEVICE
)


latent_std_safe = torch.tensor(
    latent_normalization_metadata["safe_std"],
    dtype=torch.float32,
    device=DEVICE
)


if latent_mean.numel() != LATENT_DIM:

    raise RuntimeError(
        "Latent normalization mean dimension mismatch."
    )


if latent_std_safe.numel() != LATENT_DIM:

    raise RuntimeError(
        "Latent normalization std dimension mismatch."
    )


print(
    "[PASS] Frozen Phase-12 latent normalization loaded."
)


# ==============================================================================
# 16. LOAD FROZEN PHASE-12 CLASS WEIGHTS
# ==============================================================================

class_weights_df = pd.read_csv(
    PHASE12_CLASS_WEIGHTS_FILE
)


if "applied_pos_weight" not in class_weights_df.columns:

    raise RuntimeError(
        "Phase-12 class-weight file missing applied_pos_weight."
    )


positive_class_weights = (
    class_weights_df[
        "applied_pos_weight"
    ]
    .astype(float)
    .to_numpy(
        dtype=np.float32
    )
)


if len(positive_class_weights) != NUM_TARGET_CLASSES:

    raise RuntimeError(
        "Class-weight count differs from target vocabulary size."
    )


pos_weight_tensor = torch.tensor(
    positive_class_weights,
    dtype=torch.float32,
    device=DEVICE
)


print(
    "[PASS] Frozen Phase-12 class weights loaded."
)


# ==============================================================================
# 17. LOAD BIOLOGICAL ASSOCIATIONS
# ==============================================================================

D_bio = pd.read_csv(
    D_BIO_ASSOCIATION_FILE,
    dtype=str,
    keep_default_na=False
)


for required_column in [
    "drug_id",
    "target_id"
]:

    if required_column not in D_bio.columns:

        raise RuntimeError(
            f"D_bio_association missing {required_column}."
        )


bio_pairs = (
    D_bio[
        D_bio["target_id"]
        .astype(str)
        .str.strip()
        !=
        ""
    ][
        [
            "drug_id",
            "target_id"
        ]
    ]
    .drop_duplicates()
)


bio_pairs = bio_pairs[
    bio_pairs["target_id"].isin(
        selected_target_ids
    )
]


drug_to_targets = {}


for drug_id, group in bio_pairs.groupby(
    "drug_id"
):

    drug_to_targets[
        str(drug_id)
    ] = set(
        group["target_id"].astype(str)
    )


# ==============================================================================
# 18. TARGET VECTOR HELPER
# ==============================================================================

def target_vector_for_drug(drug_id):

    target = torch.zeros(
        NUM_TARGET_CLASSES,
        dtype=torch.float32,
        device=DEVICE
    )


    associated_targets = drug_to_targets.get(
        str(drug_id),
        set()
    )


    for target_id in associated_targets:

        class_index = target_to_index.get(
            target_id
        )


        if class_index is not None:

            target[
                class_index
            ] = 1.0


    return target


# ==============================================================================
# 19. FISHER REFERENCE LABEL AUDIT
# ==============================================================================

selected_target_label_counts = []


for drug_id in D_fisher[
    "drug_id"
].astype(str):

    target = target_vector_for_drug(
        drug_id
    )


    selected_target_label_counts.append(
        int(
            target.sum().item()
        )
    )


D_fisher = D_fisher.copy()


D_fisher[
    "selected_target_label_count"
] = selected_target_label_counts


D_fisher[
    "has_selected_target"
] = (
    D_fisher[
        "selected_target_label_count"
    ]
    >
    0
)


fisher_with_targets = int(
    D_fisher[
        "has_selected_target"
    ].sum()
)


fisher_without_targets = int(
    (
        ~D_fisher[
            "has_selected_target"
        ]
    ).sum()
)


print(
    f"\nFisher molecules with ≥1 selected target : "
    f"{fisher_with_targets}"
)

print(
    f"Fisher molecules with 0 selected targets : "
    f"{fisher_without_targets}"
)


# ==============================================================================
# 20. DETERMINE GRAPH DIRECTORY
# ==============================================================================

if "GRAPH_DIR" in globals():

    GRAPH_BASE_DIR = Path(
        GRAPH_DIR
    )


else:

    graph_dir_candidates = [
        PROCESSED_DIR.parent
        / "graphs",

        PROCESSED_DIR
        / "graphs",

        Path("data")
        / "graphs"
    ]


    GRAPH_BASE_DIR = None


    for candidate in graph_dir_candidates:

        if candidate.exists():

            GRAPH_BASE_DIR = candidate

            break


if GRAPH_BASE_DIR is None:

    raise RuntimeError(
        "Unable to locate graph directory. "
        "Please ensure GRAPH_DIR from Phase 7 is available."
    )


print(
    f"\n[INFO] Graph directory : {GRAPH_BASE_DIR}"
)


# ==============================================================================
# 21. GRAPH PATH COLUMN DETECTION
# ==============================================================================

graph_path_column_candidates = [
    "graph_file",
    "graph_path",
    "file_path",
    "path"
]


GRAPH_PATH_COLUMN = None


for candidate in graph_path_column_candidates:

    if candidate in D_fisher.columns:

        GRAPH_PATH_COLUMN = candidate

        break


# ==============================================================================
# 22. GRAPH FILE RESOLVER
# ==============================================================================

def resolve_graph_file(row):

    if GRAPH_PATH_COLUMN is not None:

        path_text = str(
            row[GRAPH_PATH_COLUMN]
        ).strip()


        if path_text:

            path = Path(
                path_text
            )


            if path.exists():

                return path


            candidate = (
                GRAPH_BASE_DIR
                /
                path
            )


            if candidate.exists():

                return candidate


    graph_id = str(
        row["graph_id"]
    )


    direct_candidates = [
        GRAPH_BASE_DIR
        / f"{graph_id}.pt",

        GRAPH_BASE_DIR
        / f"{graph_id}.pth",

        GRAPH_BASE_DIR
        / f"{graph_id}.pkl"
    ]


    for candidate in direct_candidates:

        if candidate.exists():

            return candidate


    recursive_matches = []


    for extension in [
        "pt",
        "pth",
        "pkl"
    ]:

        recursive_matches.extend(
            GRAPH_BASE_DIR.rglob(
                f"{graph_id}.{extension}"
            )
        )


    if recursive_matches:

        return recursive_matches[0]


    raise FileNotFoundError(
        f"Graph file not found for graph_id={graph_id}"
    )


resolved_graph_files = []


for _, row in D_fisher.iterrows():

    resolved_graph_files.append(
        str(
            resolve_graph_file(
                row
            )
        )
    )


D_fisher[
    "resolved_graph_file"
] = resolved_graph_files


print(
    f"[PASS] All {len(D_fisher)} Fisher graph files resolved."
)


# ==============================================================================
# 23. GRAPH LOADING HELPER
# ==============================================================================

def load_graph_object(graph_path):

    return torch.load(
        graph_path,
        map_location="cpu",
        weights_only=False
    )


# ==============================================================================
# 24. GENERIC GRAPH ATTRIBUTE HELPER
# ==============================================================================

def graph_value(
    graph,
    possible_names,
    required=True
):

    for name in possible_names:

        if isinstance(graph, dict):

            if name in graph:

                return graph[name]


        else:

            if hasattr(graph, name):

                return getattr(
                    graph,
                    name
                )


    if required:

        raise KeyError(
            f"Unable to locate graph attribute among: "
            f"{possible_names}"
        )


    return None


# ==============================================================================
# 25. GRAPH TENSOR EXTRACTION
# ==============================================================================

def extract_graph_tensors(graph):

    x = graph_value(
        graph,
        [
            "x",
            "node_features",
            "node_feature"
        ]
    )


    edge_index = graph_value(
        graph,
        [
            "edge_index",
            "edges"
        ]
    )


    edge_attr = graph_value(
        graph,
        [
            "edge_attr",
            "edge_features",
            "edge_feature"
        ],
        required=False
    )


    coordinates = graph_value(
        graph,
        [
            "pos",
            "positions",
            "coordinates",
            "coords"
        ]
    )


    x = torch.as_tensor(
        x,
        dtype=torch.float32,
        device=DEVICE
    )


    edge_index = torch.as_tensor(
        edge_index,
        dtype=torch.long,
        device=DEVICE
    )


    coordinates = torch.as_tensor(
        coordinates,
        dtype=torch.float32,
        device=DEVICE
    )


    if edge_attr is not None:

        edge_attr = torch.as_tensor(
            edge_attr,
            dtype=torch.float32,
            device=DEVICE
        )


    if (
        edge_index.ndim == 2
        and
        edge_index.shape[0] != 2
        and
        edge_index.shape[1] == 2
    ):

        edge_index = edge_index.T.contiguous()


    batch = torch.zeros(
        x.shape[0],
        dtype=torch.long,
        device=DEVICE
    )


    return {
        "x":
            x,

        "edge_index":
            edge_index,

        "edge_attr":
            edge_attr,

        "pos":
            coordinates,

        "batch":
            batch
    }


# ==============================================================================
# 26. LOAD PHASE-9 FEATURE NORMALIZATION
# ==============================================================================

phase9_normalization_metadata = None


if PHASE9_FEATURE_NORMALIZATION_FILE.exists():

    with open(
        PHASE9_FEATURE_NORMALIZATION_FILE,
        "r",
        encoding="utf-8"
    ) as file:

        phase9_normalization_metadata = json.load(
            file
        )


# ==============================================================================
# 27. RECURSIVE JSON KEY SEARCH
# ==============================================================================

def recursively_find_key(
    obj,
    keys
):

    if isinstance(obj, dict):

        for key in keys:

            if key in obj:

                return obj[key]


        for value in obj.values():

            result = recursively_find_key(
                value,
                keys
            )


            if result is not None:

                return result


    return None


# ==============================================================================
# 28. NODE FEATURE NORMALIZATION
# ==============================================================================

def normalize_graph_node_features(x):

    # --------------------------------------------------------------------------
    # Prefer original notebook helper if still available.
    # --------------------------------------------------------------------------

    possible_helpers = [
        "normalize_node_features",
        "apply_node_feature_normalization",
        "normalize_graph_features"
    ]


    for helper_name in possible_helpers:

        helper = globals().get(
            helper_name
        )


        if callable(helper):

            try:

                normalized = helper(x)


                if torch.is_tensor(
                    normalized
                ):

                    return normalized.to(
                        DEVICE
                    )


            except Exception:

                pass


    # --------------------------------------------------------------------------
    # Fall back to saved Phase-9 metadata.
    # --------------------------------------------------------------------------

    if phase9_normalization_metadata is None:

        raise RuntimeError(
            "Phase-9/10 graph-feature normalization is unavailable.\n"
            "Execute the Phase-9/10 feature-normalization cell first."
        )


    indices = recursively_find_key(
        phase9_normalization_metadata,
        [
            "feature_indices",
            "normalized_feature_indices",
            "indices"
        ]
    )


    means = recursively_find_key(
        phase9_normalization_metadata,
        [
            "means",
            "mean"
        ]
    )


    stds = recursively_find_key(
        phase9_normalization_metadata,
        [
            "stds",
            "std",
            "standard_deviations"
        ]
    )


    if (
        indices is None
        or
        means is None
        or
        stds is None
    ):

        raise RuntimeError(
            "Could not interpret Phase-9 feature-normalization metadata."
        )


    indices = [
        int(index)
        for index in indices
    ]


    means = torch.tensor(
        means,
        dtype=x.dtype,
        device=x.device
    )


    stds = torch.tensor(
        stds,
        dtype=x.dtype,
        device=x.device
    )


    stds = torch.where(
        stds.abs()
        <
        1e-8,

        torch.ones_like(
            stds
        ),

        stds
    )


    normalized = x.clone()


    normalized[
        :,
        indices
    ] = (
        normalized[
            :,
            indices
        ]
        -
        means
    ) / stds


    return normalized


# ==============================================================================
# 29. LOCATE THE REAL PHASE-10 ENCODER INSTANCE
#
# IMPORTANT:
#
# Phase 13 must use the actual encoder architecture.
#
# Stored latent vectors alone cannot calculate:
#
# ∂L / ∂θE
#
# Therefore a matching in-memory nn.Module must exist.
# ==============================================================================

def module_matches_checkpoint(
    module,
    checkpoint_state
):

    try:

        state = module.state_dict()

    except Exception:

        return False


    if set(state.keys()) != set(
        checkpoint_state.keys()
    ):

        return False


    for key in state:

        if state[key].shape != checkpoint_state[key].shape:

            return False


    return True


encoder_template = None

encoder_template_name = None


for global_name, global_object in list(
    globals().items()
):

    if not isinstance(
        global_object,
        nn.Module
    ):

        continue


    if module_matches_checkpoint(
        global_object,
        encoder_checkpoint[
            "state_dict"
        ]
    ):

        encoder_template = global_object

        encoder_template_name = global_name

        break


if encoder_template is None:

    raise RuntimeError(
        "\nPHASE-10 ENCODER ARCHITECTURE NOT FOUND IN NOTEBOOK MEMORY.\n\n"
        "Phase 13 requires the actual Phase-10 encoder computational graph "
        "to calculate encoder Fisher gradients.\n\n"
        "Please execute the Phase-9 / Phase-10 cell that defines and "
        "instantiates GeometryAwareVariationalEncoder, then rerun Phase 13.\n\n"
        "Stored Phase-10 latent vectors cannot substitute for the encoder "
        "because they do not retain ∂z/∂θE."
    )


encoder = copy.deepcopy(
    encoder_template
).to(
    DEVICE
)


encoder.load_state_dict(
    encoder_checkpoint[
        "state_dict"
    ]
)


encoder.eval()


print(
    f"\n[PASS] Actual Phase-10 encoder architecture located: "
    f"{encoder_template_name}"
)


# ==============================================================================
# 30. RECREATE PHASE-12 PREDICTOR
# ==============================================================================

predictor_config = predictor_checkpoint.get(
    "config",
    {}
)


PREDICTOR_HIDDEN_DIM_1 = int(
    predictor_config.get(
        "hidden_dimension_1",
        64
    )
)


PREDICTOR_HIDDEN_DIM_2 = int(
    predictor_config.get(
        "hidden_dimension_2",
        32
    )
)


PREDICTOR_DROPOUT = float(
    predictor_config.get(
        "dropout",
        0.15
    )
)


class BiologicalTargetAssociationPredictor(nn.Module):

    def __init__(
        self,
        latent_dim,
        hidden_dim_1,
        hidden_dim_2,
        output_dim,
        dropout
    ):

        super().__init__()


        self.network = nn.Sequential(

            nn.Linear(
                latent_dim,
                hidden_dim_1
            ),

            nn.LayerNorm(
                hidden_dim_1
            ),

            nn.SiLU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                hidden_dim_1,
                hidden_dim_2
            ),

            nn.LayerNorm(
                hidden_dim_2
            ),

            nn.SiLU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                hidden_dim_2,
                output_dim
            )
        )


    def forward(
        self,
        latent
    ):

        return self.network(
            latent
        )


predictor = BiologicalTargetAssociationPredictor(
    latent_dim=LATENT_DIM,
    hidden_dim_1=PREDICTOR_HIDDEN_DIM_1,
    hidden_dim_2=PREDICTOR_HIDDEN_DIM_2,
    output_dim=NUM_TARGET_CLASSES,
    dropout=PREDICTOR_DROPOUT
).to(
    DEVICE
)


predictor.load_state_dict(
    predictor_checkpoint[
        "state_dict"
    ]
)


predictor.eval()


print(
    "[PASS] Phase-12 predictor architecture reconstructed."
)


# ==============================================================================
# 31. ENABLE GRADIENT CALCULATION
#
# IMPORTANT:
#
# requires_grad=True is needed to measure gradients.
#
# It does NOT mean that parameters will be updated.
#
# No optimizer is created anywhere in Phase 13.
# ==============================================================================

for parameter in encoder.parameters():

    parameter.requires_grad_(
        True
    )


for parameter in predictor.parameters():

    parameter.requires_grad_(
        True
    )


# ==============================================================================
# 32. FISHER TASK LOSS
# ==============================================================================

fisher_loss_function = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight_tensor,
    reduction="mean"
)


# ==============================================================================
# 33. FLEXIBLE ENCODER FORWARD HELPER
# ==============================================================================

def call_encoder(
    model,
    graph_tensors
):

    signature = inspect.signature(
        model.forward
    )


    parameter_names = set(
        signature.parameters.keys()
    )


    x = graph_tensors["x"]
    edge_index = graph_tensors["edge_index"]
    edge_attr = graph_tensors["edge_attr"]
    pos = graph_tensors["pos"]
    batch = graph_tensors["batch"]


    value_map = {
        "x":
            x,

        "node_features":
            x,

        "h":
            x,

        "edge_index":
            edge_index,

        "edge_attr":
            edge_attr,

        "edge_features":
            edge_attr,

        "pos":
            pos,

        "coords":
            pos,

        "coordinates":
            pos,

        "batch":
            batch,

        "batch_index":
            batch
    }


    kwargs = {}


    for parameter_name in parameter_names:

        if parameter_name in value_map:

            value = value_map[
                parameter_name
            ]


            if value is not None:

                kwargs[
                    parameter_name
                ] = value


    try:

        return model(
            **kwargs
        )


    except Exception as keyword_error:

        positional_attempts = [
            (
                x,
                edge_index,
                edge_attr,
                pos,
                batch
            ),

            (
                x,
                edge_index,
                pos,
                edge_attr,
                batch
            ),

            (
                x,
                edge_index,
                edge_attr,
                pos
            ),

            (
                x,
                edge_index,
                pos,
                edge_attr
            )
        ]


        last_error = keyword_error


        for attempt in positional_attempts:

            try:

                return model(
                    *attempt
                )


            except Exception as error:

                last_error = error


        raise RuntimeError(
            "Unable to call the Phase-10 encoder.\n"
            f"Forward signature: {signature}\n"
            f"Last error: {last_error}"
        )


# ==============================================================================
# 34. EXTRACT DETERMINISTIC μ
# ==============================================================================

def extract_mu(
    encoder_output
):

    # Dictionary output.

    if isinstance(
        encoder_output,
        dict
    ):

        for key in [
            "mu",
            "mean",
            "latent_mu",
            "z_mu"
        ]:

            if key in encoder_output:

                return encoder_output[
                    key
                ]


    # Tuple / list output.

    if isinstance(
        encoder_output,
        (
            tuple,
            list
        )
    ):

        candidates = [
            item
            for item in encoder_output
            if torch.is_tensor(item)
            and item.ndim >= 1
            and item.shape[-1] == LATENT_DIM
        ]


        if len(candidates) == 0:

            raise RuntimeError(
                "Encoder output contains no 32D latent tensor."
            )


        # Phase-10 implementation is expected to expose μ first.

        return candidates[0]


    # Object attributes.

    for attribute in [
        "mu",
        "mean",
        "latent_mu"
    ]:

        if hasattr(
            encoder_output,
            attribute
        ):

            return getattr(
                encoder_output,
                attribute
            )


    # Direct tensor fallback.

    if (
        torch.is_tensor(
            encoder_output
        )
        and
        encoder_output.shape[-1]
        ==
        LATENT_DIM
    ):

        return encoder_output


    raise RuntimeError(
        "Could not identify deterministic μ in encoder output."
    )


# ==============================================================================
# 35. TEST ACTUAL ENCODER PATHWAY
# ==============================================================================

first_graph_path = Path(
    D_fisher.iloc[0][
        "resolved_graph_file"
    ]
)


first_graph = load_graph_object(
    first_graph_path
)


first_tensors = extract_graph_tensors(
    first_graph
)


first_tensors[
    "x"
] = normalize_graph_node_features(
    first_tensors[
        "x"
    ]
)


encoder.zero_grad(
    set_to_none=True
)


predictor.zero_grad(
    set_to_none=True
)


first_encoder_output = call_encoder(
    encoder,
    first_tensors
)


first_mu = extract_mu(
    first_encoder_output
)


if first_mu.ndim == 1:

    first_mu = first_mu.unsqueeze(
        0
    )


if first_mu.shape[-1] != LATENT_DIM:

    raise RuntimeError(
        "Encoder latent dimension mismatch."
    )


if not torch.isfinite(
    first_mu
).all():

    raise RuntimeError(
        "Encoder produced non-finite μ."
    )


print(
    "\n[PASS] Actual encoder → μ computational pathway verified."
)

print(
    f"[INFO] μ shape : {tuple(first_mu.shape)}"
)


# ==============================================================================
# 36. INITIALIZE FISHER ACCUMULATORS
# ==============================================================================

encoder_fisher = {
    name:
        torch.zeros_like(
            parameter,
            device="cpu"
        )

    for name, parameter
    in encoder.named_parameters()
}


predictor_fisher = {
    name:
        torch.zeros_like(
            parameter,
            device="cpu"
        )

    for name, parameter
    in predictor.named_parameters()
}


print(
    f"\nEncoder Fisher tensors   : "
    f"{len(encoder_fisher)}"
)

print(
    f"Predictor Fisher tensors : "
    f"{len(predictor_fisher)}"
)


# ==============================================================================
# 37. SAVE θ* PARAMETER COPIES BEFORE FISHER
# ==============================================================================

encoder_theta_star = {
    name:
        parameter.detach().cpu().clone()

    for name, parameter
    in encoder.named_parameters()
}


predictor_theta_star = {
    name:
        parameter.detach().cpu().clone()

    for name, parameter
    in predictor.named_parameters()
}


# ==============================================================================
# 38. SAMPLE-WISE DIAGONAL EMPIRICAL FISHER
# ==============================================================================

print("\n" + "=" * 132)
print("SAMPLE-WISE DIAGONAL EMPIRICAL FISHER ESTIMATION")
print("=" * 132)


fisher_losses = []


inactive_encoder_parameters = set()

inactive_predictor_parameters = set()


fisher_start_time = time.time()


for sample_index, row in D_fisher.reset_index(
    drop=True
).iterrows():

    graph_id = str(
        row["graph_id"]
    )


    drug_id = str(
        row["drug_id"]
    )


    graph_path = Path(
        row["resolved_graph_file"]
    )


    # --------------------------------------------------------------------------
    # Zero gradients for THIS molecule.
    # --------------------------------------------------------------------------

    encoder.zero_grad(
        set_to_none=True
    )


    predictor.zero_grad(
        set_to_none=True
    )


    # --------------------------------------------------------------------------
    # Load graph.
    # --------------------------------------------------------------------------

    graph = load_graph_object(
        graph_path
    )


    graph_tensors = extract_graph_tensors(
        graph
    )


    # --------------------------------------------------------------------------
    # Reuse exact baseline graph normalization.
    # --------------------------------------------------------------------------

    graph_tensors[
        "x"
    ] = normalize_graph_node_features(
        graph_tensors[
            "x"
        ]
    )


    # --------------------------------------------------------------------------
    # θE* forward.
    # --------------------------------------------------------------------------

    encoder_output = call_encoder(
        encoder,
        graph_tensors
    )


    mu = extract_mu(
        encoder_output
    )


    if mu.ndim == 1:

        mu = mu.unsqueeze(
            0
        )


    if not torch.isfinite(
        mu
    ).all():

        raise RuntimeError(
            f"Non-finite μ for graph {graph_id}."
        )


    # --------------------------------------------------------------------------
    # Frozen Phase-12 latent normalization.
    # --------------------------------------------------------------------------

    normalized_mu = (
        mu
        -
        latent_mean.unsqueeze(
            0
        )
    ) / latent_std_safe.unsqueeze(
        0
    )


    # --------------------------------------------------------------------------
    # θP* forward.
    # --------------------------------------------------------------------------

    logits = predictor(
        normalized_mu
    )


    # --------------------------------------------------------------------------
    # Frozen Phase-12 target label space.
    # --------------------------------------------------------------------------

    target = target_vector_for_drug(
        drug_id
    ).unsqueeze(
        0
    )


    # --------------------------------------------------------------------------
    # Baseline target-association loss.
    # --------------------------------------------------------------------------

    loss = fisher_loss_function(
        logits,
        target
    )


    if not torch.isfinite(
        loss
    ):

        raise RuntimeError(
            f"Non-finite Fisher loss for graph {graph_id}."
        )


    # --------------------------------------------------------------------------
    # Backpropagation ONLY.
    #
    # No optimizer exists.
    # --------------------------------------------------------------------------

    loss.backward()


    # --------------------------------------------------------------------------
    # Encoder Fisher accumulation.
    #
    # Correct:
    #
    # accumulate g_i² PER SAMPLE.
    # --------------------------------------------------------------------------

    for name, parameter in encoder.named_parameters():

        if parameter.grad is None:

            inactive_encoder_parameters.add(
                name
            )

            continue


        gradient = parameter.grad.detach()


        if not torch.isfinite(
            gradient
        ).all():

            raise RuntimeError(
                f"Non-finite encoder gradient in {name}."
            )


        encoder_fisher[
            name
        ] += (
            gradient.cpu()
            **
            2
        )


    # --------------------------------------------------------------------------
    # Predictor Fisher accumulation.
    # --------------------------------------------------------------------------

    for name, parameter in predictor.named_parameters():

        if parameter.grad is None:

            inactive_predictor_parameters.add(
                name
            )

            continue


        gradient = parameter.grad.detach()


        if not torch.isfinite(
            gradient
        ).all():

            raise RuntimeError(
                f"Non-finite predictor gradient in {name}."
            )


        predictor_fisher[
            name
        ] += (
            gradient.cpu()
            **
            2
        )


    fisher_losses.append(
        float(
            loss.item()
        )
    )


    if (
        sample_index
        +
        1
    ) % 16 == 0:

        print(
            f"[FISHER] "
            f"{sample_index + 1:03d}/"
            f"{len(D_fisher)} molecules processed | "
            f"running mean loss="
            f"{np.mean(fisher_losses):.6f}"
        )


# ==============================================================================
# 39. AVERAGE FISHER OVER REFERENCE MOLECULES
# ==============================================================================

N_FISHER = len(
    D_fisher
)


for name in encoder_fisher:

    encoder_fisher[
        name
    ] /= float(
        N_FISHER
    )


for name in predictor_fisher:

    predictor_fisher[
        name
    ] /= float(
        N_FISHER
    )


fisher_duration_seconds = (
    time.time()
    -
    fisher_start_time
)


print(
    f"\n[PASS] Fisher averaged across "
    f"{N_FISHER} sample-wise gradients."
)


# ==============================================================================
# 40. RETURN MODELS TO FROZEN REFERENCE STATUS
# ==============================================================================

for parameter in encoder.parameters():

    parameter.requires_grad_(
        False
    )


for parameter in predictor.parameters():

    parameter.requires_grad_(
        False
    )


encoder.eval()

predictor.eval()


# ==============================================================================
# 41. VERIFY θ* PARAMETER VALUES DID NOT CHANGE
# ==============================================================================

def parameters_equal_reference(
    model,
    reference
):

    for name, parameter in model.named_parameters():

        if name not in reference:

            return False


        if not torch.equal(
            parameter.detach().cpu(),
            reference[name]
        ):

            return False


    return True


encoder_parameters_unchanged = parameters_equal_reference(
    encoder,
    encoder_theta_star
)


predictor_parameters_unchanged = parameters_equal_reference(
    predictor,
    predictor_theta_star
)


if not encoder_parameters_unchanged:

    raise RuntimeError(
        "θE* changed during Fisher estimation."
    )


if not predictor_parameters_unchanged:

    raise RuntimeError(
        "θP* changed during Fisher estimation."
    )


print(
    "[PASS] θE* parameter values unchanged."
)

print(
    "[PASS] θP* parameter values unchanged."
)


# ==============================================================================
# 42. FISHER VALIDITY
# ==============================================================================

def validate_fisher(
    fisher_dictionary,
    component_name
):

    for parameter_name, fisher_tensor in fisher_dictionary.items():

        if not torch.isfinite(
            fisher_tensor
        ).all():

            raise RuntimeError(
                f"{component_name} Fisher contains "
                f"non-finite values in {parameter_name}."
            )


        if (
            fisher_tensor
            <
            0
        ).any():

            raise RuntimeError(
                f"{component_name} Fisher contains "
                f"negative values in {parameter_name}."
            )


    return True


encoder_fisher_valid = validate_fisher(
    encoder_fisher,
    "ENCODER"
)


predictor_fisher_valid = validate_fisher(
    predictor_fisher,
    "PREDICTOR"
)


print(
    "[PASS] Encoder Fisher finite and non-negative."
)

print(
    "[PASS] Predictor Fisher finite and non-negative."
)


# ==============================================================================
# 43. PARAMETER-LEVEL STATISTICS
# ==============================================================================

def fisher_parameter_statistics(
    component,
    fisher_dictionary,
    theta_reference,
    inactive_parameters
):

    records = []


    for parameter_name, fisher_tensor in fisher_dictionary.items():

        flat_fisher = fisher_tensor.reshape(
            -1
        )


        total_elements = flat_fisher.numel()


        nonzero_elements = int(
            (
                flat_fisher
                >
                0
            ).sum().item()
        )


        zero_elements = (
            total_elements
            -
            nonzero_elements
        )


        theta_tensor = theta_reference[
            parameter_name
        ]


        records.append(
            {
                "component":
                    component,

                "parameter_name":
                    parameter_name,

                "shape":
                    str(
                        tuple(
                            fisher_tensor.shape
                        )
                    ),

                "elements":
                    total_elements,

                "inactive_under_fisher_objective":
                    parameter_name
                    in inactive_parameters,

                "nonzero_elements":
                    nonzero_elements,

                "zero_elements":
                    zero_elements,

                "nonzero_fraction":
                    (
                        nonzero_elements
                        /
                        max(
                            total_elements,
                            1
                        )
                    ),

                "fisher_min":
                    float(
                        flat_fisher.min().item()
                    ),

                "fisher_max":
                    float(
                        flat_fisher.max().item()
                    ),

                "fisher_mean":
                    float(
                        flat_fisher.mean().item()
                    ),

                "fisher_median":
                    float(
                        flat_fisher.median().item()
                    ),

                "fisher_std":
                    float(
                        flat_fisher.std(
                            unbiased=False
                        ).item()
                    ),

                "fisher_sum":
                    float(
                        flat_fisher.sum().item()
                    ),

                "theta_l2_norm":
                    float(
                        torch.linalg.vector_norm(
                            theta_tensor
                            .reshape(-1)
                            .float()
                        ).item()
                    )
            }
        )


    return records


parameter_records = []


parameter_records.extend(
    fisher_parameter_statistics(
        "ENCODER",
        encoder_fisher,
        encoder_theta_star,
        inactive_encoder_parameters
    )
)


parameter_records.extend(
    fisher_parameter_statistics(
        "PREDICTOR",
        predictor_fisher,
        predictor_theta_star,
        inactive_predictor_parameters
    )
)


parameter_statistics_df = pd.DataFrame(
    parameter_records
)


# ==============================================================================
# 44. GLOBAL FISHER STATISTICS
# ==============================================================================

def global_fisher_statistics(
    fisher_dictionary
):

    flat = torch.cat(
        [
            tensor.reshape(-1)
            for tensor
            in fisher_dictionary.values()
        ]
    )


    total_elements = flat.numel()


    nonzero_elements = int(
        (
            flat
            >
            0
        ).sum().item()
    )


    return {
        "elements":
            total_elements,

        "nonzero_elements":
            nonzero_elements,

        "zero_elements":
            total_elements
            -
            nonzero_elements,

        "nonzero_fraction":
            (
                nonzero_elements
                /
                max(
                    total_elements,
                    1
                )
            ),

        "minimum":
            float(
                flat.min().item()
            ),

        "maximum":
            float(
                flat.max().item()
            ),

        "mean":
            float(
                flat.mean().item()
            ),

        "median":
            float(
                flat.median().item()
            ),

        "standard_deviation":
            float(
                flat.std(
                    unbiased=False
                ).item()
            ),

        "sum":
            float(
                flat.sum().item()
            )
    }


encoder_global_stats = global_fisher_statistics(
    encoder_fisher
)


predictor_global_stats = global_fisher_statistics(
    predictor_fisher
)


combined_fisher_sum = (
    encoder_global_stats[
        "sum"
    ]
    +
    predictor_global_stats[
        "sum"
    ]
)


parameter_statistics_df[
    "normalized_global_importance"
] = (
    parameter_statistics_df[
        "fisher_sum"
    ]
    /
    (
        combined_fisher_sum
        +
        EPSILON
    )
)


parameter_statistics_df.to_csv(
    PHASE13_PARAMETER_STATS_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 45. LAYER-WISE STATISTICS
# ==============================================================================

def parameter_to_layer_name(
    parameter_name
):

    parts = parameter_name.split(
        "."
    )


    if len(parts) <= 1:

        return parameter_name


    return ".".join(
        parts[:-1]
    )


parameter_statistics_df[
    "layer_name"
] = parameter_statistics_df[
    "parameter_name"
].apply(
    parameter_to_layer_name
)


layer_records = []


for (
    component,
    layer_name
), group in parameter_statistics_df.groupby(
    [
        "component",
        "layer_name"
    ]
):

    total_elements = int(
        group[
            "elements"
        ].sum()
    )


    nonzero_elements = int(
        group[
            "nonzero_elements"
        ].sum()
    )


    fisher_sum = float(
        group[
            "fisher_sum"
        ].sum()
    )


    layer_records.append(
        {
            "component":
                component,

            "layer_name":
                layer_name,

            "parameter_tensors":
                len(group),

            "elements":
                total_elements,

            "nonzero_elements":
                nonzero_elements,

            "nonzero_fraction":
                (
                    nonzero_elements
                    /
                    max(
                        total_elements,
                        1
                    )
                ),

            "fisher_sum":
                fisher_sum,

            "fisher_mean_weighted":
                (
                    fisher_sum
                    /
                    max(
                        total_elements,
                        1
                    )
                ),

            "largest_parameter_fisher_value":
                float(
                    group[
                        "fisher_max"
                    ].max()
                )
        }
    )


layer_statistics_df = pd.DataFrame(
    layer_records
)


layer_statistics_df.to_csv(
    PHASE13_LAYER_STATS_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 46. TOP PARAMETER GROUPS
# ==============================================================================

top_parameters_df = (
    parameter_statistics_df
    .sort_values(
        "fisher_sum",
        ascending=False
    )
    .head(30)
    .reset_index(drop=True)
)


top_parameters_df.to_csv(
    PHASE13_TOP_PARAMETERS_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 47. SAVE FISHER TENSORS
# ==============================================================================

torch.save(
    {
        "component":
            "encoder",

        "fisher_type":
            "diagonal_empirical_fisher",

        "reference_size":
            N_FISHER,

        "objective":
            "DrugBank target-association BCE",

        "latent_policy":
            "deterministic_mu",

        "fisher":
            encoder_fisher
    },
    PHASE13_ENCODER_FISHER_FILE
)


torch.save(
    {
        "component":
            "predictor",

        "fisher_type":
            "diagonal_empirical_fisher",

        "reference_size":
            N_FISHER,

        "objective":
            "DrugBank target-association BCE",

        "fisher":
            predictor_fisher
    },
    PHASE13_PREDICTOR_FISHER_FILE
)


# ==============================================================================
# 48. SAVE θ* + F REFERENCE FOR FUTURE EWC
# ==============================================================================

parameter_reference = {
    "status":
        "BASELINE_EWC_REFERENCE",

    "created_at":
        datetime.now().isoformat(),

    "reference_size":
        N_FISHER,

    "encoder":
        {
            "theta_star":
                encoder_theta_star,

            "fisher":
                encoder_fisher
        },

    "predictor":
        {
            "theta_star":
                predictor_theta_star,

            "fisher":
                predictor_fisher
        },

    "decoder":
        {
            "theta_star_preserved":
                True,

            "fisher_calculated":
                False,

            "reason":
                (
                    "Decoder is not part of the "
                    "DrugBank target-association Fisher objective."
                )
        }
}


torch.save(
    parameter_reference,
    PHASE13_PARAMETER_REFERENCE_FILE
)


# ==============================================================================
# 49. VERIFY FISHER FILES RELOAD
# ==============================================================================

encoder_fisher_reload = torch.load(
    PHASE13_ENCODER_FISHER_FILE,
    map_location="cpu",
    weights_only=False
)


predictor_fisher_reload = torch.load(
    PHASE13_PREDICTOR_FISHER_FILE,
    map_location="cpu",
    weights_only=False
)


encoder_fisher_reloadable = (
    "fisher"
    in encoder_fisher_reload
)


predictor_fisher_reloadable = (
    "fisher"
    in predictor_fisher_reload
)


# ==============================================================================
# 50. SAVE FISHER REFERENCE MANIFEST
# ==============================================================================

manifest_columns = [
    "graph_id",
    "drug_id",
    "resolved_graph_file",
    "selected_target_label_count",
    "has_selected_target"
]


if "drug_name" in D_fisher.columns:

    manifest_columns.insert(
        2,
        "drug_name"
    )


D_fisher[
    manifest_columns
].to_csv(
    PHASE13_REFERENCE_MANIFEST_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 51. HASH CHECK AFTER FISHER ESTIMATION
# ==============================================================================

encoder_hash_after = sha256_file(
    PHASE10_ENCODER_FILE
)

decoder_hash_after = sha256_file(
    PHASE11_DECODER_FILE
)

predictor_hash_after = sha256_file(
    PHASE12_PREDICTOR_FILE
)


encoder_checkpoint_unchanged = (
    encoder_hash_before
    ==
    encoder_hash_after
)


decoder_checkpoint_unchanged = (
    decoder_hash_before
    ==
    decoder_hash_after
)


predictor_checkpoint_unchanged = (
    predictor_hash_before
    ==
    predictor_hash_after
)


if not encoder_checkpoint_unchanged:

    raise RuntimeError(
        "Encoder checkpoint changed during Phase 13."
    )


if not decoder_checkpoint_unchanged:

    raise RuntimeError(
        "Decoder checkpoint changed during Phase 13."
    )


if not predictor_checkpoint_unchanged:

    raise RuntimeError(
        "Predictor checkpoint changed during Phase 13."
    )


print(
    "\n[PASS] Encoder checkpoint hash unchanged."
)

print(
    "[PASS] Decoder checkpoint hash unchanged."
)

print(
    "[PASS] Predictor checkpoint hash unchanged."
)


# ==============================================================================
# 52. CONFIGURATION FILE
# ==============================================================================

phase13_config = {
    "phase":
        "Phase 13 — Baseline Knowledge Consolidation",

    "fisher_type":
        "diagonal empirical Fisher",

    "fisher_formula":
        "mean of squared per-sample gradients",

    "reference_size":
        N_FISHER,

    "reference_source":
        "fixed Phase-8 training-only Fisher subset",

    "fisher_batch_size":
        1,

    "objective":
        "DrugBank multi-label target-association BCEWithLogitsLoss",

    "deterministic_latent":
        "mu",

    "encoder_fisher":
        True,

    "predictor_fisher":
        True,

    "decoder_fisher":
        False,

    "phase9_10_graph_normalization_reused":
        True,

    "phase12_latent_normalization_reused":
        True,

    "phase12_target_vocabulary_reused":
        True,

    "phase12_class_weights_reused":
        True,

    "optimizer_created":
        False,

    "optimizer_step_performed":
        False,

    "parameter_update_performed":
        False,

    "ewc_applied":
        False,

    "closed_loop_adaptation":
        False
}


with open(
    PHASE13_CONFIG_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        phase13_config,
        file,
        indent=4,
        ensure_ascii=False
    )


# ==============================================================================
# 53. CONSOLIDATED KNOWLEDGE STATE K0
# ==============================================================================

knowledge_manifest = {
    "phase":
        "Phase 13 — Baseline Knowledge Consolidation",

    "created_at":
        datetime.now().isoformat(),

    "knowledge_state":
        "K0",

    "Theta_star":
        {
            "theta_E_star":
                "Phase-10 frozen encoder",

            "theta_D_star":
                "Phase-11 frozen molecular decoder",

            "theta_P_star":
                "Phase-12 frozen target-association predictor"
        },

    "Fisher":
        {
            "F_E":
                str(
                    PHASE13_ENCODER_FISHER_FILE
                ),

            "F_P":
                str(
                    PHASE13_PREDICTOR_FISHER_FILE
                ),

            "F_D":
                None
        },

    "Z0":
        str(
            PHASE12_Z0_FILE
        ),

    "parameter_reference":
        str(
            PHASE13_PARAMETER_REFERENCE_FILE
        ),

    "fisher_reference_manifest":
        str(
            PHASE13_REFERENCE_MANIFEST_FILE
        ),

    "fisher_reference_size":
        N_FISHER,

    "fisher_objective":
        "DrugBank target-association prediction",

    "parameter_updates":
        False,

    "ewc_applied":
        False,

    "closed_loop_adaptation":
        False
}


with open(
    PHASE13_KNOWLEDGE_MANIFEST_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        knowledge_manifest,
        file,
        indent=4,
        ensure_ascii=False
    )


# ==============================================================================
# 54. COMPLETION CHECKS
# ==============================================================================

completion_checks = {
    "Phase-12 Θ* loaded":
        True,

    "θE* matches Phase-12 baseline":
        encoder_matches_theta_star,

    "θD* matches Phase-12 baseline":
        decoder_matches_theta_star,

    "θP* matches Phase-12 baseline":
        predictor_matches_theta_star,

    "Fisher reference contains exactly 128 molecules":
        len(D_fisher)
        ==
        EXPECTED_FISHER_REFERENCE_SIZE,

    "Fisher reference is training-only":
        fisher_graph_ids.issubset(
            train_graph_ids
        ),

    "No validation molecules used":
        len(
            fisher_graph_ids
            &
            validation_graph_ids
        )
        ==
        0,

    "No test molecules used":
        len(
            fisher_graph_ids
            &
            test_graph_ids
        )
        ==
        0,

    "All graph files resolved":
        len(
            D_fisher[
                "resolved_graph_file"
            ]
        )
        ==
        EXPECTED_FISHER_REFERENCE_SIZE,

    "Actual Phase-10 encoder used":
        True,

    "Phase-12 predictor reconstructed":
        True,

    "Phase-9/10 graph normalization reused":
        True,

    "Phase-12 latent normalization reused":
        True,

    "Phase-12 target vocabulary reused":
        True,

    "Phase-12 class weights reused":
        True,

    "Deterministic mu used":
        True,

    "Batch size equals 1":
        FISHER_BATCH_SIZE
        ==
        1,

    "Sample-wise gradients used":
        True,

    "Gradients squared before averaging":
        True,

    "Encoder Fisher calculated":
        len(
            encoder_fisher
        )
        >
        0,

    "Predictor Fisher calculated":
        len(
            predictor_fisher
        )
        >
        0,

    "Encoder Fisher finite/non-negative":
        encoder_fisher_valid,

    "Predictor Fisher finite/non-negative":
        predictor_fisher_valid,

    "Encoder Fisher reloadable":
        encoder_fisher_reloadable,

    "Predictor Fisher reloadable":
        predictor_fisher_reloadable,

    "Inactive encoder parameters documented":
        True,

    "Inactive predictor parameters documented":
        True,

    "Parameter statistics saved":
        PHASE13_PARAMETER_STATS_FILE.exists(),

    "Layer statistics saved":
        PHASE13_LAYER_STATS_FILE.exists(),

    "θE* parameter values unchanged":
        encoder_parameters_unchanged,

    "θP* parameter values unchanged":
        predictor_parameters_unchanged,

    "Encoder checkpoint hash unchanged":
        encoder_checkpoint_unchanged,

    "Decoder checkpoint hash unchanged":
        decoder_checkpoint_unchanged,

    "Predictor checkpoint hash unchanged":
        predictor_checkpoint_unchanged,

    "θ* values saved with Fisher":
        PHASE13_PARAMETER_REFERENCE_FILE.exists(),

    "K0 knowledge state established":
        PHASE13_KNOWLEDGE_MANIFEST_FILE.exists(),

    "No optimizer created":
        True,

    "No optimizer step performed":
        True,

    "No EWC applied":
        True,

    "No closed-loop adaptation":
        True
}


# ==============================================================================
# 55. QC SUMMARY
# ==============================================================================

qc_summary = {
    "phase":
        "Phase 13 — Baseline Knowledge Consolidation",

    "created_at":
        datetime.now().isoformat(),

    "status":
        "BASELINE_KNOWLEDGE_CONSOLIDATED",

    "knowledge_state":
        "K0",

    "fisher_reference_molecules":
        N_FISHER,

    "fisher_molecules_with_selected_targets":
        fisher_with_targets,

    "fisher_molecules_without_selected_targets":
        fisher_without_targets,

    "mean_fisher_loss":
        float(
            np.mean(
                fisher_losses
            )
        ),

    "median_fisher_loss":
        float(
            np.median(
                fisher_losses
            )
        ),

    "inactive_encoder_parameter_tensors":
        len(
            inactive_encoder_parameters
        ),

    "inactive_predictor_parameter_tensors":
        len(
            inactive_predictor_parameters
        ),

    "inactive_encoder_parameters":
        sorted(
            list(
                inactive_encoder_parameters
            )
        ),

    "inactive_predictor_parameters":
        sorted(
            list(
                inactive_predictor_parameters
            )
        ),

    "encoder_fisher_statistics":
        encoder_global_stats,

    "predictor_fisher_statistics":
        predictor_global_stats,

    "combined_fisher_sum":
        combined_fisher_sum,

    "fisher_duration_seconds":
        fisher_duration_seconds,

    "decoder_fisher_calculated":
        False,

    "optimizer_created":
        False,

    "parameter_updates":
        False,

    "ewc_applied":
        False,

    "closed_loop_adaptation":
        False,

    "completion_checks":
        completion_checks
}


with open(
    PHASE13_QC_JSON_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        qc_summary,
        file,
        indent=4,
        ensure_ascii=False,
        default=str
    )


qc_rows = []


for key, value in qc_summary.items():

    if key == "completion_checks":

        for check_name, status in value.items():

            qc_rows.append(
                {
                    "metric":
                        f"completion.{check_name}",

                    "value":
                        status
                }
            )


    elif isinstance(
        value,
        dict
    ):

        for subkey, subvalue in value.items():

            qc_rows.append(
                {
                    "metric":
                        f"{key}.{subkey}",

                    "value":
                        subvalue
                }
            )


    elif isinstance(
        value,
        list
    ):

        qc_rows.append(
            {
                "metric":
                    key,

                "value":
                    "; ".join(
                        map(
                            str,
                            value
                        )
                    )
            }
        )


    else:

        qc_rows.append(
            {
                "metric":
                    key,

                "value":
                    value
            }
        )


pd.DataFrame(
    qc_rows
).to_csv(
    PHASE13_QC_CSV_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 56. DISPLAY ENCODER FISHER STATISTICS
# ==============================================================================

print("\n" + "=" * 132)
print("ENCODER FISHER STATISTICS")
print("=" * 132)


for key, value in encoder_global_stats.items():

    if isinstance(value, float):

        print(
            f"{key:<26}: "
            f"{value:.10e}"
        )

    else:

        print(
            f"{key:<26}: "
            f"{value}"
        )


# ==============================================================================
# 57. DISPLAY PREDICTOR FISHER STATISTICS
# ==============================================================================

print("\n" + "=" * 132)
print("PREDICTOR FISHER STATISTICS")
print("=" * 132)


for key, value in predictor_global_stats.items():

    if isinstance(value, float):

        print(
            f"{key:<26}: "
            f"{value:.10e}"
        )

    else:

        print(
            f"{key:<26}: "
            f"{value}"
        )


# ==============================================================================
# 58. DISPLAY INACTIVE PARAMETERS
# ==============================================================================

print("\n" + "=" * 132)
print("INACTIVE PARAMETERS UNDER THIS FISHER OBJECTIVE")
print("=" * 132)


print(
    f"Inactive Encoder Parameter Tensors   : "
    f"{len(inactive_encoder_parameters)}"
)


for name in sorted(
    inactive_encoder_parameters
):

    print(
        f"  ENCODER   : {name}"
    )


print(
    f"\nInactive Predictor Parameter Tensors : "
    f"{len(inactive_predictor_parameters)}"
)


for name in sorted(
    inactive_predictor_parameters
):

    print(
        f"  PREDICTOR : {name}"
    )


print(
    "\n[IMPORTANT] Zero/inactive Fisher does NOT mean a parameter is "
    "globally unimportant."
)

print(
    "[IMPORTANT] It means the parameter was inactive or weakly constrained "
    "under this specific deterministic target-association Fisher objective."
)


# ==============================================================================
# 59. DISPLAY TOP FISHER PARAMETER GROUPS
# ==============================================================================

print("\n" + "=" * 132)
print("TOP FISHER PARAMETER GROUPS")
print("=" * 132)


display_columns = [
    "component",
    "parameter_name",
    "elements",
    "inactive_under_fisher_objective",
    "nonzero_fraction",
    "fisher_mean",
    "fisher_max",
    "fisher_sum",
    "normalized_global_importance"
]


print(
    top_parameters_df[
        display_columns
    ].to_string(
        index=False
    )
)


# ==============================================================================
# 60. COMPLETION VALIDATION
# ==============================================================================

print("\n" + "=" * 132)
print("PHASE 13 COMPLETION VALIDATION")
print("=" * 132)


all_completion_checks_passed = True


for check_name, status in completion_checks.items():

    label = (
        "PASS"
        if bool(status)
        else "FAIL"
    )


    print(
        f"[{label:<4}] {check_name}"
    )


    if not bool(status):

        all_completion_checks_passed = False


# ==============================================================================
# 61. UPDATE DRUGBANK CONFIG
# ==============================================================================

PHASE13_DATASETS = {
    "fisher_config":
        str(
            PHASE13_CONFIG_FILE
        ),

    "fisher_reference_manifest":
        str(
            PHASE13_REFERENCE_MANIFEST_FILE
        ),

    "encoder_fisher":
        str(
            PHASE13_ENCODER_FISHER_FILE
        ),

    "predictor_fisher":
        str(
            PHASE13_PREDICTOR_FISHER_FILE
        ),

    "fisher_parameter_reference":
        str(
            PHASE13_PARAMETER_REFERENCE_FILE
        ),

    "fisher_parameter_statistics":
        str(
            PHASE13_PARAMETER_STATS_FILE
        ),

    "fisher_layer_statistics":
        str(
            PHASE13_LAYER_STATS_FILE
        ),

    "fisher_top_parameters":
        str(
            PHASE13_TOP_PARAMETERS_FILE
        ),

    "consolidated_knowledge_manifest":
        str(
            PHASE13_KNOWLEDGE_MANIFEST_FILE
        ),

    "qc_summary_csv":
        str(
            PHASE13_QC_CSV_FILE
        ),

    "qc_summary_json":
        str(
            PHASE13_QC_JSON_FILE
        ),

    "knowledge_state":
        "K0",

    "fisher_reference_size":
        N_FISHER,

    "encoder_fisher_calculated":
        True,

    "predictor_fisher_calculated":
        True,

    "decoder_fisher_calculated":
        False,

    "ewc_applied":
        False
}


DRUGBANK_CONFIG[
    "phase13_datasets"
] = PHASE13_DATASETS


# ==============================================================================
# 62. FINAL SUMMARY
# ==============================================================================

print("\n" + "=" * 132)
print("PHASE 13 SUMMARY")
print("=" * 132)


print(
    f"Fisher Reference Molecules          : "
    f"{N_FISHER}"
)

print(
    f"With ≥1 Selected Target             : "
    f"{fisher_with_targets}"
)

print(
    f"With 0 Selected Targets             : "
    f"{fisher_without_targets}"
)

print(
    f"Mean Fisher Task Loss               : "
    f"{np.mean(fisher_losses):.8f}"
)

print(
    f"Median Fisher Task Loss             : "
    f"{np.median(fisher_losses):.8f}"
)


print("\nEncoder Fisher:")

print(
    f"  Parameter Elements                : "
    f"{encoder_global_stats['elements']:,}"
)

print(
    f"  Non-Zero Elements                 : "
    f"{encoder_global_stats['nonzero_elements']:,}"
)

print(
    f"  Zero Elements                     : "
    f"{encoder_global_stats['zero_elements']:,}"
)

print(
    f"  Non-Zero Fraction                 : "
    f"{encoder_global_stats['nonzero_fraction']:.4%}"
)

print(
    f"  Minimum                           : "
    f"{encoder_global_stats['minimum']:.10e}"
)

print(
    f"  Maximum                           : "
    f"{encoder_global_stats['maximum']:.10e}"
)

print(
    f"  Mean                              : "
    f"{encoder_global_stats['mean']:.10e}"
)

print(
    f"  Median                            : "
    f"{encoder_global_stats['median']:.10e}"
)

print(
    f"  Sum                               : "
    f"{encoder_global_stats['sum']:.10e}"
)


print("\nPredictor Fisher:")

print(
    f"  Parameter Elements                : "
    f"{predictor_global_stats['elements']:,}"
)

print(
    f"  Non-Zero Elements                 : "
    f"{predictor_global_stats['nonzero_elements']:,}"
)

print(
    f"  Zero Elements                     : "
    f"{predictor_global_stats['zero_elements']:,}"
)

print(
    f"  Non-Zero Fraction                 : "
    f"{predictor_global_stats['nonzero_fraction']:.4%}"
)

print(
    f"  Minimum                           : "
    f"{predictor_global_stats['minimum']:.10e}"
)

print(
    f"  Maximum                           : "
    f"{predictor_global_stats['maximum']:.10e}"
)

print(
    f"  Mean                              : "
    f"{predictor_global_stats['mean']:.10e}"
)

print(
    f"  Median                            : "
    f"{predictor_global_stats['median']:.10e}"
)

print(
    f"  Sum                               : "
    f"{predictor_global_stats['sum']:.10e}"
)


print("\nInactive Under Current Fisher Objective:")

print(
    f"  Encoder Parameter Tensors         : "
    f"{len(inactive_encoder_parameters)}"
)

print(
    f"  Predictor Parameter Tensors       : "
    f"{len(inactive_predictor_parameters)}"
)


print("\nConsolidated Knowledge State:")

print(
    "  θE*                               : PRESERVED"
)

print(
    "  θD*                               : PRESERVED"
)

print(
    "  θP*                               : PRESERVED"
)

print(
    "  F_E                               : ESTABLISHED"
)

print(
    "  F_P                               : ESTABLISHED"
)

print(
    "  F_D                               : NOT CALCULATED"
)

print(
    "  Z0                                : PRESERVED"
)

print(
    "  K0 = {Θ*, F_E, F_P, Z0}          : ESTABLISHED"
)


print("\nIntegrity:")

print(
    f"  θE* Parameters Unchanged          : "
    f"{encoder_parameters_unchanged}"
)

print(
    f"  θP* Parameters Unchanged          : "
    f"{predictor_parameters_unchanged}"
)

print(
    f"  Encoder Hash Unchanged            : "
    f"{encoder_checkpoint_unchanged}"
)

print(
    f"  Decoder Hash Unchanged            : "
    f"{decoder_checkpoint_unchanged}"
)

print(
    f"  Predictor Hash Unchanged          : "
    f"{predictor_checkpoint_unchanged}"
)


print("\nAdaptive-Learning Status:")

print(
    "  Optimizer Created                 : False"
)

print(
    "  Optimizer Step                    : False"
)

print(
    "  EWC Applied                       : False"
)

print(
    "  Closed-Loop Adaptation            : False"
)

print(
    "  Parameter Drift                   : False"
)


print("\nSaved Files:")

print(
    f"  Fisher Config                     : "
    f"{PHASE13_CONFIG_FILE}"
)

print(
    f"  Fisher Reference Manifest         : "
    f"{PHASE13_REFERENCE_MANIFEST_FILE}"
)

print(
    f"  Encoder Fisher F_E                : "
    f"{PHASE13_ENCODER_FISHER_FILE}"
)

print(
    f"  Predictor Fisher F_P              : "
    f"{PHASE13_PREDICTOR_FISHER_FILE}"
)

print(
    f"  θ* + Fisher Reference             : "
    f"{PHASE13_PARAMETER_REFERENCE_FILE}"
)

print(
    f"  Parameter Statistics              : "
    f"{PHASE13_PARAMETER_STATS_FILE}"
)

print(
    f"  Layer Statistics                  : "
    f"{PHASE13_LAYER_STATS_FILE}"
)

print(
    f"  Top Fisher Parameters             : "
    f"{PHASE13_TOP_PARAMETERS_FILE}"
)

print(
    f"  Consolidated K0 Manifest          : "
    f"{PHASE13_KNOWLEDGE_MANIFEST_FILE}"
)

print(
    f"  QC Summary CSV                    : "
    f"{PHASE13_QC_CSV_FILE}"
)

print(
    f"  QC Summary JSON                   : "
    f"{PHASE13_QC_JSON_FILE}"
)


print("-" * 132)


if all_completion_checks_passed:

    print(
        "[STATUS] PHASE 13 COMPLETED SUCCESSFULLY."
    )

    print(
        "[READY] Diagonal empirical Fisher references "
        "F_E and F_P have been established."
    )

    print(
        "[READY] Fisher was calculated sample-wise using "
        "the fixed 128-molecule training reference."
    )

    print(
        "[READY] Gradients were squared before averaging."
    )

    print(
        "[READY] θE*, θD*, and θP* remained unchanged."
    )

    print(
        "[READY] Consolidated baseline knowledge state K0 "
        "has been established."
    )

    print(
        "[IMPORTANT] Fisher values indicate model-parameter "
        "importance under the baseline target-association task."
    )

    print(
        "[IMPORTANT] Zero Fisher does not imply global parameter "
        "irrelevance."
    )

    print(
        "[IMPORTANT] EWC has NOT yet been applied."
    )

    print(
        "[NEXT] Phase 14 — EWC-Regularized Closed-Loop Adaptation"
    )


else:

    print(
        "[STATUS] PHASE 13 REQUIRES REVIEW."
    )

    print(
        "[ACTION] Do not proceed to EWC until all failed "
        "completion checks are resolved."
    )


print("=" * 132)

PHASE 13 — BASELINE KNOWLEDGE CONSOLIDATION AND DIAGONAL EMPIRICAL FISHER INFORMATION

[DEVICE] Device                     : cpu
[CONFIG] Random Seed               : 42
[CONFIG] Latent Dimension          : 32
[CONFIG] Fisher Reference Size     : 128
[CONFIG] Fisher Batch Size         : 1
[CONFIG] Fisher Approximation      : Diagonal empirical Fisher
[CONFIG] Objective                 : DrugBank target-association BCE
[CONFIG] Fisher Components         : Encoder + Predictor
[CONFIG] Decoder Fisher            : Not calculated

[PASS] Required Phase-8, Phase-10, Phase-11 and Phase-12 files found.
[PASS] Phase-12 Θ* reference bundle loaded.
[PASS] θE*, θD*, θP* checkpoints loaded.
[PASS] θE* matches Phase-12 baseline reference.
[PASS] θD* matches Phase-12 baseline reference.
[PASS] θP* matches Phase-12 baseline reference.

PRE-FISHER BASELINE CHECKPOINT HASHES
Encoder SHA-256   : 44c3e8afe63cd7cdc055a6bdc35335c4999bb32969a12426cfee54fa77566f1a
Decoder SHA-256   : 4cee9bb5ae26206f154e67e0f7

# Phase 14 — Latent-Space Candidate Generation and Proposal Pool Construction (Cycle t=1)

**Objective:** Generate a structured proposal pool ($C_1$) of novel, chemically valid molecular candidates by exploring the continuous baseline latent representation space ($Z_0$) using the frozen generative decoder ($\theta_D^*$).

* **Generative Traversal Mechanics:**
  * **Local Perturbation:** $z' = z_{\text{seed}} + \sigma \epsilon$ with controlled perturbation scales $\sigma$ around baseline seeds.
  * **Latent Interpolation:** $z(\alpha) = (1-\alpha)z_a + \alpha z_b$ for $\alpha \in \{0.25, 0.50, 0.75\}$ between baseline seed pairs.
  * **Prior Sampling:** Exploratory background proposals drawn from $z \sim \mathcal{N}(0, I)$.
  * **Stochastic Token Sampling:** Top-$K$ and temperature-regulated sequence generation through the frozen 2-layer GRU decoder.

* **Chemical Verification & Provenance:**
  * **Validation Pipeline:** $\text{SELFIES} \to \text{RDKit Parse} \to \text{Sanitize} \to \text{Valence} \to \text{Standardize (Cleanup/Fragment/Uncharge)} \to \text{Feasibility } (4 \le N_{\text{heavy}} \le 80)$.
  * **Deduplication & Novelty:** Canonical SMILES identity filtering against the 789 standardized baseline DrugBank molecules.
  * **Pool Filtering Hierarchy:** $C_1^{\text{novel}} \subseteq C_1^{\text{unique}} \subseteq C_1^{\text{valid}} \subseteq C_1^{\text{raw}}$.
  * **Immutability Guarantee:** Model parameters $\Theta^*$ and Fisher matrices ($F_E, F_P$) remain completely frozen with zero parameter updates.

* **Primary Artifacts Exported:**
  * Candidate Pools: `phase14_generated_candidates_{raw,validated,unique,novel}.csv`.
  * Audit Manifests: `phase14_candidate_provenance.csv`, `phase14_latent_proposals.csv`.
  * Diagnostics: `phase14_generation_method_summary.csv`, `phase14_generation_qc_summary.json`.

In [14]:
# ==============================================================================
# PHASE 14 — LATENT-SPACE CANDIDATE GENERATION
# AND PROPOSAL POOL CONSTRUCTION
#
# FINAL CORRECTED VERSION
# ==============================================================================
#
# OBJECTIVE
# ---------
#
# Construct the first genuinely generated molecular proposal pool:
#
#     C1 = {c1, c2, ..., cn}
#
# from the learned 32-dimensional molecular latent space using:
#
#     1. seed latent stochastic resampling
#     2. local latent perturbation
#     3. latent interpolation
#     4. limited N(0,I) prior sampling
#     5. stochastic SELFIES decoding
#
#
# IMPORTANT CORRECTION
# --------------------
#
# The actual Phase-11 decoder contains complete latent-conditioning modules.
#
# For example:
#
#     latent_to_hidden
#     latent_to_context
#
# These may themselves contain:
#
#     Linear
#     activation
#     LayerNorm
#     etc.
#
# Therefore this implementation does NOT directly call only:
#
#     latent_to_hidden.0
#     latent_to_context.0
#
# Instead, structural discovery identifies those layers and then recovers
# and calls their COMPLETE PARENT MODULES.
#
# Thus generation follows:
#
#     z
#      ↓
#     COMPLETE latent_to_hidden module
#      ↓
#     GRU initial hidden state
#
# and:
#
#     z
#      ↓
#     COMPLETE latent_to_context module
#      ↓
#     latent context supplied at every decoding step
#
#
# PHASE 14 DOES NOT:
#
#     - update θE*
#     - update θD*
#     - update θP*
#     - change F_E
#     - change F_P
#     - apply EWC
#     - perform target ranking
#     - calculate UCB
#     - perform closed-loop parameter adaptation
#
# ==============================================================================


# ==============================================================================
# 1. IMPORTS
# ==============================================================================

from pathlib import Path
from datetime import datetime

import copy
import hashlib
import json
import random
import time
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

import selfies as sf

from rdkit import Chem
from rdkit import DataStructs
from rdkit.Chem import AllChem
from rdkit.Chem.MolStandardize import rdMolStandardize


warnings.filterwarnings("ignore")


# ==============================================================================
# 2. REPRODUCIBILITY
# ==============================================================================

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cpu")


# ==============================================================================
# 3. VERIFY PROJECT VARIABLES
# ==============================================================================

required_variables = [
    "PROCESSED_DIR",
    "DRUGBANK_CONFIG"
]

missing_variables = [
    name
    for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Required earlier-phase project variables are missing:\n"
        f"{missing_variables}"
    )

PROCESSED_DIR = Path(PROCESSED_DIR)


# ==============================================================================
# 4. PHASE-14 CONFIGURATION
# ==============================================================================

LATENT_DIM = 32
CYCLE_INDEX = 1

# Direct decoding of known training latents.
N_DIRECT_SEEDS = 40

# Local latent exploration.
N_PERTURBATION_SEEDS = 60
PERTURBATION_SIGMAS = [0.10, 0.25]
PERTURBATIONS_PER_SIGMA = 2

# Interpolation.
N_INTERPOLATION_PAIRS = 30
INTERPOLATION_ALPHAS = [0.25, 0.50, 0.75]

# Exploratory prior.
N_PRIOR_LATENTS = 60

# Decoder sampling.
DECODER_SAMPLES_PER_LATENT = 2
DECODER_TEMPERATURE = 0.90
DECODER_TOP_K = 10

# Fallback only if not found in Phase-11 metadata.
MAX_GENERATION_LENGTH_FALLBACK = 122

# Model-feasibility range established earlier.
MIN_HEAVY_ATOMS = 4
MAX_HEAVY_ATOMS = 80

# Molecular fingerprints.
FP_RADIUS = 2
FP_BITS = 2048


print("=" * 138)
print(
    "PHASE 14 — LATENT-SPACE CANDIDATE GENERATION "
    "AND PROPOSAL POOL CONSTRUCTION"
)
print("=" * 138)

print(f"\n[DEVICE] Device                         : {DEVICE}")
print(f"[CONFIG] Random Seed                   : {RANDOM_SEED}")
print(f"[CONFIG] Cycle                         : {CYCLE_INDEX}")
print(f"[CONFIG] Latent Dimension              : {LATENT_DIM}")
print(f"[CONFIG] Direct Seed Latents            : {N_DIRECT_SEEDS}")
print(f"[CONFIG] Perturbation Seeds             : {N_PERTURBATION_SEEDS}")
print(f"[CONFIG] Perturbation Sigmas            : {PERTURBATION_SIGMAS}")
print(f"[CONFIG] Perturbations/Sigma            : {PERTURBATIONS_PER_SIGMA}")
print(f"[CONFIG] Interpolation Pairs            : {N_INTERPOLATION_PAIRS}")
print(f"[CONFIG] Interpolation Alphas           : {INTERPOLATION_ALPHAS}")
print(f"[CONFIG] Prior Latents                  : {N_PRIOR_LATENTS}")
print(
    f"[CONFIG] Decoder Samples/Latent         : "
    f"{DECODER_SAMPLES_PER_LATENT}"
)
print(
    f"[CONFIG] Decoder Temperature            : "
    f"{DECODER_TEMPERATURE}"
)
print(f"[CONFIG] Decoder Top-K                  : {DECODER_TOP_K}")
print(
    f"[CONFIG] Heavy Atom Feasibility         : "
    f"{MIN_HEAVY_ATOMS}-{MAX_HEAVY_ATOMS}"
)


# ==============================================================================
# 5. INPUT FILES
# ==============================================================================

PHASE11_DECODER_FILE = (
    PROCESSED_DIR
    / "phase11_best_generative_decoder.pt"
)

PHASE11_VOCAB_FILE = (
    PROCESSED_DIR
    / "phase11_selfies_vocabulary.json"
)

PHASE11_CONFIG_FILE = (
    PROCESSED_DIR
    / "phase11_decoder_config.json"
)

PHASE12_Z0_FILE = (
    PROCESSED_DIR
    / "phase12_baseline_latent_reference.csv"
)

PHASE12_ENCODER_FILE = (
    PROCESSED_DIR
    / "phase12_thetaE_baseline_encoder.pt"
)

PHASE12_DECODER_FILE = (
    PROCESSED_DIR
    / "phase12_thetaD_baseline_decoder.pt"
)

PHASE12_PREDICTOR_FILE = (
    PROCESSED_DIR
    / "phase12_best_baseline_predictor.pt"
)

PHASE12_REFERENCE_BUNDLE_FILE = (
    PROCESSED_DIR
    / "phase12_theta_star_reference_bundle.pt"
)

PHASE13_ENCODER_FISHER_FILE = (
    PROCESSED_DIR
    / "phase13_encoder_fisher.pt"
)

PHASE13_PREDICTOR_FISHER_FILE = (
    PROCESSED_DIR
    / "phase13_predictor_fisher.pt"
)

PHASE13_REFERENCE_FILE = (
    PROCESSED_DIR
    / "phase13_fisher_parameter_reference.pt"
)


# ==============================================================================
# 6. FIND STANDARDIZED BASELINE CHEMISTRY FILE
# ==============================================================================

baseline_file_candidates = [
    PROCESSED_DIR / "D_chem_standardized.csv",
    PROCESSED_DIR / "D_chem_standardized_qc.csv",
    PROCESSED_DIR.parent / "processed" / "D_chem_standardized.csv"
]

BASELINE_CHEM_FILE = None

for candidate in baseline_file_candidates:
    if candidate.exists():
        BASELINE_CHEM_FILE = candidate
        break


required_files = [
    PHASE11_DECODER_FILE,
    PHASE11_VOCAB_FILE,
    PHASE12_Z0_FILE,
    PHASE12_ENCODER_FILE,
    PHASE12_DECODER_FILE,
    PHASE12_PREDICTOR_FILE,
    PHASE12_REFERENCE_BUNDLE_FILE,
    PHASE13_ENCODER_FISHER_FILE,
    PHASE13_PREDICTOR_FISHER_FILE,
    PHASE13_REFERENCE_FILE
]

missing_files = [
    str(path)
    for path in required_files
    if not path.exists()
]

if BASELINE_CHEM_FILE is None:
    missing_files.append("D_chem_standardized.csv")

if missing_files:
    raise FileNotFoundError(
        "Required Phase-11/12/13 files are missing:\n"
        + "\n".join(missing_files)
    )

print(
    "\n[PASS] Required Phase-11, Phase-12 and Phase-13 files found."
)


# ==============================================================================
# 7. OUTPUT FILES
# ==============================================================================

PHASE14_CONFIG_FILE = (
    PROCESSED_DIR
    / "phase14_generation_config.json"
)

PHASE14_LATENT_PROPOSALS_FILE = (
    PROCESSED_DIR
    / "phase14_latent_proposals.csv"
)

PHASE14_RAW_FILE = (
    PROCESSED_DIR
    / "phase14_generated_candidates_raw.csv"
)

PHASE14_VALIDATED_FILE = (
    PROCESSED_DIR
    / "phase14_generated_candidates_validated.csv"
)

PHASE14_UNIQUE_FILE = (
    PROCESSED_DIR
    / "phase14_generated_candidates_unique.csv"
)

PHASE14_NOVEL_FILE = (
    PROCESSED_DIR
    / "phase14_generated_candidates_novel.csv"
)

PHASE14_PROVENANCE_FILE = (
    PROCESSED_DIR
    / "phase14_candidate_provenance.csv"
)

PHASE14_METHOD_SUMMARY_FILE = (
    PROCESSED_DIR
    / "phase14_generation_method_summary.csv"
)

PHASE14_QC_CSV_FILE = (
    PROCESSED_DIR
    / "phase14_generation_qc_summary.csv"
)

PHASE14_QC_JSON_FILE = (
    PROCESSED_DIR
    / "phase14_generation_qc_summary.json"
)


# ==============================================================================
# 8. SHA-256 HELPER
# ==============================================================================

def sha256_file(
    file_path,
    chunk_size=1024 * 1024
):
    digest = hashlib.sha256()

    with open(file_path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


# ==============================================================================
# 9. RECORD K0 HASHES BEFORE GENERATION
# ==============================================================================

encoder_hash_before = sha256_file(
    PHASE12_ENCODER_FILE
)

decoder_hash_before = sha256_file(
    PHASE12_DECODER_FILE
)

predictor_hash_before = sha256_file(
    PHASE12_PREDICTOR_FILE
)

encoder_fisher_hash_before = sha256_file(
    PHASE13_ENCODER_FISHER_FILE
)

predictor_fisher_hash_before = sha256_file(
    PHASE13_PREDICTOR_FISHER_FILE
)

phase13_reference_hash_before = sha256_file(
    PHASE13_REFERENCE_FILE
)


print("\n" + "=" * 138)
print("PRE-GENERATION BASELINE K0 INTEGRITY")
print("=" * 138)

print(f"θE* SHA-256 : {encoder_hash_before}")
print(f"θD* SHA-256 : {decoder_hash_before}")
print(f"θP* SHA-256 : {predictor_hash_before}")
print(f"F_E SHA-256 : {encoder_fisher_hash_before}")
print(f"F_P SHA-256 : {predictor_fisher_hash_before}")


# ==============================================================================
# 10. LOAD Θ* BUNDLE
# ==============================================================================

phase12_bundle = torch.load(
    PHASE12_REFERENCE_BUNDLE_FILE,
    map_location="cpu",
    weights_only=False
)

for key in [
    "theta_E_star",
    "theta_D_star",
    "theta_P_star"
]:
    if key not in phase12_bundle:
        raise RuntimeError(
            f"Phase-12 Θ* bundle missing key: {key}"
        )


# ==============================================================================
# 11. LOAD PHASE-11 DECODER CHECKPOINT
# ==============================================================================

decoder_checkpoint = torch.load(
    PHASE11_DECODER_FILE,
    map_location="cpu",
    weights_only=False
)

if "state_dict" not in decoder_checkpoint:
    raise RuntimeError(
        "Phase-11 decoder checkpoint missing state_dict."
    )


# ==============================================================================
# 12. STATE-DICT EQUALITY
# ==============================================================================

def state_dicts_equal(
    state_a,
    state_b
):
    if set(state_a.keys()) != set(state_b.keys()):
        return False

    for key in state_a:
        tensor_a = state_a[key].detach().cpu()
        tensor_b = state_b[key].detach().cpu()

        if tensor_a.shape != tensor_b.shape:
            return False

        if not torch.equal(
            tensor_a,
            tensor_b
        ):
            return False

    return True


decoder_matches_theta_star = state_dicts_equal(
    decoder_checkpoint["state_dict"],
    phase12_bundle["theta_D_star"]
)

if not decoder_matches_theta_star:
    raise RuntimeError(
        "Phase-11 decoder does not match frozen Phase-12 θD*."
    )

print(
    "\n[PASS] Phase-11 decoder checkpoint exactly matches θD*."
)


# ==============================================================================
# 13. LOAD BASELINE LATENT REFERENCE Z0
# ==============================================================================

Z0 = pd.read_csv(
    PHASE12_Z0_FILE,
    dtype={
        "graph_id": str,
        "drug_id": str
    }
)

latent_columns = [
    f"z_{index}"
    for index in range(LATENT_DIM)
]

missing_latent_columns = [
    column
    for column in latent_columns
    if column not in Z0.columns
]

if missing_latent_columns:
    raise RuntimeError(
        f"Z0 missing latent columns: {missing_latent_columns}"
    )

Z0_values = (
    Z0[latent_columns]
    .astype(float)
    .to_numpy(dtype=np.float32)
)

if Z0_values.shape[1] != LATENT_DIM:
    raise RuntimeError(
        "Z0 latent dimension mismatch."
    )

if not np.isfinite(Z0_values).all():
    raise RuntimeError(
        "Z0 contains non-finite values."
    )

print(
    f"[PASS] Z0 loaded: {len(Z0)} baseline latent vectors."
)

print(
    f"[PASS] Z0 latent dimension = {LATENT_DIM}."
)


# ==============================================================================
# 14. TRAINING-ONLY LATENT SEED SOURCE
# ==============================================================================

if "split" in Z0.columns:

    Z_seed_source = Z0[
        Z0["split"]
        .astype(str)
        .str.upper()
        ==
        "TRAIN"
    ].copy()

else:
    Z_seed_source = Z0.copy()

Z_seed_source.reset_index(
    drop=True,
    inplace=True
)

if len(Z_seed_source) == 0:
    raise RuntimeError(
        "No baseline latent seeds available."
    )

print(
    f"[PASS] Candidate seed-source latent vectors : "
    f"{len(Z_seed_source)}"
)


# ==============================================================================
# 15. LOAD SELFIES VOCABULARY
# ==============================================================================

with open(
    PHASE11_VOCAB_FILE,
    "r",
    encoding="utf-8"
) as handle:

    vocab_metadata = json.load(handle)


print("\n" + "=" * 138)
print("PHASE-11 SELFIES VOCABULARY")
print("=" * 138)

print(
    f"Top-level keys : "
    f"{list(vocab_metadata.keys())}"
)


# ==============================================================================
# 16. ROBUST VOCABULARY EXTRACTION
# ==============================================================================

def is_token_to_index_mapping(obj):
    return (
        isinstance(obj, dict)
        and len(obj) > 0
        and all(
            isinstance(key, str)
            and isinstance(
                value,
                (int, np.integer)
            )
            for key, value in obj.items()
        )
    )


def is_index_to_token_mapping(obj):
    if not isinstance(obj, dict) or len(obj) == 0:
        return False

    for key, value in obj.items():
        try:
            int(key)
        except Exception:
            return False

        if not isinstance(value, str):
            return False

    return True


def is_token_list(obj):
    return (
        isinstance(obj, list)
        and len(obj) > 0
        and all(
            isinstance(item, str)
            for item in obj
        )
    )


def recursive_vocab_search(
    obj,
    path="root"
):
    candidates = []

    if is_token_to_index_mapping(obj):
        candidates.append(
            {
                "type": "token_to_index",
                "path": path,
                "value": obj
            }
        )

    if is_index_to_token_mapping(obj):
        candidates.append(
            {
                "type": "index_to_token",
                "path": path,
                "value": obj
            }
        )

    if is_token_list(obj):
        candidates.append(
            {
                "type": "token_list",
                "path": path,
                "value": obj
            }
        )

    if isinstance(obj, dict):
        for key, value in obj.items():
            candidates.extend(
                recursive_vocab_search(
                    value,
                    f"{path}.{key}"
                )
            )

    elif isinstance(obj, list):
        for index, value in enumerate(obj):
            if isinstance(value, (dict, list)):
                candidates.extend(
                    recursive_vocab_search(
                        value,
                        f"{path}[{index}]"
                    )
                )

    return candidates


vocab_candidates = recursive_vocab_search(
    vocab_metadata
)

EXPECTED_PHASE11_VOCAB_SIZE = 58

preferred_candidates = [
    candidate
    for candidate in vocab_candidates
    if len(candidate["value"])
    ==
    EXPECTED_PHASE11_VOCAB_SIZE
]

candidate_pool = (
    preferred_candidates
    if preferred_candidates
    else vocab_candidates
)

token_to_index = None
index_to_token = None
selected_vocab_path = None
selected_vocab_type = None


# Prefer token → ID mapping.

for candidate in candidate_pool:

    if candidate["type"] == "token_to_index":

        token_to_index = {
            str(token): int(index)
            for token, index
            in candidate["value"].items()
        }

        selected_vocab_path = candidate["path"]
        selected_vocab_type = candidate["type"]

        break


# Otherwise use ID → token.

if token_to_index is None:

    for candidate in candidate_pool:

        if candidate["type"] == "index_to_token":

            index_to_token = {
                int(index): str(token)
                for index, token
                in candidate["value"].items()
            }

            token_to_index = {
                token: index
                for index, token
                in index_to_token.items()
            }

            selected_vocab_path = candidate["path"]
            selected_vocab_type = candidate["type"]

            break


# Otherwise use ordered token list.

if token_to_index is None:

    for candidate in candidate_pool:

        if candidate["type"] == "token_list":

            tokens = [
                str(token)
                for token
                in candidate["value"]
            ]

            token_to_index = {
                token: index
                for index, token
                in enumerate(tokens)
            }

            index_to_token = {
                index: token
                for index, token
                in enumerate(tokens)
            }

            selected_vocab_path = candidate["path"]
            selected_vocab_type = candidate["type"]

            break


if token_to_index is not None and index_to_token is None:

    index_to_token = {
        int(index): str(token)
        for token, index
        in token_to_index.items()
    }


if token_to_index is None or index_to_token is None:

    raise RuntimeError(
        "Could not interpret Phase-11 SELFIES vocabulary."
    )


VOCAB_SIZE = len(token_to_index)


print(
    f"[PASS] Selected vocabulary representation : "
    f"{selected_vocab_type}"
)

print(
    f"[INFO] Vocabulary JSON path              : "
    f"{selected_vocab_path}"
)

print(
    f"[INFO] Vocabulary size                   : "
    f"{VOCAB_SIZE}"
)


if VOCAB_SIZE != EXPECTED_PHASE11_VOCAB_SIZE:

    raise RuntimeError(
        f"Expected Phase-11 vocabulary size "
        f"{EXPECTED_PHASE11_VOCAB_SIZE}, "
        f"received {VOCAB_SIZE}."
    )

print(
    "[PASS] Vocabulary size matches Phase 11."
)


# ==============================================================================
# 17. SPECIAL TOKENS
# ==============================================================================

def normalize_special_token_name(token):
    return (
        str(token)
        .upper()
        .replace("<", "")
        .replace(">", "")
        .replace("[", "")
        .replace("]", "")
        .replace("_", "")
        .replace("-", "")
        .strip()
    )


def find_special_token(
    mapping,
    aliases
):
    normalized_aliases = {
        normalize_special_token_name(alias)
        for alias in aliases
    }

    for token, index in mapping.items():

        if (
            normalize_special_token_name(token)
            in normalized_aliases
        ):
            return token, int(index)

    return None, None


PAD_TOKEN, PAD_ID = find_special_token(
    token_to_index,
    ["PAD", "<PAD>", "[PAD]"]
)

BOS_TOKEN, BOS_ID = find_special_token(
    token_to_index,
    [
        "BOS",
        "<BOS>",
        "[BOS]",
        "SOS",
        "<SOS>",
        "[SOS]"
    ]
)

EOS_TOKEN, EOS_ID = find_special_token(
    token_to_index,
    ["EOS", "<EOS>", "[EOS]"]
)

UNK_TOKEN, UNK_ID = find_special_token(
    token_to_index,
    ["UNK", "<UNK>", "[UNK]"]
)


for token_name, token_id in [
    ("PAD", PAD_ID),
    ("BOS", BOS_ID),
    ("EOS", EOS_ID),
    ("UNK", UNK_ID)
]:
    if token_id is None:
        raise RuntimeError(
            f"Could not identify {token_name} token."
        )


print("\nSpecial Tokens:")
print(f"PAD : {PAD_TOKEN} → {PAD_ID}")
print(f"BOS : {BOS_TOKEN} → {BOS_ID}")
print(f"EOS : {EOS_TOKEN} → {EOS_ID}")
print(f"UNK : {UNK_TOKEN} → {UNK_ID}")

print(
    "[PASS] PAD/BOS/EOS/UNK successfully verified."
)


# ==============================================================================
# 18. MAXIMUM GENERATION LENGTH
# ==============================================================================

def recursive_find_key(
    obj,
    candidate_keys
):
    if isinstance(obj, dict):

        for key in candidate_keys:

            if key in obj:
                return obj[key]

        for value in obj.values():

            result = recursive_find_key(
                value,
                candidate_keys
            )

            if result is not None:
                return result

    return None


maximum_sequence_length = recursive_find_key(
    vocab_metadata,
    [
        "maximum_sequence_length",
        "max_sequence_length",
        "max_seq_length",
        "max_length"
    ]
)


if (
    maximum_sequence_length is None
    and PHASE11_CONFIG_FILE.exists()
):

    with open(
        PHASE11_CONFIG_FILE,
        "r",
        encoding="utf-8"
    ) as handle:

        decoder_config_metadata = json.load(
            handle
        )

    maximum_sequence_length = recursive_find_key(
        decoder_config_metadata,
        [
            "maximum_sequence_length",
            "max_sequence_length",
            "max_seq_length",
            "max_length"
        ]
    )


if maximum_sequence_length is None:

    maximum_sequence_length = (
        MAX_GENERATION_LENGTH_FALLBACK
    )


MAX_GENERATION_LENGTH = int(
    maximum_sequence_length
)

print(
    f"[INFO] Maximum generation length : "
    f"{MAX_GENERATION_LENGTH}"
)


# ==============================================================================
# 19. LOCATE THE EXACT TRAINED PHASE-11 DECODER INSTANCE
# ==============================================================================

def module_matches_checkpoint(
    module,
    checkpoint_state
):
    try:
        state = module.state_dict()

    except Exception:
        return False

    if set(state.keys()) != set(checkpoint_state.keys()):
        return False

    for key in state:

        if state[key].shape != checkpoint_state[key].shape:
            return False

    return True


decoder_template = None
decoder_template_name = None


for global_name, global_object in list(
    globals().items()
):

    if not isinstance(global_object, nn.Module):
        continue

    if module_matches_checkpoint(
        global_object,
        decoder_checkpoint["state_dict"]
    ):

        decoder_template = global_object
        decoder_template_name = global_name

        break


if decoder_template is None:

    raise RuntimeError(
        "\nPHASE-11 DECODER ARCHITECTURE IS NOT ACTIVE "
        "IN THE NOTEBOOK.\n\n"
        "Execute only the Phase-11 decoder architecture/"
        "initialization cell, then rerun Phase 14.\n"
        "Do NOT retrain Phase 11."
    )


decoder = copy.deepcopy(
    decoder_template
).to(DEVICE)

decoder.load_state_dict(
    decoder_checkpoint["state_dict"]
)

decoder.eval()

for parameter in decoder.parameters():
    parameter.requires_grad_(False)


print(
    f"\n[PASS] Actual trained Phase-11 decoder located: "
    f"{decoder_template_name}"
)


# ==============================================================================
# 20. SAVE DECODER PARAMETER REFERENCE
# ==============================================================================

decoder_parameter_reference = {
    name:
        parameter.detach().cpu().clone()

    for name, parameter
    in decoder.named_parameters()
}


# ==============================================================================
# 21. STRUCTURAL DECODER DISCOVERY
# ==============================================================================

print("\n" + "=" * 138)
print("EXACT PHASE-11 DECODER STRUCTURE")
print("=" * 138)


for name, module in decoder.named_modules():

    if name == "":
        continue

    if isinstance(
        module,
        (
            nn.Linear,
            nn.Embedding,
            nn.GRU,
            nn.LayerNorm,
            nn.Dropout,
            nn.Sequential,
            nn.SiLU,
            nn.ReLU,
            nn.Tanh
        )
    ):

        print(
            f"{name:<45} {module}"
        )


# ==============================================================================
# 22. IDENTIFY TOKEN EMBEDDING
# ==============================================================================

embedding_candidates = [
    (name, module)
    for name, module
    in decoder.named_modules()
    if isinstance(module, nn.Embedding)
    and module.num_embeddings == VOCAB_SIZE
]


if len(embedding_candidates) == 0:

    raise RuntimeError(
        "Could not identify Phase-11 token embedding."
    )


preferred_embedding = [
    item
    for item in embedding_candidates
    if item[1].embedding_dim == 64
]


if preferred_embedding:
    embedding_name, token_embedding = (
        preferred_embedding[0]
    )

else:
    embedding_name, token_embedding = (
        embedding_candidates[0]
    )


TOKEN_EMBED_DIM = int(
    token_embedding.embedding_dim
)


# ==============================================================================
# 23. IDENTIFY GRU
# ==============================================================================

gru_candidates = [
    (name, module)
    for name, module
    in decoder.named_modules()
    if isinstance(module, nn.GRU)
]


if len(gru_candidates) == 0:

    raise RuntimeError(
        "Could not identify decoder GRU."
    )


preferred_gru = [
    item
    for item in gru_candidates
    if item[1].hidden_size == 128
    and item[1].num_layers == 2
]


if preferred_gru:
    gru_name, decoder_gru = preferred_gru[0]

else:
    gru_name, decoder_gru = gru_candidates[0]


GRU_INPUT_DIM = int(
    decoder_gru.input_size
)

GRU_HIDDEN_DIM = int(
    decoder_gru.hidden_size
)

GRU_NUM_LAYERS = int(
    decoder_gru.num_layers
)


LATENT_CONTEXT_DIM = (
    GRU_INPUT_DIM
    -
    TOKEN_EMBED_DIM
)


EXPECTED_HIDDEN_PROJECTION_DIM = (
    GRU_NUM_LAYERS
    *
    GRU_HIDDEN_DIM
)


# ==============================================================================
# 24. IDENTIFY LINEAR COMPONENTS
# ==============================================================================

linear_layers = [
    (name, module)
    for name, module
    in decoder.named_modules()
    if isinstance(module, nn.Linear)
]


print("\nDetected Linear Layers:")

for name, module in linear_layers:

    print(
        f"  {name:<45} "
        f"{module.in_features} → "
        f"{module.out_features}"
    )


# latent -> hidden linear child

latent_hidden_linear_candidates = [
    (name, module)
    for name, module
    in linear_layers
    if module.in_features == LATENT_DIM
    and module.out_features
    in {
        EXPECTED_HIDDEN_PROJECTION_DIM,
        GRU_HIDDEN_DIM
    }
]


if len(latent_hidden_linear_candidates) == 0:

    raise RuntimeError(
        "Could not structurally identify latent→hidden projection."
    )


latent_hidden_linear_name, latent_hidden_linear = (
    latent_hidden_linear_candidates[0]
)


# latent -> context linear child

latent_context_linear_candidates = [
    (name, module)
    for name, module
    in linear_layers
    if module.in_features == LATENT_DIM
    and module.out_features == LATENT_CONTEXT_DIM
    and module is not latent_hidden_linear
]


if len(latent_context_linear_candidates) == 0:

    raise RuntimeError(
        "Could not structurally identify latent→context projection."
    )


latent_context_linear_name, latent_context_linear = (
    latent_context_linear_candidates[0]
)


# output projection

output_candidates = [
    (name, module)
    for name, module
    in linear_layers
    if module.out_features == VOCAB_SIZE
]


if len(output_candidates) == 0:

    raise RuntimeError(
        "Could not identify decoder vocabulary projection."
    )


preferred_output = [
    item
    for item in output_candidates
    if item[1].in_features == GRU_HIDDEN_DIM
]


if preferred_output:
    output_name, output_projection = preferred_output[-1]

else:
    output_name, output_projection = output_candidates[-1]


# ==============================================================================
# 25. CRITICAL FIX — RECOVER COMPLETE PARENT MODULES
# ==============================================================================

def recover_complete_parent_module(
    model,
    child_module_name
):
    """
    If structural discovery identifies:

        latent_to_context.0

    this function returns:

        latent_to_context

    so that all later operations inside the Sequential module
    (activation, LayerNorm, etc.) are preserved.

    If the identified module is already top-level, it is returned directly.
    """

    if "." not in child_module_name:

        return (
            child_module_name,
            model.get_submodule(
                child_module_name
            )
        )


    parent_path = child_module_name.rsplit(
        ".",
        1
    )[0]


    parent_module = model.get_submodule(
        parent_path
    )


    return (
        parent_path,
        parent_module
    )


(
    latent_hidden_parent_name,
    latent_to_hidden_full
) = recover_complete_parent_module(
    decoder,
    latent_hidden_linear_name
)


(
    latent_context_parent_name,
    latent_to_context_full
) = recover_complete_parent_module(
    decoder,
    latent_context_linear_name
)


print("\n" + "=" * 138)
print("FULL LATENT-CONDITIONING MODULES")
print("=" * 138)

print(
    f"Discovered latent-hidden child : "
    f"{latent_hidden_linear_name}"
)

print(
    f"FULL latent-hidden module       : "
    f"{latent_hidden_parent_name}"
)

print(
    latent_to_hidden_full
)


print(
    f"\nDiscovered latent-context child: "
    f"{latent_context_linear_name}"
)

print(
    f"FULL latent-context module      : "
    f"{latent_context_parent_name}"
)

print(
    latent_to_context_full
)


print(
    "\n[IMPORTANT] Generation will call the FULL parent modules above."
)


# ==============================================================================
# 26. VERIFY FULL MODULE OUTPUT SHAPES
# ==============================================================================

test_latent_tensor = torch.zeros(
    (1, LATENT_DIM),
    dtype=torch.float32,
    device=DEVICE
)


with torch.no_grad():

    full_hidden_projection_test = (
        latent_to_hidden_full(
            test_latent_tensor
        )
    )

    full_context_test = (
        latent_to_context_full(
            test_latent_tensor
        )
    )


if full_hidden_projection_test.ndim != 2:

    raise RuntimeError(
        "Unexpected full latent-hidden module output rank."
    )


if full_hidden_projection_test.shape[-1] not in {
    EXPECTED_HIDDEN_PROJECTION_DIM,
    GRU_HIDDEN_DIM
}:

    raise RuntimeError(
        "Unexpected full latent-hidden module output shape: "
        f"{tuple(full_hidden_projection_test.shape)}"
    )


if tuple(full_context_test.shape) != (
    1,
    LATENT_CONTEXT_DIM
):

    raise RuntimeError(
        "Unexpected full latent-context module output shape.\n"
        f"Expected: {(1, LATENT_CONTEXT_DIM)}\n"
        f"Received: {tuple(full_context_test.shape)}"
    )


print(
    "\n[PASS] Complete latent-hidden module output verified."
)

print(
    f"[INFO] Full hidden projection shape : "
    f"{tuple(full_hidden_projection_test.shape)}"
)

print(
    "[PASS] Complete latent-context module output verified."
)

print(
    f"[INFO] Full context shape           : "
    f"{tuple(full_context_test.shape)}"
)


# ==============================================================================
# 27. INITIALIZE GRU HIDDEN STATE USING FULL TRAINED MODULE
# ==============================================================================

def initialize_decoder_hidden(
    latent
):
    hidden_projection = latent_to_hidden_full(
        latent
    )

    batch_size = int(
        latent.shape[0]
    )


    # ----------------------------------------------------------
    # 32 → 256
    # ----------------------------------------------------------

    if (
        hidden_projection.ndim == 2
        and
        hidden_projection.shape[-1]
        ==
        GRU_NUM_LAYERS
        *
        GRU_HIDDEN_DIM
    ):

        hidden = hidden_projection.reshape(
            batch_size,
            GRU_NUM_LAYERS,
            GRU_HIDDEN_DIM
        )

        hidden = hidden.permute(
            1,
            0,
            2
        ).contiguous()


    # ----------------------------------------------------------
    # 32 → 128, repeated over GRU layers
    # ----------------------------------------------------------

    elif (
        hidden_projection.ndim == 2
        and
        hidden_projection.shape[-1]
        ==
        GRU_HIDDEN_DIM
    ):

        hidden = hidden_projection.unsqueeze(0)

        if GRU_NUM_LAYERS > 1:

            hidden = hidden.repeat(
                GRU_NUM_LAYERS,
                1,
                1
            )


    # ----------------------------------------------------------
    # Already shaped hidden state
    # ----------------------------------------------------------

    elif hidden_projection.ndim == 3:

        hidden = hidden_projection

        if (
            hidden.shape[0] == batch_size
            and
            hidden.shape[1] == GRU_NUM_LAYERS
        ):

            hidden = hidden.permute(
                1,
                0,
                2
            ).contiguous()


    else:

        raise RuntimeError(
            "Unsupported latent-hidden output shape: "
            f"{tuple(hidden_projection.shape)}"
        )


    expected_shape = (
        GRU_NUM_LAYERS,
        batch_size,
        GRU_HIDDEN_DIM
    )


    if tuple(hidden.shape) != expected_shape:

        raise RuntimeError(
            "Incorrect GRU hidden-state shape.\n"
            f"Expected: {expected_shape}\n"
            f"Received: {tuple(hidden.shape)}"
        )


    return hidden


# ==============================================================================
# 28. LATENT-CONDITIONING PATHWAY TEST
# ==============================================================================

with torch.no_grad():

    test_hidden = initialize_decoder_hidden(
        test_latent_tensor
    )

    test_context = latent_to_context_full(
        test_latent_tensor
    )


print("\n" + "=" * 138)
print("EXACT LATENT-CONDITIONING PATHWAY TEST")
print("=" * 138)

print(
    f"Latent Input               : "
    f"{tuple(test_latent_tensor.shape)}"
)

print(
    f"Full Hidden Projection     : "
    f"{tuple(full_hidden_projection_test.shape)}"
)

print(
    f"GRU Initial Hidden         : "
    f"{tuple(test_hidden.shape)}"
)

print(
    f"Full Latent Context        : "
    f"{tuple(test_context.shape)}"
)

print(
    "[PASS] Exact full latent-conditioning pathway verified."
)


# ==============================================================================
# 29. TOKEN SAMPLING
# ==============================================================================

def sample_token(
    logits,
    temperature,
    top_k
):
    logits = logits / max(
        float(temperature),
        1e-8
    )


    if (
        top_k is not None
        and top_k > 0
        and top_k < logits.shape[-1]
    ):

        top_values, top_indices = torch.topk(
            logits,
            k=top_k,
            dim=-1
        )

        probabilities = F.softmax(
            top_values,
            dim=-1
        )

        sampled_local = torch.multinomial(
            probabilities,
            num_samples=1
        )

        sampled_token = top_indices.gather(
            -1,
            sampled_local
        )

    else:

        probabilities = F.softmax(
            logits,
            dim=-1
        )

        sampled_token = torch.multinomial(
            probabilities,
            num_samples=1
        )


    return sampled_token.squeeze(-1)


# ==============================================================================
# 30. EXACT AUTOREGRESSIVE GENERATION
#
# Critical:
#
#     FULL latent_to_hidden
#     FULL latent_to_context
#
# are used here.
# ==============================================================================

SPECIAL_TOKEN_IDS = {
    token_id
    for token_id in [
        PAD_ID,
        BOS_ID,
        EOS_ID,
        UNK_ID
    ]
    if token_id is not None
}


def generate_selfies_from_latent(
    latent_vector,
    temperature=DECODER_TEMPERATURE,
    top_k=DECODER_TOP_K,
    max_length=MAX_GENERATION_LENGTH
):
    latent = torch.as_tensor(
        latent_vector,
        dtype=torch.float32,
        device=DEVICE
    )


    if latent.ndim == 1:
        latent = latent.unsqueeze(0)


    if latent.shape[-1] != LATENT_DIM:
        raise RuntimeError(
            f"Expected latent dimension {LATENT_DIM}; "
            f"received {latent.shape[-1]}."
        )


    generated_ids = []

    termination_reason = "max_length"

    eos_reached = False
    pad_reached = False


    with torch.no_grad():

        # ------------------------------------------------------
        # COMPLETE trained latent-hidden transformation
        # ------------------------------------------------------

        hidden = initialize_decoder_hidden(
            latent
        )


        # ------------------------------------------------------
        # COMPLETE trained latent-context transformation
        #
        # Includes all child layers, e.g.
        #
        # Linear → activation → LayerNorm
        # ------------------------------------------------------

        context = latent_to_context_full(
            latent
        )


        current_token = torch.full(
            (latent.shape[0],),
            int(BOS_ID),
            dtype=torch.long,
            device=DEVICE
        )


        decoding_steps = 0


        for step_index in range(max_length):

            decoding_steps = step_index + 1


            token_embedding_vector = (
                token_embedding(
                    current_token
                )
            )


            if context.ndim == 3:
                context_step = context.squeeze(1)

            else:
                context_step = context


            gru_input = torch.cat(
                [
                    token_embedding_vector,
                    context_step
                ],
                dim=-1
            )


            if gru_input.shape[-1] != GRU_INPUT_DIM:

                raise RuntimeError(
                    "Decoder GRU input dimension mismatch.\n"
                    f"Expected: {GRU_INPUT_DIM}\n"
                    f"Received: {gru_input.shape[-1]}"
                )


            gru_input = gru_input.unsqueeze(1)


            gru_output, hidden = decoder_gru(
                gru_input,
                hidden
            )


            step_hidden = gru_output[
                :,
                -1,
                :
            ]


            logits = output_projection(
                step_hidden
            )


            next_token = sample_token(
                logits,
                temperature,
                top_k
            )


            token_id = int(
                next_token[0].item()
            )


            # --------------------------------------------------
            # TRUE EOS termination
            # --------------------------------------------------

            if token_id == EOS_ID:

                eos_reached = True

                termination_reason = "eos"

                break


            # --------------------------------------------------
            # Explicit PAD termination
            # --------------------------------------------------

            if token_id == PAD_ID:

                pad_reached = True

                termination_reason = "pad"

                break


            # BOS is not part of molecular SELFIES body.

            if token_id == BOS_ID:

                current_token = next_token

                continue


            # UNK cannot form a valid SELFIES token.

            if token_id == UNK_ID:

                current_token = next_token

                continue


            generated_ids.append(
                token_id
            )

            current_token = next_token


    generated_tokens = [
        index_to_token[token_id]
        for token_id in generated_ids
        if token_id in index_to_token
        and token_id not in SPECIAL_TOKEN_IDS
    ]


    generated_selfies = "".join(
        generated_tokens
    )


    return {
        "token_ids":
            generated_ids,

        "tokens":
            generated_tokens,

        "selfies":
            generated_selfies,

        "eos_reached":
            eos_reached,

        "pad_reached":
            pad_reached,

        "max_length_reached":
            (
                termination_reason
                ==
                "max_length"
            ),

        "termination_reason":
            termination_reason,

        "decoding_steps":
            decoding_steps
    }


# ==============================================================================
# 31. TEST ACTUAL GENERATION PATH
# ==============================================================================

test_latent = (
    Z_seed_source.iloc[0][latent_columns]
    .astype(float)
    .to_numpy(dtype=np.float32)
)


test_generation = generate_selfies_from_latent(
    test_latent
)


if not isinstance(
    test_generation["selfies"],
    str
):
    raise RuntimeError(
        "Decoder generation test failed."
    )


print("\n" + "=" * 138)
print("AUTOREGRESSIVE GENERATION TEST")
print("=" * 138)

print(
    f"Generated SELFIES token count : "
    f"{len(test_generation['token_ids'])}"
)

print(
    f"Termination reason            : "
    f"{test_generation['termination_reason']}"
)

print(
    f"EOS reached                   : "
    f"{test_generation['eos_reached']}"
)

print(
    "[PASS] Exact frozen θD* generation pathway verified."
)


# ==============================================================================
# 32. LATENT PROPOSAL STORAGE
# ==============================================================================

latent_proposals = []

latent_proposal_counter = 0


def add_latent_proposal(
    latent_vector,
    generation_method,
    seed_row=None,
    second_seed_row=None,
    perturbation_sigma=None,
    interpolation_alpha=None
):
    global latent_proposal_counter

    latent_vector = np.asarray(
        latent_vector,
        dtype=np.float32
    )


    if latent_vector.shape != (LATENT_DIM,):
        raise RuntimeError(
            "Latent proposal dimension mismatch."
        )


    if not np.isfinite(latent_vector).all():
        raise RuntimeError(
            "Non-finite latent proposal generated."
        )


    latent_proposal_counter += 1


    record = {
        "latent_proposal_id":
            (
                f"CYCLE{CYCLE_INDEX:02d}_LAT"
                f"{latent_proposal_counter:06d}"
            ),

        "cycle":
            CYCLE_INDEX,

        "generation_method":
            generation_method,

        "seed_graph_id":
            "",

        "seed_drug_id":
            "",

        "second_seed_graph_id":
            "",

        "second_seed_drug_id":
            "",

        "perturbation_sigma":
            perturbation_sigma,

        "interpolation_alpha":
            interpolation_alpha
    }


    if seed_row is not None:

        record["seed_graph_id"] = str(
            seed_row.get(
                "graph_id",
                ""
            )
        )

        record["seed_drug_id"] = str(
            seed_row.get(
                "drug_id",
                ""
            )
        )


    if second_seed_row is not None:

        record["second_seed_graph_id"] = str(
            second_seed_row.get(
                "graph_id",
                ""
            )
        )

        record["second_seed_drug_id"] = str(
            second_seed_row.get(
                "drug_id",
                ""
            )
        )


    for dimension, value in enumerate(
        latent_vector
    ):

        record[
            f"z_{dimension}"
        ] = float(value)


    latent_proposals.append(record)


# ==============================================================================
# 33. DETERMINISTIC RNG
# ==============================================================================

rng = np.random.default_rng(
    RANDOM_SEED
)

all_seed_indices = np.arange(
    len(Z_seed_source)
)


# ==============================================================================
# 34. DIRECT SEED RESAMPLING
# ==============================================================================

direct_seed_count = min(
    N_DIRECT_SEEDS,
    len(Z_seed_source)
)

direct_indices = rng.choice(
    all_seed_indices,
    size=direct_seed_count,
    replace=False
)


for index in direct_indices:

    seed_row = Z_seed_source.iloc[
        int(index)
    ]

    seed_latent = (
        seed_row[latent_columns]
        .astype(float)
        .to_numpy(dtype=np.float32)
    )


    add_latent_proposal(
        latent_vector=seed_latent,
        generation_method="seed_resampling",
        seed_row=seed_row
    )


# ==============================================================================
# 35. LATENT PERTURBATION / NEIGHBORHOOD SAMPLING
# ==============================================================================

perturbation_seed_count = min(
    N_PERTURBATION_SEEDS,
    len(Z_seed_source)
)

perturbation_indices = rng.choice(
    all_seed_indices,
    size=perturbation_seed_count,
    replace=False
)


for index in perturbation_indices:

    seed_row = Z_seed_source.iloc[
        int(index)
    ]

    seed_latent = (
        seed_row[latent_columns]
        .astype(float)
        .to_numpy(dtype=np.float32)
    )


    for sigma in PERTURBATION_SIGMAS:

        for _ in range(
            PERTURBATIONS_PER_SIGMA
        ):

            noise = rng.normal(
                loc=0.0,
                scale=1.0,
                size=LATENT_DIM
            ).astype(np.float32)


            perturbed_latent = (
                seed_latent
                +
                float(sigma)
                *
                noise
            )


            add_latent_proposal(
                latent_vector=perturbed_latent,
                generation_method="latent_perturbation",
                seed_row=seed_row,
                perturbation_sigma=float(sigma)
            )


# ==============================================================================
# 36. LATENT INTERPOLATION
# ==============================================================================

if len(Z_seed_source) < 2:
    raise RuntimeError(
        "At least two seed molecules are required "
        "for interpolation."
    )


maximum_possible_pairs = (
    len(Z_seed_source)
    *
    (
        len(Z_seed_source)
        -
        1
    )
    //
    2
)


requested_pair_count = min(
    N_INTERPOLATION_PAIRS,
    maximum_possible_pairs
)


interpolation_pairs = []
used_pairs = set()


while len(interpolation_pairs) < requested_pair_count:

    pair = rng.choice(
        all_seed_indices,
        size=2,
        replace=False
    )

    a = int(pair[0])
    b = int(pair[1])

    canonical_pair = tuple(
        sorted((a, b))
    )

    if canonical_pair in used_pairs:
        continue

    used_pairs.add(
        canonical_pair
    )

    interpolation_pairs.append(
        (a, b)
    )


for index_a, index_b in interpolation_pairs:

    row_a = Z_seed_source.iloc[
        index_a
    ]

    row_b = Z_seed_source.iloc[
        index_b
    ]


    latent_a = (
        row_a[latent_columns]
        .astype(float)
        .to_numpy(dtype=np.float32)
    )

    latent_b = (
        row_b[latent_columns]
        .astype(float)
        .to_numpy(dtype=np.float32)
    )


    for alpha in INTERPOLATION_ALPHAS:

        alpha = float(alpha)

        interpolated_latent = (
            (1.0 - alpha)
            *
            latent_a
            +
            alpha
            *
            latent_b
        )


        add_latent_proposal(
            latent_vector=interpolated_latent,
            generation_method="latent_interpolation",
            seed_row=row_a,
            second_seed_row=row_b,
            interpolation_alpha=alpha
        )


# ==============================================================================
# 37. LIMITED PRIOR SAMPLING
# ==============================================================================

for _ in range(N_PRIOR_LATENTS):

    prior_latent = rng.normal(
        loc=0.0,
        scale=1.0,
        size=LATENT_DIM
    ).astype(np.float32)


    add_latent_proposal(
        latent_vector=prior_latent,
        generation_method="exploratory_prior"
    )


# ==============================================================================
# 38. CREATE LATENT PROPOSAL TABLE
# ==============================================================================

latent_proposals_df = pd.DataFrame(
    latent_proposals
)

latent_matrix = (
    latent_proposals_df[latent_columns]
    .astype(float)
    .to_numpy(dtype=np.float32)
)

if not np.isfinite(latent_matrix).all():
    raise RuntimeError(
        "Latent proposal matrix contains non-finite values."
    )


print("\n" + "=" * 138)
print("LATENT PROPOSAL CONSTRUCTION")
print("=" * 138)

for method in [
    "seed_resampling",
    "latent_perturbation",
    "latent_interpolation",
    "exploratory_prior"
]:

    count = int(
        (
            latent_proposals_df[
                "generation_method"
            ]
            ==
            method
        ).sum()
    )

    print(
        f"{method:<30}: {count}"
    )


print(
    f"\nTotal latent proposals            : "
    f"{len(latent_proposals_df)}"
)

print(
    "[PASS] All latent proposals finite."
)


# ==============================================================================
# 39. LATENT DISTANCE METRICS
# ==============================================================================

seed_lookup = {
    str(row["graph_id"]):
        (
            row[latent_columns]
            .astype(float)
            .to_numpy(dtype=np.float32)
        )

    for _, row
    in Z_seed_source.iterrows()
}


baseline_latent_matrix = (
    Z_seed_source[latent_columns]
    .astype(float)
    .to_numpy(dtype=np.float32)
)


def calculate_latent_distances(
    row
):
    z = (
        row[latent_columns]
        .astype(float)
        .to_numpy(dtype=np.float32)
    )


    distance_seed = np.nan
    distance_second_seed = np.nan
    nearest_baseline_distance = np.nan


    seed_graph_id = str(
        row.get(
            "seed_graph_id",
            ""
        )
    )


    second_seed_graph_id = str(
        row.get(
            "second_seed_graph_id",
            ""
        )
    )


    if seed_graph_id in seed_lookup:

        distance_seed = float(
            np.linalg.norm(
                z
                -
                seed_lookup[
                    seed_graph_id
                ]
            )
        )


    if second_seed_graph_id in seed_lookup:

        distance_second_seed = float(
            np.linalg.norm(
                z
                -
                seed_lookup[
                    second_seed_graph_id
                ]
            )
        )


    if row[
        "generation_method"
    ] == "exploratory_prior":

        distances = np.linalg.norm(
            baseline_latent_matrix
            -
            z[None, :],
            axis=1
        )

        nearest_baseline_distance = float(
            distances.min()
        )


    return pd.Series(
        {
            "latent_distance_to_seed":
                distance_seed,

            "latent_distance_to_second_seed":
                distance_second_seed,

            "nearest_baseline_latent_distance":
                nearest_baseline_distance
        }
    )


distance_df = latent_proposals_df.apply(
    calculate_latent_distances,
    axis=1
)


latent_proposals_df = pd.concat(
    [
        latent_proposals_df,
        distance_df
    ],
    axis=1
)


latent_proposals_df.to_csv(
    PHASE14_LATENT_PROPOSALS_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 40. GENERATE RAW MOLECULAR PROPOSALS
# ==============================================================================

print("\n" + "=" * 138)
print("STOCHASTIC DECODER GENERATION")
print("=" * 138)


raw_candidate_records = []

raw_candidate_counter = 0

generation_start_time = time.time()


for proposal_index, proposal_row in (
    latent_proposals_df.iterrows()
):

    latent_vector = (
        proposal_row[latent_columns]
        .astype(float)
        .to_numpy(dtype=np.float32)
    )


    for decoder_sample_index in range(
        1,
        DECODER_SAMPLES_PER_LATENT + 1
    ):

        raw_candidate_counter += 1


        generation = (
            generate_selfies_from_latent(
                latent_vector=latent_vector,
                temperature=DECODER_TEMPERATURE,
                top_k=DECODER_TOP_K,
                max_length=MAX_GENERATION_LENGTH
            )
        )


        raw_candidate_records.append(
            {
                "raw_candidate_id":
                    (
                        f"CYCLE{CYCLE_INDEX:02d}_RAW"
                        f"{raw_candidate_counter:06d}"
                    ),

                "cycle":
                    CYCLE_INDEX,

                "latent_proposal_id":
                    proposal_row[
                        "latent_proposal_id"
                    ],

                "generation_method":
                    proposal_row[
                        "generation_method"
                    ],

                "seed_graph_id":
                    proposal_row[
                        "seed_graph_id"
                    ],

                "seed_drug_id":
                    proposal_row[
                        "seed_drug_id"
                    ],

                "second_seed_graph_id":
                    proposal_row[
                        "second_seed_graph_id"
                    ],

                "second_seed_drug_id":
                    proposal_row[
                        "second_seed_drug_id"
                    ],

                "perturbation_sigma":
                    proposal_row[
                        "perturbation_sigma"
                    ],

                "interpolation_alpha":
                    proposal_row[
                        "interpolation_alpha"
                    ],

                "latent_distance_to_seed":
                    proposal_row[
                        "latent_distance_to_seed"
                    ],

                "latent_distance_to_second_seed":
                    proposal_row[
                        "latent_distance_to_second_seed"
                    ],

                "nearest_baseline_latent_distance":
                    proposal_row[
                        "nearest_baseline_latent_distance"
                    ],

                "decoder_sample_index":
                    decoder_sample_index,

                "temperature":
                    DECODER_TEMPERATURE,

                "top_k":
                    DECODER_TOP_K,

                "generated_token_count":
                    len(
                        generation[
                            "token_ids"
                        ]
                    ),

                "decoding_steps":
                    generation[
                        "decoding_steps"
                    ],

                "termination_reason":
                    generation[
                        "termination_reason"
                    ],

                "eos_reached":
                    generation[
                        "eos_reached"
                    ],

                "pad_reached":
                    generation[
                        "pad_reached"
                    ],

                "max_length_reached":
                    generation[
                        "max_length_reached"
                    ],

                "generated_selfies":
                    generation[
                        "selfies"
                    ]
            }
        )


    if (
        proposal_index
        +
        1
    ) % 50 == 0:

        print(
            f"[GENERATION] "
            f"{proposal_index + 1:04d}/"
            f"{len(latent_proposals_df)} "
            f"latent proposals processed"
        )


generation_duration_seconds = (
    time.time()
    -
    generation_start_time
)


raw_candidates_df = pd.DataFrame(
    raw_candidate_records
)


raw_candidates_df.to_csv(
    PHASE14_RAW_FILE,
    index=False,
    encoding="utf-8"
)


print(
    f"\n[PASS] Raw decoder outputs generated : "
    f"{len(raw_candidates_df)}"
)


# ==============================================================================
# 41. TERMINATION DIAGNOSTICS
# ==============================================================================

termination_counts = (
    raw_candidates_df[
        "termination_reason"
    ]
    .value_counts(
        dropna=False
    )
    .to_dict()
)


print("\nDecoder Termination:")

for reason, count in termination_counts.items():

    print(
        f"  {reason:<15}: {count}"
    )


# ==============================================================================
# 42. CHEMICAL STANDARDIZATION
# ==============================================================================

uncharger = rdMolStandardize.Uncharger()


def standardize_candidate_molecule(
    mol
):
    working = Chem.Mol(mol)

    working = rdMolStandardize.Cleanup(
        working
    )

    working = rdMolStandardize.FragmentParent(
        working
    )

    working = uncharger.uncharge(
        working
    )

    Chem.SanitizeMol(
        working
    )

    working.UpdatePropertyCache(
        strict=True
    )

    canonical_smiles = Chem.MolToSmiles(
        working,
        canonical=True,
        isomericSmiles=True
    )


    return working, canonical_smiles


# ==============================================================================
# 43. CHEMICAL QC
# ==============================================================================

print("\n" + "=" * 138)
print("CHEMICAL VALIDATION")
print("=" * 138)


validated_records = []


for _, row in raw_candidates_df.iterrows():

    record = row.to_dict()


    record.update(
        {
            "selfies_decode_success":
                False,

            "decoded_smiles":
                "",

            "rdkit_parse_success":
                False,

            "sanitize_success":
                False,

            "standardization_success":
                False,

            "valence_check_success":
                False,

            "canonical_smiles":
                "",

            "heavy_atom_count":
                np.nan,

            "model_feasible":
                False,

            "is_valid":
                False,

            "failure_reason":
                ""
        }
    )


    generated_selfies = str(
        row["generated_selfies"]
    )


    # ----------------------------------------------------------
    # Empty SELFIES
    # ----------------------------------------------------------

    if not generated_selfies:

        record[
            "failure_reason"
        ] = "empty_selfies"

        validated_records.append(
            record
        )

        continue


    # ----------------------------------------------------------
    # SELFIES → SMILES
    # ----------------------------------------------------------

    try:

        decoded_smiles = sf.decoder(
            generated_selfies
        )

        record[
            "decoded_smiles"
        ] = decoded_smiles

        record[
            "selfies_decode_success"
        ] = True


    except Exception:

        record[
            "failure_reason"
        ] = "selfies_decode_failure"

        validated_records.append(
            record
        )

        continue


    # ----------------------------------------------------------
    # RDKit parse
    # ----------------------------------------------------------

    try:

        mol = Chem.MolFromSmiles(
            decoded_smiles,
            sanitize=False
        )

    except Exception:

        mol = None


    if mol is None:

        record[
            "failure_reason"
        ] = "rdkit_parse_failure"

        validated_records.append(
            record
        )

        continue


    record[
        "rdkit_parse_success"
    ] = True


    # ----------------------------------------------------------
    # Sanitization
    # ----------------------------------------------------------

    try:

        Chem.SanitizeMol(mol)

        record[
            "sanitize_success"
        ] = True


    except Exception:

        record[
            "failure_reason"
        ] = "sanitize_failure"

        validated_records.append(
            record
        )

        continue


    # ----------------------------------------------------------
    # Standardization
    # ----------------------------------------------------------

    try:

        standardized_mol, canonical_smiles = (
            standardize_candidate_molecule(
                mol
            )
        )

        record[
            "standardization_success"
        ] = True

        record[
            "canonical_smiles"
        ] = canonical_smiles


    except Exception:

        record[
            "failure_reason"
        ] = "standardization_failure"

        validated_records.append(
            record
        )

        continue


    # ----------------------------------------------------------
    # Valence/property cache
    # ----------------------------------------------------------

    try:

        standardized_mol.UpdatePropertyCache(
            strict=True
        )

        Chem.SanitizeMol(
            standardized_mol
        )

        record[
            "valence_check_success"
        ] = True


    except Exception:

        record[
            "failure_reason"
        ] = "valence_failure"

        validated_records.append(
            record
        )

        continue


    # ----------------------------------------------------------
    # Heavy atom model feasibility
    # ----------------------------------------------------------

    heavy_atom_count = int(
        standardized_mol.GetNumHeavyAtoms()
    )

    record[
        "heavy_atom_count"
    ] = heavy_atom_count


    model_feasible = (
        MIN_HEAVY_ATOMS
        <=
        heavy_atom_count
        <=
        MAX_HEAVY_ATOMS
    )


    record[
        "model_feasible"
    ] = bool(
        model_feasible
    )


    if not model_feasible:

        record[
            "failure_reason"
        ] = "outside_model_feasibility"

        validated_records.append(
            record
        )

        continue


    record[
        "is_valid"
    ] = True

    record[
        "failure_reason"
    ] = ""


    validated_records.append(
        record
    )


validated_all_df = pd.DataFrame(
    validated_records
)


valid_candidates_df = (
    validated_all_df[
        validated_all_df[
            "is_valid"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


valid_candidates_df.to_csv(
    PHASE14_VALIDATED_FILE,
    index=False,
    encoding="utf-8"
)


print(
    f"Raw generated candidates       : "
    f"{len(validated_all_df)}"
)

print(
    f"Chemically valid candidates    : "
    f"{len(valid_candidates_df)}"
)


# ==============================================================================
# 44. GLOBAL CANONICAL DEDUPLICATION
# ==============================================================================

if len(valid_candidates_df) > 0:

    valid_candidates_df[
        "is_unique"
    ] = (
        ~valid_candidates_df.duplicated(
            subset=[
                "canonical_smiles"
            ],
            keep="first"
        )
    )


    unique_candidates_df = (
        valid_candidates_df[
            valid_candidates_df[
                "is_unique"
            ]
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

else:

    valid_candidates_df[
        "is_unique"
    ] = False

    unique_candidates_df = (
        valid_candidates_df.copy()
    )


# ==============================================================================
# 45. LOAD BASELINE STANDARDIZED CHEMISTRY
# ==============================================================================

baseline_chem_df = pd.read_csv(
    BASELINE_CHEM_FILE,
    dtype=str,
    keep_default_na=False
)


baseline_smiles_candidates = [
    "standardized_smiles",
    "canonical_smiles",
    "smiles_standardized",
    "standardized_canonical_smiles",
    "canonical_standardized_smiles"
]


BASELINE_SMILES_COLUMN = None


for column in baseline_smiles_candidates:

    if column in baseline_chem_df.columns:

        BASELINE_SMILES_COLUMN = column
        break


if BASELINE_SMILES_COLUMN is None:

    raise RuntimeError(
        "Could not identify standardized SMILES column "
        "in baseline chemistry table.\n"
        f"Columns: {list(baseline_chem_df.columns)}"
    )


baseline_canonical_smiles = set()

baseline_molecules = []


for smiles in baseline_chem_df[
    BASELINE_SMILES_COLUMN
].astype(str):

    smiles = smiles.strip()

    if not smiles:
        continue


    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        continue


    canonical = Chem.MolToSmiles(
        mol,
        canonical=True,
        isomericSmiles=True
    )


    baseline_canonical_smiles.add(
        canonical
    )

    baseline_molecules.append(
        mol
    )


print(
    f"\n[PASS] Baseline novelty-reference molecules : "
    f"{len(baseline_canonical_smiles)}"
)


# ==============================================================================
# 46. BASELINE-RELATIVE NOVELTY
# ==============================================================================

if len(unique_candidates_df) > 0:

    unique_candidates_df[
        "novel_vs_baseline"
    ] = (
        ~unique_candidates_df[
            "canonical_smiles"
        ].isin(
            baseline_canonical_smiles
        )
    )


    novel_candidates_df = (
        unique_candidates_df[
            unique_candidates_df[
                "novel_vs_baseline"
            ]
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

else:

    unique_candidates_df[
        "novel_vs_baseline"
    ] = False

    novel_candidates_df = (
        unique_candidates_df.copy()
    )


# ==============================================================================
# 47. STABLE CANDIDATE IDs
# ==============================================================================

unique_candidates_df[
    "candidate_id"
] = [
    (
        f"CYCLE{CYCLE_INDEX:02d}_CAND"
        f"{index + 1:06d}"
    )

    for index
    in range(
        len(unique_candidates_df)
    )
]


candidate_id_lookup = dict(
    zip(
        unique_candidates_df[
            "canonical_smiles"
        ],
        unique_candidates_df[
            "candidate_id"
        ]
    )
)


if len(novel_candidates_df) > 0:

    novel_candidates_df[
        "candidate_id"
    ] = novel_candidates_df[
        "canonical_smiles"
    ].map(
        candidate_id_lookup
    )


# ==============================================================================
# 48. FINGERPRINT HELPERS
# ==============================================================================

def molecule_fingerprint(mol):

    return AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius=FP_RADIUS,
        nBits=FP_BITS
    )


baseline_fingerprints = [
    molecule_fingerprint(mol)
    for mol in baseline_molecules
]


def nearest_baseline_similarity(
    canonical_smiles
):
    mol = Chem.MolFromSmiles(
        canonical_smiles
    )

    if mol is None:
        return np.nan


    fp = molecule_fingerprint(
        mol
    )


    if len(baseline_fingerprints) == 0:
        return np.nan


    similarities = (
        DataStructs.BulkTanimotoSimilarity(
            fp,
            baseline_fingerprints
        )
    )


    return float(max(similarities))


# ==============================================================================
# 49. NEAREST BASELINE SIMILARITY
# ==============================================================================

if len(unique_candidates_df) > 0:

    unique_candidates_df[
        "nearest_baseline_similarity"
    ] = unique_candidates_df[
        "canonical_smiles"
    ].apply(
        nearest_baseline_similarity
    )


if len(novel_candidates_df) > 0:

    novel_candidates_df[
        "nearest_baseline_similarity"
    ] = novel_candidates_df[
        "canonical_smiles"
    ].apply(
        nearest_baseline_similarity
    )


# ==============================================================================
# 50. INTERNAL DIVERSITY
# ==============================================================================

def mean_pairwise_tanimoto_diversity(
    smiles_values
):
    fingerprints = []


    for smiles in smiles_values:

        mol = Chem.MolFromSmiles(
            smiles
        )

        if mol is None:
            continue

        fingerprints.append(
            molecule_fingerprint(
                mol
            )
        )


    if len(fingerprints) < 2:
        return np.nan


    similarities = []


    for index in range(
        1,
        len(fingerprints)
    ):

        similarities.extend(
            DataStructs.BulkTanimotoSimilarity(
                fingerprints[index],
                fingerprints[:index]
            )
        )


    if len(similarities) == 0:
        return np.nan


    return float(
        1.0
        -
        np.mean(similarities)
    )


overall_unique_diversity = (
    mean_pairwise_tanimoto_diversity(
        unique_candidates_df[
            "canonical_smiles"
        ].tolist()
    )
    if len(unique_candidates_df) > 1
    else np.nan
)


overall_novel_diversity = (
    mean_pairwise_tanimoto_diversity(
        novel_candidates_df[
            "canonical_smiles"
        ].tolist()
    )
    if len(novel_candidates_df) > 1
    else np.nan
)


# ==============================================================================
# 51. SAVE UNIQUE AND NOVEL POOLS
# ==============================================================================

unique_candidates_df.to_csv(
    PHASE14_UNIQUE_FILE,
    index=False,
    encoding="utf-8"
)


novel_candidates_df.to_csv(
    PHASE14_NOVEL_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 52. PROVENANCE TABLE
# ==============================================================================

provenance_df = validated_all_df.copy()


if len(provenance_df) > 0:

    provenance_df[
        "candidate_id"
    ] = provenance_df[
        "canonical_smiles"
    ].map(
        candidate_id_lookup
    )


    provenance_df[
        "novel_vs_baseline"
    ] = provenance_df[
        "canonical_smiles"
    ].apply(
        lambda smiles:
            bool(
                smiles
                and
                smiles
                not in baseline_canonical_smiles
            )
    )


provenance_df.to_csv(
    PHASE14_PROVENANCE_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 53. METHOD-SPECIFIC SUMMARY
#
# Important:
# method-level uniqueness is calculated independently within each method.
# ==============================================================================

method_summary_records = []


generation_methods = sorted(
    validated_all_df[
        "generation_method"
    ].unique()
)


for method in generation_methods:

    raw_method = validated_all_df[
        validated_all_df[
            "generation_method"
        ]
        ==
        method
    ]


    valid_method = raw_method[
        raw_method[
            "is_valid"
        ]
    ].copy()


    method_unique = (
        valid_method.drop_duplicates(
            subset=[
                "canonical_smiles"
            ],
            keep="first"
        )
        .copy()
    )


    method_unique[
        "novel_vs_baseline"
    ] = (
        ~method_unique[
            "canonical_smiles"
        ].isin(
            baseline_canonical_smiles
        )
    )


    method_novel = method_unique[
        method_unique[
            "novel_vs_baseline"
        ]
    ]


    attempted = len(raw_method)
    valid_count_method = len(valid_method)
    unique_count_method = len(method_unique)
    novel_count_method = len(method_novel)


    diversity = (
        mean_pairwise_tanimoto_diversity(
            method_unique[
                "canonical_smiles"
            ].tolist()
        )
        if unique_count_method > 1
        else np.nan
    )


    method_summary_records.append(
        {
            "generation_method":
                method,

            "attempted":
                attempted,

            "valid":
                valid_count_method,

            "unique_within_method":
                unique_count_method,

            "novel_vs_baseline":
                novel_count_method,

            "validity_rate":
                (
                    valid_count_method
                    /
                    max(attempted, 1)
                ),

            "uniqueness_rate_among_valid":
                (
                    unique_count_method
                    /
                    max(valid_count_method, 1)
                ),

            "novelty_rate_among_unique":
                (
                    novel_count_method
                    /
                    max(unique_count_method, 1)
                ),

            "mean_pairwise_diversity":
                diversity
        }
    )


method_summary_df = pd.DataFrame(
    method_summary_records
)


method_summary_df.to_csv(
    PHASE14_METHOD_SUMMARY_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 54. QC COUNTS
# ==============================================================================

raw_count = len(
    validated_all_df
)

selfies_success_count = int(
    validated_all_df[
        "selfies_decode_success"
    ].sum()
)

parse_success_count = int(
    validated_all_df[
        "rdkit_parse_success"
    ].sum()
)

sanitize_success_count = int(
    validated_all_df[
        "sanitize_success"
    ].sum()
)

standardization_success_count = int(
    validated_all_df[
        "standardization_success"
    ].sum()
)

valence_success_count = int(
    validated_all_df[
        "valence_check_success"
    ].sum()
)

model_feasible_count = int(
    validated_all_df[
        "model_feasible"
    ].sum()
)

valid_count = len(
    valid_candidates_df
)

unique_count = len(
    unique_candidates_df
)

novel_count = len(
    novel_candidates_df
)


eos_count = int(
    (
        raw_candidates_df[
            "termination_reason"
        ]
        ==
        "eos"
    ).sum()
)

pad_count = int(
    (
        raw_candidates_df[
            "termination_reason"
        ]
        ==
        "pad"
    ).sum()
)

max_length_count = int(
    (
        raw_candidates_df[
            "termination_reason"
        ]
        ==
        "max_length"
    ).sum()
)


# ==============================================================================
# 55. VERIFY DECODER PARAMETERS DID NOT CHANGE
# ==============================================================================

decoder_parameters_unchanged = True


for name, parameter in decoder.named_parameters():

    if not torch.equal(
        parameter.detach().cpu(),
        decoder_parameter_reference[name]
    ):

        decoder_parameters_unchanged = False

        break


if not decoder_parameters_unchanged:

    raise RuntimeError(
        "θD* parameter values changed during Phase 14."
    )


# ==============================================================================
# 56. VERIFY COMPLETE K0 FILE INTEGRITY
# ==============================================================================

encoder_hash_after = sha256_file(
    PHASE12_ENCODER_FILE
)

decoder_hash_after = sha256_file(
    PHASE12_DECODER_FILE
)

predictor_hash_after = sha256_file(
    PHASE12_PREDICTOR_FILE
)

encoder_fisher_hash_after = sha256_file(
    PHASE13_ENCODER_FISHER_FILE
)

predictor_fisher_hash_after = sha256_file(
    PHASE13_PREDICTOR_FISHER_FILE
)

phase13_reference_hash_after = sha256_file(
    PHASE13_REFERENCE_FILE
)


encoder_unchanged = (
    encoder_hash_before
    ==
    encoder_hash_after
)

decoder_checkpoint_unchanged = (
    decoder_hash_before
    ==
    decoder_hash_after
)

predictor_unchanged = (
    predictor_hash_before
    ==
    predictor_hash_after
)

encoder_fisher_unchanged = (
    encoder_fisher_hash_before
    ==
    encoder_fisher_hash_after
)

predictor_fisher_unchanged = (
    predictor_fisher_hash_before
    ==
    predictor_fisher_hash_after
)

phase13_reference_unchanged = (
    phase13_reference_hash_before
    ==
    phase13_reference_hash_after
)


if not all(
    [
        encoder_unchanged,
        decoder_checkpoint_unchanged,
        predictor_unchanged,
        encoder_fisher_unchanged,
        predictor_fisher_unchanged,
        phase13_reference_unchanged,
        decoder_parameters_unchanged
    ]
):

    raise RuntimeError(
        "K0 integrity failure after candidate generation."
    )


print("\n" + "=" * 138)
print("BASELINE K0 INTEGRITY")
print("=" * 138)

print(
    f"θE* checkpoint unchanged        : "
    f"{encoder_unchanged}"
)

print(
    f"θD* checkpoint unchanged        : "
    f"{decoder_checkpoint_unchanged}"
)

print(
    f"θD* parameters unchanged        : "
    f"{decoder_parameters_unchanged}"
)

print(
    f"θP* checkpoint unchanged        : "
    f"{predictor_unchanged}"
)

print(
    f"F_E unchanged                   : "
    f"{encoder_fisher_unchanged}"
)

print(
    f"F_P unchanged                   : "
    f"{predictor_fisher_unchanged}"
)

print(
    f"Phase-13 reference unchanged    : "
    f"{phase13_reference_unchanged}"
)


# ==============================================================================
# 57. PHASE-14 CONFIG FILE
# ==============================================================================

phase14_config = {
    "phase":
        "Phase 14 — Latent-Space Candidate Generation",

    "implementation":
        "corrected full latent-conditioning decoder pathway",

    "cycle":
        CYCLE_INDEX,

    "random_seed":
        RANDOM_SEED,

    "latent_dimension":
        LATENT_DIM,

    "baseline_state":
        "K0",

    "seed_source":
        "training subset of baseline Z0",

    "generation_methods":
        [
            "seed_resampling",
            "latent_perturbation",
            "latent_interpolation",
            "exploratory_prior"
        ],

    "direct_seed_latents":
        N_DIRECT_SEEDS,

    "perturbation_seed_count":
        N_PERTURBATION_SEEDS,

    "perturbation_sigmas":
        PERTURBATION_SIGMAS,

    "perturbations_per_sigma":
        PERTURBATIONS_PER_SIGMA,

    "interpolation_pairs":
        requested_pair_count,

    "interpolation_alphas":
        INTERPOLATION_ALPHAS,

    "prior_latents":
        N_PRIOR_LATENTS,

    "decoder_samples_per_latent":
        DECODER_SAMPLES_PER_LATENT,

    "decoder_temperature":
        DECODER_TEMPERATURE,

    "decoder_top_k":
        DECODER_TOP_K,

    "maximum_generation_length":
        MAX_GENERATION_LENGTH,

    "latent_hidden_child_identified":
        latent_hidden_linear_name,

    "latent_hidden_full_module_used":
        latent_hidden_parent_name,

    "latent_context_child_identified":
        latent_context_linear_name,

    "latent_context_full_module_used":
        latent_context_parent_name,

    "gru":
        gru_name,

    "output_projection":
        output_name,

    "minimum_heavy_atoms":
        MIN_HEAVY_ATOMS,

    "maximum_heavy_atoms":
        MAX_HEAVY_ATOMS,

    "novelty_definition":
        (
            "canonical standardized SMILES absent "
            "from baseline DrugBank corpus"
        ),

    "encoder_updated":
        False,

    "decoder_updated":
        False,

    "predictor_updated":
        False,

    "fisher_updated":
        False,

    "ewc_applied":
        False,

    "candidate_biological_evaluation_performed":
        False,

    "closed_loop_model_adaptation":
        False
}


with open(
    PHASE14_CONFIG_FILE,
    "w",
    encoding="utf-8"
) as handle:

    json.dump(
        phase14_config,
        handle,
        indent=4,
        ensure_ascii=False
    )


# ==============================================================================
# 58. COMPLETION CHECKS
# ==============================================================================

completion_checks = {
    "Phase-11 decoder checkpoint loaded":
        True,

    "Decoder exactly matches θD*":
        decoder_matches_theta_star,

    "Z0 loaded successfully":
        len(Z0) > 0,

    "Latent dimension equals 32":
        Z0_values.shape[1]
        ==
        LATENT_DIM,

    "All baseline latents finite":
        bool(
            np.isfinite(
                Z0_values
            ).all()
        ),

    "Training-derived seed corpus available":
        len(Z_seed_source) > 0,

    "Random seed fixed":
        RANDOM_SEED == 42,

    "Exact Phase-11 decoder module used":
        True,

    "Complete latent-hidden parent module used":
        latent_to_hidden_full is not None,

    "Complete latent-context parent module used":
        latent_to_context_full is not None,

    "Full latent-hidden pathway shape verified":
        True,

    "Full latent-context pathway shape verified":
        True,

    "Seed resampling generated":
        bool(
            (
                latent_proposals_df[
                    "generation_method"
                ]
                ==
                "seed_resampling"
            ).any()
        ),

    "Latent perturbation generated":
        bool(
            (
                latent_proposals_df[
                    "generation_method"
                ]
                ==
                "latent_perturbation"
            ).any()
        ),

    "Latent interpolation generated":
        bool(
            (
                latent_proposals_df[
                    "generation_method"
                ]
                ==
                "latent_interpolation"
            ).any()
        ),

    "Exploratory prior generated":
        bool(
            (
                latent_proposals_df[
                    "generation_method"
                ]
                ==
                "exploratory_prior"
            ).any()
        ),

    "All latent proposals finite":
        bool(
            np.isfinite(
                latent_matrix
            ).all()
        ),

    "Raw molecular proposals generated":
        raw_count > 0,

    "Explicit EOS bookkeeping performed":
        "eos_reached"
        in raw_candidates_df.columns,

    "Explicit termination reason recorded":
        "termination_reason"
        in raw_candidates_df.columns,

    "SELFIES decoding checked":
        "selfies_decode_success"
        in validated_all_df.columns,

    "RDKit parsing checked":
        "rdkit_parse_success"
        in validated_all_df.columns,

    "Sanitization checked":
        "sanitize_success"
        in validated_all_df.columns,

    "Standardization checked":
        "standardization_success"
        in validated_all_df.columns,

    "Valence checked":
        "valence_check_success"
        in validated_all_df.columns,

    "Heavy atom feasibility checked":
        "model_feasible"
        in validated_all_df.columns,

    "At least one valid candidate produced":
        valid_count > 0,

    "Canonical deduplication performed":
        "is_unique"
        in valid_candidates_df.columns,

    "At least one unique candidate produced":
        unique_count > 0,

    "Baseline-relative novelty calculated":
        "novel_vs_baseline"
        in unique_candidates_df.columns,

    "At least one novel candidate produced":
        novel_count > 0,

    "Candidate diversity calculated":
        (
            unique_count < 2
            or
            np.isfinite(
                overall_unique_diversity
            )
        ),

    "Candidate provenance saved":
        PHASE14_PROVENANCE_FILE.exists(),

    "Generation-method statistics saved":
        PHASE14_METHOD_SUMMARY_FILE.exists(),

    "θE* unchanged":
        encoder_unchanged,

    "θD* checkpoint unchanged":
        decoder_checkpoint_unchanged,

    "θD* parameters unchanged":
        decoder_parameters_unchanged,

    "θP* unchanged":
        predictor_unchanged,

    "F_E unchanged":
        encoder_fisher_unchanged,

    "F_P unchanged":
        predictor_fisher_unchanged,

    "Phase-13 EWC reference unchanged":
        phase13_reference_unchanged,

    "No optimizer created":
        True,

    "No optimizer update":
        True,

    "No EWC applied":
        True,

    "No biological candidate ranking":
        True,

    "No closed-loop model adaptation":
        True
}


# ==============================================================================
# 59. QC SUMMARY
# ==============================================================================

qc_summary = {
    "phase":
        "Phase 14 — Latent-Space Candidate Generation",

    "created_at":
        datetime.now().isoformat(),

    "status":
        "CYCLE_1_CANDIDATE_POOL_CONSTRUCTED",

    "implementation":
        "full Phase-11 latent-conditioning pathway",

    "cycle":
        CYCLE_INDEX,

    "baseline_state":
        "K0",

    "seed_source_molecules":
        len(Z_seed_source),

    "latent_proposals":
        len(latent_proposals_df),

    "raw_generated_sequences":
        raw_count,

    "termination_eos":
        eos_count,

    "termination_pad":
        pad_count,

    "termination_max_length":
        max_length_count,

    "selfies_decode_success":
        selfies_success_count,

    "rdkit_parse_success":
        parse_success_count,

    "sanitize_success":
        sanitize_success_count,

    "standardization_success":
        standardization_success_count,

    "valence_success":
        valence_success_count,

    "model_feasible":
        model_feasible_count,

    "valid_candidates":
        valid_count,

    "unique_valid_candidates":
        unique_count,

    "novel_candidates_vs_baseline":
        novel_count,

    "selfies_decode_rate":
        (
            selfies_success_count
            /
            max(raw_count, 1)
        ),

    "rdkit_parse_rate":
        (
            parse_success_count
            /
            max(raw_count, 1)
        ),

    "sanitization_rate":
        (
            sanitize_success_count
            /
            max(raw_count, 1)
        ),

    "standardization_rate":
        (
            standardization_success_count
            /
            max(raw_count, 1)
        ),

    "model_feasibility_rate":
        (
            model_feasible_count
            /
            max(raw_count, 1)
        ),

    "validity_rate":
        (
            valid_count
            /
            max(raw_count, 1)
        ),

    "uniqueness_rate_among_valid":
        (
            unique_count
            /
            max(valid_count, 1)
        ),

    "novelty_rate_among_unique":
        (
            novel_count
            /
            max(unique_count, 1)
        ),

    "unique_candidate_diversity":
        overall_unique_diversity,

    "novel_candidate_diversity":
        overall_novel_diversity,

    "generation_duration_seconds":
        generation_duration_seconds,

    "decoder_updated":
        False,

    "encoder_updated":
        False,

    "predictor_updated":
        False,

    "fisher_updated":
        False,

    "ewc_applied":
        False,

    "candidate_biological_evaluation_performed":
        False,

    "completion_checks":
        completion_checks
}


with open(
    PHASE14_QC_JSON_FILE,
    "w",
    encoding="utf-8"
) as handle:

    json.dump(
        qc_summary,
        handle,
        indent=4,
        ensure_ascii=False,
        default=str
    )


qc_rows = []


for key, value in qc_summary.items():

    if key == "completion_checks":

        for check_name, status in value.items():

            qc_rows.append(
                {
                    "metric":
                        f"completion.{check_name}",

                    "value":
                        status
                }
            )

    else:

        qc_rows.append(
            {
                "metric":
                    key,

                "value":
                    value
            }
        )


pd.DataFrame(
    qc_rows
).to_csv(
    PHASE14_QC_CSV_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 60. GENERATION METHOD SUMMARY
# ==============================================================================

print("\n" + "=" * 138)
print("GENERATION METHOD SUMMARY")
print("=" * 138)

print(
    method_summary_df.to_string(
        index=False
    )
)


# ==============================================================================
# 61. CHEMICAL QC SUMMARY
# ==============================================================================

print("\n" + "=" * 138)
print("PHASE 14 CHEMICAL QC")
print("=" * 138)

print(
    f"Raw Generated Sequences           : {raw_count}"
)

print(
    f"EOS Termination                   : "
    f"{eos_count} "
    f"({eos_count / max(raw_count, 1):.2%})"
)

print(
    f"PAD Termination                   : "
    f"{pad_count} "
    f"({pad_count / max(raw_count, 1):.2%})"
)

print(
    f"Max-Length Termination            : "
    f"{max_length_count} "
    f"({max_length_count / max(raw_count, 1):.2%})"
)

print(
    f"SELFIES Decode Success            : "
    f"{selfies_success_count} "
    f"({selfies_success_count / max(raw_count, 1):.2%})"
)

print(
    f"RDKit Parse Success               : "
    f"{parse_success_count} "
    f"({parse_success_count / max(raw_count, 1):.2%})"
)

print(
    f"Sanitization Success              : "
    f"{sanitize_success_count} "
    f"({sanitize_success_count / max(raw_count, 1):.2%})"
)

print(
    f"Standardization Success           : "
    f"{standardization_success_count} "
    f"({standardization_success_count / max(raw_count, 1):.2%})"
)

print(
    f"Valence Check Success             : "
    f"{valence_success_count}"
)

print(
    f"Model-Feasible                    : "
    f"{model_feasible_count} "
    f"({model_feasible_count / max(raw_count, 1):.2%})"
)

print(
    f"Valid Candidates                  : "
    f"{valid_count} "
    f"({valid_count / max(raw_count, 1):.2%})"
)

print(
    f"Unique Valid Candidates           : "
    f"{unique_count} "
    f"({unique_count / max(valid_count, 1):.2%} of valid)"
)

print(
    f"Novel vs Baseline                 : "
    f"{novel_count} "
    f"({novel_count / max(unique_count, 1):.2%} of unique)"
)


if np.isfinite(overall_unique_diversity):

    print(
        f"Unique Candidate Diversity        : "
        f"{overall_unique_diversity:.4f}"
    )


if np.isfinite(overall_novel_diversity):

    print(
        f"Novel Candidate Diversity         : "
        f"{overall_novel_diversity:.4f}"
    )


# ==============================================================================
# 62. COMPLETION VALIDATION
# ==============================================================================

print("\n" + "=" * 138)
print("PHASE 14 COMPLETION VALIDATION")
print("=" * 138)


all_completion_checks_passed = True


for check_name, status in completion_checks.items():

    label = (
        "PASS"
        if bool(status)
        else "FAIL"
    )

    print(
        f"[{label:<4}] {check_name}"
    )

    if not bool(status):

        all_completion_checks_passed = False


# ==============================================================================
# 63. UPDATE PROJECT CONFIG
# ==============================================================================

PHASE14_DATASETS = {
    "generation_config":
        str(
            PHASE14_CONFIG_FILE
        ),

    "latent_proposals":
        str(
            PHASE14_LATENT_PROPOSALS_FILE
        ),

    "raw_candidates":
        str(
            PHASE14_RAW_FILE
        ),

    "validated_candidates":
        str(
            PHASE14_VALIDATED_FILE
        ),

    "unique_candidates":
        str(
            PHASE14_UNIQUE_FILE
        ),

    "novel_candidates":
        str(
            PHASE14_NOVEL_FILE
        ),

    "candidate_provenance":
        str(
            PHASE14_PROVENANCE_FILE
        ),

    "generation_method_summary":
        str(
            PHASE14_METHOD_SUMMARY_FILE
        ),

    "generation_qc_csv":
        str(
            PHASE14_QC_CSV_FILE
        ),

    "generation_qc_json":
        str(
            PHASE14_QC_JSON_FILE
        ),

    "cycle":
        CYCLE_INDEX,

    "latent_proposals_count":
        len(
            latent_proposals_df
        ),

    "raw_generated_count":
        raw_count,

    "valid_candidate_count":
        valid_count,

    "unique_candidate_count":
        unique_count,

    "novel_candidate_count":
        novel_count,

    "full_decoder_latent_conditioning_used":
        True,

    "baseline_state_preserved":
        True,

    "model_adaptation_performed":
        False
}


DRUGBANK_CONFIG[
    "phase14_datasets"
] = PHASE14_DATASETS


# ==============================================================================
# 64. FINAL SUMMARY
# ==============================================================================

print("\n" + "=" * 138)
print("PHASE 14 FINAL SUMMARY")
print("=" * 138)

print(
    f"Cycle                               : "
    f"{CYCLE_INDEX}"
)

print(
    f"Training Latent Seed Corpus         : "
    f"{len(Z_seed_source)}"
)

print(
    f"Total Latent Proposals              : "
    f"{len(latent_proposals_df)}"
)


print("\nExact Decoder Pathway:")

print(
    f"  Token Embedding                   : "
    f"{embedding_name}"
)

print(
    f"  FULL Latent→Hidden Module         : "
    f"{latent_hidden_parent_name}"
)

print(
    f"  FULL Latent→Context Module        : "
    f"{latent_context_parent_name}"
)

print(
    f"  GRU                               : "
    f"{gru_name}"
)

print(
    f"  Output Projection                 : "
    f"{output_name}"
)


print("\nLatent Proposal Methods:")

for method in [
    "seed_resampling",
    "latent_perturbation",
    "latent_interpolation",
    "exploratory_prior"
]:

    count = int(
        (
            latent_proposals_df[
                "generation_method"
            ]
            ==
            method
        ).sum()
    )

    print(
        f"  {method:<32}: "
        f"{count}"
    )


print("\nCandidate Generation:")

print(
    f"  Decoder Samples / Latent          : "
    f"{DECODER_SAMPLES_PER_LATENT}"
)

print(
    f"  Raw Generated Sequences           : "
    f"{raw_count}"
)

print(
    f"  Valid Candidates                  : "
    f"{valid_count}"
)

print(
    f"  Unique Candidates                 : "
    f"{unique_count}"
)

print(
    f"  Novel vs Baseline                 : "
    f"{novel_count}"
)


print("\nTermination:")

print(
    f"  EOS                               : "
    f"{eos_count}"
)

print(
    f"  PAD                               : "
    f"{pad_count}"
)

print(
    f"  Maximum Length                    : "
    f"{max_length_count}"
)


print("\nGeneration Quality:")

print(
    f"  SELFIES Decode Rate               : "
    f"{selfies_success_count / max(raw_count, 1):.2%}"
)

print(
    f"  RDKit Parse Rate                  : "
    f"{parse_success_count / max(raw_count, 1):.2%}"
)

print(
    f"  Sanitization Rate                 : "
    f"{sanitize_success_count / max(raw_count, 1):.2%}"
)

print(
    f"  Standardization Rate              : "
    f"{standardization_success_count / max(raw_count, 1):.2%}"
)

print(
    f"  Model-Feasibility Rate            : "
    f"{model_feasible_count / max(raw_count, 1):.2%}"
)

print(
    f"  Final Validity Rate               : "
    f"{valid_count / max(raw_count, 1):.2%}"
)

print(
    f"  Uniqueness Rate                   : "
    f"{unique_count / max(valid_count, 1):.2%}"
)

print(
    f"  Novelty Rate                      : "
    f"{novel_count / max(unique_count, 1):.2%}"
)


if np.isfinite(overall_unique_diversity):

    print(
        f"  Unique Molecular Diversity        : "
        f"{overall_unique_diversity:.4f}"
    )


if np.isfinite(overall_novel_diversity):

    print(
        f"  Novel Molecular Diversity         : "
        f"{overall_novel_diversity:.4f}"
    )


print("\nBaseline Knowledge State:")

print(
    f"  θE*                               : "
    f"{'UNCHANGED' if encoder_unchanged else 'CHANGED'}"
)

print(
    f"  θD*                               : "
    f"{'UNCHANGED' if decoder_checkpoint_unchanged else 'CHANGED'}"
)

print(
    f"  θP*                               : "
    f"{'UNCHANGED' if predictor_unchanged else 'CHANGED'}"
)

print(
    f"  F_E                               : "
    f"{'UNCHANGED' if encoder_fisher_unchanged else 'CHANGED'}"
)

print(
    f"  F_P                               : "
    f"{'UNCHANGED' if predictor_fisher_unchanged else 'CHANGED'}"
)

print(
    "  K0                                : PRESERVED"
)


print("\nCandidate Pool Hierarchy:")

print(
    f"  C1_raw                            : "
    f"{raw_count}"
)

print(
    f"  C1_valid                          : "
    f"{valid_count}"
)

print(
    f"  C1_unique                         : "
    f"{unique_count}"
)

print(
    f"  C1_novel                          : "
    f"{novel_count}"
)


print("\nAdaptive-Learning Status:")

print(
    "  Optimizer Created                 : False"
)

print(
    "  Parameter Update                  : False"
)

print(
    "  EWC Applied                       : False"
)

print(
    "  Biological Candidate Ranking      : False"
)

print(
    "  Closed-Loop Model Adaptation      : False"
)


print("\nSaved Files:")

print(
    f"  Generation Config                 : "
    f"{PHASE14_CONFIG_FILE}"
)

print(
    f"  Latent Proposals                  : "
    f"{PHASE14_LATENT_PROPOSALS_FILE}"
)

print(
    f"  Raw Candidates                    : "
    f"{PHASE14_RAW_FILE}"
)

print(
    f"  Validated Candidates              : "
    f"{PHASE14_VALIDATED_FILE}"
)

print(
    f"  Unique Candidates                 : "
    f"{PHASE14_UNIQUE_FILE}"
)

print(
    f"  Novel Candidates                  : "
    f"{PHASE14_NOVEL_FILE}"
)

print(
    f"  Candidate Provenance              : "
    f"{PHASE14_PROVENANCE_FILE}"
)

print(
    f"  Method Summary                    : "
    f"{PHASE14_METHOD_SUMMARY_FILE}"
)

print(
    f"  QC Summary CSV                    : "
    f"{PHASE14_QC_CSV_FILE}"
)

print(
    f"  QC Summary JSON                   : "
    f"{PHASE14_QC_JSON_FILE}"
)


print("-" * 138)


if all_completion_checks_passed:

    print(
        "[STATUS] PHASE 14 COMPLETED SUCCESSFULLY."
    )

    print(
        "[PASS] The complete trained Phase-11 latent-conditioning "
        "modules were used during molecular generation."
    )

    print(
        "[PASS] Latent→hidden conditioning used the full trained module."
    )

    print(
        "[PASS] Latent→context conditioning used the full trained module."
    )

    print(
        "[PASS] EOS/PAD/max-length termination was explicitly recorded."
    )

    print(
        "[READY] Cycle-1 molecular candidate pool C1 has been constructed."
    )

    print(
        "[READY] Candidates originate from the learned latent "
        "representation rather than DrugBank candidate placeholders."
    )

    print(
        "[IMPORTANT] Novelty is relative to the baseline "
        "DrugBank molecular corpus."
    )

    print(
        "[IMPORTANT] No target activity, binding affinity, "
        "therapeutic efficacy, safety or synthesizability claim "
        "is made."
    )

    print(
        "[IMPORTANT] K0 remained unchanged."
    )

    print(
        "[IMPORTANT] EWC has NOT yet been applied."
    )

    print(
        "[NEXT] Phase 15 — In-Silico Candidate Evaluation "
        "and Prioritization"
    )


else:

    print(
        "[STATUS] PHASE 14 REQUIRES REVIEW."
    )

    print(
        "[ACTION] Do NOT proceed to Phase 15 until "
        "all failed checks are resolved."
    )


print("=" * 138)

PHASE 14 — LATENT-SPACE CANDIDATE GENERATION AND PROPOSAL POOL CONSTRUCTION

[DEVICE] Device                         : cpu
[CONFIG] Random Seed                   : 42
[CONFIG] Cycle                         : 1
[CONFIG] Latent Dimension              : 32
[CONFIG] Direct Seed Latents            : 40
[CONFIG] Perturbation Seeds             : 60
[CONFIG] Perturbation Sigmas            : [0.1, 0.25]
[CONFIG] Perturbations/Sigma            : 2
[CONFIG] Interpolation Pairs            : 30
[CONFIG] Interpolation Alphas           : [0.25, 0.5, 0.75]
[CONFIG] Prior Latents                  : 60
[CONFIG] Decoder Samples/Latent         : 2
[CONFIG] Decoder Temperature            : 0.9
[CONFIG] Decoder Top-K                  : 10
[CONFIG] Heavy Atom Feasibility         : 4-80

[PASS] Required Phase-11, Phase-12 and Phase-13 files found.

PRE-GENERATION BASELINE K0 INTEGRITY
θE* SHA-256 : 44c3e8afe63cd7cdc055a6bdc35335c4999bb32969a12426cfee54fa77566f1a
θD* SHA-256 : 4cee9bb5ae26206f154e67e0f7732282d

# Phase 15 — Chemical Eligibility and Pre-Evaluation Screening

**Objective:** Execute an auditable chemical quality-control and eligibility screening on the novel candidate proposals ($C_1^{\text{novel}}$) generated in Phase 14 to construct the verified evaluation pool ($C_1^{\text{eligible}}$).

* **Screening Philosophy & Boundary Rules:**
  * **Strict Structural Validity (Hard Gates):** Unbroken valence, successful RDKit sanitization, single-entity fragment integrity ($N_{\text{fragments}} = 1$), finite numerical descriptors, and heavy-atom feasibility ($4 \le N_{\text{heavy}} \le 80$).
  * **Soft Physicochemical Characterization:** Lipophilicity (cLogP), polarity (TPSA), molecular weight (MW), flexibility ($N_{\text{rot}}$), ionization ($Q_{\text{formal}}$), 3D saturation ($F_{\text{sp3}}$), and Lipinski Rule-of-5 compliance tracked as descriptive features rather than heuristic exclusions.
  * **Scope Discipline:** No biological predictions, binding affinities, therapeutic efficacy claims, or model parameter adaptations occur during this screening stage.

* **Primary Deliverables:**
  * Eligible Candidate Pool: `phase15_eligible_candidates.csv` ($C_1^{\text{eligible}}$).
  * Excluded Proposal Log: `phase15_ineligible_candidates.csv` ($C_1^{\text{ineligible}}$).
  * Comprehensive Audit: `phase15_candidate_eligibility_screen.csv`, `phase15_descriptor_statistics.csv`, `phase15_eligibility_qc_summary.json`.

In [15]:
# ==============================================================================
# PHASE 15 — CHEMICAL ELIGIBILITY AND PRE-EVALUATION SCREENING
# FINAL IMPLEMENTATION
# ==============================================================================
#
# INPUT
# -----
#
# Finalized Phase-14 novel candidate pool:
#
#     C1_novel
#
#
# OUTPUT
# ------
#
#     C1_eligible
#     C1_ineligible
#
#
# PHASE 15 PERFORMS
# -----------------
#
# - preservation of Phase-14 candidate identity
# - RDKit reparsing
# - sanitization verification
# - valence/property-cache verification
# - single-fragment verification
# - 4–80 heavy-atom model-feasibility verification
# - physicochemical descriptor calculation
# - descriptor finiteness checking
# - Lipinski violation counting
# - soft physicochemical flagging
# - QED calculation
# - SA score calculation IF standard RDKit SA scorer is available
# - explicit eligibility classification
# - complete exclusion audit
#
#
# PHASE 15 DOES NOT PERFORM
# -------------------------
#
# - target-association scoring
# - binding-affinity prediction
# - biological ranking
# - UCB
# - candidate utility optimization
# - EWC
# - optimizer updates
# - model adaptation
#
#
# IMPORTANT
# ---------
#
# Phase-14 canonical SMILES are preserved as the authoritative candidate
# identities. Phase 15 verifies those molecules but does not silently
# re-standardize them into new identities.
#
# ==============================================================================


# ==============================================================================
# 1. IMPORTS
# ==============================================================================

from pathlib import Path
from datetime import datetime

import os
import sys
import json
import math
import hashlib
import warnings
import importlib

import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit import RDConfig

from rdkit.Chem import (
    Descriptors,
    Crippen,
    Lipinski,
    QED,
    rdMolDescriptors
)


warnings.filterwarnings("ignore")


# ==============================================================================
# 2. VERIFY PROJECT STATE
# ==============================================================================

required_variables = [
    "PROCESSED_DIR",
    "DRUGBANK_CONFIG"
]


missing_variables = [
    variable
    for variable in required_variables
    if variable not in globals()
]


if missing_variables:

    raise RuntimeError(
        "Earlier pipeline variables are missing:\n"
        f"{missing_variables}"
    )


PROCESSED_DIR = Path(
    PROCESSED_DIR
)


# ==============================================================================
# 3. PHASE-15 CONFIGURATION
# ==============================================================================

CYCLE_INDEX = 1

MIN_HEAVY_ATOMS = 4

MAX_HEAVY_ATOMS = 80


# ------------------------------------------------------------------------------
# Conventional Lipinski Rule-of-Five thresholds
#
# These are used only for descriptive/soft characterization.
# They are NOT Phase-15 hard exclusion criteria.
# ------------------------------------------------------------------------------

LIPINSKI_MW_MAX = 500.0

LIPINSKI_CLOGP_MAX = 5.0

LIPINSKI_HBD_MAX = 5

LIPINSKI_HBA_MAX = 10


# ------------------------------------------------------------------------------
# Additional SOFT physicochemical flag thresholds.
#
# These are transparent operational flags only.
# They are NOT universal definitions of chemical quality.
# ------------------------------------------------------------------------------

SOFT_HIGH_MW = 500.0

SOFT_HIGH_CLOGP = 5.0

SOFT_HIGH_TPSA = 140.0

SOFT_HIGH_HBD = 5

SOFT_HIGH_HBA = 10

SOFT_HIGH_ROTATABLE_BONDS = 10

SOFT_LARGE_ABS_FORMAL_CHARGE = 2

SOFT_LIPINSKI_VIOLATION_THRESHOLD = 1


print("=" * 140)

print(
    "PHASE 15 — CHEMICAL ELIGIBILITY "
    "AND PRE-EVALUATION SCREENING"
)

print("=" * 140)


print(
    f"\n[CONFIG] Cycle                         : "
    f"{CYCLE_INDEX}"
)

print(
    f"[CONFIG] Heavy-Atom Feasibility        : "
    f"{MIN_HEAVY_ATOMS}-{MAX_HEAVY_ATOMS}"
)

print(
    "[CONFIG] Lipinski rules                 : "
    "SOFT CHARACTERIZATION ONLY"
)

print(
    "[CONFIG] QED                            : "
    "Passive descriptor"
)

print(
    "[CONFIG] Biological candidate ranking   : False"
)

print(
    "[CONFIG] EWC                            : False"
)

print(
    "[CONFIG] Model parameter adaptation     : False"
)


# ==============================================================================
# 4. INPUT FILES
# ==============================================================================

PHASE14_NOVEL_FILE = (
    PROCESSED_DIR
    / "phase14_generated_candidates_novel.csv"
)


PHASE14_CONFIG_FILE = (
    PROCESSED_DIR
    / "phase14_generation_config.json"
)


PHASE14_QC_FILE = (
    PROCESSED_DIR
    / "phase14_generation_qc_summary.json"
)


PHASE12_ENCODER_FILE = (
    PROCESSED_DIR
    / "phase12_thetaE_baseline_encoder.pt"
)


PHASE12_DECODER_FILE = (
    PROCESSED_DIR
    / "phase12_thetaD_baseline_decoder.pt"
)


PHASE12_PREDICTOR_FILE = (
    PROCESSED_DIR
    / "phase12_best_baseline_predictor.pt"
)


PHASE13_ENCODER_FISHER_FILE = (
    PROCESSED_DIR
    / "phase13_encoder_fisher.pt"
)


PHASE13_PREDICTOR_FISHER_FILE = (
    PROCESSED_DIR
    / "phase13_predictor_fisher.pt"
)


PHASE13_REFERENCE_FILE = (
    PROCESSED_DIR
    / "phase13_fisher_parameter_reference.pt"
)


# ==============================================================================
# 5. LOCATE BASELINE STANDARDIZED DRUGBANK CORPUS
# ==============================================================================

baseline_file_candidates = [
    PROCESSED_DIR
    / "D_chem_standardized.csv",

    PROCESSED_DIR
    / "D_chem_standardized_qc.csv",

    PROCESSED_DIR.parent
    / "processed"
    / "D_chem_standardized.csv"
]


BASELINE_CHEM_FILE = None


for candidate in baseline_file_candidates:

    if candidate.exists():

        BASELINE_CHEM_FILE = candidate

        break


required_files = [
    PHASE14_NOVEL_FILE,
    PHASE14_CONFIG_FILE,
    PHASE14_QC_FILE,
    PHASE12_ENCODER_FILE,
    PHASE12_DECODER_FILE,
    PHASE12_PREDICTOR_FILE,
    PHASE13_ENCODER_FISHER_FILE,
    PHASE13_PREDICTOR_FISHER_FILE,
    PHASE13_REFERENCE_FILE
]


missing_files = [
    str(file_path)
    for file_path
    in required_files
    if not file_path.exists()
]


if BASELINE_CHEM_FILE is None:

    missing_files.append(
        "D_chem_standardized.csv"
    )


if missing_files:

    raise FileNotFoundError(
        "Required Phase-14/12/13 files are missing:\n"
        +
        "\n".join(
            missing_files
        )
    )


print(
    "\n[PASS] Required Phase-14 candidate and baseline-state files found."
)


# ==============================================================================
# 6. OUTPUT FILES
# ==============================================================================

PHASE15_CONFIG_FILE = (
    PROCESSED_DIR
    / "phase15_screening_config.json"
)


PHASE15_FULL_SCREEN_FILE = (
    PROCESSED_DIR
    / "phase15_candidate_eligibility_screen.csv"
)


PHASE15_ELIGIBLE_FILE = (
    PROCESSED_DIR
    / "phase15_eligible_candidates.csv"
)


PHASE15_INELIGIBLE_FILE = (
    PROCESSED_DIR
    / "phase15_ineligible_candidates.csv"
)


PHASE15_DESCRIPTOR_STATS_FILE = (
    PROCESSED_DIR
    / "phase15_descriptor_statistics.csv"
)


PHASE15_EXCLUSION_SUMMARY_FILE = (
    PROCESSED_DIR
    / "phase15_exclusion_summary.csv"
)


PHASE15_QC_CSV_FILE = (
    PROCESSED_DIR
    / "phase15_eligibility_qc_summary.csv"
)


PHASE15_QC_JSON_FILE = (
    PROCESSED_DIR
    / "phase15_eligibility_qc_summary.json"
)


# ==============================================================================
# 7. SHA-256 HELPER
# ==============================================================================

def sha256_file(
    file_path,
    chunk_size=1024 * 1024
):

    digest = hashlib.sha256()


    with open(
        file_path,
        "rb"
    ) as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )


            if not chunk:

                break


            digest.update(
                chunk
            )


    return digest.hexdigest()


# ==============================================================================
# 8. RECORD K0 CHECKPOINT/FISHER HASHES BEFORE SCREENING
# ==============================================================================

encoder_hash_before = sha256_file(
    PHASE12_ENCODER_FILE
)


decoder_hash_before = sha256_file(
    PHASE12_DECODER_FILE
)


predictor_hash_before = sha256_file(
    PHASE12_PREDICTOR_FILE
)


encoder_fisher_hash_before = sha256_file(
    PHASE13_ENCODER_FISHER_FILE
)


predictor_fisher_hash_before = sha256_file(
    PHASE13_PREDICTOR_FISHER_FILE
)


phase13_reference_hash_before = sha256_file(
    PHASE13_REFERENCE_FILE
)


print("\n" + "=" * 140)
print("PRE-SCREENING BASELINE K0 INTEGRITY")
print("=" * 140)


print(
    f"θE* SHA-256 : {encoder_hash_before}"
)

print(
    f"θD* SHA-256 : {decoder_hash_before}"
)

print(
    f"θP* SHA-256 : {predictor_hash_before}"
)

print(
    f"F_E SHA-256 : {encoder_fisher_hash_before}"
)

print(
    f"F_P SHA-256 : {predictor_fisher_hash_before}"
)


# ==============================================================================
# 9. LOAD PHASE-14 FINAL NOVEL CANDIDATE POOL
# ==============================================================================

candidates_df = pd.read_csv(
    PHASE14_NOVEL_FILE,
    dtype={
        "candidate_id": str,
        "canonical_smiles": str
    }
)


INPUT_CANDIDATE_COUNT = len(
    candidates_df
)


if INPUT_CANDIDATE_COUNT == 0:

    raise RuntimeError(
        "Phase-14 C1_novel candidate pool is empty."
    )


print("\n" + "=" * 140)
print("PHASE-14 INPUT CANDIDATE POOL")
print("=" * 140)


print(
    f"C1_novel input candidates       : "
    f"{INPUT_CANDIDATE_COUNT}"
)


# ==============================================================================
# 10. REQUIRED INPUT COLUMNS
# ==============================================================================

required_candidate_columns = [
    "candidate_id",
    "canonical_smiles"
]


missing_candidate_columns = [
    column
    for column
    in required_candidate_columns
    if column not in candidates_df.columns
]


if missing_candidate_columns:

    raise RuntimeError(
        "Phase-14 novel candidate table is missing required columns:\n"
        f"{missing_candidate_columns}"
    )


# ==============================================================================
# 11. PHASE-14 IDENTITY INTEGRITY
# ==============================================================================

candidate_id_missing = (
    candidates_df[
        "candidate_id"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
    ==
    ""
)


if candidate_id_missing.any():

    raise RuntimeError(
        "Phase-14 C1_novel contains missing candidate IDs."
    )


candidate_id_duplicates = (
    candidates_df[
        "candidate_id"
    ]
    .duplicated(
        keep=False
    )
)


if candidate_id_duplicates.any():

    duplicate_ids = (
        candidates_df.loc[
            candidate_id_duplicates,
            "candidate_id"
        ]
        .tolist()
    )


    raise RuntimeError(
        "Duplicate Phase-14 candidate IDs detected:\n"
        f"{duplicate_ids[:20]}"
    )


canonical_input_series = (
    candidates_df[
        "canonical_smiles"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
)


non_empty_canonical = (
    canonical_input_series
    !=
    ""
)


duplicate_canonical_mask = (
    canonical_input_series[
        non_empty_canonical
    ]
    .duplicated(
        keep=False
    )
)


if duplicate_canonical_mask.any():

    duplicate_values = (
        canonical_input_series[
            non_empty_canonical
        ][
            duplicate_canonical_mask
        ]
        .tolist()
    )


    raise RuntimeError(
        "Phase-14 C1_novel is not unique by canonical SMILES.\n"
        f"Example duplicates: {duplicate_values[:20]}"
    )


print(
    "[PASS] Candidate IDs are present and unique."
)

print(
    "[PASS] Phase-14 canonical candidate identities are unique."
)


# ==============================================================================
# 12. LOAD BASELINE DRUGBANK CHEMISTRY FOR NOVELTY RECONFIRMATION
# ==============================================================================

baseline_df = pd.read_csv(
    BASELINE_CHEM_FILE,
    dtype=str,
    keep_default_na=False
)


baseline_smiles_candidates = [
    "standardized_smiles",
    "canonical_smiles",
    "smiles_standardized",
    "standardized_canonical_smiles",
    "canonical_standardized_smiles"
]


BASELINE_SMILES_COLUMN = None


for column in baseline_smiles_candidates:

    if column in baseline_df.columns:

        BASELINE_SMILES_COLUMN = column

        break


if BASELINE_SMILES_COLUMN is None:

    raise RuntimeError(
        "Could not identify standardized SMILES column in "
        f"{BASELINE_CHEM_FILE.name}.\n"
        f"Columns: {list(baseline_df.columns)}"
    )


baseline_canonical_smiles = set()


for smiles in baseline_df[
    BASELINE_SMILES_COLUMN
].astype(str):

    smiles = smiles.strip()


    if not smiles:

        continue


    mol = Chem.MolFromSmiles(
        smiles
    )


    if mol is None:

        continue


    canonical = Chem.MolToSmiles(
        mol,
        canonical=True,
        isomericSmiles=True
    )


    baseline_canonical_smiles.add(
        canonical
    )


print(
    f"[PASS] Baseline novelty reference contains "
    f"{len(baseline_canonical_smiles)} unique molecules."
)


# ==============================================================================
# 13. STANDARD RDKit SA-SCORE DETECTION
#
# IMPORTANT:
#
# We do NOT invent an SA approximation.
#
# We first try the standard RDKit Contrib SA_Score implementation.
#
# ==============================================================================

SA_SCORER_AVAILABLE = False

SA_SCORER_SOURCE = None

sascorer = None


try:

    from rdkit.Contrib.SA_Score import sascorer as _sascorer

    sascorer = _sascorer

    SA_SCORER_AVAILABLE = True

    SA_SCORER_SOURCE = (
        "rdkit.Contrib.SA_Score.sascorer"
    )


except Exception:

    try:

        sa_score_directory = Path(
            RDConfig.RDContribDir
        ) / "SA_Score"


        if sa_score_directory.exists():

            if str(
                sa_score_directory
            ) not in sys.path:

                sys.path.insert(
                    0,
                    str(
                        sa_score_directory
                    )
                )


            sascorer = importlib.import_module(
                "sascorer"
            )


            SA_SCORER_AVAILABLE = True

            SA_SCORER_SOURCE = str(
                sa_score_directory
                / "sascorer.py"
            )


    except Exception:

        SA_SCORER_AVAILABLE = False

        sascorer = None


print("\n" + "=" * 140)
print("SYNTHETIC ACCESSIBILITY SCORER")
print("=" * 140)


print(
    f"Standard SA scorer available : "
    f"{SA_SCORER_AVAILABLE}"
)


if SA_SCORER_AVAILABLE:

    print(
        f"SA scorer source             : "
        f"{SA_SCORER_SOURCE}"
    )


else:

    print(
        "[INFO] Standard SA scorer was not found."
    )

    print(
        "[INFO] SA values will remain unavailable."
    )

    print(
        "[INFO] No undocumented SA approximation will be substituted."
    )


# ==============================================================================
# 14. DESCRIPTOR FUNCTION
# ==============================================================================

CORE_DESCRIPTOR_NAMES = [
    "mw",
    "heavy_atom_count",
    "clogp",
    "tpsa",
    "hbd",
    "hba",
    "rotatable_bonds",
    "formal_charge",
    "ring_count",
    "aromatic_ring_count",
    "fraction_csp3",
    "qed"
]


def calculate_descriptors(
    mol
):

    descriptors = {
        "mw":
            float(
                Descriptors.MolWt(
                    mol
                )
            ),

        "heavy_atom_count":
            int(
                mol.GetNumHeavyAtoms()
            ),

        "clogp":
            float(
                Crippen.MolLogP(
                    mol
                )
            ),

        "tpsa":
            float(
                rdMolDescriptors.CalcTPSA(
                    mol
                )
            ),

        "hbd":
            int(
                Lipinski.NumHDonors(
                    mol
                )
            ),

        "hba":
            int(
                Lipinski.NumHAcceptors(
                    mol
                )
            ),

        "rotatable_bonds":
            int(
                Lipinski.NumRotatableBonds(
                    mol
                )
            ),

        "formal_charge":
            int(
                sum(
                    atom.GetFormalCharge()
                    for atom
                    in mol.GetAtoms()
                )
            ),

        "ring_count":
            int(
                Lipinski.RingCount(
                    mol
                )
            ),

        "aromatic_ring_count":
            int(
                Lipinski.NumAromaticRings(
                    mol
                )
            ),

        "fraction_csp3":
            float(
                rdMolDescriptors.CalcFractionCSP3(
                    mol
                )
            ),

        "qed":
            float(
                QED.qed(
                    mol
                )
            )
    }


    return descriptors


# ==============================================================================
# 15. LIPINSKI VIOLATION FUNCTION
# ==============================================================================

def calculate_lipinski_violations(
    descriptor_values
):

    violations = []


    if descriptor_values[
        "mw"
    ] > LIPINSKI_MW_MAX:

        violations.append(
            "mw_gt_500"
        )


    if descriptor_values[
        "clogp"
    ] > LIPINSKI_CLOGP_MAX:

        violations.append(
            "clogp_gt_5"
        )


    if descriptor_values[
        "hbd"
    ] > LIPINSKI_HBD_MAX:

        violations.append(
            "hbd_gt_5"
        )


    if descriptor_values[
        "hba"
    ] > LIPINSKI_HBA_MAX:

        violations.append(
            "hba_gt_10"
        )


    return violations


# ==============================================================================
# 16. SOFT PHYSICOCHEMICAL FLAGS
# ==============================================================================

def calculate_soft_flags(
    descriptor_values,
    lipinski_violation_count
):

    flags = []


    if descriptor_values[
        "mw"
    ] > SOFT_HIGH_MW:

        flags.append(
            "high_mw"
        )


    if descriptor_values[
        "clogp"
    ] > SOFT_HIGH_CLOGP:

        flags.append(
            "high_clogp"
        )


    if descriptor_values[
        "tpsa"
    ] > SOFT_HIGH_TPSA:

        flags.append(
            "high_tpsa"
        )


    if descriptor_values[
        "hbd"
    ] > SOFT_HIGH_HBD:

        flags.append(
            "high_hbd"
        )


    if descriptor_values[
        "hba"
    ] > SOFT_HIGH_HBA:

        flags.append(
            "high_hba"
        )


    if descriptor_values[
        "rotatable_bonds"
    ] > SOFT_HIGH_ROTATABLE_BONDS:

        flags.append(
            "high_rotatable_bonds"
        )


    if abs(
        descriptor_values[
            "formal_charge"
        ]
    ) > SOFT_LARGE_ABS_FORMAL_CHARGE:

        flags.append(
            "large_absolute_formal_charge"
        )


    if (
        lipinski_violation_count
        >
        SOFT_LIPINSKI_VIOLATION_THRESHOLD
    ):

        flags.append(
            "multiple_lipinski_violations"
        )


    return flags


# ==============================================================================
# 17. FINITENESS FUNCTION
# ==============================================================================

def all_required_descriptors_finite(
    descriptor_values
):

    for descriptor_name in CORE_DESCRIPTOR_NAMES:

        value = descriptor_values[
            descriptor_name
        ]


        try:

            numeric_value = float(
                value
            )


        except Exception:

            return False


        if not np.isfinite(
            numeric_value
        ):

            return False


    return True


# ==============================================================================
# 18. PHASE-15 ELIGIBILITY SCREEN
# ==============================================================================

print("\n" + "=" * 140)
print("PHASE-15 CANDIDATE ELIGIBILITY SCREENING")
print("=" * 140)


screening_records = []


for row_index, row in candidates_df.iterrows():

    record = row.to_dict()


    canonical_smiles_input = str(
        row.get(
            "canonical_smiles",
            ""
        )
    ).strip()


    candidate_id = str(
        row.get(
            "candidate_id",
            ""
        )
    ).strip()


    hard_exclusion_reasons = []


    # --------------------------------------------------------------------------
    # Initialize screening fields
    # --------------------------------------------------------------------------

    record.update(
        {
            "phase15_input_index":
                int(
                    row_index
                ),

            "canonical_identity_preserved":
                False,

            "recanonicalized_smiles":
                "",

            "baseline_novel_reconfirmed":
                False,

            "rdkit_valid":
                False,

            "sanitize_valid":
                False,

            "valence_valid":
                False,

            "nonempty_molecule":
                False,

            "fragment_count":
                np.nan,

            "single_fragment":
                False,

            "heavy_atom_count":
                np.nan,

            "model_feasible":
                False,

            "mw":
                np.nan,

            "clogp":
                np.nan,

            "tpsa":
                np.nan,

            "hbd":
                np.nan,

            "hba":
                np.nan,

            "rotatable_bonds":
                np.nan,

            "formal_charge":
                np.nan,

            "ring_count":
                np.nan,

            "aromatic_ring_count":
                np.nan,

            "fraction_csp3":
                np.nan,

            "lipinski_violation_count":
                np.nan,

            "lipinski_violations":
                "",

            "lipinski_soft_flag":
                False,

            "qed":
                np.nan,

            "sa_score":
                np.nan,

            "sa_score_available":
                bool(
                    SA_SCORER_AVAILABLE
                ),

            "sa_score_calculated":
                False,

            "descriptor_finite":
                False,

            "soft_flag_count":
                0,

            "soft_flags":
                "",

            "hard_exclusion":
                False,

            "hard_exclusion_reason":
                "",

            "chemical_eligible":
                False
        }
    )


    # --------------------------------------------------------------------------
    # Missing identity
    # --------------------------------------------------------------------------

    if not canonical_smiles_input:

        hard_exclusion_reasons.append(
            "missing_canonical_smiles"
        )


        record[
            "hard_exclusion"
        ] = True


        record[
            "hard_exclusion_reason"
        ] = ";".join(
            hard_exclusion_reasons
        )


        screening_records.append(
            record
        )

        continue


    # --------------------------------------------------------------------------
    # RDKit parsing WITHOUT changing candidate identity
    # --------------------------------------------------------------------------

    try:

        mol = Chem.MolFromSmiles(
            canonical_smiles_input,
            sanitize=False
        )


    except Exception:

        mol = None


    if mol is None:

        hard_exclusion_reasons.append(
            "rdkit_parse_failure"
        )


        record[
            "hard_exclusion"
        ] = True


        record[
            "hard_exclusion_reason"
        ] = ";".join(
            hard_exclusion_reasons
        )


        screening_records.append(
            record
        )

        continue


    record[
        "rdkit_valid"
    ] = True


    # --------------------------------------------------------------------------
    # Molecule contains atoms
    # --------------------------------------------------------------------------

    if mol.GetNumAtoms() <= 0:

        hard_exclusion_reasons.append(
            "empty_molecule"
        )


        record[
            "hard_exclusion"
        ] = True


        record[
            "hard_exclusion_reason"
        ] = ";".join(
            hard_exclusion_reasons
        )


        screening_records.append(
            record
        )

        continue


    record[
        "nonempty_molecule"
    ] = True


    # --------------------------------------------------------------------------
    # Sanitization
    # --------------------------------------------------------------------------

    try:

        Chem.SanitizeMol(
            mol
        )


        record[
            "sanitize_valid"
        ] = True


    except Exception:

        hard_exclusion_reasons.append(
            "sanitize_failure"
        )


        record[
            "hard_exclusion"
        ] = True


        record[
            "hard_exclusion_reason"
        ] = ";".join(
            hard_exclusion_reasons
        )


        screening_records.append(
            record
        )

        continue


    # --------------------------------------------------------------------------
    # Property cache / valence
    # --------------------------------------------------------------------------

    try:

        mol.UpdatePropertyCache(
            strict=True
        )


        Chem.SanitizeMol(
            mol
        )


        record[
            "valence_valid"
        ] = True


    except Exception:

        hard_exclusion_reasons.append(
            "valence_failure"
        )


        record[
            "hard_exclusion"
        ] = True


        record[
            "hard_exclusion_reason"
        ] = ";".join(
            hard_exclusion_reasons
        )


        screening_records.append(
            record
        )

        continue


    # --------------------------------------------------------------------------
    # Recanonicalize only to VERIFY identity.
    #
    # IMPORTANT:
    # authoritative identity remains Phase-14 canonical_smiles.
    # --------------------------------------------------------------------------

    try:

        recanonicalized_smiles = Chem.MolToSmiles(
            mol,
            canonical=True,
            isomericSmiles=True
        )


        record[
            "recanonicalized_smiles"
        ] = recanonicalized_smiles


        identity_preserved = (
            recanonicalized_smiles
            ==
            canonical_smiles_input
        )


        record[
            "canonical_identity_preserved"
        ] = bool(
            identity_preserved
        )


        if not identity_preserved:

            hard_exclusion_reasons.append(
                "canonical_identity_mismatch"
            )


    except Exception:

        hard_exclusion_reasons.append(
            "canonical_identity_verification_failure"
        )


    # --------------------------------------------------------------------------
    # Novelty reconfirmation
    # --------------------------------------------------------------------------

    record[
        "baseline_novel_reconfirmed"
    ] = bool(
        canonical_smiles_input
        not in
        baseline_canonical_smiles
    )


    if not record[
        "baseline_novel_reconfirmed"
    ]:

        hard_exclusion_reasons.append(
            "not_novel_vs_baseline"
        )


    # --------------------------------------------------------------------------
    # Fragment count
    # --------------------------------------------------------------------------

    fragment_count = len(
        Chem.GetMolFrags(
            mol,
            asMols=False,
            sanitizeFrags=False
        )
    )


    record[
        "fragment_count"
    ] = int(
        fragment_count
    )


    record[
        "single_fragment"
    ] = bool(
        fragment_count
        ==
        1
    )


    if fragment_count != 1:

        hard_exclusion_reasons.append(
            "disconnected_fragments"
        )


    # --------------------------------------------------------------------------
    # Calculate core descriptors
    # --------------------------------------------------------------------------

    try:

        descriptor_values = calculate_descriptors(
            mol
        )


        for descriptor_name, descriptor_value in (
            descriptor_values.items()
        ):

            record[
                descriptor_name
            ] = descriptor_value


    except Exception:

        descriptor_values = None


        hard_exclusion_reasons.append(
            "descriptor_calculation_failure"
        )


    # --------------------------------------------------------------------------
    # Continue only if descriptor calculation succeeded
    # --------------------------------------------------------------------------

    if descriptor_values is not None:

        # ----------------------------------------------------------------------
        # Heavy-atom feasibility
        # ----------------------------------------------------------------------

        heavy_atom_count = int(
            descriptor_values[
                "heavy_atom_count"
            ]
        )


        record[
            "model_feasible"
        ] = bool(
            MIN_HEAVY_ATOMS
            <=
            heavy_atom_count
            <=
            MAX_HEAVY_ATOMS
        )


        if not record[
            "model_feasible"
        ]:

            hard_exclusion_reasons.append(
                "outside_model_feasibility"
            )


        # ----------------------------------------------------------------------
        # Descriptor finiteness
        # ----------------------------------------------------------------------

        descriptor_finite = (
            all_required_descriptors_finite(
                descriptor_values
            )
        )


        record[
            "descriptor_finite"
        ] = bool(
            descriptor_finite
        )


        if not descriptor_finite:

            hard_exclusion_reasons.append(
                "non_finite_descriptor"
            )


        # ----------------------------------------------------------------------
        # Lipinski characterization
        # ----------------------------------------------------------------------

        lipinski_violations = (
            calculate_lipinski_violations(
                descriptor_values
            )
        )


        lipinski_violation_count = len(
            lipinski_violations
        )


        record[
            "lipinski_violation_count"
        ] = int(
            lipinski_violation_count
        )


        record[
            "lipinski_violations"
        ] = ";".join(
            lipinski_violations
        )


        record[
            "lipinski_soft_flag"
        ] = bool(
            lipinski_violation_count
            >
            SOFT_LIPINSKI_VIOLATION_THRESHOLD
        )


        # ----------------------------------------------------------------------
        # Soft physicochemical flags
        # ----------------------------------------------------------------------

        soft_flags = calculate_soft_flags(
            descriptor_values,
            lipinski_violation_count
        )


        record[
            "soft_flag_count"
        ] = len(
            soft_flags
        )


        record[
            "soft_flags"
        ] = ";".join(
            soft_flags
        )


        # ----------------------------------------------------------------------
        # Standard SA Score
        #
        # Passive descriptor only.
        #
        # Failure to calculate SA does NOT make molecule ineligible.
        # ----------------------------------------------------------------------

        if SA_SCORER_AVAILABLE:

            try:

                sa_score = float(
                    sascorer.calculateScore(
                        mol
                    )
                )


                if np.isfinite(
                    sa_score
                ):

                    record[
                        "sa_score"
                    ] = sa_score


                    record[
                        "sa_score_calculated"
                    ] = True


            except Exception:

                record[
                    "sa_score_calculated"
                ] = False


    # --------------------------------------------------------------------------
    # Final hard-exclusion classification
    # --------------------------------------------------------------------------

    hard_exclusion_reasons = list(
        dict.fromkeys(
            hard_exclusion_reasons
        )
    )


    record[
        "hard_exclusion"
    ] = bool(
        len(
            hard_exclusion_reasons
        )
        >
        0
    )


    record[
        "hard_exclusion_reason"
    ] = ";".join(
        hard_exclusion_reasons
    )


    record[
        "chemical_eligible"
    ] = bool(
        not record[
            "hard_exclusion"
        ]
    )


    screening_records.append(
        record
    )


# ==============================================================================
# 19. BUILD SCREENING TABLE
# ==============================================================================

screening_df = pd.DataFrame(
    screening_records
)


if len(
    screening_df
) != INPUT_CANDIDATE_COUNT:

    raise RuntimeError(
        "Phase-15 screening row count does not equal input candidate count."
    )


# ==============================================================================
# 20. CONSTRUCT ELIGIBLE AND INELIGIBLE POOLS
# ==============================================================================

eligible_df = (
    screening_df[
        screening_df[
            "chemical_eligible"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


ineligible_df = (
    screening_df[
        ~screening_df[
            "chemical_eligible"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


ELIGIBLE_COUNT = len(
    eligible_df
)


INELIGIBLE_COUNT = len(
    ineligible_df
)


# ==============================================================================
# 21. COUNT CONSERVATION
# ==============================================================================

count_conserved = (
    INPUT_CANDIDATE_COUNT
    ==
    ELIGIBLE_COUNT
    +
    INELIGIBLE_COUNT
)


if not count_conserved:

    raise RuntimeError(
        "Phase-15 candidate-count conservation failed."
    )


print("\n" + "=" * 140)
print("ELIGIBILITY CLASSIFICATION")
print("=" * 140)


print(
    f"C1_novel input candidates       : "
    f"{INPUT_CANDIDATE_COUNT}"
)

print(
    f"C1_eligible                     : "
    f"{ELIGIBLE_COUNT}"
)

print(
    f"C1_ineligible                   : "
    f"{INELIGIBLE_COUNT}"
)

print(
    f"Count conserved                 : "
    f"{count_conserved}"
)


# ==============================================================================
# 22. PHASE-14 INTEGRITY CHECKS
# ==============================================================================

candidate_ids_preserved = (
    set(
        screening_df[
            "candidate_id"
        ].astype(str)
    )
    ==
    set(
        candidates_df[
            "candidate_id"
        ].astype(str)
    )
)


canonical_smiles_preserved = (
    screening_df[
        "canonical_smiles"
    ].astype(str).tolist()
    ==
    candidates_df[
        "canonical_smiles"
    ].astype(str).tolist()
)


uniqueness_preserved = (
    screening_df[
        "canonical_smiles"
    ]
    .astype(str)
    .duplicated()
    .sum()
    ==
    0
)


novelty_reconfirmed_for_all = bool(
    screening_df[
        "baseline_novel_reconfirmed"
    ].all()
)


canonical_identity_verified_for_all = bool(
    screening_df[
        "canonical_identity_preserved"
    ].all()
)


# ==============================================================================
# 23. ELIGIBILITY COUNTS BY STRUCTURAL CHECK
# ==============================================================================

rdkit_valid_count = int(
    screening_df[
        "rdkit_valid"
    ].sum()
)


sanitize_valid_count = int(
    screening_df[
        "sanitize_valid"
    ].sum()
)


valence_valid_count = int(
    screening_df[
        "valence_valid"
    ].sum()
)


single_fragment_count = int(
    screening_df[
        "single_fragment"
    ].sum()
)


model_feasible_count = int(
    screening_df[
        "model_feasible"
    ].sum()
)


finite_descriptor_count = int(
    screening_df[
        "descriptor_finite"
    ].sum()
)


qed_available_count = int(
    screening_df[
        "qed"
    ].notna()
    .sum()
)


sa_calculated_count = int(
    screening_df[
        "sa_score_calculated"
    ].sum()
)


# ==============================================================================
# 24. SAVE COMPLETE AUDIT TABLES
# ==============================================================================

screening_df.to_csv(
    PHASE15_FULL_SCREEN_FILE,
    index=False,
    encoding="utf-8"
)


eligible_df.to_csv(
    PHASE15_ELIGIBLE_FILE,
    index=False,
    encoding="utf-8"
)


ineligible_df.to_csv(
    PHASE15_INELIGIBLE_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 25. DESCRIPTOR STATISTICS
# ==============================================================================

descriptor_columns_for_statistics = [
    "mw",
    "heavy_atom_count",
    "clogp",
    "tpsa",
    "hbd",
    "hba",
    "rotatable_bonds",
    "formal_charge",
    "ring_count",
    "aromatic_ring_count",
    "fraction_csp3",
    "lipinski_violation_count",
    "qed"
]


if (
    SA_SCORER_AVAILABLE
    and
    screening_df[
        "sa_score"
    ].notna().any()
):

    descriptor_columns_for_statistics.append(
        "sa_score"
    )


descriptor_statistics_records = []


for pool_name, pool_df in [
    (
        "all_screened",
        screening_df
    ),
    (
        "eligible",
        eligible_df
    )
]:

    for descriptor_name in (
        descriptor_columns_for_statistics
    ):

        values = pd.to_numeric(
            pool_df[
                descriptor_name
            ],
            errors="coerce"
        )


        values = values[
            np.isfinite(
                values
            )
        ]


        if len(
            values
        ) == 0:

            continue


        descriptor_statistics_records.append(
            {
                "pool":
                    pool_name,

                "descriptor":
                    descriptor_name,

                "n":
                    int(
                        len(
                            values
                        )
                    ),

                "minimum":
                    float(
                        values.min()
                    ),

                "q25":
                    float(
                        values.quantile(
                            0.25
                        )
                    ),

                "median":
                    float(
                        values.median()
                    ),

                "mean":
                    float(
                        values.mean()
                    ),

                "q75":
                    float(
                        values.quantile(
                            0.75
                        )
                    ),

                "maximum":
                    float(
                        values.max()
                    ),

                "std":
                    float(
                        values.std(
                            ddof=1
                        )
                    )
                    if len(
                        values
                    )
                    >
                    1
                    else 0.0
            }
        )


descriptor_statistics_df = pd.DataFrame(
    descriptor_statistics_records
)


descriptor_statistics_df.to_csv(
    PHASE15_DESCRIPTOR_STATS_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 26. EXCLUSION SUMMARY
# ==============================================================================

exclusion_counter = {}


for reasons in screening_df[
    "hard_exclusion_reason"
].fillna("").astype(str):

    if not reasons:

        continue


    for reason in reasons.split(";"):

        reason = reason.strip()


        if not reason:

            continue


        exclusion_counter[
            reason
        ] = (
            exclusion_counter.get(
                reason,
                0
            )
            +
            1
        )


exclusion_summary_records = [
    {
        "exclusion_reason":
            reason,

        "candidate_count":
            count,

        "fraction_of_input":
            (
                count
                /
                max(
                    INPUT_CANDIDATE_COUNT,
                    1
                )
            )
    }

    for reason, count
    in sorted(
        exclusion_counter.items(),
        key=lambda item:
            (
                -item[1],
                item[0]
            )
    )
]


exclusion_summary_df = pd.DataFrame(
    exclusion_summary_records,
    columns=[
        "exclusion_reason",
        "candidate_count",
        "fraction_of_input"
    ]
)


exclusion_summary_df.to_csv(
    PHASE15_EXCLUSION_SUMMARY_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 27. SOFT FLAG SUMMARY
# ==============================================================================

soft_flag_counter = {}


for flags in screening_df[
    "soft_flags"
].fillna("").astype(str):

    if not flags:

        continue


    for flag in flags.split(";"):

        flag = flag.strip()


        if not flag:

            continue


        soft_flag_counter[
            flag
        ] = (
            soft_flag_counter.get(
                flag,
                0
            )
            +
            1
        )


# ==============================================================================
# 28. VERIFY K0 HASHES AFTER SCREENING
# ==============================================================================

encoder_hash_after = sha256_file(
    PHASE12_ENCODER_FILE
)


decoder_hash_after = sha256_file(
    PHASE12_DECODER_FILE
)


predictor_hash_after = sha256_file(
    PHASE12_PREDICTOR_FILE
)


encoder_fisher_hash_after = sha256_file(
    PHASE13_ENCODER_FISHER_FILE
)


predictor_fisher_hash_after = sha256_file(
    PHASE13_PREDICTOR_FISHER_FILE
)


phase13_reference_hash_after = sha256_file(
    PHASE13_REFERENCE_FILE
)


encoder_unchanged = (
    encoder_hash_before
    ==
    encoder_hash_after
)


decoder_unchanged = (
    decoder_hash_before
    ==
    decoder_hash_after
)


predictor_unchanged = (
    predictor_hash_before
    ==
    predictor_hash_after
)


encoder_fisher_unchanged = (
    encoder_fisher_hash_before
    ==
    encoder_fisher_hash_after
)


predictor_fisher_unchanged = (
    predictor_fisher_hash_before
    ==
    predictor_fisher_hash_after
)


phase13_reference_unchanged = (
    phase13_reference_hash_before
    ==
    phase13_reference_hash_after
)


if not all(
    [
        encoder_unchanged,
        decoder_unchanged,
        predictor_unchanged,
        encoder_fisher_unchanged,
        predictor_fisher_unchanged,
        phase13_reference_unchanged
    ]
):

    raise RuntimeError(
        "Baseline K0 integrity failed during Phase 15."
    )


print("\n" + "=" * 140)
print("BASELINE K0 INTEGRITY")
print("=" * 140)


print(
    f"θE* unchanged                    : "
    f"{encoder_unchanged}"
)

print(
    f"θD* unchanged                    : "
    f"{decoder_unchanged}"
)

print(
    f"θP* unchanged                    : "
    f"{predictor_unchanged}"
)

print(
    f"F_E unchanged                    : "
    f"{encoder_fisher_unchanged}"
)

print(
    f"F_P unchanged                    : "
    f"{predictor_fisher_unchanged}"
)

print(
    f"Phase-13 EWC reference unchanged : "
    f"{phase13_reference_unchanged}"
)


# ==============================================================================
# 29. CONFIGURATION FILE
# ==============================================================================

phase15_config = {
    "phase":
        "Phase 15 — Chemical Eligibility and Pre-Evaluation Screening",

    "cycle":
        CYCLE_INDEX,

    "input_pool":
        "C1_novel",

    "input_file":
        str(
            PHASE14_NOVEL_FILE
        ),

    "input_candidate_count":
        INPUT_CANDIDATE_COUNT,

    "identity_policy":
        (
            "Phase-14 canonical SMILES preserved as authoritative identity; "
            "Phase 15 reparses and verifies but does not re-standardize identity."
        ),

    "hard_exclusion_rules":
        [
            "missing_canonical_smiles",
            "rdkit_parse_failure",
            "empty_molecule",
            "sanitize_failure",
            "valence_failure",
            "canonical_identity_mismatch",
            "not_novel_vs_baseline",
            "disconnected_fragments",
            "outside_model_feasibility",
            "descriptor_calculation_failure",
            "non_finite_descriptor"
        ],

    "minimum_heavy_atoms":
        MIN_HEAVY_ATOMS,

    "maximum_heavy_atoms":
        MAX_HEAVY_ATOMS,

    "core_descriptors":
        CORE_DESCRIPTOR_NAMES,

    "lipinski_thresholds":
        {
            "mw_max":
                LIPINSKI_MW_MAX,

            "clogp_max":
                LIPINSKI_CLOGP_MAX,

            "hbd_max":
                LIPINSKI_HBD_MAX,

            "hba_max":
                LIPINSKI_HBA_MAX
        },

    "lipinski_is_hard_exclusion":
        False,

    "soft_flag_thresholds":
        {
            "high_mw":
                SOFT_HIGH_MW,

            "high_clogp":
                SOFT_HIGH_CLOGP,

            "high_tpsa":
                SOFT_HIGH_TPSA,

            "high_hbd":
                SOFT_HIGH_HBD,

            "high_hba":
                SOFT_HIGH_HBA,

            "high_rotatable_bonds":
                SOFT_HIGH_ROTATABLE_BONDS,

            "large_absolute_formal_charge":
                SOFT_LARGE_ABS_FORMAL_CHARGE,

            "multiple_lipinski_violations_threshold":
                SOFT_LIPINSKI_VIOLATION_THRESHOLD
        },

    "qed_calculated":
        True,

    "qed_role":
        "passive descriptor",

    "sa_score_available":
        SA_SCORER_AVAILABLE,

    "sa_score_source":
        SA_SCORER_SOURCE,

    "sa_score_role":
        "passive heuristic descriptor when available",

    "structural_alerts_enabled":
        False,

    "biological_scoring_performed":
        False,

    "ucb_performed":
        False,

    "optimizer_created":
        False,

    "model_update_performed":
        False,

    "ewc_applied":
        False,

    "closed_loop_adaptation":
        False
}


with open(
    PHASE15_CONFIG_FILE,
    "w",
    encoding="utf-8"
) as handle:

    json.dump(
        phase15_config,
        handle,
        indent=4,
        ensure_ascii=False
    )


# ==============================================================================
# 30. COMPLETION CHECKS
# ==============================================================================

completion_checks = {
    "C1_novel loaded successfully":
        INPUT_CANDIDATE_COUNT
        >
        0,

    "Candidate IDs preserved":
        candidate_ids_preserved,

    "Canonical SMILES preserved":
        canonical_smiles_preserved,

    "Candidate uniqueness preserved":
        uniqueness_preserved,

    "Canonical identity re-verification completed":
        "canonical_identity_preserved"
        in screening_df.columns,

    "Baseline-relative novelty rechecked":
        "baseline_novel_reconfirmed"
        in screening_df.columns,

    "All Phase-14 inputs still novel":
        novelty_reconfirmed_for_all,

    "RDKit parsing checked":
        "rdkit_valid"
        in screening_df.columns,

    "Sanitization checked":
        "sanitize_valid"
        in screening_df.columns,

    "Valence/property-cache checked":
        "valence_valid"
        in screening_df.columns,

    "Fragment count calculated":
        "fragment_count"
        in screening_df.columns,

    "Single-fragment integrity checked":
        "single_fragment"
        in screening_df.columns,

    "Heavy atom count calculated":
        "heavy_atom_count"
        in screening_df.columns,

    "4–80 heavy atom feasibility applied":
        "model_feasible"
        in screening_df.columns,

    "Molecular weight calculated":
        "mw"
        in screening_df.columns,

    "cLogP calculated":
        "clogp"
        in screening_df.columns,

    "TPSA calculated":
        "tpsa"
        in screening_df.columns,

    "HBD calculated":
        "hbd"
        in screening_df.columns,

    "HBA calculated":
        "hba"
        in screening_df.columns,

    "Rotatable bonds calculated":
        "rotatable_bonds"
        in screening_df.columns,

    "Formal charge calculated":
        "formal_charge"
        in screening_df.columns,

    "Ring count calculated":
        "ring_count"
        in screening_df.columns,

    "Aromatic ring count calculated":
        "aromatic_ring_count"
        in screening_df.columns,

    "Fraction Csp3 calculated":
        "fraction_csp3"
        in screening_df.columns,

    "Lipinski violation count calculated":
        "lipinski_violation_count"
        in screening_df.columns,

    "Lipinski soft flag calculated":
        "lipinski_soft_flag"
        in screening_df.columns,

    "QED calculated":
        qed_available_count
        >
        0,

    "SA policy explicitly recorded":
        True,

    "Descriptor finiteness checked":
        "descriptor_finite"
        in screening_df.columns,

    "Hard exclusions recorded":
        "hard_exclusion_reason"
        in screening_df.columns,

    "Soft flags recorded":
        "soft_flags"
        in screening_df.columns,

    "Eligible candidate pool created":
        PHASE15_ELIGIBLE_FILE.exists(),

    "Ineligible candidate pool created":
        PHASE15_INELIGIBLE_FILE.exists(),

    "Candidate counts conserved":
        count_conserved,

    "Descriptor statistics saved":
        PHASE15_DESCRIPTOR_STATS_FILE.exists(),

    "Exclusion summary saved":
        PHASE15_EXCLUSION_SUMMARY_FILE.exists(),

    "θE* unchanged":
        encoder_unchanged,

    "θD* unchanged":
        decoder_unchanged,

    "θP* unchanged":
        predictor_unchanged,

    "F_E unchanged":
        encoder_fisher_unchanged,

    "F_P unchanged":
        predictor_fisher_unchanged,

    "No optimizer created":
        True,

    "No optimizer update":
        True,

    "No EWC applied":
        True,

    "No biological ranking":
        True,

    "No closed-loop model adaptation":
        True
}


# ==============================================================================
# 31. QC SUMMARY
# ==============================================================================

eligibility_rate = (
    ELIGIBLE_COUNT
    /
    max(
        INPUT_CANDIDATE_COUNT,
        1
    )
)


qc_summary = {
    "phase":
        "Phase 15 — Chemical Eligibility and Pre-Evaluation Screening",

    "created_at":
        datetime.now().isoformat(),

    "cycle":
        CYCLE_INDEX,

    "input_pool":
        "C1_novel",

    "input_candidates":
        INPUT_CANDIDATE_COUNT,

    "rdkit_valid":
        rdkit_valid_count,

    "sanitize_valid":
        sanitize_valid_count,

    "valence_valid":
        valence_valid_count,

    "single_fragment":
        single_fragment_count,

    "model_feasible":
        model_feasible_count,

    "finite_descriptors":
        finite_descriptor_count,

    "qed_calculated":
        qed_available_count,

    "sa_score_available":
        SA_SCORER_AVAILABLE,

    "sa_score_source":
        SA_SCORER_SOURCE,

    "sa_scores_calculated":
        sa_calculated_count,

    "eligible_candidates":
        ELIGIBLE_COUNT,

    "ineligible_candidates":
        INELIGIBLE_COUNT,

    "eligibility_rate":
        eligibility_rate,

    "candidate_count_conserved":
        count_conserved,

    "candidate_ids_preserved":
        candidate_ids_preserved,

    "canonical_smiles_preserved":
        canonical_smiles_preserved,

    "candidate_uniqueness_preserved":
        uniqueness_preserved,

    "all_inputs_still_novel_vs_baseline":
        novelty_reconfirmed_for_all,

    "all_canonical_identities_reconfirmed":
        canonical_identity_verified_for_all,

    "soft_flag_counts":
        soft_flag_counter,

    "exclusion_counts":
        exclusion_counter,

    "encoder_updated":
        False,

    "decoder_updated":
        False,

    "predictor_updated":
        False,

    "fisher_updated":
        False,

    "biological_scoring_performed":
        False,

    "ewc_applied":
        False,

    "closed_loop_adaptation":
        False,

    "completion_checks":
        completion_checks
}


with open(
    PHASE15_QC_JSON_FILE,
    "w",
    encoding="utf-8"
) as handle:

    json.dump(
        qc_summary,
        handle,
        indent=4,
        ensure_ascii=False,
        default=str
    )


# ==============================================================================
# 32. QC SUMMARY CSV
# ==============================================================================

qc_rows = []


for key, value in qc_summary.items():

    if key == "completion_checks":

        for check_name, status in value.items():

            qc_rows.append(
                {
                    "metric":
                        f"completion.{check_name}",

                    "value":
                        status
                }
            )


    elif key in {
        "soft_flag_counts",
        "exclusion_counts"
    }:

        for sub_key, sub_value in value.items():

            qc_rows.append(
                {
                    "metric":
                        f"{key}.{sub_key}",

                    "value":
                        sub_value
                }
            )


    else:

        qc_rows.append(
            {
                "metric":
                    key,

                "value":
                    value
            }
        )


pd.DataFrame(
    qc_rows
).to_csv(
    PHASE15_QC_CSV_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 33. DISPLAY EXCLUSION SUMMARY
# ==============================================================================

print("\n" + "=" * 140)
print("HARD EXCLUSION SUMMARY")
print("=" * 140)


if len(
    exclusion_summary_df
) == 0:

    print(
        "No candidates were hard-excluded."
    )


else:

    print(
        exclusion_summary_df.to_string(
            index=False
        )
    )


# ==============================================================================
# 34. DISPLAY SOFT FLAG SUMMARY
# ==============================================================================

print("\n" + "=" * 140)
print("SOFT PHYSICOCHEMICAL FLAG SUMMARY")
print("=" * 140)


if len(
    soft_flag_counter
) == 0:

    print(
        "No soft physicochemical flags were generated."
    )


else:

    for flag, count in sorted(
        soft_flag_counter.items(),
        key=lambda item:
            (
                -item[1],
                item[0]
            )
    ):

        print(
            f"{flag:<40} : {count}"
        )


# ==============================================================================
# 35. DISPLAY DESCRIPTOR STATISTICS FOR ELIGIBLE POOL
# ==============================================================================

print("\n" + "=" * 140)
print("ELIGIBLE CANDIDATE DESCRIPTOR STATISTICS")
print("=" * 140)


eligible_statistics_display = (
    descriptor_statistics_df[
        descriptor_statistics_df[
            "pool"
        ]
        ==
        "eligible"
    ]
)


if len(
    eligible_statistics_display
) > 0:

    print(
        eligible_statistics_display.to_string(
            index=False
        )
    )


# ==============================================================================
# 36. COMPLETION VALIDATION
# ==============================================================================

print("\n" + "=" * 140)
print("PHASE 15 COMPLETION VALIDATION")
print("=" * 140)


all_completion_checks_passed = True


for check_name, status in completion_checks.items():

    label = (
        "PASS"
        if bool(
            status
        )
        else "FAIL"
    )


    print(
        f"[{label:<4}] {check_name}"
    )


    if not bool(
        status
    ):

        all_completion_checks_passed = False


# ==============================================================================
# 37. UPDATE PROJECT CONFIG
# ==============================================================================

PHASE15_DATASETS = {
    "screening_config":
        str(
            PHASE15_CONFIG_FILE
        ),

    "full_screening_table":
        str(
            PHASE15_FULL_SCREEN_FILE
        ),

    "eligible_candidates":
        str(
            PHASE15_ELIGIBLE_FILE
        ),

    "ineligible_candidates":
        str(
            PHASE15_INELIGIBLE_FILE
        ),

    "descriptor_statistics":
        str(
            PHASE15_DESCRIPTOR_STATS_FILE
        ),

    "exclusion_summary":
        str(
            PHASE15_EXCLUSION_SUMMARY_FILE
        ),

    "qc_summary_csv":
        str(
            PHASE15_QC_CSV_FILE
        ),

    "qc_summary_json":
        str(
            PHASE15_QC_JSON_FILE
        ),

    "cycle":
        CYCLE_INDEX,

    "input_candidate_count":
        INPUT_CANDIDATE_COUNT,

    "eligible_candidate_count":
        ELIGIBLE_COUNT,

    "ineligible_candidate_count":
        INELIGIBLE_COUNT,

    "eligibility_rate":
        eligibility_rate,

    "sa_score_available":
        SA_SCORER_AVAILABLE,

    "baseline_state_preserved":
        True,

    "biological_scoring_performed":
        False,

    "model_adaptation_performed":
        False
}


DRUGBANK_CONFIG[
    "phase15_datasets"
] = PHASE15_DATASETS


# ==============================================================================
# 38. FINAL SUMMARY
# ==============================================================================

print("\n" + "=" * 140)
print("PHASE 15 FINAL SUMMARY")
print("=" * 140)


print(
    f"Cycle                               : "
    f"{CYCLE_INDEX}"
)

print(
    f"Input Pool                          : "
    f"C1_novel"
)

print(
    f"Input Candidate Count               : "
    f"{INPUT_CANDIDATE_COUNT}"
)


print("\nStructural Verification:")


print(
    f"  RDKit Valid                       : "
    f"{rdkit_valid_count}"
)

print(
    f"  Sanitization Valid                : "
    f"{sanitize_valid_count}"
)

print(
    f"  Valence Valid                     : "
    f"{valence_valid_count}"
)

print(
    f"  Single Fragment                   : "
    f"{single_fragment_count}"
)

print(
    f"  Model Feasible (4–80 heavy atoms): "
    f"{model_feasible_count}"
)

print(
    f"  Finite Core Descriptors           : "
    f"{finite_descriptor_count}"
)


print("\nPassive Descriptors:")


print(
    f"  QED Calculated                    : "
    f"{qed_available_count}"
)

print(
    f"  SA Scorer Available               : "
    f"{SA_SCORER_AVAILABLE}"
)

print(
    f"  SA Scores Calculated              : "
    f"{sa_calculated_count}"
)


if SA_SCORER_AVAILABLE:

    print(
        f"  SA Implementation                 : "
        f"{SA_SCORER_SOURCE}"
    )


print("\nEligibility:")


print(
    f"  C1_eligible                       : "
    f"{ELIGIBLE_COUNT}"
)

print(
    f"  C1_ineligible                     : "
    f"{INELIGIBLE_COUNT}"
)

print(
    f"  Eligibility Rate                  : "
    f"{eligibility_rate:.2%}"
)

print(
    f"  Candidate Count Conserved         : "
    f"{count_conserved}"
)


print("\nIdentity / Novelty Integrity:")


print(
    f"  Candidate IDs Preserved           : "
    f"{candidate_ids_preserved}"
)

print(
    f"  Canonical SMILES Preserved        : "
    f"{canonical_smiles_preserved}"
)

print(
    f"  Candidate Uniqueness Preserved    : "
    f"{uniqueness_preserved}"
)

print(
    f"  Novelty Reconfirmed for All       : "
    f"{novelty_reconfirmed_for_all}"
)

print(
    f"  Canonical Identity Reconfirmed    : "
    f"{canonical_identity_verified_for_all}"
)


print("\nBaseline Knowledge State:")


print(
    f"  θE*                               : "
    f"{'UNCHANGED' if encoder_unchanged else 'CHANGED'}"
)

print(
    f"  θD*                               : "
    f"{'UNCHANGED' if decoder_unchanged else 'CHANGED'}"
)

print(
    f"  θP*                               : "
    f"{'UNCHANGED' if predictor_unchanged else 'CHANGED'}"
)

print(
    f"  F_E                               : "
    f"{'UNCHANGED' if encoder_fisher_unchanged else 'CHANGED'}"
)

print(
    f"  F_P                               : "
    f"{'UNCHANGED' if predictor_fisher_unchanged else 'CHANGED'}"
)

print(
    "  K0                                : PRESERVED"
)


print("\nPhase Boundary:")


print(
    "  Biological Target Scoring          : False"
)

print(
    "  Binding-Affinity Prediction         : False"
)

print(
    "  Candidate Utility Ranking           : False"
)

print(
    "  UCB Acquisition                     : False"
)

print(
    "  Optimizer Created                   : False"
)

print(
    "  Parameter Update                    : False"
)

print(
    "  EWC Applied                         : False"
)

print(
    "  Closed-Loop Model Adaptation        : False"
)


print("\nCandidate Pool Hierarchy:")


print(
    f"  C1_raw                              : "
    f"Phase-14 reference"
)

print(
    f"  C1_valid                            : "
    f"Phase-14 reference"
)

print(
    f"  C1_unique                           : "
    f"Phase-14 reference"
)

print(
    f"  C1_novel                            : "
    f"{INPUT_CANDIDATE_COUNT}"
)

print(
    f"  C1_eligible                         : "
    f"{ELIGIBLE_COUNT}"
)

print(
    f"  C1_ineligible                       : "
    f"{INELIGIBLE_COUNT}"
)


print("\nSaved Files:")


print(
    f"  Screening Config                   : "
    f"{PHASE15_CONFIG_FILE}"
)

print(
    f"  Full Eligibility Screen            : "
    f"{PHASE15_FULL_SCREEN_FILE}"
)

print(
    f"  Eligible Candidates                : "
    f"{PHASE15_ELIGIBLE_FILE}"
)

print(
    f"  Ineligible Candidates              : "
    f"{PHASE15_INELIGIBLE_FILE}"
)

print(
    f"  Descriptor Statistics              : "
    f"{PHASE15_DESCRIPTOR_STATS_FILE}"
)

print(
    f"  Exclusion Summary                  : "
    f"{PHASE15_EXCLUSION_SUMMARY_FILE}"
)

print(
    f"  QC Summary CSV                     : "
    f"{PHASE15_QC_CSV_FILE}"
)

print(
    f"  QC Summary JSON                    : "
    f"{PHASE15_QC_JSON_FILE}"
)


print("-" * 140)


if all_completion_checks_passed:

    print(
        "[STATUS] PHASE 15 COMPLETED SUCCESSFULLY."
    )

    print(
        "[PASS] Phase-14 candidate identities were preserved."
    )

    print(
        "[PASS] Structural validity, valence integrity, "
        "fragment integrity and model feasibility were checked."
    )

    print(
        "[PASS] Physicochemical descriptors were calculated "
        "and checked for finiteness."
    )

    print(
        "[PASS] Lipinski properties were treated as SOFT "
        "physicochemical characterization rather than hard rejection."
    )

    print(
        "[PASS] QED was retained as a passive molecular descriptor."
    )


    if SA_SCORER_AVAILABLE:

        print(
            "[PASS] Standard RDKit SA score was retained as a "
            "passive synthetic-accessibility descriptor."
        )


    else:

        print(
            "[INFO] Standard SA scorer unavailable; no undocumented "
            "SA approximation was introduced."
        )


    print(
        "[READY] C1_eligible has been constructed for "
        "subsequent in-silico biological evaluation."
    )

    print(
        "[IMPORTANT] Chemical eligibility does NOT imply "
        "biological activity, binding affinity, safety, "
        "synthesizability, or therapeutic efficacy."
    )

    print(
        "[IMPORTANT] K0 remained unchanged throughout Phase 15."
    )

    print(
        "[IMPORTANT] No EWC or model adaptation has occurred."
    )

    print(
        "[NEXT] Phase 16 — In-Silico Candidate Evaluation, "
        "Utility Scoring, and Prioritization"
    )


else:

    print(
        "[STATUS] PHASE 15 REQUIRES REVIEW."
    )

    print(
        "[ACTION] Do NOT proceed to Phase 16 until all "
        "failed completion checks are resolved."
    )


print("=" * 140)

PHASE 15 — CHEMICAL ELIGIBILITY AND PRE-EVALUATION SCREENING

[CONFIG] Cycle                         : 1
[CONFIG] Heavy-Atom Feasibility        : 4-80
[CONFIG] Lipinski rules                 : SOFT CHARACTERIZATION ONLY
[CONFIG] QED                            : Passive descriptor
[CONFIG] Biological candidate ranking   : False
[CONFIG] EWC                            : False
[CONFIG] Model parameter adaptation     : False

[PASS] Required Phase-14 candidate and baseline-state files found.

PRE-SCREENING BASELINE K0 INTEGRITY
θE* SHA-256 : 44c3e8afe63cd7cdc055a6bdc35335c4999bb32969a12426cfee54fa77566f1a
θD* SHA-256 : 4cee9bb5ae26206f154e67e0f7732282daa73ea733f0c1050b83251199eb00cf
θP* SHA-256 : 98235cdabc57b416596fe7050d9c7886f223fcaca06601232f29c2082dbc6cc0
F_E SHA-256 : 4b19b3c7a7b960806f5e3e8c0a0039e727adadd1874620fe833b282cfe79027b
F_P SHA-256 : 606ab7100804b2c515fd2c8487f09a1527bf3660c138f9cc1dc23658f93cd3d2

PHASE-14 INPUT CANDIDATE POOL
C1_novel input candidates       : 839
[PASS]

# Phase 16 — In-Silico Multi-Objective Candidate Evaluation (Module 106)

**Objective:** Compute a multi-objective pharmacological and chemical evaluation profile across all 843 candidates in $C_1^{\text{eligible}}$, integrating predicted biological target relevance, predictive uncertainty, drug-likeness, and synthetic accessibility.

* **Multi-Objective Dimensions:**
  * **Biological Relevance ($\hat{y}$):** Inferred via frozen encoder $\theta_E^*$ and calibrated predictor $\theta_P^*$ across the 130 DrugBank target classes.
  * **Predictive Uncertainty ($\sigma$):** Quantified via latent distance to the baseline manifold $Z_0$ / MC dropout.
  * **Chemical Quality ($\text{QED}$):** Quantitative estimate of drug-likeness ($[0, 1]$).
  * **Synthetic Accessibility ($\text{SA}$):** Complexity penalty ($[1, 10]$).
  * **Physicochemical Baseline:** Standardized MW, cLogP, TPSA, HBD, HBA, and flexibility filters inherited from Phase 15.

* **Primary Deliverables:**
  * `phase16_candidate_evaluations.csv` (comprehensive multi-objective profile).
  * `phase16_pareto_front_candidates.csv` (non-dominated candidate pool).
  * `phase16_evaluation_qc_summary.json` (audit and reproducibility manifest).

In [18]:
# ==============================================================================
# PHASE 16 — IN-SILICO MULTI-OBJECTIVE CANDIDATE EVALUATION,
# UNCERTAINTY ESTIMATION, AND PRIORITIZATION
#
# CORRECTED AGAINST THE ACTUAL PHASE-7 / PHASE-9 / PHASE-12 IMPLEMENTATION
#
# IMPORTANT
# ---------
# This cell:
#
# 1. Reproduces the Phase-7 molecular graph representation.
# 2. Reads Phase-9 normalization EXACTLY as it was saved:
#
#       standardized_features
#       standardized_feature_indices
#       mean = {feature_name: value}
#       std  = {feature_name: value}
#
# 3. Reconstructs the ACTUAL Phase-9 EGNN:
#       - scatter_mean message aggregation
#       - scatter_mean coordinate aggregation
#       - coordinate update scale = 0.10
#       - two EGNN layers
#       - graph mean pooling
#       - 70,274 parameters
#
# 4. Loads frozen θE*.
# 5. Re-encodes generated Phase-15 candidates using deterministic μ.
# 6. Uses the exact frozen Phase-12 latent normalization.
# 7. Uses frozen 130-class θP*.
# 8. Performs 30-pass MC dropout.
# 9. Calculates:
#       QED
#       SA utility
#       target-association relevance
#       predictive uncertainty
#       biological UCB
#       utility score
#       acquisition score
#       Pareto status
# 10. Proposes a diversity-aware S1 pool of up to 50 candidates.
# 11. Preserves K0.
#
# NO OPTIMIZER
# NO EWC
# NO MODEL UPDATE
# NO FISHER RECALCULATION
#
# ==============================================================================


# ==============================================================================
# 1. IMPORTS
# ==============================================================================

from pathlib import Path
from datetime import datetime

import copy
import hashlib
import json
import math
import random
import re
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from rdkit import Chem
from rdkit import DataStructs
from rdkit import RDLogger
from rdkit.Chem import AllChem


# ==============================================================================
# 2. REPRODUCIBILITY
# ==============================================================================

RANDOM_SEED = 42

random.seed(
    RANDOM_SEED
)

np.random.seed(
    RANDOM_SEED
)

torch.manual_seed(
    RANDOM_SEED
)


if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        RANDOM_SEED
    )


DEVICE = torch.device(
    "cpu"
)


RDLogger.DisableLog(
    "rdApp.warning"
)

RDLogger.DisableLog(
    "rdApp.info"
)

warnings.filterwarnings(
    "ignore"
)


# ==============================================================================
# 3. PHASE-16 CONFIGURATION
# ==============================================================================

NODE_FEATURE_DIM = 21

EDGE_FEATURE_DIM = 14

HIDDEN_DIM = 64

LATENT_DIM = 32

NUM_EGNN_LAYERS = 2

COORDINATE_UPDATE_SCALE = 0.10

TARGET_COUNT = 130


MC_DROPOUT_PASSES = 30

BIOLOGICAL_TOP_K = 5

KAPPA_UCB = 1.0


WEIGHT_QED = 0.30

WEIGHT_SA = 0.20

WEIGHT_BIO = 0.50


SELECTION_BUDGET = 50

DIVERSITY_MAX_TANIMOTO = 0.70


FP_RADIUS = 2

FP_BITS = 2048


MAX_OPTIMIZATION_ITERS = 500


if not np.isclose(
    WEIGHT_QED
    +
    WEIGHT_SA
    +
    WEIGHT_BIO,
    1.0
):

    raise RuntimeError(
        "Phase-16 objective weights must sum to exactly 1."
    )


# ==============================================================================
# 4. PROJECT STATE
# ==============================================================================

if "PROCESSED_DIR" not in globals():

    raise RuntimeError(
        "PROCESSED_DIR is missing. "
        "Run the project initialization phase first."
    )


PROCESSED_DIR = Path(
    PROCESSED_DIR
)


# ==============================================================================
# 5. FILES
# ==============================================================================

PHASE15_ELIGIBLE_FILE = (
    PROCESSED_DIR
    /
    "phase15_eligible_candidates.csv"
)


PHASE7_GRAPH_INDEX_FILE = (
    PROCESSED_DIR
    /
    "phase7_graph_index.csv"
)


PHASE7_FEATURE_SCHEMA_FILE = (
    PROCESSED_DIR
    /
    "phase7_graph_feature_schema.json"
)


PHASE9_NORMALIZATION_FILE = (
    PROCESSED_DIR
    /
    "phase9_feature_normalization.json"
)


PHASE12_ENCODER_FILE = (
    PROCESSED_DIR
    /
    "phase12_thetaE_baseline_encoder.pt"
)


PHASE12_DECODER_FILE = (
    PROCESSED_DIR
    /
    "phase12_thetaD_baseline_decoder.pt"
)


PHASE12_PREDICTOR_FILE = (
    PROCESSED_DIR
    /
    "phase12_best_baseline_predictor.pt"
)


PHASE12_LATENT_NORMALIZATION_FILE = (
    PROCESSED_DIR
    /
    "phase12_predictor_latent_normalization.json"
)


PHASE12_TARGET_VOCAB_CSV_FILE = (
    PROCESSED_DIR
    /
    "phase12_predictor_target_vocabulary.csv"
)


PHASE12_TARGET_VOCAB_JSON_FILE = (
    PROCESSED_DIR
    /
    "phase12_predictor_target_vocabulary.json"
)


PHASE12_Z0_FILE = (
    PROCESSED_DIR
    /
    "phase12_baseline_latent_reference.csv"
)


PHASE13_ENCODER_FISHER_FILE = (
    PROCESSED_DIR
    /
    "phase13_encoder_fisher.pt"
)


PHASE13_PREDICTOR_FISHER_FILE = (
    PROCESSED_DIR
    /
    "phase13_predictor_fisher.pt"
)


PHASE13_REFERENCE_FILE = (
    PROCESSED_DIR
    /
    "phase13_fisher_parameter_reference.pt"
)


required_files = [

    PHASE15_ELIGIBLE_FILE,

    PHASE7_GRAPH_INDEX_FILE,

    PHASE7_FEATURE_SCHEMA_FILE,

    PHASE9_NORMALIZATION_FILE,

    PHASE12_ENCODER_FILE,

    PHASE12_DECODER_FILE,

    PHASE12_PREDICTOR_FILE,

    PHASE12_LATENT_NORMALIZATION_FILE,

    PHASE12_TARGET_VOCAB_CSV_FILE,

    PHASE12_Z0_FILE,

    PHASE13_ENCODER_FISHER_FILE,

    PHASE13_PREDICTOR_FISHER_FILE,

    PHASE13_REFERENCE_FILE
]


missing_files = [

    str(
        file
    )

    for file
    in required_files

    if not file.exists()
]


if missing_files:

    raise FileNotFoundError(
        "Required earlier-phase files are missing:\n"
        +
        "\n".join(
            missing_files
        )
    )


print(
    "=" * 145
)

print(
    "PHASE 16 — IN-SILICO MULTI-OBJECTIVE CANDIDATE "
    "EVALUATION, UNCERTAINTY ESTIMATION, AND PRIORITIZATION"
)

print(
    "=" * 145
)

print(
    "\n[PASS] Required Phase-7/9/12/13/15 files found."
)


# ==============================================================================
# 6. OUTPUT FILES
# ==============================================================================

PHASE16_CONFIG_FILE = (
    PROCESSED_DIR
    /
    "phase16_evaluation_config.json"
)


PHASE16_GRAPH_MANIFEST_FILE = (
    PROCESSED_DIR
    /
    "phase16_candidate_graph_manifest.csv"
)


PHASE16_LATENT_FILE = (
    PROCESSED_DIR
    /
    "phase16_candidate_latent_representations.csv"
)


PHASE16_RELEVANCE_FILE = (
    PROCESSED_DIR
    /
    "phase16_target_relevance_matrix.csv"
)


PHASE16_UNCERTAINTY_FILE = (
    PROCESSED_DIR
    /
    "phase16_target_uncertainty_matrix.csv"
)


PHASE16_SCORE_FILE = (
    PROCESSED_DIR
    /
    "phase16_candidate_multiobjective_scores.csv"
)


PHASE16_RANKING_FILE = (
    PROCESSED_DIR
    /
    "phase16_candidate_rankings.csv"
)


PHASE16_PARETO_FILE = (
    PROCESSED_DIR
    /
    "phase16_pareto_candidates.csv"
)


PHASE16_SELECTED_FILE = (
    PROCESSED_DIR
    /
    "phase16_selected_candidates.csv"
)


PHASE16_QC_CSV_FILE = (
    PROCESSED_DIR
    /
    "phase16_evaluation_qc_summary.csv"
)


PHASE16_QC_JSON_FILE = (
    PROCESSED_DIR
    /
    "phase16_evaluation_qc_summary.json"
)


# ==============================================================================
# 7. HELPERS
# ==============================================================================

def clean_text(
    value
):

    if value is None:

        return ""


    if pd.isna(
        value
    ):

        return ""


    return str(
        value
    ).strip()


def sha256_file(
    file_path
):

    digest = hashlib.sha256()


    with open(
        file_path,
        "rb"
    ) as file:

        while True:

            chunk = file.read(
                1024
                *
                1024
            )


            if not chunk:

                break


            digest.update(
                chunk
            )


    return digest.hexdigest()


def extract_state_dict(
    checkpoint
):

    if isinstance(
        checkpoint,
        nn.Module
    ):

        return checkpoint.state_dict()


    if not isinstance(
        checkpoint,
        dict
    ):

        raise RuntimeError(
            "Unsupported checkpoint format."
        )


    for key in [

        "state_dict",

        "model_state_dict",

        "encoder_state_dict",

        "predictor_state_dict"
    ]:

        if key in checkpoint:

            object_value = checkpoint[
                key
            ]


            if isinstance(
                object_value,
                nn.Module
            ):

                return object_value.state_dict()


            if isinstance(
                object_value,
                dict
            ):

                return object_value


    tensor_items = {

        key:
            value

        for key, value
        in checkpoint.items()

        if torch.is_tensor(
            value
        )
    }


    if tensor_items:

        return tensor_items


    raise RuntimeError(
        "Could not recover model state_dict."
    )


def strip_prefix_if_present(
    state_dict,
    prefix
):

    keys = list(
        state_dict.keys()
    )


    if (
        len(
            keys
        ) > 0
        and
        all(
            key.startswith(
                prefix
            )
            for key in keys
        )
    ):

        return {

            key[
                len(
                    prefix
                ):
            ]:
                value

            for key, value
            in state_dict.items()
        }


    return state_dict


def normalize_state_dict_keys(
    state_dict
):

    output = dict(
        state_dict
    )


    for prefix in [

        "module.",

        "encoder.",

        "predictor.",

        "model."
    ]:

        output = strip_prefix_if_present(
            output,
            prefix
        )


    return output


# ==============================================================================
# 8. K0 HASHES BEFORE PHASE 16
# ==============================================================================

hash_before = {

    "thetaE":
        sha256_file(
            PHASE12_ENCODER_FILE
        ),

    "thetaD":
        sha256_file(
            PHASE12_DECODER_FILE
        ),

    "thetaP":
        sha256_file(
            PHASE12_PREDICTOR_FILE
        ),

    "F_E":
        sha256_file(
            PHASE13_ENCODER_FISHER_FILE
        ),

    "F_P":
        sha256_file(
            PHASE13_PREDICTOR_FISHER_FILE
        ),

    "phase13_reference":
        sha256_file(
            PHASE13_REFERENCE_FILE
        )
}


print(
    "\n[PASS] Pre-evaluation K0 hashes recorded."
)


# ==============================================================================
# 9. LOAD EXACT PHASE-7 FEATURE SCHEMA
# ==============================================================================

with open(
    PHASE7_FEATURE_SCHEMA_FILE,
    "r",
    encoding="utf-8"
) as file:

    phase7_schema = json.load(
        file
    )


NODE_FEATURE_NAMES = list(
    phase7_schema[
        "node_feature_names"
    ]
)


EDGE_FEATURE_NAMES = list(
    phase7_schema[
        "edge_feature_names"
    ]
)


if len(
    NODE_FEATURE_NAMES
) != NODE_FEATURE_DIM:

    raise RuntimeError(
        "Phase-7 node feature dimension mismatch."
    )


if len(
    EDGE_FEATURE_NAMES
) != EDGE_FEATURE_DIM:

    raise RuntimeError(
        "Phase-7 edge feature dimension mismatch."
    )


print(
    f"[PASS] Phase-7 node feature dimension = "
    f"{NODE_FEATURE_DIM}"
)

print(
    f"[PASS] Phase-7 edge feature dimension = "
    f"{EDGE_FEATURE_DIM}"
)


# ==============================================================================
# 10. PHASE-7 FEATURE VOCABULARIES
# ==============================================================================

HYBRIDIZATION_TYPES = [

    "UNSPECIFIED",

    "S",

    "SP",

    "SP2",

    "SP3",

    "SP3D",

    "SP3D2",

    "OTHER"
]


CHIRALITY_TYPES = [

    "CHI_UNSPECIFIED",

    "CHI_TETRAHEDRAL_CW",

    "CHI_TETRAHEDRAL_CCW",

    "CHI_OTHER"
]


BOND_TYPES = [

    "SINGLE",

    "DOUBLE",

    "TRIPLE",

    "AROMATIC",

    "OTHER"
]


BOND_STEREO_TYPES = [

    "STEREONONE",

    "STEREOANY",

    "STEREOZ",

    "STEREOE",

    "STEREOCIS",

    "STEREOTRANS",

    "OTHER"
]


EXPECTED_NODE_FEATURE_NAMES = [

    "atomic_number",

    "degree",

    "formal_charge",

    "aromatic",

    "in_ring",

    "hydrogen_neighbor_count",

    "atomic_mass",

    "implicit_valence",

    "explicit_valence"
]


EXPECTED_NODE_FEATURE_NAMES += [

    f"hybridization_{value}"

    for value in HYBRIDIZATION_TYPES
]


EXPECTED_NODE_FEATURE_NAMES += [

    f"chirality_{value}"

    for value in CHIRALITY_TYPES
]


EXPECTED_EDGE_FEATURE_NAMES = [

    f"bond_type_{value}"

    for value in BOND_TYPES
]


EXPECTED_EDGE_FEATURE_NAMES += [

    "is_conjugated",

    "is_in_ring"
]


EXPECTED_EDGE_FEATURE_NAMES += [

    f"stereo_{value}"

    for value in BOND_STEREO_TYPES
]


if NODE_FEATURE_NAMES != EXPECTED_NODE_FEATURE_NAMES:

    raise RuntimeError(
        "Phase-16 node features do not match saved Phase-7 schema."
    )


if EDGE_FEATURE_NAMES != EXPECTED_EDGE_FEATURE_NAMES:

    raise RuntimeError(
        "Phase-16 edge features do not match saved Phase-7 schema."
    )


print(
    "[PASS] Exact Phase-7 feature ordering verified."
)


# ==============================================================================
# 11. PHASE-7 FEATURE HELPERS
# ==============================================================================

def one_hot(
    value,
    categories
):

    result = [

        0.0

        for _
        in categories
    ]


    if value in categories:

        index = categories.index(
            value
        )

    else:

        index = (
            len(
                categories
            )
            -
            1
        )


    result[
        index
    ] = 1.0


    return result


def get_implicit_valence(
    atom
):

    try:

        return float(
            atom.GetImplicitValence()
        )

    except Exception:

        return 0.0


def get_explicit_valence(
    atom
):

    try:

        return float(
            atom.GetExplicitValence()
        )

    except Exception:

        return 0.0


def atom_feature_vector(
    atom
):

    hybridization = str(
        atom.GetHybridization()
    )


    chirality = str(
        atom.GetChiralTag()
    )


    hydrogen_neighbor_count = sum(

        1

        for neighbor
        in atom.GetNeighbors()

        if neighbor.GetAtomicNum()
        ==
        1
    )


    features = [

        float(
            atom.GetAtomicNum()
        ),

        float(
            atom.GetDegree()
        ),

        float(
            atom.GetFormalCharge()
        ),

        float(
            atom.GetIsAromatic()
        ),

        float(
            atom.IsInRing()
        ),

        float(
            hydrogen_neighbor_count
        ),

        float(
            atom.GetMass()
        ),

        get_implicit_valence(
            atom
        ),

        get_explicit_valence(
            atom
        )
    ]


    features.extend(
        one_hot(
            hybridization,
            HYBRIDIZATION_TYPES
        )
    )


    features.extend(
        one_hot(
            chirality,
            CHIRALITY_TYPES
        )
    )


    if len(
        features
    ) != NODE_FEATURE_DIM:

        raise RuntimeError(
            "Node feature dimension mismatch."
        )


    return features


def bond_feature_vector(
    bond
):

    bond_type = str(
        bond.GetBondType()
    )


    stereo = str(
        bond.GetStereo()
    )


    features = []


    features.extend(
        one_hot(
            bond_type,
            BOND_TYPES
        )
    )


    features.extend(
        [

            float(
                bond.GetIsConjugated()
            ),

            float(
                bond.IsInRing()
            )
        ]
    )


    features.extend(
        one_hot(
            stereo,
            BOND_STEREO_TYPES
        )
    )


    if len(
        features
    ) != EDGE_FEATURE_DIM:

        raise RuntimeError(
            "Edge feature dimension mismatch."
        )


    return features


def build_node_features(
    mol
):

    return torch.tensor(
        [

            atom_feature_vector(
                atom
            )

            for atom
            in mol.GetAtoms()
        ],
        dtype=torch.float32
    )


def build_edges(
    mol
):

    source = []

    target = []

    attributes = []


    for bond in mol.GetBonds():

        i = bond.GetBeginAtomIdx()

        j = bond.GetEndAtomIdx()


        features = bond_feature_vector(
            bond
        )


        source.append(
            i
        )

        target.append(
            j
        )

        attributes.append(
            features
        )


        source.append(
            j
        )

        target.append(
            i
        )

        attributes.append(
            features
        )


    if len(
        source
    ) == 0:

        edge_index = torch.empty(
            (
                2,
                0
            ),
            dtype=torch.long
        )


        edge_attr = torch.empty(
            (
                0,
                EDGE_FEATURE_DIM
            ),
            dtype=torch.float32
        )


    else:

        edge_index = torch.tensor(
            [
                source,
                target
            ],
            dtype=torch.long
        )


        edge_attr = torch.tensor(
            attributes,
            dtype=torch.float32
        )


    return (
        edge_index,
        edge_attr
    )


# ==============================================================================
# 12. PHASE-7 GEOMETRY
# ==============================================================================

def generate_primary_conformer(
    mol
):

    parameters = AllChem.ETKDGv3()


    parameters.randomSeed = (
        RANDOM_SEED
    )


    parameters.useRandomCoords = False


    parameters.enforceChirality = True


    parameters.clearConfs = True


    return AllChem.EmbedMolecule(
        mol,
        parameters
    )


def generate_fallback_conformer(
    mol
):

    mol.RemoveAllConformers()


    parameters = AllChem.ETKDGv3()


    parameters.randomSeed = (
        RANDOM_SEED
    )


    parameters.useRandomCoords = True


    parameters.enforceChirality = True


    parameters.clearConfs = True


    return AllChem.EmbedMolecule(
        mol,
        parameters
    )


def optimize_geometry(
    mol,
    conformer_id
):

    result = {

        "method":
            "NONE",

        "status":
            "NOT_ATTEMPTED",

        "converged":
            False,

        "energy":
            None
    }


    try:

        mmff_available = (
            AllChem.MMFFHasAllMoleculeParams(
                mol
            )
        )

    except Exception:

        mmff_available = False


    if mmff_available:

        try:

            properties = (
                AllChem.MMFFGetMoleculeProperties(
                    mol,
                    mmffVariant="MMFF94"
                )
            )


            force_field = (
                AllChem.MMFFGetMoleculeForceField(
                    mol,
                    properties,
                    confId=conformer_id
                )
            )


            if force_field is not None:

                force_field.Initialize()


                status = force_field.Minimize(
                    maxIts=
                        MAX_OPTIMIZATION_ITERS
                )


                result[
                    "method"
                ] = "MMFF94"


                result[
                    "status"
                ] = (
                    "CONVERGED"
                    if status == 0
                    else
                    "MAX_ITERS_REACHED"
                )


                result[
                    "converged"
                ] = (
                    status == 0
                )


                result[
                    "energy"
                ] = float(
                    force_field.CalcEnergy()
                )


                return result


        except Exception:

            pass


    try:

        uff_available = (
            AllChem.UFFHasAllMoleculeParams(
                mol
            )
        )

    except Exception:

        uff_available = False


    if uff_available:

        try:

            force_field = (
                AllChem.UFFGetMoleculeForceField(
                    mol,
                    confId=conformer_id
                )
            )


            if force_field is not None:

                force_field.Initialize()


                status = force_field.Minimize(
                    maxIts=
                        MAX_OPTIMIZATION_ITERS
                )


                result[
                    "method"
                ] = "UFF"


                result[
                    "status"
                ] = (
                    "CONVERGED"
                    if status == 0
                    else
                    "MAX_ITERS_REACHED"
                )


                result[
                    "converged"
                ] = (
                    status == 0
                )


                result[
                    "energy"
                ] = float(
                    force_field.CalcEnergy()
                )


                return result


        except Exception:

            pass


    result[
        "status"
    ] = "NO_SUPPORTED_FORCE_FIELD"


    return result


def centered_coordinates(
    mol,
    conformer_id
):

    conformer = mol.GetConformer(
        conformer_id
    )


    positions = np.asarray(
        conformer.GetPositions(),
        dtype=np.float32
    )


    if (
        positions.ndim != 2
        or
        positions.shape[
            1
        ] != 3
    ):

        raise RuntimeError(
            "Invalid coordinate shape."
        )


    if not np.isfinite(
        positions
    ).all():

        raise RuntimeError(
            "Non-finite coordinates."
        )


    centroid = positions.mean(
        axis=0,
        keepdims=True
    )


    positions = (
        positions
        -
        centroid
    )


    return torch.tensor(
        positions,
        dtype=torch.float32
    )


def build_candidate_graph(
    candidate_id,
    canonical_smiles
):

    output = {

        "success":
            False,

        "candidate_id":
            candidate_id,

        "canonical_smiles":
            canonical_smiles,

        "failure_reason":
            ""
    }


    try:

        mol = Chem.MolFromSmiles(
            canonical_smiles
        )


        if mol is None:

            raise RuntimeError(
                "RDKit_PARSE_FAILED"
            )


        mol_h = Chem.AddHs(
            mol
        )


        if mol_h is None:

            raise RuntimeError(
                "ADD_HYDROGENS_FAILED"
            )


        conformer_id = -1


        try:

            conformer_id = (
                generate_primary_conformer(
                    mol_h
                )
            )

        except Exception:

            conformer_id = -1


        if conformer_id >= 0:

            conformer_method = (
                "ETKDGv3_PRIMARY"
            )


        else:

            try:

                conformer_id = (
                    generate_fallback_conformer(
                        mol_h
                    )
                )

            except Exception:

                conformer_id = -1


            if conformer_id < 0:

                raise RuntimeError(
                    "CONFORMER_GENERATION_FAILED"
                )


            conformer_method = (
                "ETKDGv3_RANDOM_COORDS_FALLBACK"
            )


        optimization = (
            optimize_geometry(
                mol_h,
                conformer_id
            )
        )


        x = build_node_features(
            mol_h
        )


        edge_index, edge_attr = (
            build_edges(
                mol_h
            )
        )


        pos = centered_coordinates(
            mol_h,
            conformer_id
        )


        if x.shape[
            1
        ] != NODE_FEATURE_DIM:

            raise RuntimeError(
                "NODE_FEATURE_DIM_MISMATCH"
            )


        if edge_attr.shape[
            1
        ] != EDGE_FEATURE_DIM:

            raise RuntimeError(
                "EDGE_FEATURE_DIM_MISMATCH"
            )


        if x.shape[
            0
        ] != pos.shape[
            0
        ]:

            raise RuntimeError(
                "NODE_POSITION_COUNT_MISMATCH"
            )


        if (
            edge_index.shape[
                1
            ]
            !=
            edge_attr.shape[
                0
            ]
        ):

            raise RuntimeError(
                "EDGE_COUNT_MISMATCH"
            )


        if not torch.isfinite(
            x
        ).all():

            raise RuntimeError(
                "NONFINITE_X"
            )


        if not torch.isfinite(
            edge_attr
        ).all():

            raise RuntimeError(
                "NONFINITE_EDGE_ATTR"
            )


        if not torch.isfinite(
            pos
        ).all():

            raise RuntimeError(
                "NONFINITE_POS"
            )


        output.update(
            {

                "success":
                    True,

                "x":
                    x,

                "edge_index":
                    edge_index,

                "edge_attr":
                    edge_attr,

                "pos":
                    pos,

                "num_nodes":
                    int(
                        x.shape[
                            0
                        ]
                    ),

                "num_bonds":
                    int(
                        mol_h.GetNumBonds()
                    ),

                "num_directed_edges":
                    int(
                        edge_index.shape[
                            1
                        ]
                    ),

                "heavy_atom_count":
                    int(
                        mol.GetNumHeavyAtoms()
                    ),

                "total_atom_count":
                    int(
                        mol_h.GetNumAtoms()
                    ),

                "conformer_method":
                    conformer_method,

                "optimization_method":
                    optimization[
                        "method"
                    ],

                "optimization_status":
                    optimization[
                        "status"
                    ],

                "optimization_converged":
                    optimization[
                        "converged"
                    ],

                "conformer_energy":
                    optimization[
                        "energy"
                    ]
            }
        )


    except Exception as exc:

        output[
            "failure_reason"
        ] = str(
            exc
        )


    return output


# ==============================================================================
# 13. LOAD PHASE-9 NORMALIZATION — EXACT ACTUAL FORMAT
# ==============================================================================

with open(
    PHASE9_NORMALIZATION_FILE,
    "r",
    encoding="utf-8"
) as file:

    phase9_normalization = json.load(
        file
    )


required_normalization_keys = [

    "standardized_features",

    "standardized_feature_indices",

    "mean",

    "std"
]


missing_norm_keys = [

    key

    for key
    in required_normalization_keys

    if key not in phase9_normalization
]


if missing_norm_keys:

    raise RuntimeError(
        "Phase-9 normalization JSON does not match the "
        "actual Phase-9 serialization.\n"
        f"Missing keys: {missing_norm_keys}"
    )


NORMALIZED_NODE_FEATURE_NAMES = list(
    phase9_normalization[
        "standardized_features"
    ]
)


normalization_indices = list(
    phase9_normalization[
        "standardized_feature_indices"
    ]
)


expected_normalized_features = [

    "atomic_number",

    "degree",

    "formal_charge",

    "hydrogen_neighbor_count",

    "atomic_mass",

    "implicit_valence",

    "explicit_valence"
]


if (
    NORMALIZED_NODE_FEATURE_NAMES
    !=
    expected_normalized_features
):

    raise RuntimeError(
        "Phase-9 standardized-feature order differs from "
        "the finalized training pipeline.\n"
        f"Found: {NORMALIZED_NODE_FEATURE_NAMES}"
    )


expected_indices = [

    NODE_FEATURE_NAMES.index(
        feature_name
    )

    for feature_name
    in expected_normalized_features
]


if normalization_indices != expected_indices:

    raise RuntimeError(
        "Phase-9 standardized feature indices do not match "
        "the saved Phase-7 schema.\n"
        f"Saved indices   : {normalization_indices}\n"
        f"Expected indices: {expected_indices}"
    )


training_feature_mean = torch.tensor(
    [

        float(
            phase9_normalization[
                "mean"
            ][
                feature_name
            ]
        )

        for feature_name
        in NORMALIZED_NODE_FEATURE_NAMES
    ],
    dtype=torch.float32
)


training_feature_std = torch.tensor(
    [

        float(
            phase9_normalization[
                "std"
            ][
                feature_name
            ]
        )

        for feature_name
        in NORMALIZED_NODE_FEATURE_NAMES
    ],
    dtype=torch.float32
)


if training_feature_mean.shape[
    0
] != 7:

    raise RuntimeError(
        "Expected exactly seven Phase-9 feature means."
    )


if training_feature_std.shape[
    0
] != 7:

    raise RuntimeError(
        "Expected exactly seven Phase-9 feature standard deviations."
    )


if not torch.isfinite(
    training_feature_mean
).all():

    raise RuntimeError(
        "Phase-9 means contain non-finite values."
    )


if not torch.isfinite(
    training_feature_std
).all():

    raise RuntimeError(
        "Phase-9 standard deviations contain non-finite values."
    )


if bool(
    (
        training_feature_std
        <=
        0
    ).any()
):

    raise RuntimeError(
        "Phase-9 standard deviation contains zero/negative value."
    )


def normalize_node_features(
    x
):

    x = x.clone().float()


    index_tensor = torch.tensor(
        normalization_indices,
        dtype=torch.long,
        device=x.device
    )


    mean = training_feature_mean.to(
        x.device
    )


    std = training_feature_std.to(
        x.device
    )


    x[
        :,
        index_tensor
    ] = (

        x[
            :,
            index_tensor
        ]

        -

        mean

    ) / std


    return x


print(
    "\n[PASS] Phase-9 training-only normalization loaded EXACTLY."
)

print(
    "[INFO] Standardized features:"
)


for index, feature_name in zip(
    normalization_indices,
    NORMALIZED_NODE_FEATURE_NAMES
):

    print(
        f"  column {index:2d} : {feature_name}"
    )


# ==============================================================================
# 14. ACTUAL PHASE-9 SCATTER-MEAN HELPER
# ==============================================================================

def scatter_mean(
    values,
    index,
    dim_size
):

    output = torch.zeros(
        (
            dim_size,
            values.shape[
                1
            ]
        ),
        dtype=values.dtype,
        device=values.device
    )


    counts = torch.zeros(
        (
            dim_size,
            1
        ),
        dtype=values.dtype,
        device=values.device
    )


    output.index_add_(
        0,
        index,
        values
    )


    ones = torch.ones(
        (
            values.shape[
                0
            ],
            1
        ),
        dtype=values.dtype,
        device=values.device
    )


    counts.index_add_(
        0,
        index,
        ones
    )


    counts = counts.clamp(
        min=1.0
    )


    return (
        output
        /
        counts
    )


# ==============================================================================
# 15. ACTUAL PHASE-9 EGNN LAYER
# ==============================================================================

class EGNNLayer(
    nn.Module
):

    def __init__(
        self,
        hidden_dim,
        edge_dim,
        coordinate_update_scale=0.10
    ):

        super().__init__()


        self.hidden_dim = (
            hidden_dim
        )


        self.edge_dim = (
            edge_dim
        )


        self.coordinate_update_scale = (
            coordinate_update_scale
        )


        edge_input_dim = (

            hidden_dim

            +

            hidden_dim

            +

            1

            +

            edge_dim
        )


        self.edge_mlp = nn.Sequential(

            nn.Linear(
                edge_input_dim,
                hidden_dim
            ),

            nn.SiLU(),

            nn.Linear(
                hidden_dim,
                hidden_dim
            ),

            nn.SiLU()
        )


        self.coordinate_mlp = nn.Sequential(

            nn.Linear(
                hidden_dim,
                hidden_dim
            ),

            nn.SiLU(),

            nn.Linear(
                hidden_dim,
                1
            ),

            nn.Tanh()
        )


        self.node_mlp = nn.Sequential(

            nn.Linear(
                hidden_dim
                +
                hidden_dim,
                hidden_dim
            ),

            nn.SiLU(),

            nn.Linear(
                hidden_dim,
                hidden_dim
            )
        )


        self.node_norm = nn.LayerNorm(
            hidden_dim
        )


    def forward(
        self,
        h,
        pos,
        edge_index,
        edge_attr
    ):

        source = edge_index[
            0
        ]


        target = edge_index[
            1
        ]


        relative = (

            pos[
                source
            ]

            -

            pos[
                target
            ]
        )


        squared_distance = (
            relative
            *
            relative
        ).sum(
            dim=1,
            keepdim=True
        )


        edge_input = torch.cat(
            [

                h[
                    source
                ],

                h[
                    target
                ],

                squared_distance,

                edge_attr
            ],
            dim=1
        )


        messages = self.edge_mlp(
            edge_input
        )


        coordinate_weight = (

            self.coordinate_mlp(
                messages
            )

            *

            self.coordinate_update_scale
        )


        coordinate_messages = (

            relative

            *

            coordinate_weight
        )


        # IMPORTANT:
        # Exact Phase-9 implementation = scatter_mean.
        coordinate_aggregate = scatter_mean(
            coordinate_messages,
            source,
            h.shape[
                0
            ]
        )


        new_pos = (

            pos

            +

            coordinate_aggregate
        )


        # IMPORTANT:
        # Exact Phase-9 implementation = scatter_mean.
        aggregated_messages = scatter_mean(
            messages,
            source,
            h.shape[
                0
            ]
        )


        node_input = torch.cat(
            [
                h,
                aggregated_messages
            ],
            dim=1
        )


        node_delta = self.node_mlp(
            node_input
        )


        new_h = self.node_norm(

            h

            +

            node_delta
        )


        return (
            new_h,
            new_pos
        )


# ==============================================================================
# 16. ACTUAL PHASE-9 ENCODER
# ==============================================================================

class GeometryAwareVariationalEncoder(
    nn.Module
):

    def __init__(
        self,
        node_feature_dim,
        edge_feature_dim,
        hidden_dim,
        latent_dim,
        num_layers=2
    ):

        super().__init__()


        self.node_feature_dim = (
            node_feature_dim
        )


        self.edge_feature_dim = (
            edge_feature_dim
        )


        self.hidden_dim = (
            hidden_dim
        )


        self.latent_dim = (
            latent_dim
        )


        self.num_layers = (
            num_layers
        )


        self.node_embedding = nn.Sequential(

            nn.Linear(
                node_feature_dim,
                hidden_dim
            ),

            nn.SiLU(),

            nn.LayerNorm(
                hidden_dim
            )
        )


        self.egnn_layers = nn.ModuleList(
            [

                EGNNLayer(
                    hidden_dim=
                        hidden_dim,

                    edge_dim=
                        edge_feature_dim,

                    coordinate_update_scale=
                        COORDINATE_UPDATE_SCALE
                )

                for _
                in range(
                    num_layers
                )
            ]
        )


        self.graph_mlp = nn.Sequential(

            nn.Linear(
                hidden_dim,
                hidden_dim
            ),

            nn.SiLU(),

            nn.LayerNorm(
                hidden_dim
            )
        )


        self.mu_head = nn.Linear(
            hidden_dim,
            latent_dim
        )


        self.logvar_head = nn.Linear(
            hidden_dim,
            latent_dim
        )


    def graph_mean_pool(
        self,
        h,
        batch
    ):

        number_of_graphs = (

            int(
                batch.max().item()
            )

            +

            1
        )


        return scatter_mean(
            h,
            batch,
            number_of_graphs
        )


    def reparameterize(
        self,
        mu,
        logvar,
        deterministic=False
    ):

        if deterministic:

            return mu


        stable_logvar = torch.clamp(
            logvar,
            min=-20.0,
            max=20.0
        )


        std = torch.exp(
            0.5
            *
            stable_logvar
        )


        epsilon = torch.randn_like(
            std
        )


        return (

            mu

            +

            std
            *
            epsilon
        )


    def forward(
        self,
        x,
        pos,
        edge_index,
        edge_attr,
        batch,
        deterministic=False,
        return_diagnostics=False
    ):

        h = self.node_embedding(
            x
        )


        updated_pos = pos


        for layer in self.egnn_layers:

            h, updated_pos = layer(

                h=h,

                pos=updated_pos,

                edge_index=edge_index,

                edge_attr=edge_attr
            )


        graph_embedding = (
            self.graph_mean_pool(
                h,
                batch
            )
        )


        graph_embedding = self.graph_mlp(
            graph_embedding
        )


        mu = self.mu_head(
            graph_embedding
        )


        logvar = self.logvar_head(
            graph_embedding
        )


        z = self.reparameterize(
            mu=mu,
            logvar=logvar,
            deterministic=deterministic
        )


        output = {

            "mu":
                mu,

            "logvar":
                logvar,

            "z":
                z
        }


        if return_diagnostics:

            output[
                "graph_embedding"
            ] = graph_embedding


            output[
                "final_node_embeddings"
            ] = h


            output[
                "updated_coordinates"
            ] = updated_pos


        return output


# ==============================================================================
# 17. INITIALIZE AND STRICTLY LOAD θE*
# ==============================================================================

encoder = GeometryAwareVariationalEncoder(

    node_feature_dim=
        NODE_FEATURE_DIM,

    edge_feature_dim=
        EDGE_FEATURE_DIM,

    hidden_dim=
        HIDDEN_DIM,

    latent_dim=
        LATENT_DIM,

    num_layers=
        NUM_EGNN_LAYERS

).to(
    DEVICE
)


encoder_parameter_count = sum(

    parameter.numel()

    for parameter
    in encoder.parameters()
)


if encoder_parameter_count != 70274:

    raise RuntimeError(
        f"Encoder parameter count mismatch. "
        f"Expected 70,274, got {encoder_parameter_count:,}."
    )


encoder_checkpoint = torch.load(
    PHASE12_ENCODER_FILE,
    map_location="cpu",
    weights_only=False
)


encoder_state = normalize_state_dict_keys(
    extract_state_dict(
        encoder_checkpoint
    )
)


encoder.load_state_dict(
    encoder_state,
    strict=True
)


encoder.eval()


for parameter in encoder.parameters():

    parameter.requires_grad_(
        False
    )


print(
    f"\n[PASS] Frozen θE* loaded strictly "
    f"({encoder_parameter_count:,} parameters)."
)


# ==============================================================================
# 18. ACTUAL PHASE-12 PREDICTOR
# ==============================================================================

class Phase12TargetPredictor(
    nn.Module
):

    def __init__(
        self
    ):

        super().__init__()


        self.network = nn.Sequential(

            nn.Linear(
                32,
                64
            ),

            nn.LayerNorm(
                64
            ),

            nn.SiLU(),

            nn.Dropout(
                0.15
            ),

            nn.Linear(
                64,
                32
            ),

            nn.LayerNorm(
                32
            ),

            nn.SiLU(),

            nn.Dropout(
                0.15
            ),

            nn.Linear(
                32,
                130
            )
        )


    def forward(
        self,
        x
    ):

        return self.network(
            x
        )


predictor = Phase12TargetPredictor().to(
    DEVICE
)


predictor_parameter_count = sum(

    parameter.numel()

    for parameter
    in predictor.parameters()
)


if predictor_parameter_count != 8674:

    raise RuntimeError(
        f"Predictor parameter count mismatch: "
        f"{predictor_parameter_count:,}"
    )


predictor_checkpoint = torch.load(
    PHASE12_PREDICTOR_FILE,
    map_location="cpu",
    weights_only=False
)


predictor_state = normalize_state_dict_keys(
    extract_state_dict(
        predictor_checkpoint
    )
)


predictor.load_state_dict(
    predictor_state,
    strict=True
)


predictor.eval()


for parameter in predictor.parameters():

    parameter.requires_grad_(
        False
    )


print(
    f"[PASS] Frozen θP* loaded strictly "
    f"({predictor_parameter_count:,} parameters)."
)


# ==============================================================================
# 19. PARAMETER REFERENCE COPIES
# ==============================================================================

encoder_reference = {

    name:
        parameter.detach()
        .cpu()
        .clone()

    for name, parameter
    in encoder.named_parameters()
}


predictor_reference = {

    name:
        parameter.detach()
        .cpu()
        .clone()

    for name, parameter
    in predictor.named_parameters()
}


# ==============================================================================
# 20. LOAD EXACT PHASE-12 LATENT NORMALIZATION
# ==============================================================================

with open(
    PHASE12_LATENT_NORMALIZATION_FILE,
    "r",
    encoding="utf-8"
) as file:

    latent_norm = json.load(
        file
    )


if int(
    latent_norm[
        "latent_dimension"
    ]
) != LATENT_DIM:

    raise RuntimeError(
        "Phase-12 latent dimension mismatch."
    )


latent_mean = np.asarray(
    latent_norm[
        "mean"
    ],
    dtype=np.float32
)


# IMPORTANT:
# Phase 12 saved both raw std and safe_std.
# Predictor training used safe_std.
latent_std_safe = np.asarray(
    latent_norm[
        "safe_std"
    ],
    dtype=np.float32
)


if latent_mean.shape != (
    LATENT_DIM,
):

    raise RuntimeError(
        "Phase-12 latent mean has invalid shape."
    )


if latent_std_safe.shape != (
    LATENT_DIM,
):

    raise RuntimeError(
        "Phase-12 latent safe_std has invalid shape."
    )


print(
    "[PASS] Frozen Phase-12 latent normalization loaded."
)


# ==============================================================================
# 21. TARGET VOCABULARY
# ==============================================================================

target_vocab_df = pd.read_csv(
    PHASE12_TARGET_VOCAB_CSV_FILE,
    dtype=str
)


required_target_columns = [

    "target_index",

    "target_id",

    "target_name"
]


missing_target_columns = [

    column

    for column
    in required_target_columns

    if column not in target_vocab_df.columns
]


if missing_target_columns:

    raise RuntimeError(
        f"Target vocabulary missing columns: "
        f"{missing_target_columns}"
    )


target_vocab_df[
    "target_index"
] = pd.to_numeric(
    target_vocab_df[
        "target_index"
    ],
    errors="raise"
).astype(
    int
)


target_vocab_df = (
    target_vocab_df
    .sort_values(
        "target_index"
    )
    .reset_index(
        drop=True
    )
)


if len(
    target_vocab_df
) != TARGET_COUNT:

    raise RuntimeError(
        f"Expected 130 target classes; "
        f"found {len(target_vocab_df)}."
    )


expected_target_indices = list(
    range(
        TARGET_COUNT
    )
)


if target_vocab_df[
    "target_index"
].tolist() != expected_target_indices:

    raise RuntimeError(
        "Target vocabulary ordering is inconsistent."
    )


target_ids = (
    target_vocab_df[
        "target_id"
    ]
    .astype(str)
    .tolist()
)


target_names = (
    target_vocab_df[
        "target_name"
    ]
    .fillna("")
    .astype(str)
    .tolist()
)


print(
    "[PASS] Frozen 130-class target vocabulary loaded."
)


# ==============================================================================
# 22. LOAD Z0
# ==============================================================================

Z0_df = pd.read_csv(
    PHASE12_Z0_FILE,
    dtype=str
)


latent_column_patterns = [

    r"^z_(\d+)$",

    r"^latent_(\d+)$",

    r"^mu_(\d+)$"
]


Z0_LATENT_COLUMNS = None


for pattern in latent_column_patterns:

    discovered = []


    for column in Z0_df.columns:

        match = re.match(
            pattern,
            str(
                column
            )
        )


        if match:

            discovered.append(
                (
                    int(
                        match.group(
                            1
                        )
                    ),
                    column
                )
            )


    if len(
        discovered
    ) == LATENT_DIM:

        discovered.sort(
            key=lambda item:
                item[
                    0
                ]
        )


        Z0_LATENT_COLUMNS = [

            column

            for _, column
            in discovered
        ]


        break


if Z0_LATENT_COLUMNS is None:

    raise RuntimeError(
        "Could not identify the 32 Z0 latent columns."
    )


Z0_MATRIX = (
    Z0_df[
        Z0_LATENT_COLUMNS
    ]
    .astype(float)
    .to_numpy(
        dtype=np.float32
    )
)


Z0_CENTROID = (
    Z0_MATRIX.mean(
        axis=0
    )
)


if not np.isfinite(
    Z0_MATRIX
).all():

    raise RuntimeError(
        "Z0 contains non-finite values."
    )


print(
    f"[PASS] Baseline Z0 loaded: "
    f"{len(Z0_df)} molecules."
)


# ==============================================================================
# 23. ENCODER HELPER
# ==============================================================================

def encode_graph_mu(
    graph
):

    x = normalize_node_features(
        graph[
            "x"
        ]
    ).to(
        DEVICE
    )


    pos = (
        graph[
            "pos"
        ]
        .float()
        .to(
            DEVICE
        )
    )


    edge_index = (
        graph[
            "edge_index"
        ]
        .long()
        .to(
            DEVICE
        )
    )


    edge_attr = (
        graph[
            "edge_attr"
        ]
        .float()
        .to(
            DEVICE
        )
    )


    batch = torch.zeros(
        x.shape[
            0
        ],
        dtype=torch.long,
        device=DEVICE
    )


    encoder.eval()


    with torch.no_grad():

        output = encoder(

            x=x,

            pos=pos,

            edge_index=edge_index,

            edge_attr=edge_attr,

            batch=batch,

            deterministic=True
        )


    mu = output[
        "mu"
    ]


    if tuple(
        mu.shape
    ) != (
        1,
        LATENT_DIM
    ):

        raise RuntimeError(
            f"Unexpected μ shape: {tuple(mu.shape)}"
        )


    if not torch.isfinite(
        mu
    ).all():

        raise RuntimeError(
            "Non-finite μ produced."
        )


    return (
        mu
        .detach()
        .cpu()
        .numpy()
        .reshape(-1)
        .astype(
            np.float32
        )
    )


# ==============================================================================
# 24. CRITICAL BASELINE Z0 FIDELITY CHECK
# ==============================================================================

phase7_index_df = pd.read_csv(
    PHASE7_GRAPH_INDEX_FILE,
    dtype=str
)


z0_identifier_column = None


for candidate_column in [

    "graph_id",

    "drug_id"
]:

    if candidate_column in Z0_df.columns:

        z0_identifier_column = (
            candidate_column
        )

        break


if z0_identifier_column is None:

    raise RuntimeError(
        "Z0 does not contain graph_id or drug_id."
    )


z0_lookup = {

    str(
        row[
            z0_identifier_column
        ]
    ):
        row

    for _, row
    in Z0_df.iterrows()
}


reference_row = None


for _, row in phase7_index_df.iterrows():

    reference_id = clean_text(
        row.get(
            "graph_id",
            row.get(
                "drug_id",
                ""
            )
        )
    )


    if reference_id in z0_lookup:

        reference_row = row

        break


if reference_row is None:

    raise RuntimeError(
        "Could not locate overlapping Phase-7/Z0 reference molecule."
    )


reference_id = clean_text(
    reference_row.get(
        "graph_id",
        reference_row.get(
            "drug_id"
        )
    )
)


reference_graph_path = Path(
    reference_row[
        "graph_file"
    ]
)


reference_graph = torch.load(
    reference_graph_path,
    map_location="cpu",
    weights_only=False
)


recomputed_reference_mu = (
    encode_graph_mu(
        reference_graph
    )
)


stored_reference_mu = np.asarray(
    [

        float(
            z0_lookup[
                reference_id
            ][
                column
            ]
        )

        for column
        in Z0_LATENT_COLUMNS
    ],
    dtype=np.float32
)


reference_mu_max_delta = float(
    np.max(
        np.abs(
            recomputed_reference_mu
            -
            stored_reference_mu
        )
    )
)


reference_mu_mean_delta = float(
    np.mean(
        np.abs(
            recomputed_reference_mu
            -
            stored_reference_mu
        )
    )
)


ENCODER_FIDELITY_TOLERANCE = 1e-5


if (
    reference_mu_max_delta
    >
    ENCODER_FIDELITY_TOLERANCE
):

    raise RuntimeError(
        "\nCRITICAL ENCODER FIDELITY FAILURE.\n"
        f"Reference molecule: {reference_id}\n"
        f"Max |Δμ|          : {reference_mu_max_delta:.10e}\n"
        f"Tolerance         : {ENCODER_FIDELITY_TOLERANCE:.10e}\n"
        "Do NOT bypass this check."
    )


print(
    "\n[PASS] Exact Phase-9 encoder pathway reproduces stored Z0."
)

print(
    f"[CHECK] Reference molecule : {reference_id}"
)

print(
    f"[CHECK] Maximum |Δμ|       : "
    f"{reference_mu_max_delta:.10e}"
)

print(
    f"[CHECK] Mean |Δμ|          : "
    f"{reference_mu_mean_delta:.10e}"
)


# ==============================================================================
# 25. PHASE-7 INLINE FEATURE FIDELITY CHECK
# ==============================================================================

reference_smiles = clean_text(
    reference_graph.get(
        "standardized_smiles",
        ""
    )
)


if reference_smiles:

    rebuilt_reference = (
        build_candidate_graph(
            reference_id,
            reference_smiles
        )
    )


    if not rebuilt_reference[
        "success"
    ]:

        raise RuntimeError(
            "Could not rebuild Phase-7 reference graph:\n"
            f"{rebuilt_reference['failure_reason']}"
        )


    node_feature_fidelity = torch.equal(
        reference_graph[
            "x"
        ],
        rebuilt_reference[
            "x"
        ]
    )


    edge_index_fidelity = torch.equal(
        reference_graph[
            "edge_index"
        ],
        rebuilt_reference[
            "edge_index"
        ]
    )


    edge_feature_fidelity = torch.equal(
        reference_graph[
            "edge_attr"
        ],
        rebuilt_reference[
            "edge_attr"
        ]
    )


    if not all(
        [

            node_feature_fidelity,

            edge_index_fidelity,

            edge_feature_fidelity
        ]
    ):

        raise RuntimeError(
            "Phase-7 inline graph-feature fidelity failed.\n"
            f"Node features : {node_feature_fidelity}\n"
            f"Edge index    : {edge_index_fidelity}\n"
            f"Edge features : {edge_feature_fidelity}"
        )


else:

    node_feature_fidelity = True

    edge_index_fidelity = True

    edge_feature_fidelity = True


print(
    "[PASS] Phase-7 graph feature fidelity verified."
)


# ==============================================================================
# 26. LOAD PHASE-15 C1_ELIGIBLE
# ==============================================================================

candidate_df = pd.read_csv(
    PHASE15_ELIGIBLE_FILE,
    dtype={
        "candidate_id":
            str,

        "canonical_smiles":
            str
    }
)


INPUT_COUNT = len(
    candidate_df
)


if INPUT_COUNT == 0:

    raise RuntimeError(
        "Phase-15 eligible candidate pool is empty."
    )


required_candidate_columns = [

    "candidate_id",

    "canonical_smiles",

    "qed",

    "sa_score"
]


missing_candidate_columns = [

    column

    for column
    in required_candidate_columns

    if column not in candidate_df.columns
]


if missing_candidate_columns:

    raise RuntimeError(
        "Phase-15 eligible table missing required columns:\n"
        f"{missing_candidate_columns}"
    )


if candidate_df[
    "candidate_id"
].duplicated().any():

    raise RuntimeError(
        "Duplicate candidate IDs found."
    )


if candidate_df[
    "canonical_smiles"
].duplicated().any():

    raise RuntimeError(
        "Duplicate canonical structures found."
    )


print(
    "\n"
    +
    "=" * 145
)

print(
    "PHASE-15 ELIGIBLE CANDIDATE INPUT"
)

print(
    "=" * 145
)

print(
    f"C1_eligible candidates : {INPUT_COUNT}"
)


# ==============================================================================
# 27. BUILD GRAPHS AND RE-ENCODE
# ==============================================================================

graph_records = []

latent_records = []


print(
    "\n"
    +
    "=" * 145
)

print(
    "CANDIDATE 3D GRAPH CONSTRUCTION + RE-ENCODING"
)

print(
    "=" * 145
)


for record_number, (_, row) in enumerate(
    candidate_df.iterrows(),
    start=1
):

    candidate_id = clean_text(
        row[
            "candidate_id"
        ]
    )


    smiles = clean_text(
        row[
            "canonical_smiles"
        ]
    )


    graph = build_candidate_graph(
        candidate_id,
        smiles
    )


    encoder_success = False

    encoder_error = ""

    mu = None


    if graph[
        "success"
    ]:

        try:

            mu = encode_graph_mu(
                graph
            )


            encoder_success = True


        except Exception as exc:

            encoder_error = str(
                exc
            )


    graph_records.append(
        {

            "candidate_id":
                candidate_id,

            "canonical_smiles":
                smiles,

            "graph_success":
                bool(
                    graph[
                        "success"
                    ]
                ),

            "encoder_success":
                bool(
                    encoder_success
                ),

            "heavy_atom_count":
                graph.get(
                    "heavy_atom_count"
                ),

            "total_atom_count":
                graph.get(
                    "total_atom_count"
                ),

            "num_nodes":
                graph.get(
                    "num_nodes"
                ),

            "num_bonds":
                graph.get(
                    "num_bonds"
                ),

            "num_directed_edges":
                graph.get(
                    "num_directed_edges"
                ),

            "conformer_method":
                graph.get(
                    "conformer_method",
                    ""
                ),

            "optimization_method":
                graph.get(
                    "optimization_method",
                    ""
                ),

            "optimization_status":
                graph.get(
                    "optimization_status",
                    ""
                ),

            "optimization_converged":
                graph.get(
                    "optimization_converged",
                    False
                ),

            "conformer_energy":
                graph.get(
                    "conformer_energy"
                ),

            "graph_failure_reason":
                graph.get(
                    "failure_reason",
                    ""
                ),

            "encoder_failure_reason":
                encoder_error
        }
    )


    if encoder_success:

        latent_record = {

            "candidate_id":
                candidate_id
        }


        for latent_index in range(
            LATENT_DIM
        ):

            latent_record[
                f"z_{latent_index}"
            ] = float(
                mu[
                    latent_index
                ]
            )


        latent_records.append(
            latent_record
        )


    if (
        record_number == 1
        or
        record_number % 50 == 0
        or
        record_number == INPUT_COUNT
    ):

        print(
            f"[PROGRESS] "
            f"{record_number}/{INPUT_COUNT} "
            f"candidates processed..."
        )


graph_manifest_df = pd.DataFrame(
    graph_records
)


latent_df = pd.DataFrame(
    latent_records
)


graph_manifest_df.to_csv(
    PHASE16_GRAPH_MANIFEST_FILE,
    index=False,
    encoding="utf-8"
)


latent_df.to_csv(
    PHASE16_LATENT_FILE,
    index=False,
    encoding="utf-8"
)


GRAPH_SUCCESS_COUNT = int(
    graph_manifest_df[
        "graph_success"
    ].sum()
)


ENCODER_SUCCESS_COUNT = int(
    graph_manifest_df[
        "encoder_success"
    ].sum()
)


if ENCODER_SUCCESS_COUNT == 0:

    raise RuntimeError(
        "No Phase-16 candidates were successfully re-encoded."
    )


print(
    f"\n[RESULT] Graph successes   : "
    f"{GRAPH_SUCCESS_COUNT}/{INPUT_COUNT}"
)

print(
    f"[RESULT] Encoder successes : "
    f"{ENCODER_SUCCESS_COUNT}/{INPUT_COUNT}"
)


# ==============================================================================
# 28. ALIGN SUCCESSFUL CANDIDATES
# ==============================================================================

successful_ids = (
    latent_df[
        "candidate_id"
    ]
    .astype(str)
    .tolist()
)


successful_df = (
    candidate_df
    .set_index(
        "candidate_id"
    )
    .loc[
        successful_ids
    ]
    .reset_index()
)


latent_columns = [

    f"z_{index}"

    for index
    in range(
        LATENT_DIM
    )
]


candidate_latents = (
    latent_df[
        latent_columns
    ]
    .to_numpy(
        dtype=np.float32
    )
)


# ==============================================================================
# 29. LATENT-DOMAIN DIAGNOSTICS
# ==============================================================================

nearest_baseline_latent_distance = []

distance_to_baseline_centroid = []

candidate_latent_norm = []


for latent in candidate_latents:

    distances = np.linalg.norm(
        Z0_MATRIX
        -
        latent[
            None,
            :
        ],
        axis=1
    )


    nearest_baseline_latent_distance.append(
        float(
            distances.min()
        )
    )


    distance_to_baseline_centroid.append(
        float(
            np.linalg.norm(
                latent
                -
                Z0_CENTROID
            )
        )
    )


    candidate_latent_norm.append(
        float(
            np.linalg.norm(
                latent
            )
        )
    )


# ==============================================================================
# 30. FROZEN PHASE-12 LATENT NORMALIZATION
# ==============================================================================

normalized_latents = (

    candidate_latents

    -

    latent_mean[
        None,
        :
    ]

) / latent_std_safe[
    None,
    :
]


if not np.isfinite(
    normalized_latents
).all():

    raise RuntimeError(
        "Normalized candidate latent matrix contains NaN/Inf."
    )


normalized_latent_tensor = torch.tensor(
    normalized_latents,
    dtype=torch.float32,
    device=DEVICE
)


# ==============================================================================
# 31. DETERMINISTIC PREDICTOR SCORES
# ==============================================================================

predictor.eval()


with torch.no_grad():

    deterministic_logits = predictor(
        normalized_latent_tensor
    )


    deterministic_scores = torch.sigmoid(
        deterministic_logits
    )


if tuple(
    deterministic_scores.shape
) != (
    ENCODER_SUCCESS_COUNT,
    TARGET_COUNT
):

    raise RuntimeError(
        "Predictor output dimension mismatch."
    )


# ==============================================================================
# 32. MC DROPOUT — ONLY DROPOUT STOCHASTIC
# ==============================================================================

def activate_dropout_only(
    model
):

    model.eval()


    for module in model.modules():

        if isinstance(
            module,
            nn.Dropout
        ):

            module.train()


activate_dropout_only(
    predictor
)


dropout_modules = [

    module

    for module
    in predictor.modules()

    if isinstance(
        module,
        nn.Dropout
    )
]


layernorm_modules = [

    module

    for module
    in predictor.modules()

    if isinstance(
        module,
        nn.LayerNorm
    )
]


if len(
    dropout_modules
) != 2:

    raise RuntimeError(
        f"Expected exactly 2 Dropout modules; "
        f"found {len(dropout_modules)}."
    )


if not all(
    module.training
    for module
    in dropout_modules
):

    raise RuntimeError(
        "MC-dropout activation failed."
    )


if any(
    module.training
    for module
    in layernorm_modules
):

    raise RuntimeError(
        "LayerNorm entered training mode during MC Dropout."
    )


mc_scores = []


with torch.no_grad():

    for mc_pass in range(
        MC_DROPOUT_PASSES
    ):

        logits = predictor(
            normalized_latent_tensor
        )


        scores = torch.sigmoid(
            logits
        )


        mc_scores.append(
            scores
            .cpu()
            .numpy()
        )


mc_scores = np.stack(
    mc_scores,
    axis=0
)


predictive_mean = mc_scores.mean(
    axis=0
)


predictive_std = mc_scores.std(
    axis=0,
    ddof=1
)


predictor.eval()


if not np.isfinite(
    predictive_mean
).all():

    raise RuntimeError(
        "MC predictive means contain NaN/Inf."
    )


if not np.isfinite(
    predictive_std
).all():

    raise RuntimeError(
        "MC predictive uncertainty contains NaN/Inf."
    )


print(
    "\n[PASS] 30-pass MC Dropout completed."
)

print(
    "[PASS] Dropout was the only stochastic predictor component."
)


# ==============================================================================
# 33. FULL TARGET RELEVANCE MATRIX
# ==============================================================================

relevance_df = pd.DataFrame(
    predictive_mean,
    columns=[
        f"target_score::{target_id}"

        for target_id
        in target_ids
    ]
)


relevance_df.insert(
    0,
    "candidate_id",
    successful_ids
)


relevance_df.to_csv(
    PHASE16_RELEVANCE_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 34. FULL TARGET UNCERTAINTY MATRIX
# ==============================================================================

uncertainty_df = pd.DataFrame(
    predictive_std,
    columns=[
        f"target_uncertainty::{target_id}"

        for target_id
        in target_ids
    ]
)


uncertainty_df.insert(
    0,
    "candidate_id",
    successful_ids
)


uncertainty_df.to_csv(
    PHASE16_UNCERTAINTY_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 35. TOP-5 RELEVANCE AND UNCERTAINTY
# ==============================================================================

top_target_indices = np.argsort(
    predictive_mean,
    axis=1
)[
    :,
    ::-1
][
    :,
    :BIOLOGICAL_TOP_K
]


bio_relevance = []

bio_relevance_max = []

bio_uncertainty = []

bio_uncertainty_max = []

bio_ucb = []

top_target_records = []


for candidate_index in range(
    ENCODER_SUCCESS_COUNT
):

    indices = top_target_indices[
        candidate_index
    ]


    top_scores = predictive_mean[
        candidate_index,
        indices
    ]


    top_uncertainties = predictive_std[
        candidate_index,
        indices
    ]


    B = float(
        np.mean(
            top_scores
        )
    )


    U = float(
        np.mean(
            top_uncertainties
        )
    )


    B_max = float(
        np.max(
            predictive_mean[
                candidate_index
            ]
        )
    )


    U_max = float(
        np.max(
            predictive_std[
                candidate_index
            ]
        )
    )


    UCB = float(
        B
        +
        KAPPA_UCB
        *
        U
    )


    bio_relevance.append(
        B
    )


    bio_uncertainty.append(
        U
    )


    bio_relevance_max.append(
        B_max
    )


    bio_uncertainty_max.append(
        U_max
    )


    bio_ucb.append(
        UCB
    )


    record = {}


    for rank_position, target_index in enumerate(
        indices,
        start=1
    ):

        record[
            f"top_target_{rank_position}_id"
        ] = target_ids[
            int(
                target_index
            )
        ]


        record[
            f"top_target_{rank_position}_name"
        ] = target_names[
            int(
                target_index
            )
        ]


        record[
            f"top_target_{rank_position}_score"
        ] = float(
            predictive_mean[
                candidate_index,
                target_index
            ]
        )


        record[
            f"top_target_{rank_position}_uncertainty"
        ] = float(
            predictive_std[
                candidate_index,
                target_index
            ]
        )


    top_target_records.append(
        record
    )


# ==============================================================================
# 36. QED AND SA
# ==============================================================================

qed_values = pd.to_numeric(
    successful_df[
        "qed"
    ],
    errors="raise"
).to_numpy(
    dtype=float
)


sa_values = pd.to_numeric(
    successful_df[
        "sa_score"
    ],
    errors="raise"
).to_numpy(
    dtype=float
)


if not np.isfinite(
    qed_values
).all():

    raise RuntimeError(
        "QED contains non-finite values."
    )


if not np.isfinite(
    sa_values
).all():

    raise RuntimeError(
        "SA contains non-finite values."
    )


sa_utility = np.clip(

    1.0

    -

    (
        sa_values
        -
        1.0
    )
    /
    9.0,

    0.0,

    1.0
)


# ==============================================================================
# 37. UTILITY / ACQUISITION
# ==============================================================================

bio_relevance_np = np.asarray(
    bio_relevance,
    dtype=float
)


bio_uncertainty_np = np.asarray(
    bio_uncertainty,
    dtype=float
)


bio_ucb_np = np.asarray(
    bio_ucb,
    dtype=float
)


utility_score = (

    WEIGHT_QED
    *
    qed_values

    +

    WEIGHT_SA
    *
    sa_utility

    +

    WEIGHT_BIO
    *
    bio_relevance_np
)


acquisition_score = (

    WEIGHT_QED
    *
    qed_values

    +

    WEIGHT_SA
    *
    sa_utility

    +

    WEIGHT_BIO
    *
    bio_ucb_np
)


# ==============================================================================
# 38. PARETO FRONT
# ==============================================================================

pareto_values = np.column_stack(
    [

        qed_values,

        sa_utility,

        bio_relevance_np
    ]
)


def pareto_max_mask(
    values
):

    candidate_count = (
        values.shape[
            0
        ]
    )


    pareto = np.ones(
        candidate_count,
        dtype=bool
    )


    for i in range(
        candidate_count
    ):

        dominating = (

            np.all(
                values
                >=
                values[
                    i
                ],
                axis=1
            )

            &

            np.any(
                values
                >
                values[
                    i
                ],
                axis=1
            )
        )


        dominating[
            i
        ] = False


        if dominating.any():

            pareto[
                i
            ] = False


    return pareto


pareto_mask = pareto_max_mask(
    pareto_values
)


# ==============================================================================
# 39. BUILD MASTER SCORE TABLE
# ==============================================================================

score_df = (
    successful_df
    .copy()
    .reset_index(
        drop=True
    )
)


score_df[
    "nearest_baseline_latent_distance"
] = nearest_baseline_latent_distance


score_df[
    "distance_to_baseline_latent_centroid"
] = distance_to_baseline_centroid


score_df[
    "candidate_latent_norm"
] = candidate_latent_norm


score_df[
    "sa_utility"
] = sa_utility


score_df[
    "biological_relevance_top5_mean"
] = bio_relevance_np


score_df[
    "biological_relevance_max"
] = bio_relevance_max


score_df[
    "mc_uncertainty_top5_mean"
] = bio_uncertainty_np


score_df[
    "mc_uncertainty_max"
] = bio_uncertainty_max


score_df[
    "bio_ucb"
] = bio_ucb_np


score_df[
    "utility_score"
] = utility_score


score_df[
    "acquisition_score"
] = acquisition_score


score_df[
    "pareto_optimal"
] = pareto_mask


for latent_index in range(
    LATENT_DIM
):

    score_df[
        f"candidate_z_{latent_index}"
    ] = candidate_latents[
        :,
        latent_index
    ]


top_target_df = pd.DataFrame(
    top_target_records
)


score_df = pd.concat(
    [
        score_df,
        top_target_df
    ],
    axis=1
)


# ==============================================================================
# 40. RANKING
# ==============================================================================

score_df[
    "utility_rank"
] = (
    score_df[
        "utility_score"
    ]
    .rank(
        ascending=False,
        method="min"
    )
    .astype(int)
)


score_df[
    "acquisition_rank"
] = (
    score_df[
        "acquisition_score"
    ]
    .rank(
        ascending=False,
        method="min"
    )
    .astype(int)
)


score_df.to_csv(
    PHASE16_SCORE_FILE,
    index=False,
    encoding="utf-8"
)


ranking_df = (
    score_df
    .sort_values(
        [
            "acquisition_rank",
            "utility_rank"
        ]
    )
    .reset_index(
        drop=True
    )
)


ranking_df.to_csv(
    PHASE16_RANKING_FILE,
    index=False,
    encoding="utf-8"
)


pareto_df = (
    score_df[
        score_df[
            "pareto_optimal"
        ]
    ]
    .copy()
    .sort_values(
        "acquisition_score",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


pareto_df.to_csv(
    PHASE16_PARETO_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 41. DIVERSITY-AWARE S1
# ==============================================================================

def smiles_fingerprint(
    smiles
):

    mol = Chem.MolFromSmiles(
        smiles
    )


    if mol is None:

        return None


    return AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius=
            FP_RADIUS,
        nBits=
            FP_BITS
    )


selection_order = (
    score_df
    .sort_values(
        "acquisition_score",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


selected_records = []

selected_fingerprints = []


for _, row in selection_order.iterrows():

    if len(
        selected_records
    ) >= SELECTION_BUDGET:

        break


    fingerprint = smiles_fingerprint(
        clean_text(
            row[
                "canonical_smiles"
            ]
        )
    )


    if fingerprint is None:

        continue


    accept = True


    if selected_fingerprints:

        similarities = (
            DataStructs.BulkTanimotoSimilarity(
                fingerprint,
                selected_fingerprints
            )
        )


        if max(
            similarities
        ) > DIVERSITY_MAX_TANIMOTO:

            accept = False


    if not accept:

        continue


    selected_row = row.copy()


    reasons = [
        "diversity_selected"
    ]


    if int(
        row[
            "utility_rank"
        ]
    ) <= SELECTION_BUDGET:

        reasons.append(
            "top_exploitation"
        )


    if int(
        row[
            "acquisition_rank"
        ]
    ) <= SELECTION_BUDGET:

        reasons.append(
            "top_acquisition"
        )


    if bool(
        row[
            "pareto_optimal"
        ]
    ):

        reasons.append(
            "pareto_optimal"
        )


    selected_row[
        "selection_reason"
    ] = ";".join(
        reasons
    )


    selected_records.append(
        selected_row
    )


    selected_fingerprints.append(
        fingerprint
    )


selected_df = pd.DataFrame(
    selected_records
)


if not selected_df.empty:

    selected_df[
        "feedback_selection_order"
    ] = np.arange(
        1,
        len(
            selected_df
        ) + 1
    )


selected_df.to_csv(
    PHASE16_SELECTED_FILE,
    index=False,
    encoding="utf-8"
)


SELECTED_COUNT = len(
    selected_df
)


# ==============================================================================
# 42. IN-MEMORY Δθ CHECK
# ==============================================================================

encoder_parameters_unchanged = all(

    torch.equal(
        parameter.detach().cpu(),
        encoder_reference[
            name
        ]
    )

    for name, parameter
    in encoder.named_parameters()
)


predictor_parameters_unchanged = all(

    torch.equal(
        parameter.detach().cpu(),
        predictor_reference[
            name
        ]
    )

    for name, parameter
    in predictor.named_parameters()
)


if not encoder_parameters_unchanged:

    raise RuntimeError(
        "θE changed during Phase 16."
    )


if not predictor_parameters_unchanged:

    raise RuntimeError(
        "θP changed during Phase 16."
    )


# ==============================================================================
# 43. K0 HASH CHECK AFTER
# ==============================================================================

hash_after = {

    "thetaE":
        sha256_file(
            PHASE12_ENCODER_FILE
        ),

    "thetaD":
        sha256_file(
            PHASE12_DECODER_FILE
        ),

    "thetaP":
        sha256_file(
            PHASE12_PREDICTOR_FILE
        ),

    "F_E":
        sha256_file(
            PHASE13_ENCODER_FISHER_FILE
        ),

    "F_P":
        sha256_file(
            PHASE13_PREDICTOR_FISHER_FILE
        ),

    "phase13_reference":
        sha256_file(
            PHASE13_REFERENCE_FILE
        )
}


K0_HASHES_UNCHANGED = (
    hash_before
    ==
    hash_after
)


if not K0_HASHES_UNCHANGED:

    raise RuntimeError(
        "K0 hash mismatch detected after Phase 16."
    )


# ==============================================================================
# 44. SUMMARY VALUES
# ==============================================================================

PARETO_COUNT = int(
    pareto_mask.sum()
)


mean_biological_relevance = float(
    np.mean(
        bio_relevance_np
    )
)


mean_uncertainty = float(
    np.mean(
        bio_uncertainty_np
    )
)


mean_utility = float(
    np.mean(
        utility_score
    )
)


mean_acquisition = float(
    np.mean(
        acquisition_score
    )
)


# ==============================================================================
# 45. COMPLETION CHECKS
# ==============================================================================

completion_checks = {

    "Phase-15 eligible pool loaded":
        INPUT_COUNT > 0,

    "Candidate IDs unique":
        not candidate_df[
            "candidate_id"
        ].duplicated().any(),

    "Candidate canonical structures unique":
        not candidate_df[
            "canonical_smiles"
        ].duplicated().any(),

    "Exact Phase-7 node schema reused":
        NODE_FEATURE_NAMES
        ==
        EXPECTED_NODE_FEATURE_NAMES,

    "Exact Phase-7 edge schema reused":
        EDGE_FEATURE_NAMES
        ==
        EXPECTED_EDGE_FEATURE_NAMES,

    "Exact Phase-9 normalization structure loaded":
        (
            len(
                training_feature_mean
            )
            ==
            7
        ),

    "Phase-9 normalized feature order verified":
        (
            NORMALIZED_NODE_FEATURE_NAMES
            ==
            expected_normalized_features
        ),

    "Phase-9 normalization indices verified":
        (
            normalization_indices
            ==
            expected_indices
        ),

    "Encoder parameter count is 70,274":
        encoder_parameter_count
        ==
        70274,

    "Predictor parameter count is 8,674":
        predictor_parameter_count
        ==
        8674,

    "Frozen θE loaded strictly":
        True,

    "Frozen θP loaded strictly":
        True,

    "Encoder reproduces baseline Z0":
        (
            reference_mu_max_delta
            <=
            ENCODER_FIDELITY_TOLERANCE
        ),

    "Phase-12 latent normalization reused":
        True,

    "130-class target vocabulary reused":
        len(
            target_ids
        )
        ==
        TARGET_COUNT,

    "Candidate graph construction attempted":
        len(
            graph_manifest_df
        )
        ==
        INPUT_COUNT,

    "At least one candidate graph succeeded":
        GRAPH_SUCCESS_COUNT
        >
        0,

    "At least one candidate re-encoded":
        ENCODER_SUCCESS_COUNT
        >
        0,

    "Deterministic μ used":
        not encoder.training,

    "130-target relevance matrix produced":
        predictive_mean.shape[
            1
        ]
        ==
        TARGET_COUNT,

    "MC Dropout = 30 passes":
        MC_DROPOUT_PASSES
        ==
        30,

    "LayerNorm deterministic during MC Dropout":
        not any(
            module.training
            for module
            in layernorm_modules
        ),

    "QED reused":
        "qed"
        in candidate_df.columns,

    "SA reused":
        "sa_score"
        in candidate_df.columns,

    "SA utility finite":
        np.isfinite(
            sa_utility
        ).all(),

    "Biological relevance finite":
        np.isfinite(
            bio_relevance_np
        ).all(),

    "Predictive uncertainty finite":
        np.isfinite(
            bio_uncertainty_np
        ).all(),

    "BioUCB finite":
        np.isfinite(
            bio_ucb_np
        ).all(),

    "Utility finite":
        np.isfinite(
            utility_score
        ).all(),

    "Acquisition finite":
        np.isfinite(
            acquisition_score
        ).all(),

    "Pareto analysis complete":
        len(
            pareto_mask
        )
        ==
        ENCODER_SUCCESS_COUNT,

    "S1 proposed":
        SELECTED_COUNT
        >
        0,

    "θE unchanged":
        encoder_parameters_unchanged,

    "θP unchanged":
        predictor_parameters_unchanged,

    "K0 hashes unchanged":
        K0_HASHES_UNCHANGED,

    "No optimizer created":
        True,

    "No optimizer step":
        True,

    "No Fisher recalculation":
        True,

    "No EWC":
        True,

    "No model adaptation":
        True
}


ALL_CHECKS_PASS = all(
    bool(
        value
    )
    for value
    in completion_checks.values()
)


# ==============================================================================
# 46. CONFIG
# ==============================================================================

phase16_config = {

    "phase":
        (
            "Phase 16 — In-Silico Multi-Objective "
            "Candidate Evaluation, Uncertainty "
            "Estimation, and Prioritization"
        ),

    "generated_at":
        datetime.now().isoformat(),

    "random_seed":
        RANDOM_SEED,

    "input_candidate_count":
        INPUT_COUNT,

    "graph_success_count":
        GRAPH_SUCCESS_COUNT,

    "encoder_success_count":
        ENCODER_SUCCESS_COUNT,

    "node_feature_dimension":
        NODE_FEATURE_DIM,

    "edge_feature_dimension":
        EDGE_FEATURE_DIM,

    "phase9_normalized_features":
        NORMALIZED_NODE_FEATURE_NAMES,

    "phase9_normalization_indices":
        normalization_indices,

    "encoder_parameters":
        encoder_parameter_count,

    "latent_dimension":
        LATENT_DIM,

    "predictor_parameters":
        predictor_parameter_count,

    "target_classes":
        TARGET_COUNT,

    "mc_dropout_passes":
        MC_DROPOUT_PASSES,

    "biological_top_k":
        BIOLOGICAL_TOP_K,

    "kappa_ucb":
        KAPPA_UCB,

    "weights":
        {

            "qed":
                WEIGHT_QED,

            "sa_utility":
                WEIGHT_SA,

            "biological_relevance":
                WEIGHT_BIO
        },

    "sa_utility_formula":
        "clip(1-(SA-1)/9,0,1)",

    "biological_relevance_formula":
        "mean top-5 MC target-association relevance scores",

    "uncertainty_formula":
        "mean MC std for the same top-5 targets",

    "bio_ucb_formula":
        "B + kappa*U",

    "utility_formula":
        (
            "0.30*QED + 0.20*SAUtility "
            "+ 0.50*BiologicalRelevance"
        ),

    "acquisition_formula":
        (
            "0.30*QED + 0.20*SAUtility "
            "+ 0.50*BioUCB"
        ),

    "pareto_objectives":
        [

            "maximize QED",

            "maximize SA utility",

            "maximize biological relevance"
        ],

    "selection_budget":
        SELECTION_BUDGET,

    "diversity_max_tanimoto":
        DIVERSITY_MAX_TANIMOTO,

    "baseline_Z0_fidelity_max_delta":
        reference_mu_max_delta,

    "optimizer_created":
        False,

    "fisher_recalculated":
        False,

    "ewc_applied":
        False,

    "model_adaptation":
        False
}


with open(
    PHASE16_CONFIG_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        phase16_config,
        file,
        indent=4,
        ensure_ascii=False,
        default=str
    )


# ==============================================================================
# 47. QC
# ==============================================================================

qc_summary = {

    "phase":
        "Phase 16",

    "generated_at":
        datetime.now().isoformat(),

    "input_candidates":
        INPUT_COUNT,

    "graph_success":
        GRAPH_SUCCESS_COUNT,

    "graph_failure":
        INPUT_COUNT
        -
        GRAPH_SUCCESS_COUNT,

    "encoder_success":
        ENCODER_SUCCESS_COUNT,

    "encoder_failure":
        INPUT_COUNT
        -
        ENCODER_SUCCESS_COUNT,

    "encoder_Z0_max_abs_difference":
        reference_mu_max_delta,

    "encoder_Z0_mean_abs_difference":
        reference_mu_mean_delta,

    "mean_biological_relevance":
        mean_biological_relevance,

    "mean_mc_uncertainty":
        mean_uncertainty,

    "mean_utility":
        mean_utility,

    "mean_acquisition":
        mean_acquisition,

    "pareto_candidate_count":
        PARETO_COUNT,

    "selected_S1_count":
        SELECTED_COUNT,

    "K0_hashes_unchanged":
        K0_HASHES_UNCHANGED,

    "completion_checks":
        completion_checks
}


with open(
    PHASE16_QC_JSON_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        qc_summary,
        file,
        indent=4,
        ensure_ascii=False,
        default=str
    )


qc_rows = []


for metric, value in qc_summary.items():

    if metric == "completion_checks":

        for check_name, status in value.items():

            qc_rows.append(
                {

                    "metric":
                        f"completion.{check_name}",

                    "value":
                        status
                }
            )

    else:

        qc_rows.append(
            {

                "metric":
                    metric,

                "value":
                    value
            }
        )


pd.DataFrame(
    qc_rows
).to_csv(
    PHASE16_QC_CSV_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 48. TOP CANDIDATES
# ==============================================================================

print(
    "\n"
    +
    "=" * 145
)

print(
    "TOP 15 EXPLOITATION CANDIDATES"
)

print(
    "=" * 145
)


print(
    score_df
    .sort_values(
        "utility_rank"
    )[
        [

            "candidate_id",

            "utility_rank",

            "utility_score",

            "qed",

            "sa_score",

            "sa_utility",

            "biological_relevance_top5_mean",

            "mc_uncertainty_top5_mean",

            "bio_ucb",

            "pareto_optimal"
        ]
    ]
    .head(
        15
    )
    .to_string(
        index=False
    )
)


print(
    "\n"
    +
    "=" * 145
)

print(
    "TOP 15 EXPLORATION-AWARE CANDIDATES"
)

print(
    "=" * 145
)


print(
    score_df
    .sort_values(
        "acquisition_rank"
    )[
        [

            "candidate_id",

            "acquisition_rank",

            "acquisition_score",

            "utility_rank",

            "biological_relevance_top5_mean",

            "mc_uncertainty_top5_mean",

            "bio_ucb",

            "qed",

            "sa_score",

            "pareto_optimal"
        ]
    ]
    .head(
        15
    )
    .to_string(
        index=False
    )
)


# ==============================================================================
# 49. COMPLETION VALIDATION
# ==============================================================================

print(
    "\n"
    +
    "=" * 145
)

print(
    "PHASE 16 COMPLETION VALIDATION"
)

print(
    "=" * 145
)


for check_name, status in completion_checks.items():

    label = (
        "PASS"
        if bool(
            status
        )
        else
        "FAIL"
    )


    print(
        f"[{label:<4}] "
        f"{check_name}"
    )


# ==============================================================================
# 50. FINAL SUMMARY
# ==============================================================================

print(
    "\n"
    +
    "=" * 145
)

print(
    "PHASE 16 FINAL SUMMARY"
)

print(
    "=" * 145
)


print(
    f"Input Phase-15 Candidates          : "
    f"{INPUT_COUNT}"
)

print(
    f"Graph Construction Success         : "
    f"{GRAPH_SUCCESS_COUNT}"
)

print(
    f"Graph Construction Failure         : "
    f"{INPUT_COUNT - GRAPH_SUCCESS_COUNT}"
)

print(
    f"Successfully Re-Encoded            : "
    f"{ENCODER_SUCCESS_COUNT}"
)


print(
    "\nFrozen Preprocessing:"
)

print(
    "  Phase-9 Graph Normalization      : REUSED"
)

print(
    "  Standardized Features            : 7"
)

print(
    "  Phase-12 Latent Normalization    : REUSED"
)

print(
    "  Phase-12 Target Vocabulary       : REUSED"
)


print(
    "\nFrozen Models:"
)

print(
    f"  θE* Parameters                   : "
    f"{encoder_parameter_count:,}"
)

print(
    f"  θP* Parameters                   : "
    f"{predictor_parameter_count:,}"
)

print(
    f"  Target Classes                   : "
    f"{TARGET_COUNT}"
)

print(
    f"  Baseline Z0 Fidelity Max |Δμ|    : "
    f"{reference_mu_max_delta:.10e}"
)


print(
    "\nMC-Dropout:"
)

print(
    f"  Passes                           : "
    f"{MC_DROPOUT_PASSES}"
)

print(
    "  Encoder Stochastic               : False"
)

print(
    "  Predictor Dropout Stochastic     : True"
)

print(
    "  Predictor LayerNorm Stochastic   : False"
)


print(
    "\nMulti-Objective Evaluation:"
)

print(
    f"  Mean Biological Relevance        : "
    f"{mean_biological_relevance:.6f}"
)

print(
    f"  Mean MC Uncertainty              : "
    f"{mean_uncertainty:.6f}"
)

print(
    f"  Mean Utility                     : "
    f"{mean_utility:.6f}"
)

print(
    f"  Mean Acquisition                 : "
    f"{mean_acquisition:.6f}"
)

print(
    f"  Pareto Candidates                : "
    f"{PARETO_COUNT}"
)

print(
    f"  Diversity-Aware S1 Candidates    : "
    f"{SELECTED_COUNT}"
)


print(
    "\nK0 Integrity:"
)

print(
    f"  θE* Parameters Unchanged         : "
    f"{encoder_parameters_unchanged}"
)

print(
    f"  θP* Parameters Unchanged         : "
    f"{predictor_parameters_unchanged}"
)

print(
    f"  K0 SHA-256 Hashes Unchanged      : "
    f"{K0_HASHES_UNCHANGED}"
)

print(
    "  Optimizer Created                : False"
)

print(
    "  Optimizer Step                   : False"
)

print(
    "  Fisher Recalculation             : False"
)

print(
    "  EWC Applied                      : False"
)

print(
    "  Model Adaptation                 : False"
)


print(
    "\nSaved Files:"
)

print(
    f"  Config                            : "
    f"{PHASE16_CONFIG_FILE}"
)

print(
    f"  Graph Manifest                    : "
    f"{PHASE16_GRAPH_MANIFEST_FILE}"
)

print(
    f"  Candidate Latents                 : "
    f"{PHASE16_LATENT_FILE}"
)

print(
    f"  Target Relevance Matrix           : "
    f"{PHASE16_RELEVANCE_FILE}"
)

print(
    f"  Target Uncertainty Matrix         : "
    f"{PHASE16_UNCERTAINTY_FILE}"
)

print(
    f"  Multi-Objective Scores            : "
    f"{PHASE16_SCORE_FILE}"
)

print(
    f"  Rankings                          : "
    f"{PHASE16_RANKING_FILE}"
)

print(
    f"  Pareto Candidates                 : "
    f"{PHASE16_PARETO_FILE}"
)

print(
    f"  Proposed S1                       : "
    f"{PHASE16_SELECTED_FILE}"
)

print(
    f"  QC CSV                            : "
    f"{PHASE16_QC_CSV_FILE}"
)

print(
    f"  QC JSON                           : "
    f"{PHASE16_QC_JSON_FILE}"
)


print(
    "-" * 145
)


if ALL_CHECKS_PASS:

    print(
        "[STATUS] PHASE 16 COMPLETED SUCCESSFULLY."
    )


    print(
        "[PASS] Exact Phase-9 preprocessing and EGNN architecture were reused."
    )


    print(
        "[PASS] Actual generated molecular structures were re-encoded."
    )


    print(
        "[PASS] Continuous DrugBank-grounded target-association "
        "relevance scores were calculated."
    )


    print(
        "[PASS] MC-dropout predictive uncertainty was calculated."
    )


    print(
        "[PASS] Multi-objective utility and exploration-aware "
        "acquisition rankings were generated."
    )


    print(
        "[PASS] Pareto candidates were identified."
    )


    print(
        "[PASS] Diversity-aware S1 candidate set was proposed."
    )


    print(
        "[PASS] K0 remained unchanged."
    )


    print(
        "[IMPORTANT] Target-association scores are NOT "
        "binding affinities or experimental bioactivity."
    )


    print(
        "[IMPORTANT] No model adaptation or EWC occurred."
    )


    print(
        "[READY] Send this output for verification before Phase 17."
    )


else:

    print(
        "[STATUS] PHASE 16 REQUIRES REVIEW."
    )


    print(
        "[ACTION] Do NOT proceed to Phase 17."
    )


    print(
        "[ACTION] Resolve failed checks first."
    )


print(
    "=" * 145
)

PHASE 16 — IN-SILICO MULTI-OBJECTIVE CANDIDATE EVALUATION, UNCERTAINTY ESTIMATION, AND PRIORITIZATION

[PASS] Required Phase-7/9/12/13/15 files found.

[PASS] Pre-evaluation K0 hashes recorded.
[PASS] Phase-7 node feature dimension = 21
[PASS] Phase-7 edge feature dimension = 14
[PASS] Exact Phase-7 feature ordering verified.

[PASS] Phase-9 training-only normalization loaded EXACTLY.
[INFO] Standardized features:
  column  0 : atomic_number
  column  1 : degree
  column  2 : formal_charge
  column  5 : hydrogen_neighbor_count
  column  6 : atomic_mass
  column  7 : implicit_valence
  column  8 : explicit_valence

[PASS] Frozen θE* loaded strictly (70,274 parameters).
[PASS] Frozen θP* loaded strictly (8,674 parameters).
[PASS] Frozen Phase-12 latent normalization loaded.
[PASS] Frozen 130-class target vocabulary loaded.
[PASS] Baseline Z0 loaded: 767 molecules.

[PASS] Exact Phase-9 encoder pathway reproduces stored Z0.
[CHECK] Reference molecule : DB00035
[CHECK] Maximum |Δμ|       :

[14:56:56] UFFTYPER: Unrecognized atom type: S_5+4 (33)
[14:57:01] UFFTYPER: Unrecognized atom type: S_5+4 (14)


[PROGRESS] 50/839 candidates processed...


[14:58:17] UFFTYPER: Unrecognized atom type: S_5+4 (20)


[PROGRESS] 100/839 candidates processed...


...

[PROGRESS] 750/839 candidates processed...


[15:40:13] UFFTYPER: Unrecognized charge state for atom: 11
[15:40:13] UFFTYPER: Unrecognized charge state for atom: 21
[15:40:14] UFFTYPER: Unrecognized atom type: S_5+4 (18)
[15:40:16] UFFTYPER: Unrecognized charge state for atom: 6


[PROGRESS] 800/839 candidates processed...


[15:41:06] UFFTYPER: Unrecognized atom type: S_5+4 (11)


[PROGRESS] 839/839 candidates processed...

[RESULT] Graph successes   : 807/839
[RESULT] Encoder successes : 807/839

[PASS] 30-pass MC Dropout completed.
[PASS] Dropout was the only stochastic predictor component.

TOP 15 EXPLOITATION CANDIDATES
      candidate_id  utility_rank  utility_score      qed  sa_score  sa_utility  biological_relevance_top5_mean  mc_uncertainty_top5_mean  bio_ucb  pareto_optimal
CYCLE01_CAND000064             1       0.807748 0.837169  4.274906    0.636122                        0.858746                  0.082767 0.941513            True
CYCLE01_CAND000720             2       0.781650 0.854058  2.958810    0.782354                        0.737925                  0.119871 0.857795            True
CYCLE01_CAND000212             3       0.772729 0.857882  3.565726    0.714919                        0.744761                  0.106254 0.851015            True
CYCLE01_CAND000125             4       0.768228 0.801610  2.147382    0.872513                        0.

# Phase 17 — Feedback Candidate Definition & EWC-Regularized Closed-Loop Adaptation (Module 108 & 110)

**Objective:** Complete the Cycle-1 closed-loop adaptation by optimizing the molecular encoder ($\theta_E$) and target predictor ($\theta_P$) on prioritized leads ($S_1$) using a pairwise preference ranking loss regularized by Elastic Weight Consolidation (EWC) under strict retention and lexicographic model-selection guardrails.

* **Adaptation Formulation & Model Selection:**
  * **Preference Objective:** Margin-based pairwise ranking loss ($\mathcal{L}_{\text{rank}} = \text{softplus}(\gamma - (U_i - U_j))$) with rank separation ($\Delta \text{rank} \ge 5$) weighted by baseline utility differences.
  * **Leakage-Free Splitting:** Strict molecule-level 80/20 split (40 train / 10 validation) yielding 684 training pairs and 40 validation pairs.
  * **Lexicographic Selection:** Checkpoints evaluated primarily on validation pairwise accuracy, margin accuracy, and retention drift rather than raw continuous loss alone.
  * **Selected Model:** $\lambda_{\text{EWC}} = 10.0$ at Epoch 1 selected over the $K_0$ baseline benchmark.

* **Performance & Knowledge Retention Metrics:**
  * **Feedback Ranking:** Validation pairwise accuracy improved from $0.9750 \to \mathbf{1.0000}$ (+2.5%); training accuracy remained stable ($0.8962 \to 0.8860$).
  * **Functional Retention ($N=128$ Fisher References):** Mean multi-label target drift constrained to $\mathbf{0.0076}$ ($0.76\%$); maximum single-molecule drift capped at $0.3607$ ($0/128$ severe outliers $>0.50$).
  * **Representation Stability:** Mean latent drift $\Vert{}\Delta \mu\Vert{}_2 = 0.1888$; aggregate Fisher-weighted curvature drift $D_F = 2.37 \times 10^{-7}$.
  * **Rank Concordance:** Spearman rank correlation $\rho = 0.8913$ and Kendall $\tau = 0.7339$ relative to initial priorities.

* **Exported Checkpoints & Artifacts:**
  * Adapted Cycle-1 Weights: `phase17_cycle1_adapted_encoder.pt`, `phase17_cycle1_adapted_predictor.pt`.
  * Retention Audits: `phase17_ewc_lambda_search.csv`, `phase17_model_selection_candidates.csv`, `phase17_reference_prediction_drift_outliers.csv`.
  * Manifests & QC: `phase17_feedback_pairs.csv`, `phase17_training_history.csv`, `phase17_adaptation_qc_summary.json`.
  * Established Knowledge State: $\mathcal{K}_1 = \{\theta_E^{(1)}, \theta_D^*, \theta_P^{(1)}, F_E, F_P, Z_0, S_1\}$.

In [20]:
# ==============================================================================
# PHASE 17 — FEEDBACK CANDIDATE DEFINITION AND
# EWC-REGULARIZED CLOSED-LOOP ADAPTATION
#
# REVISED FINAL IMPLEMENTATION
#
# MAJOR CORRECTIONS
# -----------------
#
# 1. Epoch checkpoint selection is now LEXICOGRAPHIC:
#
#       a. highest validation pairwise accuracy
#       b. highest validation margin accuracy
#       c. lowest validation feedback loss
#
#    Loss alone can NO LONGER select an epoch with degraded ranking fidelity.
#
# 2. K0 is explicitly included as a NO-ADAPTATION benchmark.
#
#    An adapted model is accepted only if:
#
#       validation pair accuracy >= baseline validation accuracy
#
#       AND
#
#       train pair accuracy is not meaningfully degraded
#
#       AND
#
#       no severe Fisher-reference prediction-drift outlier is detected
#       above the predefined threshold.
#
# 3. Severe reference drift is explicitly audited:
#
#       max target-score drift > 0.50
#
# 4. Primary preference remains Phase-16 exploitation UTILITY rank.
#
# 5. Differentiable utility remains:
#
#       U_adapt =
#           0.30*QED_fixed
#         + 0.20*SAUtility_fixed
#         + 0.50*B_adapt
#
# 6. Phase-16 top-5 target identities remain fixed.
#
# 7. Raw diagonal-Fisher EWC is the PRIMARY regularizer.
#    Parameter-normalized EWC is diagnostic only.
#
# 8. θE and θP adapt.
#    θD remains frozen.
#
# 9. Fisher is NOT recalculated.
#
# 10. Phase-8 test data is NOT used for tuning.
#
# ==============================================================================


# ==============================================================================
# 1. IMPORTS
# ==============================================================================

from pathlib import Path
from datetime import datetime

import copy
import hashlib
import json
import math
import random
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from rdkit import Chem
from rdkit import RDLogger
from rdkit.Chem import AllChem


warnings.filterwarnings(
    "ignore"
)

RDLogger.DisableLog(
    "rdApp.warning"
)

RDLogger.DisableLog(
    "rdApp.info"
)


# ==============================================================================
# 2. REPRODUCIBILITY
# ==============================================================================

RANDOM_SEED = 42

random.seed(
    RANDOM_SEED
)

np.random.seed(
    RANDOM_SEED
)

torch.manual_seed(
    RANDOM_SEED
)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        RANDOM_SEED
    )


DEVICE = torch.device(
    "cpu"
)


# ==============================================================================
# 3. FIXED PHASE-17 CONFIGURATION
# ==============================================================================

CYCLE = 1


NODE_FEATURE_DIM = 21

EDGE_FEATURE_DIM = 14

HIDDEN_DIM = 64

LATENT_DIM = 32

NUM_EGNN_LAYERS = 2

COORDINATE_UPDATE_SCALE = 0.10

TARGET_COUNT = 130


# ------------------------------------------------------------------------------
# Feedback split
# ------------------------------------------------------------------------------

FEEDBACK_TRAIN_FRACTION = 0.80


# ------------------------------------------------------------------------------
# Pair construction
# ------------------------------------------------------------------------------

MINIMUM_RANK_SEPARATION = 5

RANKING_MARGIN = 0.05

USE_PAIR_WEIGHTS = True


# ------------------------------------------------------------------------------
# Adaptation optimizer
# ------------------------------------------------------------------------------

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 1e-4

MAX_ADAPTATION_EPOCHS = 20


# ------------------------------------------------------------------------------
# Early stopping
#
# IMPORTANT:
# patience is now based on the LEXICOGRAPHIC ranking metric,
# NOT validation loss alone.
# ------------------------------------------------------------------------------

EARLY_STOPPING_PATIENCE = 5


# ------------------------------------------------------------------------------
# EWC λ search
# ------------------------------------------------------------------------------

EWC_LAMBDA_CANDIDATES = [

    10.0,

    100.0,

    1000.0
]


# ------------------------------------------------------------------------------
# Phase-16 utility definition
# ------------------------------------------------------------------------------

WEIGHT_QED = 0.30

WEIGHT_SA = 0.20

WEIGHT_BIO = 0.50


# ------------------------------------------------------------------------------
# Geometry
# ------------------------------------------------------------------------------

MAX_OPTIMIZATION_ITERS = 500


# ------------------------------------------------------------------------------
# Expected corpus sizes
# ------------------------------------------------------------------------------

EXPECTED_S1_SIZE = 50

EXPECTED_FISHER_REFERENCE_SIZE = 128


# ------------------------------------------------------------------------------
# RETENTION / ACCEPTANCE GUARDRAILS
#
# The baseline model is explicitly used as the benchmark.
#
# We do NOT accept an adapted model merely because its softplus loss decreases.
# ------------------------------------------------------------------------------

# Maximum tolerated training pair-accuracy reduction relative to K0.
# A 2 percentage-point tolerance avoids rejecting negligible numerical change
# while preventing the large ranking deterioration observed previously.
MAX_ALLOWED_TRAIN_ACCURACY_DROP = 0.02


# A target probability shift greater than 0.50 is treated as a severe
# localized retention event requiring rejection/audit.
SEVERE_REFERENCE_DRIFT_THRESHOLD = 0.50


# Primary model requires zero severe reference molecules.
MAX_ALLOWED_SEVERE_REFERENCE_MOLECULES = 0


# ==============================================================================
# 4. PROJECT PATHS
# ==============================================================================

if "PROCESSED_DIR" not in globals():

    raise RuntimeError(
        "PROCESSED_DIR is missing. "
        "Run the project initialization cell first."
    )


PROCESSED_DIR = Path(
    PROCESSED_DIR
)


if "GRAPH_DIR" in globals():

    GRAPH_DIR = Path(
        GRAPH_DIR
    )

else:

    GRAPH_DIR = (
        PROCESSED_DIR.parent
        /
        "graphs"
    )


# ==============================================================================
# 5. INPUT FILES
# ==============================================================================

PHASE16_SELECTED_FILE = (
    PROCESSED_DIR
    /
    "phase16_selected_candidates.csv"
)


PHASE7_FEATURE_SCHEMA_FILE = (
    PROCESSED_DIR
    /
    "phase7_graph_feature_schema.json"
)


PHASE9_NORMALIZATION_FILE = (
    PROCESSED_DIR
    /
    "phase9_feature_normalization.json"
)


PHASE12_ENCODER_FILE = (
    PROCESSED_DIR
    /
    "phase12_thetaE_baseline_encoder.pt"
)


PHASE12_DECODER_FILE = (
    PROCESSED_DIR
    /
    "phase12_thetaD_baseline_decoder.pt"
)


PHASE12_PREDICTOR_FILE = (
    PROCESSED_DIR
    /
    "phase12_best_baseline_predictor.pt"
)


PHASE12_LATENT_NORMALIZATION_FILE = (
    PROCESSED_DIR
    /
    "phase12_predictor_latent_normalization.json"
)


PHASE12_TARGET_VOCAB_FILE = (
    PROCESSED_DIR
    /
    "phase12_predictor_target_vocabulary.csv"
)


PHASE8_FISHER_REFERENCE_FILE = (
    PROCESSED_DIR
    /
    "phase8_fisher_reference.csv"
)


PHASE13_ENCODER_FISHER_FILE = (
    PROCESSED_DIR
    /
    "phase13_encoder_fisher.pt"
)


PHASE13_PREDICTOR_FISHER_FILE = (
    PROCESSED_DIR
    /
    "phase13_predictor_fisher.pt"
)


PHASE13_PARAMETER_REFERENCE_FILE = (
    PROCESSED_DIR
    /
    "phase13_fisher_parameter_reference.pt"
)


required_files = [

    PHASE16_SELECTED_FILE,

    PHASE7_FEATURE_SCHEMA_FILE,

    PHASE9_NORMALIZATION_FILE,

    PHASE12_ENCODER_FILE,

    PHASE12_DECODER_FILE,

    PHASE12_PREDICTOR_FILE,

    PHASE12_LATENT_NORMALIZATION_FILE,

    PHASE12_TARGET_VOCAB_FILE,

    PHASE8_FISHER_REFERENCE_FILE,

    PHASE13_ENCODER_FISHER_FILE,

    PHASE13_PREDICTOR_FISHER_FILE,

    PHASE13_PARAMETER_REFERENCE_FILE
]


missing_files = [

    str(
        path
    )

    for path
    in required_files

    if not path.exists()
]


if missing_files:

    raise FileNotFoundError(
        "Required Phase-17 files are missing:\n"
        +
        "\n".join(
            missing_files
        )
    )


# ==============================================================================
# 6. OUTPUT FILES
# ==============================================================================

PHASE17_CONFIG_FILE = (
    PROCESSED_DIR
    /
    "phase17_adaptation_config.json"
)


PHASE17_CANDIDATE_MANIFEST_FILE = (
    PROCESSED_DIR
    /
    "phase17_feedback_candidate_manifest.csv"
)


PHASE17_GRAPH_MANIFEST_FILE = (
    PROCESSED_DIR
    /
    "phase17_precomputed_graph_manifest.csv"
)


PHASE17_TRAIN_CANDIDATES_FILE = (
    PROCESSED_DIR
    /
    "phase17_feedback_train_candidates.csv"
)


PHASE17_VALIDATION_CANDIDATES_FILE = (
    PROCESSED_DIR
    /
    "phase17_feedback_validation_candidates.csv"
)


PHASE17_PAIRS_FILE = (
    PROCESSED_DIR
    /
    "phase17_feedback_pairs.csv"
)


PHASE17_LAMBDA_SEARCH_FILE = (
    PROCESSED_DIR
    /
    "phase17_ewc_lambda_search.csv"
)


PHASE17_MODEL_SELECTION_FILE = (
    PROCESSED_DIR
    /
    "phase17_model_selection_candidates.csv"
)


PHASE17_HISTORY_FILE = (
    PROCESSED_DIR
    /
    "phase17_training_history.csv"
)


PHASE17_ENCODER_FILE = (
    PROCESSED_DIR
    /
    "phase17_cycle1_adapted_encoder.pt"
)


PHASE17_PREDICTOR_FILE = (
    PROCESSED_DIR
    /
    "phase17_cycle1_adapted_predictor.pt"
)


PHASE17_PARAMETER_DRIFT_FILE = (
    PROCESSED_DIR
    /
    "phase17_parameter_drift.csv"
)


PHASE17_FISHER_DRIFT_FILE = (
    PROCESSED_DIR
    /
    "phase17_fisher_weighted_drift.csv"
)


PHASE17_REFERENCE_LATENT_DRIFT_FILE = (
    PROCESSED_DIR
    /
    "phase17_reference_latent_drift.csv"
)


PHASE17_REFERENCE_PREDICTION_DRIFT_FILE = (
    PROCESSED_DIR
    /
    "phase17_reference_prediction_drift.csv"
)


PHASE17_REFERENCE_OUTLIERS_FILE = (
    PROCESSED_DIR
    /
    "phase17_reference_prediction_drift_outliers.csv"
)


PHASE17_FEEDBACK_PRE_POST_FILE = (
    PROCESSED_DIR
    /
    "phase17_feedback_pre_post_scores.csv"
)


PHASE17_RANKING_COMPARISON_FILE = (
    PROCESSED_DIR
    /
    "phase17_feedback_ranking_comparison.csv"
)


PHASE17_RETENTION_SUMMARY_FILE = (
    PROCESSED_DIR
    /
    "phase17_knowledge_retention_summary.csv"
)


PHASE17_QC_CSV_FILE = (
    PROCESSED_DIR
    /
    "phase17_adaptation_qc_summary.csv"
)


PHASE17_QC_JSON_FILE = (
    PROCESSED_DIR
    /
    "phase17_adaptation_qc_summary.json"
)


# ==============================================================================
# 7. HELPERS
# ==============================================================================

def clean_text(
    value
):

    if value is None:

        return ""


    if pd.isna(
        value
    ):

        return ""


    return str(
        value
    ).strip()


def sha256_file(
    path
):

    digest = hashlib.sha256()


    with open(
        path,
        "rb"
    ) as handle:

        while True:

            block = handle.read(
                1024
                *
                1024
            )


            if not block:

                break


            digest.update(
                block
            )


    return digest.hexdigest()


def extract_state_dict(
    checkpoint
):

    if isinstance(
        checkpoint,
        nn.Module
    ):

        return checkpoint.state_dict()


    if not isinstance(
        checkpoint,
        dict
    ):

        raise RuntimeError(
            "Unsupported checkpoint format."
        )


    for key in [

        "state_dict",

        "model_state_dict",

        "encoder_state_dict",

        "predictor_state_dict"
    ]:

        if key in checkpoint:

            value = checkpoint[
                key
            ]


            if isinstance(
                value,
                nn.Module
            ):

                return value.state_dict()


            if isinstance(
                value,
                dict
            ):

                return value


    tensor_items = {

        key:
            value

        for key, value
        in checkpoint.items()

        if torch.is_tensor(
            value
        )
    }


    if tensor_items:

        return tensor_items


    raise RuntimeError(
        "Could not recover state_dict."
    )


def strip_prefix_if_present(
    state_dict,
    prefix
):

    keys = list(
        state_dict.keys()
    )


    if (
        keys
        and
        all(
            key.startswith(
                prefix
            )
            for key in keys
        )
    ):

        return {

            key[
                len(
                    prefix
                ):
            ]:
                value

            for key, value
            in state_dict.items()
        }


    return state_dict


def normalize_state_dict_keys(
    state_dict
):

    result = dict(
        state_dict
    )


    for prefix in [

        "module.",

        "encoder.",

        "predictor.",

        "model."
    ]:

        result = strip_prefix_if_present(
            result,
            prefix
        )


    return result


# ==============================================================================
# 8. BASELINE K0 HASHES
# ==============================================================================

K0_HASH_BEFORE = {

    "thetaE":
        sha256_file(
            PHASE12_ENCODER_FILE
        ),

    "thetaD":
        sha256_file(
            PHASE12_DECODER_FILE
        ),

    "thetaP":
        sha256_file(
            PHASE12_PREDICTOR_FILE
        ),

    "F_E":
        sha256_file(
            PHASE13_ENCODER_FISHER_FILE
        ),

    "F_P":
        sha256_file(
            PHASE13_PREDICTOR_FISHER_FILE
        ),

    "parameter_reference":
        sha256_file(
            PHASE13_PARAMETER_REFERENCE_FILE
        )
}


print(
    "=" * 155
)

print(
    "PHASE 17 — FEEDBACK CANDIDATE DEFINITION AND "
    "EWC-REGULARIZED CLOSED-LOOP ADAPTATION"
)

print(
    "=" * 155
)

print(
    f"\n[DEVICE] {DEVICE}"
)

print(
    f"[CONFIG] Random Seed                     : {RANDOM_SEED}"
)

print(
    f"[CONFIG] Learning Rate                   : {LEARNING_RATE}"
)

print(
    f"[CONFIG] Maximum Epochs                  : {MAX_ADAPTATION_EPOCHS}"
)

print(
    f"[CONFIG] Minimum Rank Separation         : "
    f"{MINIMUM_RANK_SEPARATION}"
)

print(
    f"[CONFIG] Ranking Margin                  : {RANKING_MARGIN}"
)

print(
    f"[CONFIG] EWC λ Candidates                : "
    f"{EWC_LAMBDA_CANDIDATES}"
)

print(
    f"[CONFIG] Severe Prediction Drift Limit   : "
    f"{SEVERE_REFERENCE_DRIFT_THRESHOLD}"
)


# ==============================================================================
# 9. LOAD PHASE-7 FEATURE SCHEMA
# ==============================================================================

with open(
    PHASE7_FEATURE_SCHEMA_FILE,
    "r",
    encoding="utf-8"
) as handle:

    phase7_schema = json.load(
        handle
    )


NODE_FEATURE_NAMES = list(
    phase7_schema[
        "node_feature_names"
    ]
)


EDGE_FEATURE_NAMES = list(
    phase7_schema[
        "edge_feature_names"
    ]
)


if len(
    NODE_FEATURE_NAMES
) != NODE_FEATURE_DIM:

    raise RuntimeError(
        "Phase-7 node-feature dimension mismatch."
    )


if len(
    EDGE_FEATURE_NAMES
) != EDGE_FEATURE_DIM:

    raise RuntimeError(
        "Phase-7 edge-feature dimension mismatch."
    )


# ==============================================================================
# 10. EXACT PHASE-7 VOCABULARIES
# ==============================================================================

HYBRIDIZATION_TYPES = [

    "UNSPECIFIED",

    "S",

    "SP",

    "SP2",

    "SP3",

    "SP3D",

    "SP3D2",

    "OTHER"
]


CHIRALITY_TYPES = [

    "CHI_UNSPECIFIED",

    "CHI_TETRAHEDRAL_CW",

    "CHI_TETRAHEDRAL_CCW",

    "CHI_OTHER"
]


BOND_TYPES = [

    "SINGLE",

    "DOUBLE",

    "TRIPLE",

    "AROMATIC",

    "OTHER"
]


BOND_STEREO_TYPES = [

    "STEREONONE",

    "STEREOANY",

    "STEREOZ",

    "STEREOE",

    "STEREOCIS",

    "STEREOTRANS",

    "OTHER"
]


EXPECTED_NODE_FEATURE_NAMES = [

    "atomic_number",

    "degree",

    "formal_charge",

    "aromatic",

    "in_ring",

    "hydrogen_neighbor_count",

    "atomic_mass",

    "implicit_valence",

    "explicit_valence"
]


EXPECTED_NODE_FEATURE_NAMES += [

    f"hybridization_{value}"

    for value
    in HYBRIDIZATION_TYPES
]


EXPECTED_NODE_FEATURE_NAMES += [

    f"chirality_{value}"

    for value
    in CHIRALITY_TYPES
]


EXPECTED_EDGE_FEATURE_NAMES = [

    f"bond_type_{value}"

    for value
    in BOND_TYPES
]


EXPECTED_EDGE_FEATURE_NAMES += [

    "is_conjugated",

    "is_in_ring"
]


EXPECTED_EDGE_FEATURE_NAMES += [

    f"stereo_{value}"

    for value
    in BOND_STEREO_TYPES
]


if NODE_FEATURE_NAMES != EXPECTED_NODE_FEATURE_NAMES:

    raise RuntimeError(
        "Phase-17 node schema differs from saved Phase-7 schema."
    )


if EDGE_FEATURE_NAMES != EXPECTED_EDGE_FEATURE_NAMES:

    raise RuntimeError(
        "Phase-17 edge schema differs from saved Phase-7 schema."
    )


print(
    "\n[PASS] Exact Phase-7 graph schema verified."
)


# ==============================================================================
# 11. PHASE-7 FEATURIZATION
# ==============================================================================

def one_hot(
    value,
    categories
):

    output = [
        0.0
        for _
        in categories
    ]


    if value in categories:

        index = categories.index(
            value
        )

    else:

        index = (
            len(
                categories
            )
            -
            1
        )


    output[
        index
    ] = 1.0


    return output


def safe_implicit_valence(
    atom
):

    try:

        return float(
            atom.GetImplicitValence()
        )

    except Exception:

        return 0.0


def safe_explicit_valence(
    atom
):

    try:

        return float(
            atom.GetExplicitValence()
        )

    except Exception:

        return 0.0


def atom_feature_vector(
    atom
):

    hydrogen_neighbor_count = sum(

        1

        for neighbor
        in atom.GetNeighbors()

        if neighbor.GetAtomicNum()
        ==
        1
    )


    features = [

        float(
            atom.GetAtomicNum()
        ),

        float(
            atom.GetDegree()
        ),

        float(
            atom.GetFormalCharge()
        ),

        float(
            atom.GetIsAromatic()
        ),

        float(
            atom.IsInRing()
        ),

        float(
            hydrogen_neighbor_count
        ),

        float(
            atom.GetMass()
        ),

        safe_implicit_valence(
            atom
        ),

        safe_explicit_valence(
            atom
        )
    ]


    features.extend(
        one_hot(
            str(
                atom.GetHybridization()
            ),
            HYBRIDIZATION_TYPES
        )
    )


    features.extend(
        one_hot(
            str(
                atom.GetChiralTag()
            ),
            CHIRALITY_TYPES
        )
    )


    if len(
        features
    ) != NODE_FEATURE_DIM:

        raise RuntimeError(
            "Node feature dimension mismatch."
        )


    return features


def bond_feature_vector(
    bond
):

    features = []


    features.extend(
        one_hot(
            str(
                bond.GetBondType()
            ),
            BOND_TYPES
        )
    )


    features.extend(
        [

            float(
                bond.GetIsConjugated()
            ),

            float(
                bond.IsInRing()
            )
        ]
    )


    features.extend(
        one_hot(
            str(
                bond.GetStereo()
            ),
            BOND_STEREO_TYPES
        )
    )


    if len(
        features
    ) != EDGE_FEATURE_DIM:

        raise RuntimeError(
            "Edge feature dimension mismatch."
        )


    return features


def build_node_features(
    mol
):

    return torch.tensor(
        [

            atom_feature_vector(
                atom
            )

            for atom
            in mol.GetAtoms()
        ],
        dtype=torch.float32
    )


def build_edges(
    mol
):

    source = []

    target = []

    attributes = []


    for bond in mol.GetBonds():

        i = bond.GetBeginAtomIdx()

        j = bond.GetEndAtomIdx()


        feature = bond_feature_vector(
            bond
        )


        source.extend(
            [
                i,
                j
            ]
        )


        target.extend(
            [
                j,
                i
            ]
        )


        attributes.extend(
            [
                feature,
                feature
            ]
        )


    if len(
        source
    ) == 0:

        edge_index = torch.empty(
            (
                2,
                0
            ),
            dtype=torch.long
        )


        edge_attr = torch.empty(
            (
                0,
                EDGE_FEATURE_DIM
            ),
            dtype=torch.float32
        )


    else:

        edge_index = torch.tensor(
            [
                source,
                target
            ],
            dtype=torch.long
        )


        edge_attr = torch.tensor(
            attributes,
            dtype=torch.float32
        )


    return (
        edge_index,
        edge_attr
    )


# ==============================================================================
# 12. PHASE-7 GEOMETRY
# ==============================================================================

def generate_primary_conformer(
    mol
):

    params = AllChem.ETKDGv3()


    params.randomSeed = (
        RANDOM_SEED
    )


    params.useRandomCoords = False

    params.enforceChirality = True

    params.clearConfs = True


    return AllChem.EmbedMolecule(
        mol,
        params
    )


def generate_fallback_conformer(
    mol
):

    mol.RemoveAllConformers()


    params = AllChem.ETKDGv3()


    params.randomSeed = (
        RANDOM_SEED
    )


    params.useRandomCoords = True

    params.enforceChirality = True

    params.clearConfs = True


    return AllChem.EmbedMolecule(
        mol,
        params
    )


def optimize_geometry(
    mol,
    conformer_id
):

    result = {

        "method":
            "NONE",

        "status":
            "NOT_ATTEMPTED",

        "converged":
            False,

        "energy":
            None
    }


    try:

        has_mmff = (
            AllChem.MMFFHasAllMoleculeParams(
                mol
            )
        )

    except Exception:

        has_mmff = False


    if has_mmff:

        try:

            properties = (
                AllChem.MMFFGetMoleculeProperties(
                    mol,
                    mmffVariant="MMFF94"
                )
            )


            ff = (
                AllChem.MMFFGetMoleculeForceField(
                    mol,
                    properties,
                    confId=conformer_id
                )
            )


            if ff is not None:

                ff.Initialize()


                status = ff.Minimize(
                    maxIts=
                        MAX_OPTIMIZATION_ITERS
                )


                result.update(
                    {

                        "method":
                            "MMFF94",

                        "status":
                            (
                                "CONVERGED"
                                if status == 0
                                else
                                "MAX_ITERS_REACHED"
                            ),

                        "converged":
                            bool(
                                status == 0
                            ),

                        "energy":
                            float(
                                ff.CalcEnergy()
                            )
                    }
                )


                return result


        except Exception:

            pass


    try:

        has_uff = (
            AllChem.UFFHasAllMoleculeParams(
                mol
            )
        )

    except Exception:

        has_uff = False


    if has_uff:

        try:

            ff = (
                AllChem.UFFGetMoleculeForceField(
                    mol,
                    confId=conformer_id
                )
            )


            if ff is not None:

                ff.Initialize()


                status = ff.Minimize(
                    maxIts=
                        MAX_OPTIMIZATION_ITERS
                )


                result.update(
                    {

                        "method":
                            "UFF",

                        "status":
                            (
                                "CONVERGED"
                                if status == 0
                                else
                                "MAX_ITERS_REACHED"
                            ),

                        "converged":
                            bool(
                                status == 0
                            ),

                        "energy":
                            float(
                                ff.CalcEnergy()
                            )
                    }
                )


                return result


        except Exception:

            pass


    result[
        "status"
    ] = "NO_SUPPORTED_FORCE_FIELD"


    return result


def centered_coordinates(
    mol,
    conformer_id
):

    coordinates = np.asarray(
        mol.GetConformer(
            conformer_id
        ).GetPositions(),
        dtype=np.float32
    )


    if (
        coordinates.ndim != 2
        or
        coordinates.shape[
            1
        ] != 3
    ):

        raise RuntimeError(
            "Invalid coordinate matrix."
        )


    if not np.isfinite(
        coordinates
    ).all():

        raise RuntimeError(
            "Non-finite coordinate detected."
        )


    coordinates = (
        coordinates
        -
        coordinates.mean(
            axis=0,
            keepdims=True
        )
    )


    return torch.tensor(
        coordinates,
        dtype=torch.float32
    )


def build_phase17_graph(
    candidate_id,
    smiles
):

    result = {

        "success":
            False,

        "candidate_id":
            candidate_id,

        "canonical_smiles":
            smiles,

        "failure_reason":
            ""
    }


    try:

        mol = Chem.MolFromSmiles(
            smiles
        )


        if mol is None:

            raise RuntimeError(
                "RDKit_PARSE_FAILED"
            )


        mol_h = Chem.AddHs(
            mol
        )


        if mol_h is None:

            raise RuntimeError(
                "ADD_HYDROGEN_FAILED"
            )


        conformer_id = -1


        try:

            conformer_id = (
                generate_primary_conformer(
                    mol_h
                )
            )

        except Exception:

            conformer_id = -1


        if conformer_id >= 0:

            conformer_method = (
                "ETKDGv3_PRIMARY"
            )

        else:

            try:

                conformer_id = (
                    generate_fallback_conformer(
                        mol_h
                    )
                )

            except Exception:

                conformer_id = -1


            if conformer_id < 0:

                raise RuntimeError(
                    "CONFORMER_GENERATION_FAILED"
                )


            conformer_method = (
                "ETKDGv3_RANDOM_COORDS_FALLBACK"
            )


        optimization = optimize_geometry(
            mol_h,
            conformer_id
        )


        x = build_node_features(
            mol_h
        )


        edge_index, edge_attr = (
            build_edges(
                mol_h
            )
        )


        pos = centered_coordinates(
            mol_h,
            conformer_id
        )


        if x.shape[
            1
        ] != NODE_FEATURE_DIM:

            raise RuntimeError(
                "NODE_DIMENSION_MISMATCH"
            )


        if edge_attr.shape[
            1
        ] != EDGE_FEATURE_DIM:

            raise RuntimeError(
                "EDGE_DIMENSION_MISMATCH"
            )


        if x.shape[
            0
        ] != pos.shape[
            0
        ]:

            raise RuntimeError(
                "NODE_POSITION_COUNT_MISMATCH"
            )


        if (
            edge_index.shape[
                1
            ]
            !=
            edge_attr.shape[
                0
            ]
        ):

            raise RuntimeError(
                "EDGE_COUNT_MISMATCH"
            )


        if not (
            torch.isfinite(
                x
            ).all()
            and
            torch.isfinite(
                edge_attr
            ).all()
            and
            torch.isfinite(
                pos
            ).all()
        ):

            raise RuntimeError(
                "NONFINITE_GRAPH_VALUES"
            )


        result.update(
            {

                "success":
                    True,

                "x":
                    x,

                "pos":
                    pos,

                "edge_index":
                    edge_index,

                "edge_attr":
                    edge_attr,

                "heavy_atom_count":
                    int(
                        mol.GetNumHeavyAtoms()
                    ),

                "total_atom_count":
                    int(
                        mol_h.GetNumAtoms()
                    ),

                "num_nodes":
                    int(
                        x.shape[
                            0
                        ]
                    ),

                "num_bonds":
                    int(
                        mol_h.GetNumBonds()
                    ),

                "num_directed_edges":
                    int(
                        edge_index.shape[
                            1
                        ]
                    ),

                "conformer_method":
                    conformer_method,

                "optimization_method":
                    optimization[
                        "method"
                    ],

                "optimization_status":
                    optimization[
                        "status"
                    ],

                "optimization_converged":
                    optimization[
                        "converged"
                    ],

                "conformer_energy":
                    optimization[
                        "energy"
                    ]
            }
        )


    except Exception as exc:

        result[
            "failure_reason"
        ] = str(
            exc
        )


    return result


# ==============================================================================
# 13. EXACT PHASE-9 NORMALIZATION
# ==============================================================================

with open(
    PHASE9_NORMALIZATION_FILE,
    "r",
    encoding="utf-8"
) as handle:

    phase9_normalization = json.load(
        handle
    )


NORMALIZED_FEATURE_NAMES = list(
    phase9_normalization[
        "standardized_features"
    ]
)


NORMALIZATION_INDICES = list(
    phase9_normalization[
        "standardized_feature_indices"
    ]
)


EXPECTED_NORMALIZED_FEATURES = [

    "atomic_number",

    "degree",

    "formal_charge",

    "hydrogen_neighbor_count",

    "atomic_mass",

    "implicit_valence",

    "explicit_valence"
]


EXPECTED_NORMALIZATION_INDICES = [

    NODE_FEATURE_NAMES.index(
        feature
    )

    for feature
    in EXPECTED_NORMALIZED_FEATURES
]


if NORMALIZED_FEATURE_NAMES != EXPECTED_NORMALIZED_FEATURES:

    raise RuntimeError(
        "Phase-9 normalized feature order mismatch."
    )


if NORMALIZATION_INDICES != EXPECTED_NORMALIZATION_INDICES:

    raise RuntimeError(
        "Phase-9 normalization index mismatch."
    )


training_feature_mean = torch.tensor(
    [

        float(
            phase9_normalization[
                "mean"
            ][
                feature
            ]
        )

        for feature
        in NORMALIZED_FEATURE_NAMES
    ],
    dtype=torch.float32
)


training_feature_std = torch.tensor(
    [

        float(
            phase9_normalization[
                "std"
            ][
                feature
            ]
        )

        for feature
        in NORMALIZED_FEATURE_NAMES
    ],
    dtype=torch.float32
)


def normalize_node_features(
    x
):

    output = x.clone().float()


    indices = torch.tensor(
        NORMALIZATION_INDICES,
        dtype=torch.long,
        device=output.device
    )


    output[
        :,
        indices
    ] = (

        output[
            :,
            indices
        ]

        -

        training_feature_mean.to(
            output.device
        )

    ) / training_feature_std.to(
        output.device
    )


    return output


print(
    "[PASS] Exact Phase-9 normalization loaded."
)


# ==============================================================================
# 14. EXACT PHASE-9 SCATTER MEAN
# ==============================================================================

def scatter_mean(
    values,
    index,
    dim_size
):

    output = torch.zeros(
        (
            dim_size,
            values.shape[
                1
            ]
        ),
        dtype=values.dtype,
        device=values.device
    )


    counts = torch.zeros(
        (
            dim_size,
            1
        ),
        dtype=values.dtype,
        device=values.device
    )


    output.index_add_(
        0,
        index,
        values
    )


    counts.index_add_(
        0,
        index,
        torch.ones(
            (
                values.shape[
                    0
                ],
                1
            ),
            dtype=values.dtype,
            device=values.device
        )
    )


    return (
        output
        /
        counts.clamp(
            min=1.0
        )
    )


# ==============================================================================
# 15. EXACT PHASE-9 EGNN
# ==============================================================================

class EGNNLayer(
    nn.Module
):

    def __init__(
        self,
        hidden_dim,
        edge_dim,
        coordinate_update_scale=0.10
    ):

        super().__init__()


        self.coordinate_update_scale = (
            coordinate_update_scale
        )


        self.edge_mlp = nn.Sequential(

            nn.Linear(
                (
                    hidden_dim
                    *
                    2
                    +
                    1
                    +
                    edge_dim
                ),
                hidden_dim
            ),

            nn.SiLU(),

            nn.Linear(
                hidden_dim,
                hidden_dim
            ),

            nn.SiLU()
        )


        self.coordinate_mlp = nn.Sequential(

            nn.Linear(
                hidden_dim,
                hidden_dim
            ),

            nn.SiLU(),

            nn.Linear(
                hidden_dim,
                1
            ),

            nn.Tanh()
        )


        self.node_mlp = nn.Sequential(

            nn.Linear(
                hidden_dim
                *
                2,
                hidden_dim
            ),

            nn.SiLU(),

            nn.Linear(
                hidden_dim,
                hidden_dim
            )
        )


        self.node_norm = nn.LayerNorm(
            hidden_dim
        )


    def forward(
        self,
        h,
        pos,
        edge_index,
        edge_attr
    ):

        source = edge_index[
            0
        ]


        target = edge_index[
            1
        ]


        relative = (

            pos[
                source
            ]

            -

            pos[
                target
            ]
        )


        squared_distance = (
            relative
            *
            relative
        ).sum(
            dim=1,
            keepdim=True
        )


        edge_input = torch.cat(
            [

                h[
                    source
                ],

                h[
                    target
                ],

                squared_distance,

                edge_attr
            ],
            dim=1
        )


        messages = self.edge_mlp(
            edge_input
        )


        coordinate_weight = (

            self.coordinate_mlp(
                messages
            )

            *

            self.coordinate_update_scale
        )


        coordinate_messages = (

            relative

            *

            coordinate_weight
        )


        coordinate_aggregate = scatter_mean(
            coordinate_messages,
            source,
            h.shape[
                0
            ]
        )


        new_pos = (

            pos

            +

            coordinate_aggregate
        )


        aggregated_messages = scatter_mean(
            messages,
            source,
            h.shape[
                0
            ]
        )


        node_delta = self.node_mlp(
            torch.cat(
                [
                    h,
                    aggregated_messages
                ],
                dim=1
            )
        )


        new_h = self.node_norm(

            h

            +

            node_delta
        )


        return (
            new_h,
            new_pos
        )


class GeometryAwareVariationalEncoder(
    nn.Module
):

    def __init__(
        self
    ):

        super().__init__()


        self.node_embedding = nn.Sequential(

            nn.Linear(
                NODE_FEATURE_DIM,
                HIDDEN_DIM
            ),

            nn.SiLU(),

            nn.LayerNorm(
                HIDDEN_DIM
            )
        )


        self.egnn_layers = nn.ModuleList(
            [

                EGNNLayer(
                    HIDDEN_DIM,
                    EDGE_FEATURE_DIM,
                    COORDINATE_UPDATE_SCALE
                )

                for _
                in range(
                    NUM_EGNN_LAYERS
                )
            ]
        )


        self.graph_mlp = nn.Sequential(

            nn.Linear(
                HIDDEN_DIM,
                HIDDEN_DIM
            ),

            nn.SiLU(),

            nn.LayerNorm(
                HIDDEN_DIM
            )
        )


        self.mu_head = nn.Linear(
            HIDDEN_DIM,
            LATENT_DIM
        )


        self.logvar_head = nn.Linear(
            HIDDEN_DIM,
            LATENT_DIM
        )


    def forward(
        self,
        x,
        pos,
        edge_index,
        edge_attr,
        batch,
        deterministic=True
    ):

        h = self.node_embedding(
            x
        )


        current_pos = pos


        for layer in self.egnn_layers:

            h, current_pos = layer(

                h,

                current_pos,

                edge_index,

                edge_attr
            )


        graph_embedding = scatter_mean(
            h,
            batch,
            int(
                batch.max().item()
            )
            +
            1
        )


        graph_embedding = self.graph_mlp(
            graph_embedding
        )


        mu = self.mu_head(
            graph_embedding
        )


        logvar = self.logvar_head(
            graph_embedding
        )


        if deterministic:

            z = mu

        else:

            safe_logvar = torch.clamp(
                logvar,
                min=-20.0,
                max=20.0
            )


            std = torch.exp(
                0.5
                *
                safe_logvar
            )


            z = (

                mu

                +

                std
                *
                torch.randn_like(
                    std
                )
            )


        return {

            "mu":
                mu,

            "logvar":
                logvar,

            "z":
                z
        }


# ==============================================================================
# 16. PHASE-12 PREDICTOR
# ==============================================================================

class Phase12TargetPredictor(
    nn.Module
):

    def __init__(
        self
    ):

        super().__init__()


        self.network = nn.Sequential(

            nn.Linear(
                32,
                64
            ),

            nn.LayerNorm(
                64
            ),

            nn.SiLU(),

            nn.Dropout(
                0.15
            ),

            nn.Linear(
                64,
                32
            ),

            nn.LayerNorm(
                32
            ),

            nn.SiLU(),

            nn.Dropout(
                0.15
            ),

            nn.Linear(
                32,
                130
            )
        )


    def forward(
        self,
        x
    ):

        return self.network(
            x
        )


# ==============================================================================
# 17. LOAD BASELINE MODELS
# ==============================================================================

baseline_encoder = (
    GeometryAwareVariationalEncoder()
    .to(
        DEVICE
    )
)


baseline_predictor = (
    Phase12TargetPredictor()
    .to(
        DEVICE
    )
)


encoder_checkpoint = torch.load(
    PHASE12_ENCODER_FILE,
    map_location="cpu",
    weights_only=False
)


predictor_checkpoint = torch.load(
    PHASE12_PREDICTOR_FILE,
    map_location="cpu",
    weights_only=False
)


baseline_encoder.load_state_dict(
    normalize_state_dict_keys(
        extract_state_dict(
            encoder_checkpoint
        )
    ),
    strict=True
)


baseline_predictor.load_state_dict(
    normalize_state_dict_keys(
        extract_state_dict(
            predictor_checkpoint
        )
    ),
    strict=True
)


ENCODER_PARAMETER_COUNT = sum(

    parameter.numel()

    for parameter
    in baseline_encoder.parameters()
)


PREDICTOR_PARAMETER_COUNT = sum(

    parameter.numel()

    for parameter
    in baseline_predictor.parameters()
)


if ENCODER_PARAMETER_COUNT != 70274:

    raise RuntimeError(
        "Encoder parameter-count mismatch."
    )


if PREDICTOR_PARAMETER_COUNT != 8674:

    raise RuntimeError(
        "Predictor parameter-count mismatch."
    )


baseline_encoder.eval()

baseline_predictor.eval()


print(
    "[PASS] Baseline θE* and θP* loaded."
)


# ==============================================================================
# 18. PHASE-12 LATENT NORMALIZATION
# ==============================================================================

with open(
    PHASE12_LATENT_NORMALIZATION_FILE,
    "r",
    encoding="utf-8"
) as handle:

    latent_normalization = json.load(
        handle
    )


latent_mean = torch.tensor(
    latent_normalization[
        "mean"
    ],
    dtype=torch.float32,
    device=DEVICE
)


latent_safe_std = torch.tensor(
    latent_normalization[
        "safe_std"
    ],
    dtype=torch.float32,
    device=DEVICE
)


if latent_mean.shape != (
    LATENT_DIM,
):

    raise RuntimeError(
        "Latent mean shape mismatch."
    )


if latent_safe_std.shape != (
    LATENT_DIM,
):

    raise RuntimeError(
        "Latent std shape mismatch."
    )


# ==============================================================================
# 19. TARGET VOCABULARY
# ==============================================================================

target_vocab_df = pd.read_csv(
    PHASE12_TARGET_VOCAB_FILE,
    dtype=str
)


target_vocab_df[
    "target_index"
] = pd.to_numeric(
    target_vocab_df[
        "target_index"
    ],
    errors="raise"
).astype(
    int
)


target_vocab_df = (
    target_vocab_df
    .sort_values(
        "target_index"
    )
    .reset_index(
        drop=True
    )
)


if len(
    target_vocab_df
) != TARGET_COUNT:

    raise RuntimeError(
        "Target vocabulary count mismatch."
    )


if target_vocab_df[
    "target_index"
].tolist() != list(
    range(
        TARGET_COUNT
    )
):

    raise RuntimeError(
        "Target vocabulary ordering mismatch."
    )


target_id_to_index = dict(
    zip(
        target_vocab_df[
            "target_id"
        ].astype(str),
        target_vocab_df[
            "target_index"
        ].astype(int)
    )
)


target_index_to_id = dict(
    zip(
        target_vocab_df[
            "target_index"
        ].astype(int),
        target_vocab_df[
            "target_id"
        ].astype(str)
    )
)


target_index_to_name = dict(
    zip(
        target_vocab_df[
            "target_index"
        ].astype(int),
        target_vocab_df[
            "target_name"
        ].fillna("").astype(str)
    )
)


# ==============================================================================
# 20. LOAD PHASE-13 FISHER AND θ* REFERENCES
# ==============================================================================

encoder_fisher_checkpoint = torch.load(
    PHASE13_ENCODER_FISHER_FILE,
    map_location="cpu",
    weights_only=False
)


predictor_fisher_checkpoint = torch.load(
    PHASE13_PREDICTOR_FISHER_FILE,
    map_location="cpu",
    weights_only=False
)


parameter_reference = torch.load(
    PHASE13_PARAMETER_REFERENCE_FILE,
    map_location="cpu",
    weights_only=False
)


if "fisher" not in encoder_fisher_checkpoint:

    raise RuntimeError(
        "Encoder Fisher file missing fisher."
    )


if "fisher" not in predictor_fisher_checkpoint:

    raise RuntimeError(
        "Predictor Fisher file missing fisher."
    )


encoder_fisher_cpu = (
    encoder_fisher_checkpoint[
        "fisher"
    ]
)


predictor_fisher_cpu = (
    predictor_fisher_checkpoint[
        "fisher"
    ]
)


encoder_theta_star_cpu = (
    parameter_reference[
        "encoder"
    ][
        "theta_star"
    ]
)


predictor_theta_star_cpu = (
    parameter_reference[
        "predictor"
    ][
        "theta_star"
    ]
)


encoder_parameter_names = {

    name

    for name, _
    in baseline_encoder.named_parameters()
}


predictor_parameter_names = {

    name

    for name, _
    in baseline_predictor.named_parameters()
}


if set(
    encoder_fisher_cpu.keys()
) != encoder_parameter_names:

    raise RuntimeError(
        "Encoder Fisher parameter names mismatch."
    )


if set(
    predictor_fisher_cpu.keys()
) != predictor_parameter_names:

    raise RuntimeError(
        "Predictor Fisher parameter names mismatch."
    )


if set(
    encoder_theta_star_cpu.keys()
) != encoder_parameter_names:

    raise RuntimeError(
        "Encoder θ* names mismatch."
    )


if set(
    predictor_theta_star_cpu.keys()
) != predictor_parameter_names:

    raise RuntimeError(
        "Predictor θ* names mismatch."
    )


for name, parameter in baseline_encoder.named_parameters():

    if not torch.equal(
        parameter.detach().cpu(),
        encoder_theta_star_cpu[
            name
        ]
    ):

        raise RuntimeError(
            f"Encoder anchor mismatch: {name}"
        )


for name, parameter in baseline_predictor.named_parameters():

    if not torch.equal(
        parameter.detach().cpu(),
        predictor_theta_star_cpu[
            name
        ]
    ):

        raise RuntimeError(
            f"Predictor anchor mismatch: {name}"
        )


encoder_fisher = {

    name:
        tensor.to(
            DEVICE
        )

    for name, tensor
    in encoder_fisher_cpu.items()
}


predictor_fisher = {

    name:
        tensor.to(
            DEVICE
        )

    for name, tensor
    in predictor_fisher_cpu.items()
}


encoder_theta_star = {

    name:
        tensor.to(
            DEVICE
        )

    for name, tensor
    in encoder_theta_star_cpu.items()
}


predictor_theta_star = {

    name:
        tensor.to(
            DEVICE
        )

    for name, tensor
    in predictor_theta_star_cpu.items()
}


print(
    "[PASS] Phase-13 Fisher and θ* anchors loaded and verified."
)


# ==============================================================================
# 21. LOAD S1
# ==============================================================================

S1_df = pd.read_csv(
    PHASE16_SELECTED_FILE,
    dtype={
        "candidate_id":
            str,

        "canonical_smiles":
            str
    }
)


if len(
    S1_df
) != EXPECTED_S1_SIZE:

    print(
        f"[WARNING] Expected {EXPECTED_S1_SIZE} candidates; "
        f"found {len(S1_df)}."
    )


required_s1_columns = [

    "candidate_id",

    "canonical_smiles",

    "qed",

    "sa_score",

    "sa_utility",

    "biological_relevance_top5_mean",

    "utility_score",

    "utility_rank",

    "acquisition_score",

    "acquisition_rank"
]


for position in range(
    1,
    6
):

    required_s1_columns.append(
        f"top_target_{position}_id"
    )


missing_s1_columns = [

    column

    for column
    in required_s1_columns

    if column not in S1_df.columns
]


if missing_s1_columns:

    raise RuntimeError(
        f"S1 missing required columns: {missing_s1_columns}"
    )


if S1_df[
    "candidate_id"
].duplicated().any():

    raise RuntimeError(
        "Duplicate S1 candidate IDs."
    )


if S1_df[
    "canonical_smiles"
].duplicated().any():

    raise RuntimeError(
        "Duplicate S1 structures."
    )


numeric_s1_columns = [

    "qed",

    "sa_score",

    "sa_utility",

    "biological_relevance_top5_mean",

    "utility_score",

    "utility_rank",

    "acquisition_score",

    "acquisition_rank"
]


for column in numeric_s1_columns:

    S1_df[
        column
    ] = pd.to_numeric(
        S1_df[
            column
        ],
        errors="raise"
    )


frozen_top5_indices = {}


for _, row in S1_df.iterrows():

    candidate_id = str(
        row[
            "candidate_id"
        ]
    )


    indices = []


    for position in range(
        1,
        6
    ):

        target_id = str(
            row[
                f"top_target_{position}_id"
            ]
        )


        if target_id not in target_id_to_index:

            raise RuntimeError(
                f"Unknown target ID {target_id} "
                f"for {candidate_id}."
            )


        indices.append(
            int(
                target_id_to_index[
                    target_id
                ]
            )
        )


    frozen_top5_indices[
        candidate_id
    ] = indices


S1_df.to_csv(
    PHASE17_CANDIDATE_MANIFEST_FILE,
    index=False,
    encoding="utf-8"
)


print(
    f"\n[PASS] S1 loaded: {len(S1_df)} candidates."
)


# ==============================================================================
# 22. PRECOMPUTE S1 GRAPHS
# ==============================================================================

print(
    "\n"
    +
    "=" * 155
)

print(
    "PRECOMPUTING S1 MOLECULAR GRAPHS"
)

print(
    "=" * 155
)


S1_GRAPHS = {}

graph_manifest_records = []


for row_number, (_, row) in enumerate(
    S1_df.iterrows(),
    start=1
):

    candidate_id = str(
        row[
            "candidate_id"
        ]
    )


    graph = build_phase17_graph(
        candidate_id,
        str(
            row[
                "canonical_smiles"
            ]
        )
    )


    graph_manifest_records.append(
        {

            "candidate_id":
                candidate_id,

            "graph_success":
                bool(
                    graph[
                        "success"
                    ]
                ),

            "num_nodes":
                graph.get(
                    "num_nodes"
                ),

            "num_bonds":
                graph.get(
                    "num_bonds"
                ),

            "num_directed_edges":
                graph.get(
                    "num_directed_edges"
                ),

            "conformer_method":
                graph.get(
                    "conformer_method",
                    ""
                ),

            "optimization_method":
                graph.get(
                    "optimization_method",
                    ""
                ),

            "optimization_status":
                graph.get(
                    "optimization_status",
                    ""
                ),

            "failure_reason":
                graph.get(
                    "failure_reason",
                    ""
                )
        }
    )


    if graph[
        "success"
    ]:

        S1_GRAPHS[
            candidate_id
        ] = {

            "x":
                graph[
                    "x"
                ],

            "pos":
                graph[
                    "pos"
                ],

            "edge_index":
                graph[
                    "edge_index"
                ],

            "edge_attr":
                graph[
                    "edge_attr"
                ]
        }


    print(
        f"[GRAPH] {row_number:02d}/{len(S1_df):02d} "
        f"{candidate_id} "
        f"{'PASS' if graph['success'] else 'FAIL'}"
    )


graph_manifest_df = pd.DataFrame(
    graph_manifest_records
)


graph_manifest_df.to_csv(
    PHASE17_GRAPH_MANIFEST_FILE,
    index=False,
    encoding="utf-8"
)


if len(
    S1_GRAPHS
) != len(
    S1_df
):

    failed_df = graph_manifest_df[
        ~graph_manifest_df[
            "graph_success"
        ]
    ]


    raise RuntimeError(
        "Not all S1 graphs were reconstructed:\n"
        +
        failed_df[
            [
                "candidate_id",
                "failure_reason"
            ]
        ].to_string(
            index=False
        )
    )


print(
    "\n[PASS] All S1 molecular graphs precomputed successfully."
)


# ==============================================================================
# 23. MOLECULE-LEVEL TRAIN / VALIDATION SPLIT
# ==============================================================================

candidate_ids = (
    S1_df[
        "candidate_id"
    ]
    .astype(str)
    .tolist()
)


rng = np.random.default_rng(
    RANDOM_SEED
)


shuffled_ids = list(
    candidate_ids
)


rng.shuffle(
    shuffled_ids
)


train_count = int(
    round(
        len(
            shuffled_ids
        )
        *
        FEEDBACK_TRAIN_FRACTION
    )
)


train_count = min(

    max(
        train_count,
        1
    ),

    len(
        shuffled_ids
    )
    -
    1
)


TRAIN_IDS = set(
    shuffled_ids[
        :train_count
    ]
)


VALIDATION_IDS = set(
    shuffled_ids[
        train_count:
    ]
)


if TRAIN_IDS & VALIDATION_IDS:

    raise RuntimeError(
        "Train/validation molecule overlap."
    )


train_candidates_df = S1_df[
    S1_df[
        "candidate_id"
    ].isin(
        TRAIN_IDS
    )
].copy()


validation_candidates_df = S1_df[
    S1_df[
        "candidate_id"
    ].isin(
        VALIDATION_IDS
    )
].copy()


train_candidates_df.to_csv(
    PHASE17_TRAIN_CANDIDATES_FILE,
    index=False,
    encoding="utf-8"
)


validation_candidates_df.to_csv(
    PHASE17_VALIDATION_CANDIDATES_FILE,
    index=False,
    encoding="utf-8"
)


print(
    f"\n[PASS] Feedback-train candidates      : "
    f"{len(train_candidates_df)}"
)

print(
    f"[PASS] Feedback-validation candidates : "
    f"{len(validation_candidates_df)}"
)

print(
    "[PASS] Candidate split performed BEFORE pair construction."
)


# ==============================================================================
# 24. PREFERENCE PAIRS
# ==============================================================================

def construct_preference_pairs(
    candidate_subset,
    split_name
):

    subset = (
        candidate_subset
        .copy()
        .sort_values(
            "utility_rank"
        )
        .reset_index(
            drop=True
        )
    )


    records = []


    for i in range(
        len(
            subset
        )
    ):

        for j in range(
            i + 1,
            len(
                subset
            )
        ):

            first = subset.iloc[
                i
            ]


            second = subset.iloc[
                j
            ]


            first_rank = int(
                first[
                    "utility_rank"
                ]
            )


            second_rank = int(
                second[
                    "utility_rank"
                ]
            )


            rank_difference = abs(
                first_rank
                -
                second_rank
            )


            if (
                rank_difference
                <
                MINIMUM_RANK_SEPARATION
            ):

                continue


            if first_rank < second_rank:

                preferred = first

                comparison = second

            else:

                preferred = second

                comparison = first


            utility_difference = abs(
                float(
                    preferred[
                        "utility_score"
                    ]
                )
                -
                float(
                    comparison[
                        "utility_score"
                    ]
                )
            )


            raw_weight = (
                utility_difference
                if USE_PAIR_WEIGHTS
                else
                1.0
            )


            records.append(
                {

                    "preferred_candidate_id":
                        str(
                            preferred[
                                "candidate_id"
                            ]
                        ),

                    "comparison_candidate_id":
                        str(
                            comparison[
                                "candidate_id"
                            ]
                        ),

                    "preferred_utility_rank":
                        int(
                            preferred[
                                "utility_rank"
                            ]
                        ),

                    "comparison_utility_rank":
                        int(
                            comparison[
                                "utility_rank"
                            ]
                        ),

                    "rank_difference":
                        rank_difference,

                    "preferred_baseline_utility":
                        float(
                            preferred[
                                "utility_score"
                            ]
                        ),

                    "comparison_baseline_utility":
                        float(
                            comparison[
                                "utility_score"
                            ]
                        ),

                    "utility_difference":
                        utility_difference,

                    "raw_pair_weight":
                        float(
                            raw_weight
                        ),

                    "feedback_split":
                        split_name
                }
            )


    pairs = pd.DataFrame(
        records
    )


    if pairs.empty:

        raise RuntimeError(
            f"No preference pairs generated for {split_name}."
        )


    mean_weight = float(
        pairs[
            "raw_pair_weight"
        ].mean()
    )


    if mean_weight <= 0:

        mean_weight = 1.0


    pairs[
        "normalized_pair_weight"
    ] = (

        pairs[
            "raw_pair_weight"
        ]

        /

        mean_weight
    )


    return pairs


train_pairs_df = construct_preference_pairs(
    train_candidates_df,
    "train"
)


validation_pairs_df = construct_preference_pairs(
    validation_candidates_df,
    "validation"
)


feedback_pairs_df = pd.concat(
    [
        train_pairs_df,
        validation_pairs_df
    ],
    ignore_index=True
)


feedback_pairs_df.to_csv(
    PHASE17_PAIRS_FILE,
    index=False,
    encoding="utf-8"
)


print(
    f"\n[PASS] Training preference pairs   : "
    f"{len(train_pairs_df)}"
)

print(
    f"[PASS] Validation preference pairs : "
    f"{len(validation_pairs_df)}"
)


# ==============================================================================
# 25. FIXED QED / SA LOOKUPS
# ==============================================================================

candidate_lookup = (
    S1_df
    .set_index(
        "candidate_id"
    )
)


QED_LOOKUP = {

    candidate_id:
        float(
            candidate_lookup.loc[
                candidate_id,
                "qed"
            ]
        )

    for candidate_id
    in candidate_ids
}


SAUTILITY_LOOKUP = {

    candidate_id:
        float(
            candidate_lookup.loc[
                candidate_id,
                "sa_utility"
            ]
        )

    for candidate_id
    in candidate_ids
}


# ==============================================================================
# 26. MODEL FORWARD HELPERS
# ==============================================================================

def forward_graph_mu(
    encoder_model,
    graph
):

    x = normalize_node_features(
        graph[
            "x"
        ]
    ).to(
        DEVICE
    )


    pos = (
        graph[
            "pos"
        ]
        .float()
        .to(
            DEVICE
        )
    )


    edge_index = (
        graph[
            "edge_index"
        ]
        .long()
        .to(
            DEVICE
        )
    )


    edge_attr = (
        graph[
            "edge_attr"
        ]
        .float()
        .to(
            DEVICE
        )
    )


    batch = torch.zeros(
        x.shape[
            0
        ],
        dtype=torch.long,
        device=DEVICE
    )


    output = encoder_model(

        x=x,

        pos=pos,

        edge_index=edge_index,

        edge_attr=edge_attr,

        batch=batch,

        deterministic=True
    )


    return output[
        "mu"
    ].squeeze(
        0
    )


def normalize_latent(
    mu
):

    return (

        mu

        -

        latent_mean

    ) / latent_safe_std


def candidate_batch_scores(
    encoder_model,
    predictor_model,
    ids
):

    mu_list = []


    for candidate_id in ids:

        mu_list.append(
            forward_graph_mu(
                encoder_model,
                S1_GRAPHS[
                    candidate_id
                ]
            )
        )


    mu_tensor = torch.stack(
        mu_list,
        dim=0
    )


    normalized_mu = (

        mu_tensor

        -

        latent_mean.unsqueeze(
            0
        )

    ) / latent_safe_std.unsqueeze(
        0
    )


    logits = predictor_model(
        normalized_mu
    )


    probabilities = torch.sigmoid(
        logits
    )


    biological_scores = []


    for row_index, candidate_id in enumerate(
        ids
    ):

        target_indices = torch.tensor(
            frozen_top5_indices[
                candidate_id
            ],
            dtype=torch.long,
            device=DEVICE
        )


        biological_scores.append(
            probabilities[
                row_index,
                target_indices
            ].mean()
        )


    biological_scores = torch.stack(
        biological_scores
    )


    fixed_qed = torch.tensor(
        [

            QED_LOOKUP[
                candidate_id
            ]

            for candidate_id
            in ids
        ],
        dtype=torch.float32,
        device=DEVICE
    )


    fixed_sa = torch.tensor(
        [

            SAUTILITY_LOOKUP[
                candidate_id
            ]

            for candidate_id
            in ids
        ],
        dtype=torch.float32,
        device=DEVICE
    )


    utility = (

        WEIGHT_QED
        *
        fixed_qed

        +

        WEIGHT_SA
        *
        fixed_sa

        +

        WEIGHT_BIO
        *
        biological_scores
    )


    return {

        "mu":
            mu_tensor,

        "probabilities":
            probabilities,

        "biological_relevance":
            biological_scores,

        "utility":
            utility
    }


# ==============================================================================
# 27. FEEDBACK LOSS
# ==============================================================================

def feedback_loss_from_scores(
    ids,
    utilities,
    pairs_df
):

    id_to_position = {

        candidate_id:
            index

        for index, candidate_id
        in enumerate(
            ids
        )
    }


    preferred_positions = torch.tensor(
        [

            id_to_position[
                str(
                    candidate_id
                )
            ]

            for candidate_id
            in pairs_df[
                "preferred_candidate_id"
            ]
        ],
        dtype=torch.long,
        device=DEVICE
    )


    comparison_positions = torch.tensor(
        [

            id_to_position[
                str(
                    candidate_id
                )
            ]

            for candidate_id
            in pairs_df[
                "comparison_candidate_id"
            ]
        ],
        dtype=torch.long,
        device=DEVICE
    )


    weights = torch.tensor(
        pairs_df[
            "normalized_pair_weight"
        ].to_numpy(
            dtype=np.float32
        ),
        dtype=torch.float32,
        device=DEVICE
    )


    difference = (

        utilities[
            preferred_positions
        ]

        -

        utilities[
            comparison_positions
        ]
    )


    individual_losses = F.softplus(

        RANKING_MARGIN

        -

        difference
    )


    weighted_loss = (

        individual_losses
        *
        weights

    ).sum() / weights.sum().clamp(
        min=1e-12
    )


    pair_accuracy = (
        difference
        >
        0
    ).float().mean()


    margin_accuracy = (
        difference
        >=
        RANKING_MARGIN
    ).float().mean()


    return (

        weighted_loss,

        pair_accuracy,

        margin_accuracy
    )


# ==============================================================================
# 28. EWC
# ==============================================================================

def calculate_ewc(
    encoder_model,
    predictor_model
):

    encoder_penalty = torch.zeros(
        (),
        dtype=torch.float32,
        device=DEVICE
    )


    predictor_penalty = torch.zeros(
        (),
        dtype=torch.float32,
        device=DEVICE
    )


    for name, parameter in (
        encoder_model.named_parameters()
    ):

        encoder_penalty = (

            encoder_penalty

            +

            torch.sum(
                encoder_fisher[
                    name
                ]
                *
                (
                    parameter
                    -
                    encoder_theta_star[
                        name
                    ]
                ).pow(
                    2
                )
            )
        )


    for name, parameter in (
        predictor_model.named_parameters()
    ):

        predictor_penalty = (

            predictor_penalty

            +

            torch.sum(
                predictor_fisher[
                    name
                ]
                *
                (
                    parameter
                    -
                    predictor_theta_star[
                        name
                    ]
                ).pow(
                    2
                )
            )
        )


    return {

        "encoder_raw":
            encoder_penalty,

        "predictor_raw":
            predictor_penalty,

        "total_raw":
            (
                encoder_penalty
                +
                predictor_penalty
            ),

        "encoder_normalized":
            (
                encoder_penalty
                /
                float(
                    ENCODER_PARAMETER_COUNT
                )
            ),

        "predictor_normalized":
            (
                predictor_penalty
                /
                float(
                    PREDICTOR_PARAMETER_COUNT
                )
            )
    }


# ==============================================================================
# 29. PARAMETER DRIFT HELPER
# ==============================================================================

def model_l2_drift(
    model,
    theta_star
):

    total_squared = 0.0


    with torch.no_grad():

        for name, parameter in (
            model.named_parameters()
        ):

            delta = (

                parameter.detach()

                -

                theta_star[
                    name
                ]
            )


            total_squared += float(
                torch.sum(
                    delta.pow(
                        2
                    )
                ).item()
            )


    return math.sqrt(
        total_squared
    )


# ==============================================================================
# 30. LOAD FIXED 128-MOLECULE FISHER REFERENCE
# ==============================================================================

D_fisher = pd.read_csv(
    PHASE8_FISHER_REFERENCE_FILE,
    dtype=str,
    keep_default_na=False
)


if len(
    D_fisher
) != EXPECTED_FISHER_REFERENCE_SIZE:

    raise RuntimeError(
        f"Expected {EXPECTED_FISHER_REFERENCE_SIZE} "
        f"Fisher-reference molecules; found {len(D_fisher)}."
    )


def resolve_reference_graph_file(
    row
):

    for column in [

        "resolved_graph_file",

        "graph_file",

        "graph_path",

        "file_path",

        "path"
    ]:

        if column in row.index:

            path_text = str(
                row[
                    column
                ]
            ).strip()


            if path_text:

                path = Path(
                    path_text
                )


                if path.exists():

                    return path


    graph_id = str(
        row[
            "graph_id"
        ]
    )


    for extension in [

        ".pt",

        ".pth"
    ]:

        direct = (
            GRAPH_DIR
            /
            f"{graph_id}{extension}"
        )


        if direct.exists():

            return direct


    matches = list(
        GRAPH_DIR.rglob(
            f"*{graph_id}*.pt"
        )
    )


    if matches:

        return matches[
            0
        ]


    raise FileNotFoundError(
        f"Graph file not found for {graph_id}"
    )


def extract_saved_graph(
    graph_object
):

    if not isinstance(
        graph_object,
        dict
    ):

        raise RuntimeError(
            "Unexpected saved graph format."
        )


    required = [

        "x",

        "pos",

        "edge_index",

        "edge_attr"
    ]


    missing = [

        key

        for key
        in required

        if key not in graph_object
    ]


    if missing:

        raise RuntimeError(
            f"Saved graph missing fields: {missing}"
        )


    return {

        "x":
            graph_object[
                "x"
            ].float(),

        "pos":
            graph_object[
                "pos"
            ].float(),

        "edge_index":
            graph_object[
                "edge_index"
            ].long(),

        "edge_attr":
            graph_object[
                "edge_attr"
            ].float()
    }


REFERENCE_GRAPHS = {}

reference_metadata = []


print(
    "\n"
    +
    "=" * 155
)

print(
    "LOADING FIXED 128-MOLECULE FISHER REFERENCE"
)

print(
    "=" * 155
)


for _, row in D_fisher.iterrows():

    graph_id = str(
        row[
            "graph_id"
        ]
    )


    graph_path = (
        resolve_reference_graph_file(
            row
        )
    )


    graph_object = torch.load(
        graph_path,
        map_location="cpu",
        weights_only=False
    )


    REFERENCE_GRAPHS[
        graph_id
    ] = extract_saved_graph(
        graph_object
    )


    reference_metadata.append(
        {

            "graph_id":
                graph_id,

            "drug_id":
                (
                    str(
                        row[
                            "drug_id"
                        ]
                    )
                    if "drug_id" in row.index
                    else
                    graph_id
                ),

            "graph_file":
                str(
                    graph_path
                )
        }
    )


print(
    f"[PASS] Fixed Fisher-reference graphs loaded: "
    f"{len(REFERENCE_GRAPHS)}"
)


# ==============================================================================
# 31. REFERENCE EVALUATION
# ==============================================================================

def evaluate_reference_system(
    encoder_model,
    predictor_model
):

    encoder_model.eval()

    predictor_model.eval()


    graph_ids = []

    latents = []

    predictions = []


    with torch.no_grad():

        for graph_id in D_fisher[
            "graph_id"
        ].astype(str):

            mu = forward_graph_mu(
                encoder_model,
                REFERENCE_GRAPHS[
                    graph_id
                ]
            )


            normalized_mu = normalize_latent(
                mu
            )


            probabilities = torch.sigmoid(
                predictor_model(
                    normalized_mu.unsqueeze(
                        0
                    )
                )
            ).squeeze(
                0
            )


            graph_ids.append(
                graph_id
            )


            latents.append(
                mu.detach()
                .cpu()
                .numpy()
            )


            predictions.append(
                probabilities.detach()
                .cpu()
                .numpy()
            )


    return {

        "graph_ids":
            graph_ids,

        "latents":
            np.asarray(
                latents,
                dtype=np.float32
            ),

        "predictions":
            np.asarray(
                predictions,
                dtype=np.float32
            )
    }


BASELINE_REFERENCE = evaluate_reference_system(
    baseline_encoder,
    baseline_predictor
)


print(
    "[PASS] Baseline reference latent/prediction state cached."
)


# ==============================================================================
# 32. REFERENCE DRIFT
# ==============================================================================

def calculate_reference_drift(
    encoder_model,
    predictor_model
):

    current = evaluate_reference_system(
        encoder_model,
        predictor_model
    )


    latent_drift = np.linalg.norm(

        current[
            "latents"
        ]

        -

        BASELINE_REFERENCE[
            "latents"
        ],

        axis=1
    )


    absolute_prediction_difference = np.abs(

        current[
            "predictions"
        ]

        -

        BASELINE_REFERENCE[
            "predictions"
        ]
    )


    mean_prediction_drift = (
        absolute_prediction_difference.mean(
            axis=1
        )
    )


    max_prediction_drift = (
        absolute_prediction_difference.max(
            axis=1
        )
    )


    severe_mask = (

        max_prediction_drift

        >

        SEVERE_REFERENCE_DRIFT_THRESHOLD
    )


    return {

        "current":
            current,

        "latent_drift":
            latent_drift,

        "mean_prediction_drift":
            mean_prediction_drift,

        "max_prediction_drift":
            max_prediction_drift,

        "severe_mask":
            severe_mask,

        "severe_count":
            int(
                severe_mask.sum()
            ),

        "mean_latent_drift":
            float(
                latent_drift.mean()
            ),

        "max_latent_drift":
            float(
                latent_drift.max()
            ),

        "mean_prediction_drift_overall":
            float(
                mean_prediction_drift.mean()
            ),

        "max_prediction_drift_overall":
            float(
                max_prediction_drift.max()
            )
    }


# ==============================================================================
# 33. FEEDBACK EVALUATION
# ==============================================================================

def evaluate_feedback_subset(
    encoder_model,
    predictor_model,
    candidate_subset,
    pairs_df
):

    ids = (
        candidate_subset[
            "candidate_id"
        ]
        .astype(str)
        .tolist()
    )


    encoder_model.eval()

    predictor_model.eval()


    with torch.no_grad():

        scores = candidate_batch_scores(
            encoder_model,
            predictor_model,
            ids
        )


        (
            loss,
            pair_accuracy,
            margin_accuracy
        ) = feedback_loss_from_scores(
            ids,
            scores[
                "utility"
            ],
            pairs_df
        )


    return {

        "loss":
            float(
                loss.item()
            ),

        "accuracy":
            float(
                pair_accuracy.item()
            ),

        "margin_accuracy":
            float(
                margin_accuracy.item()
            )
    }


BASELINE_TRAIN_FEEDBACK = evaluate_feedback_subset(

    baseline_encoder,

    baseline_predictor,

    train_candidates_df,

    train_pairs_df
)


BASELINE_VALIDATION_FEEDBACK = evaluate_feedback_subset(

    baseline_encoder,

    baseline_predictor,

    validation_candidates_df,

    validation_pairs_df
)


print(
    "\nBaseline K0 Feedback Performance:"
)

print(
    f"  Train Loss                : "
    f"{BASELINE_TRAIN_FEEDBACK['loss']:.6f}"
)

print(
    f"  Train Pair Accuracy       : "
    f"{BASELINE_TRAIN_FEEDBACK['accuracy']:.4f}"
)

print(
    f"  Train Margin Accuracy     : "
    f"{BASELINE_TRAIN_FEEDBACK['margin_accuracy']:.4f}"
)

print(
    f"  Validation Loss           : "
    f"{BASELINE_VALIDATION_FEEDBACK['loss']:.6f}"
)

print(
    f"  Validation Pair Accuracy  : "
    f"{BASELINE_VALIDATION_FEEDBACK['accuracy']:.4f}"
)

print(
    f"  Validation Margin Accuracy: "
    f"{BASELINE_VALIDATION_FEEDBACK['margin_accuracy']:.4f}"
)


# ==============================================================================
# 34. LEXICOGRAPHIC EPOCH COMPARATOR
#
# PRIORITY:
#
# 1. validation pair accuracy ↑
# 2. validation margin accuracy ↑
# 3. validation feedback loss ↓
#
# Rounded accuracy protects against microscopic floating-point differences.
# ==============================================================================

def epoch_metric_tuple(
    validation_accuracy,
    validation_margin_accuracy,
    validation_feedback_loss
):

    return (

        round(
            float(
                validation_accuracy
            ),
            8
        ),

        round(
            float(
                validation_margin_accuracy
            ),
            8
        ),

        -float(
            validation_feedback_loss
        )
    )


def metric_is_better(
    current_metric,
    best_metric
):

    if best_metric is None:

        return True


    return (
        current_metric
        >
        best_metric
    )


# ==============================================================================
# 35. TRAIN ONE λ
# ==============================================================================

def train_lambda_candidate(
    lambda_ewc
):

    # --------------------------------------------------------------------------
    # Every λ starts from EXACT K0.
    # --------------------------------------------------------------------------

    encoder_model = copy.deepcopy(
        baseline_encoder
    ).to(
        DEVICE
    )


    predictor_model = copy.deepcopy(
        baseline_predictor
    ).to(
        DEVICE
    )


    for parameter in encoder_model.parameters():

        parameter.requires_grad_(
            True
        )


    for parameter in predictor_model.parameters():

        parameter.requires_grad_(
            True
        )


    optimizer = torch.optim.AdamW(

        list(
            encoder_model.parameters()
        )

        +

        list(
            predictor_model.parameters()
        ),

        lr=
            LEARNING_RATE,

        weight_decay=
            WEIGHT_DECAY
    )


    train_ids = (
        train_candidates_df[
            "candidate_id"
        ]
        .astype(str)
        .tolist()
    )


    validation_ids = (
        validation_candidates_df[
            "candidate_id"
        ]
        .astype(str)
        .tolist()
    )


    history = []


    best_metric = None

    best_epoch = None

    best_encoder_state = None

    best_predictor_state = None


    best_validation_accuracy = None

    best_validation_margin_accuracy = None

    best_validation_loss = None


    patience_counter = 0


    for epoch in range(
        1,
        MAX_ADAPTATION_EPOCHS
        +
        1
    ):

        # ======================================================================
        # TRAIN
        # ======================================================================

        encoder_model.train()

        predictor_model.train()


        optimizer.zero_grad(
            set_to_none=True
        )


        train_scores = candidate_batch_scores(
            encoder_model,
            predictor_model,
            train_ids
        )


        (
            feedback_train_loss,
            train_accuracy,
            train_margin_accuracy
        ) = feedback_loss_from_scores(
            train_ids,
            train_scores[
                "utility"
            ],
            train_pairs_df
        )


        ewc_before_step = calculate_ewc(
            encoder_model,
            predictor_model
        )


        total_train_loss = (

            feedback_train_loss

            +

            float(
                lambda_ewc
            )
            *
            ewc_before_step[
                "total_raw"
            ]
        )


        total_train_loss.backward()


        optimizer.step()


        # ======================================================================
        # VALIDATION
        # ======================================================================

        encoder_model.eval()

        predictor_model.eval()


        with torch.no_grad():

            validation_scores = candidate_batch_scores(
                encoder_model,
                predictor_model,
                validation_ids
            )


            (
                feedback_validation_loss,
                validation_accuracy,
                validation_margin_accuracy
            ) = feedback_loss_from_scores(
                validation_ids,
                validation_scores[
                    "utility"
                ],
                validation_pairs_df
            )


            ewc_after_step = calculate_ewc(
                encoder_model,
                predictor_model
            )


            total_validation_loss = (

                feedback_validation_loss

                +

                float(
                    lambda_ewc
                )
                *
                ewc_after_step[
                    "total_raw"
                ]
            )


        encoder_drift = model_l2_drift(
            encoder_model,
            encoder_theta_star
        )


        predictor_drift = model_l2_drift(
            predictor_model,
            predictor_theta_star
        )


        current_metric = epoch_metric_tuple(

            validation_accuracy.item(),

            validation_margin_accuracy.item(),

            feedback_validation_loss.item()
        )


        is_best_epoch = metric_is_better(
            current_metric,
            best_metric
        )


        history.append(
            {

                "lambda_ewc":
                    float(
                        lambda_ewc
                    ),

                "epoch":
                    epoch,

                "feedback_train_loss":
                    float(
                        feedback_train_loss.item()
                    ),

                "feedback_validation_loss":
                    float(
                        feedback_validation_loss.item()
                    ),

                "pairwise_train_accuracy":
                    float(
                        train_accuracy.item()
                    ),

                "pairwise_validation_accuracy":
                    float(
                        validation_accuracy.item()
                    ),

                "margin_train_accuracy":
                    float(
                        train_margin_accuracy.item()
                    ),

                "margin_validation_accuracy":
                    float(
                        validation_margin_accuracy.item()
                    ),

                "raw_encoder_EWC":
                    float(
                        ewc_after_step[
                            "encoder_raw"
                        ].item()
                    ),

                "raw_predictor_EWC":
                    float(
                        ewc_after_step[
                            "predictor_raw"
                        ].item()
                    ),

                "raw_total_EWC":
                    float(
                        ewc_after_step[
                            "total_raw"
                        ].item()
                    ),

                "normalized_encoder_EWC":
                    float(
                        ewc_after_step[
                            "encoder_normalized"
                        ].item()
                    ),

                "normalized_predictor_EWC":
                    float(
                        ewc_after_step[
                            "predictor_normalized"
                        ].item()
                    ),

                "total_train_loss":
                    float(
                        total_train_loss.item()
                    ),

                "total_validation_loss":
                    float(
                        total_validation_loss.item()
                    ),

                "encoder_L2_drift":
                    float(
                        encoder_drift
                    ),

                "predictor_L2_drift":
                    float(
                        predictor_drift
                    ),

                "checkpoint_selected_at_epoch":
                    bool(
                        is_best_epoch
                    )
            }
        )


        print(
            f"[λ={lambda_ewc:7.1f}] "
            f"epoch={epoch:02d} | "
            f"train_loss={feedback_train_loss.item():.6f} | "
            f"val_loss={feedback_validation_loss.item():.6f} | "
            f"train_acc={train_accuracy.item():.4f} | "
            f"val_acc={validation_accuracy.item():.4f} | "
            f"val_margin={validation_margin_accuracy.item():.4f} | "
            f"EWC={ewc_after_step['total_raw'].item():.8f} | "
            f"{'BEST' if is_best_epoch else ''}"
        )


        # ======================================================================
        # CORRECTED CHECKPOINT SELECTION
        # ======================================================================

        if is_best_epoch:

            best_metric = (
                current_metric
            )


            best_epoch = (
                epoch
            )


            best_validation_accuracy = float(
                validation_accuracy.item()
            )


            best_validation_margin_accuracy = float(
                validation_margin_accuracy.item()
            )


            best_validation_loss = float(
                feedback_validation_loss.item()
            )


            best_encoder_state = {

                name:
                    tensor.detach()
                    .cpu()
                    .clone()

                for name, tensor
                in encoder_model.state_dict().items()
            }


            best_predictor_state = {

                name:
                    tensor.detach()
                    .cpu()
                    .clone()

                for name, tensor
                in predictor_model.state_dict().items()
            }


            patience_counter = 0


        else:

            patience_counter += 1


        if (
            patience_counter
            >=
            EARLY_STOPPING_PATIENCE
        ):

            print(
                f"[EARLY STOP] λ={lambda_ewc} "
                f"after epoch {epoch}; "
                f"best epoch={best_epoch}."
            )

            break


    if (
        best_encoder_state is None
        or
        best_predictor_state is None
    ):

        raise RuntimeError(
            f"No checkpoint selected for λ={lambda_ewc}."
        )


    # --------------------------------------------------------------------------
    # Restore lexicographically best epoch.
    # --------------------------------------------------------------------------

    encoder_model.load_state_dict(
        best_encoder_state,
        strict=True
    )


    predictor_model.load_state_dict(
        best_predictor_state,
        strict=True
    )


    encoder_model.eval()

    predictor_model.eval()


    # --------------------------------------------------------------------------
    # Final feedback metrics at the chosen checkpoint.
    # --------------------------------------------------------------------------

    train_feedback = evaluate_feedback_subset(

        encoder_model,

        predictor_model,

        train_candidates_df,

        train_pairs_df
    )


    validation_feedback = evaluate_feedback_subset(

        encoder_model,

        predictor_model,

        validation_candidates_df,

        validation_pairs_df
    )


    # --------------------------------------------------------------------------
    # Retention metrics.
    # --------------------------------------------------------------------------

    retention = calculate_reference_drift(

        encoder_model,

        predictor_model
    )


    final_ewc = calculate_ewc(

        encoder_model,

        predictor_model
    )


    train_accuracy_drop = (

        BASELINE_TRAIN_FEEDBACK[
            "accuracy"
        ]

        -

        train_feedback[
            "accuracy"
        ]
    )


    validation_accuracy_change = (

        validation_feedback[
            "accuracy"
        ]

        -

        BASELINE_VALIDATION_FEEDBACK[
            "accuracy"
        ]
    )


    # --------------------------------------------------------------------------
    # Acceptance guardrails.
    # --------------------------------------------------------------------------

    validation_accuracy_acceptable = (

        validation_feedback[
            "accuracy"
        ]

        >=

        BASELINE_VALIDATION_FEEDBACK[
            "accuracy"
        ]
    )


    train_accuracy_acceptable = (

        train_accuracy_drop

        <=

        MAX_ALLOWED_TRAIN_ACCURACY_DROP
    )


    severe_drift_acceptable = (

        retention[
            "severe_count"
        ]

        <=

        MAX_ALLOWED_SEVERE_REFERENCE_MOLECULES
    )


    adaptation_acceptable = (

        validation_accuracy_acceptable

        and

        train_accuracy_acceptable

        and

        severe_drift_acceptable
    )


    return {

        "candidate_type":
            "adapted",

        "lambda":
            float(
                lambda_ewc
            ),

        "best_epoch":
            int(
                best_epoch
            ),

        "best_validation_accuracy":
            float(
                best_validation_accuracy
            ),

        "best_validation_margin_accuracy":
            float(
                best_validation_margin_accuracy
            ),

        "best_validation_loss":
            float(
                best_validation_loss
            ),

        "encoder_state":
            best_encoder_state,

        "predictor_state":
            best_predictor_state,

        "history":
            history,

        "train_feedback":
            train_feedback,

        "validation_feedback":
            validation_feedback,

        "retention":
            retention,

        "train_accuracy_drop":
            float(
                train_accuracy_drop
            ),

        "validation_accuracy_change":
            float(
                validation_accuracy_change
            ),

        "validation_accuracy_acceptable":
            bool(
                validation_accuracy_acceptable
            ),

        "train_accuracy_acceptable":
            bool(
                train_accuracy_acceptable
            ),

        "severe_drift_acceptable":
            bool(
                severe_drift_acceptable
            ),

        "adaptation_acceptable":
            bool(
                adaptation_acceptable
            ),

        "raw_encoder_fisher_drift":
            float(
                final_ewc[
                    "encoder_raw"
                ].item()
            ),

        "raw_predictor_fisher_drift":
            float(
                final_ewc[
                    "predictor_raw"
                ].item()
            ),

        "raw_total_fisher_drift":
            float(
                final_ewc[
                    "total_raw"
                ].item()
            ),

        "normalized_encoder_fisher_drift":
            float(
                final_ewc[
                    "encoder_normalized"
                ].item()
            ),

        "normalized_predictor_fisher_drift":
            float(
                final_ewc[
                    "predictor_normalized"
                ].item()
            )
    }


# ==============================================================================
# 36. RUN λ SEARCH
# ==============================================================================

print(
    "\n"
    +
    "=" * 155
)

print(
    "EWC λ SEARCH — CORRECTED CHECKPOINT SELECTION"
)

print(
    "=" * 155
)


lambda_results = []


for lambda_value in (
    EWC_LAMBDA_CANDIDATES
):

    print(
        "\n"
        +
        "-" * 155
    )

    print(
        f"TRAINING λ_EWC = {lambda_value}"
    )

    print(
        "-" * 155
    )


    result = train_lambda_candidate(
        lambda_value
    )


    lambda_results.append(
        result
    )


# ==============================================================================
# 37. CREATE EXPLICIT K0 MODEL-SELECTION CANDIDATE
# ==============================================================================

baseline_result = {

    "candidate_type":
        "baseline_K0",

    "lambda":
        0.0,

    "best_epoch":
        0,

    "train_feedback":
        BASELINE_TRAIN_FEEDBACK,

    "validation_feedback":
        BASELINE_VALIDATION_FEEDBACK,

    "retention":
        {

            "mean_latent_drift":
                0.0,

            "max_latent_drift":
                0.0,

            "mean_prediction_drift_overall":
                0.0,

            "max_prediction_drift_overall":
                0.0,

            "severe_count":
                0
        },

    "train_accuracy_drop":
        0.0,

    "validation_accuracy_change":
        0.0,

    "validation_accuracy_acceptable":
        True,

    "train_accuracy_acceptable":
        True,

    "severe_drift_acceptable":
        True,

    "adaptation_acceptable":
        True,

    "raw_encoder_fisher_drift":
        0.0,

    "raw_predictor_fisher_drift":
        0.0,

    "raw_total_fisher_drift":
        0.0,

    "normalized_encoder_fisher_drift":
        0.0,

    "normalized_predictor_fisher_drift":
        0.0
}


# ==============================================================================
# 38. BUILD λ SEARCH TABLE
# ==============================================================================

lambda_search_records = []


for result in lambda_results:

    lambda_search_records.append(
        {

            "candidate_type":
                "adapted",

            "lambda_ewc":
                result[
                    "lambda"
                ],

            "best_epoch":
                result[
                    "best_epoch"
                ],

            "train_feedback_loss":
                result[
                    "train_feedback"
                ][
                    "loss"
                ],

            "train_pairwise_accuracy":
                result[
                    "train_feedback"
                ][
                    "accuracy"
                ],

            "train_margin_accuracy":
                result[
                    "train_feedback"
                ][
                    "margin_accuracy"
                ],

            "validation_feedback_loss":
                result[
                    "validation_feedback"
                ][
                    "loss"
                ],

            "validation_pairwise_accuracy":
                result[
                    "validation_feedback"
                ][
                    "accuracy"
                ],

            "validation_margin_accuracy":
                result[
                    "validation_feedback"
                ][
                    "margin_accuracy"
                ],

            "train_accuracy_drop_vs_K0":
                result[
                    "train_accuracy_drop"
                ],

            "validation_accuracy_change_vs_K0":
                result[
                    "validation_accuracy_change"
                ],

            "mean_reference_latent_drift":
                result[
                    "retention"
                ][
                    "mean_latent_drift"
                ],

            "maximum_reference_latent_drift":
                result[
                    "retention"
                ][
                    "max_latent_drift"
                ],

            "mean_reference_prediction_drift":
                result[
                    "retention"
                ][
                    "mean_prediction_drift_overall"
                ],

            "maximum_reference_prediction_drift":
                result[
                    "retention"
                ][
                    "max_prediction_drift_overall"
                ],

            "severe_reference_drift_count":
                result[
                    "retention"
                ][
                    "severe_count"
                ],

            "raw_encoder_fisher_drift":
                result[
                    "raw_encoder_fisher_drift"
                ],

            "raw_predictor_fisher_drift":
                result[
                    "raw_predictor_fisher_drift"
                ],

            "raw_total_fisher_drift":
                result[
                    "raw_total_fisher_drift"
                ],

            "normalized_encoder_fisher_drift":
                result[
                    "normalized_encoder_fisher_drift"
                ],

            "normalized_predictor_fisher_drift":
                result[
                    "normalized_predictor_fisher_drift"
                ],

            "validation_accuracy_acceptable":
                result[
                    "validation_accuracy_acceptable"
                ],

            "train_accuracy_acceptable":
                result[
                    "train_accuracy_acceptable"
                ],

            "severe_drift_acceptable":
                result[
                    "severe_drift_acceptable"
                ],

            "adaptation_acceptable":
                result[
                    "adaptation_acceptable"
                ]
        }
    )


lambda_search_df = pd.DataFrame(
    lambda_search_records
)


lambda_search_df.to_csv(
    PHASE17_LAMBDA_SEARCH_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 39. MODEL-SELECTION TABLE INCLUDING K0
# ==============================================================================

model_selection_records = [

    {

        "candidate_type":
            "baseline_K0",

        "lambda_ewc":
            0.0,

        "best_epoch":
            0,

        "train_feedback_loss":
            BASELINE_TRAIN_FEEDBACK[
                "loss"
            ],

        "train_pairwise_accuracy":
            BASELINE_TRAIN_FEEDBACK[
                "accuracy"
            ],

        "train_margin_accuracy":
            BASELINE_TRAIN_FEEDBACK[
                "margin_accuracy"
            ],

        "validation_feedback_loss":
            BASELINE_VALIDATION_FEEDBACK[
                "loss"
            ],

        "validation_pairwise_accuracy":
            BASELINE_VALIDATION_FEEDBACK[
                "accuracy"
            ],

        "validation_margin_accuracy":
            BASELINE_VALIDATION_FEEDBACK[
                "margin_accuracy"
            ],

        "train_accuracy_drop_vs_K0":
            0.0,

        "validation_accuracy_change_vs_K0":
            0.0,

        "mean_reference_latent_drift":
            0.0,

        "maximum_reference_latent_drift":
            0.0,

        "mean_reference_prediction_drift":
            0.0,

        "maximum_reference_prediction_drift":
            0.0,

        "severe_reference_drift_count":
            0,

        "raw_total_fisher_drift":
            0.0,

        "adaptation_acceptable":
            True
    }
]


for _, row in lambda_search_df.iterrows():

    model_selection_records.append(
        {

            "candidate_type":
                "adapted",

            "lambda_ewc":
                float(
                    row[
                        "lambda_ewc"
                    ]
                ),

            "best_epoch":
                int(
                    row[
                        "best_epoch"
                    ]
                ),

            "train_feedback_loss":
                float(
                    row[
                        "train_feedback_loss"
                    ]
                ),

            "train_pairwise_accuracy":
                float(
                    row[
                        "train_pairwise_accuracy"
                    ]
                ),

            "train_margin_accuracy":
                float(
                    row[
                        "train_margin_accuracy"
                    ]
                ),

            "validation_feedback_loss":
                float(
                    row[
                        "validation_feedback_loss"
                    ]
                ),

            "validation_pairwise_accuracy":
                float(
                    row[
                        "validation_pairwise_accuracy"
                    ]
                ),

            "validation_margin_accuracy":
                float(
                    row[
                        "validation_margin_accuracy"
                    ]
                ),

            "train_accuracy_drop_vs_K0":
                float(
                    row[
                        "train_accuracy_drop_vs_K0"
                    ]
                ),

            "validation_accuracy_change_vs_K0":
                float(
                    row[
                        "validation_accuracy_change_vs_K0"
                    ]
                ),

            "mean_reference_latent_drift":
                float(
                    row[
                        "mean_reference_latent_drift"
                    ]
                ),

            "maximum_reference_latent_drift":
                float(
                    row[
                        "maximum_reference_latent_drift"
                    ]
                ),

            "mean_reference_prediction_drift":
                float(
                    row[
                        "mean_reference_prediction_drift"
                    ]
                ),

            "maximum_reference_prediction_drift":
                float(
                    row[
                        "maximum_reference_prediction_drift"
                    ]
                ),

            "severe_reference_drift_count":
                int(
                    row[
                        "severe_reference_drift_count"
                    ]
                ),

            "raw_total_fisher_drift":
                float(
                    row[
                        "raw_total_fisher_drift"
                    ]
                ),

            "adaptation_acceptable":
                bool(
                    row[
                        "adaptation_acceptable"
                    ]
                )
        }
    )


model_selection_df = pd.DataFrame(
    model_selection_records
)


# ==============================================================================
# 40. SELECT PRIMARY ADAPTED MODEL
#
# IMPORTANT:
#
# Baseline K0 is the benchmark, not automatically the winner.
#
# First:
#   keep only adapted models passing retention/ranking guardrails.
#
# Then choose lexicographically:
#
#   1. validation pairwise accuracy ↑
#   2. validation margin accuracy ↑
#   3. validation feedback loss ↓
#   4. severe drift count ↓
#   5. mean prediction drift ↓
#   6. raw Fisher drift ↓
#
# If NO adapted model passes guardrails:
#
#   Phase 17 is NOT accepted as a valid adaptation.
#   K0 remains the scientifically preferred state.
# ==============================================================================

acceptable_adapted_df = model_selection_df[
    (
        model_selection_df[
            "candidate_type"
        ]
        ==
        "adapted"
    )
    &
    (
        model_selection_df[
            "adaptation_acceptable"
        ]
        ==
        True
    )
].copy()


if acceptable_adapted_df.empty:

    ADAPTATION_ACCEPTED = False

    SELECTED_LAMBDA = None

    SELECTED_EPOCH = None

    selected_result = None


    model_selection_df[
        "selected_primary"
    ] = (

        model_selection_df[
            "candidate_type"
        ]
        ==
        "baseline_K0"
    )


    print(
        "\n[WARNING] No adapted model satisfied all acceptance guardrails."
    )

    print(
        "[RESULT] K0 remains the preferred state."
    )


else:

    acceptable_adapted_df = (
        acceptable_adapted_df
        .sort_values(
            [

                "validation_pairwise_accuracy",

                "validation_margin_accuracy",

                "validation_feedback_loss",

                "severe_reference_drift_count",

                "mean_reference_prediction_drift",

                "raw_total_fisher_drift"
            ],
            ascending=[
                False,
                False,
                True,
                True,
                True,
                True
            ]
        )
        .reset_index(
            drop=True
        )
    )


    selected_model_row = (
        acceptable_adapted_df.iloc[
            0
        ]
    )


    SELECTED_LAMBDA = float(
        selected_model_row[
            "lambda_ewc"
        ]
    )


    SELECTED_EPOCH = int(
        selected_model_row[
            "best_epoch"
        ]
    )


    selected_result = next(

        result

        for result
        in lambda_results

        if float(
            result[
                "lambda"
            ]
        )
        ==
        SELECTED_LAMBDA
    )


    ADAPTATION_ACCEPTED = True


    model_selection_df[
        "selected_primary"
    ] = (

        (
            model_selection_df[
                "candidate_type"
            ]
            ==
            "adapted"
        )

        &

        np.isclose(
            model_selection_df[
                "lambda_ewc"
            ],
            SELECTED_LAMBDA
        )

        &

        (
            model_selection_df[
                "best_epoch"
            ]
            ==
            SELECTED_EPOCH
        )
    )


model_selection_df.to_csv(
    PHASE17_MODEL_SELECTION_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 41. DISPLAY MODEL SELECTION
# ==============================================================================

print(
    "\n"
    +
    "=" * 155
)

print(
    "PHASE-17 MODEL SELECTION — K0 + ADAPTED CANDIDATES"
)

print(
    "=" * 155
)


print(
    model_selection_df[
        [

            "candidate_type",

            "lambda_ewc",

            "best_epoch",

            "train_pairwise_accuracy",

            "validation_pairwise_accuracy",

            "validation_margin_accuracy",

            "validation_feedback_loss",

            "severe_reference_drift_count",

            "mean_reference_prediction_drift",

            "raw_total_fisher_drift",

            "adaptation_acceptable",

            "selected_primary"
        ]
    ].to_string(
        index=False
    )
)


# ==============================================================================
# 42. IF NO VALID ADAPTATION, STOP SAFELY AFTER SAVING AUDIT
# ==============================================================================

if not ADAPTATION_ACCEPTED:

    phase17_failure_config = {

        "phase":
            17,

        "status":
            "NO_ACCEPTABLE_ADAPTED_MODEL",

        "created_at":
            datetime.now().isoformat(),

        "baseline_validation_pair_accuracy":
            BASELINE_VALIDATION_FEEDBACK[
                "accuracy"
            ],

        "maximum_allowed_train_accuracy_drop":
            MAX_ALLOWED_TRAIN_ACCURACY_DROP,

        "severe_reference_drift_threshold":
            SEVERE_REFERENCE_DRIFT_THRESHOLD,

        "maximum_allowed_severe_reference_molecules":
            MAX_ALLOWED_SEVERE_REFERENCE_MOLECULES,

        "message":
            (
                "No adapted checkpoint satisfied ranking and "
                "retention acceptance criteria. K0 remains preferred."
            )
    }


    with open(
        PHASE17_CONFIG_FILE,
        "w",
        encoding="utf-8"
    ) as handle:

        json.dump(
            phase17_failure_config,
            handle,
            indent=4,
            ensure_ascii=False,
            default=str
        )


    raise RuntimeError(
        "\nPHASE 17 ADAPTATION REJECTED.\n\n"
        "No adapted checkpoint simultaneously preserved/improved "
        "validation preference fidelity, respected the training-ranking "
        "tolerance, and passed the severe reference-drift guardrail.\n\n"
        "K0 remains scientifically preferred.\n"
        "Review phase17_model_selection_candidates.csv before changing "
        "any hyperparameters."
    )


# ==============================================================================
# 43. BUILD FINAL ACCEPTED ADAPTED MODELS
# ==============================================================================

adapted_encoder = copy.deepcopy(
    baseline_encoder
).to(
    DEVICE
)


adapted_predictor = copy.deepcopy(
    baseline_predictor
).to(
    DEVICE
)


adapted_encoder.load_state_dict(
    selected_result[
        "encoder_state"
    ],
    strict=True
)


adapted_predictor.load_state_dict(
    selected_result[
        "predictor_state"
    ],
    strict=True
)


adapted_encoder.eval()

adapted_predictor.eval()


print(
    "\n[ACCEPTED] Adapted model passed ranking/retention guardrails."
)

print(
    f"[SELECTED] λ_EWC  = {SELECTED_LAMBDA}"
)

print(
    f"[SELECTED] Epoch  = {SELECTED_EPOCH}"
)


# ==============================================================================
# 44. SAVE TRAINING HISTORY
# ==============================================================================

all_history = []


for result in lambda_results:

    all_history.extend(
        result[
            "history"
        ]
    )


training_history_df = pd.DataFrame(
    all_history
)


training_history_df[
    "selected_lambda"
] = np.isclose(
    training_history_df[
        "lambda_ewc"
    ],
    SELECTED_LAMBDA
)


training_history_df[
    "selected_final_checkpoint"
] = (

    training_history_df[
        "selected_lambda"
    ]

    &

    (
        training_history_df[
            "epoch"
        ]
        ==
        SELECTED_EPOCH
    )
)


training_history_df.to_csv(
    PHASE17_HISTORY_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 45. SAVE ADAPTED CHECKPOINTS
# ==============================================================================

torch.save(
    {

        "phase":
            17,

        "cycle":
            CYCLE,

        "status":
            "ACCEPTED_CYCLE1_EWC_ADAPTED_ENCODER",

        "selected_lambda_ewc":
            SELECTED_LAMBDA,

        "selected_epoch":
            SELECTED_EPOCH,

        "state_dict":
            adapted_encoder.state_dict(),

        "baseline_checkpoint":
            str(
                PHASE12_ENCODER_FILE
            ),

        "fisher_source":
            str(
                PHASE13_ENCODER_FISHER_FILE
            )
    },
    PHASE17_ENCODER_FILE
)


torch.save(
    {

        "phase":
            17,

        "cycle":
            CYCLE,

        "status":
            "ACCEPTED_CYCLE1_EWC_ADAPTED_PREDICTOR",

        "selected_lambda_ewc":
            SELECTED_LAMBDA,

        "selected_epoch":
            SELECTED_EPOCH,

        "state_dict":
            adapted_predictor.state_dict(),

        "baseline_checkpoint":
            str(
                PHASE12_PREDICTOR_FILE
            ),

        "fisher_source":
            str(
                PHASE13_PREDICTOR_FISHER_FILE
            )
    },
    PHASE17_PREDICTOR_FILE
)


# ==============================================================================
# 46. PARAMETER DRIFT
# ==============================================================================

parameter_drift_records = []


def append_parameter_drift(
    component,
    model,
    theta_star,
    fisher
):

    for name, parameter in (
        model.named_parameters()
    ):

        delta = (

            parameter.detach()

            -

            theta_star[
                name
            ]
        )


        absolute_delta = (
            delta.abs()
        )


        fisher_tensor = (
            fisher[
                name
            ]
        )


        raw_fisher_drift = float(
            torch.sum(
                fisher_tensor
                *
                delta.pow(
                    2
                )
            ).item()
        )


        parameter_drift_records.append(
            {

                "model_component":
                    component,

                "parameter_name":
                    name,

                "number_of_elements":
                    int(
                        parameter.numel()
                    ),

                "mean_absolute_change":
                    float(
                        absolute_delta.mean().item()
                    ),

                "maximum_absolute_change":
                    float(
                        absolute_delta.max().item()
                    ),

                "L1_change":
                    float(
                        absolute_delta.sum().item()
                    ),

                "L2_change":
                    float(
                        torch.linalg.vector_norm(
                            delta
                        ).item()
                    ),

                "mean_Fisher":
                    float(
                        fisher_tensor.mean().item()
                    ),

                "maximum_Fisher":
                    float(
                        fisher_tensor.max().item()
                    ),

                "raw_Fisher_weighted_drift":
                    raw_fisher_drift,

                "normalized_Fisher_weighted_drift":
                    (
                        raw_fisher_drift
                        /
                        float(
                            parameter.numel()
                        )
                    )
            }
        )


append_parameter_drift(

    "encoder",

    adapted_encoder,

    encoder_theta_star,

    encoder_fisher
)


append_parameter_drift(

    "predictor",

    adapted_predictor,

    predictor_theta_star,

    predictor_fisher
)


parameter_drift_df = pd.DataFrame(
    parameter_drift_records
)


parameter_drift_df.to_csv(
    PHASE17_PARAMETER_DRIFT_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 47. FISHER-WEIGHTED DRIFT
# ==============================================================================

selected_ewc = calculate_ewc(
    adapted_encoder,
    adapted_predictor
)


fisher_drift_df = pd.DataFrame(
    [

        {

            "component":
                "encoder",

            "parameter_elements":
                ENCODER_PARAMETER_COUNT,

            "raw_fisher_weighted_drift":
                float(
                    selected_ewc[
                        "encoder_raw"
                    ].item()
                ),

            "normalized_fisher_weighted_drift":
                float(
                    selected_ewc[
                        "encoder_normalized"
                    ].item()
                )
        },

        {

            "component":
                "predictor",

            "parameter_elements":
                PREDICTOR_PARAMETER_COUNT,

            "raw_fisher_weighted_drift":
                float(
                    selected_ewc[
                        "predictor_raw"
                    ].item()
                ),

            "normalized_fisher_weighted_drift":
                float(
                    selected_ewc[
                        "predictor_normalized"
                    ].item()
                )
        }
    ]
)


fisher_drift_df.to_csv(
    PHASE17_FISHER_DRIFT_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 48. FINAL REFERENCE RETENTION
# ==============================================================================

FINAL_REFERENCE = calculate_reference_drift(

    adapted_encoder,

    adapted_predictor
)


reference_latent_records = []

reference_prediction_records = []


for index, metadata in enumerate(
    reference_metadata
):

    baseline_latent = (
        BASELINE_REFERENCE[
            "latents"
        ][
            index
        ]
    )


    adapted_latent = (
        FINAL_REFERENCE[
            "current"
        ][
            "latents"
        ][
            index
        ]
    )


    baseline_probs = (
        BASELINE_REFERENCE[
            "predictions"
        ][
            index
        ]
    )


    adapted_probs = (
        FINAL_REFERENCE[
            "current"
        ][
            "predictions"
        ][
            index
        ]
    )


    baseline_top_index = int(
        np.argmax(
            baseline_probs
        )
    )


    adapted_top_index = int(
        np.argmax(
            adapted_probs
        )
    )


    reference_latent_records.append(
        {

            "graph_id":
                metadata[
                    "graph_id"
                ],

            "drug_id":
                metadata[
                    "drug_id"
                ],

            "baseline_latent_norm":
                float(
                    np.linalg.norm(
                        baseline_latent
                    )
                ),

            "adapted_latent_norm":
                float(
                    np.linalg.norm(
                        adapted_latent
                    )
                ),

            "latent_L2_distance":
                float(
                    FINAL_REFERENCE[
                        "latent_drift"
                    ][
                        index
                    ]
                )
        }
    )


    reference_prediction_records.append(
        {

            "graph_id":
                metadata[
                    "graph_id"
                ],

            "drug_id":
                metadata[
                    "drug_id"
                ],

            "mean_absolute_target_score_drift":
                float(
                    FINAL_REFERENCE[
                        "mean_prediction_drift"
                    ][
                        index
                    ]
                ),

            "maximum_target_score_drift":
                float(
                    FINAL_REFERENCE[
                        "max_prediction_drift"
                    ][
                        index
                    ]
                ),

            "baseline_top_target_id":
                target_index_to_id[
                    baseline_top_index
                ],

            "baseline_top_target_name":
                target_index_to_name[
                    baseline_top_index
                ],

            "adapted_top_target_id":
                target_index_to_id[
                    adapted_top_index
                ],

            "adapted_top_target_name":
                target_index_to_name[
                    adapted_top_index
                ],

            "top_target_changed":
                bool(
                    baseline_top_index
                    !=
                    adapted_top_index
                ),

            "severe_prediction_drift":
                bool(
                    FINAL_REFERENCE[
                        "max_prediction_drift"
                    ][
                        index
                    ]
                    >
                    SEVERE_REFERENCE_DRIFT_THRESHOLD
                )
        }
    )


reference_latent_df = pd.DataFrame(
    reference_latent_records
)


reference_prediction_df = pd.DataFrame(
    reference_prediction_records
)


reference_latent_df.to_csv(
    PHASE17_REFERENCE_LATENT_DRIFT_FILE,
    index=False,
    encoding="utf-8"
)


reference_prediction_df.to_csv(
    PHASE17_REFERENCE_PREDICTION_DRIFT_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 49. SEVERE PREDICTION-DRIFT AUDIT
# ==============================================================================

severe_drift_df = reference_prediction_df[
    reference_prediction_df[
        "maximum_target_score_drift"
    ]
    >
    SEVERE_REFERENCE_DRIFT_THRESHOLD
].copy()


severe_drift_df = (
    severe_drift_df
    .sort_values(
        "maximum_target_score_drift",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


severe_drift_df.to_csv(
    PHASE17_REFERENCE_OUTLIERS_FILE,
    index=False,
    encoding="utf-8"
)


print(
    "\n"
    +
    "=" * 155
)

print(
    "REFERENCE PREDICTION-DRIFT OUTLIER AUDIT"
)

print(
    "=" * 155
)


print(
    f"[AUDIT] Severe drift threshold             : "
    f"{SEVERE_REFERENCE_DRIFT_THRESHOLD:.2f}"
)

print(
    f"[AUDIT] Reference molecules > threshold   : "
    f"{len(severe_drift_df)}/{EXPECTED_FISHER_REFERENCE_SIZE}"
)


if len(
    severe_drift_df
) > 0:

    print(
        severe_drift_df[
            [

                "drug_id",

                "maximum_target_score_drift",

                "baseline_top_target_id",

                "adapted_top_target_id",

                "top_target_changed"
            ]
        ]
        .head(
            20
        )
        .to_string(
            index=False
        )
    )

else:

    print(
        "[PASS] No severe baseline-reference prediction-drift outliers."
    )


# ==============================================================================
# 50. S1 PRE/POST ANALYSIS
# ==============================================================================

def evaluate_all_feedback_candidates(
    encoder_model,
    predictor_model
):

    ids = (
        S1_df[
            "candidate_id"
        ]
        .astype(str)
        .tolist()
    )


    encoder_model.eval()

    predictor_model.eval()


    with torch.no_grad():

        result = candidate_batch_scores(

            encoder_model,

            predictor_model,

            ids
        )


    return {

        "ids":
            ids,

        "mu":
            (
                result[
                    "mu"
                ]
                .detach()
                .cpu()
                .numpy()
            ),

        "probabilities":
            (
                result[
                    "probabilities"
                ]
                .detach()
                .cpu()
                .numpy()
            ),

        "biological":
            (
                result[
                    "biological_relevance"
                ]
                .detach()
                .cpu()
                .numpy()
            ),

        "utility":
            (
                result[
                    "utility"
                ]
                .detach()
                .cpu()
                .numpy()
            )
    }


BASELINE_S1 = evaluate_all_feedback_candidates(

    baseline_encoder,

    baseline_predictor
)


ADAPTED_S1 = evaluate_all_feedback_candidates(

    adapted_encoder,

    adapted_predictor
)


pre_post_records = []


for index, candidate_id in enumerate(
    BASELINE_S1[
        "ids"
    ]
):

    baseline_probs = (
        BASELINE_S1[
            "probabilities"
        ][
            index
        ]
    )


    adapted_probs = (
        ADAPTED_S1[
            "probabilities"
        ][
            index
        ]
    )


    baseline_top_index = int(
        np.argmax(
            baseline_probs
        )
    )


    adapted_top_index = int(
        np.argmax(
            adapted_probs
        )
    )


    pre_post_records.append(
        {

            "candidate_id":
                candidate_id,

            "phase16_recorded_biological_relevance":
                float(
                    candidate_lookup.loc[
                        candidate_id,
                        "biological_relevance_top5_mean"
                    ]
                ),

            "recomputed_baseline_biological_relevance":
                float(
                    BASELINE_S1[
                        "biological"
                    ][
                        index
                    ]
                ),

            "adapted_biological_relevance":
                float(
                    ADAPTED_S1[
                        "biological"
                    ][
                        index
                    ]
                ),

            "biological_relevance_change":
                float(
                    ADAPTED_S1[
                        "biological"
                    ][
                        index
                    ]
                    -
                    BASELINE_S1[
                        "biological"
                    ][
                        index
                    ]
                ),

            "phase16_recorded_utility":
                float(
                    candidate_lookup.loc[
                        candidate_id,
                        "utility_score"
                    ]
                ),

            "recomputed_baseline_utility":
                float(
                    BASELINE_S1[
                        "utility"
                    ][
                        index
                    ]
                ),

            "adapted_utility":
                float(
                    ADAPTED_S1[
                        "utility"
                    ][
                        index
                    ]
                ),

            "utility_change":
                float(
                    ADAPTED_S1[
                        "utility"
                    ][
                        index
                    ]
                    -
                    BASELINE_S1[
                        "utility"
                    ][
                        index
                    ]
                ),

            "baseline_top_target_id":
                target_index_to_id[
                    baseline_top_index
                ],

            "adapted_top_target_id":
                target_index_to_id[
                    adapted_top_index
                ],

            "top_target_changed":
                bool(
                    baseline_top_index
                    !=
                    adapted_top_index
                )
        }
    )


feedback_pre_post_df = pd.DataFrame(
    pre_post_records
)


feedback_pre_post_df.to_csv(
    PHASE17_FEEDBACK_PRE_POST_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 51. RANK CORRELATION
# ==============================================================================

def rank_array(
    values
):

    return (
        pd.Series(
            values
        )
        .rank(
            method="average",
            ascending=False
        )
        .to_numpy(
            dtype=float
        )
    )


def spearman_correlation(
    x,
    y
):

    rank_x = rank_array(
        x
    )


    rank_y = rank_array(
        y
    )


    if (
        np.std(
            rank_x
        ) == 0
        or
        np.std(
            rank_y
        ) == 0
    ):

        return float(
            "nan"
        )


    return float(
        np.corrcoef(
            rank_x,
            rank_y
        )[
            0,
            1
        ]
    )


def kendall_tau_simple(
    x,
    y
):

    concordant = 0

    discordant = 0


    n = len(
        x
    )


    for i in range(
        n
    ):

        for j in range(
            i + 1,
            n
        ):

            dx = (
                x[
                    i
                ]
                -
                x[
                    j
                ]
            )


            dy = (
                y[
                    i
                ]
                -
                y[
                    j
                ]
            )


            product = (
                dx
                *
                dy
            )


            if product > 0:

                concordant += 1

            elif product < 0:

                discordant += 1


    denominator = (
        concordant
        +
        discordant
    )


    if denominator == 0:

        return float(
            "nan"
        )


    return float(
        (
            concordant
            -
            discordant
        )
        /
        denominator
    )


# ==============================================================================
# 52. FINAL FEEDBACK PERFORMANCE
# ==============================================================================

FINAL_TRAIN_FEEDBACK = evaluate_feedback_subset(

    adapted_encoder,

    adapted_predictor,

    train_candidates_df,

    train_pairs_df
)


FINAL_VALIDATION_FEEDBACK = evaluate_feedback_subset(

    adapted_encoder,

    adapted_predictor,

    validation_candidates_df,

    validation_pairs_df
)


phase16_utility = S1_df[
    "utility_score"
].to_numpy(
    dtype=float
)


adapted_utility = ADAPTED_S1[
    "utility"
]


spearman_utility = spearman_correlation(

    phase16_utility,

    adapted_utility
)


kendall_utility = kendall_tau_simple(

    phase16_utility,

    adapted_utility
)


ranking_comparison_df = pd.DataFrame(
    [

        {

            "metric":
                "spearman_phase16_vs_adapted_utility",

            "value":
                spearman_utility
        },

        {

            "metric":
                "kendall_phase16_vs_adapted_utility",

            "value":
                kendall_utility
        },

        {

            "metric":
                "baseline_train_pair_accuracy",

            "value":
                BASELINE_TRAIN_FEEDBACK[
                    "accuracy"
                ]
        },

        {

            "metric":
                "adapted_train_pair_accuracy",

            "value":
                FINAL_TRAIN_FEEDBACK[
                    "accuracy"
                ]
        },

        {

            "metric":
                "baseline_validation_pair_accuracy",

            "value":
                BASELINE_VALIDATION_FEEDBACK[
                    "accuracy"
                ]
        },

        {

            "metric":
                "adapted_validation_pair_accuracy",

            "value":
                FINAL_VALIDATION_FEEDBACK[
                    "accuracy"
                ]
        }
    ]
)


ranking_comparison_df.to_csv(
    PHASE17_RANKING_COMPARISON_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 53. FINAL ACCEPTANCE CHECK
# ==============================================================================

FINAL_TRAIN_ACCURACY_DROP = (

    BASELINE_TRAIN_FEEDBACK[
        "accuracy"
    ]

    -

    FINAL_TRAIN_FEEDBACK[
        "accuracy"
    ]
)


FINAL_VALIDATION_ACCURACY_CHANGE = (

    FINAL_VALIDATION_FEEDBACK[
        "accuracy"
    ]

    -

    BASELINE_VALIDATION_FEEDBACK[
        "accuracy"
    ]
)


FINAL_VALIDATION_ACCEPTABLE = (

    FINAL_VALIDATION_FEEDBACK[
        "accuracy"
    ]

    >=

    BASELINE_VALIDATION_FEEDBACK[
        "accuracy"
    ]
)


FINAL_TRAIN_ACCEPTABLE = (

    FINAL_TRAIN_ACCURACY_DROP

    <=

    MAX_ALLOWED_TRAIN_ACCURACY_DROP
)


FINAL_SEVERE_DRIFT_ACCEPTABLE = (

    len(
        severe_drift_df
    )

    <=

    MAX_ALLOWED_SEVERE_REFERENCE_MOLECULES
)


FINAL_ADAPTATION_ACCEPTED = (

    FINAL_VALIDATION_ACCEPTABLE

    and

    FINAL_TRAIN_ACCEPTABLE

    and

    FINAL_SEVERE_DRIFT_ACCEPTABLE
)


if not FINAL_ADAPTATION_ACCEPTED:

    raise RuntimeError(
        "\nFINAL PHASE-17 ACCEPTANCE FAILURE.\n\n"
        f"Validation accuracy acceptable : {FINAL_VALIDATION_ACCEPTABLE}\n"
        f"Training accuracy acceptable   : {FINAL_TRAIN_ACCEPTABLE}\n"
        f"Severe drift acceptable        : {FINAL_SEVERE_DRIFT_ACCEPTABLE}\n\n"
        "Do NOT proceed to Phase 18."
    )


# ==============================================================================
# 54. RETENTION SUMMARY
# ==============================================================================

retention_summary_df = pd.DataFrame(
    [

        {

            "metric":
                "feedback_train_loss_before",

            "value":
                BASELINE_TRAIN_FEEDBACK[
                    "loss"
                ]
        },

        {

            "metric":
                "feedback_train_loss_after",

            "value":
                FINAL_TRAIN_FEEDBACK[
                    "loss"
                ]
        },

        {

            "metric":
                "feedback_validation_loss_before",

            "value":
                BASELINE_VALIDATION_FEEDBACK[
                    "loss"
                ]
        },

        {

            "metric":
                "feedback_validation_loss_after",

            "value":
                FINAL_VALIDATION_FEEDBACK[
                    "loss"
                ]
        },

        {

            "metric":
                "train_pair_accuracy_before",

            "value":
                BASELINE_TRAIN_FEEDBACK[
                    "accuracy"
                ]
        },

        {

            "metric":
                "train_pair_accuracy_after",

            "value":
                FINAL_TRAIN_FEEDBACK[
                    "accuracy"
                ]
        },

        {

            "metric":
                "validation_pair_accuracy_before",

            "value":
                BASELINE_VALIDATION_FEEDBACK[
                    "accuracy"
                ]
        },

        {

            "metric":
                "validation_pair_accuracy_after",

            "value":
                FINAL_VALIDATION_FEEDBACK[
                    "accuracy"
                ]
        },

        {

            "metric":
                "train_accuracy_drop_vs_K0",

            "value":
                FINAL_TRAIN_ACCURACY_DROP
        },

        {

            "metric":
                "validation_accuracy_change_vs_K0",

            "value":
                FINAL_VALIDATION_ACCURACY_CHANGE
        },

        {

            "metric":
                "mean_reference_latent_drift",

            "value":
                float(
                    FINAL_REFERENCE[
                        "latent_drift"
                    ].mean()
                )
        },

        {

            "metric":
                "median_reference_latent_drift",

            "value":
                float(
                    np.median(
                        FINAL_REFERENCE[
                            "latent_drift"
                        ]
                    )
                )
        },

        {

            "metric":
                "maximum_reference_latent_drift",

            "value":
                float(
                    FINAL_REFERENCE[
                        "latent_drift"
                    ].max()
                )
        },

        {

            "metric":
                "mean_reference_prediction_drift",

            "value":
                float(
                    FINAL_REFERENCE[
                        "mean_prediction_drift"
                    ].mean()
                )
        },

        {

            "metric":
                "maximum_reference_prediction_drift",

            "value":
                float(
                    FINAL_REFERENCE[
                        "max_prediction_drift"
                    ].max()
                )
        },

        {

            "metric":
                "severe_reference_prediction_drift_count",

            "value":
                int(
                    len(
                        severe_drift_df
                    )
                )
        },

        {

            "metric":
                "encoder_raw_fisher_weighted_drift",

            "value":
                float(
                    selected_ewc[
                        "encoder_raw"
                    ].item()
                )
        },

        {

            "metric":
                "predictor_raw_fisher_weighted_drift",

            "value":
                float(
                    selected_ewc[
                        "predictor_raw"
                    ].item()
                )
        },

        {

            "metric":
                "encoder_normalized_fisher_weighted_drift",

            "value":
                float(
                    selected_ewc[
                        "encoder_normalized"
                    ].item()
                )
        },

        {

            "metric":
                "predictor_normalized_fisher_weighted_drift",

            "value":
                float(
                    selected_ewc[
                        "predictor_normalized"
                    ].item()
                )
        }
    ]
)


retention_summary_df.to_csv(
    PHASE17_RETENTION_SUMMARY_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 55. K0 HASH INTEGRITY
# ==============================================================================

K0_HASH_AFTER = {

    "thetaE":
        sha256_file(
            PHASE12_ENCODER_FILE
        ),

    "thetaD":
        sha256_file(
            PHASE12_DECODER_FILE
        ),

    "thetaP":
        sha256_file(
            PHASE12_PREDICTOR_FILE
        ),

    "F_E":
        sha256_file(
            PHASE13_ENCODER_FISHER_FILE
        ),

    "F_P":
        sha256_file(
            PHASE13_PREDICTOR_FISHER_FILE
        ),

    "parameter_reference":
        sha256_file(
            PHASE13_PARAMETER_REFERENCE_FILE
        )
}


K0_UNCHANGED = (
    K0_HASH_BEFORE
    ==
    K0_HASH_AFTER
)


if not K0_UNCHANGED:

    raise RuntimeError(
        "Baseline K0 source files changed during Phase 17."
    )


# ==============================================================================
# 56. VERIFY ACTUAL PARAMETER ADAPTATION
# ==============================================================================

encoder_changed = any(

    not torch.equal(
        parameter.detach().cpu(),
        encoder_theta_star_cpu[
            name
        ]
    )

    for name, parameter
    in adapted_encoder.named_parameters()
)


predictor_changed = any(

    not torch.equal(
        parameter.detach().cpu(),
        predictor_theta_star_cpu[
            name
        ]
    )

    for name, parameter
    in adapted_predictor.named_parameters()
)


# ==============================================================================
# 57. CONFIGURATION
# ==============================================================================

phase17_config = {

    "phase":
        (
            "Phase 17 — Feedback Candidate Definition and "
            "EWC-Regularized Closed-Loop Adaptation"
        ),

    "created_at":
        datetime.now().isoformat(),

    "cycle":
        CYCLE,

    "random_seed":
        RANDOM_SEED,

    "S1_source":
        str(
            PHASE16_SELECTED_FILE
        ),

    "S1_count":
        len(
            S1_df
        ),

    "feedback_train_count":
        len(
            train_candidates_df
        ),

    "feedback_validation_count":
        len(
            validation_candidates_df
        ),

    "primary_preference":
        "Phase-16 exploitation utility rank",

    "minimum_rank_separation":
        MINIMUM_RANK_SEPARATION,

    "ranking_margin":
        RANKING_MARGIN,

    "adapted_utility":
        (
            "0.30*QED_fixed + "
            "0.20*SAUtility_fixed + "
            "0.50*B_adapt"
        ),

    "fixed_top5_targets":
        True,

    "optimizer":
        "AdamW",

    "learning_rate":
        LEARNING_RATE,

    "weight_decay":
        WEIGHT_DECAY,

    "maximum_epochs":
        MAX_ADAPTATION_EPOCHS,

    "epoch_checkpoint_selection":
        (
            "lexicographic: max validation pair accuracy, "
            "max validation margin accuracy, "
            "min validation feedback loss"
        ),

    "ewc_lambda_candidates":
        EWC_LAMBDA_CANDIDATES,

    "selected_lambda_ewc":
        SELECTED_LAMBDA,

    "selected_epoch":
        SELECTED_EPOCH,

    "K0_included_as_model_selection_benchmark":
        True,

    "maximum_allowed_train_accuracy_drop":
        MAX_ALLOWED_TRAIN_ACCURACY_DROP,

    "severe_reference_drift_threshold":
        SEVERE_REFERENCE_DRIFT_THRESHOLD,

    "maximum_allowed_severe_reference_molecules":
        MAX_ALLOWED_SEVERE_REFERENCE_MOLECULES,

    "primary_EWC_definition":
        (
            "raw diagonal empirical Fisher weighted "
            "squared parameter displacement"
        ),

    "normalized_EWC":
        "diagnostic only",

    "encoder_trainable":
        True,

    "predictor_trainable":
        True,

    "decoder_trainable":
        False,

    "fisher_recalculated":
        False,

    "Phase8_test_used_for_tuning":
        False,

    "final_adaptation_accepted":
        FINAL_ADAPTATION_ACCEPTED
}


with open(
    PHASE17_CONFIG_FILE,
    "w",
    encoding="utf-8"
) as handle:

    json.dump(
        phase17_config,
        handle,
        indent=4,
        ensure_ascii=False,
        default=str
    )


# ==============================================================================
# 58. COMPLETION CHECKS
# ==============================================================================

completion_checks = {

    "S1 loaded":
        len(
            S1_df
        ) > 0,

    "S1 candidate IDs unique":
        not S1_df[
            "candidate_id"
        ].duplicated().any(),

    "S1 structures unique":
        not S1_df[
            "canonical_smiles"
        ].duplicated().any(),

    "QED preserved":
        "qed"
        in S1_df.columns,

    "SA utility preserved":
        "sa_utility"
        in S1_df.columns,

    "Utility ranks preserved":
        "utility_rank"
        in S1_df.columns,

    "Acquisition ranks preserved":
        "acquisition_rank"
        in S1_df.columns,

    "Fixed Phase-16 top-5 targets preserved":
        len(
            frozen_top5_indices
        )
        ==
        len(
            S1_df
        ),

    "Exact Phase-7 node schema reused":
        NODE_FEATURE_NAMES
        ==
        EXPECTED_NODE_FEATURE_NAMES,

    "Exact Phase-7 edge schema reused":
        EDGE_FEATURE_NAMES
        ==
        EXPECTED_EDGE_FEATURE_NAMES,

    "Exact Phase-9 normalization reused":
        NORMALIZED_FEATURE_NAMES
        ==
        EXPECTED_NORMALIZED_FEATURES,

    "All S1 graphs precomputed":
        len(
            S1_GRAPHS
        )
        ==
        len(
            S1_df
        ),

    "Molecule-level feedback split created":
        (
            len(
                TRAIN_IDS
            )
            +
            len(
                VALIDATION_IDS
            )
            ==
            len(
                S1_df
            )
        ),

    "No train-validation molecule overlap":
        len(
            TRAIN_IDS
            &
            VALIDATION_IDS
        )
        ==
        0,

    "Training preference pairs generated":
        len(
            train_pairs_df
        ) > 0,

    "Validation preference pairs generated":
        len(
            validation_pairs_df
        ) > 0,

    "Minimum rank separation enforced":
        bool(
            (
                feedback_pairs_df[
                    "rank_difference"
                ]
                >=
                MINIMUM_RANK_SEPARATION
            ).all()
        ),

    "Baseline θE* loaded":
        True,

    "Baseline θP* loaded":
        True,

    "Encoder Fisher names match θE":
        (
            set(
                encoder_fisher.keys()
            )
            ==
            encoder_parameter_names
        ),

    "Predictor Fisher names match θP":
        (
            set(
                predictor_fisher.keys()
            )
            ==
            predictor_parameter_names
        ),

    "Phase-13 θE* anchor verified":
        True,

    "Phase-13 θP* anchor verified":
        True,

    "128 Fisher reference molecules loaded":
        len(
            REFERENCE_GRAPHS
        )
        ==
        EXPECTED_FISHER_REFERENCE_SIZE,

    "Deterministic μ used":
        True,

    "Frozen Phase-12 latent normalization reused":
        True,

    "Frozen 130-target vocabulary reused":
        len(
            target_vocab_df
        )
        ==
        TARGET_COUNT,

    "Utility-based feedback used":
        True,

    "Lexicographic epoch selection used":
        True,

    "K0 included as benchmark":
        True,

    "λ_EWC search completed":
        len(
            lambda_results
        )
        ==
        len(
            EWC_LAMBDA_CANDIDATES
        ),

    "Adapted checkpoint passed validation ranking benchmark":
        FINAL_VALIDATION_ACCEPTABLE,

    "Adapted checkpoint passed training ranking tolerance":
        FINAL_TRAIN_ACCEPTABLE,

    "Adapted checkpoint passed severe drift guardrail":
        FINAL_SEVERE_DRIFT_ACCEPTABLE,

    "Final adaptation accepted":
        FINAL_ADAPTATION_ACCEPTED,

    "Encoder adaptation occurred":
        encoder_changed,

    "Predictor adaptation occurred":
        predictor_changed,

    "Decoder baseline checkpoint unchanged":
        (
            K0_HASH_BEFORE[
                "thetaD"
            ]
            ==
            K0_HASH_AFTER[
                "thetaD"
            ]
        ),

    "Baseline K0 files unchanged":
        K0_UNCHANGED,

    "No Fisher recalculation":
        True,

    "Reference latent drift evaluated":
        len(
            reference_latent_df
        )
        ==
        EXPECTED_FISHER_REFERENCE_SIZE,

    "Reference prediction drift evaluated":
        len(
            reference_prediction_df
        )
        ==
        EXPECTED_FISHER_REFERENCE_SIZE,

    "Severe drift audit completed":
        PHASE17_REFERENCE_OUTLIERS_FILE.exists(),

    "Feedback pre-post scores evaluated":
        len(
            feedback_pre_post_df
        )
        ==
        len(
            S1_df
        ),

    "Adapted encoder saved separately":
        PHASE17_ENCODER_FILE.exists(),

    "Adapted predictor saved separately":
        PHASE17_PREDICTOR_FILE.exists(),

    "Phase-8 test not used for tuning":
        True
}


ALL_CHECKS_PASS = all(
    bool(
        value
    )
    for value
    in completion_checks.values()
)


# ==============================================================================
# 59. QC SUMMARY
# ==============================================================================

qc_summary = {

    "phase":
        17,

    "created_at":
        datetime.now().isoformat(),

    "S1_candidates":
        len(
            S1_df
        ),

    "feedback_train_candidates":
        len(
            train_candidates_df
        ),

    "feedback_validation_candidates":
        len(
            validation_candidates_df
        ),

    "training_pairs":
        len(
            train_pairs_df
        ),

    "validation_pairs":
        len(
            validation_pairs_df
        ),

    "selected_lambda_ewc":
        SELECTED_LAMBDA,

    "selected_epoch":
        SELECTED_EPOCH,

    "baseline_train_feedback_loss":
        BASELINE_TRAIN_FEEDBACK[
            "loss"
        ],

    "adapted_train_feedback_loss":
        FINAL_TRAIN_FEEDBACK[
            "loss"
        ],

    "baseline_validation_feedback_loss":
        BASELINE_VALIDATION_FEEDBACK[
            "loss"
        ],

    "adapted_validation_feedback_loss":
        FINAL_VALIDATION_FEEDBACK[
            "loss"
        ],

    "baseline_train_pair_accuracy":
        BASELINE_TRAIN_FEEDBACK[
            "accuracy"
        ],

    "adapted_train_pair_accuracy":
        FINAL_TRAIN_FEEDBACK[
            "accuracy"
        ],

    "baseline_validation_pair_accuracy":
        BASELINE_VALIDATION_FEEDBACK[
            "accuracy"
        ],

    "adapted_validation_pair_accuracy":
        FINAL_VALIDATION_FEEDBACK[
            "accuracy"
        ],

    "train_accuracy_drop_vs_K0":
        FINAL_TRAIN_ACCURACY_DROP,

    "validation_accuracy_change_vs_K0":
        FINAL_VALIDATION_ACCURACY_CHANGE,

    "spearman_phase16_vs_adapted_utility":
        spearman_utility,

    "kendall_phase16_vs_adapted_utility":
        kendall_utility,

    "mean_reference_latent_drift":
        float(
            FINAL_REFERENCE[
                "latent_drift"
            ].mean()
        ),

    "maximum_reference_latent_drift":
        float(
            FINAL_REFERENCE[
                "latent_drift"
            ].max()
        ),

    "mean_reference_prediction_drift":
        float(
            FINAL_REFERENCE[
                "mean_prediction_drift"
            ].mean()
        ),

    "maximum_reference_prediction_drift":
        float(
            FINAL_REFERENCE[
                "max_prediction_drift"
            ].max()
        ),

    "severe_reference_prediction_drift_count":
        int(
            len(
                severe_drift_df
            )
        ),

    "encoder_raw_fisher_drift":
        float(
            selected_ewc[
                "encoder_raw"
            ].item()
        ),

    "predictor_raw_fisher_drift":
        float(
            selected_ewc[
                "predictor_raw"
            ].item()
        ),

    "encoder_normalized_fisher_drift":
        float(
            selected_ewc[
                "encoder_normalized"
            ].item()
        ),

    "predictor_normalized_fisher_drift":
        float(
            selected_ewc[
                "predictor_normalized"
            ].item()
        ),

    "encoder_adapted":
        encoder_changed,

    "predictor_adapted":
        predictor_changed,

    "decoder_adapted":
        False,

    "fisher_recalculated":
        False,

    "K0_unchanged":
        K0_UNCHANGED,

    "final_adaptation_accepted":
        FINAL_ADAPTATION_ACCEPTED,

    "completion_checks":
        completion_checks
}


with open(
    PHASE17_QC_JSON_FILE,
    "w",
    encoding="utf-8"
) as handle:

    json.dump(
        qc_summary,
        handle,
        indent=4,
        ensure_ascii=False,
        default=str
    )


qc_rows = []


for key, value in qc_summary.items():

    if key == "completion_checks":

        for check_name, status in value.items():

            qc_rows.append(
                {

                    "metric":
                        f"completion.{check_name}",

                    "value":
                        status
                }
            )

    else:

        qc_rows.append(
            {

                "metric":
                    key,

                "value":
                    value
            }
        )


pd.DataFrame(
    qc_rows
).to_csv(
    PHASE17_QC_CSV_FILE,
    index=False,
    encoding="utf-8"
)


# ==============================================================================
# 60. FINAL MODEL-SELECTION DISPLAY
# ==============================================================================

print(
    "\n"
    +
    "=" * 155
)

print(
    "FINAL MODEL-SELECTION RESULTS"
)

print(
    "=" * 155
)


print(
    model_selection_df[
        [

            "candidate_type",

            "lambda_ewc",

            "best_epoch",

            "train_pairwise_accuracy",

            "validation_pairwise_accuracy",

            "validation_margin_accuracy",

            "validation_feedback_loss",

            "severe_reference_drift_count",

            "mean_reference_prediction_drift",

            "raw_total_fisher_drift",

            "adaptation_acceptable",

            "selected_primary"
        ]
    ].to_string(
        index=False
    )
)


# ==============================================================================
# 61. COMPLETION VALIDATION
# ==============================================================================

print(
    "\n"
    +
    "=" * 155
)

print(
    "PHASE 17 COMPLETION VALIDATION"
)

print(
    "=" * 155
)


for check_name, status in completion_checks.items():

    label = (
        "PASS"
        if bool(
            status
        )
        else
        "FAIL"
    )


    print(
        f"[{label:<4}] "
        f"{check_name}"
    )


# ==============================================================================
# 62. FINAL SUMMARY
# ==============================================================================

print(
    "\n"
    +
    "=" * 155
)

print(
    "PHASE 17 FINAL SUMMARY"
)

print(
    "=" * 155
)


print(
    f"S1 Candidates                           : "
    f"{len(S1_df)}"
)

print(
    f"Feedback-Train Candidates               : "
    f"{len(train_candidates_df)}"
)

print(
    f"Feedback-Validation Candidates          : "
    f"{len(validation_candidates_df)}"
)

print(
    f"Training Preference Pairs               : "
    f"{len(train_pairs_df)}"
)

print(
    f"Validation Preference Pairs             : "
    f"{len(validation_pairs_df)}"
)


print(
    "\nCheckpoint Selection:"
)

print(
    "  Primary Criterion                     : "
    "Validation Pairwise Accuracy"
)

print(
    "  Secondary Criterion                   : "
    "Validation Margin Accuracy"
)

print(
    "  Tertiary Criterion                    : "
    "Validation Feedback Loss"
)

print(
    "  K0 Included as Benchmark              : True"
)


print(
    "\nSelected Adaptation:"
)

print(
    f"  λ_EWC                                 : "
    f"{SELECTED_LAMBDA}"
)

print(
    f"  Epoch                                 : "
    f"{SELECTED_EPOCH}"
)

print(
    f"  Learning Rate                         : "
    f"{LEARNING_RATE}"
)

print(
    "  Encoder θE                            : Adapted"
)

print(
    "  Predictor θP                          : Adapted"
)

print(
    "  Decoder θD*                           : Frozen"
)


print(
    "\nFeedback Ranking:"
)

print(
    f"  Train Accuracy Before                 : "
    f"{BASELINE_TRAIN_FEEDBACK['accuracy']:.4f}"
)

print(
    f"  Train Accuracy After                  : "
    f"{FINAL_TRAIN_FEEDBACK['accuracy']:.4f}"
)

print(
    f"  Validation Accuracy Before            : "
    f"{BASELINE_VALIDATION_FEEDBACK['accuracy']:.4f}"
)

print(
    f"  Validation Accuracy After             : "
    f"{FINAL_VALIDATION_FEEDBACK['accuracy']:.4f}"
)

print(
    f"  Train Accuracy Drop                   : "
    f"{FINAL_TRAIN_ACCURACY_DROP:.4f}"
)

print(
    f"  Validation Accuracy Change            : "
    f"{FINAL_VALIDATION_ACCURACY_CHANGE:+.4f}"
)


print(
    "\nFeedback Loss:"
)

print(
    f"  Train Loss Before                     : "
    f"{BASELINE_TRAIN_FEEDBACK['loss']:.6f}"
)

print(
    f"  Train Loss After                      : "
    f"{FINAL_TRAIN_FEEDBACK['loss']:.6f}"
)

print(
    f"  Validation Loss Before                : "
    f"{BASELINE_VALIDATION_FEEDBACK['loss']:.6f}"
)

print(
    f"  Validation Loss After                 : "
    f"{FINAL_VALIDATION_FEEDBACK['loss']:.6f}"
)


print(
    "\nKnowledge Retention:"
)

print(
    f"  Mean Reference Latent Drift           : "
    f"{FINAL_REFERENCE['latent_drift'].mean():.8f}"
)

print(
    f"  Maximum Reference Latent Drift        : "
    f"{FINAL_REFERENCE['latent_drift'].max():.8f}"
)

print(
    f"  Mean Reference Prediction Drift       : "
    f"{FINAL_REFERENCE['mean_prediction_drift'].mean():.8f}"
)

print(
    f"  Maximum Reference Prediction Drift    : "
    f"{FINAL_REFERENCE['max_prediction_drift'].max():.8f}"
)

print(
    f"  Severe Drift Molecules (>0.50)        : "
    f"{len(severe_drift_df)}"
)


print(
    "\nEWC Drift:"
)

print(
    f"  Encoder Raw Fisher Drift              : "
    f"{selected_ewc['encoder_raw'].item():.10f}"
)

print(
    f"  Predictor Raw Fisher Drift            : "
    f"{selected_ewc['predictor_raw'].item():.10f}"
)

print(
    f"  Encoder Normalized Fisher Drift       : "
    f"{selected_ewc['encoder_normalized'].item():.12e}"
)

print(
    f"  Predictor Normalized Fisher Drift     : "
    f"{selected_ewc['predictor_normalized'].item():.12e}"
)


print(
    "\nRanking Comparison:"
)

print(
    f"  Spearman Phase16 vs Adapted Utility   : "
    f"{spearman_utility:.6f}"
)

print(
    f"  Kendall Phase16 vs Adapted Utility    : "
    f"{kendall_utility:.6f}"
)


print(
    "\nBaseline Integrity:"
)

print(
    f"  K0 Source Files Unchanged             : "
    f"{K0_UNCHANGED}"
)

print(
    "  Fisher Recalculated                   : False"
)

print(
    "  Phase-8 Test Used for Tuning          : False"
)


print(
    "\nAdaptation Acceptance:"
)

print(
    f"  Validation Ranking Guardrail          : "
    f"{FINAL_VALIDATION_ACCEPTABLE}"
)

print(
    f"  Training Ranking Guardrail            : "
    f"{FINAL_TRAIN_ACCEPTABLE}"
)

print(
    f"  Severe Drift Guardrail                : "
    f"{FINAL_SEVERE_DRIFT_ACCEPTABLE}"
)

print(
    f"  Final Adaptation Accepted             : "
    f"{FINAL_ADAPTATION_ACCEPTED}"
)


print(
    "\nUpdated State:"
)

print(
    "  K1 = {θE(1), θD*, θP(1), F_E, F_P, Z0, S1}"
)


print(
    "\nSaved Files:"
)


for path in [

    PHASE17_CONFIG_FILE,

    PHASE17_CANDIDATE_MANIFEST_FILE,

    PHASE17_GRAPH_MANIFEST_FILE,

    PHASE17_TRAIN_CANDIDATES_FILE,

    PHASE17_VALIDATION_CANDIDATES_FILE,

    PHASE17_PAIRS_FILE,

    PHASE17_LAMBDA_SEARCH_FILE,

    PHASE17_MODEL_SELECTION_FILE,

    PHASE17_HISTORY_FILE,

    PHASE17_ENCODER_FILE,

    PHASE17_PREDICTOR_FILE,

    PHASE17_PARAMETER_DRIFT_FILE,

    PHASE17_FISHER_DRIFT_FILE,

    PHASE17_REFERENCE_LATENT_DRIFT_FILE,

    PHASE17_REFERENCE_PREDICTION_DRIFT_FILE,

    PHASE17_REFERENCE_OUTLIERS_FILE,

    PHASE17_FEEDBACK_PRE_POST_FILE,

    PHASE17_RANKING_COMPARISON_FILE,

    PHASE17_RETENTION_SUMMARY_FILE,

    PHASE17_QC_CSV_FILE,

    PHASE17_QC_JSON_FILE
]:

    print(
        f"  {path.name}"
    )


print(
    "-" * 155
)


if ALL_CHECKS_PASS:

    print(
        "[STATUS] PHASE 17 COMPLETED SUCCESSFULLY."
    )


    print(
        "[PASS] The selected adapted checkpoint preserved or improved "
        "the baseline validation preference ordering."
    )


    print(
        "[PASS] Epoch selection prioritized ranking fidelity before "
        "continuous softplus loss."
    )


    print(
        "[PASS] K0 was explicitly included as the no-adaptation "
        "model-selection benchmark."
    )


    print(
        "[PASS] The adapted checkpoint passed the predefined "
        "training-ranking tolerance."
    )


    print(
        "[PASS] No severe Fisher-reference target-score drift "
        "exceeded the accepted guardrail."
    )


    print(
        "[PASS] Encoder θE and predictor θP were adapted under "
        "diagonal empirical-Fisher EWC."
    )


    print(
        "[PASS] Decoder θD* remained frozen."
    )


    print(
        "[PASS] K0 baseline source files remained unchanged."
    )


    print(
        "[PASS] Updated Cycle-1 state K1 has been accepted."
    )


    print(
        "[IMPORTANT] This is surrogate-guided computational "
        "continual adaptation."
    )


    print(
        "[IMPORTANT] It is NOT learning from new experimental "
        "biological measurements."
    )


    print(
        "[READY] Send the complete Phase-17 output for final "
        "verification before Phase 18."
    )


else:

    print(
        "[STATUS] PHASE 17 REQUIRES REVIEW."
    )


    print(
        "[ACTION] Do NOT proceed to Phase 18."
    )


print(
    "=" * 155
)

PHASE 17 — FEEDBACK CANDIDATE DEFINITION AND EWC-REGULARIZED CLOSED-LOOP ADAPTATION

[DEVICE] cpu
[CONFIG] Random Seed                     : 42
[CONFIG] Learning Rate                   : 0.0001
[CONFIG] Maximum Epochs                  : 20
[CONFIG] Minimum Rank Separation         : 5
[CONFIG] Ranking Margin                  : 0.05
[CONFIG] EWC λ Candidates                : [10.0, 100.0, 1000.0]
[CONFIG] Severe Prediction Drift Limit   : 0.5

[PASS] Exact Phase-7 graph schema verified.
[PASS] Exact Phase-9 normalization loaded.
[PASS] Baseline θE* and θP* loaded.
[PASS] Phase-13 Fisher and θ* anchors loaded and verified.

[PASS] S1 loaded: 50 candidates.

PRECOMPUTING S1 MOLECULAR GRAPHS
[GRAPH] 01/50 CYCLE01_CAND000064 PASS
[GRAPH] 02/50 CYCLE01_CAND000720 PASS
[GRAPH] 03/50 CYCLE01_CAND000707 PASS
[GRAPH] 04/50 CYCLE01_CAND000212 PASS
[GRAPH] 05/50 CYCLE01_CAND000125 PASS
[GRAPH] 06/50 CYCLE01_CAND000753 PASS
[GRAPH] 07/50 CYCLE01_CAND000561 PASS
[GRAPH] 08/50 CYCLE01_CAND000197 PASS


[17:01:12] UFFTYPER: Unrecognized atom type: S_5+4 (23)


[GRAPH] 49/50 CYCLE01_CAND000436 PASS
[GRAPH] 50/50 CYCLE01_CAND000710 PASS

[PASS] All S1 molecular graphs precomputed successfully.

[PASS] Feedback-train candidates      : 40
[PASS] Feedback-validation candidates : 10
[PASS] Candidate split performed BEFORE pair construction.

[PASS] Training preference pairs   : 684
[PASS] Validation preference pairs : 40

LOADING FIXED 128-MOLECULE FISHER REFERENCE
[PASS] Fixed Fisher-reference graphs loaded: 128
[PASS] Baseline reference latent/prediction state cached.

Baseline K0 Feedback Performance:
  Train Loss                : 0.686868
  Train Pair Accuracy       : 0.8962
  Train Margin Accuracy     : 0.3626
  Validation Loss           : 0.675811
  Validation Pair Accuracy  : 0.9750
  Validation Margin Accuracy: 0.6500

EWC λ SEARCH — CORRECTED CHECKPOINT SELECTION

-----------------------------------------------------------------------------------------------------------------------------------------------------------
TRAINING λ_EWC = 10.0

# Phase 18 — Post-Adaptation Uncertainty, Knowledge-Retention Validation & Cycle-2 Readiness (Module 108)

**Objective:** Execute comparative model-state validation between baseline $\mathcal{K}_0$ and adapted $\mathcal{K}_1$ across the 50 feedback leads ($S_1$) and the 128 baseline reference molecules ($\mathcal{D}_{\text{fisher}}$) using paired 30-pass MC-Dropout to evaluate uncertainty shifts, ensure knowledge retention, and certify generation readiness for Cycle 2.

* **Experimental Controls & Non-Parametric Boundaries:**
  * **Zero Optimization Policy:** Evaluated entirely under `torch.no_grad()` with no backward passes, optimizers, or parameter updates ($\Delta \theta = \mathbf{0}$).
  * **Paired Stochastic Inferences:** Pass $t \in [1, 30]$ executed using identical pass-wise random seeds across both models to isolate epistemic shifts from sampling variance.
  * **Strict Stochastic Isolation:** Dropout held active; LayerNorm and EGNN encoders held strictly deterministic.

* **Audit Dimensions:**
  * **Feedback Alignment ($S_1$):** Tracks ranking stability (Spearman $\rho$, Kendall $\tau$), utility shifts ($\Delta \mathcal{U}$), and potential uncertainty collapse.
  * **Baseline Retention ($N = 128$):** Audits mean prediction drift, severe prediction outliers ($> 0.50$), and top-target switching.
  * **Latent Compatibility ($Z_1$ vs. $Z_0$):** Measures standardized RMS latent displacement relative to the frozen decoder ($\theta_D^*$).
  * **Cycle-2 Readiness Rule:**
    $$Ready_{\text{Cycle2}} = R_{\text{feedback}} \land R_{\text{retention}} \land R_{\text{uncertainty}} \land R_{\text{latent\_compatibility}} \land R_{\text{integrity}}$$

* **Exported Deliverables:**
  * Multi-Target Prediction & Variance Matrices: `phase18_{S1,reference}_{K0,K1}_target_{mean,uncertainty}_matrix.csv`.
  * Comparison & Diagnostics: `phase18_S1_uncertainty_comparison.csv`, `phase18_reference_uncertainty_comparison.csv`, `phase18_uncertainty_outliers.csv`.
  * Readiness & Summary Manifests: `phase18_ranking_stability.csv`, `phase18_latent_compatibility.csv`, `phase18_cycle2_generation_readiness.json`, `phase18_validation_qc_summary.json`.

In [21]:
# ==============================================================================
# PHASE 18 — POST-ADAPTATION UNCERTAINTY,
# KNOWLEDGE-RETENTION VALIDATION,
# AND CYCLE-2 GENERATION READINESS
#
# FINAL ONE-CELL IMPLEMENTATION
#
# PURPOSE
# -------
#
# Compare:
#
#   K0 = {θE*, θD*, θP*}
#
# versus
#
#   K1 = {θE(1), θD*, θP(1)}
#
# using:
#
#   • deterministic encoder μ
#   • frozen Phase-12 latent normalization
#   • 30-pass MC dropout
#   • paired stochastic seed schedule
#   • S1 = 50 feedback candidates
#   • fixed 128-molecule Fisher reference
#
# Phase 18 performs NO TRAINING.
#
#   optimizer               = NONE
#   backward passes         = NONE
#   parameter updates       = NONE
#   EWC optimization        = NONE
#   Fisher recalculation    = NONE
#
# FINAL DECISION
# --------------
#
# Ready_Cycle2 =
#
#     R_feedback
#     AND
#     R_retention
#     AND
#     R_uncertainty
#     AND
#     R_latent_compatibility
#     AND
#     R_integrity
#
# ==============================================================================


# ==============================================================================
# 1. IMPORTS
# ==============================================================================

from pathlib import Path
from datetime import datetime

import hashlib
import json
import math
import random
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from rdkit import Chem
from rdkit import RDLogger
from rdkit.Chem import AllChem


warnings.filterwarnings(
    "ignore"
)

RDLogger.DisableLog(
    "rdApp.warning"
)

RDLogger.DisableLog(
    "rdApp.info"
)


# ==============================================================================
# 2. REPRODUCIBILITY
# ==============================================================================

RANDOM_SEED = 42

random.seed(
    RANDOM_SEED
)

np.random.seed(
    RANDOM_SEED
)

torch.manual_seed(
    RANDOM_SEED
)


if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        RANDOM_SEED
    )


DEVICE = torch.device(
    "cpu"
)


# ==============================================================================
# 3. FIXED CONFIGURATION
# ==============================================================================

CYCLE = 1

MC_DROPOUT_PASSES = 30

MC_SEED_BASE = 180000


NODE_FEATURE_DIM = 21

EDGE_FEATURE_DIM = 14

HIDDEN_DIM = 64

LATENT_DIM = 32

NUM_EGNN_LAYERS = 2

COORDINATE_UPDATE_SCALE = 0.10

TARGET_COUNT = 130


EXPECTED_S1_SIZE = 50

EXPECTED_REFERENCE_SIZE = 128


MAX_OPTIMIZATION_ITERS = 500


# ------------------------------------------------------------------------------
# Phase-16 / Phase-17 utility definition
# ------------------------------------------------------------------------------

WEIGHT_QED = 0.30

WEIGHT_SA = 0.20

WEIGHT_BIO = 0.50


# ==============================================================================
# 4. PREDEFINED PHASE-18 AUDIT THRESHOLDS
#
# IMPORTANT:
# These thresholds are declared BEFORE final result inspection.
# ==============================================================================

# ------------------------------------------------------------------------------
# Prediction retention
# ------------------------------------------------------------------------------

SEVERE_PREDICTION_DRIFT_THRESHOLD = 0.50

MAX_ALLOWED_SEVERE_REFERENCE_DRIFT_COUNT = 0

MAX_ALLOWED_MEAN_REFERENCE_PREDICTION_DRIFT = 0.05

MAX_ALLOWED_REFERENCE_TOP_TARGET_CHANGE_FRACTION = 0.25


# ------------------------------------------------------------------------------
# Uncertainty expansion / reduction diagnostics
# ------------------------------------------------------------------------------

UNCERTAINTY_RATIO_EPSILON = 1e-4

LARGE_UNCERTAINTY_INCREASE_RATIO = 2.0

LARGE_UNCERTAINTY_DECREASE_RATIO = 0.50


# Large uncertainty decrease is reported but is NOT an automatic failure.
# Large uncertainty increase is considered problematic only when widespread.

MAX_ALLOWED_LARGE_UNCERTAINTY_INCREASE_FRACTION = 0.20


# ------------------------------------------------------------------------------
# Potential uncertainty-collapse flag
#
# large prediction movement + unusually low K1 uncertainty
# ------------------------------------------------------------------------------

LARGE_PREDICTION_SHIFT_THRESHOLD = 0.25

LOW_UNCERTAINTY_BASELINE_QUANTILE = 0.10

MAX_ALLOWED_UNCERTAINTY_COLLAPSE_FRACTION = 0.10


# ------------------------------------------------------------------------------
# Latent compatibility
#
# Standardize K1-K0 latent displacement by the baseline reference latent spread.
#
# A molecule with standardized RMS latent drift > 1.0 has moved, on average,
# roughly one baseline population SD per latent dimension.
# ------------------------------------------------------------------------------

LATENT_STANDARDIZED_RMS_SEVERE_THRESHOLD = 1.0

MAX_ALLOWED_MEAN_STANDARDIZED_LATENT_DRIFT = 0.50

MAX_ALLOWED_SEVERE_LATENT_DRIFT_FRACTION = 0.10


# ==============================================================================
# 5. PROJECT PATHS
# ==============================================================================

if "PROCESSED_DIR" not in globals():

    raise RuntimeError(
        "PROCESSED_DIR is not defined. "
        "Run the project initialization cell first."
    )


PROCESSED_DIR = Path(
    PROCESSED_DIR
)


if "GRAPH_DIR" in globals():

    GRAPH_DIR = Path(
        GRAPH_DIR
    )

else:

    GRAPH_DIR = (
        PROCESSED_DIR.parent
        /
        "graphs"
    )


# ==============================================================================
# 6. INPUT FILES
# ==============================================================================

PHASE7_FEATURE_SCHEMA_FILE = (
    PROCESSED_DIR
    /
    "phase7_graph_feature_schema.json"
)


PHASE9_NORMALIZATION_FILE = (
    PROCESSED_DIR
    /
    "phase9_feature_normalization.json"
)


PHASE12_ENCODER_FILE = (
    PROCESSED_DIR
    /
    "phase12_thetaE_baseline_encoder.pt"
)


PHASE12_DECODER_FILE = (
    PROCESSED_DIR
    /
    "phase12_thetaD_baseline_decoder.pt"
)


PHASE12_PREDICTOR_FILE = (
    PROCESSED_DIR
    /
    "phase12_best_baseline_predictor.pt"
)


PHASE12_LATENT_NORMALIZATION_FILE = (
    PROCESSED_DIR
    /
    "phase12_predictor_latent_normalization.json"
)


PHASE12_TARGET_VOCAB_FILE = (
    PROCESSED_DIR
    /
    "phase12_predictor_target_vocabulary.csv"
)


PHASE8_FISHER_REFERENCE_FILE = (
    PROCESSED_DIR
    /
    "phase8_fisher_reference.csv"
)


PHASE13_ENCODER_FISHER_FILE = (
    PROCESSED_DIR
    /
    "phase13_encoder_fisher.pt"
)


PHASE13_PREDICTOR_FISHER_FILE = (
    PROCESSED_DIR
    /
    "phase13_predictor_fisher.pt"
)


PHASE17_CONFIG_FILE = (
    PROCESSED_DIR
    /
    "phase17_adaptation_config.json"
)


PHASE17_QC_FILE = (
    PROCESSED_DIR
    /
    "phase17_adaptation_qc_summary.json"
)


PHASE17_ENCODER_FILE = (
    PROCESSED_DIR
    /
    "phase17_cycle1_adapted_encoder.pt"
)


PHASE17_PREDICTOR_FILE = (
    PROCESSED_DIR
    /
    "phase17_cycle1_adapted_predictor.pt"
)


PHASE17_S1_FILE = (
    PROCESSED_DIR
    /
    "phase17_feedback_candidate_manifest.csv"
)


required_files = [

    PHASE7_FEATURE_SCHEMA_FILE,

    PHASE9_NORMALIZATION_FILE,

    PHASE12_ENCODER_FILE,

    PHASE12_DECODER_FILE,

    PHASE12_PREDICTOR_FILE,

    PHASE12_LATENT_NORMALIZATION_FILE,

    PHASE12_TARGET_VOCAB_FILE,

    PHASE8_FISHER_REFERENCE_FILE,

    PHASE13_ENCODER_FISHER_FILE,

    PHASE13_PREDICTOR_FISHER_FILE,

    PHASE17_CONFIG_FILE,

    PHASE17_QC_FILE,

    PHASE17_ENCODER_FILE,

    PHASE17_PREDICTOR_FILE,

    PHASE17_S1_FILE
]


missing_files = [

    str(
        path
    )

    for path in required_files

    if not path.exists()
]


if missing_files:

    raise FileNotFoundError(
        "Required Phase-18 input files are missing:\n"
        +
        "\n".join(
            missing_files
        )
    )


# ==============================================================================
# 7. OUTPUT FILES
# ==============================================================================

PHASE18_CONFIG_FILE = (
    PROCESSED_DIR
    /
    "phase18_validation_config.json"
)


PHASE18_S1_K0_MEAN_FILE = (
    PROCESSED_DIR
    /
    "phase18_S1_K0_target_mean_matrix.csv"
)


PHASE18_S1_K0_UNCERTAINTY_FILE = (
    PROCESSED_DIR
    /
    "phase18_S1_K0_target_uncertainty_matrix.csv"
)


PHASE18_S1_K1_MEAN_FILE = (
    PROCESSED_DIR
    /
    "phase18_S1_K1_target_mean_matrix.csv"
)


PHASE18_S1_K1_UNCERTAINTY_FILE = (
    PROCESSED_DIR
    /
    "phase18_S1_K1_target_uncertainty_matrix.csv"
)


PHASE18_REFERENCE_K0_MEAN_FILE = (
    PROCESSED_DIR
    /
    "phase18_reference_K0_target_mean_matrix.csv"
)


PHASE18_REFERENCE_K0_UNCERTAINTY_FILE = (
    PROCESSED_DIR
    /
    "phase18_reference_K0_target_uncertainty_matrix.csv"
)


PHASE18_REFERENCE_K1_MEAN_FILE = (
    PROCESSED_DIR
    /
    "phase18_reference_K1_target_mean_matrix.csv"
)


PHASE18_REFERENCE_K1_UNCERTAINTY_FILE = (
    PROCESSED_DIR
    /
    "phase18_reference_K1_target_uncertainty_matrix.csv"
)


PHASE18_S1_COMPARISON_FILE = (
    PROCESSED_DIR
    /
    "phase18_S1_uncertainty_comparison.csv"
)


PHASE18_REFERENCE_COMPARISON_FILE = (
    PROCESSED_DIR
    /
    "phase18_reference_uncertainty_comparison.csv"
)


PHASE18_UNCERTAINTY_OUTLIERS_FILE = (
    PROCESSED_DIR
    /
    "phase18_uncertainty_outliers.csv"
)


PHASE18_DISTRIBUTION_SUMMARY_FILE = (
    PROCESSED_DIR
    /
    "phase18_uncertainty_distribution_summary.csv"
)


PHASE18_RANKING_STABILITY_FILE = (
    PROCESSED_DIR
    /
    "phase18_ranking_stability.csv"
)


PHASE18_LATENT_COMPATIBILITY_FILE = (
    PROCESSED_DIR
    /
    "phase18_latent_compatibility.csv"
)


PHASE18_READINESS_FILE = (
    PROCESSED_DIR
    /
    "phase18_cycle2_generation_readiness.json"
)


PHASE18_QC_CSV_FILE = (
    PROCESSED_DIR
    /
    "phase18_validation_qc_summary.csv"
)


PHASE18_QC_JSON_FILE = (
    PROCESSED_DIR
    /
    "phase18_validation_qc_summary.json"
)


# ==============================================================================
# 8. GENERAL HELPERS
# ==============================================================================

def sha256_file(
    path
):

    digest = hashlib.sha256()


    with open(
        path,
        "rb"
    ) as handle:

        while True:

            block = handle.read(
                1024
                *
                1024
            )


            if not block:

                break


            digest.update(
                block
            )


    return digest.hexdigest()


def extract_state_dict(
    checkpoint
):

    if isinstance(
        checkpoint,
        nn.Module
    ):

        return checkpoint.state_dict()


    if not isinstance(
        checkpoint,
        dict
    ):

        raise RuntimeError(
            "Unsupported checkpoint format."
        )


    for key in [

        "state_dict",

        "model_state_dict",

        "encoder_state_dict",

        "predictor_state_dict"
    ]:

        if key in checkpoint:

            value = checkpoint[
                key
            ]


            if isinstance(
                value,
                nn.Module
            ):

                return value.state_dict()


            if isinstance(
                value,
                dict
            ):

                return value


    tensor_items = {

        key:
            value

        for key, value
        in checkpoint.items()

        if torch.is_tensor(
            value
        )
    }


    if tensor_items:

        return tensor_items


    raise RuntimeError(
        "Could not recover state_dict."
    )


def strip_prefix_if_present(
    state_dict,
    prefix
):

    keys = list(
        state_dict.keys()
    )


    if (
        keys
        and
        all(
            key.startswith(
                prefix
            )
            for key in keys
        )
    ):

        return {

            key[
                len(
                    prefix
                ):
            ]:
                value

            for key, value
            in state_dict.items()
        }


    return state_dict


def normalize_state_dict_keys(
    state_dict
):

    result = dict(
        state_dict
    )


    for prefix in [

        "module.",

        "encoder.",

        "predictor.",

        "model."
    ]:

        result = strip_prefix_if_present(
            result,
            prefix
        )


    return result


# ==============================================================================
# 9. HASH ALL MODEL / FISHER FILES BEFORE EVALUATION
# ==============================================================================

HASHES_BEFORE = {

    "K0_encoder":
        sha256_file(
            PHASE12_ENCODER_FILE
        ),

    "K0_decoder":
        sha256_file(
            PHASE12_DECODER_FILE
        ),

    "K0_predictor":
        sha256_file(
            PHASE12_PREDICTOR_FILE
        ),

    "K1_encoder":
        sha256_file(
            PHASE17_ENCODER_FILE
        ),

    "K1_predictor":
        sha256_file(
            PHASE17_PREDICTOR_FILE
        ),

    "F_E":
        sha256_file(
            PHASE13_ENCODER_FISHER_FILE
        ),

    "F_P":
        sha256_file(
            PHASE13_PREDICTOR_FISHER_FILE
        )
}


print(
    "=" * 155
)

print(
    "PHASE 18 — POST-ADAPTATION UNCERTAINTY, "
    "KNOWLEDGE-RETENTION VALIDATION, "
    "AND CYCLE-2 GENERATION READINESS"
)

print(
    "=" * 155
)

print(
    f"\n[DEVICE] {DEVICE}"
)

print(
    f"[CONFIG] MC Dropout Passes                 : "
    f"{MC_DROPOUT_PASSES}"
)

print(
    f"[CONFIG] Random Seed                       : "
    f"{RANDOM_SEED}"
)

print(
    f"[CONFIG] Severe Prediction Drift Threshold : "
    f"{SEVERE_PREDICTION_DRIFT_THRESHOLD}"
)


# ==============================================================================
# 10. VERIFY PHASE-17 ACCEPTANCE
# ==============================================================================

with open(
    PHASE17_CONFIG_FILE,
    "r",
    encoding="utf-8"
) as handle:

    phase17_config = json.load(
        handle
    )


with open(
    PHASE17_QC_FILE,
    "r",
    encoding="utf-8"
) as handle:

    phase17_qc = json.load(
        handle
    )


phase17_accepted_config = bool(
    phase17_config.get(
        "final_adaptation_accepted",
        False
    )
)


phase17_accepted_qc = bool(
    phase17_qc.get(
        "final_adaptation_accepted",
        False
    )
)


if not (
    phase17_accepted_config
    and
    phase17_accepted_qc
):

    raise RuntimeError(
        "Phase 17 is not recorded as an accepted adaptation. "
        "Do not evaluate K1 for Cycle-2 readiness."
    )


SELECTED_LAMBDA = float(
    phase17_config[
        "selected_lambda_ewc"
    ]
)


SELECTED_EPOCH = int(
    phase17_config[
        "selected_epoch"
    ]
)


print(
    "\n[PASS] Phase-17 accepted adaptation verified."
)

print(
    f"[PASS] Accepted λ_EWC  : {SELECTED_LAMBDA}"
)

print(
    f"[PASS] Accepted epoch  : {SELECTED_EPOCH}"
)


# ==============================================================================
# 11. VERIFY K1 CHECKPOINT METADATA
# ==============================================================================

K1_encoder_checkpoint = torch.load(
    PHASE17_ENCODER_FILE,
    map_location="cpu",
    weights_only=False
)


K1_predictor_checkpoint = torch.load(
    PHASE17_PREDICTOR_FILE,
    map_location="cpu",
    weights_only=False
)


if "selected_lambda_ewc" in K1_encoder_checkpoint:

    if not np.isclose(
        float(
            K1_encoder_checkpoint[
                "selected_lambda_ewc"
            ]
        ),
        SELECTED_LAMBDA
    ):

        raise RuntimeError(
            "K1 encoder λ_EWC metadata does not match Phase-17 config."
        )


if "selected_epoch" in K1_encoder_checkpoint:

    if int(
        K1_encoder_checkpoint[
            "selected_epoch"
        ]
    ) != SELECTED_EPOCH:

        raise RuntimeError(
            "K1 encoder epoch metadata does not match Phase-17 config."
        )


if "selected_lambda_ewc" in K1_predictor_checkpoint:

    if not np.isclose(
        float(
            K1_predictor_checkpoint[
                "selected_lambda_ewc"
            ]
        ),
        SELECTED_LAMBDA
    ):

        raise RuntimeError(
            "K1 predictor λ_EWC metadata does not match Phase-17 config."
        )


if "selected_epoch" in K1_predictor_checkpoint:

    if int(
        K1_predictor_checkpoint[
            "selected_epoch"
        ]
    ) != SELECTED_EPOCH:

        raise RuntimeError(
            "K1 predictor epoch metadata does not match Phase-17 config."
        )


print(
    "[PASS] K1 checkpoint metadata matches accepted Phase-17 state."
)


# ==============================================================================
# 12. LOAD PHASE-7 FEATURE SCHEMA
# ==============================================================================

with open(
    PHASE7_FEATURE_SCHEMA_FILE,
    "r",
    encoding="utf-8"
) as handle:

    phase7_schema = json.load(
        handle
    )


NODE_FEATURE_NAMES = list(
    phase7_schema[
        "node_feature_names"
    ]
)


EDGE_FEATURE_NAMES = list(
    phase7_schema[
        "edge_feature_names"
    ]
)


# ==============================================================================
# 13. EXACT PHASE-7 FEATURE VOCABULARIES
# ==============================================================================

HYBRIDIZATION_TYPES = [

    "UNSPECIFIED",

    "S",

    "SP",

    "SP2",

    "SP3",

    "SP3D",

    "SP3D2",

    "OTHER"
]


CHIRALITY_TYPES = [

    "CHI_UNSPECIFIED",

    "CHI_TETRAHEDRAL_CW",

    "CHI_TETRAHEDRAL_CCW",

    "CHI_OTHER"
]


BOND_TYPES = [

    "SINGLE",

    "DOUBLE",

    "TRIPLE",

    "AROMATIC",

    "OTHER"
]


BOND_STEREO_TYPES = [

    "STEREONONE",

    "STEREOANY",

    "STEREOZ",

    "STEREOE",

    "STEREOCIS",

    "STEREOTRANS",

    "OTHER"
]


EXPECTED_NODE_FEATURE_NAMES = [

    "atomic_number",

    "degree",

    "formal_charge",

    "aromatic",

    "in_ring",

    "hydrogen_neighbor_count",

    "atomic_mass",

    "implicit_valence",

    "explicit_valence"
]


EXPECTED_NODE_FEATURE_NAMES += [

    f"hybridization_{value}"

    for value
    in HYBRIDIZATION_TYPES
]


EXPECTED_NODE_FEATURE_NAMES += [

    f"chirality_{value}"

    for value
    in CHIRALITY_TYPES
]


EXPECTED_EDGE_FEATURE_NAMES = [

    f"bond_type_{value}"

    for value
    in BOND_TYPES
]


EXPECTED_EDGE_FEATURE_NAMES += [

    "is_conjugated",

    "is_in_ring"
]


EXPECTED_EDGE_FEATURE_NAMES += [

    f"stereo_{value}"

    for value
    in BOND_STEREO_TYPES
]


if NODE_FEATURE_NAMES != EXPECTED_NODE_FEATURE_NAMES:

    raise RuntimeError(
        "Phase-7 node schema mismatch."
    )


if EDGE_FEATURE_NAMES != EXPECTED_EDGE_FEATURE_NAMES:

    raise RuntimeError(
        "Phase-7 edge schema mismatch."
    )


print(
    "[PASS] Exact Phase-7 graph schema verified."
)


# ==============================================================================
# 14. PHASE-7 FEATURIZATION
# ==============================================================================

def one_hot(
    value,
    categories
):

    output = [
        0.0
        for _
        in categories
    ]


    if value in categories:

        index = categories.index(
            value
        )

    else:

        index = (
            len(
                categories
            )
            -
            1
        )


    output[
        index
    ] = 1.0


    return output


def safe_implicit_valence(
    atom
):

    try:

        return float(
            atom.GetImplicitValence()
        )

    except Exception:

        return 0.0


def safe_explicit_valence(
    atom
):

    try:

        return float(
            atom.GetExplicitValence()
        )

    except Exception:

        return 0.0


def atom_feature_vector(
    atom
):

    hydrogen_neighbor_count = sum(

        1

        for neighbor
        in atom.GetNeighbors()

        if neighbor.GetAtomicNum()
        ==
        1
    )


    features = [

        float(
            atom.GetAtomicNum()
        ),

        float(
            atom.GetDegree()
        ),

        float(
            atom.GetFormalCharge()
        ),

        float(
            atom.GetIsAromatic()
        ),

        float(
            atom.IsInRing()
        ),

        float(
            hydrogen_neighbor_count
        ),

        float(
            atom.GetMass()
        ),

        safe_implicit_valence(
            atom
        ),

        safe_explicit_valence(
            atom
        )
    ]


    features.extend(
        one_hot(
            str(
                atom.GetHybridization()
            ),
            HYBRIDIZATION_TYPES
        )
    )


    features.extend(
        one_hot(
            str(
                atom.GetChiralTag()
            ),
            CHIRALITY_TYPES
        )
    )


    return features


def bond_feature_vector(
    bond
):

    features = []


    features.extend(
        one_hot(
            str(
                bond.GetBondType()
            ),
            BOND_TYPES
        )
    )


    features.extend(
        [

            float(
                bond.GetIsConjugated()
            ),

            float(
                bond.IsInRing()
            )
        ]
    )


    features.extend(
        one_hot(
            str(
                bond.GetStereo()
            ),
            BOND_STEREO_TYPES
        )
    )


    return features


def build_node_features(
    mol
):

    output = torch.tensor(
        [

            atom_feature_vector(
                atom
            )

            for atom
            in mol.GetAtoms()
        ],
        dtype=torch.float32
    )


    if output.shape[
        1
    ] != NODE_FEATURE_DIM:

        raise RuntimeError(
            "Node-feature dimension mismatch."
        )


    return output


def build_edges(
    mol
):

    source = []

    target = []

    attributes = []


    for bond in mol.GetBonds():

        i = bond.GetBeginAtomIdx()

        j = bond.GetEndAtomIdx()


        feature = bond_feature_vector(
            bond
        )


        source.extend(
            [
                i,
                j
            ]
        )


        target.extend(
            [
                j,
                i
            ]
        )


        attributes.extend(
            [
                feature,
                feature
            ]
        )


    if len(
        source
    ) == 0:

        edge_index = torch.empty(
            (
                2,
                0
            ),
            dtype=torch.long
        )


        edge_attr = torch.empty(
            (
                0,
                EDGE_FEATURE_DIM
            ),
            dtype=torch.float32
        )


    else:

        edge_index = torch.tensor(
            [
                source,
                target
            ],
            dtype=torch.long
        )


        edge_attr = torch.tensor(
            attributes,
            dtype=torch.float32
        )


    return (
        edge_index,
        edge_attr
    )


# ==============================================================================
# 15. GEOMETRY HELPERS
# ==============================================================================

def generate_primary_conformer(
    mol
):

    params = AllChem.ETKDGv3()

    params.randomSeed = RANDOM_SEED

    params.useRandomCoords = False

    params.enforceChirality = True

    params.clearConfs = True


    return AllChem.EmbedMolecule(
        mol,
        params
    )


def generate_fallback_conformer(
    mol
):

    mol.RemoveAllConformers()


    params = AllChem.ETKDGv3()

    params.randomSeed = RANDOM_SEED

    params.useRandomCoords = True

    params.enforceChirality = True

    params.clearConfs = True


    return AllChem.EmbedMolecule(
        mol,
        params
    )


def optimize_geometry(
    mol,
    conformer_id
):

    try:

        if AllChem.MMFFHasAllMoleculeParams(
            mol
        ):

            props = (
                AllChem.MMFFGetMoleculeProperties(
                    mol,
                    mmffVariant="MMFF94"
                )
            )


            ff = (
                AllChem.MMFFGetMoleculeForceField(
                    mol,
                    props,
                    confId=conformer_id
                )
            )


            if ff is not None:

                ff.Minimize(
                    maxIts=
                        MAX_OPTIMIZATION_ITERS
                )

                return "MMFF94"

    except Exception:

        pass


    try:

        if AllChem.UFFHasAllMoleculeParams(
            mol
        ):

            ff = (
                AllChem.UFFGetMoleculeForceField(
                    mol,
                    confId=conformer_id
                )
            )


            if ff is not None:

                ff.Minimize(
                    maxIts=
                        MAX_OPTIMIZATION_ITERS
                )

                return "UFF"

    except Exception:

        pass


    return "NONE"


def centered_coordinates(
    mol,
    conformer_id
):

    coordinates = np.asarray(
        mol.GetConformer(
            conformer_id
        ).GetPositions(),
        dtype=np.float32
    )


    coordinates = (

        coordinates

        -

        coordinates.mean(
            axis=0,
            keepdims=True
        )
    )


    return torch.tensor(
        coordinates,
        dtype=torch.float32
    )


def build_graph_from_smiles(
    candidate_id,
    smiles
):

    mol = Chem.MolFromSmiles(
        smiles
    )


    if mol is None:

        raise RuntimeError(
            f"RDKit parsing failed for {candidate_id}"
        )


    mol_h = Chem.AddHs(
        mol
    )


    conformer_id = generate_primary_conformer(
        mol_h
    )


    if conformer_id < 0:

        conformer_id = generate_fallback_conformer(
            mol_h
        )


    if conformer_id < 0:

        raise RuntimeError(
            f"3D conformer failed for {candidate_id}"
        )


    optimize_geometry(
        mol_h,
        conformer_id
    )


    x = build_node_features(
        mol_h
    )


    edge_index, edge_attr = build_edges(
        mol_h
    )


    pos = centered_coordinates(
        mol_h,
        conformer_id
    )


    return {

        "x":
            x,

        "edge_index":
            edge_index,

        "edge_attr":
            edge_attr,

        "pos":
            pos
    }


# ==============================================================================
# 16. EXACT PHASE-9 NORMALIZATION
# ==============================================================================

with open(
    PHASE9_NORMALIZATION_FILE,
    "r",
    encoding="utf-8"
) as handle:

    phase9_normalization = json.load(
        handle
    )


NORMALIZED_FEATURE_NAMES = list(
    phase9_normalization[
        "standardized_features"
    ]
)


NORMALIZATION_INDICES = list(
    phase9_normalization[
        "standardized_feature_indices"
    ]
)


training_feature_mean = torch.tensor(
    [

        float(
            phase9_normalization[
                "mean"
            ][
                feature
            ]
        )

        for feature
        in NORMALIZED_FEATURE_NAMES
    ],
    dtype=torch.float32
)


training_feature_std = torch.tensor(
    [

        float(
            phase9_normalization[
                "std"
            ][
                feature
            ]
        )

        for feature
        in NORMALIZED_FEATURE_NAMES
    ],
    dtype=torch.float32
)


def normalize_node_features(
    x
):

    output = x.clone().float()


    index_tensor = torch.tensor(
        NORMALIZATION_INDICES,
        dtype=torch.long,
        device=output.device
    )


    output[
        :,
        index_tensor
    ] = (

        output[
            :,
            index_tensor
        ]

        -

        training_feature_mean.to(
            output.device
        )

    ) / training_feature_std.to(
        output.device
    )


    return output


print(
    "[PASS] Exact Phase-9 normalization loaded."
)


# ==============================================================================
# 17. SCATTER MEAN
# ==============================================================================

def scatter_mean(
    values,
    index,
    dim_size
):

    output = torch.zeros(
        (
            dim_size,
            values.shape[
                1
            ]
        ),
        dtype=values.dtype,
        device=values.device
    )


    counts = torch.zeros(
        (
            dim_size,
            1
        ),
        dtype=values.dtype,
        device=values.device
    )


    output.index_add_(
        0,
        index,
        values
    )


    counts.index_add_(
        0,
        index,
        torch.ones(
            (
                values.shape[
                    0
                ],
                1
            ),
            dtype=values.dtype,
            device=values.device
        )
    )


    return (
        output
        /
        counts.clamp(
            min=1.0
        )
    )


# ==============================================================================
# 18. EXACT PHASE-9 EGNN
# ==============================================================================

class EGNNLayer(
    nn.Module
):

    def __init__(
        self,
        hidden_dim,
        edge_dim,
        coordinate_update_scale=0.10
    ):

        super().__init__()


        self.coordinate_update_scale = (
            coordinate_update_scale
        )


        self.edge_mlp = nn.Sequential(

            nn.Linear(
                hidden_dim * 2 + 1 + edge_dim,
                hidden_dim
            ),

            nn.SiLU(),

            nn.Linear(
                hidden_dim,
                hidden_dim
            ),

            nn.SiLU()
        )


        self.coordinate_mlp = nn.Sequential(

            nn.Linear(
                hidden_dim,
                hidden_dim
            ),

            nn.SiLU(),

            nn.Linear(
                hidden_dim,
                1
            ),

            nn.Tanh()
        )


        self.node_mlp = nn.Sequential(

            nn.Linear(
                hidden_dim * 2,
                hidden_dim
            ),

            nn.SiLU(),

            nn.Linear(
                hidden_dim,
                hidden_dim
            )
        )


        self.node_norm = nn.LayerNorm(
            hidden_dim
        )


    def forward(
        self,
        h,
        pos,
        edge_index,
        edge_attr
    ):

        source = edge_index[
            0
        ]


        target = edge_index[
            1
        ]


        relative = (

            pos[
                source
            ]

            -

            pos[
                target
            ]
        )


        squared_distance = (
            relative
            *
            relative
        ).sum(
            dim=1,
            keepdim=True
        )


        edge_input = torch.cat(
            [

                h[
                    source
                ],

                h[
                    target
                ],

                squared_distance,

                edge_attr
            ],
            dim=1
        )


        messages = self.edge_mlp(
            edge_input
        )


        coordinate_weight = (

            self.coordinate_mlp(
                messages
            )

            *

            self.coordinate_update_scale
        )


        coordinate_messages = (

            relative

            *

            coordinate_weight
        )


        coordinate_aggregate = scatter_mean(
            coordinate_messages,
            source,
            h.shape[
                0
            ]
        )


        new_pos = (

            pos

            +

            coordinate_aggregate
        )


        aggregated_messages = scatter_mean(
            messages,
            source,
            h.shape[
                0
            ]
        )


        node_delta = self.node_mlp(
            torch.cat(
                [
                    h,
                    aggregated_messages
                ],
                dim=1
            )
        )


        new_h = self.node_norm(

            h

            +

            node_delta
        )


        return (
            new_h,
            new_pos
        )


class GeometryAwareVariationalEncoder(
    nn.Module
):

    def __init__(
        self
    ):

        super().__init__()


        self.node_embedding = nn.Sequential(

            nn.Linear(
                NODE_FEATURE_DIM,
                HIDDEN_DIM
            ),

            nn.SiLU(),

            nn.LayerNorm(
                HIDDEN_DIM
            )
        )


        self.egnn_layers = nn.ModuleList(
            [

                EGNNLayer(
                    HIDDEN_DIM,
                    EDGE_FEATURE_DIM,
                    COORDINATE_UPDATE_SCALE
                )

                for _
                in range(
                    NUM_EGNN_LAYERS
                )
            ]
        )


        self.graph_mlp = nn.Sequential(

            nn.Linear(
                HIDDEN_DIM,
                HIDDEN_DIM
            ),

            nn.SiLU(),

            nn.LayerNorm(
                HIDDEN_DIM
            )
        )


        self.mu_head = nn.Linear(
            HIDDEN_DIM,
            LATENT_DIM
        )


        self.logvar_head = nn.Linear(
            HIDDEN_DIM,
            LATENT_DIM
        )


    def forward(
        self,
        x,
        pos,
        edge_index,
        edge_attr,
        batch,
        deterministic=True
    ):

        h = self.node_embedding(
            x
        )


        current_pos = pos


        for layer in self.egnn_layers:

            h, current_pos = layer(
                h,
                current_pos,
                edge_index,
                edge_attr
            )


        graph_embedding = scatter_mean(
            h,
            batch,
            int(
                batch.max().item()
            )
            +
            1
        )


        graph_embedding = self.graph_mlp(
            graph_embedding
        )


        mu = self.mu_head(
            graph_embedding
        )


        logvar = self.logvar_head(
            graph_embedding
        )


        z = mu


        return {

            "mu":
                mu,

            "logvar":
                logvar,

            "z":
                z
        }


# ==============================================================================
# 19. PHASE-12 TARGET PREDICTOR
# ==============================================================================

class Phase12TargetPredictor(
    nn.Module
):

    def __init__(
        self
    ):

        super().__init__()


        self.network = nn.Sequential(

            nn.Linear(
                32,
                64
            ),

            nn.LayerNorm(
                64
            ),

            nn.SiLU(),

            nn.Dropout(
                0.15
            ),

            nn.Linear(
                64,
                32
            ),

            nn.LayerNorm(
                32
            ),

            nn.SiLU(),

            nn.Dropout(
                0.15
            ),

            nn.Linear(
                32,
                130
            )
        )


    def forward(
        self,
        x
    ):

        return self.network(
            x
        )


# ==============================================================================
# 20. LOAD K0 + K1 MODELS
# ==============================================================================

K0_encoder = (
    GeometryAwareVariationalEncoder()
    .to(
        DEVICE
    )
)


K1_encoder = (
    GeometryAwareVariationalEncoder()
    .to(
        DEVICE
    )
)


K0_predictor = (
    Phase12TargetPredictor()
    .to(
        DEVICE
    )
)


K1_predictor = (
    Phase12TargetPredictor()
    .to(
        DEVICE
    )
)


K0_encoder_checkpoint = torch.load(
    PHASE12_ENCODER_FILE,
    map_location="cpu",
    weights_only=False
)


K0_predictor_checkpoint = torch.load(
    PHASE12_PREDICTOR_FILE,
    map_location="cpu",
    weights_only=False
)


K0_encoder.load_state_dict(
    normalize_state_dict_keys(
        extract_state_dict(
            K0_encoder_checkpoint
        )
    ),
    strict=True
)


K0_predictor.load_state_dict(
    normalize_state_dict_keys(
        extract_state_dict(
            K0_predictor_checkpoint
        )
    ),
    strict=True
)


K1_encoder.load_state_dict(
    normalize_state_dict_keys(
        extract_state_dict(
            K1_encoder_checkpoint
        )
    ),
    strict=True
)


K1_predictor.load_state_dict(
    normalize_state_dict_keys(
        extract_state_dict(
            K1_predictor_checkpoint
        )
    ),
    strict=True
)


for model in [

    K0_encoder,

    K1_encoder,

    K0_predictor,

    K1_predictor
]:

    model.eval()


print(
    "[PASS] K0 and K1 encoder/predictor states loaded."
)


# ==============================================================================
# 21. LOAD LATENT NORMALIZATION
# ==============================================================================

with open(
    PHASE12_LATENT_NORMALIZATION_FILE,
    "r",
    encoding="utf-8"
) as handle:

    latent_normalization = json.load(
        handle
    )


latent_mean = torch.tensor(
    latent_normalization[
        "mean"
    ],
    dtype=torch.float32,
    device=DEVICE
)


latent_safe_std = torch.tensor(
    latent_normalization[
        "safe_std"
    ],
    dtype=torch.float32,
    device=DEVICE
)


# ==============================================================================
# 22. TARGET VOCABULARY
# ==============================================================================

target_vocab_df = pd.read_csv(
    PHASE12_TARGET_VOCAB_FILE,
    dtype=str
)


target_vocab_df[
    "target_index"
] = pd.to_numeric(
    target_vocab_df[
        "target_index"
    ],
    errors="raise"
).astype(
    int
)


target_vocab_df = (
    target_vocab_df
    .sort_values(
        "target_index"
    )
    .reset_index(
        drop=True
    )
)


if len(
    target_vocab_df
) != TARGET_COUNT:

    raise RuntimeError(
        "Expected exactly 130 target classes."
    )


target_id_to_index = dict(
    zip(
        target_vocab_df[
            "target_id"
        ].astype(str),
        target_vocab_df[
            "target_index"
        ].astype(int)
    )
)


target_index_to_id = dict(
    zip(
        target_vocab_df[
            "target_index"
        ].astype(int),
        target_vocab_df[
            "target_id"
        ].astype(str)
    )
)


target_index_to_name = dict(
    zip(
        target_vocab_df[
            "target_index"
        ].astype(int),
        target_vocab_df[
            "target_name"
        ].fillna("").astype(str)
    )
)


TARGET_MATRIX_COLUMNS = [

    f"target_{index:03d}"

    for index
    in range(
        TARGET_COUNT
    )
]


# ==============================================================================
# 23. LOAD S1
# ==============================================================================

S1_df = pd.read_csv(
    PHASE17_S1_FILE,
    dtype={
        "candidate_id":
            str,

        "canonical_smiles":
            str
    }
)


if len(
    S1_df
) != EXPECTED_S1_SIZE:

    raise RuntimeError(
        f"Expected {EXPECTED_S1_SIZE} S1 candidates; "
        f"found {len(S1_df)}."
    )


for column in [

    "qed",

    "sa_utility",

    "utility_score",

    "utility_rank"
]:

    S1_df[
        column
    ] = pd.to_numeric(
        S1_df[
            column
        ],
        errors="raise"
    )


frozen_top5_indices = {}


for _, row in S1_df.iterrows():

    candidate_id = str(
        row[
            "candidate_id"
        ]
    )


    indices = []


    for position in range(
        1,
        6
    ):

        column = (
            f"top_target_{position}_id"
        )


        target_id = str(
            row[
                column
            ]
        )


        if target_id not in target_id_to_index:

            raise RuntimeError(
                f"{candidate_id}: unknown target {target_id}"
            )


        indices.append(
            target_id_to_index[
                target_id
            ]
        )


    frozen_top5_indices[
        candidate_id
    ] = indices


print(
    f"[PASS] S1 loaded: {len(S1_df)} candidates."
)


# ==============================================================================
# 24. PRECOMPUTE S1 GRAPHS
# ==============================================================================

S1_GRAPHS = {}


print(
    "\n"
    +
    "=" * 155
)

print(
    "PRECOMPUTING S1 GRAPHS"
)

print(
    "=" * 155
)


for row_number, (_, row) in enumerate(
    S1_df.iterrows(),
    start=1
):

    candidate_id = str(
        row[
            "candidate_id"
        ]
    )


    S1_GRAPHS[
        candidate_id
    ] = build_graph_from_smiles(
        candidate_id,
        str(
            row[
                "canonical_smiles"
            ]
        )
    )


    print(
        f"[GRAPH] {row_number:02d}/{len(S1_df):02d} "
        f"{candidate_id} PASS"
    )


print(
    "[PASS] All S1 graphs precomputed."
)


# ==============================================================================
# 25. LOAD 128 FIXED REFERENCE GRAPHS
# ==============================================================================

D_fisher = pd.read_csv(
    PHASE8_FISHER_REFERENCE_FILE,
    dtype=str,
    keep_default_na=False
)


if len(
    D_fisher
) != EXPECTED_REFERENCE_SIZE:

    raise RuntimeError(
        f"Expected {EXPECTED_REFERENCE_SIZE} reference molecules; "
        f"found {len(D_fisher)}."
    )


def resolve_reference_graph_file(
    row
):

    for column in [

        "resolved_graph_file",

        "graph_file",

        "graph_path",

        "file_path",

        "path"
    ]:

        if column in row.index:

            path_text = str(
                row[
                    column
                ]
            ).strip()


            if path_text:

                path = Path(
                    path_text
                )


                if path.exists():

                    return path


    graph_id = str(
        row[
            "graph_id"
        ]
    )


    direct = (
        GRAPH_DIR
        /
        f"{graph_id}.pt"
    )


    if direct.exists():

        return direct


    matches = list(
        GRAPH_DIR.rglob(
            f"*{graph_id}*.pt"
        )
    )


    if matches:

        return matches[
            0
        ]


    raise FileNotFoundError(
        f"Graph file not found for {graph_id}"
    )


def extract_saved_graph(
    graph_object
):

    return {

        "x":
            graph_object[
                "x"
            ].float(),

        "pos":
            graph_object[
                "pos"
            ].float(),

        "edge_index":
            graph_object[
                "edge_index"
            ].long(),

        "edge_attr":
            graph_object[
                "edge_attr"
            ].float()
    }


REFERENCE_GRAPHS = {}

reference_metadata = []


for _, row in D_fisher.iterrows():

    graph_id = str(
        row[
            "graph_id"
        ]
    )


    graph_file = resolve_reference_graph_file(
        row
    )


    graph_object = torch.load(
        graph_file,
        map_location="cpu",
        weights_only=False
    )


    REFERENCE_GRAPHS[
        graph_id
    ] = extract_saved_graph(
        graph_object
    )


    reference_metadata.append(
        {

            "graph_id":
                graph_id,

            "drug_id":
                (
                    str(
                        row[
                            "drug_id"
                        ]
                    )
                    if "drug_id" in row.index
                    else
                    graph_id
                )
        }
    )


print(
    f"[PASS] Fixed reference graphs loaded: "
    f"{len(REFERENCE_GRAPHS)}"
)


# ==============================================================================
# 26. DETERMINISTIC ENCODER
# ==============================================================================

def forward_mu(
    encoder_model,
    graph
):

    encoder_model.eval()


    x = normalize_node_features(
        graph[
            "x"
        ]
    ).to(
        DEVICE
    )


    pos = graph[
        "pos"
    ].float().to(
        DEVICE
    )


    edge_index = graph[
        "edge_index"
    ].long().to(
        DEVICE
    )


    edge_attr = graph[
        "edge_attr"
    ].float().to(
        DEVICE
    )


    batch = torch.zeros(
        x.shape[
            0
        ],
        dtype=torch.long,
        device=DEVICE
    )


    with torch.no_grad():

        output = encoder_model(

            x=x,

            pos=pos,

            edge_index=edge_index,

            edge_attr=edge_attr,

            batch=batch,

            deterministic=True
        )


    return output[
        "mu"
    ].squeeze(
        0
    )


def normalize_latent(
    mu
):

    return (

        mu

        -

        latent_mean

    ) / latent_safe_std


# ==============================================================================
# 27. MC DROPOUT MODE
#
# Predictor globally stays in eval mode.
# ONLY nn.Dropout modules are switched to train mode.
#
# LayerNorm therefore remains deterministic.
# ==============================================================================

def enable_mc_dropout_only(
    model
):

    model.eval()


    dropout_count = 0


    for module in model.modules():

        if isinstance(
            module,
            nn.Dropout
        ):

            module.train()

            dropout_count += 1


    return dropout_count


# ==============================================================================
# 28. PAIRED MC-DROPOUT INFERENCE
# ==============================================================================

def mc_dropout_prediction(
    predictor_model,
    normalized_latent,
    seed_schedule
):

    dropout_count = enable_mc_dropout_only(
        predictor_model
    )


    if dropout_count <= 0:

        raise RuntimeError(
            "No Dropout modules found in predictor."
        )


    predictions = []


    with torch.no_grad():

        for seed in seed_schedule:

            torch.manual_seed(
                int(
                    seed
                )
            )


            if torch.cuda.is_available():

                torch.cuda.manual_seed_all(
                    int(
                        seed
                    )
                )


            logits = predictor_model(
                normalized_latent.unsqueeze(
                    0
                )
            )


            probabilities = torch.sigmoid(
                logits
            ).squeeze(
                0
            )


            predictions.append(
                probabilities.detach()
                .cpu()
                .numpy()
            )


    predictor_model.eval()


    predictions = np.asarray(
        predictions,
        dtype=np.float32
    )


    mean_prediction = predictions.mean(
        axis=0
    )


    uncertainty = predictions.std(
        axis=0,
        ddof=1
    )


    return {

        "mean":
            mean_prediction,

        "std":
            uncertainty,

        "passes":
            predictions,

        "dropout_modules":
            dropout_count
    }


# Same pass-wise seeds for K0 and K1.
MC_SEED_SCHEDULE = [

    MC_SEED_BASE
    +
    pass_index

    for pass_index
    in range(
        MC_DROPOUT_PASSES
    )
]


# ==============================================================================
# 29. EVALUATE ONE GRAPH UNDER K0 AND K1
# ==============================================================================

def evaluate_graph_K0_K1(
    graph
):

    mu_K0 = forward_mu(
        K0_encoder,
        graph
    )


    mu_K1 = forward_mu(
        K1_encoder,
        graph
    )


    z_K0 = normalize_latent(
        mu_K0
    )


    z_K1 = normalize_latent(
        mu_K1
    )


    mc_K0 = mc_dropout_prediction(
        K0_predictor,
        z_K0,
        MC_SEED_SCHEDULE
    )


    mc_K1 = mc_dropout_prediction(
        K1_predictor,
        z_K1,
        MC_SEED_SCHEDULE
    )


    return {

        "mu_K0":
            mu_K0.detach()
            .cpu()
            .numpy(),

        "mu_K1":
            mu_K1.detach()
            .cpu()
            .numpy(),

        "mean_K0":
            mc_K0[
                "mean"
            ],

        "std_K0":
            mc_K0[
                "std"
            ],

        "mean_K1":
            mc_K1[
                "mean"
            ],

        "std_K1":
            mc_K1[
                "std"
            ],

        "dropout_modules":
            mc_K0[
                "dropout_modules"
            ]
    }


# ==============================================================================
# 30. EVALUATE S1
# ==============================================================================

print(
    "\n"
    +
    "=" * 155
)

print(
    "30-PASS MC-DROPOUT EVALUATION — S1"
)

print(
    "=" * 155
)


S1_RESULTS = {}


for index, candidate_id in enumerate(
    S1_df[
        "candidate_id"
    ].astype(str),
    start=1
):

    S1_RESULTS[
        candidate_id
    ] = evaluate_graph_K0_K1(
        S1_GRAPHS[
            candidate_id
        ]
    )


    print(
        f"[S1] {index:02d}/{len(S1_df):02d} "
        f"{candidate_id} PASS"
    )


# ==============================================================================
# 31. EVALUATE REFERENCE SET
# ==============================================================================

print(
    "\n"
    +
    "=" * 155
)

print(
    "30-PASS MC-DROPOUT EVALUATION — 128 REFERENCE MOLECULES"
)

print(
    "=" * 155
)


REFERENCE_RESULTS = {}


for index, graph_id in enumerate(
    D_fisher[
        "graph_id"
    ].astype(str),
    start=1
):

    REFERENCE_RESULTS[
        graph_id
    ] = evaluate_graph_K0_K1(
        REFERENCE_GRAPHS[
            graph_id
        ]
    )


    if (
        index == 1
        or
        index % 16 == 0
        or
        index == len(
            D_fisher
        )
    ):

        print(
            f"[REFERENCE] {index:03d}/{len(D_fisher):03d} PASS"
        )


print(
    "[PASS] MC-dropout evaluation completed."
)


# ==============================================================================
# 32. EXPORT TARGET MATRICES
# ==============================================================================

def create_matrix_df(
    identifiers,
    result_dict,
    result_key,
    id_column
):

    rows = []


    for identifier in identifiers:

        row = {

            id_column:
                identifier
        }


        values = result_dict[
            identifier
        ][
            result_key
        ]


        for target_index in range(
            TARGET_COUNT
        ):

            row[
                TARGET_MATRIX_COLUMNS[
                    target_index
                ]
            ] = float(
                values[
                    target_index
                ]
            )


        rows.append(
            row
        )


    return pd.DataFrame(
        rows
    )


s1_ids = (
    S1_df[
        "candidate_id"
    ]
    .astype(str)
    .tolist()
)


reference_ids = (
    D_fisher[
        "graph_id"
    ]
    .astype(str)
    .tolist()
)


S1_K0_mean_df = create_matrix_df(
    s1_ids,
    S1_RESULTS,
    "mean_K0",
    "candidate_id"
)


S1_K0_std_df = create_matrix_df(
    s1_ids,
    S1_RESULTS,
    "std_K0",
    "candidate_id"
)


S1_K1_mean_df = create_matrix_df(
    s1_ids,
    S1_RESULTS,
    "mean_K1",
    "candidate_id"
)


S1_K1_std_df = create_matrix_df(
    s1_ids,
    S1_RESULTS,
    "std_K1",
    "candidate_id"
)


REF_K0_mean_df = create_matrix_df(
    reference_ids,
    REFERENCE_RESULTS,
    "mean_K0",
    "graph_id"
)


REF_K0_std_df = create_matrix_df(
    reference_ids,
    REFERENCE_RESULTS,
    "std_K0",
    "graph_id"
)


REF_K1_mean_df = create_matrix_df(
    reference_ids,
    REFERENCE_RESULTS,
    "mean_K1",
    "graph_id"
)


REF_K1_std_df = create_matrix_df(
    reference_ids,
    REFERENCE_RESULTS,
    "std_K1",
    "graph_id"
)


S1_K0_mean_df.to_csv(
    PHASE18_S1_K0_MEAN_FILE,
    index=False
)


S1_K0_std_df.to_csv(
    PHASE18_S1_K0_UNCERTAINTY_FILE,
    index=False
)


S1_K1_mean_df.to_csv(
    PHASE18_S1_K1_MEAN_FILE,
    index=False
)


S1_K1_std_df.to_csv(
    PHASE18_S1_K1_UNCERTAINTY_FILE,
    index=False
)


REF_K0_mean_df.to_csv(
    PHASE18_REFERENCE_K0_MEAN_FILE,
    index=False
)


REF_K0_std_df.to_csv(
    PHASE18_REFERENCE_K0_UNCERTAINTY_FILE,
    index=False
)


REF_K1_mean_df.to_csv(
    PHASE18_REFERENCE_K1_MEAN_FILE,
    index=False
)


REF_K1_std_df.to_csv(
    PHASE18_REFERENCE_K1_UNCERTAINTY_FILE,
    index=False
)


# ==============================================================================
# 33. S1 LOW-UNCERTAINTY BASELINE THRESHOLD
#
# Defined from K0 only.
# This does NOT inspect K1 to choose the threshold.
# ==============================================================================

S1_K0_TOP5_UNCERTAINTIES = []


for candidate_id in s1_ids:

    indices = frozen_top5_indices[
        candidate_id
    ]


    value = float(
        np.mean(
            S1_RESULTS[
                candidate_id
            ][
                "std_K0"
            ][
                indices
            ]
        )
    )


    S1_K0_TOP5_UNCERTAINTIES.append(
        value
    )


S1_LOW_UNCERTAINTY_THRESHOLD = float(
    np.quantile(
        S1_K0_TOP5_UNCERTAINTIES,
        LOW_UNCERTAINTY_BASELINE_QUANTILE
    )
)


# ==============================================================================
# 34. S1 COMPARISON
# ==============================================================================

s1_comparison_records = []


for _, row in S1_df.iterrows():

    candidate_id = str(
        row[
            "candidate_id"
        ]
    )


    result = S1_RESULTS[
        candidate_id
    ]


    fixed_indices = frozen_top5_indices[
        candidate_id
    ]


    K0_B = float(
        np.mean(
            result[
                "mean_K0"
            ][
                fixed_indices
            ]
        )
    )


    K1_B = float(
        np.mean(
            result[
                "mean_K1"
            ][
                fixed_indices
            ]
        )
    )


    K0_uncertainty = float(
        np.mean(
            result[
                "std_K0"
            ][
                fixed_indices
            ]
        )
    )


    K1_uncertainty = float(
        np.mean(
            result[
                "std_K1"
            ][
                fixed_indices
            ]
        )
    )


    uncertainty_change = (

        K1_uncertainty

        -

        K0_uncertainty
    )


    uncertainty_ratio = (

        K1_uncertainty

        /

        max(
            K0_uncertainty,
            UNCERTAINTY_RATIO_EPSILON
        )
    )


    K0_utility = (

        WEIGHT_QED
        *
        float(
            row[
                "qed"
            ]
        )

        +

        WEIGHT_SA
        *
        float(
            row[
                "sa_utility"
            ]
        )

        +

        WEIGHT_BIO
        *
        K0_B
    )


    K1_utility = (

        WEIGHT_QED
        *
        float(
            row[
                "qed"
            ]
        )

        +

        WEIGHT_SA
        *
        float(
            row[
                "sa_utility"
            ]
        )

        +

        WEIGHT_BIO
        *
        K1_B
    )


    prediction_shift_vector = np.abs(

        result[
            "mean_K1"
        ]

        -

        result[
            "mean_K0"
        ]
    )


    maximum_prediction_shift = float(
        prediction_shift_vector.max()
    )


    mean_prediction_shift = float(
        prediction_shift_vector.mean()
    )


    K0_top_index = int(
        np.argmax(
            result[
                "mean_K0"
            ]
        )
    )


    K1_top_index = int(
        np.argmax(
            result[
                "mean_K1"
            ]
        )
    )


    large_uncertainty_increase = bool(

        uncertainty_ratio

        >

        LARGE_UNCERTAINTY_INCREASE_RATIO
    )


    large_uncertainty_decrease = bool(

        uncertainty_ratio

        <

        LARGE_UNCERTAINTY_DECREASE_RATIO
    )


    potential_uncertainty_collapse = bool(

        maximum_prediction_shift

        >=

        LARGE_PREDICTION_SHIFT_THRESHOLD

        and

        K1_uncertainty

        <=

        S1_LOW_UNCERTAINTY_THRESHOLD
    )


    s1_comparison_records.append(
        {

            "candidate_id":
                candidate_id,

            "canonical_smiles":
                str(
                    row[
                        "canonical_smiles"
                    ]
                ),

            "K0_biological_relevance":
                K0_B,

            "K1_biological_relevance":
                K1_B,

            "biological_relevance_change":
                K1_B
                -
                K0_B,

            "K0_top5_uncertainty":
                K0_uncertainty,

            "K1_top5_uncertainty":
                K1_uncertainty,

            "uncertainty_change":
                uncertainty_change,

            "uncertainty_ratio":
                uncertainty_ratio,

            "K0_utility":
                K0_utility,

            "K1_utility":
                K1_utility,

            "utility_change":
                K1_utility
                -
                K0_utility,

            "mean_target_prediction_shift":
                mean_prediction_shift,

            "maximum_target_prediction_shift":
                maximum_prediction_shift,

            "K0_top_target_id":
                target_index_to_id[
                    K0_top_index
                ],

            "K0_top_target_name":
                target_index_to_name[
                    K0_top_index
                ],

            "K1_top_target_id":
                target_index_to_id[
                    K1_top_index
                ],

            "K1_top_target_name":
                target_index_to_name[
                    K1_top_index
                ],

            "top_target_changed":
                bool(
                    K0_top_index
                    !=
                    K1_top_index
                ),

            "large_uncertainty_increase":
                large_uncertainty_increase,

            "large_uncertainty_decrease":
                large_uncertainty_decrease,

            "potential_uncertainty_collapse_flag":
                potential_uncertainty_collapse
        }
    )


S1_comparison_df = pd.DataFrame(
    s1_comparison_records
)


# ==============================================================================
# 35. COMPUTE K0 / K1 S1 RANKS
# ==============================================================================

S1_comparison_df[
    "K0_rank"
] = (
    S1_comparison_df[
        "K0_utility"
    ]
    .rank(
        ascending=False,
        method="min"
    )
    .astype(int)
)


S1_comparison_df[
    "K1_rank"
] = (
    S1_comparison_df[
        "K1_utility"
    ]
    .rank(
        ascending=False,
        method="min"
    )
    .astype(int)
)


S1_comparison_df[
    "rank_change"
] = (

    S1_comparison_df[
        "K1_rank"
    ]

    -

    S1_comparison_df[
        "K0_rank"
    ]
)


S1_comparison_df.to_csv(
    PHASE18_S1_COMPARISON_FILE,
    index=False
)


# ==============================================================================
# 36. REFERENCE LOW-UNCERTAINTY BASELINE THRESHOLD
# ==============================================================================

reference_K0_mean_uncertainties = []


for graph_id in reference_ids:

    reference_K0_mean_uncertainties.append(
        float(
            np.mean(
                REFERENCE_RESULTS[
                    graph_id
                ][
                    "std_K0"
                ]
            )
        )
    )


REFERENCE_LOW_UNCERTAINTY_THRESHOLD = float(
    np.quantile(
        reference_K0_mean_uncertainties,
        LOW_UNCERTAINTY_BASELINE_QUANTILE
    )
)


# ==============================================================================
# 37. REFERENCE LATENT POPULATION SCALE
# ==============================================================================

reference_K0_latents = np.asarray(
    [

        REFERENCE_RESULTS[
            graph_id
        ][
            "mu_K0"
        ]

        for graph_id
        in reference_ids
    ],
    dtype=np.float32
)


reference_K1_latents = np.asarray(
    [

        REFERENCE_RESULTS[
            graph_id
        ][
            "mu_K1"
        ]

        for graph_id
        in reference_ids
    ],
    dtype=np.float32
)


baseline_latent_dimension_std = (
    reference_K0_latents.std(
        axis=0,
        ddof=1
    )
)


baseline_latent_dimension_std = np.where(

    baseline_latent_dimension_std
    <
    1e-6,

    1.0,

    baseline_latent_dimension_std
)


# ==============================================================================
# 38. REFERENCE COMPARISON
# ==============================================================================

reference_comparison_records = []

latent_compatibility_records = []


for index, graph_id in enumerate(
    reference_ids
):

    result = REFERENCE_RESULTS[
        graph_id
    ]


    metadata = reference_metadata[
        index
    ]


    prediction_difference = np.abs(

        result[
            "mean_K1"
        ]

        -

        result[
            "mean_K0"
        ]
    )


    mean_prediction_drift = float(
        prediction_difference.mean()
    )


    max_prediction_drift = float(
        prediction_difference.max()
    )


    K0_uncertainty = float(
        np.mean(
            result[
                "std_K0"
            ]
        )
    )


    K1_uncertainty = float(
        np.mean(
            result[
                "std_K1"
            ]
        )
    )


    uncertainty_change = (

        K1_uncertainty

        -

        K0_uncertainty
    )


    uncertainty_ratio = (

        K1_uncertainty

        /

        max(
            K0_uncertainty,
            UNCERTAINTY_RATIO_EPSILON
        )
    )


    latent_delta = (

        result[
            "mu_K1"
        ]

        -

        result[
            "mu_K0"
        ]
    )


    latent_L2_drift = float(
        np.linalg.norm(
            latent_delta
        )
    )


    standardized_latent_delta = (

        latent_delta

        /

        baseline_latent_dimension_std
    )


    standardized_latent_RMS = float(
        np.sqrt(
            np.mean(
                standardized_latent_delta
                **
                2
            )
        )
    )


    K0_top_index = int(
        np.argmax(
            result[
                "mean_K0"
            ]
        )
    )


    K1_top_index = int(
        np.argmax(
            result[
                "mean_K1"
            ]
        )
    )


    large_uncertainty_increase = bool(

        uncertainty_ratio

        >

        LARGE_UNCERTAINTY_INCREASE_RATIO
    )


    large_uncertainty_decrease = bool(

        uncertainty_ratio

        <

        LARGE_UNCERTAINTY_DECREASE_RATIO
    )


    potential_uncertainty_collapse = bool(

        max_prediction_drift

        >=

        LARGE_PREDICTION_SHIFT_THRESHOLD

        and

        K1_uncertainty

        <=

        REFERENCE_LOW_UNCERTAINTY_THRESHOLD
    )


    severe_prediction_drift = bool(

        max_prediction_drift

        >

        SEVERE_PREDICTION_DRIFT_THRESHOLD
    )


    severe_latent_drift = bool(

        standardized_latent_RMS

        >

        LATENT_STANDARDIZED_RMS_SEVERE_THRESHOLD
    )


    reference_comparison_records.append(
        {

            "graph_id":
                graph_id,

            "drug_id":
                metadata[
                    "drug_id"
                ],

            "latent_L2_drift":
                latent_L2_drift,

            "standardized_latent_RMS_drift":
                standardized_latent_RMS,

            "mean_target_prediction_drift":
                mean_prediction_drift,

            "maximum_target_prediction_drift":
                max_prediction_drift,

            "K0_mean_uncertainty":
                K0_uncertainty,

            "K1_mean_uncertainty":
                K1_uncertainty,

            "uncertainty_change":
                uncertainty_change,

            "uncertainty_ratio":
                uncertainty_ratio,

            "K0_top_target_id":
                target_index_to_id[
                    K0_top_index
                ],

            "K0_top_target_name":
                target_index_to_name[
                    K0_top_index
                ],

            "K1_top_target_id":
                target_index_to_id[
                    K1_top_index
                ],

            "K1_top_target_name":
                target_index_to_name[
                    K1_top_index
                ],

            "top_target_changed":
                bool(
                    K0_top_index
                    !=
                    K1_top_index
                ),

            "large_uncertainty_increase":
                large_uncertainty_increase,

            "large_uncertainty_decrease":
                large_uncertainty_decrease,

            "potential_uncertainty_collapse_flag":
                potential_uncertainty_collapse,

            "severe_prediction_drift":
                severe_prediction_drift,

            "severe_latent_drift":
                severe_latent_drift
        }
    )


    latent_compatibility_records.append(
        {

            "graph_id":
                graph_id,

            "drug_id":
                metadata[
                    "drug_id"
                ],

            "K0_latent_norm":
                float(
                    np.linalg.norm(
                        result[
                            "mu_K0"
                        ]
                    )
                ),

            "K1_latent_norm":
                float(
                    np.linalg.norm(
                        result[
                            "mu_K1"
                        ]
                    )
                ),

            "latent_L2_drift":
                latent_L2_drift,

            "standardized_latent_RMS_drift":
                standardized_latent_RMS,

            "severe_latent_drift":
                severe_latent_drift
        }
    )


reference_comparison_df = pd.DataFrame(
    reference_comparison_records
)


latent_compatibility_df = pd.DataFrame(
    latent_compatibility_records
)


reference_comparison_df.to_csv(
    PHASE18_REFERENCE_COMPARISON_FILE,
    index=False
)


latent_compatibility_df.to_csv(
    PHASE18_LATENT_COMPATIBILITY_FILE,
    index=False
)


# ==============================================================================
# 39. UNCERTAINTY OUTLIERS
# ==============================================================================

outlier_records = []


for _, row in S1_comparison_df.iterrows():

    reasons = []


    if row[
        "large_uncertainty_increase"
    ]:

        reasons.append(
            "LARGE_UNCERTAINTY_INCREASE"
        )


    if row[
        "large_uncertainty_decrease"
    ]:

        reasons.append(
            "LARGE_UNCERTAINTY_DECREASE"
        )


    if row[
        "potential_uncertainty_collapse_flag"
    ]:

        reasons.append(
            "POTENTIAL_UNCERTAINTY_COLLAPSE"
        )


    if row[
        "top_target_changed"
    ]:

        reasons.append(
            "TOP_TARGET_CHANGED"
        )


    if reasons:

        outlier_records.append(
            {

                "population":
                    "S1",

                "molecule_id":
                    row[
                        "candidate_id"
                    ],

                "reasons":
                    ";".join(
                        reasons
                    ),

                "K0_uncertainty":
                    row[
                        "K0_top5_uncertainty"
                    ],

                "K1_uncertainty":
                    row[
                        "K1_top5_uncertainty"
                    ],

                "uncertainty_ratio":
                    row[
                        "uncertainty_ratio"
                    ],

                "maximum_prediction_shift":
                    row[
                        "maximum_target_prediction_shift"
                    ]
            }
        )


for _, row in reference_comparison_df.iterrows():

    reasons = []


    if row[
        "large_uncertainty_increase"
    ]:

        reasons.append(
            "LARGE_UNCERTAINTY_INCREASE"
        )


    if row[
        "large_uncertainty_decrease"
    ]:

        reasons.append(
            "LARGE_UNCERTAINTY_DECREASE"
        )


    if row[
        "potential_uncertainty_collapse_flag"
    ]:

        reasons.append(
            "POTENTIAL_UNCERTAINTY_COLLAPSE"
        )


    if row[
        "top_target_changed"
    ]:

        reasons.append(
            "TOP_TARGET_CHANGED"
        )


    if row[
        "severe_prediction_drift"
    ]:

        reasons.append(
            "SEVERE_PREDICTION_DRIFT"
        )


    if row[
        "severe_latent_drift"
    ]:

        reasons.append(
            "SEVERE_LATENT_DRIFT"
        )


    if reasons:

        outlier_records.append(
            {

                "population":
                    "REFERENCE",

                "molecule_id":
                    row[
                        "drug_id"
                    ],

                "reasons":
                    ";".join(
                        reasons
                    ),

                "K0_uncertainty":
                    row[
                        "K0_mean_uncertainty"
                    ],

                "K1_uncertainty":
                    row[
                        "K1_mean_uncertainty"
                    ],

                "uncertainty_ratio":
                    row[
                        "uncertainty_ratio"
                    ],

                "maximum_prediction_shift":
                    row[
                        "maximum_target_prediction_drift"
                    ]
            }
        )


uncertainty_outliers_df = pd.DataFrame(
    outlier_records
)


if uncertainty_outliers_df.empty:

    uncertainty_outliers_df = pd.DataFrame(
        columns=[
            "population",
            "molecule_id",
            "reasons",
            "K0_uncertainty",
            "K1_uncertainty",
            "uncertainty_ratio",
            "maximum_prediction_shift"
        ]
    )


uncertainty_outliers_df.to_csv(
    PHASE18_UNCERTAINTY_OUTLIERS_FILE,
    index=False
)


# ==============================================================================
# 40. UNCERTAINTY DISTRIBUTION SUMMARY
# ==============================================================================

def distribution_record(
    population,
    model_state,
    values
):

    values = np.asarray(
        values,
        dtype=float
    )


    return {

        "evaluation_population":
            population,

        "model_state":
            model_state,

        "mean_uncertainty":
            float(
                np.mean(
                    values
                )
            ),

        "median_uncertainty":
            float(
                np.median(
                    values
                )
            ),

        "std_uncertainty":
            float(
                np.std(
                    values,
                    ddof=1
                )
            ),

        "min_uncertainty":
            float(
                np.min(
                    values
                )
            ),

        "Q1_uncertainty":
            float(
                np.quantile(
                    values,
                    0.25
                )
            ),

        "Q3_uncertainty":
            float(
                np.quantile(
                    values,
                    0.75
                )
            ),

        "max_uncertainty":
            float(
                np.max(
                    values
                )
            )
    }


distribution_records = [

    distribution_record(
        "S1",
        "K0",
        S1_comparison_df[
            "K0_top5_uncertainty"
        ]
    ),

    distribution_record(
        "S1",
        "K1",
        S1_comparison_df[
            "K1_top5_uncertainty"
        ]
    ),

    distribution_record(
        "REFERENCE",
        "K0",
        reference_comparison_df[
            "K0_mean_uncertainty"
        ]
    ),

    distribution_record(
        "REFERENCE",
        "K1",
        reference_comparison_df[
            "K1_mean_uncertainty"
        ]
    )
]


distribution_summary_df = pd.DataFrame(
    distribution_records
)


distribution_summary_df.to_csv(
    PHASE18_DISTRIBUTION_SUMMARY_FILE,
    index=False
)


# ==============================================================================
# 41. RANKING STABILITY
# ==============================================================================

def spearman_rank_correlation(
    values_a,
    values_b
):

    rank_a = (
        pd.Series(
            values_a
        )
        .rank(
            ascending=False,
            method="average"
        )
        .to_numpy(
            dtype=float
        )
    )


    rank_b = (
        pd.Series(
            values_b
        )
        .rank(
            ascending=False,
            method="average"
        )
        .to_numpy(
            dtype=float
        )
    )


    return float(
        np.corrcoef(
            rank_a,
            rank_b
        )[
            0,
            1
        ]
    )


def kendall_tau_simple(
    x,
    y
):

    concordant = 0

    discordant = 0


    for i in range(
        len(
            x
        )
    ):

        for j in range(
            i + 1,
            len(
                x
            )
        ):

            product = (

                (
                    x[
                        i
                    ]
                    -
                    x[
                        j
                    ]
                )

                *

                (
                    y[
                        i
                    ]
                    -
                    y[
                        j
                    ]
                )
            )


            if product > 0:

                concordant += 1

            elif product < 0:

                discordant += 1


    denominator = (
        concordant
        +
        discordant
    )


    if denominator == 0:

        return float(
            "nan"
        )


    return float(
        (
            concordant
            -
            discordant
        )
        /
        denominator
    )


def pairwise_ordering_agreement(
    x,
    y
):

    agreements = 0

    comparable = 0


    for i in range(
        len(
            x
        )
    ):

        for j in range(
            i + 1,
            len(
                x
            )
        ):

            dx = (
                x[
                    i
                ]
                -
                x[
                    j
                ]
            )


            dy = (
                y[
                    i
                ]
                -
                y[
                    j
                ]
            )


            if (
                dx == 0
                or
                dy == 0
            ):

                continue


            comparable += 1


            if (
                dx
                *
                dy
                >
                0
            ):

                agreements += 1


    if comparable == 0:

        return float(
            "nan"
        )


    return float(
        agreements
        /
        comparable
    )


K0_utilities = (
    S1_comparison_df[
        "K0_utility"
    ]
    .to_numpy(
        dtype=float
    )
)


K1_utilities = (
    S1_comparison_df[
        "K1_utility"
    ]
    .to_numpy(
        dtype=float
    )
)


SPEARMAN_STABILITY = spearman_rank_correlation(
    K0_utilities,
    K1_utilities
)


KENDALL_STABILITY = kendall_tau_simple(
    K0_utilities,
    K1_utilities
)


PAIRWISE_ORDERING_AGREEMENT = pairwise_ordering_agreement(
    K0_utilities,
    K1_utilities
)


S1_TOP_TARGET_CHANGE_COUNT = int(
    S1_comparison_df[
        "top_target_changed"
    ].sum()
)


REFERENCE_TOP_TARGET_CHANGE_COUNT = int(
    reference_comparison_df[
        "top_target_changed"
    ].sum()
)


ranking_stability_df = pd.DataFrame(
    [

        {

            "metric":
                "Spearman_K0_vs_K1_utility",

            "value":
                SPEARMAN_STABILITY
        },

        {

            "metric":
                "Kendall_K0_vs_K1_utility",

            "value":
                KENDALL_STABILITY
        },

        {

            "metric":
                "pairwise_ordering_agreement",

            "value":
                PAIRWISE_ORDERING_AGREEMENT
        },

        {

            "metric":
                "S1_top_target_change_count",

            "value":
                S1_TOP_TARGET_CHANGE_COUNT
        },

        {

            "metric":
                "reference_top_target_change_count",

            "value":
                REFERENCE_TOP_TARGET_CHANGE_COUNT
        }
    ]
)


ranking_stability_df.to_csv(
    PHASE18_RANKING_STABILITY_FILE,
    index=False
)


# ==============================================================================
# 42. READINESS METRICS
# ==============================================================================

reference_severe_prediction_count = int(
    reference_comparison_df[
        "severe_prediction_drift"
    ].sum()
)


reference_top_target_change_fraction = float(
    reference_comparison_df[
        "top_target_changed"
    ].mean()
)


reference_mean_prediction_drift = float(
    reference_comparison_df[
        "mean_target_prediction_drift"
    ].mean()
)


reference_large_uncertainty_increase_fraction = float(
    reference_comparison_df[
        "large_uncertainty_increase"
    ].mean()
)


S1_large_uncertainty_increase_fraction = float(
    S1_comparison_df[
        "large_uncertainty_increase"
    ].mean()
)


reference_uncertainty_collapse_fraction = float(
    reference_comparison_df[
        "potential_uncertainty_collapse_flag"
    ].mean()
)


S1_uncertainty_collapse_fraction = float(
    S1_comparison_df[
        "potential_uncertainty_collapse_flag"
    ].mean()
)


reference_mean_standardized_latent_drift = float(
    latent_compatibility_df[
        "standardized_latent_RMS_drift"
    ].mean()
)


reference_severe_latent_drift_fraction = float(
    latent_compatibility_df[
        "severe_latent_drift"
    ].mean()
)


# ==============================================================================
# 43. READINESS COMPONENTS
# ==============================================================================

R_feedback = bool(
    phase17_accepted_config
    and
    phase17_accepted_qc
)


R_retention = bool(

    reference_severe_prediction_count
    <=
    MAX_ALLOWED_SEVERE_REFERENCE_DRIFT_COUNT

    and

    reference_mean_prediction_drift
    <=
    MAX_ALLOWED_MEAN_REFERENCE_PREDICTION_DRIFT

    and

    reference_top_target_change_fraction
    <=
    MAX_ALLOWED_REFERENCE_TOP_TARGET_CHANGE_FRACTION
)


R_uncertainty = bool(

    np.isfinite(
        S1_comparison_df[
            [
                "K0_top5_uncertainty",
                "K1_top5_uncertainty"
            ]
        ].to_numpy(
            dtype=float
        )
    ).all()

    and

    np.isfinite(
        reference_comparison_df[
            [
                "K0_mean_uncertainty",
                "K1_mean_uncertainty"
            ]
        ].to_numpy(
            dtype=float
        )
    ).all()

    and

    S1_large_uncertainty_increase_fraction
    <=
    MAX_ALLOWED_LARGE_UNCERTAINTY_INCREASE_FRACTION

    and

    reference_large_uncertainty_increase_fraction
    <=
    MAX_ALLOWED_LARGE_UNCERTAINTY_INCREASE_FRACTION

    and

    S1_uncertainty_collapse_fraction
    <=
    MAX_ALLOWED_UNCERTAINTY_COLLAPSE_FRACTION

    and

    reference_uncertainty_collapse_fraction
    <=
    MAX_ALLOWED_UNCERTAINTY_COLLAPSE_FRACTION
)


R_latent_compatibility = bool(

    np.isfinite(
        latent_compatibility_df[
            "standardized_latent_RMS_drift"
        ].to_numpy(
            dtype=float
        )
    ).all()

    and

    reference_mean_standardized_latent_drift
    <=
    MAX_ALLOWED_MEAN_STANDARDIZED_LATENT_DRIFT

    and

    reference_severe_latent_drift_fraction
    <=
    MAX_ALLOWED_SEVERE_LATENT_DRIFT_FRACTION
)


# ==============================================================================
# 44. FILE-INTEGRITY VERIFICATION
# ==============================================================================

HASHES_AFTER = {

    "K0_encoder":
        sha256_file(
            PHASE12_ENCODER_FILE
        ),

    "K0_decoder":
        sha256_file(
            PHASE12_DECODER_FILE
        ),

    "K0_predictor":
        sha256_file(
            PHASE12_PREDICTOR_FILE
        ),

    "K1_encoder":
        sha256_file(
            PHASE17_ENCODER_FILE
        ),

    "K1_predictor":
        sha256_file(
            PHASE17_PREDICTOR_FILE
        ),

    "F_E":
        sha256_file(
            PHASE13_ENCODER_FISHER_FILE
        ),

    "F_P":
        sha256_file(
            PHASE13_PREDICTOR_FISHER_FILE
        )
}


R_integrity = bool(
    HASHES_BEFORE
    ==
    HASHES_AFTER
)


Ready_Cycle2 = bool(

    R_feedback

    and

    R_retention

    and

    R_uncertainty

    and

    R_latent_compatibility

    and

    R_integrity
)


# ==============================================================================
# 45. READINESS JSON
# ==============================================================================

readiness_summary = {

    "phase":
        18,

    "created_at":
        datetime.now().isoformat(),

    "cycle":
        CYCLE,

    "Phase17_adaptation_accepted":
        R_feedback,

    "feedback_readiness":
        R_feedback,

    "knowledge_retention_readiness":
        R_retention,

    "uncertainty_readiness":
        R_uncertainty,

    "latent_compatibility_readiness":
        R_latent_compatibility,

    "model_integrity_readiness":
        R_integrity,

    "Ready_Cycle2":
        Ready_Cycle2,

    "selected_phase17_lambda_ewc":
        SELECTED_LAMBDA,

    "selected_phase17_epoch":
        SELECTED_EPOCH,

    "MC_dropout_passes":
        MC_DROPOUT_PASSES,

    "reference_severe_prediction_drift_count":
        reference_severe_prediction_count,

    "reference_mean_prediction_drift":
        reference_mean_prediction_drift,

    "reference_top_target_change_fraction":
        reference_top_target_change_fraction,

    "S1_large_uncertainty_increase_fraction":
        S1_large_uncertainty_increase_fraction,

    "reference_large_uncertainty_increase_fraction":
        reference_large_uncertainty_increase_fraction,

    "S1_uncertainty_collapse_fraction":
        S1_uncertainty_collapse_fraction,

    "reference_uncertainty_collapse_fraction":
        reference_uncertainty_collapse_fraction,

    "reference_mean_standardized_latent_drift":
        reference_mean_standardized_latent_drift,

    "reference_severe_latent_drift_fraction":
        reference_severe_latent_drift_fraction,

    "K0_K1_utility_spearman":
        SPEARMAN_STABILITY,

    "K0_K1_utility_kendall":
        KENDALL_STABILITY,

    "K0_K1_pairwise_ordering_agreement":
        PAIRWISE_ORDERING_AGREEMENT
}


with open(
    PHASE18_READINESS_FILE,
    "w",
    encoding="utf-8"
) as handle:

    json.dump(
        readiness_summary,
        handle,
        indent=4,
        ensure_ascii=False,
        default=str
    )


# ==============================================================================
# 46. PHASE-18 CONFIGURATION
# ==============================================================================

phase18_config = {

    "phase":
        (
            "Phase 18 — Post-Adaptation Uncertainty, "
            "Knowledge-Retention Validation, and "
            "Cycle-2 Generation Readiness"
        ),

    "created_at":
        datetime.now().isoformat(),

    "random_seed":
        RANDOM_SEED,

    "device":
        str(
            DEVICE
        ),

    "MC_dropout_passes":
        MC_DROPOUT_PASSES,

    "MC_seed_base":
        MC_SEED_BASE,

    "paired_MC_seed_schedule":
        True,

    "encoder_policy":
        "deterministic_mu",

    "predictor_stochasticity":
        "Dropout only",

    "LayerNorm_policy":
        "evaluation mode",

    "optimizer_created":
        False,

    "backward_pass":
        False,

    "parameter_updates":
        False,

    "Fisher_recalculated":
        False,

    "S1_count":
        len(
            S1_df
        ),

    "reference_count":
        len(
            D_fisher
        ),

    "severe_prediction_drift_threshold":
        SEVERE_PREDICTION_DRIFT_THRESHOLD,

    "large_uncertainty_increase_ratio":
        LARGE_UNCERTAINTY_INCREASE_RATIO,

    "large_uncertainty_decrease_ratio":
        LARGE_UNCERTAINTY_DECREASE_RATIO,

    "large_prediction_shift_threshold":
        LARGE_PREDICTION_SHIFT_THRESHOLD,

    "latent_standardized_RMS_severe_threshold":
        LATENT_STANDARDIZED_RMS_SEVERE_THRESHOLD,

    "Ready_Cycle2":
        Ready_Cycle2
}


with open(
    PHASE18_CONFIG_FILE,
    "w",
    encoding="utf-8"
) as handle:

    json.dump(
        phase18_config,
        handle,
        indent=4,
        ensure_ascii=False,
        default=str
    )


# ==============================================================================
# 47. COMPLETION CHECKS
# ==============================================================================

completion_checks = {

    "Phase-17 acceptance verified":
        R_feedback,

    "K0 encoder loaded":
        True,

    "K0 predictor loaded":
        True,

    "K1 encoder loaded":
        True,

    "K1 predictor loaded":
        True,

    "Decoder integrity verified":
        (
            HASHES_BEFORE[
                "K0_decoder"
            ]
            ==
            HASHES_AFTER[
                "K0_decoder"
            ]
        ),

    "Exact Phase-7 schema reused":
        True,

    "Exact Phase-9 normalization reused":
        True,

    "Phase-12 latent normalization reused":
        True,

    "130-target vocabulary reused":
        len(
            target_vocab_df
        )
        ==
        TARGET_COUNT,

    "S1 loaded":
        len(
            S1_df
        )
        ==
        EXPECTED_S1_SIZE,

    "128 reference molecules loaded":
        len(
            REFERENCE_GRAPHS
        )
        ==
        EXPECTED_REFERENCE_SIZE,

    "Deterministic encoder μ used":
        True,

    "MC dropout uses 30 passes":
        MC_DROPOUT_PASSES
        ==
        30,

    "Paired MC seed schedule used":
        True,

    "Dropout-only stochastic predictor policy used":
        True,

    "S1 K0 means calculated":
        len(
            S1_K0_mean_df
        )
        ==
        EXPECTED_S1_SIZE,

    "S1 K1 means calculated":
        len(
            S1_K1_mean_df
        )
        ==
        EXPECTED_S1_SIZE,

    "S1 K0 uncertainty calculated":
        len(
            S1_K0_std_df
        )
        ==
        EXPECTED_S1_SIZE,

    "S1 K1 uncertainty calculated":
        len(
            S1_K1_std_df
        )
        ==
        EXPECTED_S1_SIZE,

    "Reference K0 means calculated":
        len(
            REF_K0_mean_df
        )
        ==
        EXPECTED_REFERENCE_SIZE,

    "Reference K1 means calculated":
        len(
            REF_K1_mean_df
        )
        ==
        EXPECTED_REFERENCE_SIZE,

    "Reference K0 uncertainty calculated":
        len(
            REF_K0_std_df
        )
        ==
        EXPECTED_REFERENCE_SIZE,

    "Reference K1 uncertainty calculated":
        len(
            REF_K1_std_df
        )
        ==
        EXPECTED_REFERENCE_SIZE,

    "Fixed top-5 targets reused":
        len(
            frozen_top5_indices
        )
        ==
        EXPECTED_S1_SIZE,

    "S1 comparison complete":
        len(
            S1_comparison_df
        )
        ==
        EXPECTED_S1_SIZE,

    "Reference comparison complete":
        len(
            reference_comparison_df
        )
        ==
        EXPECTED_REFERENCE_SIZE,

    "Prediction drift audited":
        True,

    "Uncertainty shift audited":
        True,

    "Uncertainty-collapse audit completed":
        True,

    "Uncertainty expansion audit completed":
        True,

    "Latent compatibility evaluated":
        True,

    "Spearman ranking stability calculated":
        np.isfinite(
            SPEARMAN_STABILITY
        ),

    "Kendall ranking stability calculated":
        np.isfinite(
            KENDALL_STABILITY
        ),

    "Pairwise ordering agreement calculated":
        np.isfinite(
            PAIRWISE_ORDERING_AGREEMENT
        ),

    "No optimizer created":
        True,

    "No backward pass":
        True,

    "No model parameter updates":
        True,

    "No Fisher recalculation":
        True,

    "K0 and K1 hashes unchanged":
        R_integrity,

    "Feedback readiness evaluated":
        True,

    "Retention readiness evaluated":
        True,

    "Uncertainty readiness evaluated":
        True,

    "Latent compatibility readiness evaluated":
        True,

    "Integrity readiness evaluated":
        True,

    "Overall Cycle-2 readiness evaluated":
        isinstance(
            Ready_Cycle2,
            bool
        )
}


ALL_COMPLETION_CHECKS_PASS = all(
    bool(
        value
    )
    for value
    in completion_checks.values()
)


# ==============================================================================
# 48. QC SUMMARY
# ==============================================================================

qc_summary = {

    "phase":
        18,

    "created_at":
        datetime.now().isoformat(),

    "S1_count":
        len(
            S1_df
        ),

    "reference_count":
        len(
            D_fisher
        ),

    "MC_dropout_passes":
        MC_DROPOUT_PASSES,

    "Phase17_feedback_readiness":
        R_feedback,

    "knowledge_retention_readiness":
        R_retention,

    "uncertainty_readiness":
        R_uncertainty,

    "latent_compatibility_readiness":
        R_latent_compatibility,

    "integrity_readiness":
        R_integrity,

    "Ready_Cycle2":
        Ready_Cycle2,

    "reference_mean_prediction_drift":
        reference_mean_prediction_drift,

    "reference_severe_prediction_count":
        reference_severe_prediction_count,

    "reference_top_target_change_fraction":
        reference_top_target_change_fraction,

    "S1_mean_K0_uncertainty":
        float(
            S1_comparison_df[
                "K0_top5_uncertainty"
            ].mean()
        ),

    "S1_mean_K1_uncertainty":
        float(
            S1_comparison_df[
                "K1_top5_uncertainty"
            ].mean()
        ),

    "reference_mean_K0_uncertainty":
        float(
            reference_comparison_df[
                "K0_mean_uncertainty"
            ].mean()
        ),

    "reference_mean_K1_uncertainty":
        float(
            reference_comparison_df[
                "K1_mean_uncertainty"
            ].mean()
        ),

    "S1_large_uncertainty_increase_fraction":
        S1_large_uncertainty_increase_fraction,

    "reference_large_uncertainty_increase_fraction":
        reference_large_uncertainty_increase_fraction,

    "S1_uncertainty_collapse_fraction":
        S1_uncertainty_collapse_fraction,

    "reference_uncertainty_collapse_fraction":
        reference_uncertainty_collapse_fraction,

    "reference_mean_standardized_latent_drift":
        reference_mean_standardized_latent_drift,

    "reference_severe_latent_drift_fraction":
        reference_severe_latent_drift_fraction,

    "Spearman_K0_K1_utility":
        SPEARMAN_STABILITY,

    "Kendall_K0_K1_utility":
        KENDALL_STABILITY,

    "pairwise_ordering_agreement":
        PAIRWISE_ORDERING_AGREEMENT,

    "completion_checks":
        completion_checks
}


with open(
    PHASE18_QC_JSON_FILE,
    "w",
    encoding="utf-8"
) as handle:

    json.dump(
        qc_summary,
        handle,
        indent=4,
        ensure_ascii=False,
        default=str
    )


qc_rows = []


for key, value in qc_summary.items():

    if key == "completion_checks":

        for check_name, status in value.items():

            qc_rows.append(
                {

                    "metric":
                        f"completion.{check_name}",

                    "value":
                        status
                }
            )

    else:

        qc_rows.append(
            {

                "metric":
                    key,

                "value":
                    value
            }
        )


pd.DataFrame(
    qc_rows
).to_csv(
    PHASE18_QC_CSV_FILE,
    index=False
)


# ==============================================================================
# 49. COMPLETION VALIDATION
# ==============================================================================

print(
    "\n"
    +
    "=" * 155
)

print(
    "PHASE 18 COMPLETION VALIDATION"
)

print(
    "=" * 155
)


for check_name, status in completion_checks.items():

    print(
        f"[{'PASS' if status else 'FAIL':<4}] "
        f"{check_name}"
    )


# ==============================================================================
# 50. FINAL SUMMARY
# ==============================================================================

print(
    "\n"
    +
    "=" * 155
)

print(
    "PHASE 18 FINAL SUMMARY"
)

print(
    "=" * 155
)


print(
    f"S1 Candidates                              : "
    f"{len(S1_df)}"
)

print(
    f"Reference Molecules                        : "
    f"{len(D_fisher)}"
)

print(
    f"MC-Dropout Passes                          : "
    f"{MC_DROPOUT_PASSES}"
)


print(
    "\nPhase-17 State:"
)

print(
    f"  Final Adaptation Accepted                : "
    f"{R_feedback}"
)

print(
    f"  Selected λ_EWC                           : "
    f"{SELECTED_LAMBDA}"
)

print(
    f"  Selected Epoch                           : "
    f"{SELECTED_EPOCH}"
)


print(
    "\nS1 Uncertainty:"
)

print(
    f"  Mean K0 Top-5 Uncertainty                : "
    f"{S1_comparison_df['K0_top5_uncertainty'].mean():.8f}"
)

print(
    f"  Mean K1 Top-5 Uncertainty                : "
    f"{S1_comparison_df['K1_top5_uncertainty'].mean():.8f}"
)

print(
    f"  Mean Uncertainty Change                  : "
    f"{S1_comparison_df['uncertainty_change'].mean():+.8f}"
)

print(
    f"  Large Uncertainty Increase Fraction      : "
    f"{S1_large_uncertainty_increase_fraction:.4f}"
)

print(
    f"  Potential Collapse Fraction              : "
    f"{S1_uncertainty_collapse_fraction:.4f}"
)


print(
    "\nReference Uncertainty:"
)

print(
    f"  Mean K0 Uncertainty                      : "
    f"{reference_comparison_df['K0_mean_uncertainty'].mean():.8f}"
)

print(
    f"  Mean K1 Uncertainty                      : "
    f"{reference_comparison_df['K1_mean_uncertainty'].mean():.8f}"
)

print(
    f"  Mean Uncertainty Change                  : "
    f"{reference_comparison_df['uncertainty_change'].mean():+.8f}"
)

print(
    f"  Large Uncertainty Increase Fraction      : "
    f"{reference_large_uncertainty_increase_fraction:.4f}"
)

print(
    f"  Potential Collapse Fraction              : "
    f"{reference_uncertainty_collapse_fraction:.4f}"
)


print(
    "\nKnowledge Retention:"
)

print(
    f"  Mean Reference Prediction Drift          : "
    f"{reference_mean_prediction_drift:.8f}"
)

print(
    f"  Severe Prediction Drift Count            : "
    f"{reference_severe_prediction_count}"
)

print(
    f"  Reference Top-Target Change Fraction     : "
    f"{reference_top_target_change_fraction:.4f}"
)


print(
    "\nLatent Compatibility:"
)

print(
    f"  Mean Standardized Latent RMS Drift       : "
    f"{reference_mean_standardized_latent_drift:.8f}"
)

print(
    f"  Severe Latent Drift Fraction             : "
    f"{reference_severe_latent_drift_fraction:.4f}"
)


print(
    "\nS1 Ranking Stability:"
)

print(
    f"  Spearman K0 vs K1 Utility                : "
    f"{SPEARMAN_STABILITY:.6f}"
)

print(
    f"  Kendall K0 vs K1 Utility                 : "
    f"{KENDALL_STABILITY:.6f}"
)

print(
    f"  Pairwise Ordering Agreement              : "
    f"{PAIRWISE_ORDERING_AGREEMENT:.6f}"
)


print(
    "\nReadiness Components:"
)

print(
    f"  R_feedback                               : "
    f"{R_feedback}"
)

print(
    f"  R_retention                              : "
    f"{R_retention}"
)

print(
    f"  R_uncertainty                            : "
    f"{R_uncertainty}"
)

print(
    f"  R_latent_compatibility                   : "
    f"{R_latent_compatibility}"
)

print(
    f"  R_integrity                              : "
    f"{R_integrity}"
)


print(
    "\n"
    +
    "-" * 155
)


print(
    f"READY FOR CYCLE-2 GENERATION               : "
    f"{Ready_Cycle2}"
)


print(
    "-" * 155
)


print(
    "\nAdaptive-Learning Status:"
)

print(
    "  Optimizer Created                       : False"
)

print(
    "  Backward Pass                           : False"
)

print(
    "  Parameter Updates                       : False"
)

print(
    "  EWC Applied                             : False"
)

print(
    "  Fisher Recalculated                     : False"
)


print(
    "\nModel Integrity:"
)

print(
    f"  K0/K1/Fisher Files Unchanged             : "
    f"{R_integrity}"
)


print(
    "\nSaved Files:"
)


for path in [

    PHASE18_CONFIG_FILE,

    PHASE18_S1_K0_MEAN_FILE,

    PHASE18_S1_K0_UNCERTAINTY_FILE,

    PHASE18_S1_K1_MEAN_FILE,

    PHASE18_S1_K1_UNCERTAINTY_FILE,

    PHASE18_REFERENCE_K0_MEAN_FILE,

    PHASE18_REFERENCE_K0_UNCERTAINTY_FILE,

    PHASE18_REFERENCE_K1_MEAN_FILE,

    PHASE18_REFERENCE_K1_UNCERTAINTY_FILE,

    PHASE18_S1_COMPARISON_FILE,

    PHASE18_REFERENCE_COMPARISON_FILE,

    PHASE18_UNCERTAINTY_OUTLIERS_FILE,

    PHASE18_DISTRIBUTION_SUMMARY_FILE,

    PHASE18_RANKING_STABILITY_FILE,

    PHASE18_LATENT_COMPATIBILITY_FILE,

    PHASE18_READINESS_FILE,

    PHASE18_QC_CSV_FILE,

    PHASE18_QC_JSON_FILE
]:

    print(
        f"  {path.name}"
    )


print(
    "\n"
    +
    "-" * 155
)


if (
    ALL_COMPLETION_CHECKS_PASS
    and
    Ready_Cycle2
):

    print(
        "[STATUS] PHASE 18 COMPLETED SUCCESSFULLY."
    )


    print(
        "[PASS] K0 and K1 were compared under paired "
        "30-pass MC-dropout inference."
    )


    print(
        "[PASS] Dropout was the intended stochastic "
        "predictor component."
    )


    print(
        "[PASS] S1 relevance and approximate epistemic "
        "uncertainty shifts were quantified."
    )


    print(
        "[PASS] Baseline-reference prediction, uncertainty, "
        "target and latent behavior were audited."
    )


    print(
        "[PASS] No optimization or parameter update occurred."
    )


    print(
        "[PASS] K0, K1 and Fisher artifacts remained unchanged."
    )


    print(
        "[PASS] Cycle-2 readiness criteria were satisfied."
    )


    print(
        "[READY] K1 may proceed to Phase 19 — "
        "Cycle-2 Molecular Generation."
    )


    print(
        "[IMPORTANT] MC-dropout uncertainty is an approximate "
        "model uncertainty measure, not experimental confidence."
    )


else:

    print(
        "[STATUS] PHASE 18 VALIDATION COMPLETED, "
        "BUT CYCLE-2 GENERATION IS NOT YET APPROVED."
    )


    if not R_feedback:

        print(
            "[FAIL] Feedback readiness failed."
        )


    if not R_retention:

        print(
            "[FAIL] Knowledge-retention readiness failed."
        )


    if not R_uncertainty:

        print(
            "[FAIL] Uncertainty readiness failed."
        )


    if not R_latent_compatibility:

        print(
            "[FAIL] Latent-compatibility readiness failed."
        )


    if not R_integrity:

        print(
            "[FAIL] Model-integrity readiness failed."
        )


    print(
        "[ACTION] Do NOT proceed to Phase 19 until the "
        "failed readiness component is reviewed."
    )


print(
    "=" * 155
)

PHASE 18 — POST-ADAPTATION UNCERTAINTY, KNOWLEDGE-RETENTION VALIDATION, AND CYCLE-2 GENERATION READINESS

[DEVICE] cpu
[CONFIG] MC Dropout Passes                 : 30
[CONFIG] Random Seed                       : 42
[CONFIG] Severe Prediction Drift Threshold : 0.5

[PASS] Phase-17 accepted adaptation verified.
[PASS] Accepted λ_EWC  : 10.0
[PASS] Accepted epoch  : 1
[PASS] K1 checkpoint metadata matches accepted Phase-17 state.
[PASS] Exact Phase-7 graph schema verified.
[PASS] Exact Phase-9 normalization loaded.
[PASS] K0 and K1 encoder/predictor states loaded.
[PASS] S1 loaded: 50 candidates.

PRECOMPUTING S1 GRAPHS
[GRAPH] 01/50 CYCLE01_CAND000064 PASS
[GRAPH] 02/50 CYCLE01_CAND000720 PASS
[GRAPH] 03/50 CYCLE01_CAND000707 PASS
[GRAPH] 04/50 CYCLE01_CAND000212 PASS
[GRAPH] 05/50 CYCLE01_CAND000125 PASS
[GRAPH] 06/50 CYCLE01_CAND000753 PASS
[GRAPH] 07/50 CYCLE01_CAND000561 PASS
[GRAPH] 08/50 CYCLE01_CAND000197 PASS
[GRAPH] 09/50 CYCLE01_CAND000669 PASS
[GRAPH] 10/50 CYCLE01_CAND000670 

[17:31:25] UFFTYPER: Unrecognized atom type: S_5+4 (23)


[GRAPH] 49/50 CYCLE01_CAND000436 PASS
[GRAPH] 50/50 CYCLE01_CAND000710 PASS
[PASS] All S1 graphs precomputed.
[PASS] Fixed reference graphs loaded: 128

30-PASS MC-DROPOUT EVALUATION — S1
[S1] 01/50 CYCLE01_CAND000064 PASS
[S1] 02/50 CYCLE01_CAND000720 PASS
[S1] 03/50 CYCLE01_CAND000707 PASS
[S1] 04/50 CYCLE01_CAND000212 PASS
[S1] 05/50 CYCLE01_CAND000125 PASS
[S1] 06/50 CYCLE01_CAND000753 PASS
[S1] 07/50 CYCLE01_CAND000561 PASS
[S1] 08/50 CYCLE01_CAND000197 PASS
[S1] 09/50 CYCLE01_CAND000669 PASS
[S1] 10/50 CYCLE01_CAND000670 PASS
[S1] 11/50 CYCLE01_CAND000407 PASS
[S1] 12/50 CYCLE01_CAND000625 PASS
[S1] 13/50 CYCLE01_CAND000495 PASS
[S1] 14/50 CYCLE01_CAND000628 PASS
[S1] 15/50 CYCLE01_CAND000057 PASS
[S1] 16/50 CYCLE01_CAND000209 PASS
[S1] 17/50 CYCLE01_CAND000704 PASS
[S1] 18/50 CYCLE01_CAND000272 PASS
[S1] 19/50 CYCLE01_CAND000451 PASS
[S1] 20/50 CYCLE01_CAND000377 PASS
[S1] 21/50 CYCLE01_CAND000220 PASS
[S1] 22/50 CYCLE01_CAND000365 PASS
[S1] 23/50 CYCLE01_CAND000692 PASS
[S1] 24

# Phase 19 — Cycle-2 Multi-Objective Latent Search & Molecular Generation (Module 104 & 108)

**Objective:** Initiate Cycle-2 candidate generation from the validated adapted knowledge state ($\mathcal{K}_1$) by conducting a multi-objective sensitivity analysis, selecting 15 diverse high-utility seeds from $S_1$, and executing three-branch latent search (exploitation, interpolation, exploration) through the frozen generative decoder ($\theta_D^*$).

* **Multi-Objective Sensitivity & Diversity Seeding:**
  * **Sensitivity Scenarios:** Benchmarked ranking stability across $W_0$ (Primary: $0.30/0.20/0.50$), $W_1$ (Chemistry: $0.40/0.30/0.30$), $W_2$ (Biology: $0.20/0.15/0.65$), and $W_3$ (Balanced: $0.33/0.33/0.33$) via Spearman $\rho$, Kendall $\tau$, and Top-$K$ overlap.
  * **Seed Selection ($N = 15$):** Selected from the top-30 high-value $S_1$ pool using greedy MaxMin Tanimoto separation over 2048-bit Morgan fingerprints.

* **Three-Tier Latent Traversal ($N = 500$ Proposals / $1000$ Raw Decodings):**
  * **Branch 1 — Local Exploitation (60%, 600 outputs):** Gaussian perturbation ($\sigma \in \{0.10, 0.25\}$) around the 15 diverse seed latents.
  * **Branch 2 — Geodesic Interpolation (25%, 250 outputs):** Linear combinations ($z_\alpha = (1-\alpha)z_i + \alpha z_j$, $\alpha \in \{0.25, 0.50, 0.75\}$) between structurally diverse seed pairs.
  * **Branch 3 — Controlled Exploration (15%, 150 outputs):** Wide perturbation ($\sigma \in \{0.50, 0.75\}$) anchored on high-uncertainty seeds.

* **Decoding, Lineage & Non-Parametric Boundaries:**
  * **Decoding Engine:** 2 samples/latent via frozen $\theta_D^*$ ($T = 0.90$, $\text{Top-}K = 10$).
  * **Lineage Auditing:** Within-cycle deduplication ($C_2^{\text{raw}} \to C_2^{\text{unique}}$) with exact match logging against Cycle 1 ($C_1$) and baseline DrugBank structures.
  * **Model Invariance:** $\theta_E^{(1)}$, $\theta_P^{(1)}$, and $\theta_D^*$ verified bit-for-bit identical via pre/post SHA-256 hashes ($\Delta \theta = \mathbf{0}$).

* **Exported Deliverables:**
  * Sensitivity Audits: `phase19_utility_weight_scenarios.csv`, `phase19_weight_sensitivity_rank_correlations.csv`, `phase19_weight_sensitivity_topK_overlap.csv`.
  * Seed Manifests: `phase19_cycle2_seed_candidates.csv`, `phase19_seed_diversity_matrix.csv`.
  * Latent Proposals: `phase19_{exploitation,interpolation,exploration}_latent_manifest.csv`, `phase19_cycle2_latent_proposals.csv`.
  * Candidate Pools: `phase19_cycle2_raw_generation.csv`, `phase19_cycle2_unique_candidates.csv` ($C_2$).
  * Diagnostics: `phase19_cycle2_generation_statistics.csv`, `phase19_generation_qc_summary.json`.

In [23]:
# ==============================================================================
# PHASE 19 — CYCLE-2 MULTI-OBJECTIVE LATENT SEARCH AND
# MOLECULAR CANDIDATE GENERATION USING THE ACCEPTED K1 STATE
#
# COMPLETE CORRECTED IMPLEMENTATION
#
# IMPORTANT FIX
# -------------
# The autoregressive decoder now keeps:
#
# current_token : (1, 1)
# embedding     : (1, 1, 64)
# context       : (1, 1, 64)
# GRU input     : (1, 1, 128)
# hidden        : (2, 1, 128)
#
# The previous:
#
#     current_token = next_token.unsqueeze(0)
#
# has been removed because next_token is already (1, 1).
#
# ==============================================================================


# ==============================================================================
# 1. IMPORTS
# ==============================================================================

from pathlib import Path
from datetime import datetime

import hashlib
import itertools
import json
import math
import random
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from rdkit import Chem
from rdkit import RDLogger
from rdkit.Chem import AllChem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.DataStructs import TanimotoSimilarity


try:

    import selfies as sf

except Exception as exc:

    raise RuntimeError(
        "The 'selfies' package is required for Phase 19. "
        "Install it using: pip install selfies"
    ) from exc


warnings.filterwarnings(
    "ignore"
)

RDLogger.DisableLog(
    "rdApp.warning"
)

RDLogger.DisableLog(
    "rdApp.info"
)


# ==============================================================================
# 2. REPRODUCIBILITY
# ==============================================================================

RANDOM_SEED = 42


random.seed(
    RANDOM_SEED
)


np.random.seed(
    RANDOM_SEED
)


torch.manual_seed(
    RANDOM_SEED
)


if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        RANDOM_SEED
    )


DEVICE = torch.device(
    "cpu"
)


# ==============================================================================
# 3. PHASE-19 CONFIGURATION
# ==============================================================================

CYCLE = 2


# ------------------------------------------------------------------------------
# Encoder
# ------------------------------------------------------------------------------

NODE_FEATURE_DIM = 21

EDGE_FEATURE_DIM = 14

HIDDEN_DIM = 64

LATENT_DIM = 32

NUM_EGNN_LAYERS = 2

COORDINATE_UPDATE_SCALE = 0.10


# ------------------------------------------------------------------------------
# Predictor
# ------------------------------------------------------------------------------

TARGET_COUNT = 130


# ------------------------------------------------------------------------------
# Decoder
#
# Exact architecture already established in the project.
# ------------------------------------------------------------------------------

VOCAB_SIZE = 58

TOKEN_EMBED_DIM = 64

DECODER_HIDDEN_DIM = 128

DECODER_GRU_LAYERS = 2


# ------------------------------------------------------------------------------
# Generation
# ------------------------------------------------------------------------------

DECODER_TEMPERATURE = 0.90

DECODER_TOP_K = 10

DECODER_SAMPLES_PER_LATENT = 2


# ------------------------------------------------------------------------------
# Utility-weight sensitivity analysis
# ------------------------------------------------------------------------------

UTILITY_WEIGHT_SCENARIOS = {

    "W0_PRIMARY": {

        "qed":
            0.30,

        "sa":
            0.20,

        "bio":
            0.50
    },

    "W1_CHEMISTRY_FAVORING": {

        "qed":
            0.40,

        "sa":
            0.30,

        "bio":
            0.30
    },

    "W2_BIOLOGY_FAVORING": {

        "qed":
            0.20,

        "sa":
            0.15,

        "bio":
            0.65
    },

    "W3_BALANCED": {

        "qed":
            1.0 / 3.0,

        "sa":
            1.0 / 3.0,

        "bio":
            1.0 / 3.0
    }
}


PRIMARY_WEIGHT_SCENARIO = (
    "W0_PRIMARY"
)


# ------------------------------------------------------------------------------
# Diversity-aware seed selection
# ------------------------------------------------------------------------------

HIGH_VALUE_POOL_SIZE = 30

FINAL_SEED_COUNT = 15


# ------------------------------------------------------------------------------
# Cycle-2 latent search
#
# 500 latent proposals
#
# × 2 stochastic decoder samples
#
# = 1000 raw decoder outputs
# ------------------------------------------------------------------------------

EXPLOITATION_PROPOSALS = 300

INTERPOLATION_PROPOSALS = 125

EXPLORATION_PROPOSALS = 75


EXPECTED_TOTAL_LATENT_PROPOSALS = (

    EXPLOITATION_PROPOSALS

    +

    INTERPOLATION_PROPOSALS

    +

    EXPLORATION_PROPOSALS
)


EXPECTED_RAW_DECODER_OUTPUTS = (

    EXPECTED_TOTAL_LATENT_PROPOSALS

    *

    DECODER_SAMPLES_PER_LATENT
)


# ------------------------------------------------------------------------------
# Latent perturbation
# ------------------------------------------------------------------------------

EXPLOITATION_SIGMAS = [

    0.10,

    0.25
]


EXPLORATION_SIGMAS = [

    0.50,

    0.75
]


INTERPOLATION_ALPHAS = [

    0.25,

    0.50,

    0.75
]


# ------------------------------------------------------------------------------
# Molecular diversity
# ------------------------------------------------------------------------------

MORGAN_RADIUS = 2

MORGAN_BITS = 2048


# ------------------------------------------------------------------------------
# Geometry
# ------------------------------------------------------------------------------

MAX_OPTIMIZATION_ITERS = 500


# ==============================================================================
# 4. PROJECT PATH
# ==============================================================================

if "PROCESSED_DIR" not in globals():

    raise RuntimeError(
        "PROCESSED_DIR is missing. "
        "Run the project initialization cell first."
    )


PROCESSED_DIR = Path(
    PROCESSED_DIR
)


# ==============================================================================
# 5. INPUT FILES
# ==============================================================================

PHASE18_READINESS_FILE = (
    PROCESSED_DIR
    /
    "phase18_cycle2_generation_readiness.json"
)


PHASE18_S1_COMPARISON_FILE = (
    PROCESSED_DIR
    /
    "phase18_S1_uncertainty_comparison.csv"
)


PHASE17_S1_FILE = (
    PROCESSED_DIR
    /
    "phase17_feedback_candidate_manifest.csv"
)


PHASE17_ENCODER_FILE = (
    PROCESSED_DIR
    /
    "phase17_cycle1_adapted_encoder.pt"
)


PHASE17_PREDICTOR_FILE = (
    PROCESSED_DIR
    /
    "phase17_cycle1_adapted_predictor.pt"
)


PHASE12_DECODER_FILE = (
    PROCESSED_DIR
    /
    "phase12_thetaD_baseline_decoder.pt"
)


PHASE11_VOCAB_FILE = (
    PROCESSED_DIR
    /
    "phase11_selfies_vocabulary.json"
)


PHASE9_NORMALIZATION_FILE = (
    PROCESSED_DIR
    /
    "phase9_feature_normalization.json"
)


PHASE7_FEATURE_SCHEMA_FILE = (
    PROCESSED_DIR
    /
    "phase7_graph_feature_schema.json"
)


PHASE14_CYCLE1_UNIQUE_FILE = (
    PROCESSED_DIR
    /
    "phase14_generated_candidates_unique.csv"
)


BASELINE_DRUGBANK_FILE = (
    PROCESSED_DIR
    /
    "D_chem_standardized.csv"
)


required_files = [

    PHASE18_READINESS_FILE,

    PHASE18_S1_COMPARISON_FILE,

    PHASE17_S1_FILE,

    PHASE17_ENCODER_FILE,

    PHASE17_PREDICTOR_FILE,

    PHASE12_DECODER_FILE,

    PHASE11_VOCAB_FILE,

    PHASE9_NORMALIZATION_FILE,

    PHASE7_FEATURE_SCHEMA_FILE,

    PHASE14_CYCLE1_UNIQUE_FILE,

    BASELINE_DRUGBANK_FILE
]


missing_files = [

    str(
        path
    )

    for path
    in required_files

    if not path.exists()
]


if missing_files:

    raise FileNotFoundError(
        "Required Phase-19 input files are missing:\n"
        +
        "\n".join(
            missing_files
        )
    )


# ==============================================================================
# 6. OUTPUT FILES
# ==============================================================================

PHASE19_CONFIG_FILE = (
    PROCESSED_DIR
    /
    "phase19_generation_config.json"
)


PHASE19_WEIGHT_SCENARIOS_FILE = (
    PROCESSED_DIR
    /
    "phase19_utility_weight_scenarios.csv"
)


PHASE19_SENSITIVITY_SCORES_FILE = (
    PROCESSED_DIR
    /
    "phase19_S1_weight_sensitivity_scores.csv"
)


PHASE19_RANK_CORRELATIONS_FILE = (
    PROCESSED_DIR
    /
    "phase19_weight_sensitivity_rank_correlations.csv"
)


PHASE19_TOPK_OVERLAP_FILE = (
    PROCESSED_DIR
    /
    "phase19_weight_sensitivity_topK_overlap.csv"
)


PHASE19_SEEDS_FILE = (
    PROCESSED_DIR
    /
    "phase19_cycle2_seed_candidates.csv"
)


PHASE19_SEED_DIVERSITY_FILE = (
    PROCESSED_DIR
    /
    "phase19_seed_diversity_matrix.csv"
)


PHASE19_SEED_SUMMARY_FILE = (
    PROCESSED_DIR
    /
    "phase19_seed_selection_summary.json"
)


PHASE19_EXPLOITATION_LATENT_FILE = (
    PROCESSED_DIR
    /
    "phase19_exploitation_latent_manifest.csv"
)


PHASE19_INTERPOLATION_LATENT_FILE = (
    PROCESSED_DIR
    /
    "phase19_interpolation_latent_manifest.csv"
)


PHASE19_EXPLORATION_LATENT_FILE = (
    PROCESSED_DIR
    /
    "phase19_exploration_latent_manifest.csv"
)


PHASE19_ALL_LATENT_FILE = (
    PROCESSED_DIR
    /
    "phase19_cycle2_latent_proposals.csv"
)


PHASE19_RAW_GENERATION_FILE = (
    PROCESSED_DIR
    /
    "phase19_cycle2_raw_generation.csv"
)


PHASE19_GENERATION_MANIFEST_FILE = (
    PROCESSED_DIR
    /
    "phase19_cycle2_generation_manifest.csv"
)


PHASE19_UNIQUE_CANDIDATES_FILE = (
    PROCESSED_DIR
    /
    "phase19_cycle2_unique_candidates.csv"
)


PHASE19_DUPLICATE_SUMMARY_FILE = (
    PROCESSED_DIR
    /
    "phase19_cycle2_duplicate_summary.csv"
)


PHASE19_GENERATION_STATS_FILE = (
    PROCESSED_DIR
    /
    "phase19_cycle2_generation_statistics.csv"
)


PHASE19_QC_CSV_FILE = (
    PROCESSED_DIR
    /
    "phase19_generation_qc_summary.csv"
)


PHASE19_QC_JSON_FILE = (
    PROCESSED_DIR
    /
    "phase19_generation_qc_summary.json"
)


# ==============================================================================
# 7. GENERAL HELPERS
# ==============================================================================

def sha256_file(
    path
):

    digest = hashlib.sha256()


    with open(
        path,
        "rb"
    ) as handle:

        while True:

            block = handle.read(
                1024
                *
                1024
            )


            if not block:

                break


            digest.update(
                block
            )


    return digest.hexdigest()


def extract_state_dict(
    checkpoint
):

    if isinstance(
        checkpoint,
        nn.Module
    ):

        return checkpoint.state_dict()


    if not isinstance(
        checkpoint,
        dict
    ):

        raise RuntimeError(
            "Unsupported checkpoint format."
        )


    for key in [

        "state_dict",

        "model_state_dict",

        "encoder_state_dict",

        "decoder_state_dict",

        "predictor_state_dict"
    ]:

        if key in checkpoint:

            value = checkpoint[
                key
            ]


            if isinstance(
                value,
                nn.Module
            ):

                return value.state_dict()


            if isinstance(
                value,
                dict
            ):

                return value


    tensor_items = {

        key:
            value

        for key, value
        in checkpoint.items()

        if torch.is_tensor(
            value
        )
    }


    if tensor_items:

        return tensor_items


    raise RuntimeError(
        "Could not recover state_dict."
    )


def strip_prefix_if_present(
    state_dict,
    prefix
):

    keys = list(
        state_dict.keys()
    )


    if (
        keys
        and
        all(
            key.startswith(
                prefix
            )
            for key in keys
        )
    ):

        return {

            key[
                len(
                    prefix
                ):
            ]:
                value

            for key, value
            in state_dict.items()
        }


    return state_dict


def normalize_state_dict_keys(
    state_dict
):

    result = dict(
        state_dict
    )


    for prefix in [

        "module.",

        "encoder.",

        "decoder.",

        "predictor.",

        "model."
    ]:

        result = strip_prefix_if_present(
            result,
            prefix
        )


    return result


def canonicalize_smiles_safe(
    smiles
):

    if smiles is None:

        return ""


    smiles = str(
        smiles
    ).strip()


    if not smiles:

        return ""


    try:

        mol = Chem.MolFromSmiles(
            smiles
        )


        if mol is None:

            return ""


        return Chem.MolToSmiles(
            mol,
            canonical=True,
            isomericSmiles=True
        )


    except Exception:

        return ""


# ==============================================================================
# 8. VERIFY PHASE-18 READINESS
# ==============================================================================

with open(
    PHASE18_READINESS_FILE,
    "r",
    encoding="utf-8"
) as handle:

    phase18_readiness = json.load(
        handle
    )


if not bool(
    phase18_readiness.get(
        "Ready_Cycle2",
        False
    )
):

    raise RuntimeError(
        "Phase 18 did not approve Cycle-2 generation."
    )


print(
    "=" * 155
)

print(
    "PHASE 19 — CYCLE-2 MULTI-OBJECTIVE LATENT SEARCH "
    "AND MOLECULAR CANDIDATE GENERATION"
)

print(
    "=" * 155
)

print(
    "\n[PASS] Phase-18 Ready_Cycle2 = True"
)


# ==============================================================================
# 9. CHECKPOINT HASHES BEFORE PHASE 19
# ==============================================================================

MODEL_HASHES_BEFORE = {

    "K1_encoder":
        sha256_file(
            PHASE17_ENCODER_FILE
        ),

    "K1_predictor":
        sha256_file(
            PHASE17_PREDICTOR_FILE
        ),

    "decoder":
        sha256_file(
            PHASE12_DECODER_FILE
        )
}


# ==============================================================================
# 10. LOAD EXACT PHASE-7 GRAPH SCHEMA
# ==============================================================================

with open(
    PHASE7_FEATURE_SCHEMA_FILE,
    "r",
    encoding="utf-8"
) as handle:

    phase7_schema = json.load(
        handle
    )


NODE_FEATURE_NAMES = list(
    phase7_schema[
        "node_feature_names"
    ]
)


EDGE_FEATURE_NAMES = list(
    phase7_schema[
        "edge_feature_names"
    ]
)


if len(
    NODE_FEATURE_NAMES
) != NODE_FEATURE_DIM:

    raise RuntimeError(
        "Phase-7 node dimension mismatch."
    )


if len(
    EDGE_FEATURE_NAMES
) != EDGE_FEATURE_DIM:

    raise RuntimeError(
        "Phase-7 edge dimension mismatch."
    )


# ==============================================================================
# 11. PHASE-7 VOCABULARIES
# ==============================================================================

HYBRIDIZATION_TYPES = [

    "UNSPECIFIED",

    "S",

    "SP",

    "SP2",

    "SP3",

    "SP3D",

    "SP3D2",

    "OTHER"
]


CHIRALITY_TYPES = [

    "CHI_UNSPECIFIED",

    "CHI_TETRAHEDRAL_CW",

    "CHI_TETRAHEDRAL_CCW",

    "CHI_OTHER"
]


BOND_TYPES = [

    "SINGLE",

    "DOUBLE",

    "TRIPLE",

    "AROMATIC",

    "OTHER"
]


BOND_STEREO_TYPES = [

    "STEREONONE",

    "STEREOANY",

    "STEREOZ",

    "STEREOE",

    "STEREOCIS",

    "STEREOTRANS",

    "OTHER"
]


EXPECTED_NODE_FEATURE_NAMES = [

    "atomic_number",

    "degree",

    "formal_charge",

    "aromatic",

    "in_ring",

    "hydrogen_neighbor_count",

    "atomic_mass",

    "implicit_valence",

    "explicit_valence"
]


EXPECTED_NODE_FEATURE_NAMES += [

    f"hybridization_{value}"

    for value
    in HYBRIDIZATION_TYPES
]


EXPECTED_NODE_FEATURE_NAMES += [

    f"chirality_{value}"

    for value
    in CHIRALITY_TYPES
]


EXPECTED_EDGE_FEATURE_NAMES = [

    f"bond_type_{value}"

    for value
    in BOND_TYPES
]


EXPECTED_EDGE_FEATURE_NAMES += [

    "is_conjugated",

    "is_in_ring"
]


EXPECTED_EDGE_FEATURE_NAMES += [

    f"stereo_{value}"

    for value
    in BOND_STEREO_TYPES
]


if NODE_FEATURE_NAMES != EXPECTED_NODE_FEATURE_NAMES:

    raise RuntimeError(
        "Saved Phase-7 node schema mismatch."
    )


if EDGE_FEATURE_NAMES != EXPECTED_EDGE_FEATURE_NAMES:

    raise RuntimeError(
        "Saved Phase-7 edge schema mismatch."
    )


print(
    "[PASS] Exact Phase-7 feature schema verified."
)


# ==============================================================================
# 12. GRAPH FEATURIZATION HELPERS
# ==============================================================================

def one_hot(
    value,
    categories
):

    output = [
        0.0
        for _
        in categories
    ]


    if value in categories:

        index = categories.index(
            value
        )

    else:

        index = (
            len(
                categories
            )
            -
            1
        )


    output[
        index
    ] = 1.0


    return output


def atom_feature_vector(
    atom
):

    hydrogen_neighbor_count = sum(

        1

        for neighbour
        in atom.GetNeighbors()

        if neighbour.GetAtomicNum()
        ==
        1
    )


    try:

        implicit_valence = float(
            atom.GetImplicitValence()
        )

    except Exception:

        implicit_valence = 0.0


    try:

        explicit_valence = float(
            atom.GetExplicitValence()
        )

    except Exception:

        explicit_valence = 0.0


    features = [

        float(
            atom.GetAtomicNum()
        ),

        float(
            atom.GetDegree()
        ),

        float(
            atom.GetFormalCharge()
        ),

        float(
            atom.GetIsAromatic()
        ),

        float(
            atom.IsInRing()
        ),

        float(
            hydrogen_neighbor_count
        ),

        float(
            atom.GetMass()
        ),

        implicit_valence,

        explicit_valence
    ]


    features.extend(
        one_hot(
            str(
                atom.GetHybridization()
            ),
            HYBRIDIZATION_TYPES
        )
    )


    features.extend(
        one_hot(
            str(
                atom.GetChiralTag()
            ),
            CHIRALITY_TYPES
        )
    )


    if len(
        features
    ) != NODE_FEATURE_DIM:

        raise RuntimeError(
            "Node feature dimension mismatch."
        )


    return features


def bond_feature_vector(
    bond
):

    features = []


    features.extend(
        one_hot(
            str(
                bond.GetBondType()
            ),
            BOND_TYPES
        )
    )


    features.extend(
        [

            float(
                bond.GetIsConjugated()
            ),

            float(
                bond.IsInRing()
            )
        ]
    )


    features.extend(
        one_hot(
            str(
                bond.GetStereo()
            ),
            BOND_STEREO_TYPES
        )
    )


    if len(
        features
    ) != EDGE_FEATURE_DIM:

        raise RuntimeError(
            "Edge feature dimension mismatch."
        )


    return features


def build_edges(
    mol
):

    source = []

    target = []

    attributes = []


    for bond in mol.GetBonds():

        i = bond.GetBeginAtomIdx()

        j = bond.GetEndAtomIdx()


        feature = bond_feature_vector(
            bond
        )


        source.extend(
            [
                i,
                j
            ]
        )


        target.extend(
            [
                j,
                i
            ]
        )


        attributes.extend(
            [
                feature,
                feature
            ]
        )


    if len(
        source
    ) == 0:

        edge_index = torch.empty(
            (
                2,
                0
            ),
            dtype=torch.long
        )


        edge_attr = torch.empty(
            (
                0,
                EDGE_FEATURE_DIM
            ),
            dtype=torch.float32
        )


    else:

        edge_index = torch.tensor(
            [
                source,
                target
            ],
            dtype=torch.long
        )


        edge_attr = torch.tensor(
            attributes,
            dtype=torch.float32
        )


    return (
        edge_index,
        edge_attr
    )


# ==============================================================================
# 13. EXACT S1 GRAPH CONSTRUCTION
# ==============================================================================

def build_graph(
    smiles
):

    mol = Chem.MolFromSmiles(
        smiles
    )


    if mol is None:

        raise RuntimeError(
            "RDKit parsing failed."
        )


    mol_h = Chem.AddHs(
        mol
    )


    if mol_h is None:

        raise RuntimeError(
            "Adding explicit hydrogens failed."
        )


    params = AllChem.ETKDGv3()

    params.randomSeed = RANDOM_SEED

    params.useRandomCoords = False

    params.enforceChirality = True

    params.clearConfs = True


    conformer_id = AllChem.EmbedMolecule(
        mol_h,
        params
    )


    if conformer_id < 0:

        mol_h.RemoveAllConformers()


        params = AllChem.ETKDGv3()

        params.randomSeed = RANDOM_SEED

        params.useRandomCoords = True

        params.enforceChirality = True

        params.clearConfs = True


        conformer_id = AllChem.EmbedMolecule(
            mol_h,
            params
        )


    if conformer_id < 0:

        raise RuntimeError(
            "3D conformer generation failed."
        )


    # --------------------------------------------------------------------------
    # MMFF94 first, then UFF
    # --------------------------------------------------------------------------

    optimized = False


    try:

        if AllChem.MMFFHasAllMoleculeParams(
            mol_h
        ):

            AllChem.MMFFOptimizeMolecule(
                mol_h,
                mmffVariant="MMFF94",
                confId=conformer_id,
                maxIters=
                    MAX_OPTIMIZATION_ITERS
            )


            optimized = True

    except Exception:

        optimized = False


    if not optimized:

        try:

            if AllChem.UFFHasAllMoleculeParams(
                mol_h
            ):

                AllChem.UFFOptimizeMolecule(
                    mol_h,
                    confId=conformer_id,
                    maxIters=
                        MAX_OPTIMIZATION_ITERS
                )

        except Exception:

            pass


    x = torch.tensor(
        [

            atom_feature_vector(
                atom
            )

            for atom
            in mol_h.GetAtoms()
        ],
        dtype=torch.float32
    )


    edge_index, edge_attr = build_edges(
        mol_h
    )


    coordinates = np.asarray(
        mol_h.GetConformer(
            conformer_id
        ).GetPositions(),
        dtype=np.float32
    )


    if not np.isfinite(
        coordinates
    ).all():

        raise RuntimeError(
            "Non-finite coordinates."
        )


    coordinates = (

        coordinates

        -

        coordinates.mean(
            axis=0,
            keepdims=True
        )
    )


    pos = torch.tensor(
        coordinates,
        dtype=torch.float32
    )


    if x.shape[
        1
    ] != NODE_FEATURE_DIM:

        raise RuntimeError(
            "Node dimension mismatch."
        )


    if edge_attr.shape[
        1
    ] != EDGE_FEATURE_DIM:

        raise RuntimeError(
            "Edge dimension mismatch."
        )


    if x.shape[
        0
    ] != pos.shape[
        0
    ]:

        raise RuntimeError(
            "Node-coordinate mismatch."
        )


    return {

        "x":
            x,

        "edge_index":
            edge_index,

        "edge_attr":
            edge_attr,

        "pos":
            pos
    }


# ==============================================================================
# 14. EXACT PHASE-9 NODE NORMALIZATION
# ==============================================================================

with open(
    PHASE9_NORMALIZATION_FILE,
    "r",
    encoding="utf-8"
) as handle:

    phase9_normalization = json.load(
        handle
    )


NORMALIZATION_INDICES = list(
    phase9_normalization[
        "standardized_feature_indices"
    ]
)


NORMALIZED_FEATURE_NAMES = list(
    phase9_normalization[
        "standardized_features"
    ]
)


EXPECTED_NORMALIZED_FEATURE_NAMES = [

    "atomic_number",

    "degree",

    "formal_charge",

    "hydrogen_neighbor_count",

    "atomic_mass",

    "implicit_valence",

    "explicit_valence"
]


if NORMALIZED_FEATURE_NAMES != (
    EXPECTED_NORMALIZED_FEATURE_NAMES
):

    raise RuntimeError(
        "Phase-9 normalized feature mismatch."
    )


training_feature_mean = torch.tensor(
    [

        float(
            phase9_normalization[
                "mean"
            ][
                name
            ]
        )

        for name
        in NORMALIZED_FEATURE_NAMES
    ],
    dtype=torch.float32
)


training_feature_std = torch.tensor(
    [

        float(
            phase9_normalization[
                "std"
            ][
                name
            ]
        )

        for name
        in NORMALIZED_FEATURE_NAMES
    ],
    dtype=torch.float32
)


def normalize_nodes(
    x
):

    output = x.clone().float()


    indices = torch.tensor(
        NORMALIZATION_INDICES,
        dtype=torch.long,
        device=output.device
    )


    output[
        :,
        indices
    ] = (

        output[
            :,
            indices
        ]

        -

        training_feature_mean.to(
            output.device
        )

    ) / training_feature_std.to(
        output.device
    )


    return output


print(
    "[PASS] Exact Phase-9 normalization loaded."
)


# ==============================================================================
# 15. SCATTER MEAN
# ==============================================================================

def scatter_mean(
    values,
    index,
    dim_size
):

    output = torch.zeros(
        (
            dim_size,
            values.shape[
                1
            ]
        ),
        dtype=values.dtype,
        device=values.device
    )


    counts = torch.zeros(
        (
            dim_size,
            1
        ),
        dtype=values.dtype,
        device=values.device
    )


    output.index_add_(
        0,
        index,
        values
    )


    counts.index_add_(
        0,
        index,
        torch.ones(
            (
                values.shape[
                    0
                ],
                1
            ),
            dtype=values.dtype,
            device=values.device
        )
    )


    return (

        output

        /

        counts.clamp(
            min=1.0
        )
    )


# ==============================================================================
# 16. EXACT K1 ENCODER ARCHITECTURE
# ==============================================================================

class EGNNLayer(
    nn.Module
):

    def __init__(
        self
    ):

        super().__init__()


        self.edge_mlp = nn.Sequential(

            nn.Linear(
                HIDDEN_DIM
                *
                2
                +
                1
                +
                EDGE_FEATURE_DIM,
                HIDDEN_DIM
            ),

            nn.SiLU(),

            nn.Linear(
                HIDDEN_DIM,
                HIDDEN_DIM
            ),

            nn.SiLU()
        )


        self.coordinate_mlp = nn.Sequential(

            nn.Linear(
                HIDDEN_DIM,
                HIDDEN_DIM
            ),

            nn.SiLU(),

            nn.Linear(
                HIDDEN_DIM,
                1
            ),

            nn.Tanh()
        )


        self.node_mlp = nn.Sequential(

            nn.Linear(
                HIDDEN_DIM
                *
                2,
                HIDDEN_DIM
            ),

            nn.SiLU(),

            nn.Linear(
                HIDDEN_DIM,
                HIDDEN_DIM
            )
        )


        self.node_norm = nn.LayerNorm(
            HIDDEN_DIM
        )


    def forward(
        self,
        h,
        pos,
        edge_index,
        edge_attr
    ):

        source = edge_index[
            0
        ]


        target = edge_index[
            1
        ]


        relative = (

            pos[
                source
            ]

            -

            pos[
                target
            ]
        )


        squared_distance = (
            relative
            *
            relative
        ).sum(
            dim=1,
            keepdim=True
        )


        edge_input = torch.cat(
            [

                h[
                    source
                ],

                h[
                    target
                ],

                squared_distance,

                edge_attr
            ],
            dim=1
        )


        messages = self.edge_mlp(
            edge_input
        )


        coordinate_weight = (

            self.coordinate_mlp(
                messages
            )

            *

            COORDINATE_UPDATE_SCALE
        )


        coordinate_messages = (

            relative

            *

            coordinate_weight
        )


        coordinate_aggregate = scatter_mean(
            coordinate_messages,
            source,
            h.shape[
                0
            ]
        )


        new_pos = (

            pos

            +

            coordinate_aggregate
        )


        aggregated_messages = scatter_mean(
            messages,
            source,
            h.shape[
                0
            ]
        )


        node_delta = self.node_mlp(
            torch.cat(
                [
                    h,
                    aggregated_messages
                ],
                dim=1
            )
        )


        new_h = self.node_norm(

            h

            +

            node_delta
        )


        return (
            new_h,
            new_pos
        )


class GeometryAwareVariationalEncoder(
    nn.Module
):

    def __init__(
        self
    ):

        super().__init__()


        self.node_embedding = nn.Sequential(

            nn.Linear(
                NODE_FEATURE_DIM,
                HIDDEN_DIM
            ),

            nn.SiLU(),

            nn.LayerNorm(
                HIDDEN_DIM
            )
        )


        self.egnn_layers = nn.ModuleList(
            [

                EGNNLayer()

                for _
                in range(
                    NUM_EGNN_LAYERS
                )
            ]
        )


        self.graph_mlp = nn.Sequential(

            nn.Linear(
                HIDDEN_DIM,
                HIDDEN_DIM
            ),

            nn.SiLU(),

            nn.LayerNorm(
                HIDDEN_DIM
            )
        )


        self.mu_head = nn.Linear(
            HIDDEN_DIM,
            LATENT_DIM
        )


        self.logvar_head = nn.Linear(
            HIDDEN_DIM,
            LATENT_DIM
        )


    def forward(
        self,
        x,
        pos,
        edge_index,
        edge_attr,
        batch,
        deterministic=True
    ):

        h = self.node_embedding(
            x
        )


        current_pos = pos


        for layer in self.egnn_layers:

            h, current_pos = layer(

                h,

                current_pos,

                edge_index,

                edge_attr
            )


        graph_embedding = scatter_mean(
            h,
            batch,
            int(
                batch.max().item()
            )
            +
            1
        )


        graph_embedding = self.graph_mlp(
            graph_embedding
        )


        mu = self.mu_head(
            graph_embedding
        )


        logvar = self.logvar_head(
            graph_embedding
        )


        if deterministic:

            z = mu

        else:

            safe_logvar = torch.clamp(
                logvar,
                min=-20.0,
                max=20.0
            )


            std = torch.exp(
                0.5
                *
                safe_logvar
            )


            z = (

                mu

                +

                std
                *
                torch.randn_like(
                    std
                )
            )


        return {

            "mu":
                mu,

            "logvar":
                logvar,

            "z":
                z
        }


# ==============================================================================
# 17. K1 PREDICTOR ARCHITECTURE
# ==============================================================================

class Phase12TargetPredictor(
    nn.Module
):

    def __init__(
        self
    ):

        super().__init__()


        self.network = nn.Sequential(

            nn.Linear(
                LATENT_DIM,
                64
            ),

            nn.LayerNorm(
                64
            ),

            nn.SiLU(),

            nn.Dropout(
                0.15
            ),

            nn.Linear(
                64,
                32
            ),

            nn.LayerNorm(
                32
            ),

            nn.SiLU(),

            nn.Dropout(
                0.15
            ),

            nn.Linear(
                32,
                TARGET_COUNT
            )
        )


    def forward(
        self,
        x
    ):

        return self.network(
            x
        )


# ==============================================================================
# 18. LOAD ACCEPTED K1 ENCODER + PREDICTOR
# ==============================================================================

K1_encoder = (
    GeometryAwareVariationalEncoder()
    .to(
        DEVICE
    )
)


K1_predictor = (
    Phase12TargetPredictor()
    .to(
        DEVICE
    )
)


K1_encoder_checkpoint = torch.load(
    PHASE17_ENCODER_FILE,
    map_location="cpu",
    weights_only=False
)


K1_predictor_checkpoint = torch.load(
    PHASE17_PREDICTOR_FILE,
    map_location="cpu",
    weights_only=False
)


K1_encoder.load_state_dict(
    normalize_state_dict_keys(
        extract_state_dict(
            K1_encoder_checkpoint
        )
    ),
    strict=True
)


K1_predictor.load_state_dict(
    normalize_state_dict_keys(
        extract_state_dict(
            K1_predictor_checkpoint
        )
    ),
    strict=True
)


K1_encoder.eval()

K1_predictor.eval()


ENCODER_PARAMETER_COUNT = sum(

    parameter.numel()

    for parameter
    in K1_encoder.parameters()
)


PREDICTOR_PARAMETER_COUNT = sum(

    parameter.numel()

    for parameter
    in K1_predictor.parameters()
)


if ENCODER_PARAMETER_COUNT != 70274:

    raise RuntimeError(
        f"K1 encoder parameter count mismatch: "
        f"{ENCODER_PARAMETER_COUNT}"
    )


if PREDICTOR_PARAMETER_COUNT != 8674:

    raise RuntimeError(
        f"K1 predictor parameter count mismatch: "
        f"{PREDICTOR_PARAMETER_COUNT}"
    )


for parameter in K1_encoder.parameters():

    parameter.requires_grad_(
        False
    )


for parameter in K1_predictor.parameters():

    parameter.requires_grad_(
        False
    )


print(
    f"[PASS] K1 encoder loaded: "
    f"{ENCODER_PARAMETER_COUNT:,} parameters."
)

print(
    f"[PASS] K1 predictor loaded: "
    f"{PREDICTOR_PARAMETER_COUNT:,} parameters."
)


# ==============================================================================
# 19. EXACT FROZEN SELFIES DECODER
# ==============================================================================

class LatentConditionedSELFIESDecoder(
    nn.Module
):

    def __init__(
        self
    ):

        super().__init__()


        self.token_embedding = nn.Embedding(
            VOCAB_SIZE,
            TOKEN_EMBED_DIM,
            padding_idx=0
        )


        self.latent_to_hidden = nn.Sequential(

            nn.Linear(
                LATENT_DIM,
                DECODER_HIDDEN_DIM
                *
                DECODER_GRU_LAYERS
            ),

            nn.Tanh()
        )


        self.latent_to_context = nn.Sequential(

            nn.Linear(
                LATENT_DIM,
                TOKEN_EMBED_DIM
            ),

            nn.SiLU(),

            nn.LayerNorm(
                TOKEN_EMBED_DIM
            )
        )


        self.gru = nn.GRU(

            input_size=
                TOKEN_EMBED_DIM
                *
                2,

            hidden_size=
                DECODER_HIDDEN_DIM,

            num_layers=
                DECODER_GRU_LAYERS,

            batch_first=True
        )


        self.output_projection = nn.Linear(
            DECODER_HIDDEN_DIM,
            VOCAB_SIZE
        )


# ==============================================================================
# 20. LOAD FROZEN DECODER
# ==============================================================================

decoder = (
    LatentConditionedSELFIESDecoder()
    .to(
        DEVICE
    )
)


decoder_checkpoint = torch.load(
    PHASE12_DECODER_FILE,
    map_location="cpu",
    weights_only=False
)


decoder.load_state_dict(
    normalize_state_dict_keys(
        extract_state_dict(
            decoder_checkpoint
        )
    ),
    strict=True
)


decoder.eval()


DECODER_PARAMETER_COUNT = sum(

    parameter.numel()

    for parameter
    in decoder.parameters()
)


if DECODER_PARAMETER_COUNT != 220026:

    raise RuntimeError(
        f"Decoder parameter-count mismatch: "
        f"{DECODER_PARAMETER_COUNT}"
    )


for parameter in decoder.parameters():

    parameter.requires_grad_(
        False
    )


print(
    f"[PASS] Frozen decoder loaded: "
    f"{DECODER_PARAMETER_COUNT:,} parameters."
)


# ==============================================================================
# 21. LOAD SELFIES VOCABULARY
# ==============================================================================

with open(
    PHASE11_VOCAB_FILE,
    "r",
    encoding="utf-8"
) as handle:

    vocabulary = json.load(
        handle
    )


token_to_id = {

    str(
        token
    ):
        int(
            index
        )

    for token, index
    in vocabulary[
        "token_to_id"
    ].items()
}


id_to_token = {

    int(
        index
    ):
        token

    for token, index
    in token_to_id.items()
}


PAD_ID = int(
    vocabulary[
        "pad_id"
    ]
)


BOS_ID = int(
    vocabulary[
        "bos_id"
    ]
)


EOS_ID = int(
    vocabulary[
        "eos_id"
    ]
)


UNK_ID = int(
    vocabulary[
        "unk_id"
    ]
)


MAX_SEQUENCE_LENGTH = int(
    vocabulary[
        "maximum_sequence_length"
    ]
)


if len(
    token_to_id
) != VOCAB_SIZE:

    raise RuntimeError(
        f"Vocabulary size mismatch: "
        f"{len(token_to_id)} != {VOCAB_SIZE}"
    )


print(
    "[PASS] SELFIES vocabulary loaded."
)

print(
    f"[PASS] Vocabulary size       : "
    f"{len(token_to_id)}"
)

print(
    f"[PASS] Maximum sequence len : "
    f"{MAX_SEQUENCE_LENGTH}"
)


# ==============================================================================
# 22. CORRECTED AUTOREGRESSIVE SELFIES DECODER
#
# CRITICAL SHAPE INVARIANTS
#
# current_token = (1,1)
#
# embedding =
#     (1,1,64)
#
# context =
#     (1,64)
#
# step_context =
#     (1,1,64)
#
# gru_input =
#     (1,1,128)
#
# hidden =
#     (2,1,128)
#
# next_token =
#     (1,1)
#
# ==============================================================================

def generate_selfies_tokens(
    latent_vector,
    seed
):

    # --------------------------------------------------------------------------
    # Latent
    # --------------------------------------------------------------------------

    latent = torch.as_tensor(
        latent_vector,
        dtype=torch.float32,
        device=DEVICE
    ).reshape(
        1,
        LATENT_DIM
    )


    if latent.shape != (
        1,
        LATENT_DIM
    ):

        raise RuntimeError(
            f"Unexpected latent shape: "
            f"{tuple(latent.shape)}"
        )


    if not torch.isfinite(
        latent
    ).all():

        raise RuntimeError(
            "Decoder received non-finite latent values."
        )


    # --------------------------------------------------------------------------
    # Reproducible stochastic sampling
    # --------------------------------------------------------------------------

    torch.manual_seed(
        int(
            seed
        )
    )


    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            int(
                seed
            )
        )


    # --------------------------------------------------------------------------
    # Hidden-state initialization
    #
    # latent:
    #
    #     (1,32)
    #
    # latent_to_hidden:
    #
    #     (1,256)
    #
    # reshape:
    #
    #     (1,2,128)
    #
    # permute:
    #
    #     (2,1,128)
    # --------------------------------------------------------------------------

    hidden_projection = decoder.latent_to_hidden(
        latent
    )


    expected_hidden_projection_shape = (

        1,

        DECODER_GRU_LAYERS
        *
        DECODER_HIDDEN_DIM
    )


    if hidden_projection.shape != (
        expected_hidden_projection_shape
    ):

        raise RuntimeError(
            "Unexpected decoder hidden projection shape: "
            f"{tuple(hidden_projection.shape)}; "
            f"expected "
            f"{expected_hidden_projection_shape}"
        )


    hidden = (

        hidden_projection

        .reshape(
            1,
            DECODER_GRU_LAYERS,
            DECODER_HIDDEN_DIM
        )

        .permute(
            1,
            0,
            2
        )

        .contiguous()
    )


    expected_hidden_shape = (

        DECODER_GRU_LAYERS,

        1,

        DECODER_HIDDEN_DIM
    )


    if hidden.shape != (
        expected_hidden_shape
    ):

        raise RuntimeError(
            "Unexpected decoder hidden shape: "
            f"{tuple(hidden.shape)}; "
            f"expected {expected_hidden_shape}"
        )


    # --------------------------------------------------------------------------
    # Latent context
    #
    # (1,32)
    #    ↓
    # (1,64)
    # --------------------------------------------------------------------------

    context = decoder.latent_to_context(
        latent
    )


    expected_context_shape = (

        1,

        TOKEN_EMBED_DIM
    )


    if context.shape != (
        expected_context_shape
    ):

        raise RuntimeError(
            "Unexpected decoder context shape: "
            f"{tuple(context.shape)}; "
            f"expected {expected_context_shape}"
        )


    # --------------------------------------------------------------------------
    # BOS token
    #
    # ALWAYS 2-D
    #
    # (1,1)
    # --------------------------------------------------------------------------

    current_token = torch.full(
        (
            1,
            1
        ),
        fill_value=BOS_ID,
        dtype=torch.long,
        device=DEVICE
    )


    generated_ids = []


    termination_reason = (
        "max_length"
    )


    # --------------------------------------------------------------------------
    # Autoregressive decoding
    # --------------------------------------------------------------------------

    with torch.no_grad():

        for decoding_step in range(
            MAX_SEQUENCE_LENGTH
        ):

            # ------------------------------------------------------------------
            # FORCE TOKEN TO 2-D
            # ------------------------------------------------------------------

            current_token = current_token.reshape(
                1,
                1
            )


            if current_token.shape != (
                1,
                1
            ):

                raise RuntimeError(
                    f"Token shape failure at step "
                    f"{decoding_step}: "
                    f"{tuple(current_token.shape)}"
                )


            # ------------------------------------------------------------------
            # Token embedding
            #
            # (1,1)
            #   ↓
            # (1,1,64)
            # ------------------------------------------------------------------

            embedding = decoder.token_embedding(
                current_token
            )


            expected_embedding_shape = (

                1,

                1,

                TOKEN_EMBED_DIM
            )


            if embedding.shape != (
                expected_embedding_shape
            ):

                raise RuntimeError(
                    f"Unexpected embedding shape at "
                    f"decoder step {decoding_step}: "
                    f"{tuple(embedding.shape)}; "
                    f"expected "
                    f"{expected_embedding_shape}"
                )


            # ------------------------------------------------------------------
            # Context
            #
            # (1,64)
            #
            # →
            #
            # (1,1,64)
            # ------------------------------------------------------------------

            step_context = context.reshape(
                1,
                1,
                TOKEN_EMBED_DIM
            )


            expected_step_context_shape = (

                1,

                1,

                TOKEN_EMBED_DIM
            )


            if step_context.shape != (
                expected_step_context_shape
            ):

                raise RuntimeError(
                    f"Unexpected decoder context shape "
                    f"at step {decoding_step}: "
                    f"{tuple(step_context.shape)}"
                )


            # ------------------------------------------------------------------
            # Concatenate
            #
            # embedding:
            #
            #     (1,1,64)
            #
            # context:
            #
            #     (1,1,64)
            #
            # GRU input:
            #
            #     (1,1,128)
            # ------------------------------------------------------------------

            gru_input = torch.cat(
                [
                    embedding,
                    step_context
                ],
                dim=-1
            )


            expected_gru_input_shape = (

                1,

                1,

                TOKEN_EMBED_DIM
                *
                2
            )


            if gru_input.shape != (
                expected_gru_input_shape
            ):

                raise RuntimeError(
                    f"Unexpected GRU input shape "
                    f"at decoder step {decoding_step}: "
                    f"{tuple(gru_input.shape)}; "
                    f"expected "
                    f"{expected_gru_input_shape}"
                )


            # ------------------------------------------------------------------
            # GRU
            # ------------------------------------------------------------------

            output, hidden = decoder.gru(
                gru_input,
                hidden
            )


            expected_output_shape = (

                1,

                1,

                DECODER_HIDDEN_DIM
            )


            if output.shape != (
                expected_output_shape
            ):

                raise RuntimeError(
                    f"Unexpected GRU output shape at "
                    f"step {decoding_step}: "
                    f"{tuple(output.shape)}"
                )


            if hidden.shape != (
                expected_hidden_shape
            ):

                raise RuntimeError(
                    f"Unexpected GRU hidden shape at "
                    f"step {decoding_step}: "
                    f"{tuple(hidden.shape)}"
                )


            # ------------------------------------------------------------------
            # Output projection
            #
            # (1,128)
            #
            # →
            #
            # (1,58)
            # ------------------------------------------------------------------

            logits = decoder.output_projection(
                output[
                    :,
                    -1,
                    :
                ]
            )


            expected_logits_shape = (

                1,

                VOCAB_SIZE
            )


            if logits.shape != (
                expected_logits_shape
            ):

                raise RuntimeError(
                    f"Unexpected logits shape at "
                    f"step {decoding_step}: "
                    f"{tuple(logits.shape)}"
                )


            # ------------------------------------------------------------------
            # Temperature
            # ------------------------------------------------------------------

            logits = (

                logits

                /

                float(
                    DECODER_TEMPERATURE
                )
            )


            # ------------------------------------------------------------------
            # Top-k stochastic sampling
            # ------------------------------------------------------------------

            top_k = min(
                int(
                    DECODER_TOP_K
                ),
                int(
                    logits.shape[
                        -1
                    ]
                )
            )


            top_values, top_indices = torch.topk(
                logits,
                k=top_k,
                dim=-1
            )


            probabilities = torch.softmax(
                top_values,
                dim=-1
            )


            sampled_position = torch.multinomial(
                probabilities,
                num_samples=1
            )


            # ------------------------------------------------------------------
            # IMPORTANT
            #
            # gather returns:
            #
            #     (1,1)
            #
            # Therefore:
            #
            # DO NOT DO:
            #
            #     next_token.unsqueeze(0)
            #
            # ------------------------------------------------------------------

            next_token = top_indices.gather(
                1,
                sampled_position
            )


            next_token = next_token.reshape(
                1,
                1
            )


            if next_token.shape != (
                1,
                1
            ):

                raise RuntimeError(
                    f"Unexpected next-token shape at "
                    f"step {decoding_step}: "
                    f"{tuple(next_token.shape)}"
                )


            token_id = int(
                next_token[
                    0,
                    0
                ].item()
            )


            # ------------------------------------------------------------------
            # EOS
            # ------------------------------------------------------------------

            if token_id == EOS_ID:

                termination_reason = (
                    "eos"
                )

                break


            # ------------------------------------------------------------------
            # PAD
            # ------------------------------------------------------------------

            if token_id == PAD_ID:

                termination_reason = (
                    "pad"
                )

                break


            # ------------------------------------------------------------------
            # Record generated token
            # ------------------------------------------------------------------

            generated_ids.append(
                token_id
            )


            # ------------------------------------------------------------------
            # CRITICAL FIX
            #
            # Keep token exactly:
            #
            #     (1,1)
            #
            # NOT:
            #
            #     next_token.unsqueeze(0)
            #
            # ------------------------------------------------------------------

            current_token = next_token.reshape(
                1,
                1
            )


    # --------------------------------------------------------------------------
    # Token IDs → SELFIES tokens
    # --------------------------------------------------------------------------

    tokens = []


    unknown_token_encountered = False


    for token_id in generated_ids:

        token = id_to_token.get(
            token_id
        )


        if token is None:

            unknown_token_encountered = True

            continue


        if token_id in {

            PAD_ID,

            BOS_ID,

            EOS_ID
        }:

            continue


        # Do not write special UNK token into a SELFIES string.
        if token_id == UNK_ID:

            unknown_token_encountered = True

            continue


        tokens.append(
            token
        )


    selfies_string = "".join(
        tokens
    )


    return {

        "token_ids":
            generated_ids,

        "tokens":
            tokens,

        "selfies":
            selfies_string,

        "termination_reason":
            termination_reason,

        "eos_reached":
            bool(
                termination_reason
                ==
                "eos"
            ),

        "unknown_token_encountered":
            unknown_token_encountered,

        "generated_token_count":
            len(
                generated_ids
            )
    }


# ==============================================================================
# 23. SELFIES → SMILES
# ==============================================================================

def decode_selfies_to_smiles(
    selfies_string
):

    result = {

        "selfies_decode_success":
            False,

        "raw_smiles":
            "",

        "rdkit_parse_success":
            False,

        "canonicalization_success":
            False,

        "canonical_smiles":
            "",

        "failure_reason":
            ""
    }


    if not selfies_string:

        result[
            "failure_reason"
        ] = "EMPTY_SELFIES"

        return result


    # --------------------------------------------------------------------------
    # SELFIES decoding
    # --------------------------------------------------------------------------

    try:

        raw_smiles = sf.decoder(
            selfies_string
        )


        result[
            "raw_smiles"
        ] = raw_smiles


        result[
            "selfies_decode_success"
        ] = True


    except Exception as exc:

        result[
            "failure_reason"
        ] = (

            "SELFIES_DECODE_FAILED:"

            +

            str(
                exc
            )
        )


        return result


    # --------------------------------------------------------------------------
    # RDKit parsing
    # --------------------------------------------------------------------------

    try:

        mol = Chem.MolFromSmiles(
            raw_smiles
        )


        if mol is None:

            result[
                "failure_reason"
            ] = "RDKIT_PARSE_FAILED"

            return result


        result[
            "rdkit_parse_success"
        ] = True


        canonical = Chem.MolToSmiles(
            mol,
            canonical=True,
            isomericSmiles=True
        )


        if not canonical:

            result[
                "failure_reason"
            ] = "CANONICALIZATION_FAILED"

            return result


        result[
            "canonical_smiles"
        ] = canonical


        result[
            "canonicalization_success"
        ] = True


        return result


    except Exception as exc:

        result[
            "failure_reason"
        ] = (

            "CHEMICAL_PROCESSING_FAILED:"

            +

            str(
                exc
            )
        )


        return result


# ==============================================================================
# 24. LOAD S1 + PHASE-18 K1 EVALUATION
# ==============================================================================

S1_manifest_df = pd.read_csv(
    PHASE17_S1_FILE,
    dtype={
        "candidate_id":
            str,

        "canonical_smiles":
            str
    }
)


phase18_s1_df = pd.read_csv(
    PHASE18_S1_COMPARISON_FILE,
    dtype={
        "candidate_id":
            str
    }
)


required_phase18_columns = [

    "candidate_id",

    "K1_biological_relevance",

    "K1_top5_uncertainty"
]


missing_phase18_columns = [

    column

    for column
    in required_phase18_columns

    if column not in phase18_s1_df.columns
]


if missing_phase18_columns:

    raise RuntimeError(
        "Phase-18 S1 comparison is missing columns: "
        f"{missing_phase18_columns}"
    )


S1_df = S1_manifest_df.merge(
    phase18_s1_df[
        required_phase18_columns
    ],
    on="candidate_id",
    how="inner",
    validate="one_to_one"
)


if len(
    S1_df
) != 50:

    raise RuntimeError(
        f"Expected 50 S1 candidates; "
        f"found {len(S1_df)}."
    )


for column in [

    "qed",

    "sa_utility",

    "K1_biological_relevance",

    "K1_top5_uncertainty"
]:

    S1_df[
        column
    ] = pd.to_numeric(
        S1_df[
            column
        ],
        errors="raise"
    )


if S1_df[
    "candidate_id"
].duplicated().any():

    raise RuntimeError(
        "Duplicate candidate IDs in S1."
    )


print(
    f"[PASS] S1 K1 evaluation loaded: "
    f"{len(S1_df)} candidates."
)


# ==============================================================================
# 25. UTILITY-WEIGHT SCENARIOS
# ==============================================================================

weight_records = []


for scenario_name, weights in (
    UTILITY_WEIGHT_SCENARIOS.items()
):

    total_weight = (

        weights[
            "qed"
        ]

        +

        weights[
            "sa"
        ]

        +

        weights[
            "bio"
        ]
    )


    if not np.isclose(
        total_weight,
        1.0,
        atol=1e-12
    ):

        raise RuntimeError(
            f"Utility weights do not sum to 1 "
            f"for {scenario_name}."
        )


    if any(
        float(
            value
        ) < 0.0

        for value
        in weights.values()
    ):

        raise RuntimeError(
            f"Negative utility weight in {scenario_name}."
        )


    weight_records.append(
        {

            "scenario":
                scenario_name,

            "qed_weight":
                weights[
                    "qed"
                ],

            "sa_utility_weight":
                weights[
                    "sa"
                ],

            "biological_relevance_weight":
                weights[
                    "bio"
                ],

            "weight_sum":
                total_weight,

            "is_primary":
                bool(
                    scenario_name
                    ==
                    PRIMARY_WEIGHT_SCENARIO
                )
        }
    )


weight_scenarios_df = pd.DataFrame(
    weight_records
)


weight_scenarios_df.to_csv(
    PHASE19_WEIGHT_SCENARIOS_FILE,
    index=False
)


# ==============================================================================
# 26. S1 WEIGHT-SENSITIVITY SCORES
# ==============================================================================

sensitivity_df = S1_df[
    [

        "candidate_id",

        "canonical_smiles",

        "qed",

        "sa_utility",

        "K1_biological_relevance",

        "K1_top5_uncertainty"
    ]
].copy()


for scenario_name, weights in (
    UTILITY_WEIGHT_SCENARIOS.items()
):

    utility_column = (
        f"{scenario_name}_utility"
    )


    rank_column = (
        f"{scenario_name}_rank"
    )


    sensitivity_df[
        utility_column
    ] = (

        float(
            weights[
                "qed"
            ]
        )
        *
        sensitivity_df[
            "qed"
        ]

        +

        float(
            weights[
                "sa"
            ]
        )
        *
        sensitivity_df[
            "sa_utility"
        ]

        +

        float(
            weights[
                "bio"
            ]
        )
        *
        sensitivity_df[
            "K1_biological_relevance"
        ]
    )


    sensitivity_df[
        rank_column
    ] = (

        sensitivity_df[
            utility_column
        ]
        .rank(
            ascending=False,
            method="min"
        )
        .astype(
            int
        )
    )


sensitivity_df.to_csv(
    PHASE19_SENSITIVITY_SCORES_FILE,
    index=False
)


# ==============================================================================
# 27. RANK ROBUSTNESS
# ==============================================================================

def spearman_rank_correlation(
    rank_a,
    rank_b
):

    rank_a = np.asarray(
        rank_a,
        dtype=float
    )


    rank_b = np.asarray(
        rank_b,
        dtype=float
    )


    if (
        np.std(
            rank_a
        ) == 0.0
        or
        np.std(
            rank_b
        ) == 0.0
    ):

        return float(
            "nan"
        )


    return float(
        np.corrcoef(
            rank_a,
            rank_b
        )[
            0,
            1
        ]
    )


def kendall_tau_simple(
    rank_a,
    rank_b
):

    rank_a = np.asarray(
        rank_a,
        dtype=float
    )


    rank_b = np.asarray(
        rank_b,
        dtype=float
    )


    concordant = 0

    discordant = 0


    n = len(
        rank_a
    )


    for i in range(
        n
    ):

        for j in range(
            i + 1,
            n
        ):

            delta_a = (
                rank_a[
                    i
                ]
                -
                rank_a[
                    j
                ]
            )


            delta_b = (
                rank_b[
                    i
                ]
                -
                rank_b[
                    j
                ]
            )


            product = (

                delta_a

                *

                delta_b
            )


            if product > 0:

                concordant += 1


            elif product < 0:

                discordant += 1


    denominator = (

        concordant

        +

        discordant
    )


    if denominator == 0:

        return float(
            "nan"
        )


    return float(
        (
            concordant
            -
            discordant
        )
        /
        denominator
    )


primary_rank_column = (
    f"{PRIMARY_WEIGHT_SCENARIO}_rank"
)


rank_correlation_records = []

topk_records = []


for scenario_name in (
    UTILITY_WEIGHT_SCENARIOS.keys()
):

    if scenario_name == (
        PRIMARY_WEIGHT_SCENARIO
    ):

        continue


    alternative_rank_column = (
        f"{scenario_name}_rank"
    )


    primary_ranks = sensitivity_df[
        primary_rank_column
    ].to_numpy(
        dtype=float
    )


    alternative_ranks = sensitivity_df[
        alternative_rank_column
    ].to_numpy(
        dtype=float
    )


    rank_correlation_records.append(
        {

            "primary_scenario":
                PRIMARY_WEIGHT_SCENARIO,

            "comparison_scenario":
                scenario_name,

            "spearman":
                spearman_rank_correlation(
                    primary_ranks,
                    alternative_ranks
                ),

            "kendall":
                kendall_tau_simple(
                    primary_ranks,
                    alternative_ranks
                )
        }
    )


    for top_k in [

        10,

        20
    ]:

        primary_top = set(
            sensitivity_df.nsmallest(
                top_k,
                primary_rank_column
            )[
                "candidate_id"
            ].astype(
                str
            )
        )


        alternative_top = set(
            sensitivity_df.nsmallest(
                top_k,
                alternative_rank_column
            )[
                "candidate_id"
            ].astype(
                str
            )
        )


        overlap_count = len(

            primary_top

            &

            alternative_top
        )


        topk_records.append(
            {

                "primary_scenario":
                    PRIMARY_WEIGHT_SCENARIO,

                "comparison_scenario":
                    scenario_name,

                "top_k":
                    top_k,

                "overlap_count":
                    overlap_count,

                "overlap_fraction":
                    (
                        overlap_count
                        /
                        float(
                            top_k
                        )
                    )
            }
        )


rank_correlations_df = pd.DataFrame(
    rank_correlation_records
)


topk_overlap_df = pd.DataFrame(
    topk_records
)


rank_correlations_df.to_csv(
    PHASE19_RANK_CORRELATIONS_FILE,
    index=False
)


topk_overlap_df.to_csv(
    PHASE19_TOPK_OVERLAP_FILE,
    index=False
)


# ==============================================================================
# 28. MOLECULAR FINGERPRINTS
# ==============================================================================

fingerprint_generator = (
    rdFingerprintGenerator.GetMorganGenerator(
        radius=
            MORGAN_RADIUS,
        fpSize=
            MORGAN_BITS
    )
)


def fingerprint_from_smiles(
    smiles
):

    mol = Chem.MolFromSmiles(
        str(
            smiles
        )
    )


    if mol is None:

        raise RuntimeError(
            f"Could not generate fingerprint: "
            f"{smiles}"
        )


    return fingerprint_generator.GetFingerprint(
        mol
    )


# ==============================================================================
# 29. HIGH-VALUE S1 POOL
# ==============================================================================

PRIMARY_UTILITY_COLUMN = (
    f"{PRIMARY_WEIGHT_SCENARIO}_utility"
)


PRIMARY_RANK_COLUMN = (
    f"{PRIMARY_WEIGHT_SCENARIO}_rank"
)


high_value_pool_df = (

    sensitivity_df

    .sort_values(
        PRIMARY_UTILITY_COLUMN,
        ascending=False
    )

    .head(
        HIGH_VALUE_POOL_SIZE
    )

    .reset_index(
        drop=True
    )
)


pool_fingerprints = {

    str(
        row[
            "candidate_id"
        ]
    ):
        fingerprint_from_smiles(
            row[
                "canonical_smiles"
            ]
        )

    for _, row
    in high_value_pool_df.iterrows()
}


# ==============================================================================
# 30. DIVERSITY-AWARE SEED SELECTION
# ==============================================================================

first_seed_id = str(
    high_value_pool_df.iloc[
        0
    ][
        "candidate_id"
    ]
)


selected_seed_ids = [

    first_seed_id
]


while len(
    selected_seed_ids
) < FINAL_SEED_COUNT:

    best_candidate = None

    best_min_distance = (
        -float(
            "inf"
        )
    )

    best_utility = (
        -float(
            "inf"
        )
    )


    for _, row in (
        high_value_pool_df.iterrows()
    ):

        candidate_id = str(
            row[
                "candidate_id"
            ]
        )


        if candidate_id in (
            selected_seed_ids
        ):

            continue


        similarities = [

            TanimotoSimilarity(

                pool_fingerprints[
                    candidate_id
                ],

                pool_fingerprints[
                    selected_candidate
                ]
            )

            for selected_candidate
            in selected_seed_ids
        ]


        maximum_similarity = max(
            similarities
        )


        minimum_distance_to_selected = (

            1.0

            -

            maximum_similarity
        )


        utility = float(
            row[
                PRIMARY_UTILITY_COLUMN
            ]
        )


        if (
            minimum_distance_to_selected
            >
            best_min_distance
        ):

            best_candidate = (
                candidate_id
            )


            best_min_distance = (
                minimum_distance_to_selected
            )


            best_utility = (
                utility
            )


        elif np.isclose(
            minimum_distance_to_selected,
            best_min_distance
        ):

            if utility > best_utility:

                best_candidate = (
                    candidate_id
                )


                best_min_distance = (
                    minimum_distance_to_selected
                )


                best_utility = (
                    utility
                )


    if best_candidate is None:

        raise RuntimeError(
            "Diversity-aware seed selection failed."
        )


    selected_seed_ids.append(
        best_candidate
    )


seed_df = sensitivity_df[
    sensitivity_df[
        "candidate_id"
    ].astype(str).isin(
        selected_seed_ids
    )
].copy()


selection_order_lookup = {

    candidate_id:
        index + 1

    for index, candidate_id
    in enumerate(
        selected_seed_ids
    )
}


seed_df[
    "selection_order"
] = seed_df[
    "candidate_id"
].astype(str).map(
    selection_order_lookup
)


seed_df = (

    seed_df

    .sort_values(
        "selection_order"
    )

    .reset_index(
        drop=True
    )
)


# ==============================================================================
# 31. SEED DIVERSITY MATRIX
# ==============================================================================

seed_diversity_records = []


for candidate_i in (
    selected_seed_ids
):

    record = {

        "candidate_id":
            candidate_i
    }


    for candidate_j in (
        selected_seed_ids
    ):

        similarity = TanimotoSimilarity(

            pool_fingerprints[
                candidate_i
            ],

            pool_fingerprints[
                candidate_j
            ]
        )


        record[
            candidate_j
        ] = (

            1.0

            -

            similarity
        )


    seed_diversity_records.append(
        record
    )


seed_diversity_df = pd.DataFrame(
    seed_diversity_records
)


seed_diversity_df.to_csv(
    PHASE19_SEED_DIVERSITY_FILE,
    index=False
)


# ==============================================================================
# 32. ENCODE SEEDS WITH ACCEPTED K1 ENCODER
# ==============================================================================

def encode_smiles_K1(
    smiles
):

    graph = build_graph(
        smiles
    )


    x = normalize_nodes(
        graph[
            "x"
        ]
    ).to(
        DEVICE
    )


    pos = graph[
        "pos"
    ].float().to(
        DEVICE
    )


    edge_index = graph[
        "edge_index"
    ].long().to(
        DEVICE
    )


    edge_attr = graph[
        "edge_attr"
    ].float().to(
        DEVICE
    )


    batch = torch.zeros(
        x.shape[
            0
        ],
        dtype=torch.long,
        device=DEVICE
    )


    K1_encoder.eval()


    with torch.no_grad():

        output = K1_encoder(

            x=x,

            pos=pos,

            edge_index=edge_index,

            edge_attr=edge_attr,

            batch=batch,

            deterministic=True
        )


    mu = (

        output[
            "mu"
        ]

        .squeeze(
            0
        )

        .detach()

        .cpu()

        .numpy()

        .astype(
            np.float32
        )
    )


    if mu.shape != (
        LATENT_DIM,
    ):

        raise RuntimeError(
            f"Unexpected latent shape: "
            f"{mu.shape}"
        )


    if not np.isfinite(
        mu
    ).all():

        raise RuntimeError(
            "Non-finite K1 latent representation."
        )


    return mu


SEED_LATENTS = {}


print(
    "\n"
    +
    "=" * 155
)

print(
    "K1 SEED LATENT ENCODING"
)

print(
    "=" * 155
)


for index, (_, row) in enumerate(
    seed_df.iterrows(),
    start=1
):

    candidate_id = str(
        row[
            "candidate_id"
        ]
    )


    latent = encode_smiles_K1(
        row[
            "canonical_smiles"
        ]
    )


    SEED_LATENTS[
        candidate_id
    ] = latent


    print(
        f"[SEED] "
        f"{index:02d}/{len(seed_df):02d} "
        f"{candidate_id} PASS"
    )


seed_df[
    "latent_norm"
] = [

    float(
        np.linalg.norm(
            SEED_LATENTS[
                str(
                    candidate_id
                )
            ]
        )
    )

    for candidate_id
    in seed_df[
        "candidate_id"
    ]
]


seed_df.to_csv(
    PHASE19_SEEDS_FILE,
    index=False
)


seed_summary = {

    "high_value_pool_size":
        HIGH_VALUE_POOL_SIZE,

    "final_seed_count":
        FINAL_SEED_COUNT,

    "primary_weight_scenario":
        PRIMARY_WEIGHT_SCENARIO,

    "selection_policy":
        (
            "Highest-primary-utility initial seed followed by "
            "greedy maximum-minimum Morgan fingerprint diversity."
        ),

    "selected_seed_ids":
        selected_seed_ids
}


with open(
    PHASE19_SEED_SUMMARY_FILE,
    "w",
    encoding="utf-8"
) as handle:

    json.dump(
        seed_summary,
        handle,
        indent=4,
        ensure_ascii=False
    )


print(
    "\n[PASS] Diversity-aware Cycle-2 seeds selected."
)


# ==============================================================================
# 33. LATENT PROPOSAL REGISTRATION
# ==============================================================================

latent_records = []

latent_vectors = {}

latent_counter = 0


def register_latent(
    latent,
    generation_branch,
    **metadata
):

    global latent_counter


    latent = np.asarray(
        latent,
        dtype=np.float32
    ).reshape(
        -1
    )


    if latent.shape != (
        LATENT_DIM,
    ):

        raise RuntimeError(
            f"Latent proposal shape mismatch: "
            f"{latent.shape}"
        )


    if not np.isfinite(
        latent
    ).all():

        raise RuntimeError(
            "Latent proposal contains non-finite values."
        )


    latent_counter += 1


    latent_id = (
        f"CYCLE02_LATENT"
        f"{latent_counter:06d}"
    )


    latent_vectors[
        latent_id
    ] = latent.copy()


    record = {

        "latent_id":
            latent_id,

        "generation_branch":
            generation_branch,

        "latent_norm":
            float(
                np.linalg.norm(
                    latent
                )
            )
    }


    record.update(
        metadata
    )


    latent_records.append(
        record
    )


    return latent_id


# ==============================================================================
# 34. EXPLOITATION LATENT SEARCH
# ==============================================================================

rng = np.random.default_rng(
    RANDOM_SEED
)


seed_cycle = itertools.cycle(
    selected_seed_ids
)


for proposal_index in range(
    EXPLOITATION_PROPOSALS
):

    seed_id = next(
        seed_cycle
    )


    sigma = float(
        EXPLOITATION_SIGMAS[
            proposal_index
            %
            len(
                EXPLOITATION_SIGMAS
            )
        ]
    )


    seed_latent = SEED_LATENTS[
        seed_id
    ]


    noise = rng.normal(
        loc=0.0,
        scale=sigma,
        size=LATENT_DIM
    ).astype(
        np.float32
    )


    proposed_latent = (

        seed_latent

        +

        noise
    )


    seed_row = seed_df[
        seed_df[
            "candidate_id"
        ].astype(str)
        ==
        seed_id
    ].iloc[
        0
    ]


    register_latent(

        proposed_latent,

        generation_branch=
            "exploitation",

        seed_candidate_id=
            seed_id,

        seed_K1_utility=
            float(
                seed_row[
                    PRIMARY_UTILITY_COLUMN
                ]
            ),

        seed_K1_uncertainty=
            float(
                seed_row[
                    "K1_top5_uncertainty"
                ]
            ),

        perturbation_scale=
            sigma,

        interpolation_parent_1=
            "",

        interpolation_parent_2=
            "",

        interpolation_alpha=
            np.nan,

        parent_diversity=
            np.nan
    )


# ==============================================================================
# 35. INTERPOLATION LATENT SEARCH
# ==============================================================================

parent_pair_records = []


for parent_1, parent_2 in itertools.combinations(
    selected_seed_ids,
    2
):

    similarity = TanimotoSimilarity(

        pool_fingerprints[
            parent_1
        ],

        pool_fingerprints[
            parent_2
        ]
    )


    diversity = (

        1.0

        -

        similarity
    )


    parent_pair_records.append(
        (

            parent_1,

            parent_2,

            diversity
        )
    )


parent_pair_records = sorted(
    parent_pair_records,
    key=lambda item:
        item[
            2
        ],
    reverse=True
)


parent_pair_cycle = itertools.cycle(
    parent_pair_records
)


for proposal_index in range(
    INTERPOLATION_PROPOSALS
):

    (
        parent_1,
        parent_2,
        parent_diversity
    ) = next(
        parent_pair_cycle
    )


    alpha = float(
        INTERPOLATION_ALPHAS[
            proposal_index
            %
            len(
                INTERPOLATION_ALPHAS
            )
        ]
    )


    proposed_latent = (

        (
            1.0
            -
            alpha
        )

        *

        SEED_LATENTS[
            parent_1
        ]

        +

        alpha

        *

        SEED_LATENTS[
            parent_2
        ]
    )


    register_latent(

        proposed_latent,

        generation_branch=
            "interpolation",

        seed_candidate_id=
            "",

        seed_K1_utility=
            np.nan,

        seed_K1_uncertainty=
            np.nan,

        perturbation_scale=
            np.nan,

        interpolation_parent_1=
            parent_1,

        interpolation_parent_2=
            parent_2,

        interpolation_alpha=
            alpha,

        parent_diversity=
            float(
                parent_diversity
            )
    )


# ==============================================================================
# 36. EXPLORATION LATENT SEARCH
# ==============================================================================

exploration_seed_ids = (

    seed_df

    .sort_values(
        "K1_top5_uncertainty",
        ascending=False
    )

    [
        "candidate_id"
    ]

    .astype(
        str
    )

    .tolist()
)


exploration_seed_cycle = itertools.cycle(
    exploration_seed_ids
)


for proposal_index in range(
    EXPLORATION_PROPOSALS
):

    seed_id = next(
        exploration_seed_cycle
    )


    sigma = float(
        EXPLORATION_SIGMAS[
            proposal_index
            %
            len(
                EXPLORATION_SIGMAS
            )
        ]
    )


    noise = rng.normal(
        loc=0.0,
        scale=sigma,
        size=LATENT_DIM
    ).astype(
        np.float32
    )


    proposed_latent = (

        SEED_LATENTS[
            seed_id
        ]

        +

        noise
    )


    seed_row = seed_df[
        seed_df[
            "candidate_id"
        ].astype(str)
        ==
        seed_id
    ].iloc[
        0
    ]


    register_latent(

        proposed_latent,

        generation_branch=
            "exploration",

        seed_candidate_id=
            seed_id,

        seed_K1_utility=
            float(
                seed_row[
                    PRIMARY_UTILITY_COLUMN
                ]
            ),

        seed_K1_uncertainty=
            float(
                seed_row[
                    "K1_top5_uncertainty"
                ]
            ),

        perturbation_scale=
            sigma,

        interpolation_parent_1=
            "",

        interpolation_parent_2=
            "",

        interpolation_alpha=
            np.nan,

        parent_diversity=
            np.nan
    )


# ==============================================================================
# 37. VERIFY + SAVE LATENT PROPOSALS
# ==============================================================================

if len(
    latent_records
) != EXPECTED_TOTAL_LATENT_PROPOSALS:

    raise RuntimeError(
        "Unexpected total latent proposal count: "
        f"{len(latent_records)}"
    )


if len(
    latent_vectors
) != EXPECTED_TOTAL_LATENT_PROPOSALS:

    raise RuntimeError(
        "Latent-vector dictionary size mismatch."
    )


all_latent_df = pd.DataFrame(
    latent_records
)


all_latent_df.to_csv(
    PHASE19_ALL_LATENT_FILE,
    index=False
)


all_latent_df[
    all_latent_df[
        "generation_branch"
    ]
    ==
    "exploitation"
].to_csv(
    PHASE19_EXPLOITATION_LATENT_FILE,
    index=False
)


all_latent_df[
    all_latent_df[
        "generation_branch"
    ]
    ==
    "interpolation"
].to_csv(
    PHASE19_INTERPOLATION_LATENT_FILE,
    index=False
)


all_latent_df[
    all_latent_df[
        "generation_branch"
    ]
    ==
    "exploration"
].to_csv(
    PHASE19_EXPLORATION_LATENT_FILE,
    index=False
)


print(
    "\n"
    +
    "=" * 155
)

print(
    "CYCLE-2 LATENT SEARCH"
)

print(
    "=" * 155
)

print(
    f"Exploitation proposals  : "
    f"{EXPLOITATION_PROPOSALS}"
)

print(
    f"Interpolation proposals : "
    f"{INTERPOLATION_PROPOSALS}"
)

print(
    f"Exploration proposals   : "
    f"{EXPLORATION_PROPOSALS}"
)

print(
    f"Total latent proposals  : "
    f"{len(all_latent_df)}"
)


# ==============================================================================
# 38. LOAD CYCLE-1 AND DRUGBANK REFERENCE STRUCTURES
# ==============================================================================

cycle1_df = pd.read_csv(
    PHASE14_CYCLE1_UNIQUE_FILE,
    dtype=str
)


drugbank_df = pd.read_csv(
    BASELINE_DRUGBANK_FILE,
    dtype=str
)


def resolve_smiles_column(
    dataframe,
    candidate_columns
):

    for column in candidate_columns:

        if column in dataframe.columns:

            return column


    raise RuntimeError(
        "Could not locate a SMILES column. "
        f"Available columns: "
        f"{list(dataframe.columns)}"
    )


cycle1_smiles_column = resolve_smiles_column(
    cycle1_df,
    [

        "canonical_smiles",

        "standardized_smiles",

        "smiles",

        "SMILES"
    ]
)


drugbank_smiles_column = resolve_smiles_column(
    drugbank_df,
    [

        "canonical_smiles",

        "standardized_smiles",

        "smiles",

        "SMILES"
    ]
)


# ------------------------------------------------------------------------------
# Recanonicalize for robust exact comparison.
# ------------------------------------------------------------------------------

CYCLE1_CANONICAL_SET = set()


for smiles in (
    cycle1_df[
        cycle1_smiles_column
    ]
    .dropna()
    .astype(str)
):

    canonical = canonicalize_smiles_safe(
        smiles
    )


    if canonical:

        CYCLE1_CANONICAL_SET.add(
            canonical
        )


DRUGBANK_CANONICAL_SET = set()


for smiles in (
    drugbank_df[
        drugbank_smiles_column
    ]
    .dropna()
    .astype(str)
):

    canonical = canonicalize_smiles_safe(
        smiles
    )


    if canonical:

        DRUGBANK_CANONICAL_SET.add(
            canonical
        )


print(
    f"\n[PASS] Cycle-1 exact-overlap reference : "
    f"{len(CYCLE1_CANONICAL_SET)} unique structures"
)

print(
    f"[PASS] DrugBank exact-overlap reference: "
    f"{len(DRUGBANK_CANONICAL_SET)} unique structures"
)


# ==============================================================================
# 39. CYCLE-2 DECODER GENERATION
# ==============================================================================

print(
    "\n"
    +
    "=" * 155
)

print(
    "CYCLE-2 STOCHASTIC DECODER GENERATION"
)

print(
    "=" * 155
)


raw_generation_records = []


generation_index = 0


for latent_position, latent_record in enumerate(
    latent_records,
    start=1
):

    latent_id = str(
        latent_record[
            "latent_id"
        ]
    )


    latent_vector = latent_vectors[
        latent_id
    ]


    for decoder_sample_index in range(
        DECODER_SAMPLES_PER_LATENT
    ):

        generation_index += 1


        decoder_seed = (

            RANDOM_SEED
            *
            100000

            +

            generation_index
        )


        try:

            generation = generate_selfies_tokens(
                latent_vector,
                decoder_seed
            )


            chemistry = decode_selfies_to_smiles(
                generation[
                    "selfies"
                ]
            )


            decoder_execution_success = (
                True
            )


            decoder_exception = ""


        except Exception as exc:

            generation = {

                "token_ids":
                    [],

                "tokens":
                    [],

                "selfies":
                    "",

                "termination_reason":
                    "decoder_exception",

                "eos_reached":
                    False,

                "unknown_token_encountered":
                    False,

                "generated_token_count":
                    0
            }


            chemistry = {

                "selfies_decode_success":
                    False,

                "raw_smiles":
                    "",

                "rdkit_parse_success":
                    False,

                "canonicalization_success":
                    False,

                "canonical_smiles":
                    "",

                "failure_reason":
                    "DECODER_EXCEPTION"
            }


            decoder_execution_success = (
                False
            )


            decoder_exception = str(
                exc
            )


        canonical_smiles = str(
            chemistry[
                "canonical_smiles"
            ]
        )


        exact_cycle1_overlap = bool(

            canonical_smiles

            and

            canonical_smiles
            in
            CYCLE1_CANONICAL_SET
        )


        exact_drugbank_overlap = bool(

            canonical_smiles

            and

            canonical_smiles
            in
            DRUGBANK_CANONICAL_SET
        )


        raw_generation_records.append(
            {

                "generation_index":
                    generation_index,

                "latent_id":
                    latent_id,

                "generation_branch":
                    latent_record[
                        "generation_branch"
                    ],

                "latent_norm":
                    latent_record[
                        "latent_norm"
                    ],

                "seed_candidate_id":
                    latent_record.get(
                        "seed_candidate_id",
                        ""
                    ),

                "seed_K1_utility":
                    latent_record.get(
                        "seed_K1_utility",
                        np.nan
                    ),

                "seed_K1_uncertainty":
                    latent_record.get(
                        "seed_K1_uncertainty",
                        np.nan
                    ),

                "perturbation_scale":
                    latent_record.get(
                        "perturbation_scale",
                        np.nan
                    ),

                "interpolation_parent_1":
                    latent_record.get(
                        "interpolation_parent_1",
                        ""
                    ),

                "interpolation_parent_2":
                    latent_record.get(
                        "interpolation_parent_2",
                        ""
                    ),

                "interpolation_alpha":
                    latent_record.get(
                        "interpolation_alpha",
                        np.nan
                    ),

                "parent_diversity":
                    latent_record.get(
                        "parent_diversity",
                        np.nan
                    ),

                "decoder_sample_index":
                    decoder_sample_index,

                "decoder_seed":
                    decoder_seed,

                "decoder_temperature":
                    DECODER_TEMPERATURE,

                "decoder_top_k":
                    DECODER_TOP_K,

                "decoder_execution_success":
                    decoder_execution_success,

                "decoder_exception":
                    decoder_exception,

                "termination_reason":
                    generation[
                        "termination_reason"
                    ],

                "eos_reached":
                    generation[
                        "eos_reached"
                    ],

                "unknown_token_encountered":
                    generation[
                        "unknown_token_encountered"
                    ],

                "generated_token_count":
                    generation[
                        "generated_token_count"
                    ],

                "generated_selfies":
                    generation[
                        "selfies"
                    ],

                "selfies_decode_success":
                    chemistry[
                        "selfies_decode_success"
                    ],

                "raw_smiles":
                    chemistry[
                        "raw_smiles"
                    ],

                "rdkit_parse_success":
                    chemistry[
                        "rdkit_parse_success"
                    ],

                "canonicalization_success":
                    chemistry[
                        "canonicalization_success"
                    ],

                "canonical_smiles":
                    canonical_smiles,

                "exact_cycle1_overlap":
                    exact_cycle1_overlap,

                "exact_drugbank_overlap":
                    exact_drugbank_overlap,

                "failure_reason":
                    chemistry[
                        "failure_reason"
                    ]
            }
        )


    if (
        latent_position % 25 == 0

        or

        latent_position
        ==
        len(
            latent_records
        )
    ):

        print(
            f"[GENERATION] "
            f"{latent_position:04d}/"
            f"{len(latent_records):04d} "
            f"latent proposals processed"
        )


raw_generation_df = pd.DataFrame(
    raw_generation_records
)


if len(
    raw_generation_df
) != EXPECTED_RAW_DECODER_OUTPUTS:

    raise RuntimeError(
        "Unexpected raw generation count: "
        f"{len(raw_generation_df)} "
        f"!= "
        f"{EXPECTED_RAW_DECODER_OUTPUTS}"
    )


raw_generation_df.to_csv(
    PHASE19_RAW_GENERATION_FILE,
    index=False
)


# ==============================================================================
# 40. BASIC PHASE-19 SUCCESS
# ==============================================================================

generation_manifest_df = (
    raw_generation_df
    .copy()
)


generation_manifest_df[
    "basic_phase19_success"
] = (

    generation_manifest_df[
        "decoder_execution_success"
    ]

    &

    generation_manifest_df[
        "selfies_decode_success"
    ]

    &

    generation_manifest_df[
        "rdkit_parse_success"
    ]

    &

    generation_manifest_df[
        "canonicalization_success"
    ]
)


generation_manifest_df.to_csv(
    PHASE19_GENERATION_MANIFEST_FILE,
    index=False
)


successful_df = generation_manifest_df[
    generation_manifest_df[
        "basic_phase19_success"
    ]
].copy()


# ==============================================================================
# 41. WITHIN-CYCLE DEDUPLICATION
# ==============================================================================

if len(
    successful_df
) > 0:

    successful_df[
        "within_cycle_duplicate"
    ] = successful_df[
        "canonical_smiles"
    ].duplicated(
        keep="first"
    )


else:

    successful_df[
        "within_cycle_duplicate"
    ] = pd.Series(
        dtype=bool
    )


duplicate_summary_df = successful_df[
    successful_df[
        "within_cycle_duplicate"
    ]
].copy()


duplicate_summary_df.to_csv(
    PHASE19_DUPLICATE_SUMMARY_FILE,
    index=False
)


unique_df = successful_df[
    ~successful_df[
        "within_cycle_duplicate"
    ]
].copy()


unique_df = unique_df.reset_index(
    drop=True
)


# ==============================================================================
# 42. ASSIGN CYCLE-2 CANDIDATE IDS
# ==============================================================================

unique_df[
    "candidate_id"
] = [

    f"CYCLE02_CAND{index:06d}"

    for index
    in range(
        1,
        len(
            unique_df
        )
        +
        1
    )
]


output_columns = [

    "candidate_id"
] + [

    column

    for column
    in unique_df.columns

    if column
    !=
    "candidate_id"
]


unique_df = unique_df[
    output_columns
]


unique_df.to_csv(
    PHASE19_UNIQUE_CANDIDATES_FILE,
    index=False
)


# ==============================================================================
# 43. GENERATION STATISTICS
# ==============================================================================

generation_statistics = []


for branch in [

    "exploitation",

    "interpolation",

    "exploration"
]:

    branch_raw = raw_generation_df[
        raw_generation_df[
            "generation_branch"
        ]
        ==
        branch
    ]


    branch_successful = successful_df[
        successful_df[
            "generation_branch"
        ]
        ==
        branch
    ]


    branch_unique = unique_df[
        unique_df[
            "generation_branch"
        ]
        ==
        branch
    ]


    attempted = len(
        branch_raw
    )


    decoder_success_count = int(
        branch_raw[
            "decoder_execution_success"
        ].sum()
    )


    selfies_decode_count = int(
        branch_raw[
            "selfies_decode_success"
        ].sum()
    )


    parse_count = int(
        branch_raw[
            "rdkit_parse_success"
        ].sum()
    )


    canonical_count = int(
        branch_raw[
            "canonicalization_success"
        ].sum()
    )


    unique_count = len(
        branch_unique
    )


    duplicate_count = int(
        branch_successful[
            "within_cycle_duplicate"
        ].sum()
    ) if len(
        branch_successful
    ) else 0


    cycle1_overlap_count = int(
        branch_unique[
            "exact_cycle1_overlap"
        ].sum()
    ) if unique_count else 0


    drugbank_overlap_count = int(
        branch_unique[
            "exact_drugbank_overlap"
        ].sum()
    ) if unique_count else 0


    generation_statistics.append(
        {

            "generation_branch":
                branch,

            "attempted":
                attempted,

            "decoder_execution_success":
                decoder_success_count,

            "selfies_decode_success":
                selfies_decode_count,

            "rdkit_parse_success":
                parse_count,

            "canonicalization_success":
                canonical_count,

            "unique_candidates":
                unique_count,

            "within_cycle_duplicates":
                duplicate_count,

            "exact_cycle1_overlap":
                cycle1_overlap_count,

            "exact_drugbank_overlap":
                drugbank_overlap_count,

            "decoder_execution_rate":
                (
                    decoder_success_count
                    /
                    attempted
                    if attempted
                    else
                    np.nan
                ),

            "selfies_decode_rate":
                (
                    selfies_decode_count
                    /
                    attempted
                    if attempted
                    else
                    np.nan
                ),

            "parse_success_rate":
                (
                    parse_count
                    /
                    attempted
                    if attempted
                    else
                    np.nan
                ),

            "unique_yield":
                (
                    unique_count
                    /
                    attempted
                    if attempted
                    else
                    np.nan
                ),

            "cycle1_overlap_rate_among_unique":
                (
                    cycle1_overlap_count
                    /
                    unique_count
                    if unique_count
                    else
                    np.nan
                ),

            "drugbank_overlap_rate_among_unique":
                (
                    drugbank_overlap_count
                    /
                    unique_count
                    if unique_count
                    else
                    np.nan
                )
        }
    )


# ------------------------------------------------------------------------------
# Overall statistics
# ------------------------------------------------------------------------------

attempted_total = len(
    raw_generation_df
)


unique_total = len(
    unique_df
)


generation_statistics.append(
    {

        "generation_branch":
            "OVERALL",

        "attempted":
            attempted_total,

        "decoder_execution_success":
            int(
                raw_generation_df[
                    "decoder_execution_success"
                ].sum()
            ),

        "selfies_decode_success":
            int(
                raw_generation_df[
                    "selfies_decode_success"
                ].sum()
            ),

        "rdkit_parse_success":
            int(
                raw_generation_df[
                    "rdkit_parse_success"
                ].sum()
            ),

        "canonicalization_success":
            int(
                raw_generation_df[
                    "canonicalization_success"
                ].sum()
            ),

        "unique_candidates":
            unique_total,

        "within_cycle_duplicates":
            int(
                successful_df[
                    "within_cycle_duplicate"
                ].sum()
            ) if len(
                successful_df
            ) else 0,

        "exact_cycle1_overlap":
            int(
                unique_df[
                    "exact_cycle1_overlap"
                ].sum()
            ) if unique_total else 0,

        "exact_drugbank_overlap":
            int(
                unique_df[
                    "exact_drugbank_overlap"
                ].sum()
            ) if unique_total else 0,

        "decoder_execution_rate":
            float(
                raw_generation_df[
                    "decoder_execution_success"
                ].mean()
            ),

        "selfies_decode_rate":
            float(
                raw_generation_df[
                    "selfies_decode_success"
                ].mean()
            ),

        "parse_success_rate":
            float(
                raw_generation_df[
                    "rdkit_parse_success"
                ].mean()
            ),

        "unique_yield":
            (
                unique_total
                /
                attempted_total
            ),

        "cycle1_overlap_rate_among_unique":
            (
                float(
                    unique_df[
                        "exact_cycle1_overlap"
                    ].mean()
                )
                if unique_total
                else
                np.nan
            ),

        "drugbank_overlap_rate_among_unique":
            (
                float(
                    unique_df[
                        "exact_drugbank_overlap"
                    ].mean()
                )
                if unique_total
                else
                np.nan
            )
    }
)


generation_stats_df = pd.DataFrame(
    generation_statistics
)


generation_stats_df.to_csv(
    PHASE19_GENERATION_STATS_FILE,
    index=False
)


# ==============================================================================
# 44. VERIFY CHECKPOINT INTEGRITY
# ==============================================================================

MODEL_HASHES_AFTER = {

    "K1_encoder":
        sha256_file(
            PHASE17_ENCODER_FILE
        ),

    "K1_predictor":
        sha256_file(
            PHASE17_PREDICTOR_FILE
        ),

    "decoder":
        sha256_file(
            PHASE12_DECODER_FILE
        )
}


MODEL_FILES_UNCHANGED = (

    MODEL_HASHES_BEFORE

    ==

    MODEL_HASHES_AFTER
)


if not MODEL_FILES_UNCHANGED:

    raise RuntimeError(
        "Model checkpoint integrity failure."
    )


# ==============================================================================
# 45. PHASE-19 CONFIGURATION ARTIFACT
# ==============================================================================

phase19_config = {

    "phase":
        (
            "Phase 19 — Cycle-2 Multi-Objective "
            "Latent Search and Molecular Candidate Generation "
            "Using the Accepted K1 State"
        ),

    "created_at":
        datetime.now().isoformat(),

    "cycle":
        CYCLE,

    "device":
        str(
            DEVICE
        ),

    "random_seed":
        RANDOM_SEED,

    "Phase18_Ready_Cycle2":
        True,

    "primary_weight_scenario":
        PRIMARY_WEIGHT_SCENARIO,

    "utility_weight_scenarios":
        UTILITY_WEIGHT_SCENARIOS,

    "high_value_pool_size":
        HIGH_VALUE_POOL_SIZE,

    "final_seed_count":
        FINAL_SEED_COUNT,

    "seed_selection_policy":
        (
            "Primary utility followed by greedy maximum-minimum "
            "Morgan-fingerprint diversity."
        ),

    "Morgan_radius":
        MORGAN_RADIUS,

    "Morgan_bits":
        MORGAN_BITS,

    "exploitation_proposals":
        EXPLOITATION_PROPOSALS,

    "interpolation_proposals":
        INTERPOLATION_PROPOSALS,

    "exploration_proposals":
        EXPLORATION_PROPOSALS,

    "total_latent_proposals":
        EXPECTED_TOTAL_LATENT_PROPOSALS,

    "decoder_samples_per_latent":
        DECODER_SAMPLES_PER_LATENT,

    "expected_raw_decoder_outputs":
        EXPECTED_RAW_DECODER_OUTPUTS,

    "exploitation_sigmas":
        EXPLOITATION_SIGMAS,

    "exploration_sigmas":
        EXPLORATION_SIGMAS,

    "interpolation_alphas":
        INTERPOLATION_ALPHAS,

    "decoder_temperature":
        DECODER_TEMPERATURE,

    "decoder_top_k":
        DECODER_TOP_K,

    "decoder_max_sequence_length":
        MAX_SEQUENCE_LENGTH,

    "decoder_shape_fix":
        (
            "current_token preserved as shape (1,1); "
            "embedding (1,1,64); context (1,1,64); "
            "GRU input (1,1,128); hidden (2,1,128)"
        ),

    "encoder_trainable":
        False,

    "predictor_trainable":
        False,

    "decoder_trainable":
        False,

    "optimizer_created":
        False,

    "backward_pass":
        False,

    "parameter_updates":
        False,

    "Fisher_recalculated":
        False,

    "full_chemical_eligibility_deferred_to_phase20":
        True
}


with open(
    PHASE19_CONFIG_FILE,
    "w",
    encoding="utf-8"
) as handle:

    json.dump(
        phase19_config,
        handle,
        indent=4,
        ensure_ascii=False,
        default=str
    )


# ==============================================================================
# 46. PHASE-19 COMPLETION CHECKS
# ==============================================================================

completion_checks = {

    "Phase-18 Ready_Cycle2 verified":
        True,

    "K1 encoder loaded":
        True,

    "K1 predictor loaded":
        True,

    "Frozen decoder loaded":
        True,

    "K1 encoder parameter count correct":
        ENCODER_PARAMETER_COUNT
        ==
        70274,

    "K1 predictor parameter count correct":
        PREDICTOR_PARAMETER_COUNT
        ==
        8674,

    "Decoder parameter count correct":
        DECODER_PARAMETER_COUNT
        ==
        220026,

    "SELFIES vocabulary size correct":
        len(
            token_to_id
        )
        ==
        VOCAB_SIZE,

    "Exact Phase-7 graph schema reused":
        True,

    "Exact Phase-9 normalization reused":
        True,

    "Primary utility W0 retained":
        PRIMARY_WEIGHT_SCENARIO
        ==
        "W0_PRIMARY",

    "All utility scenarios sum to one":
        all(

            np.isclose(
                sum(
                    scenario.values()
                ),
                1.0
            )

            for scenario
            in UTILITY_WEIGHT_SCENARIOS.values()
        ),

    "Sensitivity scores calculated":
        len(
            sensitivity_df
        )
        ==
        50,

    "Spearman sensitivity calculated":
        (
            len(
                rank_correlations_df
            )
            ==
            3
            and
            rank_correlations_df[
                "spearman"
            ].notna().all()
        ),

    "Kendall sensitivity calculated":
        (
            len(
                rank_correlations_df
            )
            ==
            3
            and
            rank_correlations_df[
                "kendall"
            ].notna().all()
        ),

    "Top-10 overlap calculated":
        (
            topk_overlap_df[
                "top_k"
            ]
            ==
            10
        ).sum()
        ==
        3,

    "Top-20 overlap calculated":
        (
            topk_overlap_df[
                "top_k"
            ]
            ==
            20
        ).sum()
        ==
        3,

    "High-value S1 pool defined":
        len(
            high_value_pool_df
        )
        ==
        HIGH_VALUE_POOL_SIZE,

    "Diversity-aware seeds selected":
        len(
            seed_df
        )
        ==
        FINAL_SEED_COUNT,

    "K1 deterministic seed latents calculated":
        len(
            SEED_LATENTS
        )
        ==
        FINAL_SEED_COUNT,

    "Exploitation proposals generated":
        int(
            (
                all_latent_df[
                    "generation_branch"
                ]
                ==
                "exploitation"
            ).sum()
        )
        ==
        EXPLOITATION_PROPOSALS,

    "Interpolation proposals generated":
        int(
            (
                all_latent_df[
                    "generation_branch"
                ]
                ==
                "interpolation"
            ).sum()
        )
        ==
        INTERPOLATION_PROPOSALS,

    "Exploration proposals generated":
        int(
            (
                all_latent_df[
                    "generation_branch"
                ]
                ==
                "exploration"
            ).sum()
        )
        ==
        EXPLORATION_PROPOSALS,

    "All latent proposals finite":
        all(

            np.isfinite(
                latent
            ).all()

            for latent
            in latent_vectors.values()
        ),

    "Expected raw generation count produced":
        len(
            raw_generation_df
        )
        ==
        EXPECTED_RAW_DECODER_OUTPUTS,

    "Decoder tensor-shape execution completed":
        bool(
            raw_generation_df[
                "decoder_execution_success"
            ].any()
        ),

    "SELFIES decoding audited":
        "selfies_decode_success"
        in
        raw_generation_df.columns,

    "RDKit parsing audited":
        "rdkit_parse_success"
        in
        raw_generation_df.columns,

    "Canonicalization audited":
        "canonicalization_success"
        in
        raw_generation_df.columns,

    "Within-cycle duplicates audited":
        "within_cycle_duplicate"
        in
        successful_df.columns,

    "Cycle-1 exact overlap audited":
        "exact_cycle1_overlap"
        in
        unique_df.columns,

    "DrugBank exact overlap audited":
        "exact_drugbank_overlap"
        in
        unique_df.columns,

    "Unique Cycle-2 IDs assigned":
        bool(
            len(
                unique_df
            )
            >
            0

            and

            unique_df[
                "candidate_id"
            ]
            .astype(str)
            .str.startswith(
                "CYCLE02_CAND"
            )
            .all()
        ),

    "Generation provenance saved":
        PHASE19_GENERATION_MANIFEST_FILE.exists(),

    "Generation statistics saved":
        PHASE19_GENERATION_STATS_FILE.exists(),

    "No optimizer created":
        True,

    "No backward pass":
        True,

    "No model parameter update":
        True,

    "No Fisher recalculation":
        True,

    "K1 encoder checkpoint unchanged":
        (
            MODEL_HASHES_BEFORE[
                "K1_encoder"
            ]
            ==
            MODEL_HASHES_AFTER[
                "K1_encoder"
            ]
        ),

    "K1 predictor checkpoint unchanged":
        (
            MODEL_HASHES_BEFORE[
                "K1_predictor"
            ]
            ==
            MODEL_HASHES_AFTER[
                "K1_predictor"
            ]
        ),

    "Decoder checkpoint unchanged":
        (
            MODEL_HASHES_BEFORE[
                "decoder"
            ]
            ==
            MODEL_HASHES_AFTER[
                "decoder"
            ]
        )
}


ALL_CHECKS_PASS = all(

    bool(
        value
    )

    for value
    in completion_checks.values()
)


# ==============================================================================
# 47. QC SUMMARY
# ==============================================================================

qc_summary = {

    "phase":
        19,

    "created_at":
        datetime.now().isoformat(),

    "S1_candidates":
        len(
            S1_df
        ),

    "high_value_pool":
        HIGH_VALUE_POOL_SIZE,

    "selected_cycle2_seeds":
        FINAL_SEED_COUNT,

    "latent_proposals":
        len(
            all_latent_df
        ),

    "raw_decoder_outputs":
        len(
            raw_generation_df
        ),

    "decoder_execution_success":
        int(
            raw_generation_df[
                "decoder_execution_success"
            ].sum()
        ),

    "selfies_decode_success":
        int(
            raw_generation_df[
                "selfies_decode_success"
            ].sum()
        ),

    "rdkit_parse_success":
        int(
            raw_generation_df[
                "rdkit_parse_success"
            ].sum()
        ),

    "canonicalization_success":
        int(
            raw_generation_df[
                "canonicalization_success"
            ].sum()
        ),

    "basic_successful_outputs":
        len(
            successful_df
        ),

    "unique_cycle2_candidates":
        len(
            unique_df
        ),

    "within_cycle_duplicates":
        (
            int(
                successful_df[
                    "within_cycle_duplicate"
                ].sum()
            )
            if len(
                successful_df
            )
            else
            0
        ),

    "exact_cycle1_overlap":
        (
            int(
                unique_df[
                    "exact_cycle1_overlap"
                ].sum()
            )
            if len(
                unique_df
            )
            else
            0
        ),

    "exact_drugbank_overlap":
        (
            int(
                unique_df[
                    "exact_drugbank_overlap"
                ].sum()
            )
            if len(
                unique_df
            )
            else
            0
        ),

    "decoder_execution_rate":
        float(
            raw_generation_df[
                "decoder_execution_success"
            ].mean()
        ),

    "selfies_decode_rate":
        float(
            raw_generation_df[
                "selfies_decode_success"
            ].mean()
        ),

    "parse_success_rate":
        float(
            raw_generation_df[
                "rdkit_parse_success"
            ].mean()
        ),

    "canonicalization_rate":
        float(
            raw_generation_df[
                "canonicalization_success"
            ].mean()
        ),

    "unique_yield":
        float(
            len(
                unique_df
            )
            /
            len(
                raw_generation_df
            )
        ),

    "cycle1_overlap_rate_among_unique":
        (
            float(
                unique_df[
                    "exact_cycle1_overlap"
                ].mean()
            )
            if len(
                unique_df
            )
            else
            np.nan
        ),

    "drugbank_overlap_rate_among_unique":
        (
            float(
                unique_df[
                    "exact_drugbank_overlap"
                ].mean()
            )
            if len(
                unique_df
            )
            else
            np.nan
        ),

    "model_files_unchanged":
        MODEL_FILES_UNCHANGED,

    "completion_checks":
        completion_checks
}


with open(
    PHASE19_QC_JSON_FILE,
    "w",
    encoding="utf-8"
) as handle:

    json.dump(
        qc_summary,
        handle,
        indent=4,
        ensure_ascii=False,
        default=str
    )


qc_records = []


for metric_name, metric_value in (
    qc_summary.items()
):

    if metric_name == (
        "completion_checks"
    ):

        for check_name, check_status in (
            metric_value.items()
        ):

            qc_records.append(
                {

                    "metric":
                        f"completion.{check_name}",

                    "value":
                        check_status
                }
            )


    else:

        qc_records.append(
            {

                "metric":
                    metric_name,

                "value":
                    metric_value
            }
        )


pd.DataFrame(
    qc_records
).to_csv(
    PHASE19_QC_CSV_FILE,
    index=False
)


# ==============================================================================
# 48. DISPLAY WEIGHT-SENSITIVITY RESULTS
# ==============================================================================

print(
    "\n"
    +
    "=" * 155
)

print(
    "UTILITY-WEIGHT SENSITIVITY"
)

print(
    "=" * 155
)


print(
    rank_correlations_df.to_string(
        index=False
    )
)


print(
    "\nTOP-K OVERLAP"
)


print(
    topk_overlap_df.to_string(
        index=False
    )
)


# ==============================================================================
# 49. DISPLAY GENERATION STATISTICS
# ==============================================================================

print(
    "\n"
    +
    "=" * 155
)

print(
    "CYCLE-2 GENERATION STATISTICS"
)

print(
    "=" * 155
)


print(
    generation_stats_df.to_string(
        index=False
    )
)


# ==============================================================================
# 50. PHASE-19 COMPLETION VALIDATION
# ==============================================================================

print(
    "\n"
    +
    "=" * 155
)

print(
    "PHASE 19 COMPLETION VALIDATION"
)

print(
    "=" * 155
)


for check_name, status in (
    completion_checks.items()
):

    print(
        f"[{'PASS' if bool(status) else 'FAIL':<4}] "
        f"{check_name}"
    )


# ==============================================================================
# 51. FINAL SUMMARY
# ==============================================================================

print(
    "\n"
    +
    "=" * 155
)

print(
    "PHASE 19 FINAL SUMMARY"
)

print(
    "=" * 155
)


print(
    f"Cycle                                     : "
    f"{CYCLE}"
)

print(
    f"S1 Candidates                             : "
    f"{len(S1_df)}"
)

print(
    f"High-Value S1 Pool                       : "
    f"{HIGH_VALUE_POOL_SIZE}"
)

print(
    f"Diversity-Aware Seeds                    : "
    f"{FINAL_SEED_COUNT}"
)


print(
    "\nPrimary Utility:"
)

print(
    "  U = 0.30 QED + "
    "0.20 SAUtility + "
    "0.50 BiologicalRelevance"
)


print(
    "\nUtility Sensitivity Scenarios:"
)

print(
    "  W0 Primary                             : "
    "(0.30, 0.20, 0.50)"
)

print(
    "  W1 Chemistry-Favoring                  : "
    "(0.40, 0.30, 0.30)"
)

print(
    "  W2 Biology-Favoring                    : "
    "(0.20, 0.15, 0.65)"
)

print(
    "  W3 Balanced                            : "
    "(1/3, 1/3, 1/3)"
)


print(
    "\nLatent Search:"
)

print(
    f"  Exploitation Proposals                 : "
    f"{EXPLOITATION_PROPOSALS}"
)

print(
    f"  Interpolation Proposals                : "
    f"{INTERPOLATION_PROPOSALS}"
)

print(
    f"  Exploration Proposals                  : "
    f"{EXPLORATION_PROPOSALS}"
)

print(
    f"  Total Latent Proposals                 : "
    f"{len(all_latent_df)}"
)


print(
    "\nFrozen Decoder:"
)

print(
    f"  Parameter Count                        : "
    f"{DECODER_PARAMETER_COUNT:,}"
)

print(
    f"  Samples per Latent                     : "
    f"{DECODER_SAMPLES_PER_LATENT}"
)

print(
    f"  Temperature                            : "
    f"{DECODER_TEMPERATURE}"
)

print(
    f"  Top-K                                  : "
    f"{DECODER_TOP_K}"
)

print(
    f"  Maximum Sequence Length                : "
    f"{MAX_SEQUENCE_LENGTH}"
)


print(
    "\nDecoder Shape Policy:"
)

print(
    "  current_token                          : (1, 1)"
)

print(
    "  token_embedding                        : (1, 1, 64)"
)

print(
    "  latent_context                         : (1, 1, 64)"
)

print(
    "  GRU input                              : (1, 1, 128)"
)

print(
    "  GRU hidden                             : (2, 1, 128)"
)

print(
    "  next_token                             : (1, 1)"
)


print(
    "\nGeneration Outcome:"
)

print(
    f"  Raw Decoder Outputs                    : "
    f"{len(raw_generation_df)}"
)

print(
    f"  Decoder Execution Success              : "
    f"{int(raw_generation_df['decoder_execution_success'].sum())}"
)

print(
    f"  SELFIES Decode Success                 : "
    f"{int(raw_generation_df['selfies_decode_success'].sum())}"
)

print(
    f"  RDKit Parse Success                    : "
    f"{int(raw_generation_df['rdkit_parse_success'].sum())}"
)

print(
    f"  Canonicalization Success               : "
    f"{int(raw_generation_df['canonicalization_success'].sum())}"
)

print(
    f"  Basic Successful Outputs               : "
    f"{len(successful_df)}"
)

print(
    f"  Within-Cycle Duplicates                : "
    f"{int(successful_df['within_cycle_duplicate'].sum()) if len(successful_df) else 0}"
)

print(
    f"  Unique Cycle-2 Candidates              : "
    f"{len(unique_df)}"
)


print(
    "\nExact Novelty Bookkeeping:"
)

print(
    f"  Exact Cycle-1 Overlap                  : "
    f"{int(unique_df['exact_cycle1_overlap'].sum()) if len(unique_df) else 0}"
)

print(
    f"  Exact DrugBank Overlap                 : "
    f"{int(unique_df['exact_drugbank_overlap'].sum()) if len(unique_df) else 0}"
)


print(
    "\nGeneration Rates:"
)

print(
    f"  Decoder Execution Rate                 : "
    f"{raw_generation_df['decoder_execution_success'].mean():.2%}"
)

print(
    f"  SELFIES Decode Rate                    : "
    f"{raw_generation_df['selfies_decode_success'].mean():.2%}"
)

print(
    f"  RDKit Parse Rate                       : "
    f"{raw_generation_df['rdkit_parse_success'].mean():.2%}"
)

print(
    f"  Canonicalization Rate                  : "
    f"{raw_generation_df['canonicalization_success'].mean():.2%}"
)

print(
    f"  Unique Yield                           : "
    f"{len(unique_df) / len(raw_generation_df):.2%}"
)


print(
    "\nModel-State Integrity:"
)

print(
    f"  K1 Encoder Unchanged                   : "
    f"{MODEL_HASHES_BEFORE['K1_encoder'] == MODEL_HASHES_AFTER['K1_encoder']}"
)

print(
    f"  K1 Predictor Unchanged                 : "
    f"{MODEL_HASHES_BEFORE['K1_predictor'] == MODEL_HASHES_AFTER['K1_predictor']}"
)

print(
    f"  Frozen Decoder Unchanged               : "
    f"{MODEL_HASHES_BEFORE['decoder'] == MODEL_HASHES_AFTER['decoder']}"
)

print(
    "  Optimizer Created                      : False"
)

print(
    "  Backward Pass                          : False"
)

print(
    "  Model Adaptation                       : False"
)

print(
    "  Fisher Recalculated                    : False"
)


print(
    "\nCycle-2 Candidate Pool:"
)

print(
    f"  C2_raw                                 : "
    f"{len(raw_generation_df)}"
)

print(
    f"  C2_parseable                           : "
    f"{len(successful_df)}"
)

print(
    f"  C2_unique                              : "
    f"{len(unique_df)}"
)


print(
    "\nSaved Files:"
)


for path in [

    PHASE19_CONFIG_FILE,

    PHASE19_WEIGHT_SCENARIOS_FILE,

    PHASE19_SENSITIVITY_SCORES_FILE,

    PHASE19_RANK_CORRELATIONS_FILE,

    PHASE19_TOPK_OVERLAP_FILE,

    PHASE19_SEEDS_FILE,

    PHASE19_SEED_DIVERSITY_FILE,

    PHASE19_SEED_SUMMARY_FILE,

    PHASE19_EXPLOITATION_LATENT_FILE,

    PHASE19_INTERPOLATION_LATENT_FILE,

    PHASE19_EXPLORATION_LATENT_FILE,

    PHASE19_ALL_LATENT_FILE,

    PHASE19_RAW_GENERATION_FILE,

    PHASE19_GENERATION_MANIFEST_FILE,

    PHASE19_UNIQUE_CANDIDATES_FILE,

    PHASE19_DUPLICATE_SUMMARY_FILE,

    PHASE19_GENERATION_STATS_FILE,

    PHASE19_QC_CSV_FILE,

    PHASE19_QC_JSON_FILE
]:

    print(
        f"  {path.name}"
    )


print(
    "\n"
    +
    "-" * 155
)


if ALL_CHECKS_PASS:

    print(
        "[STATUS] PHASE 19 COMPLETED SUCCESSFULLY."
    )


    print(
        "[PASS] Phase-18 Cycle-2 readiness was verified."
    )


    print(
        "[PASS] The primary Cycle-1 utility definition "
        "was retained for Cycle 2."
    )


    print(
        "[PASS] Alternative weight combinations were used "
        "only for sensitivity analysis."
    )


    print(
        "[PASS] Diversity-aware K1 latent seeds were selected."
    )


    print(
        "[PASS] Cycle-2 latent search combined exploitation, "
        "interpolation and controlled exploration."
    )


    print(
        "[PASS] The corrected autoregressive SELFIES decoder "
        "maintained valid tensor shapes throughout decoding."
    )


    print(
        "[PASS] The frozen decoder generated Cycle-2 "
        "molecular sequences."
    )


    print(
        "[PASS] Basic SELFIES decoding, RDKit parsing and "
        "canonicalization were completed."
    )


    print(
        "[PASS] Within-cycle duplicate, Cycle-1 overlap "
        "and DrugBank overlap were recorded."
    )


    print(
        "[PASS] K1 encoder, K1 predictor and decoder "
        "checkpoints remained unchanged."
    )


    print(
        "[IMPORTANT] Phase 19 performs Cycle-2 generation "
        "and basic parsing only."
    )


    print(
        "[IMPORTANT] Full Cycle-2 chemical eligibility, "
        "novelty and physicochemical screening belongs "
        "to Phase 20."
    )


    print(
        "[READY] Send me the complete Phase-19 FINAL SUMMARY "
        "and GENERATION STATISTICS before Phase 20."
    )


else:

    print(
        "[STATUS] PHASE 19 REQUIRES REVIEW."
    )


    print(
        "[ACTION] Do NOT proceed to Phase 20 until "
        "failed QC checks are resolved."
    )


print(
    "=" * 155
)

PHASE 19 — CYCLE-2 MULTI-OBJECTIVE LATENT SEARCH AND MOLECULAR CANDIDATE GENERATION

[PASS] Phase-18 Ready_Cycle2 = True
[PASS] Exact Phase-7 feature schema verified.
[PASS] Exact Phase-9 normalization loaded.
[PASS] K1 encoder loaded: 70,274 parameters.
[PASS] K1 predictor loaded: 8,674 parameters.
[PASS] Frozen decoder loaded: 220,026 parameters.
[PASS] SELFIES vocabulary loaded.
[PASS] Vocabulary size       : 58
[PASS] Maximum sequence len : 122
[PASS] S1 K1 evaluation loaded: 50 candidates.

K1 SEED LATENT ENCODING
[SEED] 01/15 CYCLE01_CAND000064 PASS
[SEED] 02/15 CYCLE01_CAND000228 PASS
[SEED] 03/15 CYCLE01_CAND000300 PASS
[SEED] 04/15 CYCLE01_CAND000220 PASS
[SEED] 05/15 CYCLE01_CAND000280 PASS
[SEED] 06/15 CYCLE01_CAND000753 PASS
[SEED] 07/15 CYCLE01_CAND000197 PASS
[SEED] 08/15 CYCLE01_CAND000057 PASS
[SEED] 09/15 CYCLE01_CAND000704 PASS
[SEED] 10/15 CYCLE01_CAND000722 PASS
[SEED] 11/15 CYCLE01_CAND000710 PASS
[SEED] 12/15 CYCLE01_CAND000495 PASS
[SEED] 13/15 CYCLE01_CAND000670

# Phase 20 — Cycle-2 Chemical Eligibility, Uniqueness & Novelty Screening (Module 106)

**Objective:** Execute chemical quality-control and multi-corpus novelty screening on the 969 unique Cycle-2 candidate proposals ($C_2^{\text{unique}}$) from Phase 19 to establish the verified evaluation cohort ($C_2^{\text{eligible}}$) for downstream active learning.

* **Eligibility Architecture (Hard Exclusion Gates):**
  * **Syntax & Valence Integrity:** RDKit reparsing, strict sanitization, property-cache updates, and explicit valence verification.
  * **Topological Consistency:** Single-entity fragment filter ($N_{\text{frags}} = 1$) to eliminate disconnected salts and counterions.
  * **Feasibility Boundary:** Heavy atom count constrained to $4 \le N_{\text{heavy}} \le 80$.
  * **Numerical Validity:** All computed descriptors must be strictly finite (no NaN / $\pm\infty$).

* **Physicochemical Characterization & Lineage:**
  * Continuous property profiling: MW, cLogP, TPSA, HBD, HBA, rotatable bonds, formal charge, ring counts, and $F_{\text{sp3}}$.
  * Drug-likeness ($\text{QED}$) and synthetic accessibility ($\text{SA}$) score calculation.
  * Exact novelty auditing relative to both Cycle 1 ($C_1$) and the baseline DrugBank reference set.

* **Primary Deliverables:**
  * Eligible Pool: `phase20_cycle2_eligible_candidates.csv` ($C_2^{\text{eligible}}$).
  * Exclusion Log: `phase20_cycle2_ineligible_candidates.csv` ($C_2^{\text{ineligible}}$).
  * Comprehensive Audit: `phase20_cycle2_eligibility_screening.csv`, `phase20_cycle2_novelty_summary.csv`, `phase20_cycle2_qc_summary.json`.

In [24]:
# ==============================================================================
# PHASE 20 — CYCLE-2 CHEMICAL ELIGIBILITY, UNIQUENESS, NOVELTY,
# AND PHYSICOCHEMICAL SCREENING
#
# Module 106
#
# INPUT:
#     phase19_cycle2_unique_candidates.csv
#
# OUTPUT:
#     C2_eligible
#     C2_ineligible
#     complete QC / novelty / descriptor audit
#
# IMPORTANT:
#     - No EGNN
#     - No predictor inference
#     - No MC Dropout
#     - No UCB
#     - No optimizer
#     - No backward pass
#     - No Fisher recalculation
#     - No parameter adaptation
#
# ==============================================================================


# ==============================================================================
# 1. IMPORTS
# ==============================================================================

from pathlib import Path
from datetime import datetime

import hashlib
import json
import math
import warnings
from collections import Counter

import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit import RDLogger
from rdkit.Chem import (
    Crippen,
    Descriptors,
    Lipinski,
    QED,
    rdMolDescriptors
)


# ==============================================================================
# 2. RDKit SA SCORE
# ==============================================================================

try:

    from rdkit.Contrib.SA_Score import sascorer

    SA_SCORER_AVAILABLE = True

    SA_SCORER_SOURCE = (
        "rdkit.Contrib.SA_Score.sascorer"
    )


except Exception as exc:

    SA_SCORER_AVAILABLE = False

    SA_SCORER_SOURCE = (
        "UNAVAILABLE"
    )

    raise RuntimeError(
        "RDKit Contrib SA_Score is required for Phase 20. "
        "The Cycle-1 methodology used "
        "rdkit.Contrib.SA_Score.sascorer."
    ) from exc


# ==============================================================================
# 3. ENVIRONMENT
# ==============================================================================

warnings.filterwarnings(
    "ignore"
)


RDLogger.DisableLog(
    "rdApp.warning"
)


RDLogger.DisableLog(
    "rdApp.info"
)


# ==============================================================================
# 4. PHASE-20 CONFIGURATION
# ==============================================================================

CYCLE = 2

EXPECTED_INPUT_COUNT = 969


HEAVY_ATOM_MIN = 4

HEAVY_ATOM_MAX = 80


# ------------------------------------------------------------------------------
# Novelty eligibility policy
#
# True:
#     exact Cycle-1 / DrugBank overlap excludes the candidate from C2_eligible.
#
# False:
#     novelty remains annotation-only.
#
# For the finalized methodology, True is recommended.
# ------------------------------------------------------------------------------

NOVELTY_REQUIRED_FOR_ELIGIBILITY = True


# ------------------------------------------------------------------------------
# Identity policy
#
# Phase-19 canonical SMILES remain authoritative.
#
# A textual recanonicalization mismatch is NOT automatically exclusion.
#
# Molecular identity is additionally checked using InChIKey where available.
# ------------------------------------------------------------------------------

PRESERVE_PHASE19_CANONICAL_SMILES = True


# ------------------------------------------------------------------------------
# Cycle-1 SAUtility transformation
#
# Verified from the accepted Cycle-1 values:
#
#     SAUtility = clip((10 - SA_score) / 9, 0, 1)
#
# Example:
#
#     SA = 4.274906
#
#     (10 - 4.274906) / 9
#     = 0.63612156
#
# matching Cycle-1:
#
#     0.636122
# ------------------------------------------------------------------------------

SA_MIN_THEORETICAL = 1.0

SA_MAX_THEORETICAL = 10.0


# ==============================================================================
# 5. PROJECT DIRECTORY
# ==============================================================================

if "PROCESSED_DIR" not in globals():

    raise RuntimeError(
        "PROCESSED_DIR is not defined. "
        "Run the project initialization cell first."
    )


PROCESSED_DIR = Path(
    PROCESSED_DIR
)


if not PROCESSED_DIR.exists():

    raise FileNotFoundError(
        f"PROCESSED_DIR does not exist: "
        f"{PROCESSED_DIR}"
    )


# ==============================================================================
# 6. INPUT FILES
# ==============================================================================

PHASE19_C2_UNIQUE_FILE = (
    PROCESSED_DIR
    /
    "phase19_cycle2_unique_candidates.csv"
)


# ------------------------------------------------------------------------------
# Cycle-1 novelty reference
# ------------------------------------------------------------------------------

PHASE14_C1_UNIQUE_FILE = (
    PROCESSED_DIR
    /
    "phase14_generated_candidates_unique.csv"
)


# ------------------------------------------------------------------------------
# DrugBank standardized reference
# ------------------------------------------------------------------------------

DRUGBANK_REFERENCE_FILE = (
    PROCESSED_DIR
    /
    "D_chem_standardized.csv"
)


# ------------------------------------------------------------------------------
# K0 model state
# ------------------------------------------------------------------------------

K0_ENCODER_FILE = (
    PROCESSED_DIR
    /
    "phase12_thetaE_baseline_encoder.pt"
)


K0_DECODER_FILE = (
    PROCESSED_DIR
    /
    "phase12_thetaD_baseline_decoder.pt"
)


K0_PREDICTOR_FILE = (
    PROCESSED_DIR
    /
    "phase12_best_baseline_predictor.pt"
)


# ------------------------------------------------------------------------------
# K1 adapted model state
# ------------------------------------------------------------------------------

K1_ENCODER_FILE = (
    PROCESSED_DIR
    /
    "phase17_cycle1_adapted_encoder.pt"
)


K1_PREDICTOR_FILE = (
    PROCESSED_DIR
    /
    "phase17_cycle1_adapted_predictor.pt"
)


# ------------------------------------------------------------------------------
# Frozen Fisher artifacts
# ------------------------------------------------------------------------------

ENCODER_FISHER_FILE = (
    PROCESSED_DIR
    /
    "phase13_encoder_fisher.pt"
)


PREDICTOR_FISHER_FILE = (
    PROCESSED_DIR
    /
    "phase13_predictor_fisher.pt"
)


PARAMETER_REFERENCE_FILE = (
    PROCESSED_DIR
    /
    "phase13_fisher_parameter_reference.pt"
)


required_files = [

    PHASE19_C2_UNIQUE_FILE,

    PHASE14_C1_UNIQUE_FILE,

    DRUGBANK_REFERENCE_FILE,

    K0_ENCODER_FILE,

    K0_DECODER_FILE,

    K0_PREDICTOR_FILE,

    K1_ENCODER_FILE,

    K1_PREDICTOR_FILE,

    ENCODER_FISHER_FILE,

    PREDICTOR_FISHER_FILE,

    PARAMETER_REFERENCE_FILE
]


missing_files = [

    str(
        path
    )

    for path
    in required_files

    if not path.exists()
]


if missing_files:

    raise FileNotFoundError(
        "Required Phase-20 files are missing:\n"
        +
        "\n".join(
            missing_files
        )
    )


# ==============================================================================
# 7. OUTPUT FILES
# ==============================================================================

PHASE20_CONFIG_FILE = (
    PROCESSED_DIR
    /
    "phase20_screening_config.json"
)


PHASE20_SCREENING_FILE = (
    PROCESSED_DIR
    /
    "phase20_cycle2_eligibility_screening.csv"
)


PHASE20_ELIGIBLE_FILE = (
    PROCESSED_DIR
    /
    "phase20_cycle2_eligible_candidates.csv"
)


PHASE20_INELIGIBLE_FILE = (
    PROCESSED_DIR
    /
    "phase20_cycle2_ineligible_candidates.csv"
)


PHASE20_DESCRIPTOR_FILE = (
    PROCESSED_DIR
    /
    "phase20_cycle2_physicochemical_descriptors.csv"
)


PHASE20_NOVELTY_SUMMARY_FILE = (
    PROCESSED_DIR
    /
    "phase20_cycle2_novelty_summary.csv"
)


PHASE20_DESCRIPTOR_SUMMARY_FILE = (
    PROCESSED_DIR
    /
    "phase20_cycle2_descriptor_summary.csv"
)


PHASE20_EXCLUSION_SUMMARY_FILE = (
    PROCESSED_DIR
    /
    "phase20_cycle2_exclusion_summary.csv"
)


PHASE20_QC_CSV_FILE = (
    PROCESSED_DIR
    /
    "phase20_cycle2_qc_summary.csv"
)


PHASE20_QC_JSON_FILE = (
    PROCESSED_DIR
    /
    "phase20_cycle2_qc_summary.json"
)


# ==============================================================================
# 8. GENERAL HELPERS
# ==============================================================================

def sha256_file(
    path
):

    digest = hashlib.sha256()


    with open(
        path,
        "rb"
    ) as handle:

        while True:

            block = handle.read(
                1024
                *
                1024
            )


            if not block:

                break


            digest.update(
                block
            )


    return digest.hexdigest()


def resolve_column(
    dataframe,
    candidates,
    label
):

    for column in candidates:

        if column in dataframe.columns:

            return column


    raise RuntimeError(
        f"Could not locate {label}. "
        f"Available columns: "
        f"{list(dataframe.columns)}"
    )


def is_finite_number(
    value
):

    try:

        return bool(
            np.isfinite(
                float(
                    value
                )
            )
        )


    except Exception:

        return False


# ==============================================================================
# 9. EXACT CYCLE-1 SAUTILITY TRANSFORMATION
# ==============================================================================

def sa_to_utility(
    sa_score
):

    sa_score = float(
        sa_score
    )


    utility = (

        SA_MAX_THEORETICAL

        -

        sa_score

    ) / (

        SA_MAX_THEORETICAL

        -

        SA_MIN_THEORETICAL
    )


    return float(
        np.clip(
            utility,
            0.0,
            1.0
        )
    )


# ------------------------------------------------------------------------------
# Self-test using known Cycle-1 accepted value.
# ------------------------------------------------------------------------------

SA_SELF_TEST_SCORE = (
    4.274906
)


SA_SELF_TEST_EXPECTED = (
    0.636122
)


SA_SELF_TEST_CALCULATED = (
    sa_to_utility(
        SA_SELF_TEST_SCORE
    )
)


if not np.isclose(
    SA_SELF_TEST_CALCULATED,
    SA_SELF_TEST_EXPECTED,
    atol=1e-6
):

    raise RuntimeError(
        "SAUtility transformation self-test failed. "
        f"Calculated={SA_SELF_TEST_CALCULATED}, "
        f"expected≈{SA_SELF_TEST_EXPECTED}"
    )


# ==============================================================================
# 10. CHECKPOINT HASHES — BEFORE PHASE 20
# ==============================================================================

FROZEN_ARTIFACTS = {

    "K0_encoder_thetaE_star":
        K0_ENCODER_FILE,

    "K0_decoder_thetaD_star":
        K0_DECODER_FILE,

    "K0_predictor_thetaP_star":
        K0_PREDICTOR_FILE,

    "K1_encoder_thetaE_1":
        K1_ENCODER_FILE,

    "K1_predictor_thetaP_1":
        K1_PREDICTOR_FILE,

    "F_E":
        ENCODER_FISHER_FILE,

    "F_P":
        PREDICTOR_FISHER_FILE,

    "Phase13_parameter_reference":
        PARAMETER_REFERENCE_FILE
}


HASHES_BEFORE = {

    name:
        sha256_file(
            path
        )

    for name, path
    in FROZEN_ARTIFACTS.items()
}


# ==============================================================================
# 11. START
# ==============================================================================

print(
    "=" * 160
)

print(
    "PHASE 20 — CYCLE-2 CHEMICAL ELIGIBILITY, UNIQUENESS, "
    "NOVELTY, AND PHYSICOCHEMICAL SCREENING"
)

print(
    "=" * 160
)


print(
    f"\n[CONFIG] Cycle                          : "
    f"{CYCLE}"
)

print(
    f"[CONFIG] Expected C2_unique             : "
    f"{EXPECTED_INPUT_COUNT}"
)

print(
    f"[CONFIG] Heavy-Atom Feasibility         : "
    f"{HEAVY_ATOM_MIN}-{HEAVY_ATOM_MAX}"
)

print(
    f"[CONFIG] Novelty Required for Eligibility: "
    f"{NOVELTY_REQUIRED_FOR_ELIGIBILITY}"
)

print(
    f"[CONFIG] SA Implementation              : "
    f"{SA_SCORER_SOURCE}"
)

print(
    "[CONFIG] SAUtility                      : "
    "clip((10 - SA) / 9, 0, 1)"
)


# ==============================================================================
# 12. LOAD PHASE-19 C2 UNIQUE POPULATION
# ==============================================================================

phase19_df = pd.read_csv(
    PHASE19_C2_UNIQUE_FILE,
    dtype={
        "candidate_id":
            str,

        "canonical_smiles":
            str
    }
)


required_phase19_columns = [

    "candidate_id",

    "canonical_smiles"
]


missing_phase19_columns = [

    column

    for column
    in required_phase19_columns

    if column not in phase19_df.columns
]


if missing_phase19_columns:

    raise RuntimeError(
        "Phase-19 unique candidate file is missing "
        f"required columns: {missing_phase19_columns}"
    )


if len(
    phase19_df
) != EXPECTED_INPUT_COUNT:

    raise RuntimeError(
        f"Expected {EXPECTED_INPUT_COUNT} Phase-19 "
        f"unique candidates, but found "
        f"{len(phase19_df)}."
    )


if phase19_df[
    "candidate_id"
].isna().any():

    raise RuntimeError(
        "Missing Phase-19 candidate IDs."
    )


if phase19_df[
    "candidate_id"
].duplicated().any():

    duplicates = (
        phase19_df.loc[
            phase19_df[
                "candidate_id"
            ].duplicated(
                keep=False
            ),
            "candidate_id"
        ]
        .tolist()
    )


    raise RuntimeError(
        "Duplicate Phase-19 candidate IDs detected:\n"
        f"{duplicates[:20]}"
    )


if phase19_df[
    "canonical_smiles"
].isna().any():

    raise RuntimeError(
        "Missing Phase-19 canonical SMILES values."
    )


PHASE19_ID_SET = set(
    phase19_df[
        "candidate_id"
    ].astype(
        str
    )
)


print(
    f"\n[PASS] Phase-19 C2_unique loaded: "
    f"{len(phase19_df)}"
)

print(
    "[PASS] Candidate IDs present and unique."
)


# ==============================================================================
# 13. LOAD CYCLE-1 NOVELTY REFERENCE
# ==============================================================================

cycle1_df = pd.read_csv(
    PHASE14_C1_UNIQUE_FILE,
    dtype=str
)


cycle1_smiles_column = resolve_column(
    cycle1_df,
    [

        "canonical_smiles",

        "standardized_smiles",

        "smiles",

        "SMILES"
    ],
    "Cycle-1 SMILES column"
)


# ==============================================================================
# 14. LOAD DRUGBANK NOVELTY REFERENCE
# ==============================================================================

drugbank_df = pd.read_csv(
    DRUGBANK_REFERENCE_FILE,
    dtype=str
)


drugbank_smiles_column = resolve_column(
    drugbank_df,
    [

        "canonical_smiles",

        "standardized_smiles",

        "smiles",

        "SMILES"
    ],
    "DrugBank SMILES column"
)


# ==============================================================================
# 15. STRICT MOLECULAR VALIDATION HELPER
# ==============================================================================

def strict_validate_smiles(
    smiles
):

    result = {

        "rdkit_parse_success":
            False,

        "sanitization_success":
            False,

        "property_cache_success":
            False,

        "valence_validation_success":
            False,

        "mol":
            None,

        "recanonicalized_smiles":
            "",

        "inchi_key":
            "",

        "failure_reasons":
            []
    }


    if smiles is None:

        result[
            "failure_reasons"
        ].append(
            "MISSING_CANONICAL_SMILES"
        )


        return result


    smiles = str(
        smiles
    ).strip()


    if not smiles:

        result[
            "failure_reasons"
        ].append(
            "MISSING_CANONICAL_SMILES"
        )


        return result


    # --------------------------------------------------------------------------
    # Parse without automatic sanitization first.
    # --------------------------------------------------------------------------

    try:

        mol = Chem.MolFromSmiles(
            smiles,
            sanitize=False
        )


    except Exception:

        mol = None


    if mol is None:

        result[
            "failure_reasons"
        ].append(
            "RDKIT_PARSE_FAILED"
        )


        return result


    result[
        "rdkit_parse_success"
    ] = True


    # --------------------------------------------------------------------------
    # Strict sanitization
    # --------------------------------------------------------------------------

    try:

        Chem.SanitizeMol(
            mol
        )


        result[
            "sanitization_success"
        ] = True


    except Exception:

        result[
            "failure_reasons"
        ].append(
            "SANITIZATION_FAILED"
        )


        return result


    # --------------------------------------------------------------------------
    # Strict property cache
    # --------------------------------------------------------------------------

    try:

        mol.UpdatePropertyCache(
            strict=True
        )


        result[
            "property_cache_success"
        ] = True


    except Exception:

        result[
            "failure_reasons"
        ].append(
            "PROPERTY_CACHE_FAILED"
        )


        return result


    # --------------------------------------------------------------------------
    # Explicit valence access as validation
    # --------------------------------------------------------------------------

    try:

        for atom in mol.GetAtoms():

            _ = atom.GetExplicitValence()

            _ = atom.GetImplicitValence()

            _ = atom.GetTotalValence()


        result[
            "valence_validation_success"
        ] = True


    except Exception:

        result[
            "failure_reasons"
        ].append(
            "VALENCE_VALIDATION_FAILED"
        )


        return result


    # --------------------------------------------------------------------------
    # Recanonicalization
    # --------------------------------------------------------------------------

    try:

        recanonicalized = Chem.MolToSmiles(
            mol,
            canonical=True,
            isomericSmiles=True
        )


    except Exception:

        recanonicalized = ""


    result[
        "recanonicalized_smiles"
    ] = recanonicalized


    # --------------------------------------------------------------------------
    # InChIKey for identity support
    # --------------------------------------------------------------------------

    try:

        inchi_key = Chem.MolToInchiKey(
            mol
        )


    except Exception:

        inchi_key = ""


    result[
        "inchi_key"
    ] = inchi_key


    result[
        "mol"
    ] = mol


    return result


# ==============================================================================
# 16. BUILD AUTHORITATIVE CYCLE-1 REFERENCE SET
# ==============================================================================

CYCLE1_CANONICAL_SET = set()

CYCLE1_INCHIKEY_SET = set()


for smiles in (
    cycle1_df[
        cycle1_smiles_column
    ]
    .dropna()
    .astype(
        str
    )
):

    validation = strict_validate_smiles(
        smiles
    )


    mol = validation[
        "mol"
    ]


    if mol is None:

        continue


    canonical = validation[
        "recanonicalized_smiles"
    ]


    inchi_key = validation[
        "inchi_key"
    ]


    if canonical:

        CYCLE1_CANONICAL_SET.add(
            canonical
        )


    if inchi_key:

        CYCLE1_INCHIKEY_SET.add(
            inchi_key
        )


# ==============================================================================
# 17. BUILD DRUGBANK REFERENCE SET
# ==============================================================================

DRUGBANK_CANONICAL_SET = set()

DRUGBANK_INCHIKEY_SET = set()


for smiles in (
    drugbank_df[
        drugbank_smiles_column
    ]
    .dropna()
    .astype(
        str
    )
):

    validation = strict_validate_smiles(
        smiles
    )


    mol = validation[
        "mol"
    ]


    if mol is None:

        continue


    canonical = validation[
        "recanonicalized_smiles"
    ]


    inchi_key = validation[
        "inchi_key"
    ]


    if canonical:

        DRUGBANK_CANONICAL_SET.add(
            canonical
        )


    if inchi_key:

        DRUGBANK_INCHIKEY_SET.add(
            inchi_key
        )


print(
    f"\n[PASS] Cycle-1 canonical reference : "
    f"{len(CYCLE1_CANONICAL_SET)}"
)

print(
    f"[PASS] DrugBank canonical reference: "
    f"{len(DRUGBANK_CANONICAL_SET)}"
)


# ==============================================================================
# 18. DESCRIPTOR CALCULATION
# ==============================================================================

def calculate_descriptors(
    mol
):

    descriptor_values = {

        "atom_count":
            float(
                mol.GetNumAtoms()
            ),

        "heavy_atom_count":
            float(
                mol.GetNumHeavyAtoms()
            ),

        "fragment_count":
            float(
                len(
                    Chem.GetMolFrags(
                        mol
                    )
                )
            ),

        "molecular_weight":
            float(
                Descriptors.MolWt(
                    mol
                )
            ),

        "clogp":
            float(
                Crippen.MolLogP(
                    mol
                )
            ),

        "tpsa":
            float(
                rdMolDescriptors.CalcTPSA(
                    mol
                )
            ),

        "hbd":
            float(
                Lipinski.NumHDonors(
                    mol
                )
            ),

        "hba":
            float(
                Lipinski.NumHAcceptors(
                    mol
                )
            ),

        "rotatable_bonds":
            float(
                Lipinski.NumRotatableBonds(
                    mol
                )
            ),

        "formal_charge":
            float(
                Chem.GetFormalCharge(
                    mol
                )
            ),

        "ring_count":
            float(
                rdMolDescriptors.CalcNumRings(
                    mol
                )
            ),

        "aromatic_ring_count":
            float(
                rdMolDescriptors.CalcNumAromaticRings(
                    mol
                )
            ),

        "fraction_csp3":
            float(
                rdMolDescriptors.CalcFractionCSP3(
                    mol
                )
            ),

        "qed":
            float(
                QED.qed(
                    mol
                )
            ),

        "sa_score":
            float(
                sascorer.calculateScore(
                    mol
                )
            )
    }


    descriptor_values[
        "sa_utility"
    ] = sa_to_utility(
        descriptor_values[
            "sa_score"
        ]
    )


    return descriptor_values


# ==============================================================================
# 19. PHASE-20 SCREENING
# ==============================================================================

print(
    "\n"
    +
    "=" * 160
)

print(
    "PHASE-20 CYCLE-2 CHEMICAL SCREENING"
)

print(
    "=" * 160
)


screening_records = []


for row_index, row in (
    phase19_df.iterrows()
):

    candidate_id = str(
        row[
            "candidate_id"
        ]
    )


    phase19_smiles = str(
        row[
            "canonical_smiles"
        ]
    ).strip()


    validation = strict_validate_smiles(
        phase19_smiles
    )


    exclusion_reasons = list(
        validation[
            "failure_reasons"
        ]
    )


    recanonicalized_smiles = (
        validation[
            "recanonicalized_smiles"
        ]
    )


    inchi_key = validation[
        "inchi_key"
    ]


    mol = validation[
        "mol"
    ]


    # --------------------------------------------------------------------------
    # Identity checks
    # --------------------------------------------------------------------------

    exact_text_identity = bool(
        recanonicalized_smiles
        ==
        phase19_smiles
        and
        recanonicalized_smiles
        !=
        ""
    )


    identity_representation_mismatch = bool(

        recanonicalized_smiles

        and

        recanonicalized_smiles
        !=
        phase19_smiles
    )


    # Since recanonicalization is generated directly from the parsed Phase-19
    # structure, a text mismatch alone is not sufficient for exclusion.
    #
    # Identity is considered verifiable if:
    #
    # 1. exact canonical text matches
    #
    # OR
    #
    # 2. a valid molecular graph + InChIKey were recovered.
    # --------------------------------------------------------------------------

    identity_consistency = bool(

        exact_text_identity

        or

        (
            mol is not None
            and
            bool(
                inchi_key
            )
        )
    )


    if (
        not identity_consistency
        and
        validation[
            "rdkit_parse_success"
        ]
    ):

        exclusion_reasons.append(
            "IDENTITY_VERIFICATION_FAILED"
        )


    # --------------------------------------------------------------------------
    # Descriptor defaults
    # --------------------------------------------------------------------------

    descriptor_values = {

        "atom_count":
            np.nan,

        "heavy_atom_count":
            np.nan,

        "fragment_count":
            np.nan,

        "molecular_weight":
            np.nan,

        "clogp":
            np.nan,

        "tpsa":
            np.nan,

        "hbd":
            np.nan,

        "hba":
            np.nan,

        "rotatable_bonds":
            np.nan,

        "formal_charge":
            np.nan,

        "ring_count":
            np.nan,

        "aromatic_ring_count":
            np.nan,

        "fraction_csp3":
            np.nan,

        "qed":
            np.nan,

        "sa_score":
            np.nan,

        "sa_utility":
            np.nan
    }


    descriptor_calculation_success = (
        False
    )


    descriptors_finite = (
        False
    )


    # --------------------------------------------------------------------------
    # Descriptor calculation only after structural validation
    # --------------------------------------------------------------------------

    if (
        mol is not None
        and
        validation[
            "sanitization_success"
        ]
        and
        validation[
            "property_cache_success"
        ]
        and
        validation[
            "valence_validation_success"
        ]
    ):

        try:

            descriptor_values = (
                calculate_descriptors(
                    mol
                )
            )


            descriptor_calculation_success = (
                True
            )


            descriptors_finite = all(

                is_finite_number(
                    value
                )

                for value
                in descriptor_values.values()
            )


        except Exception:

            descriptor_calculation_success = (
                False
            )


            descriptors_finite = (
                False
            )


    # --------------------------------------------------------------------------
    # Empty molecule
    # --------------------------------------------------------------------------

    if descriptor_calculation_success:

        if int(
            descriptor_values[
                "atom_count"
            ]
        ) <= 0:

            exclusion_reasons.append(
                "EMPTY_MOLECULE"
            )


    # --------------------------------------------------------------------------
    # Fragment requirement
    # --------------------------------------------------------------------------

    if descriptor_calculation_success:

        fragment_count = int(
            descriptor_values[
                "fragment_count"
            ]
        )


        if fragment_count != 1:

            exclusion_reasons.append(
                "DISCONNECTED_FRAGMENTS"
            )


    # --------------------------------------------------------------------------
    # Heavy-atom model-feasibility range
    # --------------------------------------------------------------------------

    if descriptor_calculation_success:

        heavy_atom_count = int(
            descriptor_values[
                "heavy_atom_count"
            ]
        )


        if heavy_atom_count < (
            HEAVY_ATOM_MIN
        ):

            exclusion_reasons.append(
                "HEAVY_ATOM_COUNT_BELOW_MODEL_RANGE"
            )


        if heavy_atom_count > (
            HEAVY_ATOM_MAX
        ):

            exclusion_reasons.append(
                "HEAVY_ATOM_COUNT_ABOVE_MODEL_RANGE"
            )


    # --------------------------------------------------------------------------
    # Numerical validity
    # --------------------------------------------------------------------------

    if (
        descriptor_calculation_success
        and
        not descriptors_finite
    ):

        exclusion_reasons.append(
            "NONFINITE_DESCRIPTOR"
        )


    if not descriptor_calculation_success:

        if (
            validation[
                "rdkit_parse_success"
            ]
            and
            validation[
                "sanitization_success"
            ]
            and
            validation[
                "property_cache_success"
            ]
            and
            validation[
                "valence_validation_success"
            ]
        ):

            exclusion_reasons.append(
                "DESCRIPTOR_CALCULATION_FAILED"
            )


    # --------------------------------------------------------------------------
    # Novelty checks
    # --------------------------------------------------------------------------

    comparison_smiles = (
        recanonicalized_smiles
        if recanonicalized_smiles
        else
        phase19_smiles
    )


    exact_cycle1_overlap = bool(

        comparison_smiles
        in
        CYCLE1_CANONICAL_SET

        or

        (
            inchi_key
            and
            inchi_key
            in
            CYCLE1_INCHIKEY_SET
        )
    )


    exact_drugbank_overlap = bool(

        comparison_smiles
        in
        DRUGBANK_CANONICAL_SET

        or

        (
            inchi_key
            and
            inchi_key
            in
            DRUGBANK_INCHIKEY_SET
        )
    )


    novel_vs_cycle1 = (
        not exact_cycle1_overlap
    )


    novel_vs_drugbank = (
        not exact_drugbank_overlap
    )


    jointly_novel = (

        novel_vs_cycle1

        and

        novel_vs_drugbank
    )


    if NOVELTY_REQUIRED_FOR_ELIGIBILITY:

        if exact_cycle1_overlap:

            exclusion_reasons.append(
                "EXACT_CYCLE1_OVERLAP"
            )


        if exact_drugbank_overlap:

            exclusion_reasons.append(
                "EXACT_DRUGBANK_OVERLAP"
            )


    # --------------------------------------------------------------------------
    # Record
    # --------------------------------------------------------------------------

    record = {

        "candidate_id":
            candidate_id,

        "cycle":
            CYCLE,

        "phase19_canonical_smiles":
            phase19_smiles,

        "phase20_recanonicalized_smiles":
            recanonicalized_smiles,

        "inchi_key":
            inchi_key,

        "exact_text_identity":
            exact_text_identity,

        "identity_representation_mismatch":
            identity_representation_mismatch,

        "identity_consistency":
            identity_consistency,

        "rdkit_parse_success":
            validation[
                "rdkit_parse_success"
            ],

        "sanitization_success":
            validation[
                "sanitization_success"
            ],

        "property_cache_success":
            validation[
                "property_cache_success"
            ],

        "valence_validation_success":
            validation[
                "valence_validation_success"
            ],

        "descriptor_calculation_success":
            descriptor_calculation_success,

        **descriptor_values,

        "descriptors_finite":
            descriptors_finite,

        "exact_cycle1_overlap":
            exact_cycle1_overlap,

        "novel_vs_cycle1":
            novel_vs_cycle1,

        "exact_drugbank_overlap":
            exact_drugbank_overlap,

        "novel_vs_drugbank":
            novel_vs_drugbank,

        "jointly_novel":
            jointly_novel,

        "unexpected_cycle2_duplicate":
            False,

        "eligible":
            False,

        "exclusion_reasons":
            ""
    }


    # --------------------------------------------------------------------------
    # Preserve useful Phase-19 provenance
    # --------------------------------------------------------------------------

    provenance_columns = [

        "generation_index",

        "latent_id",

        "generation_branch",

        "seed_candidate_id",

        "seed_K1_utility",

        "seed_K1_uncertainty",

        "perturbation_scale",

        "interpolation_parent_1",

        "interpolation_parent_2",

        "interpolation_alpha",

        "decoder_sample_index",

        "decoder_seed",

        "decoder_temperature",

        "decoder_top_k"
    ]


    for column in provenance_columns:

        if column in phase19_df.columns:

            record[
                column
            ] = row[
                column
            ]


    record[
        "exclusion_reasons"
    ] = ";".join(
        sorted(
            set(
                exclusion_reasons
            )
        )
    )


    screening_records.append(
        record
    )


    if (
        (
            row_index
            +
            1
        )
        %
        100
        ==
        0

        or

        (
            row_index
            +
            1
        )
        ==
        len(
            phase19_df
        )
    ):

        print(
            f"[SCREEN] "
            f"{row_index + 1:04d}/"
            f"{len(phase19_df):04d} processed"
        )


screening_df = pd.DataFrame(
    screening_records
)


# ==============================================================================
# 20. INDEPENDENT WITHIN-CYCLE UNIQUENESS RECONFIRMATION
# ==============================================================================

valid_identity_mask = (

    screening_df[
        "phase20_recanonicalized_smiles"
    ]
    .astype(str)
    .str.len()
    >
    0
)


duplicate_mask = pd.Series(
    False,
    index=screening_df.index
)


duplicate_mask.loc[
    valid_identity_mask
] = (

    screening_df.loc[
        valid_identity_mask,
        "phase20_recanonicalized_smiles"
    ]
    .duplicated(
        keep=False
    )
)


screening_df[
    "unexpected_cycle2_duplicate"
] = (
    duplicate_mask
)


for index in screening_df.index[

    screening_df[
        "unexpected_cycle2_duplicate"
    ]

]:

    reasons = screening_df.at[
        index,
        "exclusion_reasons"
    ]


    reason_list = [

        reason

        for reason
        in str(
            reasons
        ).split(
            ";"
        )

        if reason
    ]


    if (
        "UNEXPECTED_CYCLE2_DUPLICATE"
        not in reason_list
    ):

        reason_list.append(
            "UNEXPECTED_CYCLE2_DUPLICATE"
        )


    screening_df.at[
        index,
        "exclusion_reasons"
    ] = ";".join(
        sorted(
            set(
                reason_list
            )
        )
    )


# ==============================================================================
# 21. FINAL ELIGIBILITY
# ==============================================================================

screening_df[
    "eligible"
] = (

    screening_df[
        "exclusion_reasons"
    ]
    .fillna(
        ""
    )
    .astype(str)
    .str.strip()
    ==
    ""
)


# ==============================================================================
# 22. AUTHORITATIVE SMILES PRESERVATION CHECK
# ==============================================================================

if not np.array_equal(

    screening_df[
        "phase19_canonical_smiles"
    ].astype(str).to_numpy(),

    phase19_df[
        "canonical_smiles"
    ].astype(str).to_numpy()
):

    raise RuntimeError(
        "Authoritative Phase-19 canonical SMILES "
        "were not preserved."
    )


# ==============================================================================
# 23. CANDIDATE-ID PRESERVATION CHECK
# ==============================================================================

PHASE20_ID_SET = set(
    screening_df[
        "candidate_id"
    ].astype(str)
)


IDS_PRESERVED = (

    PHASE19_ID_SET

    ==

    PHASE20_ID_SET
)


if not IDS_PRESERVED:

    lost_ids = sorted(
        PHASE19_ID_SET
        -
        PHASE20_ID_SET
    )


    new_ids = sorted(
        PHASE20_ID_SET
        -
        PHASE19_ID_SET
    )


    raise RuntimeError(
        "Candidate-ID conservation failed.\n"
        f"Lost IDs: {lost_ids[:20]}\n"
        f"Unexpected IDs: {new_ids[:20]}"
    )


if screening_df[
    "candidate_id"
].duplicated().any():

    raise RuntimeError(
        "Duplicate candidate IDs detected in Phase 20."
    )


# ==============================================================================
# 24. SPLIT ELIGIBLE / INELIGIBLE
# ==============================================================================

eligible_df = screening_df[
    screening_df[
        "eligible"
    ]
].copy()


ineligible_df = screening_df[
    ~screening_df[
        "eligible"
    ]
].copy()


eligible_df = eligible_df.reset_index(
    drop=True
)


ineligible_df = ineligible_df.reset_index(
    drop=True
)


# ==============================================================================
# 25. POPULATION CONSERVATION
# ==============================================================================

POPULATION_CONSERVED = (

    len(
        screening_df
    )
    ==
    len(
        eligible_df
    )
    +
    len(
        ineligible_df
    )
    ==
    EXPECTED_INPUT_COUNT
)


if not POPULATION_CONSERVED:

    raise RuntimeError(
        "Population conservation failed:\n"
        f"C2_unique={len(screening_df)}\n"
        f"C2_eligible={len(eligible_df)}\n"
        f"C2_ineligible={len(ineligible_df)}"
    )


# ==============================================================================
# 26. EVERY INELIGIBLE RECORD MUST HAVE A REASON
# ==============================================================================

if len(
    ineligible_df
) > 0:

    missing_reason_mask = (

        ineligible_df[
            "exclusion_reasons"
        ]
        .fillna(
            ""
        )
        .astype(str)
        .str.strip()
        ==
        ""
    )


    if missing_reason_mask.any():

        raise RuntimeError(
            "At least one ineligible candidate has "
            "no exclusion reason."
        )


# ==============================================================================
# 27. SAVE FULL SCREENING TABLE
# ==============================================================================

screening_df.to_csv(
    PHASE20_SCREENING_FILE,
    index=False
)


eligible_df.to_csv(
    PHASE20_ELIGIBLE_FILE,
    index=False
)


ineligible_df.to_csv(
    PHASE20_INELIGIBLE_FILE,
    index=False
)


# ==============================================================================
# 28. SAVE PHYSICOCHEMICAL DESCRIPTORS
# ==============================================================================

descriptor_columns = [

    "candidate_id",

    "phase19_canonical_smiles",

    "phase20_recanonicalized_smiles",

    "atom_count",

    "heavy_atom_count",

    "fragment_count",

    "molecular_weight",

    "clogp",

    "tpsa",

    "hbd",

    "hba",

    "rotatable_bonds",

    "formal_charge",

    "ring_count",

    "aromatic_ring_count",

    "fraction_csp3",

    "qed",

    "sa_score",

    "sa_utility",

    "descriptors_finite",

    "eligible"
]


screening_df[
    descriptor_columns
].to_csv(
    PHASE20_DESCRIPTOR_FILE,
    index=False
)


# ==============================================================================
# 29. NOVELTY SUMMARY
# ==============================================================================

novelty_summary_df = pd.DataFrame(
    [

        {

            "metric":
                "total_candidates",

            "value":
                len(
                    screening_df
                )
        },

        {

            "metric":
                "exact_cycle1_overlap",

            "value":
                int(
                    screening_df[
                        "exact_cycle1_overlap"
                    ].sum()
                )
        },

        {

            "metric":
                "novel_vs_cycle1",

            "value":
                int(
                    screening_df[
                        "novel_vs_cycle1"
                    ].sum()
                )
        },

        {

            "metric":
                "exact_drugbank_overlap",

            "value":
                int(
                    screening_df[
                        "exact_drugbank_overlap"
                    ].sum()
                )
        },

        {

            "metric":
                "novel_vs_drugbank",

            "value":
                int(
                    screening_df[
                        "novel_vs_drugbank"
                    ].sum()
                )
        },

        {

            "metric":
                "jointly_novel",

            "value":
                int(
                    screening_df[
                        "jointly_novel"
                    ].sum()
                )
        }
    ]
)


novelty_summary_df.to_csv(
    PHASE20_NOVELTY_SUMMARY_FILE,
    index=False
)


# ==============================================================================
# 30. DESCRIPTOR DISTRIBUTION SUMMARY
# ==============================================================================

SUMMARY_DESCRIPTORS = [

    "molecular_weight",

    "heavy_atom_count",

    "clogp",

    "tpsa",

    "hbd",

    "hba",

    "rotatable_bonds",

    "formal_charge",

    "ring_count",

    "aromatic_ring_count",

    "fraction_csp3",

    "qed",

    "sa_score",

    "sa_utility"
]


descriptor_summary_records = []


for pool_name, pool_df in [

    (
        "ALL",
        screening_df
    ),

    (
        "ELIGIBLE",
        eligible_df
    )
]:

    for descriptor in (
        SUMMARY_DESCRIPTORS
    ):

        values = pd.to_numeric(
            pool_df[
                descriptor
            ],
            errors="coerce"
        )


        values = values[
            np.isfinite(
                values
            )
        ]


        if len(
            values
        ) == 0:

            record = {

                "pool":
                    pool_name,

                "descriptor":
                    descriptor,

                "n":
                    0,

                "minimum":
                    np.nan,

                "q25":
                    np.nan,

                "median":
                    np.nan,

                "mean":
                    np.nan,

                "q75":
                    np.nan,

                "maximum":
                    np.nan,

                "std":
                    np.nan
            }


        else:

            record = {

                "pool":
                    pool_name,

                "descriptor":
                    descriptor,

                "n":
                    int(
                        len(
                            values
                        )
                    ),

                "minimum":
                    float(
                        values.min()
                    ),

                "q25":
                    float(
                        values.quantile(
                            0.25
                        )
                    ),

                "median":
                    float(
                        values.median()
                    ),

                "mean":
                    float(
                        values.mean()
                    ),

                "q75":
                    float(
                        values.quantile(
                            0.75
                        )
                    ),

                "maximum":
                    float(
                        values.max()
                    ),

                "std":
                    float(
                        values.std(
                            ddof=1
                        )
                    )
                    if len(
                        values
                    ) > 1
                    else
                    0.0
            }


        descriptor_summary_records.append(
            record
        )


descriptor_summary_df = pd.DataFrame(
    descriptor_summary_records
)


descriptor_summary_df.to_csv(
    PHASE20_DESCRIPTOR_SUMMARY_FILE,
    index=False
)


# ==============================================================================
# 31. EXCLUSION SUMMARY
# ==============================================================================

exclusion_counter = Counter()


for reasons in (
    ineligible_df[
        "exclusion_reasons"
    ]
    .fillna(
        ""
    )
    .astype(str)
):

    for reason in reasons.split(
        ";"
    ):

        reason = reason.strip()


        if reason:

            exclusion_counter[
                reason
            ] += 1


exclusion_summary_records = []


for reason, count in sorted(
    exclusion_counter.items(),
    key=lambda item:
        (
            -item[
                1
            ],
            item[
                0
            ]
        )
):

    exclusion_summary_records.append(
        {

            "exclusion_reason":
                reason,

            "candidate_count":
                int(
                    count
                ),

            "fraction_of_C2_unique":
                float(
                    count
                    /
                    EXPECTED_INPUT_COUNT
                )
        }
    )


exclusion_summary_df = pd.DataFrame(
    exclusion_summary_records,
    columns=[
        "exclusion_reason",
        "candidate_count",
        "fraction_of_C2_unique"
    ]
)


exclusion_summary_df.to_csv(
    PHASE20_EXCLUSION_SUMMARY_FILE,
    index=False
)


# ==============================================================================
# 32. CHECKPOINT HASHES — AFTER PHASE 20
# ==============================================================================

HASHES_AFTER = {

    name:
        sha256_file(
            path
        )

    for name, path
    in FROZEN_ARTIFACTS.items()
}


ARTIFACT_INTEGRITY = {

    name:
        (
            HASHES_BEFORE[
                name
            ]
            ==
            HASHES_AFTER[
                name
            ]
        )

    for name
    in FROZEN_ARTIFACTS
}


ALL_FROZEN_ARTIFACTS_UNCHANGED = all(
    ARTIFACT_INTEGRITY.values()
)


if not ALL_FROZEN_ARTIFACTS_UNCHANGED:

    failed = [

        name

        for name, status
        in ARTIFACT_INTEGRITY.items()

        if not status
    ]


    raise RuntimeError(
        "Frozen artifact integrity failure:\n"
        +
        "\n".join(
            failed
        )
    )


# ==============================================================================
# 33. PHASE-20 CONFIGURATION FILE
# ==============================================================================

phase20_config = {

    "phase":
        (
            "Phase 20 — Cycle-2 Chemical Eligibility, "
            "Uniqueness, Novelty, and Physicochemical Screening"
        ),

    "module":
        106,

    "created_at":
        datetime.now().isoformat(),

    "cycle":
        CYCLE,

    "input_file":
        PHASE19_C2_UNIQUE_FILE.name,

    "expected_input_population":
        EXPECTED_INPUT_COUNT,

    "heavy_atom_minimum":
        HEAVY_ATOM_MIN,

    "heavy_atom_maximum":
        HEAVY_ATOM_MAX,

    "heavy_atom_constraint_type":
        "MODEL_FEASIBILITY_CONSTRAINT",

    "fragment_requirement":
        1,

    "preserve_phase19_candidate_ids":
        True,

    "preserve_phase19_canonical_smiles":
        PRESERVE_PHASE19_CANONICAL_SMILES,

    "identity_policy":
        (
            "Phase-19 canonical SMILES retained as authoritative; "
            "RDKit canonical isomeric SMILES recalculated for audit; "
            "text mismatch alone is not exclusion if molecular identity "
            "remains verifiable."
        ),

    "qed_implementation":
        "rdkit.Chem.QED.qed",

    "sa_implementation":
        SA_SCORER_SOURCE,

    "sa_utility_transformation":
        "clip((10 - SA_score) / 9, 0, 1)",

    "sa_utility_transformation_source":
        (
            "Frozen Cycle-1 utility transformation; "
            "verified against accepted Phase-16 SA/SAUtility values."
        ),

    "cycle1_novelty_reference":
        PHASE14_C1_UNIQUE_FILE.name,

    "drugbank_novelty_reference":
        DRUGBANK_REFERENCE_FILE.name,

    "novelty_required_for_eligibility":
        NOVELTY_REQUIRED_FOR_ELIGIBILITY,

    "checkpoint_integrity_policy":
        "SHA-256 before == SHA-256 after",

    "optimizer_created":
        False,

    "backward_pass":
        False,

    "gradient_update":
        False,

    "encoder_update":
        False,

    "predictor_update":
        False,

    "decoder_update":
        False,

    "ewc_applied":
        False,

    "fisher_recalculated":
        False,

    "frozen_artifacts":
        {

            name:
                path.name

            for name, path
            in FROZEN_ARTIFACTS.items()
        }
}


with open(
    PHASE20_CONFIG_FILE,
    "w",
    encoding="utf-8"
) as handle:

    json.dump(
        phase20_config,
        handle,
        indent=4,
        ensure_ascii=False
    )


# ==============================================================================
# 34. PHASE-20 COMPLETION CHECKS
# ==============================================================================

completion_checks = {

    "Phase-19 C2_unique loaded":
        len(
            phase19_df
        )
        ==
        EXPECTED_INPUT_COUNT,

    "Expected input population verified":
        len(
            phase19_df
        )
        ==
        969,

    "Candidate IDs preserved exactly":
        IDS_PRESERVED,

    "Candidate IDs remain unique":
        not screening_df[
            "candidate_id"
        ].duplicated().any(),

    "Phase-19 canonical SMILES preserved":
        np.array_equal(

            screening_df[
                "phase19_canonical_smiles"
            ].astype(str).to_numpy(),

            phase19_df[
                "canonical_smiles"
            ].astype(str).to_numpy()
        ),

    "All candidates screened":
        len(
            screening_df
        )
        ==
        EXPECTED_INPUT_COUNT,

    "RDKit reparsing performed":
        "rdkit_parse_success"
        in
        screening_df.columns,

    "Strict sanitization performed":
        "sanitization_success"
        in
        screening_df.columns,

    "Property-cache validation performed":
        "property_cache_success"
        in
        screening_df.columns,

    "Valence integrity evaluated":
        "valence_validation_success"
        in
        screening_df.columns,

    "Fragment counts calculated":
        screening_df[
            "fragment_count"
        ].notna().any(),

    "Heavy atom counts calculated":
        screening_df[
            "heavy_atom_count"
        ].notna().any(),

    "MW calculated":
        screening_df[
            "molecular_weight"
        ].notna().any(),

    "cLogP calculated":
        screening_df[
            "clogp"
        ].notna().any(),

    "TPSA calculated":
        screening_df[
            "tpsa"
        ].notna().any(),

    "HBD calculated":
        screening_df[
            "hbd"
        ].notna().any(),

    "HBA calculated":
        screening_df[
            "hba"
        ].notna().any(),

    "Rotatable bonds calculated":
        screening_df[
            "rotatable_bonds"
        ].notna().any(),

    "Formal charge calculated":
        screening_df[
            "formal_charge"
        ].notna().any(),

    "Ring count calculated":
        screening_df[
            "ring_count"
        ].notna().any(),

    "Aromatic ring count calculated":
        screening_df[
            "aromatic_ring_count"
        ].notna().any(),

    "Fraction Csp3 calculated":
        screening_df[
            "fraction_csp3"
        ].notna().any(),

    "QED calculated":
        screening_df[
            "qed"
        ].notna().any(),

    "SA score calculated":
        screening_df[
            "sa_score"
        ].notna().any(),

    "Cycle-1 SAUtility transform reused":
        bool(
            np.isclose(
                SA_SELF_TEST_CALCULATED,
                SA_SELF_TEST_EXPECTED,
                atol=1e-6
            )
        ),

    "Numerical finiteness evaluated":
        "descriptors_finite"
        in
        screening_df.columns,

    "Within-Cycle-2 uniqueness reconfirmed":
        "unexpected_cycle2_duplicate"
        in
        screening_df.columns,

    "Cycle-1 exact overlap independently reconfirmed":
        "exact_cycle1_overlap"
        in
        screening_df.columns,

    "DrugBank exact overlap independently reconfirmed":
        "exact_drugbank_overlap"
        in
        screening_df.columns,

    "Novelty annotations generated":
        (
            "novel_vs_cycle1"
            in
            screening_df.columns
            and
            "novel_vs_drugbank"
            in
            screening_df.columns
            and
            "jointly_novel"
            in
            screening_df.columns
        ),

    "Every ineligible candidate has exclusion reason":
        bool(
            len(
                ineligible_df
            )
            ==
            0
            or
            (
                ineligible_df[
                    "exclusion_reasons"
                ]
                .fillna(
                    ""
                )
                .astype(str)
                .str.len()
                >
                0
            ).all()
        ),

    "Eligible candidates saved":
        PHASE20_ELIGIBLE_FILE.exists(),

    "Ineligible candidates saved":
        PHASE20_INELIGIBLE_FILE.exists(),

    "Complete screening table saved":
        PHASE20_SCREENING_FILE.exists(),

    "Population conservation satisfied":
        POPULATION_CONSERVED,

    "No Phase-19 candidate lost":
        IDS_PRESERVED,

    "No candidate re-indexing":
        PHASE19_ID_SET
        ==
        PHASE20_ID_SET,

    "K0 encoder unchanged":
        ARTIFACT_INTEGRITY[
            "K0_encoder_thetaE_star"
        ],

    "K0 decoder unchanged":
        ARTIFACT_INTEGRITY[
            "K0_decoder_thetaD_star"
        ],

    "K0 predictor unchanged":
        ARTIFACT_INTEGRITY[
            "K0_predictor_thetaP_star"
        ],

    "K1 encoder unchanged":
        ARTIFACT_INTEGRITY[
            "K1_encoder_thetaE_1"
        ],

    "K1 predictor unchanged":
        ARTIFACT_INTEGRITY[
            "K1_predictor_thetaP_1"
        ],

    "Encoder Fisher unchanged":
        ARTIFACT_INTEGRITY[
            "F_E"
        ],

    "Predictor Fisher unchanged":
        ARTIFACT_INTEGRITY[
            "F_P"
        ],

    "Phase-13 parameter reference unchanged":
        ARTIFACT_INTEGRITY[
            "Phase13_parameter_reference"
        ],

    "No optimizer created":
        True,

    "No backward pass":
        True,

    "No gradient update":
        True,

    "No encoder update":
        True,

    "No predictor update":
        True,

    "No decoder update":
        True,

    "No EWC applied":
        True,

    "No Fisher recalculation":
        True
}


ALL_COMPLETION_CHECKS_PASS = all(

    bool(
        value
    )

    for value
    in completion_checks.values()
)


# ==============================================================================
# 35. QC SUMMARY
# ==============================================================================

qc_summary = {

    "phase":
        20,

    "module":
        106,

    "created_at":
        datetime.now().isoformat(),

    "input_population_C2_unique":
        int(
            len(
                screening_df
            )
        ),

    "eligible_population_C2_eligible":
        int(
            len(
                eligible_df
            )
        ),

    "ineligible_population_C2_ineligible":
        int(
            len(
                ineligible_df
            )
        ),

    "population_conserved":
        bool(
            POPULATION_CONSERVED
        ),

    "candidate_ids_preserved":
        bool(
            IDS_PRESERVED
        ),

    "identity_representation_mismatch_count":
        int(
            screening_df[
                "identity_representation_mismatch"
            ].sum()
        ),

    "rdkit_parse_success_count":
        int(
            screening_df[
                "rdkit_parse_success"
            ].sum()
        ),

    "sanitization_success_count":
        int(
            screening_df[
                "sanitization_success"
            ].sum()
        ),

    "property_cache_success_count":
        int(
            screening_df[
                "property_cache_success"
            ].sum()
        ),

    "valence_validation_success_count":
        int(
            screening_df[
                "valence_validation_success"
            ].sum()
        ),

    "finite_descriptor_count":
        int(
            screening_df[
                "descriptors_finite"
            ].sum()
        ),

    "unexpected_cycle2_duplicate_count":
        int(
            screening_df[
                "unexpected_cycle2_duplicate"
            ].sum()
        ),

    "exact_cycle1_overlap_count":
        int(
            screening_df[
                "exact_cycle1_overlap"
            ].sum()
        ),

    "novel_vs_cycle1_count":
        int(
            screening_df[
                "novel_vs_cycle1"
            ].sum()
        ),

    "exact_drugbank_overlap_count":
        int(
            screening_df[
                "exact_drugbank_overlap"
            ].sum()
        ),

    "novel_vs_drugbank_count":
        int(
            screening_df[
                "novel_vs_drugbank"
            ].sum()
        ),

    "jointly_novel_count":
        int(
            screening_df[
                "jointly_novel"
            ].sum()
        ),

    "eligibility_rate":
        float(
            len(
                eligible_df
            )
            /
            len(
                screening_df
            )
        ),

    "sa_scorer_available":
        bool(
            SA_SCORER_AVAILABLE
        ),

    "sa_scorer_source":
        SA_SCORER_SOURCE,

    "sa_utility_transformation":
        "clip((10 - SA_score) / 9, 0, 1)",

    "novelty_required_for_eligibility":
        bool(
            NOVELTY_REQUIRED_FOR_ELIGIBILITY
        ),

    "frozen_artifacts_unchanged":
        bool(
            ALL_FROZEN_ARTIFACTS_UNCHANGED
        ),

    "completion_checks":
        completion_checks
}


with open(
    PHASE20_QC_JSON_FILE,
    "w",
    encoding="utf-8"
) as handle:

    json.dump(
        qc_summary,
        handle,
        indent=4,
        ensure_ascii=False,
        default=str
    )


# ------------------------------------------------------------------------------
# CSV QC summary
# ------------------------------------------------------------------------------

qc_records = []


for metric, value in (
    qc_summary.items()
):

    if metric == (
        "completion_checks"
    ):

        for check_name, status in (
            value.items()
        ):

            qc_records.append(
                {

                    "metric":
                        f"completion.{check_name}",

                    "value":
                        status
                }
            )


    else:

        qc_records.append(
            {

                "metric":
                    metric,

                "value":
                    value
            }
        )


pd.DataFrame(
    qc_records
).to_csv(
    PHASE20_QC_CSV_FILE,
    index=False
)


# ==============================================================================
# 36. DISPLAY EXCLUSION SUMMARY
# ==============================================================================

print(
    "\n"
    +
    "=" * 160
)

print(
    "PHASE-20 EXCLUSION SUMMARY"
)

print(
    "=" * 160
)


if len(
    exclusion_summary_df
) == 0:

    print(
        "No exclusions."
    )


else:

    print(
        exclusion_summary_df.to_string(
            index=False
        )
    )


# ==============================================================================
# 37. DISPLAY NOVELTY SUMMARY
# ==============================================================================

print(
    "\n"
    +
    "=" * 160
)

print(
    "PHASE-20 NOVELTY SUMMARY"
)

print(
    "=" * 160
)


print(
    novelty_summary_df.to_string(
        index=False
    )
)


# ==============================================================================
# 38. DISPLAY ELIGIBLE DESCRIPTOR SUMMARY
# ==============================================================================

print(
    "\n"
    +
    "=" * 160
)

print(
    "ELIGIBLE CYCLE-2 DESCRIPTOR SUMMARY"
)

print(
    "=" * 160
)


eligible_descriptor_display = (
    descriptor_summary_df[
        descriptor_summary_df[
            "pool"
        ]
        ==
        "ELIGIBLE"
    ]
)


print(
    eligible_descriptor_display.to_string(
        index=False
    )
)


# ==============================================================================
# 39. PHASE-20 COMPLETION VALIDATION
# ==============================================================================

print(
    "\n"
    +
    "=" * 160
)

print(
    "PHASE 20 COMPLETION VALIDATION"
)

print(
    "=" * 160
)


for check_name, status in (
    completion_checks.items()
):

    print(
        f"[{'PASS' if bool(status) else 'FAIL':<4}] "
        f"{check_name}"
    )


# ==============================================================================
# 40. PHASE-20 FINAL SUMMARY
# ==============================================================================

print(
    "\n"
    +
    "=" * 160
)

print(
    "PHASE 20 FINAL SUMMARY"
)

print(
    "=" * 160
)


print(
    f"Cycle                                      : "
    f"{CYCLE}"
)


print(
    f"Input C2_unique                            : "
    f"{len(screening_df)}"
)


print(
    "\nStructural / Chemical QC:"
)


print(
    f"  RDKit Parse Success                     : "
    f"{int(screening_df['rdkit_parse_success'].sum())}"
)


print(
    f"  Sanitization Success                    : "
    f"{int(screening_df['sanitization_success'].sum())}"
)


print(
    f"  Property-Cache Success                  : "
    f"{int(screening_df['property_cache_success'].sum())}"
)


print(
    f"  Valence Validation Success              : "
    f"{int(screening_df['valence_validation_success'].sum())}"
)


print(
    f"  Finite Descriptor Sets                  : "
    f"{int(screening_df['descriptors_finite'].sum())}"
)


print(
    "\nIdentity:"
)


print(
    f"  Exact Canonical Text Match              : "
    f"{int(screening_df['exact_text_identity'].sum())}"
)


print(
    f"  Representation Mismatches               : "
    f"{int(screening_df['identity_representation_mismatch'].sum())}"
)


print(
    f"  Verified Molecular Identity             : "
    f"{int(screening_df['identity_consistency'].sum())}"
)


print(
    "\nNovelty:"
)


print(
    f"  Exact Cycle-1 Overlap                   : "
    f"{int(screening_df['exact_cycle1_overlap'].sum())}"
)


print(
    f"  Novel vs Cycle-1                        : "
    f"{int(screening_df['novel_vs_cycle1'].sum())}"
)


print(
    f"  Exact DrugBank Overlap                  : "
    f"{int(screening_df['exact_drugbank_overlap'].sum())}"
)


print(
    f"  Novel vs DrugBank                       : "
    f"{int(screening_df['novel_vs_drugbank'].sum())}"
)


print(
    f"  Jointly Novel                           : "
    f"{int(screening_df['jointly_novel'].sum())}"
)


print(
    "\nEligibility:"
)


print(
    f"  C2_eligible                             : "
    f"{len(eligible_df)}"
)


print(
    f"  C2_ineligible                           : "
    f"{len(ineligible_df)}"
)


print(
    f"  Eligibility Rate                        : "
    f"{len(eligible_df) / len(screening_df):.2%}"
)


print(
    f"  Population Conserved                    : "
    f"{POPULATION_CONSERVED}"
)


print(
    "\nDescriptor Policy:"
)


print(
    f"  QED Implementation                      : "
    f"rdkit.Chem.QED.qed"
)


print(
    f"  SA Implementation                       : "
    f"{SA_SCORER_SOURCE}"
)


print(
    "  SAUtility                               : "
    "clip((10 - SA) / 9, 0, 1)"
)


print(
    "\nModel / Knowledge Integrity:"
)


for artifact_name, status in (
    ARTIFACT_INTEGRITY.items()
):

    print(
        f"  {artifact_name:<40}: "
        f"{status}"
    )


print(
    "\nAdaptive-Learning Status:"
)


print(
    "  Optimizer Created                       : False"
)

print(
    "  Backward Pass                           : False"
)

print(
    "  Gradient Update                         : False"
)

print(
    "  Encoder Update                          : False"
)

print(
    "  Predictor Update                        : False"
)

print(
    "  Decoder Update                          : False"
)

print(
    "  EWC Applied                             : False"
)

print(
    "  Fisher Recalculated                     : False"
)


print(
    "\nSaved Files:"
)


for path in [

    PHASE20_CONFIG_FILE,

    PHASE20_SCREENING_FILE,

    PHASE20_ELIGIBLE_FILE,

    PHASE20_INELIGIBLE_FILE,

    PHASE20_DESCRIPTOR_FILE,

    PHASE20_NOVELTY_SUMMARY_FILE,

    PHASE20_DESCRIPTOR_SUMMARY_FILE,

    PHASE20_EXCLUSION_SUMMARY_FILE,

    PHASE20_QC_CSV_FILE,

    PHASE20_QC_JSON_FILE
]:

    print(
        f"  {path.name}"
    )


print(
    "\n"
    +
    "-" * 160
)


if ALL_COMPLETION_CHECKS_PASS:

    print(
        "[STATUS] PHASE 20 COMPLETED SUCCESSFULLY."
    )


    print(
        "[PASS] All 969 Phase-19 Cycle-2 candidates "
        "were explicitly classified."
    )


    print(
        "[PASS] Phase-19 candidate IDs and authoritative "
        "canonical SMILES were preserved."
    )


    print(
        "[PASS] RDKit parse, sanitization, property-cache "
        "and valence integrity were audited."
    )


    print(
        "[PASS] The established 4–80 heavy-atom "
        "model-feasibility constraint was applied."
    )


    print(
        "[PASS] Physicochemical descriptors, QED, SA and "
        "Cycle-1-consistent SAUtility were calculated."
    )


    print(
        "[PASS] Cycle-1 and DrugBank exact novelty were "
        "independently re-evaluated."
    )


    print(
        "[PASS] Eligible and ineligible populations satisfy "
        "exact population conservation."
    )


    print(
        "[PASS] K0, K1 and Fisher artifacts remained "
        "cryptographically unchanged."
    )


    print(
        "[IMPORTANT] Phase 20 establishes chemical / "
        "computational eligibility only."
    )


    print(
        "[IMPORTANT] Eligibility does NOT imply biological "
        "activity, binding affinity, therapeutic efficacy, "
        "experimental stability or confirmed synthesizability."
    )


    print(
        "[READY] C2_eligible may proceed to Phase 21 — "
        "Multi-Objective Evaluation, MC-Dropout Utility "
        "Uncertainty, UCB Acquisition and Diversity-Aware "
        "Feedback Selection."
    )


else:

    print(
        "[STATUS] PHASE 20 REQUIRES REVIEW."
    )


    print(
        "[ACTION] Do NOT begin Phase 21 until every failed "
        "Phase-20 completion check is resolved."
    )


print(
    "=" * 160
)

PHASE 20 — CYCLE-2 CHEMICAL ELIGIBILITY, UNIQUENESS, NOVELTY, AND PHYSICOCHEMICAL SCREENING

[CONFIG] Cycle                          : 2
[CONFIG] Expected C2_unique             : 969
[CONFIG] Heavy-Atom Feasibility         : 4-80
[CONFIG] Novelty Required for Eligibility: True
[CONFIG] SA Implementation              : rdkit.Contrib.SA_Score.sascorer
[CONFIG] SAUtility                      : clip((10 - SA) / 9, 0, 1)

[PASS] Phase-19 C2_unique loaded: 969
[PASS] Candidate IDs present and unique.


[18:36:56] Invalid InChI prefix in generating InChI Key
[18:36:56] Invalid InChI prefix in generating InChI Key



[PASS] Cycle-1 canonical reference : 846
[PASS] DrugBank canonical reference: 789

PHASE-20 CYCLE-2 CHEMICAL SCREENING
[SCREEN] 0100/0969 processed
[SCREEN] 0200/0969 processed
[SCREEN] 0300/0969 processed
[SCREEN] 0400/0969 processed
[SCREEN] 0500/0969 processed
[SCREEN] 0600/0969 processed
[SCREEN] 0700/0969 processed
[SCREEN] 0800/0969 processed
[SCREEN] 0900/0969 processed
[SCREEN] 0969/0969 processed

PHASE-20 EXCLUSION SUMMARY
    exclusion_reason  candidate_count  fraction_of_C2_unique
EXACT_CYCLE1_OVERLAP                2               0.002064

PHASE-20 NOVELTY SUMMARY
                metric  value
      total_candidates    969
  exact_cycle1_overlap      2
       novel_vs_cycle1    967
exact_drugbank_overlap      0
     novel_vs_drugbank    969
         jointly_novel    967

ELIGIBLE CYCLE-2 DESCRIPTOR SUMMARY
    pool          descriptor   n   minimum        q25     median       mean        q75    maximum       std
ELIGIBLE    molecular_weight 967 57.052000 155.197000 200.1